In [2]:

import pandas as pd
import re
from pathlib import Path

# --------------------------------------------------
# 1. Input file
# --------------------------------------------------
file_path = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Digestive_diseases\digestive_surgical_episodes.csv"
)

# --------------------------------------------------
# 2. Read dataset
# --------------------------------------------------
df = pd.read_csv(file_path, encoding="utf-8-sig")

# Persian/Arabic-script character pattern
persian_pattern = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")

# --------------------------------------------------
# 3. Find columns containing Persian values
# --------------------------------------------------
persian_columns = []

for col in df.columns:
    if df[col].dropna().astype(str).str.contains(
        persian_pattern, na=False
    ).any():
        persian_columns.append(col)

print("\n" + "=" * 80)
print("EXACT DISTINCT PERSIAN VALUES BY COLUMN")
print("=" * 80)

# --------------------------------------------------
# 4. Print every distinct Persian value and frequency
# --------------------------------------------------
output_lines = []

for col in persian_columns:
    # Keep only values that actually contain Persian text
    values = df[col].dropna().astype(str).str.strip()

    persian_values = values[
        values.str.contains(persian_pattern, na=False)
    ]

    counts = persian_values.value_counts()

    section = [
        "\n" + "=" * 80,
        f"COLUMN: {col}",
        f"Number of distinct Persian values: {len(counts)}",
        "-" * 80
    ]

    for value, frequency in counts.items():
        section.append(f"{value}\tFREQUENCY: {frequency}")

    section_text = "\n".join(section)

    print(section_text)
    output_lines.append(section_text)

# --------------------------------------------------
# 5. Save the exact output to a text file
# --------------------------------------------------
output_file = file_path.with_name(
    "exact_persian_column_values.txt"
)

output_file.write_text(
    "\n".join(output_lines),
    encoding="utf-8-sig"
)

print("\n" + "=" * 80)
print(f"Total columns containing Persian: {len(persian_columns)}")
print(f"Output saved to: {output_file}")
print("=" * 80)



EXACT DISTINCT PERSIAN VALUES BY COLUMN

COLUMN: specialty
Number of distinct Persian values: 66
--------------------------------------------------------------------------------
متخصص جراحی عمومی	FREQUENCY: 8539
دکترای پزشکی	FREQUENCY: 4291
متخصص جراحی کلیه و مجاری ادراری تناسلی	FREQUENCY: 856
دانشجوی تخصص جراحی عمومی	FREQUENCY: 303
متخصص بیماریهای داخلی	FREQUENCY: 240
متخصص طب اورژانس	FREQUENCY: 218
ارولوژی	FREQUENCY: 171
فوق تخصص جراحی اطفال	FREQUENCY: 99
فلوشیپ جراحی کولورکتال	FREQUENCY: 67
فوق تخصص جراحی قفسه صدری	FREQUENCY: 65
فوق تخصص گوارش و کبد بالغین	FREQUENCY: 48
فلوشیپ آندویورولوژی	FREQUENCY: 42
فوق تخصص جراحی عروق	FREQUENCY: 39
دانشجوی تخصص جراحی کلیه و مجاری ادراری تناسلی	FREQUENCY: 37
فلوشیپ پیوند کلیه	FREQUENCY: 36
متخصص بیماریهای عفونی و گرمسیری	FREQUENCY: 33
فلوشیپ پیوند کبد	FREQUENCY: 33
متخصص زنان و زایمان	FREQUENCY: 29
فلوشیپ جراحی عمومی درون بین- لاپاراسکوپی	FREQUENCY: 26
متخصص کودکان	FREQUENCY: 14
متخصص بیماریهای قلب و عروق	FREQUENCY: 12
فلوشیپ پیوند کلیه و اوروا

In [3]:

import pandas as pd
from pathlib import Path

# ============================================================
# 1. INPUT AND OUTPUT FILES
# ============================================================

input_file = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Digestive_diseases\digestive_surgical_episodes.csv"
)

output_file = input_file.with_name(
    "digestive_surgical_episodes_english.csv"
)

# ============================================================
# 2. READ ORIGINAL DATA
# ============================================================

df = pd.read_csv(input_file, encoding="utf-8-sig")

print(f"Original dataset: {df.shape[0]:,} rows, "
      f"{df.shape[1]} columns")

# ============================================================
# 3. EXPLICIT PERSIAN-TO-ENGLISH TRANSLATION DICTIONARIES
# ============================================================

translation_maps = {

    # --------------------------------------------------------
    # A. MEDICAL SPECIALTY
    # --------------------------------------------------------
    "specialty": {
        "متخصص جراحی عمومی":
            "General Surgery Specialist",

        "دکترای پزشکی":
            "Doctor of Medicine",

        "متخصص جراحی کلیه و مجاری ادراری تناسلی":
            "Urology Specialist",

        "دانشجوی تخصص جراحی عمومی":
            "General Surgery Resident",

        "متخصص بیماریهای داخلی":
            "Internal Medicine Specialist",

        "متخصص طب اورژانس":
            "Emergency Medicine Specialist",

        "ارولوژی":
            "Urology",

        "فوق تخصص جراحی اطفال":
            "Pediatric Surgery Subspecialist",

        "فلوشیپ جراحی کولورکتال":
            "Colorectal Surgery Fellow",

        "فوق تخصص جراحی قفسه صدری":
            "Thoracic Surgery Subspecialist",

        "فوق تخصص گوارش و کبد بالغین":
            "Adult Gastroenterology and Hepatology Subspecialist",

        "فلوشیپ آندویورولوژی":
            "Endourology Fellow",

        "فوق تخصص جراحی عروق":
            "Vascular Surgery Subspecialist",

        "دانشجوی تخصص جراحی کلیه و مجاری ادراری تناسلی":
            "Urology Resident",

        "فلوشیپ پیوند کلیه":
            "Kidney Transplantation Fellow",

        "متخصص بیماریهای عفونی و گرمسیری":
            "Infectious and Tropical Diseases Specialist",

        "فلوشیپ پیوند کبد":
            "Liver Transplantation Fellow",

        "متخصص زنان و زایمان":
            "Obstetrics and Gynecology Specialist",

        "فلوشیپ جراحی عمومی درون بین- لاپاراسکوپی":
            "General Surgery and Laparoscopic Surgery Fellow",

        "متخصص کودکان":
            "Pediatrics Specialist",

        "متخصص بیماریهای قلب و عروق":
            "Cardiology Specialist",

        "فلوشیپ پیوند کلیه و اوروانکولوژی":
            "Kidney Transplantation and Urologic Oncology Fellow",

        "متخصص ارتوپدی":
            "Orthopedic Surgery Specialist",

        "فوق تخصص جراحی قلب و عروق":
            "Cardiovascular Surgery Subspecialist",

        "دانشجوی تخصص بیماریهای داخلی":
            "Internal Medicine Resident",

        "جراحی عمومی (دستیاری تخصصی بالینی)":
            "General Surgery Clinical Resident",

        "متخصص بیهوشی":
            "Anesthesiology Specialist",

        "متخصص گوش و حلق و بینی و جراحی سر و گردن":
            "Otolaryngology and Head and Neck Surgery Specialist",

        "فوق تخصص گوارش اطفال":
            "Pediatric Gastroenterology Subspecialist",

        "دانشجوی تخصص طب اورژانس":
            "Emergency Medicine Resident",

        "متخصص بیماریهای مغز و اعصاب":
            "Neurology Specialist",

        "متخصص رادیولوژی":
            "Radiology Specialist",

        "فلوشیپ جراحی پستان":
            "Breast Surgery Fellow",

        "فوق تخصص روماتولوژی":
            "Rheumatology Subspecialist",

        "فوق تخصص بیماریهای ریه":
            "Pulmonology Subspecialist",

        "جراحی کلیه و مجاری ادراری تناسلی(دستیاری تخصصی بالینی)":
            "Urology Clinical Resident",

        "فلوشیپ جراحی عروق":
            "Vascular Surgery Fellow",

        "فوق تخصص کلیه اطفال":
            "Pediatric Nephrology Subspecialist",

        "فلوشیپ جراحی سرطان":
            "Surgical Oncology Fellow",

        "متخصص جراحی مغز و اعصاب":
            "Neurosurgery Specialist",

        "فوق تخصص عفونی بالغین":
            "Adult Infectious Diseases Subspecialist",

        "فوق تخصص خون و سرطان بالغین":
            "Adult Hematology and Oncology Subspecialist",

        "دانشجوی تخصص کودکان":
            "Pediatrics Resident",

        "متخصص بیماریهای پوست":
            "Dermatology Specialist",

        "فوق تخصص نفرولوژی":
            "Nephrology Subspecialist",

        "HIV/AIDS دانشجوی فلوشیپ بالینی":
            "HIV/AIDS Clinical Fellow",

        "فلوشیپ اقدامات مداخله‌ای قلب بزرگسال":
            "Adult Interventional Cardiology Fellow",

        "طب اورژانس (دستیاری تخصصی بالینی)":
            "Emergency Medicine Clinical Resident",

        "فوق تخصص عفونی اطفال":
            "Pediatric Infectious Diseases Subspecialist",

        "دانشجوی فوق تخصص گوارش و کبد بالغین":
            "Adult Gastroenterology and Hepatology Subspecialty Trainee",

        "عفونی":
            "Infectious Diseases",

        "فوق تخصص جراحی پلاستیک، ترمیمی و سوختگی":
            "Plastic, Reconstructive and Burn Surgery Subspecialist",

        "فلوشیپ سوختگی":
            "Burn Surgery Fellow",

        "فلوشیپ جراحی ستون فقرات":
            "Spine Surgery Fellow",

        "دکترای دندانپزشکی":
            "Doctor of Dental Surgery",

        "دانشجوی فلوشیپ ارولوژی اطفال":
            "Pediatric Urology Fellow",

        "فلوشیپ اقدامات مداخله‌ای قلب و عروق کودکان":
            "Pediatric Interventional Cardiology Fellow",

        "فوق تخصص مغز و اعصاب کودکان":
            "Pediatric Neurology Subspecialist",

        "فلوشیپ ارولوژی اطفال":
            "Pediatric Urology Fellow",

        "فلوشیپ مراقبت های ویژه کودکان":
            "Pediatric Critical Care Fellow",

        "دانشجوی تخصص زنان و زایمان":
            "Obstetrics and Gynecology Resident",

        "فلوشیپ ارولوژی ترمیمی":
            "Reconstructive Urology Fellow",

        "فوق تخصص بیماریهای قلب و عروق":
            "Cardiovascular Diseases Subspecialist",

        "متخصص پرتودرمانی":
            "Radiation Oncology Specialist",

        "فوق تخصص طب نوزادی و پیرامون تولد":
            "Neonatology Subspecialist",

        "متخصص چشم پزشکی":
            "Ophthalmology Specialist",
    },

    # --------------------------------------------------------
    # B. HOSPITAL OWNERSHIP
    # --------------------------------------------------------
    "ownership": {
        "دولتی دانشگاهی":
            "Public University-Affiliated",

        "خصوصی":
            "Private",

        "عمومی غیردولتی":
            "Non-Governmental Public",

        "دولتی غیر دانشگاهی":
            "Public Non-University-Affiliated",

        "خیریه":
            "Charitable",
    },

    # --------------------------------------------------------
    # C. INSURANCE FUND
    # --------------------------------------------------------
    "insurance_fund": {
        "روستایی":
            "Rural Health Insurance Fund",

        "کارکنان دولت":
            "Government Employees Health Insurance Fund",

        "ایرانیان":
            "Iranians Health Insurance Fund",

        "سلامت همگانی":
            "Universal Health Insurance Fund",

        "سایر اقشار":
            "Other Population Groups",

        "سایر اقشار نظام ارجاع یک":
            "Other Population Groups - Level 1 Referral System",

        "سایر اقشار اتباع":
            "Other Population Groups - Foreign Nationals",

        "بیمه تامین اجتماعی":
            "Social Security Insurance",

        "سایر اقشار مجهول الهویه":
            "Other Population Groups - Unidentified Individuals",

        "سایر اقشار زندانیان":
            "Other Population Groups - Prisoners",
    },

    # --------------------------------------------------------
    # D. RECORD TYPE / CARE CATEGORY
    # --------------------------------------------------------
    "record_type": {
        "گلوبال":
            "Global",

        "بستری":
            "Inpatient",

        "اورژانس تحت نظر":
            "Emergency Observation",

        "بستری موقت":
            "Temporary Inpatient Admission",
    },

    # --------------------------------------------------------
    # E. DISCHARGE STATUS
    # --------------------------------------------------------
    "discharge_status": {
        "بهبودی کامل":
            "Complete Recovery",

        "بهبودی نسبی":
            "Partial Recovery",

        "ترخیص با میل شخصی":
            "Discharge at Patient's Request",

        "پیگیری":
            "Follow-Up",

        "انتقال به مرکز دیگر":
            "Transfer to Another Facility",

        "فوت":
            "Death",

        "فرار":
            "Absconded from Hospital",

        "تبدیل به بستری":
            "Converted to Inpatient Admission",

        "ارجاع برای اهدای عضو":
            "Referral for Organ Donation",
    },

    # --------------------------------------------------------
    # F. PROVINCE
    # --------------------------------------------------------
    "province": {
        "خوزستان":
            "Khuzestan",

        "آذربایجان شرقی":
            "East Azerbaijan",

        "خراسان رضوی":
            "Razavi Khorasan",

        "سیستان وبلوچستان":
            "Sistan and Baluchestan",

        "اصفهان":
            "Isfahan",

        "کردستان":
            "Kurdistan",

        "تهران":
            "Tehran",

        "کرمانشاه":
            "Kermanshah",

        "مازندران":
            "Mazandaran",

        "فارس":
            "Fars",

        "آذربایجان غربی":
            "West Azerbaijan",

        "لرستان":
            "Lorestan",

        "خراسان جنوبی":
            "South Khorasan",

        "یزد":
            "Yazd",

        "ایلام":
            "Ilam",

        "کرمان":
            "Kerman",

        "اردبیل":
            "Ardabil",

        "همدان":
            "Hamadan",

        "مرکزی":
            "Markazi",

        "البرز":
            "Alborz",

        "بوشهر":
            "Bushehr",

        "سمنان":
            "Semnan",

        "کهگیلویه وبویراحمد":
            "Kohgiluyeh and Boyer-Ahmad",

        "گیلان":
            "Gilan",

        "گلستان":
            "Golestan",

        "قزوین":
            "Qazvin",

        "خراسان شمالی":
            "North Khorasan",

        "چهارمحال وبختیاری":
            "Chaharmahal and Bakhtiari",

        "زنجان":
            "Zanjan",

        "هرمزگان":
            "Hormozgan",

        "قم":
            "Qom",
    },

    # --------------------------------------------------------
    # G. SEX
    # --------------------------------------------------------
    "sex": {
        "مرد": "Male",
        "زن": "Female",
    },
}

# ============================================================
# 4. APPLY TRANSLATIONS
# ============================================================

translation_audit = []

for column, mapping in translation_maps.items():

    if column not in df.columns:
        print(f"WARNING: Column '{column}' not found.")
        continue

    # Normalize leading/trailing spaces for matching.
    original_values = df[column].copy()
    normalized_values = original_values.map(
        lambda value: (
            value.strip() if isinstance(value, str)
            else value
        )
    )

    # Identify Persian values not covered by the dictionary.
    persian_mask = normalized_values.map(
        lambda value: (
            isinstance(value, str)
            and any("\u0600" <= char <= "\u06FF"
                    for char in value)
        )
    )

    unmapped_values = (
        normalized_values[persian_mask]
        .loc[lambda series: ~series.isin(mapping.keys())]
        .value_counts()
    )

    # Translate known categories; preserve unknown values.
    df[column] = normalized_values.map(
        lambda value: mapping.get(value, value)
        if isinstance(value, str)
        else value
    )

    for value, frequency in unmapped_values.items():
        translation_audit.append({
            "column": column,
            "untranslated_value": value,
            "frequency": int(frequency),
        })

    print(
        f"{column}: "
        f"{len(mapping) - len(set(mapping) - set(normalized_values.dropna()))} "
        f"dictionary values observed; "
        f"{len(unmapped_values)} unmapped Persian categories."
    )

# ============================================================
# 5. CHECK FOR REMAINING PERSIAN VALUES
# ============================================================

import re

persian_pattern = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")

print("\nRemaining Persian values in translated columns:")

remaining_persian = False

for column in translation_maps:
    if column not in df.columns:
        continue

    values = df[column].dropna().astype(str)

    remaining = values[
        values.str.contains(persian_pattern, na=False)
    ].value_counts()

    if not remaining.empty:
        remaining_persian = True
        print(f"\nColumn: {column}")

        for value, frequency in remaining.items():
            print(f"  {value} | frequency: {frequency}")

if not remaining_persian:
    print("No Persian values remain in the translated columns.")

# ============================================================
# 6. SAVE TRANSLATED DATASET AND AUDIT REPORT
# ============================================================

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nTranslation completed.")
print(f"Output file: {output_file}")
print(f"Dataset dimensions: {df.shape}")

# Save any untranslated categories separately for review.
if translation_audit:
    audit_file = input_file.with_name(
        "untranslated_persian_values.csv"
    )

    pd.DataFrame(translation_audit).to_csv(
        audit_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Review untranslated categories: {audit_file}")
else:
    print("All observed Persian categories were mapped.")


Original dataset: 15,601 rows, 25 columns
specialty: 66 dictionary values observed; 0 unmapped Persian categories.
ownership: 5 dictionary values observed; 0 unmapped Persian categories.
insurance_fund: 10 dictionary values observed; 0 unmapped Persian categories.
record_type: 4 dictionary values observed; 0 unmapped Persian categories.
discharge_status: 9 dictionary values observed; 0 unmapped Persian categories.
province: 31 dictionary values observed; 0 unmapped Persian categories.
sex: 2 dictionary values observed; 0 unmapped Persian categories.

Remaining Persian values in translated columns:
No Persian values remain in the translated columns.

Translation completed.
Output file: C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases\digestive_surgical_episodes_english.csv
Dataset dimensions: (15601, 25)
All observed Persian categories were mapped.


In [10]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Stage C of the analysis protocol (final): cleaning and preprocessing of the episode file
(digestive_surgical_episodes_english.csv) into a frozen analysis file.

Protocol: "Analysis protocol: cost, resource use and patient cost-sharing in
common digestive-surgical admissions in Iran".

Steps (in protocol order)
  1. Load and verify (15,601 rows, 25 columns, episode_no unique, types). STOP on any mismatch.
  2. Exclude impossible values (each rule counted separately, then sequentially).
  2b. Data-quality checks (flags only, nothing deleted).
  3. Convert currency (Int$ = IRR / 118,411.24).
  4. Reconcile financing (gap <= 5% and both shares present).
  5. Define outcomes (patient share in [0, 1]; overnight stay = LOS >= 1 day).
  6. Flag (do not delete) extremes: LOS > P99 and cost > P99 within group, gap > 50%.
  7. Build exposures (billing mode, age group, ...).
  8. Collapse rare categories (< 30 episodes -> "Other"), using counts only.
  9. Cluster id for cluster-robust SEs (patient_no; anonymous episodes = own cluster).
 10. Freeze: analysis file, data dictionary, flow table, metadata (seed, versions, hashes).

Changes against the first cleaning script (marked ADDED in the log and dictionary)
  * "-" placeholders in sex / specialty / other categorical fields are treated as missing
    (labelled 'Missing'); the missing K64 diagnosis name is filled with its ICD-10 title.
  * Case-variant labels merged; whitespace collapsed.
  * Extra impossible-value rules: admission month outside 1-12, claim lines < 1.
  * Flag-only checks (duplicates, patient conflicts, money inconsistencies, LOS > 365 d ...).
  * Analytic-sample flags in_O1, in_O2, in_O4; discharge_status_model; specialty_h3 (hypothesis 3).
  * O3 scenarios: 1 recorded lower bound, 2 ownership-group mean, 3 upper bound. The upper
    bound can no longer fall below the recorded value. (No cholera-paper scenario.)
  * Console output saved to cleaning_log.txt; QC table saved to qc_checks.csv.

NOT POSSIBLE HERE (needs the raw claim lines, i.e. the build script, Stage B):
  exact duplicate claim lines across sheets, los_from_dates, anonymous share per sheet.

Usage
  python clean_digestive_surgical_episodes_final.py
  python clean_digestive_surgical_episodes_final.py <input.csv> [<output_dir>]
"""

import hashlib
import json
import os
import platform
import stat
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# CONFIGURATION (all values fixed by the protocol unless marked ADDED)
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\digestive_surgical_episodes_english.csv"
)
OUT_DIR = INPUT_CSV.parent / "analysis_ready_final"

SEED = 42                      # recorded; the cleaning itself uses no randomness
IRR_PER_INTL = 118_411.24      # 2024 PPP factor (same as the cholera paper)
RECON_TOL = 0.05               # reconciliation tolerance
GAP_FLAG = 0.50                # financing gap flag (sensitivity analysis 4)
RARE_MIN = 30                  # rare-category threshold
AGE_MAX = 110
P_MAIN = 0.99                  # P99 cap / flag
P_SENS = 0.95                  # P95 cap (sensitivity analysis 7)
EXPECTED_ROWS = 15_601
EXPECTED_GROUP_SIZES = [8517, 3760, 3324]   # Hernia, Gallbladder, Anorectal
FORCE_OVERWRITE = False        # a frozen file is never overwritten unless True

# ADDED (plausibility thresholds; flags only, nothing is deleted by these)
LOS_MAX_FLAG = 365             # days
COST_MIN_INTL_FLAG = 1.0       # Int$
MONEY_TOL = 0.01               # tolerance (share of total cost) for paid > requested
GLOBAL_SHARE_EXPECTED = 0.50   # protocol: "more than half of episodes are Global"
DOCTOR_SHARE_RANGE = (0.20, 0.35)   # protocol: about 27% "Doctor of medicine"
YEAR_RANGE = (1400, 1405)      # Jalali years expected for 2024-2026 admissions; report only

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# Command-line paths are used only when run as a plain script. In Jupyter, sys.argv
# holds the kernel's own arguments (e.g. "-f kernel.json"), which must be ignored.
if not IN_NOTEBOOK:
    if len(sys.argv) > 1:
        INPUT_CSV = Path(sys.argv[1])
        OUT_DIR = INPUT_CSV.parent / "analysis_ready_final"
    if len(sys.argv) > 2:
        OUT_DIR = Path(sys.argv[2])

EXPECTED_COLUMNS = [
    "episode_no", "anonymous", "icd10", "diagnosis_name", "specialty", "ownership",
    "insurance_fund", "record_type", "discharge_status", "province", "sex", "age",
    "claim_lines", "n_codes", "n_service_groups", "total_cost_irr", "insured_share_irr",
    "requested_total_irr", "paid_total_irr", "los_days", "any_surgical_service", "group",
    "patient_no", "adm_year", "adm_month",
]
NUMERIC_COLS = [
    "episode_no", "age", "claim_lines", "n_codes", "n_service_groups", "total_cost_irr",
    "insured_share_irr", "requested_total_irr", "paid_total_irr", "los_days",
    "patient_no", "adm_year", "adm_month",
]
BOOL_COLS = ["anonymous", "any_surgical_service"]
TEXT_COLS = [
    "icd10", "diagnosis_name", "specialty", "ownership", "insurance_fund", "record_type",
    "discharge_status", "province", "sex", "group",
]
HARMONISE_COLS = [c for c in TEXT_COLS if c not in ("icd10", "group")]
CATEGORICAL_FILL = [
    "specialty", "ownership", "insurance_fund", "record_type", "discharge_status",
    "province", "sex",
]
MONEY_COLS = ["total_cost_irr", "insured_share_irr", "requested_total_irr", "paid_total_irr"]

KEPT_CODES = ["K40", "K41", "K42", "K43", "K45", "K46",   # Hernia
              "K80", "K81",                               # Gallbladder
              "K60", "K61", "K62", "K64"]                 # Anorectal

# Fields allowed to be missing (all others must be complete)
MAY_BE_MISSING = {"age", "los_days", "insured_share_irr", "requested_total_irr",
                  "paid_total_irr", "patient_no"}

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

FLOW = []   # rows of the flow table
QC = []     # rows of the data-quality table
_LOG = {"file": None, "orig": None}


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for x in self.streams:
            x.write(s)

    def flush(self):
        for x in self.streams:
            x.flush()


def close_log():
    if _LOG["orig"] is not None:
        sys.stdout = _LOG["orig"]
        _LOG["file"].close()
        _LOG["orig"] = None


def fail(msg):
    print(f"\nSTOP: {msg}")
    close_log()
    if IN_NOTEBOOK:
        # sys.exit() only produces a confusing warning in Jupyter; raise a clear error instead
        raise RuntimeError(f"STOP: {msg}")
    sys.exit(1)


def log(step, description, n_remaining, n_removed=None, sample="all"):
    FLOW.append({"step": step, "sample": sample, "description": description,
                 "n_removed": n_removed, "n_remaining": int(n_remaining)})
    rem = "" if n_removed is None else f"  (removed {n_removed:,})"
    print(f"[{step}] {description}: {int(n_remaining):,}{rem}")


def qc(check, n, denom, action):
    QC.append({"check": check, "n": int(n), "pct": round(100 * n / denom, 2) if denom else np.nan,
               "action": action})
    print(f"  [QC] {check}: {int(n):,} ({100 * n / denom:.2f}%) -> {action}" if denom
          else f"  [QC] {check}: {int(n):,}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def pct(x, n):
    return f"{100 * x / n:.1f}%" if n else "n/a"


# =============================================================================
# 0. Guard against overwriting a frozen file
# =============================================================================
out_csv = OUT_DIR / "digestive_surgical_analysis_file.csv"
if out_csv.exists() and not FORCE_OVERWRITE:
    fail(f"{out_csv} already exists and is frozen. Protocol: do not edit the analysis "
         f"file after any model has been run. Set FORCE_OVERWRITE = True only if no "
         f"model has been run on it.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
_LOG["file"] = open(OUT_DIR / "cleaning_log.txt", "w", encoding="utf-8")
_LOG["orig"] = sys.stdout
sys.stdout = Tee(sys.stdout, _LOG["file"])
print(f"Run started {datetime.now().isoformat(timespec='seconds')}; input {INPUT_CSV}")

# =============================================================================
# 1. LOAD AND VERIFY
# =============================================================================
print("=" * 78)
print("1. LOAD AND VERIFY")
print("=" * 78)
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")

raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
raw.columns = [c.strip() for c in raw.columns]
df = raw.copy()

if len(df) != EXPECTED_ROWS:
    fail(f"Expected {EXPECTED_ROWS:,} rows, found {len(df):,}.")
if df.shape[1] != len(EXPECTED_COLUMNS):
    fail(f"Expected {len(EXPECTED_COLUMNS)} columns, found {df.shape[1]}.")
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]
if missing_cols or extra_cols:
    fail(f"Column mismatch. Missing: {missing_cols}; unexpected: {extra_cols}")

# numeric types
for c in NUMERIC_COLS:
    conv = pd.to_numeric(df[c], errors="coerce")
    bad = conv.isna() & df[c].notna()
    if bad.any():
        fail(f"Column '{c}' has {int(bad.sum())} non-numeric values, e.g. "
             f"{df.loc[bad, c].head(3).tolist()}")
    df[c] = conv.astype("float64")

# boolean types
TRUE_SET, FALSE_SET = {"TRUE", "T", "1", "YES"}, {"FALSE", "F", "0", "NO"}
for c in BOOL_COLS:
    s = df[c].astype("string").str.strip().str.upper()
    ok = s.isin(TRUE_SET | FALSE_SET)
    if not ok.all():
        fail(f"Column '{c}' has {int((~ok).sum())} values that are not TRUE/FALSE.")
    df[c] = s.isin(TRUE_SET)

# text columns: trim, collapse internal whitespace; empty -> missing
for c in TEXT_COLS:
    s = df[c].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    df[c] = s.mask(s == "").astype(object)
df["icd10"] = df["icd10"].str.upper()

# ADDED: placeholders. A lone dash (or n/a) in a categorical field means "not recorded".
PLACEHOLDERS = {"-", "--", "\u2014", "\u2013", "n/a", "na"}
for c in CATEGORICAL_FILL:
    ph = df[c].astype("string").str.lower().isin(PLACEHOLDERS).fillna(False).astype(bool)
    if ph.any():
        df[c] = df[c].mask(ph)
        print(f"  placeholder '-' treated as missing in '{c}': {int(ph.sum()):,}")
        QC.append({"check": f"placeholder treated as missing in {c}", "n": int(ph.sum()),
                   "pct": round(100 * ph.sum() / len(df), 2), "action": "labelled 'Missing'"})
# ADDED: diagnosis_name is only a label. Where it is a dash for K64, use the ICD-10 title.
m64 = (df["icd10"] == "K64") & (df["diagnosis_name"].fillna("-").str.strip() == "-")
if m64.any():
    df.loc[m64, "diagnosis_name"] = "Haemorrhoids and perianal venous thrombosis"
    print(f"  diagnosis_name filled with the ICD-10 title for K64: {int(m64.sum()):,}")

# ADDED: label hygiene. Labels that differ only in letter case are merged into the most
# frequent spelling, so that one category is never split in two.
for c in HARMONISE_COLS:
    s = df[c]
    key = s.astype("string").str.lower()
    tmp = pd.DataFrame({"raw": s, "key": key}).dropna()
    canon = tmp.groupby("key")["raw"].agg(lambda x: x.value_counts().index[0])
    new = key.map(canon).astype(object)
    changed = int((new.notna() & (new != s)).sum())
    df[c] = new.where(new.notna(), s)
    if changed:
        print(f"  label hygiene: '{c}' {changed:,} values merged into a case-variant spelling")
        QC.append({"check": f"case-variant labels merged in {c}", "n": changed,
                   "pct": round(100 * changed / len(df), 2), "action": "merged"})

# episode_no unique and complete
if df["episode_no"].isna().any():
    fail("episode_no has missing values.")
if not df["episode_no"].is_unique:
    fail(f"episode_no is not unique ({int(df['episode_no'].duplicated().sum())} duplicates).")
df["episode_no"] = df["episode_no"].astype("int64")

# completeness of fields that must be complete
for c in EXPECTED_COLUMNS:
    if c not in MAY_BE_MISSING and df[c].isna().any():
        n_na = int(df[c].isna().sum())
        if c in CATEGORICAL_FILL:
            print(f"  note: '{c}' has {n_na} missing values (will be labelled 'Missing').")
        else:
            fail(f"Column '{c}' has {n_na} missing values but must be complete.")

# fixed code list and group consistency
bad_codes = sorted(set(df["icd10"].dropna()) - set(KEPT_CODES))
if bad_codes:
    fail(f"ICD-10 codes outside the fixed list of 12: {bad_codes}")
if df["icd10"].isna().any():
    fail("icd10 has missing values.")
if (df.groupby("icd10")["group"].nunique() > 1).any():
    fail("Some ICD-10 codes map to more than one group.")
if df["group"].nunique() != 3:
    fail(f"Expected 3 groups, found {df['group'].nunique()}: {df['group'].unique().tolist()}")
sizes = sorted(df["group"].value_counts().tolist(), reverse=True)
if sizes != EXPECTED_GROUP_SIZES:
    print(f"  WARNING: group sizes {sizes} differ from the protocol {EXPECTED_GROUP_SIZES}.")

log("C1", "Episodes loaded and verified", len(df))
print("\nEpisodes by group:\n", df["group"].value_counts().to_string())
print("\nRecord types:\n", df["record_type"].value_counts(dropna=False).to_string())
print("\nSex values:\n", df["sex"].value_counts(dropna=False).to_string())

# Missingness of key variables (information only)
miss = df[["age", "los_days", "insured_share_irr", "requested_total_irr",
           "paid_total_irr", "patient_no"]].isna().sum()
print("\nMissing values in fields that may be missing:\n", miss.to_string())

# =============================================================================
# 2. EXCLUDE IMPOSSIBLE VALUES (counted separately, then sequentially)
# =============================================================================
print("\n" + "=" * 78)
print("2. EXCLUDE IMPOSSIBLE VALUES")
print("=" * 78)
rules = {
    "total cost <= 0 (or missing)": df["total_cost_irr"].isna() | (df["total_cost_irr"] <= 0),
    f"age < 0 or > {AGE_MAX}": (df["age"] < 0) | (df["age"] > AGE_MAX),
    "negative LOS": df["los_days"] < 0,
    "negative money value (any of 4 columns)": (df[MONEY_COLS] < 0).any(axis=1),
    "ADDED: admission month outside 1-12": ~df["adm_month"].between(1, 12),
    "ADDED: claim lines < 1": df["claim_lines"] < 1,
}
print("Independent counts (each rule applied to all episodes):")
for name, m in rules.items():
    print(f"  {name}: {int(m.sum()):,}")

keep = pd.Series(True, index=df.index)
n_before = len(df)
for name, m in rules.items():
    newly = m & keep
    keep &= ~m
    log("C2", f"Exclude: {name}", keep.sum(), int(newly.sum()))

excluded = df.loc[~keep].copy()
if len(excluded):
    R = pd.DataFrame(rules).loc[~keep]
    excluded["exclusion_reasons"] = R.apply(lambda r: "; ".join(r.index[r.values]), axis=1)
d = df.loc[keep].copy().reset_index(drop=True)
log("C2", "Valid episodes (primary sample for O1 and O2)", len(d), n_before - len(d), "O1/O2")

# =============================================================================
# 2b. ADDED: DATA-QUALITY CHECKS (flag, never delete)
# =============================================================================
print("\n" + "=" * 78)
print("2b. DATA-QUALITY CHECKS (flags only; nothing below removes an episode)")
print("=" * 78)
N = len(d)

dup_cols = [c for c in EXPECTED_COLUMNS if c != "episode_no"]
d["flag_dup_episode"] = d.duplicated(subset=dup_cols, keep="first")
qc("Exact duplicate episodes (all fields except episode_no; first copy kept unflagged)",
   d["flag_dup_episode"].sum(), N, "flagged, kept; see keep_xsens_consistent")

has_pid = d["patient_no"].notna() & (~d["anonymous"])
sx = d.loc[has_pid & (d["sex"].fillna("Missing") != "Missing")].groupby("patient_no")["sex"].nunique()
conflict_ids = sx[sx > 1].index
d["flag_patient_conflict"] = d["patient_no"].isin(conflict_ids) & has_pid
qc("Episodes of patients recorded with more than one sex", d["flag_patient_conflict"].sum(), N,
   "flagged, kept")
qc("Non-anonymous episodes without patient_no", (~d["anonymous"] & d["patient_no"].isna()).sum(), N,
   "each becomes its own cluster")

t = d["total_cost_irr"]
c_ins = d["insured_share_irr"] > t
c_req = d["requested_total_irr"] > t
c_paid = d["paid_total_irr"] > d["requested_total_irr"] + MONEY_TOL * t
d["flag_money_inconsistent"] = c_ins | c_req | c_paid
qc("Insured share > total cost", c_ins.sum(), N, "flagged, kept")
qc("Requested amount > total cost", c_req.sum(), N, "flagged, kept")
qc(f"Paid > requested (+{MONEY_TOL:.0%} of total)", c_paid.sum(), N, "flagged, kept")

d["flag_los_gt365"] = d["los_days"] > LOS_MAX_FLAG
qc(f"LOS > {LOS_MAX_FLAG} days", d["flag_los_gt365"].sum(), N, "flagged, kept (P99 flag also applies)")
qc("LOS = 0 days", (d["los_days"] == 0).sum(), N, "information; counted as no overnight stay")
qc(f"Total cost < Int$ {COST_MIN_INTL_FLAG:g}", (t / IRR_PER_INTL < COST_MIN_INTL_FLAG).sum(), N,
   "information")
qc("n_codes < 1 or n_service_groups < 1", ((d["n_codes"] < 1) | (d["n_service_groups"] < 1)).sum(), N,
   "information")
yr = d["adm_year"]
qc(f"Admission year outside {YEAR_RANGE[0]}-{YEAR_RANGE[1]}", (~yr.between(*YEAR_RANGE)).sum(), N,
   "information; check the Jalali conversion if > 0")
print("  Admission years:", yr.value_counts().sort_index().to_dict())

# Global share and surgical service (protocol expectations)
is_global = d["record_type"].fillna("Missing").str.lower().str.contains("global")
gs = is_global.mean()
print(f"  Global share: {100 * gs:.1f}% (protocol: more than half)")
if gs <= GLOBAL_SHARE_EXPECTED:
    print("  WARNING: Global share is not above 50%; check record_type labels (crosstab below).")
print("\n  record_type labels classified as Global:")
print(d.assign(is_global=is_global).groupby("record_type")["is_global"].agg(["first", "size"])
      .rename(columns={"first": "classified_global", "size": "episodes"}).to_string())
surg_by_group = d.groupby("group")["any_surgical_service"].mean().mul(100).round(1)
print("\n  Share with a surgical service by group (%):", surg_by_group.to_dict())
if (surg_by_group < 50).any():
    print("  WARNING: a group has < 50% surgical-service episodes; sensitivity analysis 1 will "
          "remove a large part of it.")

# Missingness pattern (informative-missingness check): % missing by group and billing mode
d["_billing"] = np.where(is_global, "Global", "Other")
mv = ["age", "los_days", "insured_share_irr", "requested_total_irr", "paid_total_irr", "patient_no"]
print("\n  % missing by group:")
print(d.groupby("group")[mv].apply(lambda g: g.isna().mean() * 100).round(1).to_string())
print("\n  % missing by billing mode:")
print(d.groupby("_billing")[mv].apply(lambda g: g.isna().mean() * 100).round(1).to_string())
d.drop(columns="_billing", inplace=True)

anon_tab = d.groupby(["adm_year", "group"])["anonymous"].mean().mul(100).round(1).unstack()
print("\n  % anonymous episodes by admission year and group:")
print(anon_tab.to_string())

# =============================================================================
# 3. CONVERT CURRENCY
# =============================================================================
d["cost_intl"] = d["total_cost_irr"] / IRR_PER_INTL
d["insured_share_intl"] = d["insured_share_irr"] / IRR_PER_INTL
d["requested_total_intl"] = d["requested_total_irr"] / IRR_PER_INTL
d["paid_total_intl"] = d["paid_total_irr"] / IRR_PER_INTL

# =============================================================================
# 4. RECONCILE FINANCING
# =============================================================================
both = d["insured_share_irr"].notna() & d["requested_total_irr"].notna()
d["fin_gap"] = np.where(
    both,
    (d["total_cost_irr"] - (d["insured_share_irr"] + d["requested_total_irr"])).abs()
    / d["total_cost_irr"],
    np.nan,
)
d["reconciled"] = (d["fin_gap"] <= RECON_TOL) & both

# =============================================================================
# 5. DEFINE OUTCOMES
# =============================================================================
d["patient_share_raw"] = d["insured_share_irr"] / d["total_cost_irr"]
d["patient_share_gt1"] = d["patient_share_raw"] > 1
d["patient_share"] = d["patient_share_raw"].where(d["patient_share_raw"].between(0, 1))

ov = (d["los_days"] >= 1).astype("Int64")
ov[d["los_days"].isna()] = pd.NA
d["overnight"] = ov

# =============================================================================
# 6. FLAG (DO NOT DELETE) EXTREMES  -- P99 computed within group on the valid sample
# =============================================================================
grp = d.groupby("group")
d["los_p99_group"] = grp["los_days"].transform(lambda s: s.quantile(P_MAIN))
d["los_p95_group"] = grp["los_days"].transform(lambda s: s.quantile(P_SENS))
d["cost_p99_group_irr"] = grp["total_cost_irr"].transform(lambda s: s.quantile(P_MAIN))

d["flag_los_p99"] = d["los_days"] > d["los_p99_group"]
d["flag_cost_p99"] = d["total_cost_irr"] > d["cost_p99_group_irr"]
d["flag_gap_gt50"] = d["fin_gap"] > GAP_FLAG
d["flag_extreme_any"] = d["flag_los_p99"] | d["flag_cost_p99"] | d["flag_gap_gt50"]

# LOS capped (used for models among overnight stays; P95 for sensitivity analysis 7)
d["los_capped_p99"] = np.minimum(d["los_days"], d["los_p99_group"])
d["los_capped_p95"] = np.minimum(d["los_days"], d["los_p95_group"])

# =============================================================================
# 7. BUILD EXPOSURES
# =============================================================================
for c in CATEGORICAL_FILL:
    d[c] = d[c].fillna("Missing")

d["billing_mode"] = np.where(
    d["record_type"].str.lower().str.contains("global", na=False), "Global", "Other")
if (d["billing_mode"] == "Global").sum() == 0:
    fail("No record_type contains 'global'; check the record_type labels printed above.")

d["age_group"] = pd.cut(
    d["age"], bins=[0, 18, 40, 60, 75, 111], right=False,
    labels=["0-17", "18-39", "40-59", "60-74", "75+"],
).astype(object)
d["age_group"] = d["age_group"].fillna("Missing")
d["adm_year"] = d["adm_year"].astype("Int64")
d["adm_month"] = d["adm_month"].astype("Int64")

# =============================================================================
# 8. COLLAPSE RARE CATEGORIES (counts only, outcomes never used)
# =============================================================================
collapse_log = []
for col, new in [("specialty", "specialty_model"), ("province", "province_model"),
                 ("insurance_fund", "insurance_fund_model")]:
    counts = d[col].value_counts()
    rare = counts[counts < RARE_MIN].index.tolist()
    d[new] = d[col].where(~d[col].isin(rare), "Other")
    for r in rare:
        collapse_log.append({"variable": col, "category": r, "episodes": int(counts[r])})
    print(f"  {col}: {len(counts)} categories, {len(rare)} collapsed to 'Other' "
          f"(n = {int(counts[rare].sum()) if rare else 0})")

counts = d["discharge_status"].value_counts()
rare = counts[counts < RARE_MIN].index.tolist()
d["discharge_status_model"] = d["discharge_status"].where(~d["discharge_status"].isin(rare), "Other")
for r in rare:
    collapse_log.append({"variable": "discharge_status", "category": r, "episodes": int(counts[r])})
print(f"  discharge_status: {len(counts)} categories, {len(rare)} collapsed to 'Other' "
      f"(n = {int(counts[rare].sum()) if rare else 0})")

_sp = d["specialty"].str.lower()
d["specialty_h3"] = np.select(
    [_sp == "missing", _sp.str.startswith("general surgery"), _sp.str.contains("urolog")],
    ["Missing", "General surgery", "Urology"], default="Other")
print("  specialty_h3 (hypothesis 3):", d["specialty_h3"].value_counts().to_dict())

# =============================================================================
# 9. REPEATED PATIENTS: CLUSTER ID
# =============================================================================
has_id = d["patient_no"].notna() & (~d["anonymous"])
pid = "P" + d["patient_no"].astype("Int64").astype(str)
aid = "A" + d["episode_no"].astype("int64").astype(str)
d["cluster_id"] = pid.where(has_id, aid)

# =============================================================================
# SAMPLE MEMBERSHIP (primary samples and sensitivity analyses 1-6)
# =============================================================================
n_valid = len(d)
n_both = int(both.sum())
n_recon = int(d["reconciled"].sum())
n_gt1_recon = int((d["reconciled"] & d["patient_share_gt1"]).sum())
d["in_O3_O4"] = d["reconciled"] & d["patient_share"].notna()
log("C4", "Episodes with both insured and requested shares present", n_both, None, "O1/O2")
log("C4", "Reconciled (gap <= 5%, both shares present)", n_recon, None, "O3/O4")
log("C5", "Patient share > 1 among reconciled (excluded from O3, counted)",
    n_recon - n_gt1_recon, n_gt1_recon, "O3/O4")
log("C5", "O3 / O4 primary sample (reconciled, patient share in [0, 1])",
    int(d["in_O3_O4"].sum()), None, "O3/O4")

# ADDED: complete-case definitions of the analytic samples (explicit, so no model silently
# drops rows). Categorical covariates are never missing (labelled 'Missing').
d["in_O1"] = d["claim_lines"].notna() & d["overnight"].notna()
d["in_O2"] = d["cost_intl"].notna() & d["los_days"].notna()
d["in_O4"] = (d["in_O3_O4"] & d["los_days"].notna() & d["claim_lines"].notna()
              & d["cost_intl"].notna())
log("C6", "O1 sample (claim lines and LOS present)", int(d["in_O1"].sum()), None, "O1")
log("C6", "O2 sample (cost and LOS present)", int(d["in_O2"].sum()), None, "O2")
log("C6", "O4 sample (O3/O4 sample with LOS, claim lines and cost present)",
    int(d["in_O4"].sum()), None, "O4")

d["keep_sens1_surgical"] = d["any_surgical_service"]
d["keep_sens2_itemised"] = d["billing_mode"] != "Global"
d["keep_sens3_not_K81"] = d["icd10"] != "K81"
d["keep_sens4_no_extremes"] = ~d["flag_extreme_any"]
d["keep_sens5_not_anonymous"] = ~d["anonymous"]
d["keep_sens6_not_doctor_of_medicine"] = d["specialty"].str.lower() != "doctor of medicine"
# ADDED (exploratory, label as such): drops duplicates, conflicting patients, money inconsistencies
d["keep_xsens_consistent"] = ~(d["flag_dup_episode"] | d["flag_patient_conflict"]
                               | d["flag_money_inconsistent"])
qc("Episodes dropped by keep_xsens_consistent", (~d["keep_xsens_consistent"]).sum(), n_valid,
   "exploratory sensitivity only")

dm = 1 - d["keep_sens6_not_doctor_of_medicine"].mean()
print(f"\n  'Doctor of medicine' share: {100 * dm:.1f}% (protocol: about 27%)")
if not (DOCTOR_SHARE_RANGE[0] <= dm <= DOCTOR_SHARE_RANGE[1]):
    print("  WARNING: share outside the expected range; sensitivity 6 may be using the wrong "
          "label. Specialty labels containing 'doctor':")
    print(d.loc[d["specialty"].str.lower().str.contains("doctor"), "specialty"]
          .value_counts().to_string())

# =============================================================================
# O3 RECONCILIATION SCENARIOS (patient share recomputed for ALL valid episodes)
#   Reconciled episodes keep their observed patient share in every scenario.
# =============================================================================
total = d["total_cost_irr"]
o3 = d["in_O3_O4"]
d["ps_scen1_recorded_lower"] = (d["insured_share_irr"] / total).clip(0, 1)
own_mean = d.loc[o3].groupby("ownership")["patient_share"].mean()
d["ps_scen2_ownership_mean"] = np.where(o3, d["patient_share"], d["ownership"].map(own_mean))
# Upper bound: the whole financing gap goes to the patient. When insured share + requested
# exceeds the total there is no gap to assign, so the bound never falls below the recorded value.
upper = np.fmax((total - d["requested_total_irr"]) / total, d["ps_scen1_recorded_lower"]).clip(0, 1)
d["ps_scen3_upper_bound"] = np.where(o3, d["patient_share"], upper)

# =============================================================================
# CHECKS AND SUMMARIES
# =============================================================================
print("\n" + "=" * 78)
print("CHECKS AND SUMMARIES")
print("=" * 78)
print(f"Valid episodes: {n_valid:,}; Global: {pct((d.billing_mode == 'Global').sum(), n_valid)}; "
      f"surgical service: {pct(d.any_surgical_service.sum(), n_valid)}; "
      f"anonymous: {pct(d.anonymous.sum(), n_valid)}")
print(f"Flags: LOS>P99 {int(d.flag_los_p99.sum()):,}; cost>P99 {int(d.flag_cost_p99.sum()):,}; "
      f"gap>50% {int(d.flag_gap_gt50.sum()):,}; any {int(d.flag_extreme_any.sum()):,}")
print(f"Financing gap not assessable (a share missing): {int((~both).sum()):,}")
print(f"Patient share > 1 (all valid episodes): {int(d.patient_share_gt1.sum()):,}")
print(f"Clusters: {d.cluster_id.nunique():,} for {n_valid:,} episodes")

recon_by_code = (d.groupby("icd10")["reconciled"].agg(["sum", "count", "mean"])
                 .rename(columns={"sum": "reconciled", "count": "episodes", "mean": "share"}))
recon_by_code["share"] = (100 * recon_by_code["share"]).round(1)
print("\nReconciliation by ICD-10 code (protocol check: K62 about 23.6%, K45 about 81.9%):")
print(recon_by_code.to_string())

o3_by_group = d.groupby("group")["in_O3_O4"].agg(["sum", "count"]).rename(
    columns={"sum": "O3_episodes", "count": "valid_episodes"})
o3_by_own = d.groupby("ownership")["in_O3_O4"].agg(["sum", "count"]).rename(
    columns={"sum": "O3_episodes", "count": "valid_episodes"})
print("\nO3 sample by group:\n", o3_by_group.to_string())
print("\nO3 sample by ownership:\n", o3_by_own.to_string())

scen = {
    "1 recorded lower bound": d["ps_scen1_recorded_lower"],
    "2 ownership-group mean": d["ps_scen2_ownership_mean"],
    "3 upper bound": d["ps_scen3_upper_bound"],
}
print("\nPooled mean patient share by scenario (non-missing episodes):")
for k, s in scen.items():
    print(f"  Scenario {k}: {100 * s.mean():.1f}%  (n = {int(s.notna().sum()):,})")
print(f"  Primary reconciled sample: {100 * d.loc[o3, 'patient_share'].mean():.1f}%  "
      f"(n = {int(o3.sum()):,})")

print("\nMedian cost (Int$) by admission year (NOMINAL IRR / one 2024 PPP factor; not inflation-adjusted):")
print(d.groupby("adm_year")["cost_intl"].agg(["size", "median"]).round(1).to_string())

first = d[d["episode_no"] == 1]
if len(first):
    r = first.iloc[0]
    print("\nWorked example, episode 1 (should be flagged and kept):")
    print(f"  {r.icd10}, age {r.age:.0f}, LOS {r.los_days:.2f} d, cost Int$ {r.cost_intl:,.0f}, "
          f"gap {100 * r.fin_gap:.1f}%, reconciled={r.reconciled}, "
          f"flag LOS P99={r.flag_los_p99}, flag cost P99={r.flag_cost_p99}, "
          f"flag gap>50%={r.flag_gap_gt50}")

# audit tables that do not depend on the freeze decision
pd.DataFrame(QC).to_csv(OUT_DIR / "qc_checks.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(FLOW).to_csv(OUT_DIR / "flow_table.csv", index=False, encoding="utf-8-sig")

# =============================================================================
# 10. FREEZE: analysis file, data dictionary, flow table, metadata
# =============================================================================
DICTIONARY = {
    # original columns
    "episode_no": ("episode_no", "Unique episode identifier (one row per hospital admission)"),
    "anonymous": ("anonymous", "TRUE if the episode key was built from a signature because patient ID was missing"),
    "icd10": ("icd10", "ICD-10 code of the highest-cost line (12 codes kept)"),
    "diagnosis_name": ("diagnosis_name", "English name of the ICD-10 code"),
    "specialty": ("specialty", "Specialty of the highest-cost line (as recorded)"),
    "ownership": ("ownership", "Hospital ownership (as recorded)"),
    "insurance_fund": ("insurance_fund", "Insurance fund (as recorded)"),
    "record_type": ("record_type", "Record type, e.g. Global or itemised (as recorded)"),
    "discharge_status": ("discharge_status", "Discharge status of the highest-cost line"),
    "province": ("province", "Province of the claim (as recorded)"),
    "sex": ("sex", "Sex"),
    "age": ("age", "Age in years"),
    "claim_lines": ("claim_lines", "Number of claim lines in the episode (O1 outcome)"),
    "n_codes": ("n_codes", "Number of distinct ICD-10 codes in the episode"),
    "n_service_groups": ("n_service_groups", "Number of distinct service groups in the episode"),
    "total_cost_irr": ("total_cost_irr", "Total episode cost, IRR (sum of lines)"),
    "insured_share_irr": ("insured_share_irr", "Insured (patient) share, IRR"),
    "requested_total_irr": ("requested_total_irr", "Amount requested from the insurer, IRR"),
    "paid_total_irr": ("paid_total_irr", "Amount paid by the insurer, IRR"),
    "los_days": ("los_days", "Length of stay, fractional days (maximum of the line values)"),
    "any_surgical_service": ("any_surgical_service", "Any operating-room, surgical or anaesthesia line"),
    "group": ("group", "Hernia, Gallbladder or Anorectal"),
    "patient_no": ("patient_no", "Anonymised patient number (random integer, seed 42)"),
    "adm_year": ("adm_year", "Jalali admission year"),
    "adm_month": ("adm_month", "Jalali admission month"),
    # derived columns
    "cost_intl": ("total_cost_irr", f"Total cost in Int$ = IRR / {IRR_PER_INTL} (O2 outcome)"),
    "insured_share_intl": ("insured_share_irr", "Insured share in Int$"),
    "requested_total_intl": ("requested_total_irr", "Requested total in Int$"),
    "paid_total_intl": ("paid_total_irr", "Paid total in Int$"),
    "fin_gap": ("total, insured, requested", "|total - (insured + requested)| / total; missing if a share is missing"),
    "reconciled": ("fin_gap", f"TRUE if gap <= {RECON_TOL:.0%} and both shares present (V3)"),
    "patient_share_raw": ("insured_share_irr, total_cost_irr", "Insured share / total cost, unrestricted"),
    "patient_share_gt1": ("patient_share_raw", "TRUE if raw patient share > 1 (excluded from O3)"),
    "patient_share": ("patient_share_raw", "Patient share, valid only within [0, 1] (O3 outcome)"),
    "overnight": ("los_days", "1 if LOS >= 1 day, 0 otherwise, missing if LOS missing (O1 outcome)"),
    "los_p99_group": ("los_days, group", "99th percentile of LOS within group (valid sample)"),
    "los_p95_group": ("los_days, group", "95th percentile of LOS within group (sensitivity 7)"),
    "cost_p99_group_irr": ("total_cost_irr, group", "99th percentile of total cost (IRR) within group"),
    "flag_los_p99": ("los_days", "LOS above group P99 (flag, kept)"),
    "flag_cost_p99": ("total_cost_irr", "Cost above group P99 (flag, kept)"),
    "flag_gap_gt50": ("fin_gap", f"Financing gap above {GAP_FLAG:.0%} (flag, kept)"),
    "flag_extreme_any": ("flags", "Any of the three flags (sensitivity 4)"),
    "los_capped_p99": ("los_days", "LOS capped at group P99 (LOS models)"),
    "los_capped_p95": ("los_days", "LOS capped at group P95 (sensitivity 7)"),
    "billing_mode": ("record_type", "Global versus Other record types"),
    "age_group": ("age", "0-17, 18-39, 40-59, 60-74, 75+, or Missing"),
    "specialty_model": ("specialty", f"Specialty, categories with < {RARE_MIN} episodes set to 'Other'"),
    "province_model": ("province", f"Province, categories with < {RARE_MIN} episodes set to 'Other'"),
    "insurance_fund_model": ("insurance_fund", f"Fund, categories with < {RARE_MIN} episodes set to 'Other'"),
    "cluster_id": ("patient_no, anonymous, episode_no", "Cluster for robust SEs: patient number; each anonymous episode is its own cluster"),
    "in_O3_O4": ("reconciled, patient_share", "TRUE if in the O3/O4 primary sample (reconciled, share in [0, 1])"),
    "keep_sens1_surgical": ("any_surgical_service", "Sensitivity 1: episodes with a surgical service"),
    "keep_sens2_itemised": ("billing_mode", "Sensitivity 2: exclude Global packages"),
    "keep_sens3_not_K81": ("icd10", "Sensitivity 3: exclude K81"),
    "keep_sens4_no_extremes": ("flag_extreme_any", "Sensitivity 4: exclude flagged extremes"),
    "keep_sens5_not_anonymous": ("anonymous", "Sensitivity 5: exclude anonymous episodes"),
    "keep_sens6_not_doctor_of_medicine": ("specialty", "Sensitivity 6: exclude specialty 'Doctor of medicine'"),
    "ps_scen1_recorded_lower": ("insured_share_irr, total_cost_irr", "O3 scenario 1: recorded lower bound (insured / total, capped at 1), all valid episodes"),
    "ps_scen2_ownership_mean": ("patient_share, ownership", "O3 scenario 2: unreconciled episodes get the mean of reconciled episodes in the same ownership group"),
    "ps_scen3_upper_bound": ("total, requested, insured", "O3 scenario 3 (upper bound): unreconciled episodes get max(recorded share, (total - requested) / total), capped to [0, 1]"),
    "discharge_status_model": ("discharge_status", f"ADDED: discharge status, categories with < {RARE_MIN} episodes set to 'Other'"),
    "specialty_h3": ("specialty", "ADDED: for hypothesis 3 (K40, general surgery vs urology): General surgery / Urology / Other / Missing"),
    # ADDED in v2
    "flag_dup_episode": ("all original columns except episode_no", "ADDED: exact duplicate of an earlier episode (flag, kept)"),
    "flag_patient_conflict": ("patient_no, sex", "ADDED: patient recorded with more than one sex (flag, kept)"),
    "flag_money_inconsistent": ("money columns", "ADDED: insured or requested > total, or paid > requested (flag, kept)"),
    "flag_los_gt365": ("los_days", f"ADDED: LOS above {LOS_MAX_FLAG} days (flag, kept)"),
    "in_O1": ("claim_lines, overnight", "ADDED: O1 complete-case sample"),
    "in_O2": ("cost_intl, los_days", "ADDED: O2 complete-case sample"),
    "in_O4": ("in_O3_O4, los_days, claim_lines, cost_intl", "ADDED: O4 complete-case sample"),
    "keep_xsens_consistent": ("flag_dup_episode, flag_patient_conflict, flag_money_inconsistent", "ADDED, exploratory sensitivity: exclude flagged duplicates, conflicting patients and money inconsistencies"),
}

ORIGINAL = EXPECTED_COLUMNS
DERIVED = [c for c in DICTIONARY if c not in ORIGINAL]
OUT_COLS = ORIGINAL + DERIVED
unlisted = [c for c in d.columns if c not in OUT_COLS]
if unlisted:
    print(f"\nNote: columns not written to the analysis file (kept out of the dictionary): {unlisted}")
d_out = d[OUT_COLS].copy()

OUT_DIR.mkdir(parents=True, exist_ok=True)
if out_csv.exists():
    os.chmod(out_csv, stat.S_IWRITE | stat.S_IREAD)
d_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

dd = pd.DataFrame(
    [{"variable": c, "source_columns": DICTIONARY[c][0], "definition": DICTIONARY[c][1],
      "dtype": str(d_out[c].dtype), "n_missing": int(d_out[c].isna().sum())} for c in OUT_COLS])
dd.to_csv(OUT_DIR / "data_dictionary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(FLOW).to_csv(OUT_DIR / "flow_table.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(collapse_log).to_csv(OUT_DIR / "rare_category_collapse.csv", index=False,
                                  encoding="utf-8-sig")
recon_by_code.to_csv(OUT_DIR / "reconciliation_by_code.csv", encoding="utf-8-sig")
o3_by_group.to_csv(OUT_DIR / "o3_sample_by_group.csv", encoding="utf-8-sig")
o3_by_own.to_csv(OUT_DIR / "o3_sample_by_ownership.csv", encoding="utf-8-sig")
if len(excluded):
    excluded.to_csv(OUT_DIR / "excluded_impossible_values.csv", index=False, encoding="utf-8-sig")

try:
    script_hash = sha256(Path(__file__))
except Exception:
    script_hash = None
meta = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "script_version": "final",
    "script_sha256": script_hash,
    "input_file": str(INPUT_CSV),
    "input_sha256": sha256(INPUT_CSV),
    "output_file": str(out_csv),
    "output_sha256": sha256(out_csv),
    "rows_in": int(len(df)),
    "rows_out": int(len(d_out)),
    "random_seed": SEED,
    "parameters": {
        "irr_per_intl": IRR_PER_INTL, "recon_tolerance": RECON_TOL, "gap_flag": GAP_FLAG,
        "rare_category_min": RARE_MIN, "age_max": AGE_MAX, "p_main": P_MAIN, "p_sens": P_SENS,
        "los_max_flag": LOS_MAX_FLAG, "money_tol": MONEY_TOL,
    },
    "software": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "platform": platform.platform()},
}
with open(OUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

os.chmod(out_csv, stat.S_IREAD)   # read-only: the file is frozen

print("\n" + "=" * 78)
print(f"FROZEN: {len(d_out):,} episodes x {d_out.shape[1]} columns -> {out_csv}")
print(f"Outputs in {OUT_DIR}: analysis file, data_dictionary, flow_table, qc_checks, "
      f"cleaning_log, metadata, and audit tables.")
print("Do not edit the analysis file after any model has been run.")
print("=" * 78)
close_log()

Run started 2026-10-27T22:59:54; input C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases\digestive_surgical_episodes_english.csv
1. LOAD AND VERIFY
  placeholder '-' treated as missing in 'specialty': 283
  placeholder '-' treated as missing in 'sex': 2
  diagnosis_name filled with the ICD-10 title for K64: 2,247
  note: 'specialty' has 283 missing values (will be labelled 'Missing').
  note: 'sex' has 2 missing values (will be labelled 'Missing').
[C1] Episodes loaded and verified: 15,601

Episodes by group:
 group
Hernia         8517
Gallbladder    3760
Anorectal      3324

Record types:
 record_type
Global                           8462
Inpatient                        6223
Emergency Observation             889
Temporary Inpatient Admission      27

Sex values:
 sex
Male      9190
Female    6409
NaN          2

Missing values in fields that may be missing:
 age                    0
los_days               0
insured_share_irr      0
requested_total_irr    0
paid_to

In [12]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 1 diagnostics. Run this BEFORE trusting or reporting the Objective 1 results.

It reads the frozen analysis file (and, for D10, the Objective 1 output tables) and writes
d1_...d11_ CSV files to <objective1_outputs>/diagnostics. Nothing is modified.

  D1  Sparse cells: levels with < 30 episodes, or < 10 overnight / non-overnight episodes
  D2  LOS distribution: heaping, mass between 0.5 and 1 day, other overnight cut-offs
  D3  Record types: Global vs Inpatient vs Emergency Observation vs Temporary admission
  D4  Firth logit: sandwich SE vs model-based SE vs ordinary ML logit (sparse-cell CIs)
  D5  Clustering: patient clusters are almost all singletons; SE when clustering on province
  D6  Leave-one-province-out stability of billing-mode, group and surgical-service ratios
  D7  Billing-mode definition: 4 record types, drop Emergency Observation, strata Global / Other
  D8  Interaction tests (billing mode x group, billing mode x surgical service)
  D9  Overnight cut-off sensitivity (0.5, 0.75, 1.0 day)
  D10 Sensitivity-direction rule redone with confidence intervals (needs the O1 output tables)
  D11 Influence of the tiny cells (sex 'Missing', admission year 1400)

Usage
  python objective1_diagnostics.py
  python objective1_diagnostics.py <analysis_file.csv> [--no-models]

--no-models runs only the pandas checks (D1, D2, D3, D5 counts, D10) and needs no statsmodels.
"""

import argparse
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--no-models", action="store_true")
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_MODELS = not args.no_models
O1_DIR = INPUT_CSV.parent / "objective1_outputs"
OUT = O1_DIR / "diagnostics"
OUT.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

if RUN_MODELS:
    try:
        import patsy
        import statsmodels.api as sm
        import statsmodels.formula.api as smf
        from scipy import stats
    except ImportError as e:
        print(f"statsmodels/patsy/scipy not available ({e}); running with --no-models.")
        RUN_MODELS = False


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def save(df, name):
    df.to_csv(OUT / name, index=False, encoding="utf-8-sig")


# =============================================================================
# LOAD AND PREPARE (same preparation as objective1_resource_utilisation.py)
# =============================================================================
if not INPUT_CSV.exists():
    sys.exit(f"Input file not found: {INPUT_CSV}")
D = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


D["any_surgical_service"] = np.where(to_bool(D["any_surgical_service"]), "Yes", "No")
D["adm_year"] = D["adm_year"].astype("Int64").astype(str)
D["adm_month"] = D["adm_month"].astype("Int64").astype(str)
D["cluster_id"] = D["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "billing_mode", "group", "icd10", "record_type"]:
    D[c] = D[c].fillna("Missing").astype(str)
if "discharge_status_model" not in D.columns:
    cnt = D["discharge_status"].value_counts()
    D["discharge_status_model"] = D["discharge_status"].where(
        ~D["discharge_status"].isin(cnt[cnt < 30].index), "Other")
for c in ["claim_lines", "los_days", "overnight", "los_capped_p99"]:
    D[c] = pd.to_numeric(D[c], errors="coerce")
D["overnight_t05"] = (D["los_days"] >= 0.5).astype(float)
D["overnight_t075"] = (D["los_days"] >= 0.75).astype(float)

TIME = ["adm_year", "adm_month"]
M3 = ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode", "group",
      "any_surgical_service", "discharge_status_model"]
M5 = M3 + ["province_model", "specialty_model"]
CAT = sorted(set(M5) | set(TIME) | {"record_type"})
REFS = {v: D[v].value_counts().index[0] for v in CAT}
REFS.update({"billing_mode": "Other", "any_surgical_service": "No", "adm_month": "1",
             "record_type": "Inpatient"})
print(f"Rows: {len(D):,}; overnight share {100 * D.overnight.mean():.1f}%")


def categorise(d):
    d = d.copy()
    for v in CAT:
        s = d[v].astype(str)
        lv = s.value_counts().index.tolist()
        ref = REFS[v] if REFS[v] in lv else lv[0]
        d[v] = pd.Categorical(s, categories=[ref] + [x for x in lv if x != ref])
    return d


# =============================================================================
# D1. SPARSE CELLS
# =============================================================================
hdr("D1. SPARSE CELLS (n < 30, or < 10 overnight / non-overnight episodes)")
rows = []
for v in M5 + TIME:
    for lev, g in D.groupby(v):
        ev = int((g["overnight"] == 1).sum())
        rows.append({"variable": v, "level": lev, "n": len(g), "overnight": ev,
                     "not_overnight": len(g) - ev, "flag_n_lt30": len(g) < 30,
                     "flag_separation_risk": ev < 10 or (len(g) - ev) < 10,
                     "overnight_pct": round(100 * ev / len(g), 1)})
d1 = pd.DataFrame(rows)
save(d1, "d1_sparse_cells.csv")
print(d1[d1.flag_n_lt30 | d1.flag_separation_risk].to_string(index=False))

# =============================================================================
# D2. LOS DISTRIBUTION
# =============================================================================
hdr("D2. LOS DISTRIBUTION (is 'LOS >= 1 day' a sound overnight definition?)")
bins = [0, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 3, 5, np.inf]
D["los_bin"] = pd.cut(D["los_days"], bins, right=False).astype(str)
order = [str(x) for x in pd.cut(pd.Series([0.1]), bins, right=False).cat.categories]
t = pd.crosstab(D["group"], D["los_bin"], normalize="index").mul(100).round(1)
t = t[[c for c in order if c in t.columns]]
t["all_groups_in_0.5_to_1"] = D.groupby("group")["los_days"].apply(lambda s: round(100 * s.between(0.5, 1, inclusive="left").mean(), 1))
d2a = t.reset_index()
save(d2a, "d2a_los_bins_by_group.csv")
print("% of episodes by LOS band (days):\n", d2a.to_string(index=False))
top = D["los_days"].round(2).value_counts().head(15)
d2b = pd.DataFrame({"los_rounded": top.index, "episodes": top.values,
                    "pct": (100 * top.values / len(D)).round(2)})
save(d2b, "d2b_los_most_frequent_values.csv")
print("\nMost frequent LOS values (heaping suggests tariff-like or hospital-fixed LOS):\n",
      d2b.to_string(index=False))
d2c = pd.DataFrame([{"cutoff_days": c, "overnight_pct": round(100 * (D.los_days >= c).mean(), 1)}
                    for c in [0.25, 0.5, 0.75, 1.0, 1.25]])
save(d2c, "d2c_overnight_prevalence_by_cutoff.csv")
print("\nShare called 'overnight' under different cut-offs:\n", d2c.to_string(index=False))

# =============================================================================
# D3. RECORD TYPES
# =============================================================================
hdr("D3. RECORD TYPES (billing mode 'Other' mixes four record types)")


def iqr(s, k=1):
    return f"{s.median():.{k}f} ({s.quantile(.25):.{k}f}-{s.quantile(.75):.{k}f})"


d3 = (D.groupby("record_type")
      .apply(lambda g: pd.Series({"episodes": len(g), "claim_lines": iqr(g.claim_lines, 0),
                                  "LOS_days": iqr(g.los_days, 2),
                                  "overnight_pct": round(100 * g.overnight.mean(), 1),
                                  "surgical_service_pct": round(100 * (g.any_surgical_service == "Yes").mean(), 1),
                                  "total_cost_median_IRR": round(g.total_cost_irr.median())}))
      .reset_index())
save(d3, "d3_record_types.csv")
print(d3.to_string(index=False))
gl = D[D.record_type.str.lower().str.contains("global")]["claim_lines"].value_counts(normalize=True).head(6).mul(100).round(1)
print("\nGlobal claim lines, most frequent values (% of Global episodes):", gl.to_dict())
sp = D["specialty_model"].where(D["specialty_model"].isin(["Emergency Medicine Specialist", "Missing"]), "All other specialties")
d3b = pd.crosstab(sp, D["record_type"]).reset_index()
save(d3b, "d3b_specialty_by_record_type.csv")
print("\nEmergency-medicine and 'Missing' specialty by record type:\n", d3b.to_string(index=False))

# =============================================================================
# D5 (counts part). PATIENT CLUSTERS
# =============================================================================
hdr("D5. CLUSTERING")
cs = D["cluster_id"].value_counts()
print(f"Patient clusters: {len(cs):,} for {len(D):,} episodes; singletons {100 * (cs == 1).mean():.1f}% "
      f"(max episodes per patient {cs.max()}). Clustering on patient therefore adds almost nothing;\n"
      f"episodes are really correlated within hospital, which the file does not identify.")
pc = D["province_model"].value_counts()
print(f"Province clusters (province_model): {len(pc)}; smallest {pc.min()}, largest {pc.max()}")

# =============================================================================
# MODEL-BASED DIAGNOSTICS
# =============================================================================
if RUN_MODELS:
    DM = categorise(D)

    def build(y, terms, d, extra=""):
        use = [t for t in terms + TIME if d[t].nunique() >= 2]
        return f"{y} ~ " + " + ".join(f"C({t})" for t in use) + extra

    def prep(d, y, los=False):
        sub = d.dropna(subset=[y]).copy()
        if los:
            sub = sub[(sub["overnight"] == 1) & (sub[y] > 0)]
        for v in CAT:
            sub[v] = sub[v].cat.remove_unused_categories()
        return sub

    FAMS = {"claim_lines": ("claim_lines", sm.families.Poisson(), False),
            "los": ("los_capped_p99", sm.families.Gamma(sm.families.links.Log()), True),
            "overnight": ("overnight", sm.families.Binomial(), False)}

    def fit(d, key, terms, cluster="cluster_id", extra="", ycol=None):
        y, fam, los = FAMS[key]
        y = ycol or y
        sub = prep(d, y, los)
        f = build(y, terms, sub, extra)
        g = pd.factorize(sub[cluster])[0]
        res = smf.glm(f, sub, family=fam).fit(cov_type="cluster", cov_kwds={"groups": g}, maxiter=200)
        return res, sub

    def tidy(res, key, scenario, only=None):
        z = stats.norm.ppf(0.975)
        out = []
        for term in res.params.index:
            if term == "Intercept" or (only and not any(o in term for o in only)):
                continue
            b, se = res.params[term], res.bse[term]
            out.append({"scenario": scenario, "outcome": key, "term": term, "n": int(res.nobs),
                        "ratio": np.exp(b), "lo": np.exp(b - z * se), "hi": np.exp(b + z * se)})
        return out

    KEY = ["billing_mode", "record_type", "group", "any_surgical_service"]

    # ---------------------------------------------------------------- D4
    hdr("D4. FIRTH LOGIT: SANDWICH SE vs MODEL-BASED SE vs ORDINARY LOGIT (overnight, M5)")

    def firth_logit(X, y, groups, maxiter=200, tol=1e-8):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        n, k = X.shape

        def pen_ll(beta):
            eta = X @ beta
            p = 1 / (1 + np.exp(-eta))
            w = p * (1 - p)
            ev = np.linalg.eigvalsh(X.T @ (X * w[:, None]))
            ev = ev[ev > 1e-10]
            return np.sum(y * eta - np.logaddexp(0, eta)) + 0.5 * np.sum(np.log(ev))

        beta = np.zeros(k)
        conv = False
        for _ in range(maxiter):
            p = 1 / (1 + np.exp(-(X @ beta)))
            w = p * (1 - p)
            inv = np.linalg.pinv(X.T @ (X * w[:, None]))
            A = X * np.sqrt(w)[:, None]
            h = np.einsum("ij,ij->i", A @ inv, A)
            step = inv @ (X.T @ (y - p + h * (0.5 - p)))
            mx = np.abs(step).max()
            if mx > 5:
                step *= 5 / mx
            f0, t = pen_ll(beta), 1.0
            while t > 1e-4 and pen_ll(beta + t * step) < f0 - 1e-10:
                t /= 2
            beta = beta + t * step
            if np.abs(t * step).max() < tol:
                conv = True
                break
        p = 1 / (1 + np.exp(-(X @ beta)))
        w = p * (1 - p)
        inv = np.linalg.pinv(X.T @ (X * w[:, None]))
        A = X * np.sqrt(w)[:, None]
        h = np.einsum("ij,ij->i", A @ inv, A)
        S = X * (y - p + h * (0.5 - p))[:, None]
        G = pd.DataFrame(S).groupby(groups).sum().values
        g = G.shape[0]
        corr = (g / (g - 1)) * ((n - 1) / max(n - k, 1))
        return beta, corr * inv @ (G.T @ G) @ inv, inv, conv

    sub = prep(DM, "overnight")
    fml = build("overnight", M5, sub)
    yv, X = patsy.dmatrices(fml, sub, return_type="dataframe")
    X = X[[c for c in X.columns if c == "Intercept" or X[c].std() > 0]]
    groups = pd.factorize(sub["cluster_id"])[0]
    b, cov_rob, cov_mod, conv = firth_logit(X.values, yv.iloc[:, 0].values, groups)
    d4 = pd.DataFrame({"term": X.columns, "beta_firth": b,
                       "se_sandwich": np.sqrt(np.diag(cov_rob)), "se_model": np.sqrt(np.diag(cov_mod))})
    try:
        ml = smf.glm(fml, sub, family=sm.families.Binomial()).fit(maxiter=200)
        d4 = d4.merge(pd.DataFrame({"term": ml.params.index, "beta_ml": ml.params.values,
                                    "se_ml": ml.bse.values}), on="term", how="left")
        d4["beta_diff_firth_minus_ml"] = d4["beta_firth"] - d4["beta_ml"]
    except Exception as e:
        print("Ordinary logit failed (separation?):", e)
    d4["se_ratio_model_over_sandwich"] = d4["se_model"] / d4["se_sandwich"]
    z = 1.96
    d4["OR_firth"] = np.exp(d4["beta_firth"])
    d4["CI_model_lo"] = np.exp(d4["beta_firth"] - z * d4["se_model"])
    d4["CI_model_hi"] = np.exp(d4["beta_firth"] + z * d4["se_model"])
    d4["flag_sandwich_too_narrow"] = d4["se_ratio_model_over_sandwich"] > 2
    save(d4, "d4_firth_vs_ml.csv")
    print(f"Firth converged: {conv}. Terms where the model-based SE is more than twice the sandwich SE "
          f"(sandwich CI too narrow):")
    print(d4[d4.flag_sandwich_too_narrow].sort_values("se_ratio_model_over_sandwich", ascending=False)
          [["term", "OR_firth", "se_sandwich", "se_model", "CI_model_lo", "CI_model_hi"]]
          .round(3).to_string(index=False))

    # ---------------------------------------------------------------- D5 (SE by clustering level)
    hdr("D5b. SE WHEN CLUSTERING ON PROVINCE (31 clusters) vs PATIENT (M5)")
    rows = []
    for key in ["claim_lines", "los", "overnight"]:
        try:
            r1, _ = fit(DM, key, M5, "cluster_id")
            r2, _ = fit(DM, key, M5, "province_model")
            for term in r1.params.index:
                if term == "Intercept" or any(s in term for s in ["province_model", "adm_month"]):
                    continue
                rows.append({"outcome": key, "term": term, "ratio": np.exp(r1.params[term]),
                             "se_patient": r1.bse[term], "se_province": r2.bse[term],
                             "se_ratio_province_over_patient": r2.bse[term] / r1.bse[term]})
        except Exception as e:
            print(key, "failed:", e)
    d5 = pd.DataFrame(rows)
    save(d5, "d5_se_by_cluster_level.csv")
    print(d5[d5.term.str.contains("billing_mode|group|any_surgical")].round(3).to_string(index=False))
    print("\nMedian SE ratio (province / patient) by outcome:",
          d5.groupby("outcome")["se_ratio_province_over_patient"].median().round(2).to_dict())

    # ICC of Pearson residuals by province after M3 (no province in the model)
    def icc1(r, g):
        t = pd.DataFrame({"r": np.asarray(r), "g": np.asarray(g)})
        k, n = t.g.nunique(), len(t)
        gm = t.groupby("g")["r"].agg(["mean", "size"])
        msb = (gm["size"] * (gm["mean"] - t.r.mean()) ** 2).sum() / (k - 1)
        msw = ((t.r - t.g.map(gm["mean"])) ** 2).sum() / (n - k)
        n0 = (n - (gm["size"] ** 2).sum() / n) / (k - 1)
        return (msb - msw) / (msb + (n0 - 1) * msw)

    rows = []
    for key in ["claim_lines", "los", "overnight"]:
        try:
            r3, s3 = fit(DM, key, M3)
            rows.append({"outcome": key, "icc_province_after_M3": icc1(r3.resid_pearson, s3["province_model"])})
        except Exception as e:
            print(key, "ICC failed:", e)
    d5b = pd.DataFrame(rows)
    save(d5b, "d5b_icc_province.csv")
    print("\nResidual intra-province correlation after M3 (province not in the model):\n", d5b.round(4).to_string(index=False))

    # ---------------------------------------------------------------- D6
    hdr("D6. LEAVE-ONE-PROVINCE-OUT (10 largest provinces)")
    top_prov = [p for p in DM["province_model"].value_counts().index if p != "Other"][:10]
    rows = []
    for key in ["claim_lines", "los", "overnight"]:
        for p in top_prov:
            try:
                res, _ = fit(DM[DM["province_model"] != p], key, M5)
                rows += tidy(res, key, f"without {p}", only=["billing_mode", "group", "any_surgical"])
            except Exception as e:
                print(key, p, "failed:", e)
    d6 = pd.DataFrame(rows)
    save(d6, "d6_leave_one_province_out.csv")
    print(d6.groupby(["outcome", "term"])["ratio"].agg(["min", "max"]).round(3).to_string())

    # ---------------------------------------------------------------- D7
    hdr("D7. BILLING-MODE DEFINITION AND STRATA (M5)")
    rows = []
    # a) four record types instead of Global vs Other
    terms_rt = [t for t in M5 if t != "billing_mode"] + ["record_type"]
    # b) drop Emergency Observation and Temporary Inpatient admission (not operated inpatient care)
    keep_adm = DM[~DM["record_type"].isin(["Emergency Observation", "Temporary Inpatient Admission"])]
    # c) strata
    glob = DM[DM["billing_mode"] == "Global"]
    oth = DM[DM["billing_mode"] == "Other"]
    scen = [("a) four record types", DM, terms_rt),
            ("b) without Emergency Observation / Temporary admission", keep_adm, M5),
            ("c1) Global only", glob, [t for t in M5 if t != "billing_mode"]),
            ("c2) Other only", oth, [t for t in M5 if t != "billing_mode"])]
    for name, d, terms in scen:
        for key in ["claim_lines", "los", "overnight"]:
            try:
                res, _ = fit(d, key, terms)
                rows += tidy(res, key, name, only=["billing_mode", "record_type", "group", "any_surgical"])
            except Exception as e:
                print(name, key, "failed:", e)
    d7 = pd.DataFrame(rows)
    save(d7, "d7_billing_definition_and_strata.csv")
    print(d7.round(3).to_string(index=False))

    # ---------------------------------------------------------------- D8
    hdr("D8. INTERACTION TESTS (Wald, cluster-robust)")
    rows = []
    for label, extra in [("billing_mode x group", " + C(billing_mode):C(group)"),
                         ("billing_mode x surgical service", " + C(billing_mode):C(any_surgical_service)"),
                         ("group x surgical service", " + C(group):C(any_surgical_service)")]:
        for key in ["claim_lines", "los", "overnight"]:
            try:
                res, _ = fit(DM, key, M5, extra=extra)
                idx = [i for i, n in enumerate(res.params.index) if ":" in n]
                R = np.eye(len(res.params))[idx]
                w = res.wald_test(R, scalar=True)
                rows.append({"interaction": label, "outcome": key, "chi2": float(w.statistic),
                             "df": len(idx), "p_value": float(w.pvalue)})
            except Exception as e:
                print(label, key, "failed:", e)
    d8 = pd.DataFrame(rows)
    save(d8, "d8_interaction_tests.csv")
    print(d8.round(4).to_string(index=False))

    # ---------------------------------------------------------------- D9
    hdr("D9. OVERNIGHT CUT-OFF SENSITIVITY (ordinary logit, M5)")
    rows = []
    for ycol, label in [("overnight_t05", "LOS >= 0.5 d"), ("overnight_t075", "LOS >= 0.75 d"),
                        ("overnight", "LOS >= 1 d (protocol)")]:
        try:
            res, _ = fit(DM, "overnight", M5, ycol=ycol)
            rows += tidy(res, "overnight", label, only=["billing_mode", "group", "any_surgical"])
        except Exception as e:
            print(label, "failed:", e)
    d9 = pd.DataFrame(rows)
    save(d9, "d9_overnight_cutoffs.csv")
    print(d9.round(3).to_string(index=False))

    # ---------------------------------------------------------------- D11
    hdr("D11. INFLUENCE OF TINY CELLS (sex 'Missing' n = 2, admission year 1400 n = 10)")
    tiny = (DM["sex"] == "Missing") | (DM["adm_year"] == "1400")
    rows = []
    for name, d in [("all episodes", DM), ("without tiny-cell episodes", DM[~tiny])]:
        for key in ["claim_lines", "los", "overnight"]:
            try:
                res, _ = fit(d, key, M5)
                rows += tidy(res, key, name, only=["billing_mode", "group", "any_surgical"])
            except Exception as e:
                print(name, key, "failed:", e)
    d11 = pd.DataFrame(rows)
    save(d11, "d11_tiny_cells.csv")
    print(f"Episodes removed: {int(tiny.sum())}")
    print(d11.pivot_table(index=["outcome", "term"], columns="scenario", values="ratio").round(3).to_string())

# =============================================================================
# D10. SENSITIVITY DIRECTION REDONE WITH CONFIDENCE INTERVALS
# =============================================================================
hdr("D10. SENSITIVITY DIRECTION RULE USING CONFIDENCE INTERVALS")
fm, fs = O1_DIR / "o1_models_long.csv", O1_DIR / "o1_sensitivity_long.csv"
if fm.exists() and fs.exists():
    M = pd.read_csv(fm)
    S = pd.read_csv(fs)
    M = M[(M.model == "M5") & (M.outcome.isin(["claim_lines", "overnight", "los"]))]
    M = M[~M.term.str.contains("adm_|age_group|sex|discharge|insurance_fund")]
    M["main_sig"] = (M.lo > 1) | (M.hi < 1)
    rows = []
    for _, r in M.iterrows():
        s = S[(S.outcome == r.outcome) & (S.term == r.term)]
        if s.empty:
            continue
        same = ((s.ratio > 1) == (r.ratio > 1))
        sig_same = same & ((s.lo > 1) | (s.hi < 1))
        rows.append({"outcome": r.outcome, "term": r.term, "main_ratio": round(r.ratio, 3),
                     "main_CI_excludes_1": bool(r.main_sig), "n_sens": len(s),
                     "same_direction": int(same.sum()), "same_direction_and_significant": int(sig_same.sum()),
                     "robust_if_main_significant": bool(r.main_sig and same.all() and sig_same.all())})
    d10 = pd.DataFrame(rows)
    save(d10, "d10_sensitivity_direction_with_ci.csv")
    print("Terms whose main CI excludes 1 (only these can be called robust or not):")
    print(d10[d10.main_CI_excludes_1].to_string(index=False))
    print("\nTerms with main CI including 1: a flipped sign there is noise, not fragility "
          f"({int((~d10.main_CI_excludes_1).sum())} terms).")
else:
    print("o1_models_long.csv / o1_sensitivity_long.csv not found in", O1_DIR)

print("\nDone. Tables in", OUT)

Rows: 15,601; overnight share 68.4%

D1. SPARSE CELLS (n < 30, or < 10 overnight / non-overnight episodes)
              variable                                       level  n  overnight  not_overnight  flag_n_lt30  flag_separation_risk  overnight_pct
                   sex                                     Missing  2          2              0         True                  True          100.0
  insurance_fund_model                                       Other  4          2              2         True                  True           50.0
discharge_status_model                                       Other 32         24              8        False                  True           75.0
        province_model                              North Khorasan 34          7             27        False                  True           20.6
        province_model                                      Qazvin 35         28              7        False                  True           80.0
       specialty_

In [14]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 1 (resource utilisation) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".   VERSION 3

Runs on the FROZEN analysis file written by clean_digestive_surgical_episodes.py (read only).
Needs only numpy, pandas and scipy (no statsmodels / patsy): the GLM, Firth and cluster-robust
engines are written out below.  They were checked against statsmodels 0.14.5 on the real file:
dispersion, rate ratios and Firth odds ratios agree to four decimals.

OUTCOMES                                                         MODEL
  1. Claim lines per episode        quasi-Poisson, log link       rate ratio (RR)
  2. Any overnight stay (LOS >= 1)  Firth penalised logistic      odds ratio (OR)
  3. LOS among overnight stays      Gamma-log GLM on P99-capped   ratio of means
                                    LOS; quasi-Poisson as check
BLOCKS (Andersen), every model also carries Jalali year and month
  M1 age group, sex | M2 + fund, ownership, billing mode | M3 + group, surgical service, discharge
  status | M4 + province | M5 + specialty

------------------------------------------------------------------------------------------------
WHAT CHANGED IN VERSION 3 AND WHY (all of it is driven by the diagnostic output on the real file)
------------------------------------------------------------------------------------------------
  1. PRIMARY SAMPLE = ADMISSIONS (record types Global + Inpatient, n = 14,685).
     Emergency Observation (889 episodes: LOS 0.08 d, 1% overnight, 8% surgical) and Temporary
     inpatient admission (27) are not admissions, yet they sat inside the "Other" billing mode.
     They inflated the surgical-service odds ratio from about 5 to 12.5 and pulled every Global-vs-
     other contrast.  The protocol's sample (all valid episodes) is still fitted and compared
     (o1_sample_comparison.csv; use --sample all to make it primary).  THIS IS A DEVIATION FROM THE
     FROZEN PROTOCOL: state it in the Methods and label it as made after seeing the diagnostics.
  2. BILLING MODE IS ANALYSED BY PROVINCE.  The Global-minus-itemised gap in overnight share runs from
     -21 to +66 percentage points across provinces (billing x province chi2 = 501).  A pooled OR is an
     average of opposite effects.  The script now fits province-specific billing effects, pools them
     with a DerSimonian-Laird random-effects model and reports I2, tau and the 95% prediction interval.
  3. MODELS STRATIFIED BY BILLING MODE (Global only / Inpatient only) because group x billing and
     surgical service x billing interactions are large (e.g. Gallbladder overnight OR 9.1 in Global vs
     3.1 in Inpatient; claim-line RR 0.93 vs 1.15).  Surgical service is almost constant inside Global
     (99.6%), so its effect is only estimable in Inpatient.
  4. TWO SETS OF STANDARD ERRORS.  Patient clusters are almost singletons (15,214 clusters for 15,601
     episodes), so they equal ordinary robust SEs; province clusters are 2-8x wider for facility-level
     exposures (ICC after M3 = 0.18 for claim lines).  Both are reported; province-cluster CIs use t with
     G-1 df.  Province coefficients themselves have no province-cluster SE (they are absorbed).
  5. SPARSE CELLS (kept from v2): level flagged when n < 30 or < 10 in an outcome class; Firth sandwich
     SE replaced by profile penalised-likelihood CI (M5) or the larger model SE when the model SE is
     > 1.5x the sandwich SE.
  6. EXPLICIT RELIABILITY VERDICT per main-table estimate (o1_reliability_summary.csv):
        A  direction and size hold      B  direction holds, size varies by billing mode
        C  not robust (the column `why` lists the failed criteria)
     The pre-specified rule (direction holds in all sensitivity analyses) is one of the criteria.
  7. Tables now print the number of episodes behind each level; the specialty and province rows with
     fewer than 100 episodes are marked as not interpretable on their own.
  Sensitivity 8 (province random intercept) is still NOT run here; the province-clustered models and
  the by-province random-effects analysis are Python stand-ins, not replacements.

OUTPUTS (folder objective1_outputs next to the analysis file)
  o1_sample_flow.csv, o1_descriptives.csv
  o1_models_long.csv (patient clusters) / o1_models_province_cluster.csv (M5, province clusters)
  o1_main_table.csv, o1_main_table_stratified.csv
  o1_billing_by_province.csv, o1_billing_pooled.csv
  o1_reliability_summary.csv
  o1_sensitivity_long.csv, o1_sensitivity_direction.csv
  o1_sample_comparison.csv, o1_exploratory_long.csv, o1_exploratory_core_terms.csv
  o1_diagnostics_*.csv (dispersion, gvif, m5_vs_m3, sparse_cells, firth_vs_ml, interactions,
                        icc_province, se_by_cluster_level, los_bins, los_top_values, overnight_cutoffs,
                        record_type)
  o1_run_info.json

USAGE
  python objective1_resource_utilisation.py [analysis_file.csv] [--sample admissions|all]
                                            [--no-sens] [--no-extra] [--no-profile]
"""

import argparse
import hashlib
import json
import platform
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import expit

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
RARE_MIN = 30             # protocol rule (discharge status, counts only)
VIF_FLAG = 5.0
TOP_PROVINCES = 5
SPARSE_N_MIN = 30         # level flagged when it has fewer episodes than this ...
SPARSE_CELL_MIN = 10      # ... or fewer than this in either outcome class (separation risk)
SE_NARROW_RATIO = 1.5     # model SE / sandwich SE above this -> sandwich not trusted
MIN_LEVEL_N = 100         # levels below this are shown but marked "not interpretable"
SPREAD_FLAG = 1.5         # stratum-specific ratios differing by more than this factor = "size varies"
PROV_MIN_PER_MODE = 30    # provinces enter the billing-by-province analysis with >= this many per mode
OVERNIGHT_CUTOFFS = [0.5, 0.75]
LOPO_N = 8
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--no-profile", action="store_true")
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_EXTRA, RUN_PROFILE = not args.no_sens, not args.no_extra, not args.no_profile
OUT_DIR = INPUT_CSV.parent / "objective1_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode",
           "group", "any_surgical_service", "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
TIME = ["adm_year", "adm_month"]
NO_BILLING = lambda b: [t for t in b if t != "billing_mode"]
OUTCOMES = {"claim_lines": ("claim_lines", "quasipoisson"),
            "overnight": ("overnight", "firth"),
            "los": ("los_capped_p99", "gamma")}
EFFECT_LABEL = {"claim_lines": "RR", "overnight": "OR", "los": "ratio", "los_qp": "RR"}
CORE_VARS = ["age_group", "billing_mode", "group", "any_surgical_service"]
REPORT_VARS = ["ownership", "billing_mode", "group", "any_surgical_service", "specialty_model", "province_model"]


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


# =============================================================================
# 1. LOAD, VERIFY, DEFINE THE PRIMARY SAMPLE
# =============================================================================
print("=" * 78)
print("1. LOAD AND VERIFY")
print("=" * 78)
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need_cols = ["episode_no", "icd10", "group", "claim_lines", "los_days", "overnight", "los_capped_p99",
             "los_capped_p95", "cluster_id", "any_surgical_service", "specialty_model", "province_model",
             "insurance_fund_model", "ownership", "billing_mode", "record_type", "discharge_status",
             "age_group", "sex", "adm_year", "adm_month", "anonymous", "keep_sens1_surgical",
             "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
             "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine"]
missing = [c for c in need_cols if c not in raw.columns]
if missing:
    fail(f"Columns missing from the analysis file: {missing}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous"] + [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["claim_lines", "los_days", "overnight", "los_capped_p99", "los_capped_p95"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")

cnt = DALL["discharge_status"].value_counts()
rare = cnt[cnt < RARE_MIN].index.tolist()
DALL["discharge_status_model"] = DALL["discharge_status"].where(~DALL["discharge_status"].isin(rare), "Other")
print(f"discharge_status: {len(cnt)} categories, {len(rare)} collapsed to 'Other'.")

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
flow.append({"step": "  of which record type Global", "n": int((DALL.record_type == "Global").sum())})
flow.append({"step": "  of which record type Inpatient (itemised)", "n": int((DALL.record_type == "Inpatient").sum())})
for t in NON_ADMISSION:
    flow.append({"step": f"  of which record type {t} (not an admission)", "n": int((DALL.record_type == t).sum())})
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types found: {unknown_rt}; they are treated as admissions.")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
D = DALL[is_adm].copy() if args.sample == "admissions" else DALL.copy()
ALT = DALL.copy() if args.sample == "admissions" else DALL[is_adm].copy()
SAMPLE_LABEL = ("admissions (Global + Inpatient)" if args.sample == "admissions"
                else "all valid episodes (protocol sample)")
ALT_LABEL = "all valid episodes (protocol sample)" if args.sample == "admissions" else "admissions only"
flow.append({"step": f"PRIMARY SAMPLE: {SAMPLE_LABEL}", "n": len(D)})
save(pd.DataFrame(flow), "o1_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))

print("\nOutcome checks (primary sample)")
print(f"  claim_lines: median {D.claim_lines.median():.0f}, IQR {D.claim_lines.quantile(.25):.0f}-"
      f"{D.claim_lines.quantile(.75):.0f}, min {D.claim_lines.min():.0f}, max {D.claim_lines.max():.0f}, "
      f"missing {D.claim_lines.isna().sum()}")
print(f"  overnight stay: {100 * D.overnight.mean():.1f}% (missing {D.overnight.isna().sum()})")
print(f"  LOS among overnight stays: n = {int((D.overnight == 1).sum()):,}, "
      f"median {D.loc[D.overnight == 1, 'los_days'].median():.2f} d")

CAT_VARS = sorted({v for b in BLOCKS.values() for v in b} | set(TIME))
REFS = {v: D[v].value_counts().index[0] for v in CAT_VARS}
REFS["billing_mode"] = "Other"             # Other = itemised inpatient in the primary sample
REFS["any_surgical_service"] = "No"
REFS["adm_month"] = "1"
print("Reference levels:", {k: REFS[k] for k in BLOCKS["M5"]})
CUT_COL = {}
for c in OVERNIGHT_CUTOFFS:
    col = f"overnight_c{int(round(c * 100)):03d}"
    DALL[col] = np.where(DALL["los_days"].notna(), (DALL["los_days"] >= c).astype(float), np.nan)
    CUT_COL[c] = col
    D[col] = DALL.loc[D.index, col]
    ALT[col] = DALL.loc[ALT.index, col]


# =============================================================================
# 2. MODEL ENGINE (numpy / scipy)
# =============================================================================
def design(d, terms, extra_inter=()):
    """Dummy design matrix. Columns are named like C(var)[T.level]; the reference is REFS[var]
    (or the most common level in this sample when REFS[var] is absent). Constant columns are dropped."""
    cols, names, levs = [np.ones(len(d))], ["Intercept"], {}
    for v in terms:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = REFS.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        for lev in levs[v][1:]:
            cols.append((s == lev).to_numpy(float))
            names.append(f"C({v})[T.{lev}]")
    for a, b in extra_inter:
        for la in levs[a][1:]:
            for lb in levs[b][1:]:
                cols.append(((d[a].astype(str) == la) & (d[b].astype(str) == lb)).to_numpy(float))
                names.append(f"C({a})[T.{la}]:C({b})[T.{lb}]")
    X = np.column_stack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], levs


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def glm_log(X, y, groups, fam):
    """Log-link GLM (Poisson or Gamma) by IRLS with Liang-Zeger cluster-robust covariance (CR1)."""
    n, k = X.shape
    wf = (lambda m: m) if fam == "poisson" else (lambda m: np.ones_like(m))
    beta = np.zeros(k)
    beta[0] = np.log(y.mean())
    mu = np.full(n, y.mean())
    eta = np.log(mu)
    for _ in range(100):
        w = wf(mu)
        z = eta + (y - mu) / mu
        XtW = X.T * w
        b_new = np.linalg.solve(XtW @ X, XtW @ z)
        eta = X @ b_new
        mu = np.exp(np.clip(eta, -30, 30))
        done = np.abs(b_new - beta).max() < 1e-10
        beta = b_new
        if done:
            break
    bread = np.linalg.inv((X.T * wf(mu)) @ X)
    V = mu if fam == "poisson" else mu ** 2
    meat, g = cluster_meat(X * ((y - mu) / V * mu)[:, None], groups)
    cov = (g / (g - 1)) * ((n - 1) / (n - k)) * bread @ meat @ bread
    return {"beta": beta, "cov": cov, "G": g, "disp": float(np.sum((y - mu) ** 2 / V) / (n - k)),
            "resid": (y - mu) / np.sqrt(V), "conv": bool(done)}


def _pen_ll(beta, X, y):
    eta = X @ beta
    w = expit(eta) * (1 - expit(eta))
    ev = np.linalg.eigvalsh(X.T @ (X * w[:, None]))
    return np.sum(y * eta - np.logaddexp(0, eta)) + 0.5 * np.sum(np.log(ev[ev > 1e-10]))


def firth_irls(X, y, fixed=None, beta0=None, maxiter=200, tol=1e-8):
    """Firth penalised logistic regression; fixed=(j, value) holds beta_j fixed (profile likelihood)."""
    n, k = X.shape
    beta = np.zeros(k) if beta0 is None else np.array(beta0, float)
    free = np.arange(k)
    if fixed is not None:
        beta[fixed[0]] = fixed[1]
        free = np.array([i for i in range(k) if i != fixed[0]])
    conv = False
    for _ in range(maxiter):
        p = expit(X @ beta)
        w = p * (1 - p)
        info = X.T @ (X * w[:, None])
        inv = np.linalg.pinv(info)
        A = X * np.sqrt(w)[:, None]
        h = np.einsum("ij,ij->i", A @ inv, A)
        u = X.T @ (y - p + h * (0.5 - p))
        step = np.zeros(k)
        if fixed is None:
            step = inv @ u
        else:
            step[free] = np.linalg.pinv(info[np.ix_(free, free)]) @ u[free]
        mx = np.abs(step).max()
        if mx > 5:
            step *= 5 / mx
        f0, t = _pen_ll(beta, X, y), 1.0
        while t > 1e-4 and _pen_ll(beta + t * step, X, y) < f0 - 1e-10:
            t /= 2
        beta = beta + t * step
        # converged when the step is tiny OR the penalised score is ~0 (the line search can stall on
        # floating-point noise in sparse designs although the penalised likelihood is already flat)
        if np.abs(t * step).max() < tol or np.abs(u[free]).max() < 1e-4:
            conv = True
            break
    return beta, conv, _pen_ll(beta, X, y)


def firth_fit(X, y, groups):
    n, k = X.shape
    beta, conv, pl = firth_irls(X, y)
    p = expit(X @ beta)
    w = p * (1 - p)
    inv = np.linalg.pinv(X.T @ (X * w[:, None]))
    A = X * np.sqrt(w)[:, None]
    h = np.einsum("ij,ij->i", A @ inv, A)
    meat, g = cluster_meat(X * (y - p + h * (0.5 - p))[:, None], groups)
    cov = ((g / (g - 1)) * ((n - 1) / max(n - k, 1)) if g > 1 else 1.0) * inv @ meat @ inv
    return {"beta": beta, "cov": cov, "cov_model": inv, "G": g, "conv": conv, "pl": pl,
            "resid": (y - p) / np.sqrt(w)}


def firth_profile_ci(X, y, j, beta, pl_max, se_hint, level=0.95):
    """Profile penalised-likelihood CI and LR p-value for coefficient j (valid under separation)."""
    crit = stats.chi2.ppf(level, 1)

    def g(b):
        return 2 * (pl_max - firth_irls(X, y, fixed=(j, b), beta0=beta)[2]) - crit

    bounds = []
    for sgn in (-1, 1):
        step, found = max(se_hint, 0.25), None
        for _ in range(10):
            if g(beta[j] + sgn * step) > 0:
                found = beta[j] + sgn * step
                break
            step *= 2
        if found is None:
            bounds.append(sgn * np.inf)
        else:
            a, c = (beta[j], found) if sgn > 0 else (found, beta[j])
            bounds.append(optimize.brentq(g, a, c, xtol=1e-4))
    pl0 = firth_irls(X, y, fixed=(j, 0.0), beta0=beta)[2]
    return bounds[0], bounds[1], stats.chi2.sf(max(2 * (pl_max - pl0), 0.0), 1)


def ml_logit(X, y, maxiter=60):
    """Plain maximum-likelihood logistic regression (damped Newton), only to show separation."""
    beta = np.zeros(X.shape[1])
    for _ in range(maxiter):
        p = expit(X @ beta)
        w = np.clip(p * (1 - p), 1e-12, None)
        step = np.linalg.pinv(X.T @ (X * w[:, None])) @ (X.T @ (y - p))
        mx = np.abs(step).max()
        if mx > 3:
            step *= 3 / mx
        beta = beta + step
        if np.abs(step).max() < 1e-8:
            break
    w = np.clip(expit(X @ beta) * (1 - expit(X @ beta)), 1e-12, None)
    return beta, np.sqrt(np.diag(np.linalg.pinv(X.T @ (X * w[:, None]))))


def icc_anova(resid, groups):
    df = pd.DataFrame({"r": np.asarray(resid, float), "g": np.asarray(groups)})
    gm = df.groupby("g")["r"]
    k, n = gm.ngroups, len(df)
    ni = gm.size().to_numpy()
    ssb = np.sum(ni * (gm.mean().to_numpy() - df["r"].mean()) ** 2)
    ssw = np.sum((df["r"] - gm.transform("mean")) ** 2)
    msb, msw = ssb / (k - 1), ssw / (n - k)
    n0 = (n - np.sum(ni ** 2) / n) / (k - 1)
    return (msb - msw) / (msb + (n0 - 1) * msw)


def term_flags(X, names, y_bin):
    """Sparse-cell flag per column: level n < 30, or (binary outcome) < 10 in an outcome class."""
    flags, ns = {}, {}
    for j, nm in enumerate(names):
        if nm == "Intercept":
            continue
        n_l = int(X[:, j].sum())
        f = n_l < SPARSE_N_MIN
        if y_bin is not None and n_l > 0:
            n1 = int((X[:, j] * y_bin).sum())
            f = f or min(n1, n_l - n1) < SPARSE_CELL_MIN
        flags[nm], ns[nm] = bool(f), n_l
    return flags, ns


def fit_core(d, key, terms, y_col=None, family=None, cluster="patient", extra_inter=(), profile=False):
    outcome_col, fam = OUTCOMES[key]
    outcome_col = y_col or outcome_col
    fam = family or fam
    sub = d.dropna(subset=[outcome_col]).copy()
    if key == "los":                                   # LOS among overnight stays only
        sub = sub[(sub["overnight"] == 1) & (sub[outcome_col] > 0)]
    X, names, _ = design(sub, terms + TIME, extra_inter)
    y = sub[outcome_col].to_numpy(float)
    groups = pd.factorize(sub["cluster_id" if cluster == "patient" else "province_model"])[0]
    core = {"n": len(sub), "fam": fam, "cluster": cluster, "names": names, "profile": {}, "sub_index": sub.index}
    if fam == "firth":
        f = firth_fit(X, y, groups)
        se_rob, se_mod = np.sqrt(np.diag(f["cov"])), np.sqrt(np.diag(f["cov_model"]))
        narrow = (se_mod / np.where(se_rob > 0, se_rob, np.nan) > SE_NARROW_RATIO) & (np.array(names) != "Intercept")
        se = np.where(narrow, np.maximum(se_rob, se_mod), se_rob)
        if profile and narrow.any():
            for j in np.where(narrow)[0]:
                try:
                    core["profile"][int(j)] = firth_profile_ci(X, y, int(j), f["beta"], f["pl"], se_mod[j])
                except Exception as e:
                    print(f"    profile CI failed for {names[j]}: {type(e).__name__}: {e}")
        core.update(beta=f["beta"], se=se, se_sandwich=se_rob, se_model=se_mod, cov=f["cov"], G=f["G"],
                    narrow=narrow, converged=f["conv"], dispersion=np.nan, family_label="Firth logit",
                    resid=f["resid"], X=X, y=y)
        core["flags"], core["lev_n"] = term_flags(X, names, y)
    else:
        r = glm_log(X, y, groups, "poisson" if fam == "quasipoisson" else "gamma")
        core.update(beta=r["beta"], se=np.sqrt(np.diag(r["cov"])), se_sandwich=np.sqrt(np.diag(r["cov"])),
                    se_model=np.full(len(names), np.nan), cov=r["cov"], G=r["G"], narrow=np.zeros(len(names), bool),
                    converged=r["conv"], dispersion=r["disp"], resid=r["resid"], X=X, y=y,
                    family_label="quasi-Poisson (Poisson + robust SE)" if fam == "quasipoisson" else "Gamma-log")
        core["flags"], core["lev_n"] = term_flags(X, names, None)
    return core


def make_tab(core, label, model, extra=None):
    beta, se, names, G = core["beta"], core["se"], core["names"], core["G"]
    use_t = core["cluster"] == "province"
    crit = stats.t.ppf(0.975, G - 1) if use_t else stats.norm.ppf(0.975)
    lo, hi = beta - crit * se, beta + crit * se
    z = beta / np.where(se > 0, se, np.nan)
    p = 2 * (stats.t.sf(np.abs(z), G - 1) if use_t else stats.norm.sf(np.abs(z)))
    base = "cluster-robust SE (province, t)" if use_t else "cluster-robust SE (patient)"
    method = np.array([base] * len(names), dtype=object)
    for j in np.where(core["narrow"])[0]:
        method[j] = "model-based SE (sandwich too narrow)"
    for j, (l, h_, pj) in core["profile"].items():
        lo[j], hi[j], p[j] = l, h_, pj
        method[j] = "profile penalised likelihood"
    out = pd.DataFrame({
        "outcome": label, "model": model, "term": names, "n": core["n"], "level_n": [core["lev_n"].get(t, np.nan) for t in names],
        "ratio": np.exp(beta), "lo": np.exp(lo), "hi": np.exp(hi), "p": p, "se_log": se,
        "se_log_sandwich": core["se_sandwich"], "se_log_model": core["se_model"], "ci_method": method,
        "sparse_flag": [bool(core["flags"].get(t, False)) for t in names], "cluster": core["cluster"]})
    if extra:
        for k, v in extra.items():
            out[k] = v
    return out[out["term"] != "Intercept"]


def fit_one(d, key, model, terms, label=None, quiet=False, **kw):
    label = label or key
    try:
        core = fit_core(d, key, terms, **kw)
        tab = make_tab(core, label, model)
        disp = {"outcome": label, "model": model, "n": core["n"], "family": core["family_label"],
                "dispersion": core["dispersion"], "converged": core["converged"], "cluster": core["cluster"]}
        if not quiet:
            extra = f", {int(core['narrow'].sum())} term(s) with sandwich SE too narrow" if core["narrow"].any() else ""
            print(f"  {label:<12} {model}: n = {core['n']:,}, terms = {len(tab)}"
                  + (f", dispersion = {disp['dispersion']:.2f}" if np.isfinite(disp["dispersion"]) else "")
                  + extra + ("" if disp["converged"] else "  ** did not converge **"))
        return tab, disp, core
    except Exception as e:
        print(f"  {label:<12} {model}: FAILED ({type(e).__name__}: {e})")
        return None, None, None


# =============================================================================
# 3. DESCRIPTIVES
# =============================================================================
def iqr_str(s, dec=1):
    s = s.dropna()
    return "" if len(s) == 0 else f"{s.median():.{dec}f} ({s.quantile(.25):.{dec}f}-{s.quantile(.75):.{dec}f})"


def describe(d, var, label):
    rows = []
    for lev, g in ([("All", d)] if var is None else d.groupby(var, observed=True)):
        gi = g[g["overnight"] == 1]
        rows.append({"variable": label, "level": lev, "episodes": len(g),
                     "claim_lines_median_IQR": iqr_str(g["claim_lines"], 0),
                     "overnight_n": int((g["overnight"] == 1).sum()),
                     "overnight_pct": round(100 * g["overnight"].mean(), 1),
                     "LOS_all_median_IQR": iqr_str(g["los_days"], 2),
                     "LOS_overnight_median_IQR": iqr_str(gi["los_days"], 2),
                     "surgical_service_pct": round(100 * (g["any_surgical_service"] == "Yes").mean(), 1)})
    return rows


print("\n" + "=" * 78)
print("2. DESCRIPTIVES")
print("=" * 78)
desc = describe(DALL, None, "All valid episodes (protocol sample)") + describe(DALL, "record_type", "Record type (all episodes)")
desc += describe(D, None, "PRIMARY SAMPLE")
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("billing_mode", "Billing mode"),
                 ("specialty_model", "Specialty")]:
    desc += describe(D, var, lab)
desc = pd.DataFrame(desc)
save(desc, "o1_descriptives.csv")
print(desc[desc.variable.isin(["All valid episodes (protocol sample)", "Record type (all episodes)", "PRIMARY SAMPLE",
                               "Billing mode", "Group"])].to_string(index=False))

lb = pd.cut(DALL["los_days"], [0, .25, .5, .75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0, np.inf], right=False)
los_bins = (pd.crosstab(D["group"], pd.cut(D["los_days"], lb.cat.categories, right=False), normalize="index") * 100).round(1)
los_bins.columns = [str(c) for c in los_bins.columns]
save(los_bins.reset_index(), "o1_diagnostics_los_bins.csv")
top = D["los_days"].round(2).value_counts().head(15).rename_axis("los_rounded").reset_index(name="episodes")
save(top, "o1_diagnostics_los_top_values.csv")
cut = pd.DataFrame({"cutoff_days": [0.25, 0.5, 0.75, 1.0, 1.25]})
cut["overnight_pct"] = [round(100 * (D["los_days"].dropna() >= c).mean(), 1) for c in cut.cutoff_days]
save(cut, "o1_diagnostics_overnight_cutoffs.csv")
save(pd.DataFrame(describe(DALL, "record_type", "Record type")), "o1_diagnostics_record_type.csv")

sp_rows = []
for v in sorted(set(BLOCKS["M5"]) | set(TIME)):
    for lev, r in D.groupby(v)["overnight"].agg(n="size", overnight="sum").iterrows():
        n_l, n1 = int(r["n"]), int(r["overnight"])
        sp_rows.append({"variable": v, "level": lev, "n": n_l, "overnight": n1, "not_overnight": n_l - n1,
                        "flag_n_lt30": n_l < SPARSE_N_MIN, "flag_n_lt100": n_l < MIN_LEVEL_N,
                        "flag_separation_risk": min(n1, n_l - n1) < SPARSE_CELL_MIN or n_l < SPARSE_N_MIN})
SPARSE = pd.DataFrame(sp_rows)
save(SPARSE, "o1_diagnostics_sparse_cells.csv")
print(f"\nSparse cells (n < {SPARSE_N_MIN} or < {SPARSE_CELL_MIN} in an outcome class): "
      f"{int(SPARSE.flag_separation_risk.sum())} levels; levels with n < {MIN_LEVEL_N}: {int(SPARSE.flag_n_lt100.sum())}.")

# =============================================================================
# 4. MAIN MODELS M1-M5 (patient clusters, as pre-specified)
# =============================================================================
print("\n" + "=" * 78)
print(f"3. MODELS M1-M5, primary sample: {SAMPLE_LABEL}, n = {len(D):,}")
print("=" * 78)
warnings.filterwarnings("ignore")
long_tabs, disp_rows, CORE = [], [], {}
for model, blocks in BLOCKS.items():
    prof = RUN_PROFILE and model == "M5"
    for key in OUTCOMES:
        tab, disp, core = fit_one(D, key, model, blocks, profile=prof)
        if tab is not None:
            long_tabs.append(tab); disp_rows.append(disp); CORE[(key, model)] = core
    tab, disp, core = fit_one(D, "los", model, blocks, family="quasipoisson", label="los_qp")
    if tab is not None:
        long_tabs.append(tab); disp_rows.append(disp); CORE[("los_qp", model)] = core
LONG = pd.concat(long_tabs, ignore_index=True)
LONG["analysis_status"] = "pre-specified models; primary sample redefined after diagnostics"
save(LONG, "o1_models_long.csv")
save(pd.DataFrame(disp_rows), "o1_diagnostics_dispersion.csv")

# M5 with province-clustered SEs (all outcomes)
print("\nModel M5 with province-clustered SEs (28 clusters; t with G-1 df)")
pc_tabs = []
for key in list(OUTCOMES) + ["los_qp"]:
    k_, fam_ = ("los", "quasipoisson") if key == "los_qp" else (key, None)
    tab, _, _ = fit_one(D, k_, "M5", BLOCKS["M5"], label=key, family=fam_, cluster="province")
    if tab is not None:
        pc_tabs.append(tab)
PC = pd.concat(pc_tabs, ignore_index=True)
PC = PC[~PC.term.str.contains("province_model")]       # province coefficients have no province-cluster SE
PC["analysis_status"] = "exploratory: province-clustered SE"
save(PC, "o1_models_province_cluster.csv")

# =============================================================================
# 5. STRATIFIED MODELS BY BILLING MODE, M5 (Global only / itemised only)
# =============================================================================
print("\n" + "=" * 78)
print("4. M5 STRATIFIED BY BILLING MODE (Global only | Other = itemised only)")
print("=" * 78)
ST_tabs = []
for lab, mask in [("Global", D.billing_mode == "Global"), ("Other", D.billing_mode != "Global")]:
    sub = D[mask]
    for key in OUTCOMES:
        for cl in ["patient", "province"]:
            tab, _, _ = fit_one(sub, key, f"M5 | {lab}", NO_BILLING(BLOCKS["M5"]), label=key, cluster=cl,
                                profile=RUN_PROFILE and cl == "patient", quiet=(cl == "province"))
            if tab is not None:
                tab["stratum"] = lab
                ST_tabs.append(tab)
ST = pd.concat(ST_tabs, ignore_index=True)
ST["analysis_status"] = "exploratory (added after diagnostics)"
save(ST, "o1_models_stratified.csv")

# =============================================================================
# 6. BILLING MODE BY PROVINCE, random-effects pooling
# =============================================================================
print("\n" + "=" * 78)
print("5. BILLING-MODE EFFECT BY PROVINCE (Global vs itemised), random-effects pooling")
print("=" * 78)


def drsl(b, se):
    w = 1 / se ** 2
    fe = np.sum(w * b) / w.sum()
    Q = np.sum(w * (b - fe) ** 2)
    k = len(b)
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - np.sum(w ** 2) / w.sum()))
    ws = 1 / (se ** 2 + tau2)
    re, se_re = np.sum(ws * b) / ws.sum(), np.sqrt(1 / ws.sum())
    t = stats.t.ppf(.975, k - 2)
    pi_half = t * np.sqrt(tau2 + se_re ** 2)
    return {"k": k, "fixed": fe, "re": re, "se_re": se_re, "tau": np.sqrt(tau2),
            "I2_pct": 100 * max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0, "Q": Q,
            "Q_p": stats.chi2.sf(Q, k - 1), "pi_lo": re - pi_half, "pi_hi": re + pi_half}


BP_rows, BP_pool = [], []
if args.sample == "all":
    print("  (skipped: the billing-by-province analysis needs the admissions-only sample, where "
          "Other = itemised inpatient)")
    BPD = None
else:
    ct = pd.crosstab(D.province_model, D.billing_mode)
    PROV = [p for p in ct.index if ct.loc[p].min() >= PROV_MIN_PER_MODE and p != "Other"]
    BPD = D[D.province_model.isin(PROV)]
    ref_prov = BPD.province_model.value_counts().index[0]
    REFS_SAVE = REFS.get("province_model")
    REFS["province_model"] = ref_prov
    print(f"  {len(PROV)} provinces with >= {PROV_MIN_PER_MODE} Global and >= {PROV_MIN_PER_MODE} itemised "
          f"episodes ({len(BPD):,} episodes); reference province: {ref_prov}")
    for key in OUTCOMES:
        core = fit_core(BPD, key, BLOCKS["M5"], extra_inter=[("billing_mode", "province_model")])
        names, beta, cov = core["names"], core["beta"], core["cov"]
        jb = names.index("C(billing_mode)[T.Global]")
        est, sel, used = [], [], []
        for p in PROV:
            L = np.zeros(len(names)); L[jb] = 1
            nm = f"C(billing_mode)[T.Global]:C(province_model)[T.{p}]"
            if nm in names:
                L[names.index(nm)] = 1
            elif p != ref_prov:
                continue
            e, s = float(L @ beta), float(np.sqrt(L @ cov @ L))
            g_ = BPD[BPD.province_model == p]
            raw_ = {"overnight_Global_pct": 100 * g_[g_.billing_mode == "Global"].overnight.mean(),
                    "overnight_Other_pct": 100 * g_[g_.billing_mode != "Global"].overnight.mean()}
            BP_rows.append({"outcome": key, "province": p, "n_Global": int((g_.billing_mode == "Global").sum()),
                            "n_Other": int((g_.billing_mode != "Global").sum()), "ratio": np.exp(e),
                            "lo": np.exp(e - 1.96 * s), "hi": np.exp(e + 1.96 * s), **raw_})
            est.append(e); sel.append(s); used.append(p)
        est, sel = np.array(est), np.array(sel)
        r = drsl(est, sel)
        pool = {"outcome": key, "provinces": r["k"], "pooled_ratio_RE": np.exp(r["re"]),
                "pooled_lo": np.exp(r["re"] - 1.96 * r["se_re"]), "pooled_hi": np.exp(r["re"] + 1.96 * r["se_re"]),
                "pooled_ratio_fixed": np.exp(r["fixed"]), "I2_pct": r["I2_pct"], "tau_log": r["tau"],
                "Q_p": r["Q_p"], "pred_interval_lo": np.exp(r["pi_lo"]), "pred_interval_hi": np.exp(r["pi_hi"]),
                "provinces_ratio_below_1": int((est < 0).sum()), "provinces_ratio_above_1": int((est > 0).sum()),
                "min_province_ratio": np.exp(est.min()), "max_province_ratio": np.exp(est.max()),
                "same_direction_all_provinces": bool((est < 0).all() or (est > 0).all())}
        BP_pool.append(pool)
        print(f"  {key:<12} RE ratio {pool['pooled_ratio_RE']:.2f} ({pool['pooled_lo']:.2f}-{pool['pooled_hi']:.2f}); "
              f"I2 = {pool['I2_pct']:.0f}%; 95% PI {pool['pred_interval_lo']:.2f}-{pool['pred_interval_hi']:.2f}; "
              f"provinces <1 / >1: {pool['provinces_ratio_below_1']}/{pool['provinces_ratio_above_1']}")
    if REFS_SAVE is not None:
        REFS["province_model"] = REFS_SAVE
    BP = pd.DataFrame(BP_rows)
    BP["analysis_status"] = "exploratory (added after diagnostics)"
    BPP = pd.DataFrame(BP_pool)
    BPP["analysis_status"] = "exploratory (added after diagnostics)"
    save(BP, "o1_billing_by_province.csv")
    save(BPP, "o1_billing_pooled.csv")

# =============================================================================
# 7. MAIN TABLE (M5)
# =============================================================================
vc = D["province_model"].value_counts()
TOP_PROV = [p for p in vc.index if p != "Other"][:TOP_PROVINCES]
SELECT = [("billing_mode", None), ("group", None), ("specialty_model", None), ("province_model", TOP_PROV)]
LEVELS = {v: [REFS[v]] + [x for x in D[v].value_counts().index if x != REFS[v]] for v in REFS if v in D.columns}


def fmt(r, mark_small=True):
    s = f"{r.ratio:.2f} ({r.lo:.2f}-{r.hi:.2f})"
    s += "" if pd.isna(r.p) else (f"; p={r.p:.3f}" if r.p >= 0.001 else "; p<0.001")
    if r.sparse_flag:
        s += " \u2020"
    elif mark_small and pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N:
        s += " \u2021"
    return s


def lookup(tabs, key, model, term, cluster="patient"):
    m = tabs[(tabs.outcome == key) & (tabs.model == model) & (tabs.term == term) & (tabs.cluster == cluster)]
    return m.iloc[0] if len(m) else None


def main_table(model="M5"):
    rows = []
    for var, only in SELECT:
        levels = LEVELS[var] if only is None else [REFS[var]] + [x for x in only if x != REFS[var]]
        for lev in levels:
            row = {"variable": var, "level": lev, "episodes": int((D[var] == lev).sum())}
            for key in ["claim_lines", "overnight", "los", "los_qp"]:
                lab = f"{EFFECT_LABEL[key]} {key}"
                if lev == REFS[var]:
                    row[lab] = "1 (ref)"
                    row[lab + " | province-cluster CI"] = "1 (ref)"
                    continue
                term = f"C({var})[T.{lev}]"
                a = lookup(LONG, key, model, term)
                row[lab] = fmt(a) if a is not None else ""
                b = lookup(PC.assign(model=model), key, model, term, "province")
                row[lab + " | province-cluster CI"] = (
                    f"{b.ratio:.2f} ({b.lo:.2f}-{b.hi:.2f}); p={b.p:.3f}" if b is not None
                    else ("not defined" if var == "province_model" else ""))
            rows.append(row)
    return pd.DataFrame(rows)


MAIN = main_table()
save(MAIN, "o1_main_table.csv")
print("\n" + "=" * 78)
print("6. MAIN TABLE, model M5 (patient-clustered CI | province-clustered CI)")
print("=" * 78)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 38)
print(MAIN[["variable", "level", "episodes", "RR claim_lines", "OR overnight", "ratio los"]].to_string(index=False))
print("\u2020 sparse cell (< 30 episodes or < 10 in an outcome class); \u2021 fewer than "
      f"{MIN_LEVEL_N} episodes: not interpretable on its own.")
print("The billing-mode row is a pooled average across provinces whose effects differ; "
      "use section 5 (by province, random effects) for billing mode.")

ST_MAIN_ROWS = []
for var, only in [("group", None), ("any_surgical_service", None), ("ownership", None), ("specialty_model", None),
                  ("province_model", TOP_PROV)]:
    levels = LEVELS[var] if only is None else [REFS[var]] + [x for x in only if x != REFS[var]]
    for lev in levels:
        row = {"variable": var, "level": lev}
        for stratum in ["Global", "Other"]:
            sub = D[D.billing_mode == "Global"] if stratum == "Global" else D[D.billing_mode != "Global"]
            row[f"episodes_{stratum}"] = int((sub[var] == lev).sum())
            for key in ["claim_lines", "overnight", "los"]:
                lab = f"{EFFECT_LABEL[key]} {key} | {stratum}"
                if lev == REFS[var]:
                    row[lab] = "1 (ref)"
                    continue
                m = ST[(ST.outcome == key) & (ST.stratum == stratum) & (ST.term == f"C({var})[T.{lev}]") & (ST.cluster == "patient")]
                row[lab] = fmt(m.iloc[0]) if len(m) else "not estimable"
        ST_MAIN_ROWS.append(row)
STM = pd.DataFrame(ST_MAIN_ROWS)
save(STM, "o1_main_table_stratified.csv")
print("\nStratified by billing mode (group and surgical service):")
print(STM[STM.variable.isin(["group", "any_surgical_service"])][["variable", "level", "RR claim_lines | Global",
      "RR claim_lines | Other", "OR overnight | Global", "OR overnight | Other"]].to_string(index=False))

# =============================================================================
# 8. DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 78)
print("7. DIAGNOSTICS")
print("=" * 78)


def gvif_table(X, names):
    keep = [i for i, nm in enumerate(names) if nm != "Intercept" and X[:, i].std() > 0]
    term_of = {i: names[i].split("[T.")[0] for i in keep}
    R = np.corrcoef(X[:, keep], rowvar=False)
    sign, ld_all = np.linalg.slogdet(R)
    rows = []
    for term in dict.fromkeys(term_of.values()):
        idx = [j for j, i in enumerate(keep) if term_of[i] == term]
        rest = [j for j in range(len(keep)) if j not in set(idx)]
        s1, ld_t = np.linalg.slogdet(R[np.ix_(idx, idx)])
        s2, ld_r = np.linalg.slogdet(R[np.ix_(rest, rest)])
        g = np.exp(ld_t + ld_r - ld_all) if sign > 0 and s1 > 0 and s2 > 0 else np.nan
        rows.append({"term": term, "df": len(idx), "GVIF": g, "VIF_equivalent": g ** (1 / len(idx)) if np.isfinite(g) else np.nan})
    out = pd.DataFrame(rows)
    out["flag_gt5"] = out["VIF_equivalent"] > VIF_FLAG
    return out


c5 = CORE[("claim_lines", "M5")]
GV = gvif_table(c5["X"], c5["names"])
save(GV, "o1_diagnostics_gvif.csv")
print("GVIF (M5): VIF_equivalent = GVIF^(1/(2df)) flag > 5 ->", "none flagged" if not GV.flag_gt5.any()
      else GV[GV.flag_gt5].term.tolist())

shared = [t for t in LONG.term.unique() if "province_model" not in t and "specialty_model" not in t]
m3 = LONG[(LONG.model == "M3") & LONG.term.isin(shared)].set_index(["outcome", "term"])["ratio"]
m5 = LONG[(LONG.model == "M5") & LONG.term.isin(shared)].set_index(["outcome", "term"])["ratio"]
CH = pd.concat([m3.rename("ratio_M3"), m5.rename("ratio_M5")], axis=1).dropna()
CH["pct_change_M5_vs_M3"] = 100 * (CH["ratio_M5"] / CH["ratio_M3"] - 1)
CH = CH.reset_index()
save(CH, "o1_diagnostics_m5_vs_m3.csv")
big = CH[(CH.pct_change_M5_vs_M3.abs() > 10) & CH.term.str.contains("ownership|billing|group|surgical")]
print(f"M5 vs M3: {int((CH.pct_change_M5_vs_M3.abs() > 10).sum())} of {len(CH)} estimates move > 10%; among the "
      f"main exposures: {len(big)}")
if len(big):
    print(big.round(2).to_string(index=False))
print("Dispersion, M3/M5:")
print(pd.DataFrame(disp_rows).query("model in ['M3','M5']").round(2).to_string(index=False))

try:
    cf = CORE[("overnight", "M5")]
    bml, seml = ml_logit(cf["X"], cf["y"])
    FM = pd.DataFrame({"term": cf["names"], "beta_firth": cf["beta"], "se_sandwich": cf["se_sandwich"],
                       "se_model": cf["se_model"], "beta_ml": bml, "se_ml": seml})
    FM["se_ratio_model_over_sandwich"] = FM.se_model / FM.se_sandwich
    FM["flag_sandwich_too_narrow"] = (FM.se_ratio_model_over_sandwich > SE_NARROW_RATIO) & (FM.term != "Intercept")
    save(FM, "o1_diagnostics_firth_vs_ml.csv")
    print("Sandwich SE too narrow (separation) for:", FM[FM.flag_sandwich_too_narrow].term.tolist() or "none")
except Exception as e:
    print(f"Firth vs ML failed: {type(e).__name__}: {e}")

icc_rows = []
for key in OUTCOMES:
    c3 = CORE[(key, "M3")]
    prov = D.loc[c3["sub_index"], "province_model"].to_numpy()
    icc_rows.append({"outcome": key, "icc_province_after_M3": icc_anova(c3["resid"], prov),
                     "n": c3["n"], "provinces": int(pd.Series(prov).nunique())})
ICC = pd.DataFrame(icc_rows)
save(ICC, "o1_diagnostics_icc_province.csv")
print("Province ICC after M3:", {r.outcome: round(r.icc_province_after_M3, 3) for r in ICC.itertuples()})

M5L = LONG[(LONG.model == "M5") & LONG.outcome.isin(list(OUTCOMES))]
CMP = M5L[["outcome", "term", "ratio", "lo", "hi", "se_log"]].merge(
    PC[PC.outcome.isin(list(OUTCOMES))][["outcome", "term", "ratio", "lo", "hi", "se_log"]],
    on=["outcome", "term"], suffixes=("_patient", "_province"))
CMP["se_ratio_province_over_patient"] = CMP.se_log_province / CMP.se_log_patient
CMP["sig_patient"] = (CMP.lo_patient > 1) | (CMP.hi_patient < 1)
CMP["sig_province"] = (CMP.lo_province > 1) | (CMP.hi_province < 1)
save(CMP, "o1_diagnostics_se_by_cluster_level.csv")
mainx = CMP[CMP.term.str.contains("ownership|billing|group|surgical")]
print(f"Province-clustered SE is {mainx.se_ratio_province_over_patient.median():.1f}x the patient-clustered SE "
      f"(median, main exposures); significance lost with province clusters: "
      f"{int((mainx.sig_patient & ~mainx.sig_province).sum())} of {int(mainx.sig_patient.sum())} significant terms.")

# interaction (heterogeneity) tests
INT_ROWS = []
for a, b in [("billing_mode", "group"), ("billing_mode", "any_surgical_service"), ("group", "any_surgical_service"),
             ("billing_mode", "province_model")]:
    if args.sample == "all" and a == "billing_mode" and b == "province_model":
        continue
    for key in OUTCOMES:
        try:
            core = fit_core(D, key, BLOCKS["M5"], extra_inter=[(a, b)])
            idx = [i for i, nm in enumerate(core["names"]) if ":" in nm and f"C({a})" in nm and f"C({b})" in nm]
            bb, V = core["beta"][idx], core["cov"][np.ix_(idx, idx)]
            chi2 = float(bb @ np.linalg.pinv(V) @ bb)
            INT_ROWS.append({"interaction": f"{a} x {b}", "outcome": key, "chi2": chi2, "df": len(idx),
                             "p_value": stats.chi2.sf(chi2, len(idx))})
        except Exception as e:
            print(f"  interaction {a} x {b} / {key} FAILED: {type(e).__name__}: {e}")
INT = pd.DataFrame(INT_ROWS)
INT["significant_bonferroni"] = INT.p_value < 0.05 / max(len(INT), 1)
save(INT, "o1_diagnostics_interactions.csv")
print("Interaction (heterogeneity) tests, M5:")
print(INT.round(4).to_string(index=False))

# =============================================================================
# 9. SENSITIVITY ANALYSES 1-7 (M5, primary sample)
# =============================================================================
SENS = {1: ("Surgical service only", "keep_sens1_surgical", None),
        2: ("Exclude Global packages", "keep_sens2_itemised", None),
        3: ("Exclude K81", "keep_sens3_not_K81", None),
        4: ("Exclude flagged extremes", "keep_sens4_no_extremes", None),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous", None),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine", None),
        7: ("LOS capped at P95", None, "los_capped_p95")}
SL = pd.DataFrame()
if RUN_SENS:
    print("\n" + "=" * 78)
    print("8. SENSITIVITY ANALYSES 1-7 (M5, primary sample)")
    print("=" * 78)
    tabs = []
    for s, (name, keepcol, los_col) in SENS.items():
        print(f"\nSensitivity {s}: {name}")
        sub = D[D[keepcol]] if keepcol else D
        for key in OUTCOMES:
            if s == 7 and key != "los":
                continue
            tab, _, _ = fit_one(sub, key, "M5", BLOCKS["M5"], y_col=los_col if key == "los" else None)
            if tab is not None:
                tab["sens"], tab["sens_name"] = s, name
                tabs.append(tab)
    SL = pd.concat(tabs, ignore_index=True)
    SL["analysis_status"] = "pre-specified analyses; primary sample redefined after diagnostics"
    save(SL, "o1_sensitivity_long.csv")

# =============================================================================
# 10. EXPLORATORY: protocol sample, cut-offs, tiny cells, leave-one-province-out
# =============================================================================
EXP = []


def add_exp(code, name, d, keys=("claim_lines", "overnight", "los"), y_cols=None, core_only=True, label_n=True):
    print(f"\nExploratory {code}: {name} (n = {len(d):,})")
    for key in keys:
        tab, _, _ = fit_one(d, key, "M5", BLOCKS["M5"], y_col=(y_cols or {}).get(key))
        if tab is not None:
            if core_only:
                tab = tab[tab.term.str.startswith(tuple(f"C({v})[" for v in CORE_VARS))]
            tab["analysis"], tab["analysis_name"] = code, name
            EXP.append(tab)


if RUN_EXTRA:
    print("\n" + "=" * 78)
    print("9. EXPLORATORY ANALYSES (report as exploratory)")
    print("=" * 78)
    add_exp("X0", f"Other sample: {ALT_LABEL}", ALT, core_only=False)
    for c in OVERNIGHT_CUTOFFS:
        add_exp(f"X5_{c}", f"Overnight stay defined as LOS >= {c} d", D, keys=("overnight",),
                y_cols={"overnight": CUT_COL[c]})
    tiny = pd.Series(False, index=D.index)
    for v in ["sex", "insurance_fund_model", "adm_year"]:
        vcn = D[v].value_counts()
        tiny |= D[v].isin(vcn[vcn < SPARSE_N_MIN].index)
    add_exp("X6", f"Episodes in cells < {SPARSE_N_MIN} (sex, fund, year) removed ({int(tiny.sum())})", D[~tiny])
    for p in [p for p in D["province_model"].value_counts().index if p != "Other"][:LOPO_N]:
        add_exp(f"X7_{p}", f"Without {p}", D[D["province_model"] != p])
    EXPL = pd.concat(EXP, ignore_index=True)
    EXPL["analysis_status"] = "exploratory (added after diagnostics)"
    save(EXPL, "o1_exploratory_long.csv")
    core_tab = EXPL[EXPL.term.str.startswith(tuple(f"C({v})[" for v in CORE_VARS))].merge(
        LONG[LONG.model == "M5"][["outcome", "term", "ratio"]].rename(columns={"ratio": "primary_M5_ratio"}),
        on=["outcome", "term"], how="left")
    save(core_tab[["analysis", "analysis_name", "outcome", "term", "n", "ratio", "lo", "hi", "p", "primary_M5_ratio",
                   "sparse_flag", "ci_method"]], "o1_exploratory_core_terms.csv")
    x0 = EXPL[EXPL.analysis == "X0"].merge(LONG[LONG.model == "M5"][["outcome", "term", "ratio", "lo", "hi"]],
                                           on=["outcome", "term"], suffixes=("_other_sample", "_primary"))
    x0 = x0[x0.term.str.contains("billing|group|surgical|ownership|age_group")]
    x0["pct_difference"] = 100 * (x0.ratio_other_sample / x0.ratio_primary - 1)
    x0["other_sample"] = ALT_LABEL
    save(x0[["outcome", "term", "other_sample", "ratio_primary", "ratio_other_sample", "pct_difference"]],
         "o1_sample_comparison.csv")
    print("\nSample comparison (main exposures), primary vs", ALT_LABEL)
    print(x0[x0.term.str.contains("billing|surgical|group")][["outcome", "term", "ratio_primary", "ratio_other_sample",
          "pct_difference"]].round(2).to_string(index=False))

# =============================================================================
# 11. RELIABILITY SUMMARY
# =============================================================================
print("\n" + "=" * 78)
print("10. RELIABILITY SUMMARY OF MAIN-TABLE ESTIMATES")
print("=" * 78)
prefix = tuple(f"C({v})[" for v in REPORT_VARS)
rel_rows = []
for key in OUTCOMES:
    main = LONG[(LONG.model == "M5") & (LONG.outcome == key) & LONG.term.str.startswith(prefix)]
    for _, r in main.iterrows():
        sig_p = bool((r.lo > 1) or (r.hi < 1))
        pcr = PC[(PC.outcome == key) & (PC.term == r.term)]
        sig_pc = None if "province_model" in r.term else (bool((pcr.iloc[0].lo > 1) or (pcr.iloc[0].hi < 1)) if len(pcr) else None)
        s = SL[(SL.outcome == key) & (SL.term == r.term)] if len(SL) else pd.DataFrame()
        sens_n = len(s)
        sens_dir = int(((s.ratio > 1) == (r.ratio > 1)).sum()) if sens_n else 0
        sens_sig = int((((s.ratio > 1) == (r.ratio > 1)) & ((s.lo > 1) | (s.hi < 1))).sum()) if sens_n else 0
        st = ST[(ST.outcome == key) & (ST.term == r.term) & (ST.cluster == "patient")]
        rg = st[st.stratum == "Global"].ratio
        ro = st[st.stratum == "Other"].ratio
        if "billing_mode" in r.term:
            strata_dir, spread = None, np.nan
        elif len(rg) and len(ro):
            strata_dir = bool(((rg.iloc[0] > 1) == (r.ratio > 1)) and ((ro.iloc[0] > 1) == (r.ratio > 1)))
            spread = float(max(rg.iloc[0], ro.iloc[0]) / min(rg.iloc[0], ro.iloc[0]))
        else:
            strata_dir, spread = None, np.nan
        why = []
        if "billing_mode" in r.term:
            bpp = BPP[BPP.outcome == key] if args.sample == "admissions" else pd.DataFrame()
            if len(bpp):
                b = bpp.iloc[0]
                re_sig = (b.pooled_lo > 1) or (b.pooled_hi < 1)
                if not b.same_direction_all_provinces:
                    why.append(f"direction differs across provinces ({int(b.provinces_ratio_below_1)} below 1, "
                               f"{int(b.provinces_ratio_above_1)} above 1)")
                if not re_sig:
                    why.append("random-effects pooled CI includes 1")
                verdict = "B" if (b.same_direction_all_provinces and re_sig) else "C"
                if verdict == "B":
                    why.append("same direction in every province; size varies (see I2 / prediction interval)")
            else:
                verdict = "C"; why.append("billing-by-province analysis not run")
        else:
            if not sig_p:
                why.append("CI includes 1 (patient clusters)")
            if sig_pc is False:
                why.append("CI includes 1 with province clusters")
            if sens_n and sens_dir < sens_n:
                why.append(f"direction fails in {sens_n - sens_dir} of {sens_n} sensitivity analyses")
            if strata_dir is False:
                why.append("direction differs between Global-only and itemised-only models")
            if strata_dir is None:
                why.append("not estimable in one billing stratum")
            if r.sparse_flag:
                why.append("sparse cell")
            if pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N:
                why.append(f"fewer than {MIN_LEVEL_N} episodes")
            hard = [w for w in why if not w.startswith("not estimable")]
            ok = (not hard) and strata_dir is not False
            if "province_model" in r.term:
                ok = sig_p and (not r.sparse_flag) and (sens_n == 0 or sens_dir == sens_n) and strata_dir is not False \
                    and not (pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N)
                why.append("province coefficient: cluster-robust SE not defined at province level; "
                           "treat as descriptive of recording/case-mix differences")
            if not ok:
                verdict = "C"
            elif pd.notna(spread) and spread > SPREAD_FLAG:
                verdict = "B"; why.append(f"size differs between billing strata (ratio of ratios {spread:.1f})")
            else:
                verdict = "A"
        rel_rows.append({"outcome": key, "term": r.term, "level_n": r.level_n, "ratio": round(r.ratio, 3),
                         "CI_patient": f"{r.lo:.2f}-{r.hi:.2f}",
                         "CI_province": (f"{pcr.iloc[0].lo:.2f}-{pcr.iloc[0].hi:.2f}" if len(pcr) else ""),
                         "sens_tested": sens_n, "sens_same_direction": sens_dir, "sens_same_direction_and_sig": sens_sig,
                         "ratio_Global_only": round(rg.iloc[0], 3) if len(rg) else np.nan,
                         "ratio_Other_only": round(ro.iloc[0], 3) if len(ro) else np.nan,
                         "strata_spread": round(spread, 2) if pd.notna(spread) else np.nan,
                         "sparse_flag": bool(r.sparse_flag), "verdict": verdict, "why": "; ".join(why)})
REL = pd.DataFrame(rel_rows)
save(REL, "o1_reliability_summary.csv")
save(REL[REL.term.str.contains("ownership|billing|group|surgical")] if len(REL) else REL, "o1_reliability_main_exposures.csv")
print(REL.groupby(["outcome", "verdict"]).size().unstack(fill_value=0))
print("\nMain exposures:")
print(REL[REL.term.str.contains("billing|group|surgical|ownership")][["outcome", "term", "ratio", "CI_patient", "CI_province",
      "verdict", "why"]].to_string(index=False, max_colwidth=70))

# =============================================================================
# 12. RUN INFO
# =============================================================================
info = {"script_version": "3", "input_file": str(INPUT_CSV), "input_sha256": sha256(INPUT_CSV),
        "rows_in_file": int(len(DALL)), "primary_sample": SAMPLE_LABEL, "primary_sample_n": int(len(D)),
        "reference_levels": {k: REFS[k] for k in BLOCKS["M5"] + TIME}, "top_provinces_in_table": TOP_PROV,
        "settings": {"sparse_n_min": SPARSE_N_MIN, "sparse_cell_min": SPARSE_CELL_MIN, "se_narrow_ratio": SE_NARROW_RATIO,
                     "min_level_n": MIN_LEVEL_N, "spread_flag": SPREAD_FLAG, "prov_min_per_mode": PROV_MIN_PER_MODE,
                     "profile_run": RUN_PROFILE, "overnight_cutoffs_exploratory": OVERNIGHT_CUTOFFS, "lopo_provinces": LOPO_N},
        "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                     "scipy": scipy.__version__},
        "notes": ["Primary sample redefined to admissions (Global + Inpatient) after the diagnostics: a deviation from the frozen protocol",
                  "Sensitivity 8 (province random intercept) not run here; province-clustered SEs and the by-province "
                  "random-effects analysis are Python stand-ins",
                  "Firth sandwich SE replaced by profile likelihood CI (M5) or the larger model SE where the model SE "
                  "is > 1.5x the sandwich SE",
                  "Billing x province, stratified, X0-X7, interaction and reliability analyses were added after the "
                  "diagnostic run and must be labelled exploratory"]}
with open(OUT_DIR / "o1_run_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2)
print("\n" + "=" * 78)
print(f"DONE. Outputs in {OUT_DIR}")
print("Sensitivity 8 (province as random intercept) is not run here; run it in R (lme4::glmer / glmmTMB) on the same file.")
print("=" * 78)

1. LOAD AND VERIFY
Rows: 15,601; columns: 70
discharge_status: 9 categories, 4 collapsed to 'Other'.
                                                                   step     n
                                   Episodes in the frozen analysis file 15601
                                            of which record type Global  8462
                              of which record type Inpatient (itemised)  6223
          of which record type Emergency Observation (not an admission)   889
  of which record type Temporary Inpatient Admission (not an admission)    27
                        PRIMARY SAMPLE: admissions (Global + Inpatient) 14685

Outcome checks (primary sample)
  claim_lines: median 4, IQR 2-10, min 1, max 23, missing 0
  overnight stay: 72.6% (missing 0)
  LOS among overnight stays: n = 10,660, median 1.61 d
Reference levels: {'age_group': '40-59', 'sex': 'Male', 'insurance_fund_model': 'Rural Health Insurance Fund', 'ownership': 'Public University-Affiliated', 'billing_mode

In [15]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 1, sensitivity analysis 8: province as a RANDOM INTERCEPT instead of fixed effects.

Same model blocks, sample and outcomes as objective1_resource_utilisation.py (primary sample =
admissions, model M5), but the province fixed effects are replaced by a random intercept
    eta_ij = x_ij' beta + u_j,   u_j ~ N(0, sigma_u^2),   j = province.
Fitted by maximum likelihood with the Laplace approximation (the default of lme4::glmer, nAGQ = 1).
Needs only numpy, pandas and scipy.  No R required.

  claim lines   Poisson, log link          -> rate ratio
  overnight     logistic (ordinary ML)     -> odds ratio   (Firth is not available in a mixed model)
  LOS           Gamma, log link            -> ratio of means (LOS among overnight stays, P99-capped)

Standard errors are model-based (inverse observed information, numerical).  Two cautions:
  * claim lines are UNDER-dispersed (Pearson dispersion about 0.5), so Poisson model SEs are too wide
    by roughly 1/sqrt(0.5); read those CIs as conservative.
  * a random INTERCEPT assumes the same billing-mode effect in every province. It does not fix the
    province-by-billing heterogeneity found in the main script (see o1_billing_by_province.csv); use that
    analysis for billing mode. Facility-level exposures (ownership, billing mode) also keep optimistic
    CIs here, because there is no hospital identifier.
  * overnight ML logistic does not handle sparse cells (separation): the levels flagged 'sparse'
    (n < 30 or < 10 in an outcome class) are not estimable and should be ignored.

Usage:  python objective1_sens8_province_random_intercept.py [analysis_file.csv]
Outputs (folder objective1_outputs next to the file):
  o1_sens8_random_intercept_long.csv        all fixed-effect terms
  o1_sens8_main_terms.csv                   main exposures next to the fixed-effect M5 results
  o1_sens8_variance_components.csv          sigma_u, ICC (latent scale for the logistic model)
"""
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize, stats
from scipy.special import expit, gammaln

INPUT_CSV = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
                 r"\analysis_ready_final\digestive_surgical_analysis_file.csv")
if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
    INPUT_CSV = Path(sys.argv[1])
OUT_DIR = INPUT_CSV.parent / "objective1_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
SPARSE_N_MIN, SPARSE_CELL_MIN, RARE_MIN = 30, 10, 30
M5_NO_PROV = ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode", "group",
              "any_surgical_service", "discharge_status_model", "specialty_model", "adm_year", "adm_month"]


# ----------------------------------------------------------------------------- GLMM engine
class Laplace:
    """Random-intercept GLMM, family in {'poisson', 'binomial', 'gamma'}, Laplace-approximated ML."""

    def __init__(self, X, y, g, fam):
        self.X, self.y, self.g, self.fam = X, y, g, fam
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.ng = np.bincount(g, minlength=self.G)
        self.u = np.zeros(self.G)
        self.const = -gammaln(y + 1).sum() if fam == "poisson" else 0.0

    def _l(self, eta, nu):
        y = self.y
        if self.fam == "poisson":
            return y * eta - np.exp(eta), y - np.exp(eta), -np.exp(eta)
        if self.fam == "binomial":
            p = expit(eta)
            return y * eta - np.logaddexp(0, eta), y - p, -p * (1 - p)
        e = np.exp(-eta)                                   # gamma, shape nu, mean exp(eta)
        l = nu * np.log(nu) - gammaln(nu) + (nu - 1) * np.log(y) - nu * y * e - nu * eta
        return l, nu * (y * e - 1), -nu * y * e

    def negll(self, theta, return_u=False):
        beta, sig = theta[:self.k], np.exp(theta[self.k])
        nu = np.exp(theta[self.k + 1]) if self.fam == "gamma" else None
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):                                # Newton for each province's mode
            l, d1, d2 = self._l(xb + u[self.g], nu)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        l, d1, d2 = self._l(xb + u[self.g], nu)
        h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
        ll = l.sum() + self.const - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h))
        return (-ll, u) if return_u else -ll

    def fit(self, beta0):
        th0 = np.concatenate([beta0, [np.log(0.5)], [np.log(3.0)] if self.fam == "gamma" else []])
        self.u[:] = 0
        res = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        res = optimize.minimize(self.negll, res.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        self.theta, self.res = res.x, res
        return res

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))

        def grad(t):
            return optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p); e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


def design(d, terms, refs):
    cols, names = [np.ones(len(d))], ["Intercept"]
    for v in terms:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        for lev in [ref] + [x for x in order if x != ref]:
            if lev == ref:
                continue
            cols.append((s == lev).to_numpy(float))
            names.append(f"C({v})[T.{lev}]")
    X = np.column_stack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep]


def poisson_start(X, y):
    beta = np.zeros(X.shape[1]); beta[0] = np.log(y.mean())
    for _ in range(50):
        mu = np.exp(X @ beta)
        b = np.linalg.solve((X.T * mu) @ X + 1e-8 * np.eye(X.shape[1]), (X.T * mu) @ (X @ beta + (y - mu) / mu))
        if np.abs(b - beta).max() < 1e-9:
            beta = b; break
        beta = b
    return beta


def logit_start(X, y):
    beta = np.zeros(X.shape[1])
    for _ in range(50):
        p = expit(X @ beta); w = p * (1 - p) + 1e-9
        b = beta + np.linalg.solve((X.T * w) @ X + 1e-4 * np.eye(X.shape[1]), X.T @ (y - p) - 1e-4 * beta)
        if np.abs(b - beta).max() < 1e-8:
            beta = b; break
        beta = b
    return beta


def gamma_start(X, y):
    beta = np.zeros(X.shape[1]); beta[0] = np.log(y.mean())
    for _ in range(50):
        mu = np.exp(X @ beta)
        b = np.linalg.solve(X.T @ X + 1e-8 * np.eye(X.shape[1]), X.T @ (X @ beta + (y - mu) / mu))
        if np.abs(b - beta).max() < 1e-9:
            beta = b; break
        beta = b
    return beta


def fit_model(d, y_col, fam, label):
    t0 = time.time()
    X, names = design(d, M5_NO_PROV, REFS)
    y = d[y_col].to_numpy(float)
    g = pd.factorize(d["province_model"])[0]
    m = Laplace(X, y, g, fam)
    start = {"poisson": poisson_start, "binomial": logit_start, "gamma": gamma_start}[fam](X, y)
    m.fit(start)
    theta = m.theta
    H = m.hessian(theta)
    for _ in range(2):                       # Newton polish: removes the last bit of gradient left by BFGS
        gr = optimize.approx_fprime(theta, m.negll, 1e-6)
        try:
            cand = theta - np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        if m.negll(cand) <= m.negll(theta) + 1e-9:
            theta = cand
            m.theta = theta
            H = m.hessian(theta)
    try:
        cov = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(H)
    se = np.sqrt(np.clip(np.diag(cov), 0, None))
    k = m.k
    z = stats.norm.ppf(0.975)
    flags = {}
    for j, nm in enumerate(names):
        if nm == "Intercept":
            continue
        n_l = int(X[:, j].sum())
        f = n_l < SPARSE_N_MIN
        if fam == "binomial":
            n1 = int((X[:, j] * y).sum())
            f = f or min(n1, n_l - n1) < SPARSE_CELL_MIN
        flags[nm] = f
    tab = pd.DataFrame({"outcome": label, "term": names[1:], "n": len(d),
                        "level_n": [int(X[:, j].sum()) for j in range(1, k)],
                        "ratio": np.exp(theta[1:k]), "lo": np.exp(theta[1:k] - z * se[1:k]),
                        "hi": np.exp(theta[1:k] + z * se[1:k]),
                        "p": 2 * stats.norm.sf(np.abs(theta[1:k] / np.where(se[1:k] > 0, se[1:k], np.nan))),
                        "sparse_flag": [flags[nm] for nm in names[1:]]})
    sig = float(np.exp(theta[k]))
    if fam == "binomial":
        icc = sig ** 2 / (sig ** 2 + np.pi ** 2 / 3)
    else:
        icc = np.nan
    vc = {"outcome": label, "family": fam, "n": len(d), "provinces": m.G, "sigma_u": sig,
          "sigma_u_lo": float(np.exp(theta[k] - z * se[k])), "sigma_u_hi": float(np.exp(theta[k] + z * se[k])),
          "ICC_latent_logit": icc, "loglik": -float(m.negll(theta)),
          "converged": bool(float(optimize.approx_fprime(theta, m.negll, 1e-6) @ np.linalg.pinv(H)
                                  @ optimize.approx_fprime(theta, m.negll, 1e-6)) < 1e-4)}   # Newton decrement
    if fam == "gamma":
        vc["gamma_dispersion"] = float(1 / np.exp(theta[k + 1]))
    print(f"  {label:<12} n = {len(d):,}, provinces = {m.G}, sigma_u = {sig:.3f}, "
          f"max|grad| = {np.abs(optimize.approx_fprime(theta, m.negll, 1e-6)).max():.1e}, {time.time() - t0:.0f}s")
    return tab, vc


# ----------------------------------------------------------------------------- data
if __name__ == "__main__":
    print("Objective 1, sensitivity 8: province random intercept (Laplace ML)")
    if not INPUT_CSV.exists():
        sys.exit(f"Input file not found: {INPUT_CSV}")
    raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
    D = raw[~raw["record_type"].astype(str).isin(NON_ADMISSION)].copy()
    D["any_surgical_service"] = np.where(D["any_surgical_service"].astype(str).str.strip().str.upper()
                                         .isin(["TRUE", "T", "1", "YES"]), "Yes", "No")
    D["adm_year"] = D["adm_year"].astype("Int64").astype(str)
    D["adm_month"] = D["adm_month"].astype("Int64").astype(str)
    for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
              "specialty_model", "billing_mode", "group"]:
        D[c] = D[c].fillna("Missing").astype(str)
    cnt = D["discharge_status"].value_counts()
    D["discharge_status_model"] = D["discharge_status"].where(~D["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
    for c in ["claim_lines", "los_days", "overnight", "los_capped_p99"]:
        D[c] = pd.to_numeric(D[c], errors="coerce")
    REFS = {v: D[v].value_counts().index[0] for v in M5_NO_PROV}
    REFS.update({"billing_mode": "Other", "any_surgical_service": "No", "adm_month": "1"})
    print(f"Primary sample (admissions): {len(D):,} episodes, {D.province_model.nunique()} provinces\n")

    tabs, vcs = [], []
    for label, ycol, fam, sub in [("claim_lines", "claim_lines", "poisson", D),
                                   ("overnight", "overnight", "binomial", D),
                                   ("los", "los_capped_p99", "gamma", D[(D.overnight == 1) & (D.los_capped_p99 > 0)])]:
        tab, vc = fit_model(sub, ycol, fam, label)
        tabs.append(tab); vcs.append(vc)
    LONG = pd.concat(tabs, ignore_index=True)
    LONG["analysis_status"] = "pre-specified sensitivity 8 (Laplace GLMM, province random intercept)"
    VC = pd.DataFrame(vcs)
    LONG.to_csv(OUT_DIR / "o1_sens8_random_intercept_long.csv", index=False, encoding="utf-8-sig")
    VC.to_csv(OUT_DIR / "o1_sens8_variance_components.csv", index=False, encoding="utf-8-sig")

    main_path = OUT_DIR / "o1_models_long.csv"
    keep = LONG[LONG.term.str.contains("billing_mode|group|any_surgical|ownership")].copy()
    if main_path.exists():
        ML = pd.read_csv(main_path)
        ML = ML[(ML.model == "M5") & ML.outcome.isin(["claim_lines", "overnight", "los"])][["outcome", "term", "ratio", "lo", "hi"]]
        keep = keep.merge(ML, on=["outcome", "term"], how="left", suffixes=("_random_intercept", "_fixed_province_M5"))
        keep["same_direction"] = (keep.ratio_random_intercept > 1) == (keep.ratio_fixed_province_M5 > 1)
        keep["pct_difference"] = 100 * (keep.ratio_random_intercept / keep.ratio_fixed_province_M5 - 1)
    keep.to_csv(OUT_DIR / "o1_sens8_main_terms.csv", index=False, encoding="utf-8-sig")
    pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
    print("\nVariance components:"); print(VC.round(3).to_string(index=False))
    print("\nMain exposures, random-intercept model vs fixed-province M5:")
    print(keep.round(3).to_string(index=False))
    print(f"\nDONE. Outputs in {OUT_DIR}")

Objective 1, sensitivity 8: province random intercept (Laplace ML)
Primary sample (admissions): 14,685 episodes, 28 provinces

  claim_lines  n = 14,685, provinces = 28, sigma_u = 0.161, max|grad| = 1.2e-02, 66s
  overnight    n = 14,685, provinces = 28, sigma_u = 0.864, max|grad| = 2.4e-04, 179s
  los          n = 10,660, provinces = 28, sigma_u = 0.120, max|grad| = 7.8e-03, 52s

Variance components:
    outcome   family     n  provinces  sigma_u  sigma_u_lo  sigma_u_hi  ICC_latent_logit     loglik  converged  gamma_dispersion
claim_lines  poisson 14685         28    0.161       0.123       0.211               NaN -28317.005       True               NaN
  overnight binomial 14685         28    0.864       0.655       1.138             0.185  -6278.670       True               NaN
        los    gamma 10660         28    0.120       0.090       0.160               NaN -11540.901       True             0.164

Main exposures, random-intercept model vs fixed-province M5:
    outcome      

In [16]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 2 (expenditure) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy
(matplotlib optional, for figures). Same engine style as objective1_resource_utilisation.py v3.

OUTCOME   Total episode cost in Int$ (cost_intl = IRR / 118,411.24, nominal; not inflation-adjusted,
          so Jalali year/month are in every model).
MODEL     Gamma-log GLM, blocks M1-M5 exactly as in Objective 1; LOS (capped at group P99, per day)
          is added in M3 together with surgical service and discharge status (protocol).
          Cost ratios = exp(coef) with cluster-robust 95% CI (patient_no clusters, as pre-specified;
          province clusters, t with G-1 df, reported next to them because facility-level exposures
          are correlated within province and the file has no hospital id).

PRIMARY SAMPLE  Same as Objective 1 v3: admissions (record types Global + Inpatient). Emergency
          Observation and Temporary admission are not admissions and distort the Global-vs-other
          contrast. The protocol sample (all valid episodes) is fitted and compared (X0).
          This is a deviation from the frozen protocol: state it in the Methods. (--sample all
          makes the protocol sample primary.)

SECTIONS
  2  Descriptives: median, IQR, mean of cost by group, code, billing mode, specialty, province
  3  Hypothesis 1: Global vs itemised cost variation within the same ICD-10 code (near-fixed tariff)
  4  Models M1-M5 (long table), main table (billing mode, group, specialty, 5 largest provinces)
  5  M5 stratified by billing mode (+ residual dispersion per stratum)
  6  Billing-mode effect by province, random-effects pooling (I2, prediction interval)
  7  Average marginal costs by group and billing mode (g-computation, delta-method CIs)
  8  Diagnostics: Park test, link test, Cook's distance (+ refit), GVIF, M5 vs M3, interactions
  9  Five-fold out-of-fold prediction error (MAE, calibration by decile)
 10  Concentration: top 10% share, Gini, Lorenz curves by group
 11  Hypothesis 2 (provinces) and Hypothesis 3 (K40: general surgery vs urology)
 12  Sensitivity analyses 1-8 (8 = province random intercept, Laplace ML)
 13  Exploratory (label as such): other sample, quasi-Poisson variance check, model without LOS,
     tiny cells removed, leave-one-province-out
 14  Reliability verdict (A / B / C) per main-table estimate

USAGE
  python objective2_expenditure.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 300]
Outputs: folder objective2_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import gammaln

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
SEED = 42
RARE_MIN = 30
VIF_FLAG = 5.0
TOP_PROVINCES = 5
SPARSE_N_MIN = 30
MIN_LEVEL_N = 100
SPREAD_FLAG = 1.5
PROV_MIN_PER_MODE = 30
LOPO_N = 8
COOK_MULT = 4.0                      # Cook's distance threshold = COOK_MULT / n
N_FOLDS = 5
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=300)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")

TIME = ["adm_year", "adm_month"]
BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode",
           "group", "any_surgical_service", "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
NUMS = {"M1": [], "M2": [], "M3": ["los_cov"], "M4": ["los_cov"], "M5": ["los_cov"]}   # LOS enters in M3
NO_BILLING = lambda b: [t for t in b if t != "billing_mode"]
REPORT_VARS = ["ownership", "billing_mode", "group", "any_surgical_service", "specialty_model",
               "province_model", "los_cov"]
CORE_VARS = ["age_group", "billing_mode", "group", "any_surgical_service", "los_cov"]
Z975 = stats.norm.ppf(0.975)


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE THE PRIMARY SAMPLE
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "cost_intl", "total_cost_irr", "los_days", "los_capped_p99",
        "los_capped_p95", "cluster_id", "any_surgical_service", "specialty_model", "specialty_h3",
        "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O2",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O2", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "specialty_h3", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["cost_intl", "total_cost_irr", "los_days", "los_capped_p99", "los_capped_p95"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
if (DALL["cost_intl"] <= 0).any() or DALL["cost_intl"].isna().any():
    fail("cost_intl has missing or non-positive values; the frozen file should not.")
DALL["los_cov"] = DALL["los_capped_p99"]                    # LOS covariate (P99 cap); sensitivity 7 swaps it

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
valid = DALL["in_O2"] & DALL["los_cov"].notna()
flow.append({"step": "O2 complete cases (cost and LOS present), all record types", "n": int(valid.sum())})
D = (DALL[is_adm & valid] if args.sample == "admissions" else DALL[valid]).copy()
ALT = (DALL[valid] if args.sample == "admissions" else DALL[is_adm & valid]).copy()
SAMPLE_LABEL = "admissions (Global + Inpatient)" if args.sample == "admissions" else "all valid episodes (protocol sample)"
ALT_LABEL = "all valid episodes (protocol sample)" if args.sample == "admissions" else "admissions only"
flow.append({"step": f"PRIMARY SAMPLE O2: {SAMPLE_LABEL}", "n": len(D)})
save(pd.DataFrame(flow), "o2_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
print(f"\nCost (Int$): median {D.cost_intl.median():,.0f}, IQR {D.cost_intl.quantile(.25):,.0f}-"
      f"{D.cost_intl.quantile(.75):,.0f}, mean {D.cost_intl.mean():,.0f}, max {D.cost_intl.max():,.0f}")

CAT_VARS = sorted({v for b in BLOCKS.values() for v in b} | set(TIME))
REFS = {v: D[v].value_counts().index[0] for v in CAT_VARS}
REFS.update({"billing_mode": "Other", "any_surgical_service": "No", "adm_month": "1",
             "specialty_h3": "General surgery"})
print("Reference levels:", {k: REFS[k] for k in BLOCKS["M5"]})


# =============================================================================
# 2. MODEL ENGINE (numpy / scipy): Gamma-log and quasi-Poisson-log, cluster-robust
# =============================================================================
def build_spec(d, cats, nums, inter=(), refs=None):
    refs = refs or REFS
    cols, names, levs = [np.ones(len(d))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        for lev in levs[v][1:]:
            cols.append((s == lev).to_numpy(float))
            names.append(f"C({v})[T.{lev}]")
    for v in nums:
        cols.append(d[v].to_numpy(float))
        names.append(v)
    for a, b in inter:
        for la in levs[a][1:]:
            for lb in levs[b][1:]:
                cols.append(((d[a].astype(str) == la) & (d[b].astype(str) == lb)).to_numpy(float))
                names.append(f"C({a})[T.{la}]:C({b})[T.{lb}]")
    X = np.column_stack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    spec = {"cats": list(cats), "nums": list(nums), "inter": list(inter), "levs": levs, "keep": keep}
    return X[:, keep], [names[i] for i in keep], spec


def apply_spec(d, spec):
    cols = [np.ones(len(d))]
    for v in spec["cats"]:
        s = d[v].astype(str)
        for lev in spec["levs"][v][1:]:
            cols.append((s == lev).to_numpy(float))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float))
    for a, b in spec["inter"]:
        for la in spec["levs"][a][1:]:
            for lb in spec["levs"][b][1:]:
                cols.append(((d[a].astype(str) == la) & (d[b].astype(str) == lb)).to_numpy(float))
    return np.column_stack(cols)[:, spec["keep"]]


def fit_glm(X, y, fam="gamma"):
    """Log-link GLM by IRLS. Gamma: V = mu^2 (working weight 1). quasi-Poisson: V = mu."""
    n, k = X.shape
    beta = np.zeros(k)
    beta[0] = np.log(y.mean())
    eta = X @ beta
    mu = np.exp(eta)
    conv = False
    for _ in range(200):
        w = np.ones(n) if fam == "gamma" else mu
        z = eta + (y - mu) / mu
        A, rhs = (X.T * w) @ X, (X.T * w) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        eta = np.clip(X @ b_new, -30, 30)
        mu = np.exp(eta)
        done = np.abs(b_new - beta).max() < 1e-9
        beta = b_new
        if done:
            conv = True
            break
    w = np.ones(n) if fam == "gamma" else mu
    V = mu ** 2 if fam == "gamma" else mu
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * w) @ X), "conv": conv,
            "phi": float(np.sum((y - mu) ** 2 / V) / max(n - k, 1)), "V": V}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups, fam="gamma"):
    n, k = X.shape
    s = (y - f["mu"]) / f["mu"] if fam == "gamma" else (y - f["mu"])
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_core(d, cats, nums, fam="gamma", inter=(), refs=None):
    cats = list(dict.fromkeys(list(cats) + TIME))
    X, names, spec = build_spec(d, cats, nums, inter, refs)
    y = d["cost_intl"].to_numpy(float)
    f = fit_glm(X, y, fam)
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl], fam) for cl in groups}
    lev_n = {nm: (int(X[:, j].sum()) if nm.startswith("C(") else np.nan) for j, nm in enumerate(names)}
    return {"n": len(d), "fam": fam, "names": names, "beta": f["beta"], "covs": covs, "X": X, "y": y,
            "mu": f["mu"], "eta": f["eta"], "phi": f["phi"], "conv": f["conv"], "spec": spec,
            "lev_n": lev_n, "groups": groups, "sub_index": d.index, "bread": f["bread"], "V": f["V"]}


def make_tab(core, label, model):
    out = []
    names, beta = core["names"], core["beta"]
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = beta / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term": names, "n": core["n"],
                          "level_n": [core["lev_n"][nm] for nm in names], "ratio": np.exp(beta),
                          "lo": np.exp(beta - crit * se), "hi": np.exp(beta + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[t["term"] != "Intercept"])
    return pd.concat(out, ignore_index=True)


def fit_one(d, model, cats, nums, label="cost", fam="gamma", quiet=False, inter=(), refs=None):
    try:
        core = fit_core(d, cats, nums, fam, inter, refs)
        tab = make_tab(core, label, model)
        if not quiet:
            print(f"  {label:<10} {model}: n = {core['n']:,}, terms = {len(core['names']) - 1}, "
                  f"dispersion = {core['phi']:.2f}" + ("" if core["conv"] else "  ** did not converge **"))
        return tab, core
    except Exception as e:
        print(f"  {label:<10} {model}: FAILED ({type(e).__name__}: {e})")
        return None, None


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


def hit(t):  # CI excludes 1
    return bool((t.lo > 1) or (t.hi < 1))


# =============================================================================
# 3. DESCRIPTIVES
# =============================================================================
hdr("2. DESCRIPTIVES (cost, Int$)")


def cstats(s):
    s = s.dropna()
    m = s.mean()
    return {"n": len(s), "mean": m, "sd": s.std(), "median": s.median(), "p25": s.quantile(.25),
            "p75": s.quantile(.75), "p5": s.quantile(.05), "p95": s.quantile(.95), "p99": s.quantile(.99),
            "max": s.max(), "cv": s.std() / m if m > 0 else np.nan,
            "median_IQR": f"{s.median():,.0f} ({s.quantile(.25):,.0f}-{s.quantile(.75):,.0f})"}


def describe(d, var, label):
    rows = []
    for lev, g in ([("All", d)] if var is None else d.groupby(var, observed=True)):
        rows.append({"variable": label, "level": lev, **cstats(g["cost_intl"])})
    return rows


desc = describe(DALL[valid], None, "All valid episodes (protocol sample)")
desc += describe(DALL[valid], "record_type", "Record type (all valid episodes)")
desc += describe(D, None, "PRIMARY SAMPLE")
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("billing_mode", "Billing mode"),
                 ("specialty_model", "Specialty"), ("province_model", "Province"), ("ownership", "Ownership")]:
    desc += describe(D, var, lab)
DESC = pd.DataFrame(desc)
save(DESC, "o2_descriptives.csv")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
print(DESC[DESC.variable.isin(["All valid episodes (protocol sample)", "Record type (all valid episodes)",
                               "PRIMARY SAMPLE", "Group", "Billing mode"])]
      [["variable", "level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
by_code = DESC[DESC.variable == "ICD-10 code"][["level", "n", "median_IQR", "mean", "cv"]]
print("\nBy ICD-10 code:\n", by_code.round(2).to_string(index=False))

# =============================================================================
# 4. HYPOTHESIS 1: Global vs itemised cost variation within the same code
# =============================================================================
hdr("3. HYPOTHESIS 1: cost variation, Global vs itemised, same ICD-10 code")


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i])
        adj[i] = min(run, 1.0)
    return adj


def tariff_stats(x):
    med = x.median()
    return {"n": len(x), "median": med, "mean": x.mean(), "sd": x.std(), "cv": x.std() / x.mean(),
            "sd_log": np.log(x).std(), "qcd": (x.quantile(.75) - x.quantile(.25)) / (x.quantile(.75) + x.quantile(.25)),
            "pct_within_10pct_of_median": 100 * ((x - med).abs() <= 0.10 * med).mean(),
            "pct_within_25pct_of_median": 100 * ((x - med).abs() <= 0.25 * med).mean()}


H1_cells, H1_cmp = [], []
if D.billing_mode.nunique() == 2:
    for code, g in D.groupby("icd10"):
        gg = g.loc[g.billing_mode == "Global", "cost_intl"]
        oo = g.loc[g.billing_mode != "Global", "cost_intl"]
        for lab, x in [("Global", gg), ("Other", oo)]:
            if len(x) >= 5:
                H1_cells.append({"icd10": code, "billing_mode": lab, **tariff_stats(x)})
        if len(gg) < 10 or len(oo) < 10:
            H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo), "note": "too few in a mode"})
            continue
        lg, lo_ = np.log(gg.to_numpy()), np.log(oo.to_numpy())
        bf = stats.levene(lg, lo_, center="median")
        sr, cr = lg.std() / lo_.std(), (gg.std() / gg.mean()) / (oo.std() / oo.mean())
        bs, bc = [], []
        for _ in range(NBOOT):
            a, b = RNG.choice(gg.to_numpy(), len(gg)), RNG.choice(oo.to_numpy(), len(oo))
            bs.append(np.log(a).std() / np.log(b).std())
            bc.append((a.std() / a.mean()) / (b.std() / b.mean()))
        H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo), "sd_log_ratio_G_over_O": sr,
                       "sd_log_lo": np.percentile(bs, 2.5), "sd_log_hi": np.percentile(bs, 97.5),
                       "cv_ratio_G_over_O": cr, "cv_lo": np.percentile(bc, 2.5), "cv_hi": np.percentile(bc, 97.5),
                       "median_ratio_G_over_O": gg.median() / oo.median(),
                       "brown_forsythe_p": float(bf.pvalue), "note": ""})
    H1c = pd.DataFrame(H1_cmp)
    if "brown_forsythe_p" in H1c:
        ok = H1c["brown_forsythe_p"].notna()
        H1c.loc[ok, "p_holm"] = holm(H1c.loc[ok, "brown_forsythe_p"])
        H1c["Global_less_variable"] = (H1c["sd_log_ratio_G_over_O"] < 1) & (H1c["p_holm"] < 0.05)
    # pooled: deviation of log cost from the median of its code x billing-mode cell
    lc = np.log(D["cost_intl"])
    dev = (lc - lc.groupby([D["icd10"], D["billing_mode"]]).transform("median")).abs()
    isg = D["billing_mode"] == "Global"
    mw = stats.mannwhitneyu(dev[isg], dev[~isg], alternative="less")
    pooled = {"median_absdev_log_Global": float(dev[isg].median()), "median_absdev_log_Other": float(dev[~isg].median()),
              "mannwhitney_p_Global_less": float(mw.pvalue)}
    save(pd.DataFrame(H1_cells), "o2_h1_cost_by_code_and_billing.csv")
    save(H1c, "o2_h1_variation_comparison.csv")
    save(pd.DataFrame([pooled]), "o2_h1_pooled.csv")
    print(pd.DataFrame(H1_cells).round(2).to_string(index=False))
    print("\nGlobal vs Other, ratio of SD(log cost) and CV (ratio < 1: Global less variable):")
    print(H1c.round(3).to_string(index=False))
    print("\nPooled within-cell |log deviation|:", {k: round(v, 4) for k, v in pooled.items()})
else:
    print("Only one billing mode in the primary sample; hypothesis 1 skipped.")

# =============================================================================
# 5. MODELS M1-M5 (Gamma-log; patient and province clusters)
# =============================================================================
hdr(f"4. MODELS M1-M5, primary sample: {SAMPLE_LABEL}, n = {len(D):,}")
tabs, CORE, disp_rows = [], {}, []
for model, cats in BLOCKS.items():
    tab, core = fit_one(D, model, cats, NUMS[model])
    if tab is not None:
        tabs.append(tab)
        CORE[model] = core
        disp_rows.append({"model": model, "n": core["n"], "dispersion": core["phi"], "converged": core["conv"]})
ALLT = pd.concat(tabs, ignore_index=True)
ALLT["analysis_status"] = "pre-specified models; primary sample redefined after Objective 1 diagnostics"
LONG = ALLT[ALLT.cluster == "patient"].copy()
PC = ALLT[(ALLT.cluster == "province") & (ALLT.model == "M5") & ~ALLT.term.str.contains("province_model")].copy()
save(LONG, "o2_models_long.csv")
save(PC, "o2_models_province_cluster.csv")
save(pd.DataFrame(disp_rows), "o2_diagnostics_dispersion.csv")
print(pd.DataFrame(disp_rows).round(3).to_string(index=False))

vc = D["province_model"].value_counts()
TOP_PROV = [p for p in vc.index if p != "Other"][:TOP_PROVINCES]
LEVELS = {v: [REFS[v]] + [x for x in D[v].value_counts().index if x != REFS[v]] for v in REFS if v in D.columns}


def fmt(r, mark=True):
    s = f"{r.ratio:.2f} ({r.lo:.2f}-{r.hi:.2f})"
    s += "" if pd.isna(r.p) else (f"; p={r.p:.3f}" if r.p >= 0.001 else "; p<0.001")
    if bool(r.sparse_flag):
        s += " \u2020"
    elif mark and pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N:
        s += " \u2021"
    return s


def lookup(tabs_, model, term, cluster="patient", outcome="cost"):
    m = tabs_[(tabs_.outcome == outcome) & (tabs_.model == model) & (tabs_.term == term) & (tabs_.cluster == cluster)]
    return m.iloc[0] if len(m) else None


rows = []
for var, only in [("billing_mode", None), ("group", None), ("ownership", None), ("specialty_model", None),
                  ("province_model", TOP_PROV)]:
    levels = LEVELS[var] if only is None else [REFS[var]] + [x for x in only if x != REFS[var]]
    for lev in levels:
        row = {"variable": var, "level": lev, "episodes": int((D[var] == lev).sum())}
        if lev == REFS[var]:
            row["cost ratio (patient-clustered CI)"] = "1 (ref)"
            row["cost ratio (province-clustered CI)"] = "1 (ref)"
        else:
            a = lookup(ALLT, "M5", f"C({var})[T.{lev}]", "patient")
            b = lookup(ALLT, "M5", f"C({var})[T.{lev}]", "province")
            row["cost ratio (patient-clustered CI)"] = fmt(a) if a is not None else ""
            row["cost ratio (province-clustered CI)"] = (
                "not defined (absorbed)" if var == "province_model"
                else (f"{b.ratio:.2f} ({b.lo:.2f}-{b.hi:.2f}); p={b.p:.3f}" if b is not None else ""))
        rows.append(row)
for var in ["los_cov", "any_surgical_service"]:
    terms = ["los_cov"] if var == "los_cov" else [f"C({var})[T.Yes]"]
    for term in terms:
        a = lookup(ALLT, "M5", term, "patient")
        b = lookup(ALLT, "M5", term, "province")
        if a is not None:
            rows.append({"variable": var + (" (per day)" if var == "los_cov" else ""), "level": "" if var == "los_cov" else "Yes",
                         "episodes": int(len(D)) if var == "los_cov" else int((D[var] == "Yes").sum()),
                         "cost ratio (patient-clustered CI)": fmt(a),
                         "cost ratio (province-clustered CI)": f"{b.ratio:.2f} ({b.lo:.2f}-{b.hi:.2f}); p={b.p:.3f}" if b is not None else ""})
MAIN = pd.DataFrame(rows)
save(MAIN, "o2_main_table.csv")
hdr("MAIN TABLE, model M5: cost ratios (Gamma-log)")
print(MAIN.to_string(index=False))
print("\u2020 sparse cell (< 30 episodes); \u2021 fewer than 100 episodes: not interpretable on its own.")
print("Billing mode is a pooled average across provinces whose effects may differ; use section 6.")

# =============================================================================
# 6. M5 STRATIFIED BY BILLING MODE
# =============================================================================
hdr("5. M5 STRATIFIED BY BILLING MODE (Global only | Other only)")
ST_tabs, ST_disp = [], []
if D.billing_mode.nunique() == 2:
    for lab, mask in [("Global", D.billing_mode == "Global"), ("Other", D.billing_mode != "Global")]:
        tab, core = fit_one(D[mask], f"M5 | {lab}", NO_BILLING(BLOCKS["M5"]), NUMS["M5"])
        if tab is not None:
            tab["stratum"] = lab
            ST_tabs.append(tab)
            ST_disp.append({"stratum": lab, "n": core["n"], "dispersion_gamma": core["phi"]})
ST = pd.concat(ST_tabs, ignore_index=True) if ST_tabs else pd.DataFrame(
    columns=["outcome", "term", "stratum", "ratio", "cluster"])
if len(ST):
    ST["analysis_status"] = "exploratory (added after Objective 1 diagnostics)"
    save(ST, "o2_models_stratified.csv")
    save(pd.DataFrame(ST_disp), "o2_stratified_dispersion.csv")
    print("Residual dispersion (smaller = costs closer to the model prediction):")
    print(pd.DataFrame(ST_disp).round(3).to_string(index=False))
    srows = []
    for var, only in [("group", None), ("any_surgical_service", None), ("ownership", None), ("specialty_model", None),
                      ("province_model", TOP_PROV)]:
        levels = LEVELS[var] if only is None else [REFS[var]] + [x for x in only if x != REFS[var]]
        for lev in levels:
            row = {"variable": var, "level": lev}
            for stratum in ["Global", "Other"]:
                sub = D[D.billing_mode == "Global"] if stratum == "Global" else D[D.billing_mode != "Global"]
                row[f"episodes_{stratum}"] = int((sub[var] == lev).sum())
                if lev == REFS[var]:
                    row[f"cost ratio | {stratum}"] = "1 (ref)"
                else:
                    m = ST[(ST.stratum == stratum) & (ST.term == f"C({var})[T.{lev}]") & (ST.cluster == "patient")]
                    row[f"cost ratio | {stratum}"] = fmt(m.iloc[0]) if len(m) else "not estimable"
            srows.append(row)
    STM = pd.DataFrame(srows)
    save(STM, "o2_main_table_stratified.csv")
    print(STM[STM.variable.isin(["group", "any_surgical_service"])].to_string(index=False))

# =============================================================================
# 7. BILLING MODE BY PROVINCE, random-effects pooling
# =============================================================================
hdr("6. BILLING-MODE COST RATIO BY PROVINCE (Global vs itemised), random-effects pooling")


def drsl(b, se):
    w = 1 / se ** 2
    fe = np.sum(w * b) / w.sum()
    Q = np.sum(w * (b - fe) ** 2)
    k = len(b)
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - np.sum(w ** 2) / w.sum()))
    ws = 1 / (se ** 2 + tau2)
    re, se_re = np.sum(ws * b) / ws.sum(), np.sqrt(1 / ws.sum())
    t = stats.t.ppf(.975, max(k - 2, 1))
    ph = t * np.sqrt(tau2 + se_re ** 2)
    return {"k": k, "fixed": fe, "re": re, "se_re": se_re, "tau": np.sqrt(tau2),
            "I2_pct": 100 * max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0, "Q_p": stats.chi2.sf(Q, k - 1),
            "pi_lo": re - ph, "pi_hi": re + ph}


BPP = pd.DataFrame()
BP = pd.DataFrame()
if args.sample == "admissions" and D.billing_mode.nunique() == 2:
    ct = pd.crosstab(D.province_model, D.billing_mode)
    PROV = [p for p in ct.index if ct.loc[p].min() >= PROV_MIN_PER_MODE and p != "Other"]
    BPD = D[D.province_model.isin(PROV)]
    ref_prov = BPD.province_model.value_counts().index[0]
    refs2 = {**REFS, "province_model": ref_prov}
    print(f"  {len(PROV)} provinces with >= {PROV_MIN_PER_MODE} Global and >= {PROV_MIN_PER_MODE} itemised "
          f"episodes ({len(BPD):,} episodes); reference province: {ref_prov}")
    core = fit_core(BPD, BLOCKS["M5"], NUMS["M5"], inter=[("billing_mode", "province_model")], refs=refs2)
    names, beta, cov = core["names"], core["beta"], core["covs"]["patient"][0]
    jb = names.index("C(billing_mode)[T.Global]")
    est, sel, used, rws = [], [], [], []
    for p in PROV:
        L = np.zeros(len(names))
        L[jb] = 1
        nm = f"C(billing_mode)[T.Global]:C(province_model)[T.{p}]"
        if nm in names:
            L[names.index(nm)] = 1
        elif p != ref_prov:
            continue
        e, s = float(L @ beta), float(np.sqrt(L @ cov @ L))
        g_ = BPD[BPD.province_model == p]
        rws.append({"province": p, "n_Global": int((g_.billing_mode == "Global").sum()),
                    "n_Other": int((g_.billing_mode != "Global").sum()),
                    "median_cost_Global": g_.loc[g_.billing_mode == "Global", "cost_intl"].median(),
                    "median_cost_Other": g_.loc[g_.billing_mode != "Global", "cost_intl"].median(),
                    "ratio": np.exp(e), "lo": np.exp(e - Z975 * s), "hi": np.exp(e + Z975 * s)})
        est.append(e)
        sel.append(s)
    est, sel = np.array(est), np.array(sel)
    r = drsl(est, sel)
    BPP = pd.DataFrame([{"outcome": "cost", "provinces": r["k"], "pooled_ratio_RE": np.exp(r["re"]),
                         "pooled_lo": np.exp(r["re"] - Z975 * r["se_re"]), "pooled_hi": np.exp(r["re"] + Z975 * r["se_re"]),
                         "pooled_ratio_fixed": np.exp(r["fixed"]), "I2_pct": r["I2_pct"], "tau_log": r["tau"], "Q_p": r["Q_p"],
                         "pred_interval_lo": np.exp(r["pi_lo"]), "pred_interval_hi": np.exp(r["pi_hi"]),
                         "provinces_ratio_below_1": int((est < 0).sum()), "provinces_ratio_above_1": int((est > 0).sum()),
                         "min_province_ratio": np.exp(est.min()), "max_province_ratio": np.exp(est.max()),
                         "same_direction_all_provinces": bool((est < 0).all() or (est > 0).all())}])
    BP = pd.DataFrame(rws)
    BPP["analysis_status"] = BP["analysis_status"] = "exploratory (added after Objective 1 diagnostics)"
    save(BP, "o2_billing_by_province.csv")
    save(BPP, "o2_billing_pooled.csv")
    b = BPP.iloc[0]
    print(f"  cost: RE ratio {b.pooled_ratio_RE:.2f} ({b.pooled_lo:.2f}-{b.pooled_hi:.2f}); I2 = {b.I2_pct:.0f}%; "
          f"95% PI {b.pred_interval_lo:.2f}-{b.pred_interval_hi:.2f}; provinces <1 / >1: "
          f"{b.provinces_ratio_below_1}/{b.provinces_ratio_above_1}")
else:
    print("  skipped (needs the admissions-only sample with both billing modes).")

# =============================================================================
# 8. AVERAGE MARGINAL COSTS (g-computation, delta method)
# =============================================================================
hdr("7. AVERAGE MARGINAL COSTS (Int$) BY GROUP AND BILLING MODE")


def predict_margin(core, d, over):
    d2 = d.copy()
    for k_, v_ in over.items():
        if v_ not in core["spec"]["levs"].get(k_, []):
            return None
        d2[k_] = v_
    X2 = apply_spec(d2, core["spec"])
    mu = np.exp(np.clip(X2 @ core["beta"], -30, 30))
    return mu.mean(), (X2 * mu[:, None]).mean(0)


def amc_table(core, d, model, cluster):
    cov, G = core["covs"][cluster]
    crit = stats.t.ppf(0.975, max(G - 1, 1)) if cluster == "province" else Z975
    groups = [g for g in core["spec"]["levs"].get("group", [])]
    modes = core["spec"]["levs"].get("billing_mode", [])
    settings = [("overall", {}, "all", "observed")]
    settings += [("group", {"group": g}, g, "observed") for g in groups]
    settings += [("billing mode", {"billing_mode": m}, "all", m) for m in modes]
    settings += [("group x billing mode", {"group": g, "billing_mode": m}, g, m) for g in groups for m in modes]
    cache, rows = {}, []
    for kind, over, g, m in settings:
        res = predict_margin(core, d, over)
        if res is None:
            continue
        cache[(g, m)] = res
        mean, grad = res
        se = float(np.sqrt(grad @ cov @ grad))
        obs = d
        if g != "all":
            obs = obs[obs["group"] == g]
        if m not in ("observed", "all"):
            obs = obs[(obs["billing_mode"] == m)]
        rows.append({"model": model, "cluster": cluster, "kind": kind, "group": g, "billing_mode": m,
                     "amc_intl": mean, "lo": mean * np.exp(-crit * se / mean), "hi": mean * np.exp(crit * se / mean),
                     "observed_mean_in_cell": obs["cost_intl"].mean() if len(obs) else np.nan, "n_observed": len(obs)})
    # Global minus Other within group and overall
    diffs = []
    for g in groups + ["all"]:
        a, b_ = cache.get((g, "Global")), cache.get((g, "Other"))
        if a is None or b_ is None:
            continue
        dm, dg = a[0] - b_[0], a[1] - b_[1]
        se = float(np.sqrt(dg @ cov @ dg))
        lr = np.log(a[0] / b_[0])
        gr = a[1] / a[0] - b_[1] / b_[0]
        sl = float(np.sqrt(gr @ cov @ gr))
        diffs.append({"model": model, "cluster": cluster, "group": g, "diff_Global_minus_Other_intl": dm,
                      "diff_lo": dm - crit * se, "diff_hi": dm + crit * se, "ratio_Global_over_Other": np.exp(lr),
                      "ratio_lo": np.exp(lr - crit * sl), "ratio_hi": np.exp(lr + crit * sl)})
    return pd.DataFrame(rows), pd.DataFrame(diffs)


AM, AD = [], []
for model, cl in [("M5", "patient"), ("M3", "province"), ("M3", "patient")]:
    if model in CORE:
        a, b = amc_table(CORE[model], D, model, cl)
        AM.append(a)
        AD.append(b)
AMC = pd.concat(AM, ignore_index=True) if AM else pd.DataFrame()
AMD = pd.concat(AD, ignore_index=True) if AD else pd.DataFrame()
save(AMC, "o2_average_marginal_costs.csv")
save(AMD, "o2_marginal_cost_global_vs_other.csv")
if len(AMC):
    print("M5 (patient-clustered CI). Province-clustered CIs (from M3, which has no province terms) are in the CSV.")
    print(AMC[(AMC.model == "M5") & (AMC.kind.isin(["overall", "group", "billing mode", "group x billing mode"]))]
          .round(1).drop(columns=["model", "cluster"]).to_string(index=False))
    print("\nGlobal minus Other, same covariates (M5, patient clusters):")
    print(AMD[(AMD.model == "M5")].round(2).drop(columns=["model", "cluster"]).to_string(index=False))
    print("Note: marginal costs hold LOS and surgical service at observed values; LOS is partly a "
          "mechanism of billing-mode differences (see exploratory X9, model without LOS).")

# =============================================================================
# 9. DIAGNOSTICS
# =============================================================================
hdr("8. DIAGNOSTICS")


def ols_cluster(x, y, groups):
    X = np.column_stack([np.ones(len(x)), x])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ b
    bread = np.linalg.inv(X.T @ X)
    meat, g = cluster_meat(X * res[:, None], groups)
    n, k = X.shape
    cov = (g / (g - 1)) * ((n - 1) / (n - k)) * bread @ meat @ bread
    return b, np.sqrt(np.diag(cov)), g


park_rows, link_rows = [], []
for model, core in CORE.items():
    # Park test: regress ln(e^2) on ln(mu); slope ~ 2 -> Gamma, ~ 1 -> Poisson-like, ~ 3 -> inverse Gaussian
    e2 = (core["y"] - core["mu"]) ** 2
    m = e2 > 0
    b, se, g = ols_cluster(np.log(core["mu"][m]), np.log(e2[m]), core["groups"]["patient"][m])
    nearest = min([(0, "Gaussian"), (1, "Poisson-like"), (2, "Gamma"), (3, "inverse Gaussian")], key=lambda t: abs(t[0] - b[1]))[1]
    park_rows.append({"model": model, "n": int(m.sum()), "slope": b[1], "se": se[1], "lo": b[1] - Z975 * se[1],
                      "hi": b[1] + Z975 * se[1], "CI_includes_2": bool(b[1] - Z975 * se[1] <= 2 <= b[1] + Z975 * se[1]),
                      "nearest_variance_function": nearest})
    # Link test (Pregibon): add squared linear predictor; a significant coefficient rejects the log link
    eta = core["eta"]
    sq = ((eta - eta.mean()) / eta.std()) ** 2
    X2 = np.column_stack([core["X"], sq])
    f2 = fit_glm(X2, core["y"], "gamma")
    cv2, _ = cov_robust(X2, core["y"], f2, core["groups"]["patient"], "gamma")
    bb, ss = f2["beta"][-1], float(np.sqrt(cv2[-1, -1]))
    link_rows.append({"model": model, "coef_eta_sq": bb, "se": ss, "z": bb / ss, "p": 2 * stats.norm.sf(abs(bb / ss))})
PARK, LINK = pd.DataFrame(park_rows), pd.DataFrame(link_rows)
save(PARK, "o2_diagnostics_park.csv")
save(LINK, "o2_diagnostics_link.csv")
print("Park test (Gamma variance function: slope near 2):\n", PARK.round(3).to_string(index=False))
print("\nLink test (squared linear predictor; p < 0.05 rejects the log link):\n", LINK.round(4).to_string(index=False))

# Cook's distance on M5
c5 = CORE["M5"]
Xm = c5["X"]
hat = np.clip(np.einsum("ij,jk,ik->i", Xm, np.linalg.pinv(Xm.T @ Xm), Xm), 0, 0.999)
rP = (c5["y"] - c5["mu"]) / c5["mu"]
kk = Xm.shape[1]
cook = (rP ** 2 * hat) / (kk * c5["phi"] * (1 - hat) ** 2)
thr = COOK_MULT / len(cook)
flag_cook = cook > thr
COOKDF = D.loc[c5["sub_index"], ["episode_no", "icd10", "group", "billing_mode", "province_model", "cost_intl",
                                  "los_days"]].assign(cooks_d=cook, leverage=hat)
save(COOKDF.sort_values("cooks_d", ascending=False).head(100), "o2_diagnostics_cook_top100.csv")
print(f"\nCook's distance (M5): threshold 4/n = {thr:.2e}; {int(flag_cook.sum()):,} episodes above "
      f"({100 * flag_cook.mean():.1f}%); max {cook.max():.3f}")
print("Highest-influence episodes (cost Int$):")
print(COOKDF.sort_values("cooks_d", ascending=False).head(8).round(3).to_string(index=False))
tab_nc, core_nc = fit_one(D[~flag_cook], "M5 (no Cook > 4/n)", BLOCKS["M5"], NUMS["M5"], label="cost")
if tab_nc is not None:
    a = LONG[LONG.model == "M5"][["term", "ratio", "lo", "hi"]]
    b = tab_nc[tab_nc.cluster == "patient"][["term", "ratio", "lo", "hi"]]
    CK = a.merge(b, on="term", suffixes=("_full", "_no_influential"))
    CK["pct_change"] = 100 * (CK.ratio_no_influential / CK.ratio_full - 1)
    CK["same_direction"] = (CK.ratio_full > 1) == (CK.ratio_no_influential > 1)
    CK["n_full"], CK["n_refit"] = len(D), int((~flag_cook).sum())
    save(CK, "o2_diagnostics_cook_refit.csv")
    mainc = CK[CK.term.str.contains("billing|group|surgical|ownership|los_cov")]
    print(f"Refit without them: {int((CK.same_direction).sum())}/{len(CK)} terms keep their direction; "
          f"main exposures, largest changes:")
    print(mainc.reindex(mainc["pct_change"].abs().sort_values(ascending=False).index).head(10).round(3).to_string(index=False))


def gvif_table(X, names):
    keep = [i for i, nm in enumerate(names) if nm != "Intercept" and X[:, i].std() > 0]
    term_of = {i: names[i].split("[T.")[0] for i in keep}
    R = np.corrcoef(X[:, keep], rowvar=False)
    sign, ld_all = np.linalg.slogdet(R)
    rows_ = []
    for term in dict.fromkeys(term_of.values()):
        idx = [j for j, i in enumerate(keep) if term_of[i] == term]
        rest = [j for j in range(len(keep)) if j not in set(idx)]
        s1, ld_t = np.linalg.slogdet(R[np.ix_(idx, idx)])
        s2, ld_r = np.linalg.slogdet(R[np.ix_(rest, rest)])
        g = np.exp(ld_t + ld_r - ld_all) if sign > 0 and s1 > 0 and s2 > 0 else np.nan
        rows_.append({"term": term, "df": len(idx), "GVIF": g, "VIF_equivalent": g ** (1 / (2 * len(idx))) if np.isfinite(g) else np.nan})
    out = pd.DataFrame(rows_)
    out["flag_gt5"] = out["VIF_equivalent"] > VIF_FLAG
    return out


GV = gvif_table(c5["X"], c5["names"])
save(GV, "o2_diagnostics_gvif.csv")
print("\nGVIF (M5), VIF_equivalent = GVIF^(1/(2df)); flag > 5 ->",
      "none flagged" if not GV.flag_gt5.any() else GV[GV.flag_gt5].term.tolist())
print(GV.round(2).to_string(index=False))

shared = [t for t in LONG.term.unique() if "province_model" not in t and "specialty_model" not in t]
m3 = LONG[(LONG.model == "M3") & LONG.term.isin(shared)].set_index("term")["ratio"]
m5 = LONG[(LONG.model == "M5") & LONG.term.isin(shared)].set_index("term")["ratio"]
CH = pd.concat([m3.rename("ratio_M3"), m5.rename("ratio_M5")], axis=1).dropna().reset_index()
CH["pct_change_M5_vs_M3"] = 100 * (CH.ratio_M5 / CH.ratio_M3 - 1)
save(CH, "o2_diagnostics_m5_vs_m3.csv")
big = CH[(CH.pct_change_M5_vs_M3.abs() > 10) & CH.term.str.contains("ownership|billing|group|surgical|los_cov")]
print(f"\nM5 vs M3: {int((CH.pct_change_M5_vs_M3.abs() > 10).sum())} of {len(CH)} estimates move > 10%; "
      f"main exposures: {len(big)}")
if len(big):
    print(big.round(3).to_string(index=False))

# interaction (heterogeneity) tests
INT_ROWS = []
pairs = [("billing_mode", "group"), ("billing_mode", "any_surgical_service"), ("group", "any_surgical_service"),
         ("billing_mode", "province_model")]
for a_, b_ in pairs:
    if a_ == "billing_mode" and D.billing_mode.nunique() < 2:
        continue
    try:
        core = fit_core(D, BLOCKS["M5"], NUMS["M5"], inter=[(a_, b_)])
        idx = [i for i, nm in enumerate(core["names"]) if ":" in nm and f"C({a_})" in nm and f"C({b_})" in nm]
        chi2, df_, p = wald(core, idx)
        INT_ROWS.append({"interaction": f"{a_} x {b_}", "chi2": chi2, "df": df_, "p_value": p})
    except Exception as e:
        print(f"  interaction {a_} x {b_} FAILED: {type(e).__name__}: {e}")
INT = pd.DataFrame(INT_ROWS)
if len(INT):
    INT["significant_bonferroni"] = INT.p_value < 0.05 / len(INT)
save(INT, "o2_diagnostics_interactions.csv")
print("\nInteraction (heterogeneity) tests, M5, Wald with patient clusters:\n", INT.round(4).to_string(index=False))

# =============================================================================
# 10. FIVE-FOLD OUT-OF-FOLD PREDICTION ERROR
# =============================================================================
hdr("9. FIVE-FOLD OUT-OF-FOLD PREDICTION ERROR (folds split by patient cluster)")
cl_ids = pd.Series(c5["groups"]["patient"])
uniq = cl_ids.unique()
RNG.shuffle(uniq)
fold_of = dict(zip(uniq, np.arange(len(uniq)) % N_FOLDS))
FOLD = cl_ids.map(fold_of).to_numpy()
oof, oof_rows, OOF_PRED = {}, [], {}
for model, core in CORE.items():
    X_, y_ = core["X"], core["y"]
    pred, null = np.zeros(len(y_)), np.zeros(len(y_))
    for f_ in range(N_FOLDS):
        tr, te = FOLD != f_, FOLD == f_
        fit = fit_glm(X_[tr], y_[tr], "gamma")
        eta = np.clip(X_[te] @ fit["beta"], np.log(y_[tr].min()), np.log(y_[tr].max()))
        pred[te] = np.exp(eta)
        null[te] = y_[tr].mean()
    OOF_PRED[model] = pred
    ae = np.abs(y_ - pred)
    mae_null = np.abs(y_ - null).mean()
    oof_rows.append({"model": model, "MAE_intl": ae.mean(), "MedAE_intl": np.median(ae),
                     "RMSE_intl": float(np.sqrt(np.mean((y_ - pred) ** 2))), "MAE_pct_of_mean_cost": 100 * ae.mean() / y_.mean(),
                     "MAE_log": float(np.mean(np.abs(np.log(y_) - np.log(pred)))),
                     "mean_pred_over_mean_obs": pred.mean() / y_.mean(), "MAE_null_mean_model": mae_null,
                     "skill_vs_null_pct": 100 * (1 - ae.mean() / mae_null)})
OOF = pd.DataFrame(oof_rows)
save(OOF, "o2_oof_error_by_model.csv")
print(OOF.round(3).to_string(index=False))
pred5, y5 = OOF_PRED["M5"], c5["y"]
dd = D.loc[c5["sub_index"]]
by_rows = []
for var in ["billing_mode", "group"]:
    for lev, idx in dd.groupby(var).indices.items():
        by_rows.append({"stratum": f"{var}={lev}", "n": len(idx), "MAE_intl": np.abs(y5[idx] - pred5[idx]).mean(),
                        "mean_obs": y5[idx].mean(), "mean_pred": pred5[idx].mean()})
save(pd.DataFrame(by_rows), "o2_oof_error_by_stratum_M5.csv")
print("\nM5 out-of-fold error by stratum:\n", pd.DataFrame(by_rows).round(1).to_string(index=False))
CAL_ALL = []
for model in ["M3", "M5"]:
    if model not in OOF_PRED:
        continue
    pr = OOF_PRED[model]
    dec = pd.qcut(pd.Series(pr).rank(method="first"), 10, labels=False)
    cal = pd.DataFrame({"model": model, "decile": dec + 1, "pred": pr, "obs": CORE[model]["y"]}).groupby(["model", "decile"]).agg(
        n=("obs", "size"), mean_predicted=("pred", "mean"), mean_observed=("obs", "mean")).reset_index()
    cal["obs_over_pred"] = cal.mean_observed / cal.mean_predicted
    cal["MAE"] = [np.abs(CORE[model]["y"][(dec == k).to_numpy()] - pr[(dec == k).to_numpy()]).mean() for k in range(10)]
    CAL_ALL.append(cal)
CAL = pd.concat(CAL_ALL, ignore_index=True)
save(CAL, "o2_oof_calibration_deciles.csv")
print("\nCalibration by decile of out-of-fold prediction (M5):\n", CAL[CAL.model == "M5"].round(2).to_string(index=False))

# =============================================================================
# 11. CONCENTRATION
# =============================================================================
hdr("10. CONCENTRATION OF EXPENDITURE")


def conc_stats(x):
    x = np.sort(np.asarray(x, float))
    n, tot = len(x), x.sum()
    cum = np.concatenate([[0], np.cumsum(x) / tot])
    p = np.linspace(0, 1, n + 1)
    gini = 1 - 2 * float(np.sum((cum[1:] + cum[:-1]) / 2 * np.diff(p)))
    out = {"n": n, "total_intl": tot, "gini": gini}
    for q in (1, 5, 10, 20):
        out[f"top{q}pct_share"] = 100 * x[-max(int(np.ceil(q / 100 * n)), 1):].sum() / tot
    return out


def conc_boot(x, B):
    x = np.asarray(x, float)
    r = [conc_stats(RNG.choice(x, len(x))) for _ in range(B)]
    return (np.percentile([v["top10pct_share"] for v in r], [2.5, 97.5]), np.percentile([v["gini"] for v in r], [2.5, 97.5]))


crow = []
for lab, mask in [("All (primary sample)", pd.Series(True, index=D.index))] + \
        [(f"group={g}", D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())] + \
        [(f"{g} x {b}", (D.group == g) & (D.billing_mode == b)) for g in sorted(D.group.unique()) for b in sorted(D.billing_mode.unique())] + \
        [("All, flagged extremes removed", ~D.flag_extreme_any)]:
    x = D.loc[mask, "cost_intl"]
    if len(x) < 20:
        continue
    s = conc_stats(x)
    (t_lo, t_hi), (g_lo, g_hi) = conc_boot(x, NBOOT)
    crow.append({"stratum": lab, **s, "top10_lo": t_lo, "top10_hi": t_hi, "gini_lo": g_lo, "gini_hi": g_hi})
CONC = pd.DataFrame(crow)
save(CONC, "o2_concentration.csv")
print(CONC.round(2).to_string(index=False))
lz = []
for lab, mask in [("All", pd.Series(True, index=D.index))] + [(g, D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())]:
    xs = np.sort(D.loc[mask, "cost_intl"].to_numpy())
    cum = np.concatenate([[0], np.cumsum(xs) / xs.sum()])
    pp = np.linspace(0, 1, len(xs) + 1)
    for q in np.linspace(0, 1, 101):
        lz.append({"stratum": lab, "cum_share_episodes": q, "cum_share_cost": float(np.interp(q, pp, cum))})
LZ = pd.DataFrame(lz)
save(LZ, "o2_lorenz_points.csv")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
    for lab in ["All"] + sorted(D.group.unique()):
        s = LZ[LZ.stratum == lab]
        ax[0].plot(s.cum_share_episodes, s.cum_share_cost, label=lab, lw=2 if lab == "All" else 1.4)
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
    ax[0].set(title="Lorenz curve of episode cost by group", xlabel="Cumulative share of episodes (ordered by cost)",
              ylabel="Cumulative share of expenditure")
    ax[0].legend(frameon=False)
    c8 = CAL[CAL.model == "M5"]
    ax[1].plot(c8.mean_predicted, c8.mean_observed, "o-")
    lim = [0, max(c8.mean_predicted.max(), c8.mean_observed.max()) * 1.05]
    ax[1].plot(lim, lim, "k--", lw=.8)
    ax[1].set(title="M5 calibration by decile (out-of-fold)", xlabel="Mean predicted cost (Int$)", ylabel="Mean observed cost (Int$)")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o2_lorenz_and_calibration.png", dpi=200)
    plt.close(fig)
    print("Figure saved: o2_lorenz_and_calibration.png")
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")

# =============================================================================
# 12. HYPOTHESES 2 AND 3
# =============================================================================
hdr("11. HYPOTHESIS 2 (provinces) AND HYPOTHESIS 3 (K40: general surgery vs urology)")
h2 = []
for model in ["M4", "M5"]:
    core = CORE[model]
    idx = [i for i, nm in enumerate(core["names"]) if nm.startswith("C(province_model)[T.")]
    chi2, df_, p = wald(core, idx)
    pr = np.exp(core["beta"][idx])
    h2.append({"model": model, "wald_chi2": chi2, "df": df_, "p_value": p, "min_province_ratio": pr.min(),
               "max_province_ratio": pr.max(), "sd_log_ratio": np.log(pr).std(),
               "note": "patient-clustered Wald: optimistic for facility-level structure; see sens 8 sigma_u"})
H2 = pd.DataFrame(h2)
save(H2, "o2_h2_province_wald.csv")
print("H2: joint Wald test of the province terms (cost differs across provinces after case-mix adjustment):")
print(H2.round(4).to_string(index=False))

H3_rows = []
d3 = D[(D.icd10 == "K40") & D.specialty_h3.isin(["General surgery", "Urology"])].copy()
tab_h3 = d3.groupby(["specialty_h3", "billing_mode"])["cost_intl"].agg(["size", "median", "mean"]).reset_index()
save(tab_h3, "o2_h3_descriptives.csv")
print(f"\nH3: K40 episodes with general surgery or urology as specialty: {len(d3):,}")
print(tab_h3.round(1).to_string(index=False))
COV3 = ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode", "any_surgical_service",
        "discharge_status_model"]
if d3.specialty_h3.nunique() == 2 and d3.specialty_h3.value_counts().min() >= 10:
    pv = pd.crosstab(d3.province_model, d3.specialty_h3)
    both = pv[(pv >= 10).all(axis=1)].index.tolist()
    variants = [("a) unadjusted", d3, ["specialty_h3"], []),
                ("b) case-mix adjusted, no province", d3, ["specialty_h3"] + COV3, ["los_cov"]),
                ("c) + province fixed effects", d3, ["specialty_h3"] + COV3 + ["province_model"], ["los_cov"]),
                ("d) c, provinces with >= 10 of each specialty", d3[d3.province_model.isin(both)],
                 ["specialty_h3"] + COV3 + ["province_model"], ["los_cov"]),
                ("e1) b, Global only", d3[d3.billing_mode == "Global"], ["specialty_h3"] + NO_BILLING(COV3), ["los_cov"]),
                ("e2) b, Other only", d3[d3.billing_mode != "Global"], ["specialty_h3"] + NO_BILLING(COV3), ["los_cov"])]
    for name, dd_, cats, nums in variants:
        if len(dd_) < 40 or dd_.specialty_h3.nunique() < 2 or dd_.specialty_h3.value_counts().min() < 5:
            H3_rows.append({"variant": name, "n": len(dd_), "note": "too few episodes"})
            continue
        try:
            core = fit_core(dd_, cats, nums)
            term = "C(specialty_h3)[T.Urology]"
            if term not in core["names"]:
                H3_rows.append({"variant": name, "n": len(dd_), "note": "specialty term dropped"})
                continue
            j = core["names"].index(term)
            row = {"variant": name, "n": len(dd_), "n_urology": int((dd_.specialty_h3 == "Urology").sum()),
                   "n_general_surgery": int((dd_.specialty_h3 == "General surgery").sum()),
                   "provinces": int(dd_.province_model.nunique()), "note": ""}
            for cl in ("patient", "province"):
                cov, G = core["covs"][cl]
                se = float(np.sqrt(cov[j, j]))
                crit = stats.t.ppf(.975, max(G - 1, 1)) if cl == "province" else Z975
                row[f"cost_ratio_urology_vs_gensurg_{cl}"] = float(np.exp(core["beta"][j]))
                row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(core["beta"][j] - crit * se)), float(np.exp(core["beta"][j] + crit * se))
                row[f"p_{cl}"] = float(2 * (stats.t.sf(abs(core["beta"][j] / se), max(G - 1, 1)) if cl == "province"
                                            else stats.norm.sf(abs(core["beta"][j] / se))))
            if "province_model" in cats:
                row["lo_province"] = row["hi_province"] = row["p_province"] = np.nan   # absorbed by province terms
            H3_rows.append(row)
        except Exception as e:
            H3_rows.append({"variant": name, "n": len(dd_), "note": f"failed: {type(e).__name__}"})
else:
    H3_rows.append({"variant": "all", "note": "too few K40 episodes in one specialty"})
H3 = pd.DataFrame(H3_rows)
save(H3, "o2_h3_k40_specialty.csv")
print("\nH3: cost ratio Urology vs General surgery (K40):\n", H3.round(3).to_string(index=False))
print("Caveat: specialty is tied to hospital and province; variant d is the closest comparison within province.")

# =============================================================================
# 13. SENSITIVITY ANALYSES 1-8
# =============================================================================
SENS = {1: ("Surgical service only", "keep_sens1_surgical"),
        2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"),
        4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS covariate capped at P95", None)}
SL = pd.DataFrame()
if RUN_SENS:
    hdr("12. SENSITIVITY ANALYSES 1-7 (M5, primary sample)")
    stabs = []
    for s, (name, keepcol) in SENS.items():
        print(f"\nSensitivity {s}: {name}")
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        if s == 7:
            sub["los_cov"] = sub["los_capped_p95"]
        tab, _ = fit_one(sub, "M5", BLOCKS["M5"], NUMS["M5"])
        if tab is not None:
            tab = tab[tab.cluster == "patient"].copy()
            tab["sens"], tab["sens_name"] = s, name
            stabs.append(tab)
    SL = pd.concat(stabs, ignore_index=True)


class LaplaceGamma:
    """Gamma-log GLMM with a province random intercept, Laplace-approximated ML (as lme4::glmer nAGQ = 1)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def _l(self, eta, nu):
        e = np.exp(-eta)
        l = nu * np.log(nu) - gammaln(nu) + (nu - 1) * np.log(self.y) - nu * self.y * e - nu * eta
        return l, nu * (self.y * e - 1), -nu * self.y * e

    def negll(self, theta):
        beta, sig, nu = theta[:self.k], np.exp(theta[self.k]), np.exp(theta[self.k + 1])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            l, d1, d2 = self._l(xb + u[self.g], nu)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        l, d1, d2 = self._l(xb + u[self.g], nu)
        h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
        return -(l.sum() - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0, nu0):
        th0 = np.concatenate([beta0, [np.log(0.3)], [np.log(nu0)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()
if RUN_SENS8:
    hdr("12b. SENSITIVITY 8: province as a random intercept (Gamma-log, Laplace ML)")
    t0 = time.time()
    try:
        cats = [v for v in BLOCKS["M5"] if v != "province_model"] + TIME
        X8, names8, spec8 = build_spec(D, cats, ["los_cov"])
        y8 = D["cost_intl"].to_numpy(float)
        g8 = pd.factorize(D["province_model"])[0]
        f0 = fit_glm(X8, y8, "gamma")
        m8 = LaplaceGamma(X8, y8, g8)
        theta = m8.fit(f0["beta"], 1 / max(f0["phi"], 1e-3))
        H = m8.hessian(theta)
        for _ in range(2):
            gr = optimize.approx_fprime(theta, m8.negll, 1e-6)
            try:
                cand = theta - np.linalg.solve(H, gr)
            except np.linalg.LinAlgError:
                break
            if m8.negll(cand) <= m8.negll(theta) + 1e-9:
                theta = cand
                H = m8.hessian(theta)
        cov8 = np.linalg.pinv(H)
        se8 = np.sqrt(np.clip(np.diag(cov8), 0, None))
        k8 = m8.k
        S8 = pd.DataFrame({"outcome": "cost", "model": "M5 random intercept", "term": names8[1:], "n": len(D),
                           "level_n": [int(X8[:, j].sum()) if names8[j].startswith("C(") else np.nan for j in range(1, k8)],
                           "ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                           "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]),
                           "p": 2 * stats.norm.sf(np.abs(theta[1:k8] / np.where(se8[1:k8] > 0, se8[1:k8], np.nan))),
                           "sens": 8, "sens_name": "Province random intercept"})
        sig = float(np.exp(theta[k8]))
        VC8 = pd.DataFrame([{"outcome": "cost", "n": len(D), "provinces": m8.G, "sigma_u": sig,
                             "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])),
                             "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                             "province_cost_ratio_1sd": float(np.exp(sig)),
                             "gamma_dispersion": float(1 / np.exp(theta[k8 + 1])), "loglik": -float(m8.negll(theta)),
                             "max_abs_grad": float(np.abs(optimize.approx_fprime(theta, m8.negll, 1e-6)).max())}])
        save(S8, "o2_sens8_random_intercept_long.csv")
        save(VC8, "o2_sens8_variance_components.csv")
        print(f"  n = {len(D):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (a province one SD above the mean costs "
              f"x{np.exp(sig):.2f}), {time.time() - t0:.0f}s")
        cmpx = S8[S8.term.str.contains("billing|group|surgical|ownership|los_cov")].merge(
            LONG[LONG.model == "M5"][["term", "ratio", "lo", "hi"]], on="term", suffixes=("_RI", "_fixed"))
        cmpx["same_direction"] = (cmpx.ratio_RI > 1) == (cmpx.ratio_fixed > 1)
        cmpx["pct_difference"] = 100 * (cmpx.ratio_RI / cmpx.ratio_fixed - 1)
        save(cmpx, "o2_sens8_vs_fixed_M5.csv")
        print(cmpx[["term", "ratio_RI", "lo_RI", "hi_RI", "ratio_fixed", "same_direction", "pct_difference"]].round(3).to_string(index=False))
        SL = pd.concat([SL, S8], ignore_index=True)
    except Exception as e:
        print(f"  Sensitivity 8 FAILED ({type(e).__name__}: {e})")
if len(SL):
    SL["analysis_status"] = "pre-specified sensitivity analyses; primary sample redefined after Objective 1 diagnostics"
    save(SL, "o2_sensitivity_long.csv")
    sd_rows = []
    for _, r in LONG[LONG.model == "M5"].iterrows():
        s = SL[SL.term == r.term]
        if len(s):
            same = (s.ratio > 1) == (r.ratio > 1)
            sd_rows.append({"term": r.term, "main_ratio": r.ratio, "main_CI_excludes_1": hit(r), "n_sens": len(s),
                            "same_direction": int(same.sum()), "same_direction_and_significant": int((same & ((s.lo > 1) | (s.hi < 1))).sum())})
    save(pd.DataFrame(sd_rows), "o2_sensitivity_direction.csv")

# =============================================================================
# 14. EXPLORATORY
# =============================================================================
EXPL = []


def add_exp(code, name, d, fam="gamma", nums=None, core_only=True):
    print(f"\nExploratory {code}: {name} (n = {len(d):,})")
    tab, _ = fit_one(d, "M5", BLOCKS["M5"], NUMS["M5"] if nums is None else nums, fam=fam)
    if tab is not None:
        tab = tab[tab.cluster == "patient"].copy()
        if core_only:
            tab = tab[tab.term.str.startswith(tuple(f"C({v})[" for v in CORE_VARS) + ("los_cov",))]
        tab["analysis"], tab["analysis_name"] = code, name
        EXPL.append(tab)


if RUN_EXTRA:
    hdr("13. EXPLORATORY ANALYSES (report as exploratory)")
    add_exp("X0", f"Other sample: {ALT_LABEL}", ALT, core_only=False)
    add_exp("X8", "Quasi-Poisson (variance proportional to mean) instead of Gamma", D, fam="quasipoisson", core_only=False)
    add_exp("X9", "Model without LOS (LOS is a mechanism of billing and surgical effects)", D, nums=[], core_only=False)
    tiny = pd.Series(False, index=D.index)
    for v in ["sex", "insurance_fund_model", "adm_year"]:
        vcn = D[v].value_counts()
        tiny |= D[v].isin(vcn[vcn < SPARSE_N_MIN].index)
    add_exp("X6", f"Episodes in cells < {SPARSE_N_MIN} (sex, fund, year) removed ({int(tiny.sum())})", D[~tiny])
    for p in [p for p in D.province_model.value_counts().index if p != "Other"][:LOPO_N]:
        add_exp(f"X7_{p}", f"Without {p}", D[D.province_model != p])
    if EXPL:
        EX = pd.concat(EXPL, ignore_index=True)
        EX["analysis_status"] = "exploratory (added after Objective 1 diagnostics)"
        save(EX, "o2_exploratory_long.csv")
        base = LONG[LONG.model == "M5"][["term", "ratio"]].rename(columns={"ratio": "primary_M5_ratio"})
        core_tab = EX[EX.term.str.startswith(tuple(f"C({v})[" for v in CORE_VARS) + ("los_cov",))].merge(base, on="term", how="left")
        core_tab["pct_vs_primary"] = 100 * (core_tab.ratio / core_tab.primary_M5_ratio - 1)
        save(core_tab[["analysis", "analysis_name", "term", "n", "ratio", "lo", "hi", "p", "primary_M5_ratio", "pct_vs_primary"]],
             "o2_exploratory_core_terms.csv")
        print("\nSample / variance-function / LOS comparisons (main exposures):")
        sel_ = core_tab[core_tab.analysis.isin(["X0", "X8", "X9"]) & core_tab.term.str.contains("billing|group|surgical|los_cov")]
        print(sel_[["analysis", "term", "ratio", "primary_M5_ratio", "pct_vs_primary"]].round(3).to_string(index=False))

# =============================================================================
# 15. RELIABILITY SUMMARY
# =============================================================================
hdr("14. RELIABILITY SUMMARY OF MAIN-TABLE ESTIMATES")
prefix = tuple(f"C({v})[" for v in REPORT_VARS if v != "los_cov") + ("los_cov",)
rel = []
main = LONG[(LONG.model == "M5") & LONG.term.str.startswith(prefix)]
for _, r in main.iterrows():
    sig_p = hit(r)
    pcr = PC[PC.term == r.term]
    sig_pc = None if "province_model" in r.term else (hit(pcr.iloc[0]) if len(pcr) else None)
    s = SL[SL.term == r.term] if len(SL) else pd.DataFrame()
    sens_n = len(s)
    same = ((s.ratio > 1) == (r.ratio > 1)) if sens_n else pd.Series(dtype=bool)
    sens_dir = int(same.sum()) if sens_n else 0
    sens_sig = int((same & ((s.lo > 1) | (s.hi < 1))).sum()) if sens_n else 0
    st = ST[(ST.term == r.term) & (ST.cluster == "patient")] if len(ST) else pd.DataFrame()
    rg = st[st.stratum == "Global"].ratio if len(st) else pd.Series(dtype=float)
    ro = st[st.stratum == "Other"].ratio if len(st) else pd.Series(dtype=float)
    if "billing_mode" in r.term:
        strata_dir, spread = None, np.nan
    elif len(rg) and len(ro):
        strata_dir = bool(((rg.iloc[0] > 1) == (r.ratio > 1)) and ((ro.iloc[0] > 1) == (r.ratio > 1)))
        spread = float(max(rg.iloc[0], ro.iloc[0]) / min(rg.iloc[0], ro.iloc[0]))
    else:
        strata_dir, spread = None, np.nan
    why = []
    if "billing_mode" in r.term:
        if len(BPP):
            b = BPP.iloc[0]
            re_sig = (b.pooled_lo > 1) or (b.pooled_hi < 1)
            if not b.same_direction_all_provinces:
                why.append(f"direction differs across provinces ({int(b.provinces_ratio_below_1)} below 1, {int(b.provinces_ratio_above_1)} above 1)")
            if not re_sig:
                why.append("random-effects pooled CI includes 1")
            if sens_n and sens_dir < sens_n:
                why.append(f"direction fails in {sens_n - sens_dir} of {sens_n} sensitivity analyses")
            verdict = "B" if (b.same_direction_all_provinces and re_sig and not (sens_n and sens_dir < sens_n)) else "C"
            if verdict == "B":
                why.append("same direction in every province; size varies (see I2 / prediction interval)")
        else:
            verdict = "C"
            why.append("billing-by-province analysis not run")
    else:
        if not sig_p:
            why.append("CI includes 1 (patient clusters)")
        if sig_pc is False:
            why.append("CI includes 1 with province clusters")
        if sens_n and sens_dir < sens_n:
            why.append(f"direction fails in {sens_n - sens_dir} of {sens_n} sensitivity analyses")
        if strata_dir is False:
            why.append("direction differs between Global-only and itemised-only models")
        if r.sparse_flag:
            why.append("sparse cell")
        if pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N:
            why.append(f"fewer than {MIN_LEVEL_N} episodes")
        ok = not why
        if "province_model" in r.term:
            ok = sig_p and not r.sparse_flag and (sens_n == 0 or sens_dir == sens_n) and strata_dir is not False \
                and not (pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N)
            why.append("province coefficient: no province-level robust SE; read as descriptive of case-mix and tariff differences")
        if not ok:
            verdict = "C"
        elif pd.notna(spread) and spread > SPREAD_FLAG:
            verdict = "B"
            why.append(f"size differs between billing strata (ratio of ratios {spread:.1f})")
        else:
            verdict = "A"
    rel.append({"term": r.term, "level_n": r.level_n, "ratio": round(r.ratio, 3), "CI_patient": f"{r.lo:.2f}-{r.hi:.2f}",
                "CI_province": f"{pcr.iloc[0].lo:.2f}-{pcr.iloc[0].hi:.2f}" if len(pcr) else "",
                "sens_tested": sens_n, "sens_same_direction": sens_dir, "sens_same_direction_and_sig": sens_sig,
                "ratio_Global_only": round(rg.iloc[0], 3) if len(rg) else np.nan,
                "ratio_Other_only": round(ro.iloc[0], 3) if len(ro) else np.nan,
                "strata_spread": round(spread, 2) if pd.notna(spread) else np.nan, "verdict": verdict, "why": "; ".join(why)})
REL = pd.DataFrame(rel)
save(REL, "o2_reliability_summary.csv")
print("Verdict counts (A direction and size hold; B direction holds, size varies; C not robust):")
print(REL.verdict.value_counts().to_string())
print("\nMain exposures:")
print(REL[REL.term.str.contains("billing|group|surgical|ownership|los_cov")][["term", "ratio", "CI_patient", "CI_province", "verdict", "why"]]
      .to_string(index=False, max_colwidth=70))

# =============================================================================
# 16. RUN INFO
# =============================================================================
info = {"script": "objective2_expenditure.py", "input_file": str(INPUT_CSV), "input_sha256": sha256(INPUT_CSV),
        "rows_in_file": int(len(DALL)), "primary_sample": SAMPLE_LABEL, "primary_sample_n": int(len(D)), "seed": SEED,
        "bootstrap_reps": NBOOT, "reference_levels": {k: REFS[k] for k in BLOCKS["M5"] + TIME},
        "settings": {"sparse_n_min": SPARSE_N_MIN, "min_level_n": MIN_LEVEL_N, "cook_threshold": f"{COOK_MULT}/n",
                     "folds": N_FOLDS, "prov_min_per_mode": PROV_MIN_PER_MODE, "irr_per_intl": 118411.24},
        "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "scipy": scipy.__version__},
        "notes": ["Primary sample = admissions (Global + Inpatient), as in Objective 1 v3: a deviation from the frozen protocol",
                  "Costs are nominal IRR converted with one 2024 PPP factor; year and month are in every model",
                  "Billing-by-province, stratified, interaction, X0-X9, H3 variants and reliability analyses are exploratory",
                  "Province-clustered SEs: t with G-1 df; province coefficients have no province-level SE"]}
with open(OUT_DIR / "o2_run_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, default=str)
print("\n" + "=" * 78)
print(f"DONE. Outputs in {OUT_DIR}")
print("=" * 78)


1. LOAD AND VERIFY
Rows: 15,601; columns: 70
                                                      step     n
                      Episodes in the frozen analysis file 15601
                                        record type Global  8462
                                     record type Inpatient  6223
                         record type Emergency Observation   889
                 record type Temporary Inpatient Admission    27
O2 complete cases (cost and LOS present), all record types 15601
        PRIMARY SAMPLE O2: admissions (Global + Inpatient) 14685

Cost (Int$): median 436, IQR 301-708, mean 602, max 37,984
Reference levels: {'age_group': '40-59', 'sex': 'Male', 'insurance_fund_model': 'Rural Health Insurance Fund', 'ownership': 'Public University-Affiliated', 'billing_mode': 'Other', 'group': 'Hernia', 'any_surgical_service': 'No', 'discharge_status_model': 'Complete Recovery', 'province_model': 'Khuzestan', 'specialty_model': 'General Surgery Specialist'}

2. DESCRIPTIVES 

In [18]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 2 (expenditure) of the analysis protocol  --  VERSION 2 (corrected after diagnostics on the real file)
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional).

WHAT CHANGED FROM v1 (each change was forced by a diagnostic on digestive_surgical_analysis_file.csv)
  1  PRIMARY MODEL IS STRATIFIED BY BILLING MODE (Global package vs itemised "Other"), fitted as ONE fully
     interacted block-diagonal Gamma-log GLM (so patient/province-clustered covariances and Global-vs-Other
     contrasts are valid). v1 forced one set of covariate effects and one constant Global/Other ratio on two
     cost-generating processes that differ enormously (dispersion 0.05 vs 0.24, different ownership/surgical/
     province effects). The additive pooled model is still fitted (M1-M5, protocol comparable) but is labelled
     "pooled additive, known to be mis-specified" and is not used for conclusions.
  2  TIME: the nominal tariff moves in steps (about +45-50% in the first two months of 1402) and drifts inside
     1401. Separate year and month dummies cannot represent this; v2 uses year-month (period) dummies.
     The 10 episodes of 1400-12 are merged into 1401-01; any further period with < RARE_MIN episodes in a
     stratum is merged into its chronological neighbour.
  3  Categories with < RARE_MIN (30) episodes WITHIN A STRATUM are collapsed (to "Other", or to the modal level
     if the Other bucket would still be < 30) so no coefficient rests on a handful of episodes.
  4  POSITIVITY: any_surgical_service is not estimable in the Global stratum (only ~31 non-surgical Global
     episodes) and is dropped there; it stays in the itemised stratum.
  5  AVERAGE MARGINAL COSTS come from the stratified model (g-computation, delta-method CIs with patient AND
     province clusters) and are compared with the observed cell means; the Global-vs-itemised contrast is
     the standardised effect on the Global episodes (ATT): what the Global episodes would have cost if billed
     itemised, with the share of episodes on common support reported.
  6  Diagnostics (Park, link, Cook + refit, quasi-Poisson check) are run WITHIN each stratum; five-fold
     patient-grouped out-of-fold validation compares pooled vs stratified models by stratum.
  7  H1 compares variation after removing tariff steps (ICD-10 x period, and x province cells).
     H2 is tested within each stratum, H3 is shown crude / adjusted / province FE with a province-level
     cluster bootstrap CI (the province coefficient has no valid province-clustered SE otherwise).
  8  Sensitivity analyses 1-8 are re-run on the stratified specification; the A/B/C reliability verdict is
     computed from them (sparse level, direction, size of change, Cook refit, quasi-Poisson, province
     cluster, province random intercept).

PRIMARY SAMPLE  Admissions (record types Global + Inpatient), as in Objective 1 v3. Protocol sample (all valid
          episodes) is fitted as exploratory X0. This and the billing-mode stratification are deviations from
          the frozen protocol: state them in the Methods and label the analyses post hoc.
OUTCOME   cost_intl (Int$ = IRR/118,411.24, nominal, not inflation-adjusted).

USAGE
  python objective2_expenditure.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 300]
Outputs: folder objective2_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import gammaln

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "v2-stratified"
SEED = 42
RARE_MIN = 30                       # minimum episodes per category level within a stratum
SPARSE_N_MIN = 30
TOP_PROVINCES = 5
COOK_MULT = 4.0                     # Cook's distance threshold = COOK_MULT / n
N_FOLDS = 5
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
STABLE_PCT = 15.0                   # max % change across sensitivity analyses for an "A" estimate
COOK_PCT = 10.0                     # max % change in the Cook refit / quasi-Poisson for an "A" estimate

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=300)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service",
           "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
NUMS = {"M1": [], "M2": [], "M3": ["los_cov"], "M4": ["los_cov"], "M5": ["los_cov"]}   # LOS enters in M3
COLLAPSE_VARS = ["ownership", "specialty_model", "province_model", "insurance_fund_model",
                 "discharge_status_model", "sex", "age_group", "period"]
KEY_RE = r"C\(group\)|any_surgical_service|C\(ownership\)|los_cov"
Z975 = stats.norm.ppf(0.975)


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE THE PRIMARY SAMPLE
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "cost_intl", "total_cost_irr", "los_days", "los_capped_p99",
        "los_capped_p95", "cluster_id", "any_surgical_service", "specialty_model", "specialty_h3",
        "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O2",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O2", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "specialty_h3", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["cost_intl", "total_cost_irr", "los_days", "los_capped_p99", "los_capped_p95"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
if (DALL["cost_intl"] <= 0).any() or DALL["cost_intl"].isna().any():
    fail("cost_intl has missing or non-positive values; the frozen file should not.")
DALL["los_cov"] = DALL["los_capped_p99"]
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"          # 10 episodes; see header
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
valid = DALL["in_O2"] & DALL["los_cov"].notna()
flow.append({"step": "O2 complete cases (cost and LOS present), all record types", "n": int(valid.sum())})
D = (DALL[is_adm & valid] if args.sample == "admissions" else DALL[valid]).copy().reset_index(drop=True)
ALT = (DALL[valid] if args.sample == "admissions" else DALL[is_adm & valid]).copy().reset_index(drop=True)
SAMPLE_LABEL = "admissions (Global + Inpatient)" if args.sample == "admissions" else "all valid episodes (protocol sample)"
ALT_LABEL = "all valid episodes (protocol sample)" if args.sample == "admissions" else "admissions only"
flow.append({"step": f"PRIMARY SAMPLE O2: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o2_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
print(f"\nCost (Int$): median {D.cost_intl.median():,.0f}, IQR {D.cost_intl.quantile(.25):,.0f}-"
      f"{D.cost_intl.quantile(.75):,.0f}, mean {D.cost_intl.mean():,.0f}, max {D.cost_intl.max():,.0f}")
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary sample.")


# =============================================================================
# 2. MODEL ENGINE: block-diagonal (stratified) or pooled Gamma-log / quasi-Poisson-log GLM
# =============================================================================
def build_spec(d, cats, nums, refs=None):
    refs = refs if refs is not None else {}
    cols, names, levs = [np.ones(len(d))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        for lev in levs[v][1:]:
            cols.append((s == lev).to_numpy(float))
            names.append(f"C({v})[T.{lev}]")
    for v in nums:
        cols.append(d[v].to_numpy(float))
        names.append(v)
    X = np.column_stack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], {"cats": list(cats), "nums": list(nums), "levs": levs, "keep": keep}


def apply_spec(d, spec):
    cols = [np.ones(len(d))]
    for v in spec["cats"]:
        s = d[v].astype(str)
        for lev in spec["levs"][v][1:]:
            cols.append((s == lev).to_numpy(float))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float))
    return np.column_stack(cols)[:, spec["keep"]]


def merge_periods(vc, minn):
    groups = [[p, [p], int(vc[p])] for p in sorted(vc.index)]          # [label, members, count]
    while len(groups) > 1 and min(g[2] for g in groups) < minn:
        i = int(np.argmin([g[2] for g in groups]))
        j = i + 1 if i + 1 < len(groups) else i - 1
        groups[j][1] += groups[i][1]
        groups[j][2] += groups[i][2]
        groups.pop(i)
    return {m: g[0] for g in groups for m in g[1]}


def make_cmap(ds, vars_, minn=RARE_MIN):
    """Collapse levels with < minn episodes inside this stratum."""
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        if v == "period":
            m = merge_periods(vc, minn)
        else:
            rare = [l for l in vc.index if vc[l] < minn]
            m = {}
            if rare:
                total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
                tgt = "Other" if total >= minn else vc.index[0]
                m = {l: tgt for l in rare}
                if tgt == "Other" and "Other" in vc.index:
                    pass
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def apply_cmap(d, cm):
    d = d.copy()
    for v, m in cm["map"].items():
        s = d[v].astype(str).map(lambda x: m.get(x, x))
        seen = cm["seen"][v]
        d[v] = s.where(s.isin(seen), "Other" if "Other" in seen else "__unseen__")
    return d


def fit_glm(X, y, fam="gamma"):
    """Log-link GLM by IRLS with step-halving on the deviance (plain IRLS diverges on the itemised stratum).
    Gamma: V = mu^2 (working weight 1). quasi-Poisson ('qp'): V = mu."""
    n, k = X.shape

    def dev(mu):
        if fam == "gamma":
            return 2 * np.sum(-np.log(y / mu) + (y - mu) / mu)
        return 2 * np.sum(y * np.log(y / mu) - (y - mu))

    beta = np.zeros(k)
    beta[0] = np.log(y.mean())
    eta = X @ beta
    mu = np.exp(eta)
    d_old = dev(mu)
    conv = False
    for _ in range(300):
        w = np.ones(n) if fam == "gamma" else mu
        z = eta + (y - mu) / mu
        A, rhs = (X.T * w) @ X, (X.T * w) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        step = 1.0
        for _h in range(30):
            cand = beta + step * (b_new - beta)
            eta_c = np.clip(X @ cand, -30, 30)
            mu_c = np.exp(eta_c)
            d_new = dev(mu_c)
            if np.isfinite(d_new) and d_new <= d_old + 1e-10 * abs(d_old):
                break
            step /= 2
        done = np.abs(cand - beta).max() < 1e-9 or abs(d_old - d_new) < 1e-12 * abs(d_old)
        beta, eta, mu, d_old = cand, eta_c, mu_c, d_new
        if done:
            conv = True
            break
    w = np.ones(n) if fam == "gamma" else mu
    V = mu ** 2 if fam == "gamma" else mu
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * w) @ X), "conv": conv,
            "phi": float(np.sum((y - mu) ** 2 / V) / max(n - k, 1)), "V": V}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups, fam="gamma"):
    n, k = X.shape
    s = (y - f["mu"]) / f["mu"] if fam == "gamma" else (y - f["mu"])
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_block(d, cats, nums, mode="stratified", fam="gamma", cov=True, minn=RARE_MIN):
    """mode 'stratified': one separate model per billing-mode stratum, fitted jointly as a block-diagonal GLM
    (any_surgical_service is dropped in the Global stratum). mode 'pooled': one additive model, billing_mode
    as a covariate (v1 specification, kept for comparison only)."""
    d = d.reset_index(drop=True)
    if mode == "pooled":
        parts = {"All": np.ones(len(d), bool)}
    else:
        parts = {s: (d["stratum"] == s).to_numpy() for s in ["Global", "Other"] if (d["stratum"] == s).sum() >= 50}
    blocks, Xs, names, off = {}, [], [], 0
    for s, rows in parts.items():
        ds = d.loc[rows]
        cs = [c for c in cats if not (mode == "stratified" and c in ("any_surgical_service", "billing_mode") and
                                      (s == "Global" or c == "billing_mode"))]
        cs = list(dict.fromkeys(cs + ["period"]))
        cm = make_cmap(ds, [v for v in cs if v in COLLAPSE_VARS], minn)
        dm = apply_cmap(ds, cm)
        refs = {"period": min(dm["period"].unique())}
        X_s, nm_s, spec = build_spec(dm, cs, nums, refs)
        blocks[s] = {"rows": np.where(rows)[0], "cols": slice(off, off + X_s.shape[1]), "names": nm_s,
                     "spec": spec, "cmap": cm, "X": X_s,
                     "lev_n": [int(X_s[:, j].sum()) if nm.startswith("C(") else np.nan for j, nm in enumerate(nm_s)]}
        off += X_s.shape[1]
        Xs.append((rows, X_s))
    K = off
    X = np.zeros((len(d), K))
    for (rows, X_s), b in zip(Xs, blocks.values()):
        X[np.ix_(np.where(rows)[0], np.arange(b["cols"].start, b["cols"].stop))] = X_s
    y = d["cost_intl"].to_numpy(float)
    f = fit_glm(X, y, fam)
    pre = {s: s for s in blocks}
    allnames = [(f"{s}|{nm}" if mode == "stratified" else nm) for s, b in blocks.items() for nm in b["names"]]
    lev_n = [x for b in blocks.values() for x in b["lev_n"]]
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl], fam) for cl in groups} if cov else {}
    phi = {}
    for s, b in blocks.items():
        r = b["rows"]
        phi[s] = float(np.sum((y[r] - f["mu"][r]) ** 2 / f["V"][r]) / max(len(r) - b["X"].shape[1], 1))
    return {"mode": mode, "fam": fam, "n": len(d), "names": allnames, "lev_n": lev_n, "beta": f["beta"],
            "covs": covs, "X": X, "y": y, "mu": f["mu"], "eta": f["eta"], "phi": phi, "conv": f["conv"],
            "blocks": blocks, "d": d, "bread": f["bread"], "V": f["V"], "K": K, "groups": groups}


def make_tab(core, model, label="cost"):
    out = []
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = core["beta"] / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term_full": core["names"], "n": core["n"],
                          "level_n": core["lev_n"], "ratio": np.exp(core["beta"]),
                          "lo": np.exp(core["beta"] - crit * se), "hi": np.exp(core["beta"] + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        if core["mode"] == "stratified":
            t["stratum"] = t.term_full.str.split("|", n=1).str[0]
            t["term"] = t.term_full.str.split("|", n=1).str[1]
        else:
            t["stratum"], t["term"] = "All", t["term_full"]
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[(t["term"] != "Intercept") & ~t["term"].str.startswith("C(period)")])
    return pd.concat(out, ignore_index=True)


def hit(r):
    return bool((r.lo > 1) or (r.hi < 1))


def crit_p(core, cl, est, se):
    G = core["covs"][cl][1]
    if cl == "province":
        c = stats.t.ppf(0.975, G - 1)
        return c, 2 * stats.t.sf(abs(est / se), G - 1) if se > 0 else np.nan
    return Z975, 2 * stats.norm.sf(abs(est / se)) if se > 0 else np.nan


# --- prediction, g-computation margins with gradients ------------------------
def pred_part(core, s, dpop, over=None):
    b = core["blocks"][s]
    dm = apply_cmap(dpop, b["cmap"])
    for k, v in (over or {}).items():
        dm[k] = v
    X = apply_spec(dm, b["spec"])
    return np.exp(np.clip(X @ core["beta"][b["cols"]], -30, 30)), X


def margin(core, s, dpop, over=None):
    mu, X = pred_part(core, s, dpop, over)
    g = np.zeros(core["K"])
    g[core["blocks"][s]["cols"]] = (mu @ X) / len(mu)
    return float(mu.mean()), g


def predict_rows(core, dte):
    mu = np.full(len(dte), np.nan)
    for s, b in core["blocks"].items():
        m = (dte["stratum"] == s).to_numpy() if core["mode"] == "stratified" else np.ones(len(dte), bool)
        if m.any():
            mu[m] = pred_part(core, s, dte[m])[0]
    return mu


def est_ci(core, est, grad, as_ratio=True):
    rows = {}
    for cl in core["covs"]:
        cov = core["covs"][cl][0]
        se = float(np.sqrt(max(grad @ cov @ grad, 0)))
        c, p = crit_p(core, cl, est, se)
        rows[cl] = (se, c, p)
    return rows


def support_share(core, s, dpop):
    b = core["blocks"][s]
    ok = np.ones(len(dpop), bool)
    for v in COLLAPSE_VARS:
        if v in b["cmap"]["map"] and v not in ("period", "sex", "age_group"):
            m = b["cmap"]["map"][v]
            ok &= ~dpop[v].astype(str).isin(m.keys()).to_numpy()
    return float(ok.mean())


# =============================================================================
# 3. DESCRIPTIVES
# =============================================================================
hdr("2. DESCRIPTIVES (cost, Int$)")


def cstats(s):
    s = s.dropna()
    m = s.mean()
    return {"n": len(s), "mean": m, "sd": s.std(), "median": s.median(), "p25": s.quantile(.25),
            "p75": s.quantile(.75), "p5": s.quantile(.05), "p95": s.quantile(.95), "p99": s.quantile(.99),
            "max": s.max(), "cv": s.std() / m if m > 0 else np.nan,
            "median_IQR": f"{s.median():,.0f} ({s.quantile(.25):,.0f}-{s.quantile(.75):,.0f})"}


def describe(d, var, label):
    rows = []
    for lev, g in ([("All", d)] if var is None else d.groupby(var, observed=True)):
        rows.append({"variable": label, "level": lev, **cstats(g["cost_intl"])})
    return rows


desc = describe(DALL[valid], None, "All valid episodes (protocol sample)")
desc += describe(DALL[valid], "record_type", "Record type (all valid episodes)")
desc += describe(D, None, "PRIMARY SAMPLE")
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("billing_mode", "Billing mode"),
                 ("specialty_model", "Specialty"), ("province_model", "Province"), ("ownership", "Ownership"),
                 ("period", "Period (Jalali year-month)")]:
    desc += describe(D, var, lab)
D["grp_x_bill"] = D["group"] + " | " + D["billing_mode"]
desc += describe(D, "grp_x_bill", "Group x billing mode")
DESC = pd.DataFrame(desc)
save(DESC, "o2_descriptives.csv")
print(DESC[DESC.variable.isin(["All valid episodes (protocol sample)", "Record type (all valid episodes)",
                               "PRIMARY SAMPLE", "Group", "Billing mode", "Group x billing mode"])]
      [["variable", "level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
print("\nBy ICD-10 code:\n", DESC[DESC.variable == "ICD-10 code"][["level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
print("\nBy period:\n", DESC[DESC.variable == "Period (Jalali year-month)"][["level", "n", "median_IQR", "mean"]].round(1).to_string(index=False))


# =============================================================================
# 4. HYPOTHESIS 1: Global vs itemised cost variation within the same code (tariff steps removed)
# =============================================================================
hdr("3. HYPOTHESIS 1: Global vs itemised cost variation, same ICD-10 code")


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj, run = np.empty_like(p), 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i])
        adj[i] = min(run, 1.0)
    return adj


def tariff_stats(x):
    med = x.median()
    return {"n": len(x), "median": med, "mean": x.mean(), "sd": x.std(), "cv": x.std() / x.mean(),
            "sd_log": np.log(x).std(),
            "qcd": (x.quantile(.75) - x.quantile(.25)) / (x.quantile(.75) + x.quantile(.25)),
            "pct_within_10pct_of_median": 100 * ((x - med).abs() <= 0.10 * med).mean(),
            "pct_within_25pct_of_median": 100 * ((x - med).abs() <= 0.25 * med).mean()}


H1_cells, H1_cmp = [], []
for code, g in D.groupby("icd10"):
    gg, oo = g.loc[g.stratum == "Global", "cost_intl"], g.loc[g.stratum == "Other", "cost_intl"]
    for lab, x in [("Global", gg), ("Other", oo)]:
        if len(x) >= 5:
            H1_cells.append({"icd10": code, "billing_mode": lab, **tariff_stats(x)})
    if len(gg) < 10 or len(oo) < 10:
        H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo), "note": "too few in a mode"})
        continue
    lg, lo_ = np.log(gg.to_numpy()), np.log(oo.to_numpy())
    bf = stats.levene(lg, lo_, center="median")
    bs = [np.log(RNG.choice(gg.to_numpy(), len(gg))).std() / np.log(RNG.choice(oo.to_numpy(), len(oo))).std()
          for _ in range(NBOOT)]
    H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo),
                   "sd_log_ratio_G_over_O": lg.std() / lo_.std(), "sd_log_lo": np.percentile(bs, 2.5),
                   "sd_log_hi": np.percentile(bs, 97.5), "cv_ratio_G_over_O": (gg.std() / gg.mean()) / (oo.std() / oo.mean()),
                   "median_ratio_G_over_O": gg.median() / oo.median(), "brown_forsythe_p": float(bf.pvalue), "note": ""})
H1c = pd.DataFrame(H1_cmp)
ok = H1c["brown_forsythe_p"].notna()
H1c.loc[ok, "p_holm"] = holm(H1c.loc[ok, "brown_forsythe_p"])
H1c["Global_less_variable"] = (H1c["sd_log_ratio_G_over_O"] < 1) & (H1c["p_holm"] < 0.05)
save(pd.DataFrame(H1_cells), "o2_h1_cost_by_code_and_billing.csv")
save(H1c, "o2_h1_variation_by_code.csv")
print(pd.DataFrame(H1_cells).round(2).to_string(index=False))
print("\nPer code (ratio of SD(log cost), Global/itemised; < 1 = Global less variable). The itemised cost also moves "
      "with the nominal tariff, so see the adjusted comparison below:")
print(H1c.round(3).to_string(index=False))


def pooled_within(lc, cell):
    n = np.bincount(cell)
    m = np.bincount(cell, lc) / np.maximum(n, 1)
    ss = float(np.sum((lc - m[cell]) ** 2))
    df = len(lc) - np.count_nonzero(n)
    tot = float(np.sum((lc - lc.mean()) ** 2))
    return np.sqrt(ss / max(df, 1)), 1 - ss / tot


h1a = []
for lab, keys in [("ICD-10 code", ["icd10"]), ("ICD-10 x period", ["icd10", "period"]),
                  ("ICD-10 x period x province", ["icd10", "period", "province_model"])]:
    cellid = D[keys].astype(str).agg("|".join, axis=1)
    cnt = pd.crosstab(cellid, D.stratum)
    good = cnt.index[(cnt["Global"] >= 5) & (cnt["Other"] >= 5)]
    sub = D[cellid.isin(good)]
    if len(sub) < 100:
        continue
    cid = pd.factorize(cellid[cellid.isin(good)])[0]
    lc = np.log(sub.cost_intl.to_numpy())
    res = {}
    for s in ["Global", "Other"]:
        m = (sub.stratum == s).to_numpy()
        res[s] = pooled_within(lc[m], pd.factorize(cid[m])[0])
    bs = []
    for _ in range(NBOOT):
        r = {}
        for s in ["Global", "Other"]:
            idx = np.where((sub.stratum == s).to_numpy())[0]
            idx = RNG.choice(idx, len(idx))
            r[s] = pooled_within(lc[idx], pd.factorize(cid[idx])[0])[0]
        bs.append(r["Global"] / r["Other"])
    sub_s = sub.assign(_c=cellid[cellid.isin(good)])
    pct = {}
    for s in ["Global", "Other"]:
        x = sub_s[sub_s.stratum == s]
        med = x.groupby("_c").cost_intl.transform("median")
        pct[s] = (100 * ((x.cost_intl - med).abs() <= .10 * med).mean(), 100 * ((x.cost_intl - med).abs() <= .25 * med).mean())
    h1a.append({"cells": lab, "n_cells": len(good), "n_Global": int((sub.stratum == "Global").sum()),
                "n_Other": int((sub.stratum == "Other").sum()),
                "within_cell_SD_log_Global": res["Global"][0], "within_cell_SD_log_Other": res["Other"][0],
                "SD_ratio_G_over_O": res["Global"][0] / res["Other"][0],
                "ratio_lo": np.percentile(bs, 2.5), "ratio_hi": np.percentile(bs, 97.5),
                "R2_cells_Global": res["Global"][1], "R2_cells_Other": res["Other"][1],
                "pct_within10_Global": pct["Global"][0], "pct_within10_Other": pct["Other"][0],
                "pct_within25_Global": pct["Global"][1], "pct_within25_Other": pct["Other"][1]})
H1A = pd.DataFrame(h1a)
H1A["supports_H1"] = H1A["ratio_hi"] < 1
save(H1A, "o2_h1_adjusted_variation.csv")
print("\nPooled within-cell variation (cells with >= 5 episodes in BOTH modes; bootstrap CI):")
print(H1A.round(3).to_string(index=False))
print("Reading: after removing code and period (tariff steps), the Global package varies less than itemised billing "
      "if the SD ratio and its CI are below 1. The R2 columns show how much of the Global cost variance is a tariff cell.")


# =============================================================================
# 5. MODELS M1-M5 : pooled additive (comparison) and stratified (primary)
# =============================================================================
hdr("4. MODELS M1-M5 (Gamma-log; patient and province clusters)")
CATS_ALL = BLOCKS["M5"] + ["billing_mode"]


def blk_cats(m, pooled):
    c = list(BLOCKS[m])
    return c + ["billing_mode"] if pooled and m != "M1" else c


CORE_P, CORE_S, TABS = {}, {}, []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cp = fit_block(D, blk_cats(m, True), NUMS[m], "pooled")
    cs = fit_block(D, blk_cats(m, False), NUMS[m], "stratified")
    CORE_P[m], CORE_S[m] = cp, cs
    TABS += [make_tab(cp, m).assign(spec="pooled additive (comparison only)"),
             make_tab(cs, m).assign(spec="stratified by billing mode (PRIMARY)")]
    print(f"  {m}: pooled terms {len(cp['names']) - 1}, dispersion {cp['phi']['All']:.3f} | stratified terms "
          f"{len(cs['names']) - len(cs['blocks'])}, dispersion " + ", ".join(f"{s} {v:.3f}" for s, v in cs["phi"].items()))
LONG = pd.concat(TABS, ignore_index=True)
save(LONG, "o2_models_long.csv")
LONG_S = LONG[LONG.spec.str.contains("PRIMARY")]
M5S, M5P = CORE_S["M5"], CORE_P["M5"]
prov_n = D.province_model.value_counts()
top_prov = prov_n.index[:TOP_PROVINCES].tolist()


def fmt(r):
    star = " *" if hit(r) else ""
    sp = " [n<30]" if r.sparse_flag else ""
    return f"{r.ratio:.2f} ({r.lo:.2f}-{r.hi:.2f}){star}{sp}"


def main_table(core, model):
    t = make_tab(core, model)
    rows = []
    for s in ["Global", "Other"]:
        for _, r in t[(t.stratum == s) & (t.cluster == "patient")].iterrows():
            if not (r.term.startswith(("C(group)", "C(any_surgical", "C(ownership", "C(specialty_model")) or
                    r.term == "los_cov" or any(r.term.endswith(f"[T.{p}]") and "province" in r.term for p in top_prov)):
                continue
            q = t[(t.stratum == s) & (t.cluster == "province") & (t.term == r.term)].iloc[0]
            rows.append({"stratum": s, "term": r.term, "level_n": r.level_n, "ratio": r.ratio, "lo_patient": r.lo,
                         "hi_patient": r.hi, "p_patient": r.p, "lo_province": q.lo, "hi_province": q.hi,
                         "p_province": q.p, "patient_CI": fmt(r), "province_CI": fmt(q),
                         "sparse_flag": r.sparse_flag})
    return pd.DataFrame(rows)


MAIN = main_table(M5S, "M5")
save(MAIN, "o2_main_table_M5_stratified.csv")
print("\nMAIN TABLE (stratified M5): cost ratio (95% CI) with patient clusters and with province clusters (t, G-1 df).\n"
      "* = CI excludes 1. Province-cluster CIs are the more honest ones for facility-level terms (ownership, specialty,\n"
      "province); province terms themselves have no valid province-clustered SE.")
print(MAIN[["stratum", "term", "level_n", "patient_CI", "province_CI"]].to_string(index=False))

# pooled additive billing-mode ratio (v1 type), for comparison with the stratified standardised ratio
pb = LONG[(LONG.spec.str.contains("pooled")) & (LONG.cluster == "patient") & (LONG.term.str.contains("billing_mode"))]
print("\nPooled ADDITIVE billing-mode ratio (Global vs itemised) by block [v1-type estimate; assumes one constant ratio]:")
print(pb[["model", "term", "ratio", "lo", "hi"]].round(3).to_string(index=False))
save(pb, "o2_pooled_billing_ratio_by_block.csv")


# =============================================================================
# 6. AVERAGE MARGINAL COSTS + STANDARDISED GLOBAL-vs-ITEMISED CONTRAST
# =============================================================================
hdr("5. AVERAGE MARGINAL COSTS (g-computation from the stratified model)")


def amc_tables(core, d, model):
    own, att = [], []
    groups = ["All"] + sorted(d.group.unique())
    for s in ["Global", "Other"]:
        pop_s = d[d.stratum == s]
        for g in groups:
            over = None if g == "All" else {"group": g}
            pop = pop_s if g == "All" else pop_s[pop_s.group == g]
            if len(pop) < 20:
                continue
            m, gr = margin(core, s, pop)                                  # fitted mean in the observed group
            lm, lg = np.log(m), gr / m
            ci = est_ci(core, lm, lg)
            row = {"model": model, "stratum": s, "group": g, "n": len(pop), "observed_mean": pop.cost_intl.mean(),
                   "observed_median": pop.cost_intl.median(), "AMC": m, "obs_over_AMC": pop.cost_intl.mean() / m}
            for cl, (se, c, p) in ci.items():
                row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(lm - c * se)), float(np.exp(lm + c * se))
            own.append(row)
        # group counterfactual (everyone in the stratum set to group g): standardised AMC
        if True:
            for g in groups[1:]:
                m, gr = margin(core, s, pop_s, {"group": g})
                ci = est_ci(core, np.log(m), gr / m)
                row = {"model": model, "stratum": s, "group": f"standardised: all {s} episodes as {g}", "n": len(pop_s),
                       "observed_mean": np.nan, "observed_median": np.nan, "AMC": m, "obs_over_AMC": np.nan}
                for cl, (se, c, p) in ci.items():
                    row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(np.log(m) - c * se)), float(np.exp(np.log(m) + c * se))
                own.append(row)
    popG = d[d.stratum == "Global"]
    for g in groups:
        pop = popG if g == "All" else popG[popG.group == g]
        if len(pop) < 20:
            continue
        m1, g1 = margin(core, "Global", pop)
        m2, g2 = margin(core, "Other", pop)
        lr, glr = np.log(m1 / m2), g1 / m1 - g2 / m2
        cir = est_ci(core, lr, glr)
        cid = est_ci(core, m1 - m2, g1 - g2)
        row = {"model": model, "population": f"Global episodes ({g})", "n_Global": len(pop),
               "share_on_common_support": support_share(core, "Other", pop), "observed_Global_mean": pop.cost_intl.mean(),
               "fitted_Global_mean": m1, "if_itemised_mean": m2, "ratio_Global_over_itemised": m1 / m2,
               "diff_Int$": m1 - m2}
        for cl, (se, c, p) in cir.items():
            row[f"ratio_lo_{cl}"], row[f"ratio_hi_{cl}"], row[f"ratio_p_{cl}"] = \
                float(np.exp(lr - c * se)), float(np.exp(lr + c * se)), p
        for cl, (se, c, p) in cid.items():
            row[f"diff_lo_{cl}"], row[f"diff_hi_{cl}"] = (m1 - m2) - c * se, (m1 - m2) + c * se
        att.append(row)
    return pd.DataFrame(own), pd.DataFrame(att)


# The marginal costs (means) come from the quasi-Poisson-log fit: its score equations make the fitted mean equal the
# observed mean in every cell of the categorical covariates, so it is the mean-preserving estimator of expenditure.
# The Gamma-log fit (used for the cost RATIOS, as in the protocol) over-weights low-cost episodes in the heavy-tailed
# itemised stratum and misses the observed itemised Gallbladder mean by ~13%; its AMCs are saved for comparison.
M5Q = fit_block(D, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp")
AMC, ATT = amc_tables(M5Q, D, "M5 quasi-Poisson-log (primary for means)")
AMC_G, ATT_G = amc_tables(M5S, D, "M5 Gamma-log (comparison)")


def boot_ci(d, B, key):
    """Cluster bootstrap (whole refit) for the AMCs and the Global/itemised ratio. The delta-method CI above is
    conditional on the observed covariate mix, so it is too narrow for population means (it is ~0 for the stratum
    mean under the quasi-Poisson fit); the bootstrap includes that sampling variation."""
    cl = d[key].astype(str).to_numpy()
    uc, inv = np.unique(cl, return_inverse=True)
    idx_by = [np.where(inv == i)[0] for i in range(len(uc))]
    ra, rt = [], []
    for _ in range(B):
        pick = RNG.integers(0, len(uc), len(uc))
        db = d.iloc[np.concatenate([idx_by[i] for i in pick])].reset_index(drop=True)
        try:
            cb = fit_block(db, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp", cov=False)
            a_, t_ = amc_tables(cb, db, "b")
        except Exception:
            continue
        ra.append(a_.set_index(["stratum", "group"]).AMC)
        rt.append(t_.set_index("population")[["ratio_Global_over_itemised", "diff_Int$"]])
    ra, rt = pd.concat(ra, axis=1), pd.concat(rt, axis=1)
    return ra, rt


NB = min(NBOOT, 200)
for key, tag in [("cluster_id", "patient"), ("province_model", "province")]:
    ra, rt = boot_ci(D, NB, key)
    q = ra.quantile([.025, .975], axis=1).T
    AMC[f"lo_boot_{tag}"] = [q.loc[(r.stratum, r.group), 0.025] if (r.stratum, r.group) in q.index else np.nan for r in AMC.itertuples()]
    AMC[f"hi_boot_{tag}"] = [q.loc[(r.stratum, r.group), 0.975] if (r.stratum, r.group) in q.index else np.nan for r in AMC.itertuples()]
    ratio = pd.concat([rt.iloc[:, 0::2]], axis=1) if False else rt
    rr = rt["ratio_Global_over_itemised"]
    dd = rt["diff_Int$"]
    qr, qd = rr.quantile([.025, .975], axis=1).T, dd.quantile([.025, .975], axis=1).T
    ATT[f"ratio_lo_boot_{tag}"] = [qr.loc[p_, 0.025] for p_ in ATT.population]
    ATT[f"ratio_hi_boot_{tag}"] = [qr.loc[p_, 0.975] for p_ in ATT.population]
    ATT[f"diff_lo_boot_{tag}"] = [qd.loc[p_, 0.025] for p_ in ATT.population]
    ATT[f"diff_hi_boot_{tag}"] = [qd.loc[p_, 0.975] for p_ in ATT.population]
# the bootstrap CIs replace the delta-method CIs as the reported ones (delta columns are kept, suffix _delta)
for tag in ["patient", "province"]:
    AMC = AMC.rename(columns={f"lo_{tag}": f"lo_{tag}_delta", f"hi_{tag}": f"hi_{tag}_delta"})
    ATT = ATT.rename(columns={f"ratio_lo_{tag}": f"ratio_lo_{tag}_delta", f"ratio_hi_{tag}": f"ratio_hi_{tag}_delta"})
    AMC[f"lo_{tag}"], AMC[f"hi_{tag}"] = AMC[f"lo_boot_{tag}"], AMC[f"hi_boot_{tag}"]
    ATT[f"ratio_lo_{tag}"], ATT[f"ratio_hi_{tag}"] = ATT[f"ratio_lo_boot_{tag}"], ATT[f"ratio_hi_boot_{tag}"]

save(AMC, "o2_amc_stratified_M5.csv")
save(ATT, "o2_global_vs_itemised_standardised_M5.csv")
save(AMC_G, "o2_amc_stratified_M5_gamma.csv")
save(ATT_G, "o2_global_vs_itemised_standardised_M5_gamma.csv")
print("Average marginal cost (Int$) by stratum and group, with the observed mean (obs_over_AMC should be ~ 1):")
print(AMC[AMC.n.notna()][["stratum", "group", "n", "observed_mean", "AMC", "lo_patient", "hi_patient", "lo_province",
                          "hi_province", "obs_over_AMC"]].round(2).to_string(index=False))
print("\nStandardised Global-vs-itemised contrast on the Global episodes (what they would have cost if billed itemised):")
print(ATT[["population", "n_Global", "share_on_common_support", "observed_Global_mean", "if_itemised_mean",
           "ratio_Global_over_itemised", "ratio_lo_patient", "ratio_hi_patient", "ratio_lo_province", "ratio_hi_province"]]
      .round(3).to_string(index=False))
ATT_BY_BLOCK = []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    a = amc_tables(fit_block(D, blk_cats(m, False), NUMS[m], "stratified", fam="qp"), D, m)[1]
    ATT_BY_BLOCK.append(a[a.population == "Global episodes (All)"])
ATT_BY_BLOCK = pd.concat(ATT_BY_BLOCK)
save(ATT_BY_BLOCK, "o2_global_vs_itemised_by_block.csv")
print("\nGlobal/itemised ratio across blocks (stratified quasi-Poisson models):")
print(ATT_BY_BLOCK[["model", "fitted_Global_mean", "if_itemised_mean", "ratio_Global_over_itemised",
                    "ratio_lo_patient", "ratio_hi_patient"]].round(3).to_string(index=False))

# check of the v1 additive AMC against observed cells
chk = []
for s in ["Global", "Other"]:
    for g in sorted(D.group.unique()):
        pop = D[(D.stratum == s) & (D.group == g)]
        mu_p = pred_part(M5P, "All", pop)[0].mean()
        a = AMC_G[(AMC.stratum == s) & (AMC.group == g)]
        chk.append({"stratum": s, "group": g, "n": len(pop), "observed_mean": pop.cost_intl.mean(),
                    "pooled_additive_fitted": mu_p, "stratified_gamma_fitted": float(a.AMC.iloc[0]),
                    "stratified_qp_fitted": float(AMC[(AMC.stratum == s) & (AMC.group == g)].AMC.iloc[0]),
                    "pooled_error_pct": 100 * (mu_p / pop.cost_intl.mean() - 1),
                    "stratified_gamma_error_pct": 100 * (float(a.AMC.iloc[0]) / pop.cost_intl.mean() - 1),
                    "stratified_qp_error_pct": 100 * (float(AMC[(AMC.stratum == s) & (AMC.group == g)].AMC.iloc[0]) / pop.cost_intl.mean() - 1)})
CHK = pd.DataFrame(chk)
save(CHK, "o2_amc_fit_to_observed_pooled_vs_stratified.csv")
print("\nFitted vs observed cell means (quasi-Poisson reproduces them by construction, Gamma-log and the additive model do not):")
print(CHK.round(2).to_string(index=False))


# =============================================================================
# 7. DIAGNOSTICS, WITHIN EACH STRATUM
# =============================================================================
hdr("6. DIAGNOSTICS (Park, link, Cook's distance + refit, quasi-Poisson), within strata")


def ols_cluster(x, y, groups):
    Xo = np.column_stack([np.ones(len(x)), x])
    b = np.linalg.solve(Xo.T @ Xo, Xo.T @ y)
    e = y - Xo @ b
    meat, g = cluster_meat(Xo * e[:, None], groups)
    bread = np.linalg.inv(Xo.T @ Xo)
    V = (g / (g - 1)) * bread @ meat @ bread
    return b, np.sqrt(np.diag(V)), g


diag, cook_tabs = [], []
cook_flag_all = np.zeros(len(M5S["d"]), bool)
for s, b in M5S["blocks"].items():
    r = b["rows"]
    Xs, ys, mu, eta = b["X"], M5S["y"][r], M5S["mu"][r], M5S["eta"][r]
    pat = M5S["groups"]["patient"][r]
    e = ys - mu
    ok = e != 0
    (a0, a1), (s0, s1), g = ols_cluster(np.log(mu[ok]), np.log(e[ok] ** 2), pat[ok])
    # link test
    Xl = np.column_stack([Xs, eta ** 2 - 0])
    fl = fit_glm(Xl, ys)
    covl, _ = cov_robust(Xl, ys, fl, pat)
    tl = fl["beta"][-1] / np.sqrt(covl[-1, -1])
    # Cook's distance (Gamma, working weight 1)
    brd = np.linalg.pinv(Xs.T @ Xs)
    h = np.einsum("ij,jk,ik->i", Xs, brd, Xs)
    rp = (ys - mu) / mu
    k = Xs.shape[1]
    cook = rp ** 2 * h / (k * M5S["phi"][s] * (1 - h) ** 2)
    flag = cook > COOK_MULT / len(r)
    cook_flag_all[r[flag]] = True
    diag.append({"stratum": s, "n": len(r), "terms": k, "dispersion": M5S["phi"][s],
                 "park_slope": a1, "park_se": s1, "park_reading": ("Gamma (slope ~2)" if abs(a1 - 2) < 2 * s1 else
                                                                  ("closer to inverse Gaussian (3)" if a1 > 2 else "closer to Poisson-like (1)")),
                 "link_t": float(tl), "link_p": float(2 * stats.norm.sf(abs(tl))), "max_cook": float(cook.max()),
                 "n_cook_flag": int(flag.sum()), "max_leverage": float(h.max()), "n_leverage_gt_0.2": int((h > .2).sum())})
DIAG = pd.DataFrame(diag)
save(DIAG, "o2_diagnostics_by_stratum.csv")
print(DIAG.round(3).to_string(index=False))
print("Park slope within each stratum guides the variance function (2 = Gamma). Robust (sandwich) SEs are used throughout,\n"
      "so a moderate departure widens nothing; the quasi-Poisson refit below shows whether it matters for the estimates.")

dref = M5S["d"][~cook_flag_all]
M5C = fit_block(dref, BLOCKS["M5"], NUMS["M5"], "stratified")
tm, tc, tq = make_tab(M5S, "M5"), make_tab(M5C, "M5 cook-excl"), make_tab(M5Q, "M5 quasi-Poisson")
tm = tm[tm.cluster == "patient"].set_index("term_full")
tc, tq = tc[tc.cluster == "patient"].set_index("term_full"), tq[tq.cluster == "patient"].set_index("term_full")
ROB = tm[["stratum", "term", "level_n", "ratio", "lo", "hi"]].copy()
ROB["ratio_cook_excl"] = tc["ratio"].reindex(ROB.index)
ROB["ratio_quasiPoisson"] = tq["ratio"].reindex(ROB.index)
ROB["cook_pct_change"] = 100 * (ROB.ratio_cook_excl / ROB.ratio - 1)
ROB["qp_pct_change"] = 100 * (ROB.ratio_quasiPoisson / ROB.ratio - 1)
ROB = ROB.reset_index(drop=True)
save(ROB, "o2_robustness_cook_quasipoisson.csv")
print(f"\nCook refit excludes {int(cook_flag_all.sum())} episodes (> {COOK_MULT:g}/n within stratum). Key terms:")
kt = ROB[ROB.term.str.contains(KEY_RE)]
print(kt.round(3).to_string(index=False))
# AMC/ATT effect of the influential episodes
attC = amc_tables(M5C, M5C["d"], "M5 cook-excl")[1]
attQ = ATT_G
print("\nGlobal/itemised ratio: quasi-Poisson (primary)", ATT.ratio_Global_over_itemised.iloc[0].round(3), "| Gamma-log",
      ATT_G.ratio_Global_over_itemised.iloc[0].round(3), "| Gamma, Cook-excluded", attC.ratio_Global_over_itemised.iloc[0].round(3))


# =============================================================================
# 8. FIVE-FOLD OUT-OF-FOLD PREDICTION ERROR
# =============================================================================
hdr("7. OUT-OF-FOLD PREDICTION (5 folds, grouped by patient)")
ids = pd.factorize(D.cluster_id)[0]
uid = np.unique(ids)
RNG2 = np.random.default_rng(SEED)
RNG2.shuffle(uid)
fm = dict(zip(uid, np.arange(len(uid)) % N_FOLDS))
FOLD = np.array([fm[i] for i in ids])
oof_rows, cal_rows = [], []
for m in ["M1", "M3", "M5"]:
    for mode in ["pooled", "stratified", "stratified_qp"]:
        pred = np.full(len(D), np.nan)
        for k in range(N_FOLDS):
            tr, te = D[FOLD != k], D[FOLD == k]
            core = fit_block(tr, blk_cats(m, mode == "pooled"), NUMS[m], mode.replace("_qp", ""), fam="qp" if mode.endswith("qp") else "gamma", cov=False)
            p = predict_rows(core, te)
            lo_, hi_ = tr.cost_intl.min(), tr.cost_intl.max()
            pred[(FOLD == k)] = np.clip(p, lo_, hi_)
        y = D.cost_intl.to_numpy()
        for s in ["All", "Global", "Other"]:
            mk = np.ones(len(D), bool) if s == "All" else (D.stratum == s).to_numpy()
            err = y[mk] - pred[mk]
            oof_rows.append({"model": m, "spec": mode, "stratum": s, "n": int(mk.sum()), "MAE": np.abs(err).mean(),
                             "RMSE": np.sqrt((err ** 2).mean()), "MAE_log": np.abs(np.log(y[mk]) - np.log(pred[mk])).mean(),
                             "mean_observed": y[mk].mean(), "mean_predicted": pred[mk].mean(),
                             "calibration_ratio": pred[mk].mean() / y[mk].mean(),
                             "relative_MAE_pct": 100 * np.abs(err).mean() / y[mk].mean()})
            if m == "M5" and s != "All" or (m == "M5" and s == "All"):
                q = pd.qcut(pd.Series(pred[mk]).rank(method="first"), 10, labels=False)
                c = pd.DataFrame({"decile": q + 1, "pred": pred[mk], "obs": y[mk]}).groupby("decile").mean().reset_index()
                c["model"], c["spec"], c["stratum"] = m, mode, s
                cal_rows.append(c.rename(columns={"pred": "mean_predicted", "obs": "mean_observed"}))
OOF = pd.DataFrame(oof_rows)
CAL = pd.concat(cal_rows, ignore_index=True)
save(OOF, "o2_oof_error.csv")
save(CAL, "o2_oof_calibration_deciles.csv")
print(OOF.round(3).to_string(index=False))
c5 = CAL[(CAL.model == "M5") & (CAL.stratum == "All")].copy()
c5["ratio"] = c5.mean_predicted / c5.mean_observed
print("\nM5 calibration by predicted decile, all admissions (ratio = predicted/observed):")
print(c5.pivot(index="decile", columns="spec", values="ratio").round(3).to_string())


# =============================================================================
# 9. CONCENTRATION
# =============================================================================
hdr("8. CONCENTRATION OF EXPENDITURE")


def conc_stats(x):
    x = np.sort(np.asarray(x, float))
    n, tot = len(x), x.sum()
    cum = np.concatenate([[0], np.cumsum(x) / tot])
    p = np.linspace(0, 1, n + 1)
    gini = 1 - 2 * float(np.sum((cum[1:] + cum[:-1]) / 2 * np.diff(p)))
    out = {"n": n, "total_intl": tot, "gini": gini}
    for q in (1, 5, 10, 20):
        out[f"top{q}pct_share"] = 100 * x[-max(int(np.ceil(q / 100 * n)), 1):].sum() / tot
    return out


def conc_boot(x, B):
    x = np.asarray(x, float)
    r = [conc_stats(RNG.choice(x, len(x))) for _ in range(B)]
    return (np.percentile([v["top10pct_share"] for v in r], [2.5, 97.5]), np.percentile([v["gini"] for v in r], [2.5, 97.5]))


crow = []
masks = [("All (primary sample)", pd.Series(True, index=D.index))] + \
        [(f"group={g}", D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())] + \
        [(f"{g} x {b}", (D.group == g) & (D.billing_mode == b)) for g in sorted(D.group.unique()) for b in sorted(D.billing_mode.unique())] + \
        [("All, flagged extremes removed", ~D.flag_extreme_any)]
for lab, mask in masks:
    x = D.loc[mask, "cost_intl"]
    if len(x) < 20:
        continue
    (t_lo, t_hi), (g_lo, g_hi) = conc_boot(x, NBOOT)
    crow.append({"stratum": lab, **conc_stats(x), "top10_lo": t_lo, "top10_hi": t_hi, "gini_lo": g_lo, "gini_hi": g_hi})
CONC = pd.DataFrame(crow)
save(CONC, "o2_concentration.csv")
print(CONC.round(2).to_string(index=False))
lz = []
for lab, mask in [("All", pd.Series(True, index=D.index))] + [(g, D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())]:
    xs = np.sort(D.loc[mask, "cost_intl"].to_numpy())
    cum = np.concatenate([[0], np.cumsum(xs) / xs.sum()])
    pp = np.linspace(0, 1, len(xs) + 1)
    for q in np.linspace(0, 1, 101):
        lz.append({"stratum": lab, "cum_share_episodes": q, "cum_share_cost": float(np.interp(q, pp, cum))})
LZ = pd.DataFrame(lz)
save(LZ, "o2_lorenz_points.csv")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for lab in ["All"] + sorted(D.group.unique()):
        s = LZ[LZ.stratum == lab]
        ax[0].plot(s.cum_share_episodes, s.cum_share_cost, label=lab, lw=2 if lab == "All" else 1.4)
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
    ax[0].set(title="Lorenz curve of episode cost by group", xlabel="Cumulative share of episodes",
              ylabel="Cumulative share of expenditure")
    ax[0].legend(frameon=False)
    for k, spec in enumerate(["pooled", "stratified_qp"]):
        for s, col in [("Global", "C0"), ("Other", "C3")]:
            c8 = CAL[(CAL.model == "M5") & (CAL.stratum == s) & (CAL.spec == spec)]
            ax[1 + k].plot(c8.mean_predicted, c8.mean_observed, "o-", color=col, label=s)
    for a, t in zip(ax[1:], ["M5 pooled additive: OOF calibration", "M5 stratified quasi-Poisson: OOF calibration"]):
        lim = [0, max(CAL.mean_predicted.max(), CAL.mean_observed.max()) * 1.05]
        a.plot(lim, lim, "k--", lw=.8)
        a.set(title=t, xlabel="Mean predicted (Int$)", ylabel="Mean observed (Int$)")
        a.legend(frameon=False, title="billing mode")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o2_lorenz_and_calibration.png", dpi=200)
    plt.close(fig)
    print("Figure saved: o2_lorenz_and_calibration.png")
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")


# =============================================================================
# 10. HYPOTHESES 2 AND 3
# =============================================================================
hdr("9. HYPOTHESIS 2 (provinces) AND HYPOTHESIS 3 (K40: general surgery vs urology)")


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


h2 = []
for m in ["M3", "M4", "M5"]:
    core = CORE_S[m]
    for s, b in core["blocks"].items():
        idx = [b["cols"].start + j for j, nm in enumerate(b["names"]) if nm.startswith("C(province_model)")]
        if m == "M3":
            kw = stats.kruskal(*[g.cost_intl.values for _, g in D[D.stratum == s].groupby("province_model") if len(g) >= 5])
            h2.append({"model": "crude (Kruskal-Wallis)", "stratum": s, "df": np.nan, "chi2": float(kw.statistic),
                       "p": float(kw.pvalue), "note": "unadjusted"})
        if idx and m != "M3":
            chi2, df, p = wald(core, idx)
            prov = make_tab(core, m)
            prov = prov[(prov.stratum == s) & (prov.cluster == "patient") & prov.term.str.startswith("C(province_model)")]
            prov = prov[~prov.sparse_flag]
            h2.append({"model": m, "stratum": s, "df": df, "chi2": chi2, "p": p,
                       "n_provinces_modelled": df + 1, "min_ratio": prov.ratio.min(), "max_ratio": prov.ratio.max(),
                       "note": "Wald, patient-clustered; province-level SE not valid (28 clusters)"})
H2 = pd.DataFrame(h2)
save(H2, "o2_h2_province_tests.csv")
print(H2.round(4).to_string(index=False))
prov_main = make_tab(M5S, "M5")
prov_main = prov_main[prov_main.term.str.startswith("C(province_model)") & (prov_main.cluster == "patient")]
save(prov_main.drop(columns=["term_full"]), "o2_h2_province_ratios_M5.csv")
for s in ["Global", "Other"]:
    pm = prov_main[(prov_main.stratum == s) & ~prov_main.sparse_flag].sort_values("ratio")
    print(f"\n{s}: adjusted province cost ratios (vs the largest province in that stratum), extremes:")
    print(pm[["term", "level_n", "ratio", "lo", "hi"]].iloc[[0, 1, 2, -3, -2, -1]].round(2).to_string(index=False))

SIG = []
for s in ["Global", "Other"]:
    pass  # random-intercept variance components are computed with sensitivity 8 below


def h3_variant(d, which, cl_boot=False):
    cats = ["age_group", "sex", "insurance_fund_model", "ownership", "discharge_status_model", "any_surgical_service"]
    if which == "crude":
        cats = []
    if which == "adjusted_provFE":
        cats = cats + ["province_model"]
    d = d.copy()
    d["spec_h3"] = d["specialty_h3"].where(d["specialty_h3"].isin(["General surgery", "Urology"]), "x")
    d = d[d.spec_h3 != "x"]
    return d, cats + ["spec_h3"], ["los_cov"] if which != "crude" else []


h3 = []
K40 = D[D.icd10 == "K40"].copy()
for s in ["Other", "Global", "Both (pooled, +billing mode)"]:
    sub = K40 if s.startswith("Both") else K40[K40.stratum == s]
    for which in ["crude", "adjusted", "adjusted_provFE"]:
        d3, c3, n3 = h3_variant(sub, which)
        nu = (d3.specialty_h3 == "Urology").sum()
        if nu < 30 or (d3.specialty_h3 == "General surgery").sum() < 30:
            h3.append({"stratum": s, "variant": which, "n": len(d3), "n_urology": int(nu), "note": "too few urology episodes"})
            continue
        if s.startswith("Both"):
            c3 = c3 + ["billing_mode"]
        core = fit_block(d3.assign(stratum="Other"), c3, n3, "pooled")
        tt = make_tab(core, "K40")
        tt = tt[tt.term.str.contains("spec_h3")]
        rp = tt[tt.cluster == "patient"].iloc[0]
        rq = tt[tt.cluster == "province"].iloc[0]
        # province-level cluster bootstrap CI
        provs = d3.province_model.unique()
        bs = []
        grp = {p: d3[d3.province_model == p] for p in provs}
        for _ in range(min(NBOOT, 200)):
            pick = RNG.choice(provs, len(provs))
            db = pd.concat([grp[p] for p in pick], ignore_index=True)
            if db.spec_h3.nunique() < 2:
                continue
            try:
                cb = fit_block(db.assign(stratum="Other"), c3, n3, "pooled", cov=False)
                nmb = [i for i, nm in enumerate(cb["names"]) if "spec_h3" in nm]
                if nmb:
                    bs.append(np.exp(cb["beta"][nmb[0]]))
            except Exception:
                pass
        h3.append({"stratum": s, "variant": which, "n": len(d3), "n_urology": int(nu), "urology_vs_general_ratio": rp.ratio,
                   "lo_patient": rp.lo, "hi_patient": rp.hi, "p_patient": rp.p,
                   "lo_province_t": rq.lo, "hi_province_t": rq.hi, "p_province_t": rq.p,
                   "lo_province_bootstrap": np.percentile(bs, 2.5) if len(bs) > 20 else np.nan,
                   "hi_province_bootstrap": np.percentile(bs, 97.5) if len(bs) > 20 else np.nan, "note": ""})
H3 = pd.DataFrame(h3)
save(H3, "o2_h3_k40_specialty.csv")
print("\nH3 (K40, urology vs general surgery). Urology is concentrated in itemised billing, so the itemised stratum is "
      "the primary comparison; the specialty term is a facility-level exposure, so the province CIs are the honest ones:")
print(H3.round(3).to_string(index=False))


# =============================================================================
# 11. SENSITIVITY ANALYSES 1-8 (stratified M5)
# =============================================================================
hdr("10. SENSITIVITY ANALYSES 1-8 (stratified M5)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"),
        2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"),
        4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS covariate capped at P95", None)}
SL, SATT = pd.DataFrame(), pd.DataFrame()


def key_rows(core, s_id, name):
    t = make_tab(core, "M5")
    t = t[(t.cluster == "patient") & t.term.str.contains(KEY_RE)].copy()
    t["sens"], t["sens_name"] = s_id, name
    return t


if RUN_SENS:
    stabs, satt = [], []
    for s_id, (name, keepcol) in SENS.items():
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        if s_id == 7:
            sub["los_cov"] = sub["los_capped_p95"]
        core = fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified")
        stabs.append(key_rows(core, s_id, name))
        if "Global" in core["blocks"] and "Other" in core["blocks"]:
            coreq = fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp")
            a = amc_tables(coreq, sub.reset_index(drop=True), "M5")[1].assign(sens=s_id, sens_name=name)
            satt.append(a)
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}, strata = {list(core['blocks'])}")
    SL = pd.concat(stabs, ignore_index=True)
    SATT = pd.concat(satt, ignore_index=True) if satt else pd.DataFrame()


class LaplaceGamma:
    """Gamma-log GLMM with a province random intercept, Laplace-approximated ML (as lme4::glmer nAGQ = 1)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def _l(self, eta, nu):
        e = np.exp(-eta)
        l = nu * np.log(nu) - gammaln(nu) + (nu - 1) * np.log(self.y) - nu * self.y * e - nu * eta
        return l, nu * (self.y * e - 1), -nu * self.y * e

    def negll(self, theta):
        beta, sig, nu = theta[:self.k], np.exp(theta[self.k]), np.exp(theta[self.k + 1])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            l, d1, d2 = self._l(xb + u[self.g], nu)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        l, d1, d2 = self._l(xb + u[self.g], nu)
        h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
        return -(l.sum() - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0, nu0):
        th0 = np.concatenate([beta0, [np.log(0.3)], [np.log(nu0)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()
if RUN_SENS8:
    hdr("10b. SENSITIVITY 8: province as a random intercept (Gamma-log, Laplace ML), within each stratum")
    s8, vc8 = [], []
    for s, b in M5S["blocks"].items():
        t0 = time.time()
        try:
            ds = D[D.stratum == s].reset_index(drop=True)
            cats = [v for v in BLOCKS["M5"] if v != "province_model" and not (s == "Global" and v == "any_surgical_service")]
            cats = list(dict.fromkeys(cats + ["period"]))
            cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
            dm = apply_cmap(ds, cm)
            X8, nm8, _ = build_spec(dm, cats, ["los_cov"], {"period": min(dm["period"].unique())})
            y8 = ds.cost_intl.to_numpy(float)
            g8 = pd.factorize(ds.province_model)[0]
            f0 = fit_glm(X8, y8)
            m8 = LaplaceGamma(X8, y8, g8)
            theta = m8.fit(f0["beta"], 1 / max(f0["phi"], 1e-3))
            H = m8.hessian(theta)
            for _ in range(2):
                gr = optimize.approx_fprime(theta, m8.negll, 1e-6)
                try:
                    cand = theta - np.linalg.solve(H, gr)
                except np.linalg.LinAlgError:
                    break
                if m8.negll(cand) <= m8.negll(theta) + 1e-9:
                    theta = cand
                    H = m8.hessian(theta)
            se8 = np.sqrt(np.clip(np.diag(np.linalg.pinv(H)), 0, None))
            k8 = m8.k
            t8 = pd.DataFrame({"outcome": "cost", "model": "M5 random intercept", "stratum": s, "term": nm8[1:], "n": len(ds),
                               "level_n": [int(X8[:, j].sum()) if nm8[j].startswith("C(") else np.nan for j in range(1, k8)],
                               "ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                               "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]), "sens": 8,
                               "sens_name": "Province random intercept"})
            s8.append(t8)
            sig = float(np.exp(theta[k8]))
            vc8.append({"stratum": s, "n": len(ds), "provinces": m8.G, "sigma_u": sig,
                        "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])),
                        "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                        "province_cost_ratio_1sd": float(np.exp(sig)),
                        "gamma_dispersion": float(1 / np.exp(theta[k8 + 1]))})
            print(f"  {s}: n = {len(ds):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (a province 1 SD above the mean costs "
                  f"x{np.exp(sig):.2f}), {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  Sensitivity 8 ({s}) FAILED ({type(e).__name__}: {e})")
    if s8:
        S8, VC8 = pd.concat(s8, ignore_index=True), pd.DataFrame(vc8)
        save(S8, "o2_sens8_random_intercept_long.csv")
        save(VC8, "o2_sens8_variance_components.csv")
        print(VC8.round(3).to_string(index=False))
        SL = pd.concat([SL, S8[S8.term.str.contains(KEY_RE)]], ignore_index=True)
if len(SL):
    SL["analysis_status"] = "pre-specified sensitivity analyses, re-run on the post-hoc stratified specification"
    save(SL, "o2_sensitivity_long.csv")
if len(SATT):
    save(SATT, "o2_sensitivity_global_vs_itemised.csv")


# =============================================================================
# 12. EXPLORATORY
# =============================================================================
hdr("11. EXPLORATORY (label as such)")
if RUN_EXTRA and len(ALT):
    try:
        core0 = fit_block(ALT, BLOCKS["M5"], NUMS["M5"], "stratified")
        x0 = make_tab(core0, "X0 " + ALT_LABEL)
        save(x0.drop(columns=["term_full"]), "o2_X0_other_sample_M5.csv")
        k0 = x0[(x0.cluster == "patient") & x0.term.str.contains(KEY_RE)]
        print(f"X0: stratified M5 on {ALT_LABEL} (n = {len(ALT):,}); key terms:")
        print(k0[["stratum", "term", "ratio", "lo", "hi"]].round(3).to_string(index=False))
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    # leave-one-province-out on the Global/itemised ratio and group ratios
    lop = []
    for p in top_prov:
        sub = D[D.province_model != p]
        a = amc_tables(fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp"), sub.reset_index(drop=True), "M5")[1]
        lop.append({"province_left_out": p, "n": len(sub), "ratio_Global_over_itemised": a.ratio_Global_over_itemised.iloc[0]})
    LOP = pd.DataFrame(lop)
    save(LOP, "o2_X_leave_one_province_out.csv")
    print("\nLeave-one-province-out (largest provinces): Global/itemised standardised ratio")
    print(LOP.round(3).to_string(index=False))


# =============================================================================
# 13. RELIABILITY VERDICT
# =============================================================================
hdr("12. RELIABILITY VERDICT (A / B / C) for each stratified M5 estimate")
rows = []
base = make_tab(M5S, "M5")
bp = base[(base.cluster == "patient") & base.term.str.contains(KEY_RE)]
bq = base[(base.cluster == "province")].set_index("term_full")
for _, r in bp.iterrows():
    s = SL[(SL.stratum == r.stratum) & (SL.term == r.term)] if len(SL) else pd.DataFrame()
    s = s[s.sens.notna()] if len(s) else s
    if "stratum" not in SL.columns or not len(s):
        s = pd.DataFrame(columns=["ratio"])
    dev = 100 * np.abs(s.ratio / r.ratio - 1) if len(s) else pd.Series(dtype=float)
    same_dir = bool(((s.ratio > 1) == (r.ratio > 1)).all()) if len(s) else True
    rb = ROB[(ROB.stratum == r.stratum) & (ROB.term == r.term)]
    cook_ch = float(abs(rb.cook_pct_change.iloc[0])) if len(rb) else np.nan
    qp_ch = float(abs(rb.qp_pct_change.iloc[0])) if len(rb) else np.nan
    q = bq.loc[r.term_full]
    sig_p, sig_v = hit(r), hit(q)
    reasons = []
    if r.sparse_flag:
        reasons.append("level has <30 episodes")
    if not same_dir:
        reasons.append("direction changes in a sensitivity analysis")
    if len(dev) and dev.max() > STABLE_PCT:
        reasons.append(f"changes up to {dev.max():.0f}% across sensitivity analyses")
    if cook_ch > COOK_PCT:
        reasons.append(f"Cook refit changes it by {cook_ch:.0f}%")
    if qp_ch > COOK_PCT:
        reasons.append(f"quasi-Poisson changes it by {qp_ch:.0f}%")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    verdict = "C" if (r.sparse_flag or not same_dir) else ("A" if not reasons else "B")
    rows.append({"stratum": r.stratum, "term": r.term, "ratio": r.ratio, "lo": r.lo, "hi": r.hi, "CI_excludes_1": sig_p,
                 "level_n": r.level_n, "n_sens": len(s), "max_pct_change_sens": float(dev.max()) if len(dev) else np.nan,
                 "same_direction_all": same_dir, "cook_pct": cook_ch, "qp_pct": qp_ch,
                 "province_cluster_CI_excludes_1": sig_v, "verdict": verdict, "reasons": "; ".join(reasons)})
# the standardised Global vs itemised contrast
for _, r in ATT.iterrows():
    pop = r.population
    s = SATT[SATT.population == pop] if len(SATT) else pd.DataFrame()
    dev = 100 * np.abs(s.ratio_Global_over_itemised / r.ratio_Global_over_itemised - 1) if len(s) else pd.Series(dtype=float)
    same = bool(((s.ratio_Global_over_itemised > 1) == (r.ratio_Global_over_itemised > 1)).all()) if len(s) else True
    ratio_hit = lambda lo, hi: bool(lo > 1 or hi < 1)
    sp, sv = ratio_hit(r.ratio_lo_patient, r.ratio_hi_patient), ratio_hit(r.ratio_lo_province, r.ratio_hi_province)
    reasons = []
    if r.share_on_common_support < 0.9:
        reasons.append(f"only {100 * r.share_on_common_support:.0f}% of Global episodes on common support")
    if not same:
        reasons.append("direction changes")
    if len(dev) and dev.max() > STABLE_PCT:
        reasons.append(f"changes up to {dev.max():.0f}% across sensitivity analyses")
    if sp != sv:
        reasons.append("conclusion differs between patient and province clusters")
    rows.append({"stratum": "Global vs itemised (ATT)", "term": pop, "ratio": r.ratio_Global_over_itemised,
                 "lo": r.ratio_lo_patient, "hi": r.ratio_hi_patient, "CI_excludes_1": sp, "level_n": r.n_Global,
                 "n_sens": len(s), "max_pct_change_sens": float(dev.max()) if len(dev) else np.nan,
                 "same_direction_all": same, "cook_pct": np.nan, "qp_pct": np.nan, "province_cluster_CI_excludes_1": sv,
                 "verdict": "C" if not same else ("A" if not reasons else "B"), "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o2_reliability_verdict.csv")
print("A = stable under every check; B = direction stable but size or inference depends on the check named; "
      "C = sparse level or direction not stable (do not interpret).")
print(REL[["stratum", "term", "ratio", "lo", "hi", "verdict", "reasons"]].round(2).to_string(index=False))
print("\nVerdict counts:", REL.verdict.value_counts().to_dict())


# =============================================================================
# 14. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)),
        "n_Global": int((D.stratum == "Global").sum()), "n_Other": int((D.stratum == "Other").sum()),
        "rare_min": RARE_MIN, "boot": NBOOT, "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "deviations_from_protocol": ["admissions are the primary sample", "model stratified by billing mode",
                                     "year-month period dummies instead of year + month",
                                     "rare categories collapsed within strata",
                                     "any_surgical_service not estimated in the Global stratum"]}
(OUT_DIR / "o2_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")


1. LOAD AND VERIFY
Rows: 15,601; columns: 70
                                                      step     n
                      Episodes in the frozen analysis file 15601
                                        record type Global  8462
                                     record type Inpatient  6223
                         record type Emergency Observation   889
                 record type Temporary Inpatient Admission    27
O2 complete cases (cost and LOS present), all record types 15601
        PRIMARY SAMPLE O2: admissions (Global + Inpatient) 14685
                                            stratum Global  8462
                                             stratum Other  6223

Cost (Int$): median 436, IQR 301-708, mean 602, max 37,984

2. DESCRIPTIVES (cost, Int$)
                            variable                         level     n      median_IQR   mean   cv
All valid episodes (protocol sample)                           All 15601   417 (280-680) 569.93 1.29
    Record t

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 2 (expenditure) of the analysis protocol  --  VERSION 2 (corrected after diagnostics on the real file)
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional).

WHAT CHANGED FROM v1 (each change was forced by a diagnostic on digestive_surgical_analysis_file.csv)
  1  PRIMARY MODEL IS STRATIFIED BY BILLING MODE (Global package vs itemised "Other"), fitted as ONE fully
     interacted block-diagonal Gamma-log GLM (so patient/province-clustered covariances and Global-vs-Other
     contrasts are valid). v1 forced one set of covariate effects and one constant Global/Other ratio on two
     cost-generating processes that differ enormously (dispersion 0.05 vs 0.24, different ownership/surgical/
     province effects). The additive pooled model is still fitted (M1-M5, protocol comparable) but is labelled
     "pooled additive, known to be mis-specified" and is not used for conclusions.
  2  TIME: the nominal tariff moves in steps (about +45-50% in the first two months of 1402) and drifts inside
     1401. Separate year and month dummies cannot represent this; v2 uses year-month (period) dummies.
     The 10 episodes of 1400-12 are merged into 1401-01; any further period with < RARE_MIN episodes in a
     stratum is merged into its chronological neighbour.
  3  Categories with < RARE_MIN (30) episodes WITHIN A STRATUM are collapsed (to "Other", or to the modal level
     if the Other bucket would still be < 30) so no coefficient rests on a handful of episodes.
  4  POSITIVITY: any_surgical_service is not estimable in the Global stratum (only ~31 non-surgical Global
     episodes) and is dropped there; it stays in the itemised stratum.
  5  AVERAGE MARGINAL COSTS come from the stratified model (g-computation, delta-method CIs with patient AND
     province clusters) and are compared with the observed cell means; the Global-vs-itemised contrast is
     the standardised effect on the Global episodes (ATT): what the Global episodes would have cost if billed
     itemised, with the share of episodes on common support reported.
  6  Diagnostics (Park, link, Cook + refit, quasi-Poisson check) are run WITHIN each stratum; five-fold
     patient-grouped out-of-fold validation compares pooled vs stratified models by stratum.
  7  H1 compares variation after removing tariff steps (ICD-10 x period, and x province cells).
     H2 is tested within each stratum, H3 is shown crude / adjusted / province FE with a province-level
     cluster bootstrap CI (the province coefficient has no valid province-clustered SE otherwise).
  8  Sensitivity analyses 1-8 are re-run on the stratified specification; the A/B/C reliability verdict is
     computed from them (sparse level, direction, size of change, Cook refit, quasi-Poisson, province
     cluster, province random intercept).

PRIMARY SAMPLE  Admissions (record types Global + Inpatient), as in Objective 1 v3. Protocol sample (all valid
          episodes) is fitted as exploratory X0. This and the billing-mode stratification are deviations from
          the frozen protocol: state them in the Methods and label the analyses post hoc.
OUTCOME   cost_intl (Int$ = IRR/118,411.24, nominal, not inflation-adjusted).

USAGE
  python objective2_expenditure.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 200]   (AMC and H3 bootstraps use at most 100 replicates)
Outputs: folder objective2_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import gammaln

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "v2-stratified"
SEED = 42
RARE_MIN = 30                       # minimum episodes per category level within a stratum
SPARSE_N_MIN = 30
TOP_PROVINCES = 5
COOK_MULT = 4.0                     # Cook's distance threshold = COOK_MULT / n
N_FOLDS = 5
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
STABLE_PCT = 15.0                   # max % change across sensitivity analyses for an "A" estimate
COOK_PCT = 10.0                     # max % change in the Cook refit / quasi-Poisson for an "A" estimate

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=200)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service",
           "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
NUMS = {"M1": [], "M2": [], "M3": ["los_cov"], "M4": ["los_cov"], "M5": ["los_cov"]}   # LOS enters in M3
COLLAPSE_VARS = ["ownership", "specialty_model", "province_model", "insurance_fund_model",
                 "discharge_status_model", "sex", "age_group", "period"]
KEY_RE = r"C\(group\)|any_surgical_service|C\(ownership\)|los_cov"
Z975 = stats.norm.ppf(0.975)


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


T0 = time.time()


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE THE PRIMARY SAMPLE
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "cost_intl", "total_cost_irr", "los_days", "los_capped_p99",
        "los_capped_p95", "cluster_id", "any_surgical_service", "specialty_model", "specialty_h3",
        "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O2",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O2", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "specialty_h3", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["cost_intl", "total_cost_irr", "los_days", "los_capped_p99", "los_capped_p95"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
if (DALL["cost_intl"] <= 0).any() or DALL["cost_intl"].isna().any():
    fail("cost_intl has missing or non-positive values; the frozen file should not.")
DALL["los_cov"] = DALL["los_capped_p99"]
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"          # 10 episodes; see header
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
valid = DALL["in_O2"] & DALL["los_cov"].notna()
flow.append({"step": "O2 complete cases (cost and LOS present), all record types", "n": int(valid.sum())})
D = (DALL[is_adm & valid] if args.sample == "admissions" else DALL[valid]).copy().reset_index(drop=True)
ALT = (DALL[valid] if args.sample == "admissions" else DALL[is_adm & valid]).copy().reset_index(drop=True)
SAMPLE_LABEL = "admissions (Global + Inpatient)" if args.sample == "admissions" else "all valid episodes (protocol sample)"
ALT_LABEL = "all valid episodes (protocol sample)" if args.sample == "admissions" else "admissions only"
flow.append({"step": f"PRIMARY SAMPLE O2: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o2_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
print(f"\nCost (Int$): median {D.cost_intl.median():,.0f}, IQR {D.cost_intl.quantile(.25):,.0f}-"
      f"{D.cost_intl.quantile(.75):,.0f}, mean {D.cost_intl.mean():,.0f}, max {D.cost_intl.max():,.0f}")
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary sample.")


# =============================================================================
# 2. MODEL ENGINE: block-diagonal (stratified) or pooled Gamma-log / quasi-Poisson-log GLM
# =============================================================================
def build_spec(d, cats, nums, refs=None):
    refs = refs if refs is not None else {}
    cols, names, levs = [np.ones((len(d), 1))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        cols.append(_dummies(s, levs[v]))
        names += [f"C({v})[T.{lev}]" for lev in levs[v][1:]]
    for v in nums:
        cols.append(d[v].to_numpy(float)[:, None])
        names.append(v)
    X = np.hstack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], {"cats": list(cats), "nums": list(nums), "levs": levs, "keep": keep}


def _dummies(series, levels):
    """One-hot (non-reference levels) in a single vectorised pass."""
    codes = pd.Categorical(series.astype(str), categories=levels).codes
    return (codes[:, None] == np.arange(1, len(levels))[None, :]).astype(float)


def apply_spec(d, spec):
    cols = [np.ones((len(d), 1))]
    for v in spec["cats"]:
        cols.append(_dummies(d[v], spec["levs"][v]))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float)[:, None])
    return np.hstack(cols)[:, spec["keep"]]


def merge_periods(vc, minn):
    groups = [[p, [p], int(vc[p])] for p in sorted(vc.index)]          # [label, members, count]
    while len(groups) > 1 and min(g[2] for g in groups) < minn:
        i = int(np.argmin([g[2] for g in groups]))
        j = i + 1 if i + 1 < len(groups) else i - 1
        groups[j][1] += groups[i][1]
        groups[j][2] += groups[i][2]
        groups.pop(i)
    return {m: g[0] for g in groups for m in g[1]}


def make_cmap(ds, vars_, minn=RARE_MIN):
    """Collapse levels with < minn episodes inside this stratum."""
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        if v == "period":
            m = merge_periods(vc, minn)
        else:
            rare = [l for l in vc.index if vc[l] < minn]
            m = {}
            if rare:
                total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
                tgt = "Other" if total >= minn else vc.index[0]
                m = {l: tgt for l in rare}
                if tgt == "Other" and "Other" in vc.index:
                    pass
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def apply_cmap(d, cm):
    d = d.copy()
    for v, m in cm["map"].items():
        s0 = d[v].astype(str)
        s = s0.map(m).fillna(s0) if m else s0
        seen = cm["seen"][v]
        d[v] = s.where(s.isin(seen), "Other" if "Other" in seen else "__unseen__")
    return d


def fit_glm(X, y, fam="gamma"):
    """Log-link GLM by IRLS with step-halving on the deviance (plain IRLS diverges on the itemised stratum).
    Gamma: V = mu^2 (working weight 1). quasi-Poisson ('qp'): V = mu."""
    n, k = X.shape

    def dev(mu):
        if fam == "gamma":
            return 2 * np.sum(-np.log(y / mu) + (y - mu) / mu)
        return 2 * np.sum(y * np.log(y / mu) - (y - mu))

    beta = np.zeros(k)
    beta[0] = np.log(y.mean())
    eta = X @ beta
    mu = np.exp(eta)
    d_old = dev(mu)
    conv = False
    for _ in range(300):
        w = np.ones(n) if fam == "gamma" else mu
        z = eta + (y - mu) / mu
        A, rhs = (X.T * w) @ X, (X.T * w) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        step = 1.0
        for _h in range(30):
            cand = beta + step * (b_new - beta)
            eta_c = np.clip(X @ cand, -30, 30)
            mu_c = np.exp(eta_c)
            d_new = dev(mu_c)
            if np.isfinite(d_new) and d_new <= d_old + 1e-10 * abs(d_old):
                break
            step /= 2
        done = np.abs(cand - beta).max() < 1e-9 or abs(d_old - d_new) < 1e-12 * abs(d_old)
        beta, eta, mu, d_old = cand, eta_c, mu_c, d_new
        if done:
            conv = True
            break
    w = np.ones(n) if fam == "gamma" else mu
    V = mu ** 2 if fam == "gamma" else mu
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * w) @ X), "conv": conv,
            "phi": float(np.sum((y - mu) ** 2 / V) / max(n - k, 1)), "V": V}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups, fam="gamma"):
    n, k = X.shape
    s = (y - f["mu"]) / f["mu"] if fam == "gamma" else (y - f["mu"])
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_block(d, cats, nums, mode="stratified", fam="gamma", cov=True, minn=RARE_MIN):
    """mode 'stratified': one separate model per billing-mode stratum, fitted jointly as a block-diagonal GLM
    (any_surgical_service is dropped in the Global stratum). mode 'pooled': one additive model, billing_mode
    as a covariate (v1 specification, kept for comparison only)."""
    d = d.reset_index(drop=True)
    if mode == "pooled":
        parts = {"All": np.ones(len(d), bool)}
    else:
        parts = {s: (d["stratum"] == s).to_numpy() for s in ["Global", "Other"] if (d["stratum"] == s).sum() >= 50}
    blocks, Xs, names, off = {}, [], [], 0
    for s, rows in parts.items():
        ds = d.loc[rows]
        cs = [c for c in cats if not (mode == "stratified" and c in ("any_surgical_service", "billing_mode") and
                                      (s == "Global" or c == "billing_mode"))]
        cs = list(dict.fromkeys(cs + ["period"]))
        cm = make_cmap(ds, [v for v in cs if v in COLLAPSE_VARS], minn)
        dm = apply_cmap(ds, cm)
        refs = {"period": min(dm["period"].unique())}
        X_s, nm_s, spec = build_spec(dm, cs, nums, refs)
        blocks[s] = {"rows": np.where(rows)[0], "cols": slice(off, off + X_s.shape[1]), "names": nm_s,
                     "spec": spec, "cmap": cm, "X": X_s,
                     "lev_n": [int(X_s[:, j].sum()) if nm.startswith("C(") else np.nan for j, nm in enumerate(nm_s)]}
        off += X_s.shape[1]
        Xs.append((rows, X_s))
    K = off
    y = d["cost_intl"].to_numpy(float)
    # each stratum is fitted on its own (identical to the block-diagonal joint fit, ~5x cheaper)
    beta, eta, mu, V = np.zeros(K), np.zeros(len(d)), np.zeros(len(d)), np.zeros(len(d))
    bread, conv = np.zeros((K, K)), True
    for b in blocks.values():
        r = b["rows"]
        fb = fit_glm(b["X"], y[r], fam)
        beta[b["cols"]], eta[r], mu[r], V[r] = fb["beta"], fb["eta"], fb["mu"], fb["V"]
        bread[b["cols"], b["cols"]] = fb["bread"] if cov else 0
        conv &= fb["conv"]
    f = {"beta": beta, "eta": eta, "mu": mu, "V": V, "bread": bread, "conv": conv}
    X = None
    if cov:   # dense joint design only needed for the cluster-robust covariance (clusters cross strata)
        X = np.zeros((len(d), K))
        for (rows, X_s), b in zip(Xs, blocks.values()):
            X[np.ix_(np.where(rows)[0], np.arange(b["cols"].start, b["cols"].stop))] = X_s
    pre = {s: s for s in blocks}
    allnames = [(f"{s}|{nm}" if mode == "stratified" else nm) for s, b in blocks.items() for nm in b["names"]]
    lev_n = [x for b in blocks.values() for x in b["lev_n"]]
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl], fam) for cl in groups} if cov else {}
    phi = {}
    for s, b in blocks.items():
        r = b["rows"]
        phi[s] = float(np.sum((y[r] - f["mu"][r]) ** 2 / f["V"][r]) / max(len(r) - b["X"].shape[1], 1))
    return {"mode": mode, "fam": fam, "n": len(d), "names": allnames, "lev_n": lev_n, "beta": f["beta"],
            "covs": covs, "X": X, "y": y, "mu": f["mu"], "eta": f["eta"], "phi": phi, "conv": f["conv"],
            "blocks": blocks, "d": d, "bread": f["bread"], "V": f["V"], "K": K, "groups": groups}


def make_tab(core, model, label="cost"):
    out = []
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = core["beta"] / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term_full": core["names"], "n": core["n"],
                          "level_n": core["lev_n"], "ratio": np.exp(core["beta"]),
                          "lo": np.exp(core["beta"] - crit * se), "hi": np.exp(core["beta"] + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        if core["mode"] == "stratified":
            t["stratum"] = t.term_full.str.split("|", n=1).str[0]
            t["term"] = t.term_full.str.split("|", n=1).str[1]
        else:
            t["stratum"], t["term"] = "All", t["term_full"]
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[(t["term"] != "Intercept") & ~t["term"].str.startswith("C(period)")])
    return pd.concat(out, ignore_index=True)


def hit(r):
    return bool((r.lo > 1) or (r.hi < 1))


def crit_p(core, cl, est, se):
    G = core["covs"][cl][1]
    if cl == "province":
        c = stats.t.ppf(0.975, G - 1)
        return c, 2 * stats.t.sf(abs(est / se), G - 1) if se > 0 else np.nan
    return Z975, 2 * stats.norm.sf(abs(est / se)) if se > 0 else np.nan


# --- prediction, g-computation margins with gradients ------------------------
def pred_part(core, s, dpop, over=None):
    b = core["blocks"][s]
    dm = apply_cmap(dpop, b["cmap"])
    for k, v in (over or {}).items():
        dm[k] = v
    X = apply_spec(dm, b["spec"])
    return np.exp(np.clip(X @ core["beta"][b["cols"]], -30, 30)), X


def margin(core, s, dpop, over=None):
    mu, X = pred_part(core, s, dpop, over)
    g = np.zeros(core["K"])
    g[core["blocks"][s]["cols"]] = (mu @ X) / len(mu)
    return float(mu.mean()), g


def predict_rows(core, dte):
    mu = np.full(len(dte), np.nan)
    for s, b in core["blocks"].items():
        m = (dte["stratum"] == s).to_numpy() if core["mode"] == "stratified" else np.ones(len(dte), bool)
        if m.any():
            mu[m] = pred_part(core, s, dte[m])[0]
    return mu


def est_ci(core, est, grad, as_ratio=True):
    rows = {}
    for cl in core["covs"]:
        cov = core["covs"][cl][0]
        se = float(np.sqrt(max(grad @ cov @ grad, 0)))
        c, p = crit_p(core, cl, est, se)
        rows[cl] = (se, c, p)
    return rows


def support_share(core, s, dpop):
    b = core["blocks"][s]
    ok = np.ones(len(dpop), bool)
    for v in COLLAPSE_VARS:
        if v in b["cmap"]["map"] and v not in ("period", "sex", "age_group"):
            m = b["cmap"]["map"][v]
            ok &= ~dpop[v].astype(str).isin(m.keys()).to_numpy()
    return float(ok.mean())


# =============================================================================
# 3. DESCRIPTIVES
# =============================================================================
hdr("2. DESCRIPTIVES (cost, Int$)")


def cstats(s):
    s = s.dropna()
    m = s.mean()
    return {"n": len(s), "mean": m, "sd": s.std(), "median": s.median(), "p25": s.quantile(.25),
            "p75": s.quantile(.75), "p5": s.quantile(.05), "p95": s.quantile(.95), "p99": s.quantile(.99),
            "max": s.max(), "cv": s.std() / m if m > 0 else np.nan,
            "median_IQR": f"{s.median():,.0f} ({s.quantile(.25):,.0f}-{s.quantile(.75):,.0f})"}


def describe(d, var, label):
    rows = []
    for lev, g in ([("All", d)] if var is None else d.groupby(var, observed=True)):
        rows.append({"variable": label, "level": lev, **cstats(g["cost_intl"])})
    return rows


desc = describe(DALL[valid], None, "All valid episodes (protocol sample)")
desc += describe(DALL[valid], "record_type", "Record type (all valid episodes)")
desc += describe(D, None, "PRIMARY SAMPLE")
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("billing_mode", "Billing mode"),
                 ("specialty_model", "Specialty"), ("province_model", "Province"), ("ownership", "Ownership"),
                 ("period", "Period (Jalali year-month)")]:
    desc += describe(D, var, lab)
D["grp_x_bill"] = D["group"] + " | " + D["billing_mode"]
desc += describe(D, "grp_x_bill", "Group x billing mode")
DESC = pd.DataFrame(desc)
save(DESC, "o2_descriptives.csv")
print(DESC[DESC.variable.isin(["All valid episodes (protocol sample)", "Record type (all valid episodes)",
                               "PRIMARY SAMPLE", "Group", "Billing mode", "Group x billing mode"])]
      [["variable", "level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
print("\nBy ICD-10 code:\n", DESC[DESC.variable == "ICD-10 code"][["level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
print("\nBy period:\n", DESC[DESC.variable == "Period (Jalali year-month)"][["level", "n", "median_IQR", "mean"]].round(1).to_string(index=False))


# =============================================================================
# 4. HYPOTHESIS 1: Global vs itemised cost variation within the same code (tariff steps removed)
# =============================================================================
hdr("3. HYPOTHESIS 1: Global vs itemised cost variation, same ICD-10 code")


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj, run = np.empty_like(p), 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i])
        adj[i] = min(run, 1.0)
    return adj


def tariff_stats(x):
    med = x.median()
    return {"n": len(x), "median": med, "mean": x.mean(), "sd": x.std(), "cv": x.std() / x.mean(),
            "sd_log": np.log(x).std(),
            "qcd": (x.quantile(.75) - x.quantile(.25)) / (x.quantile(.75) + x.quantile(.25)),
            "pct_within_10pct_of_median": 100 * ((x - med).abs() <= 0.10 * med).mean(),
            "pct_within_25pct_of_median": 100 * ((x - med).abs() <= 0.25 * med).mean()}


H1_cells, H1_cmp = [], []
for code, g in D.groupby("icd10"):
    gg, oo = g.loc[g.stratum == "Global", "cost_intl"], g.loc[g.stratum == "Other", "cost_intl"]
    for lab, x in [("Global", gg), ("Other", oo)]:
        if len(x) >= 5:
            H1_cells.append({"icd10": code, "billing_mode": lab, **tariff_stats(x)})
    if len(gg) < 10 or len(oo) < 10:
        H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo), "note": "too few in a mode"})
        continue
    lg, lo_ = np.log(gg.to_numpy()), np.log(oo.to_numpy())
    bf = stats.levene(lg, lo_, center="median")
    bs = [np.log(RNG.choice(gg.to_numpy(), len(gg))).std() / np.log(RNG.choice(oo.to_numpy(), len(oo))).std()
          for _ in range(NBOOT)]
    H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo),
                   "sd_log_ratio_G_over_O": lg.std() / lo_.std(), "sd_log_lo": np.percentile(bs, 2.5),
                   "sd_log_hi": np.percentile(bs, 97.5), "cv_ratio_G_over_O": (gg.std() / gg.mean()) / (oo.std() / oo.mean()),
                   "median_ratio_G_over_O": gg.median() / oo.median(), "brown_forsythe_p": float(bf.pvalue), "note": ""})
H1c = pd.DataFrame(H1_cmp)
ok = H1c["brown_forsythe_p"].notna()
H1c.loc[ok, "p_holm"] = holm(H1c.loc[ok, "brown_forsythe_p"])
H1c["Global_less_variable"] = (H1c["sd_log_ratio_G_over_O"] < 1) & (H1c["p_holm"] < 0.05)
save(pd.DataFrame(H1_cells), "o2_h1_cost_by_code_and_billing.csv")
save(H1c, "o2_h1_variation_by_code.csv")
print(pd.DataFrame(H1_cells).round(2).to_string(index=False))
print("\nPer code (ratio of SD(log cost), Global/itemised; < 1 = Global less variable). The itemised cost also moves "
      "with the nominal tariff, so see the adjusted comparison below:")
print(H1c.round(3).to_string(index=False))


def pooled_within(lc, cell):
    n = np.bincount(cell)
    m = np.bincount(cell, lc) / np.maximum(n, 1)
    ss = float(np.sum((lc - m[cell]) ** 2))
    df = len(lc) - np.count_nonzero(n)
    tot = float(np.sum((lc - lc.mean()) ** 2))
    return np.sqrt(ss / max(df, 1)), 1 - ss / tot


h1a = []
for lab, keys in [("ICD-10 code", ["icd10"]), ("ICD-10 x period", ["icd10", "period"]),
                  ("ICD-10 x period x province", ["icd10", "period", "province_model"])]:
    cellid = D[keys].astype(str).agg("|".join, axis=1)
    cnt = pd.crosstab(cellid, D.stratum)
    good = cnt.index[(cnt["Global"] >= 5) & (cnt["Other"] >= 5)]
    sub = D[cellid.isin(good)]
    if len(sub) < 100:
        continue
    cid = pd.factorize(cellid[cellid.isin(good)])[0]
    lc = np.log(sub.cost_intl.to_numpy())
    res = {}
    for s in ["Global", "Other"]:
        m = (sub.stratum == s).to_numpy()
        res[s] = pooled_within(lc[m], pd.factorize(cid[m])[0])
    bs = []
    for _ in range(NBOOT):
        r = {}
        for s in ["Global", "Other"]:
            idx = np.where((sub.stratum == s).to_numpy())[0]
            idx = RNG.choice(idx, len(idx))
            r[s] = pooled_within(lc[idx], pd.factorize(cid[idx])[0])[0]
        bs.append(r["Global"] / r["Other"])
    sub_s = sub.assign(_c=cellid[cellid.isin(good)])
    pct = {}
    for s in ["Global", "Other"]:
        x = sub_s[sub_s.stratum == s]
        med = x.groupby("_c").cost_intl.transform("median")
        pct[s] = (100 * ((x.cost_intl - med).abs() <= .10 * med).mean(), 100 * ((x.cost_intl - med).abs() <= .25 * med).mean())
    h1a.append({"cells": lab, "n_cells": len(good), "n_Global": int((sub.stratum == "Global").sum()),
                "n_Other": int((sub.stratum == "Other").sum()),
                "within_cell_SD_log_Global": res["Global"][0], "within_cell_SD_log_Other": res["Other"][0],
                "SD_ratio_G_over_O": res["Global"][0] / res["Other"][0],
                "ratio_lo": np.percentile(bs, 2.5), "ratio_hi": np.percentile(bs, 97.5),
                "R2_cells_Global": res["Global"][1], "R2_cells_Other": res["Other"][1],
                "pct_within10_Global": pct["Global"][0], "pct_within10_Other": pct["Other"][0],
                "pct_within25_Global": pct["Global"][1], "pct_within25_Other": pct["Other"][1]})
H1A = pd.DataFrame(h1a)
H1A["supports_H1"] = H1A["ratio_hi"] < 1
save(H1A, "o2_h1_adjusted_variation.csv")
print("\nPooled within-cell variation (cells with >= 5 episodes in BOTH modes; bootstrap CI):")
print(H1A.round(3).to_string(index=False))
print("Reading: after removing code and period (tariff steps), the Global package varies less than itemised billing "
      "if the SD ratio and its CI are below 1. The R2 columns show how much of the Global cost variance is a tariff cell.")


# =============================================================================
# 5. MODELS M1-M5 : pooled additive (comparison) and stratified (primary)
# =============================================================================
hdr("4. MODELS M1-M5 (Gamma-log; patient and province clusters)")
CATS_ALL = BLOCKS["M5"] + ["billing_mode"]


def blk_cats(m, pooled):
    c = list(BLOCKS[m])
    return c + ["billing_mode"] if pooled and m != "M1" else c


CORE_P, CORE_S, TABS = {}, {}, []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cp = fit_block(D, blk_cats(m, True), NUMS[m], "pooled")
    cs = fit_block(D, blk_cats(m, False), NUMS[m], "stratified")
    CORE_P[m], CORE_S[m] = cp, cs
    TABS += [make_tab(cp, m).assign(spec="pooled additive (comparison only)"),
             make_tab(cs, m).assign(spec="stratified by billing mode (PRIMARY)")]
    print(f"  {m}: pooled terms {len(cp['names']) - 1}, dispersion {cp['phi']['All']:.3f} | stratified terms "
          f"{len(cs['names']) - len(cs['blocks'])}, dispersion " + ", ".join(f"{s} {v:.3f}" for s, v in cs["phi"].items()))
LONG = pd.concat(TABS, ignore_index=True)
save(LONG, "o2_models_long.csv")
LONG_S = LONG[LONG.spec.str.contains("PRIMARY")]
M5S, M5P = CORE_S["M5"], CORE_P["M5"]
prov_n = D.province_model.value_counts()
top_prov = prov_n.index[:TOP_PROVINCES].tolist()


def fmt(r):
    star = " *" if hit(r) else ""
    sp = " [n<30]" if r.sparse_flag else ""
    return f"{r.ratio:.2f} ({r.lo:.2f}-{r.hi:.2f}){star}{sp}"


def main_table(core, model):
    t = make_tab(core, model)
    rows = []
    for s in ["Global", "Other"]:
        for _, r in t[(t.stratum == s) & (t.cluster == "patient")].iterrows():
            if not (r.term.startswith(("C(group)", "C(any_surgical", "C(ownership", "C(specialty_model")) or
                    r.term == "los_cov" or any(r.term.endswith(f"[T.{p}]") and "province" in r.term for p in top_prov)):
                continue
            q = t[(t.stratum == s) & (t.cluster == "province") & (t.term == r.term)].iloc[0]
            rows.append({"stratum": s, "term": r.term, "level_n": r.level_n, "ratio": r.ratio, "lo_patient": r.lo,
                         "hi_patient": r.hi, "p_patient": r.p, "lo_province": q.lo, "hi_province": q.hi,
                         "p_province": q.p, "patient_CI": fmt(r), "province_CI": fmt(q),
                         "sparse_flag": r.sparse_flag})
    return pd.DataFrame(rows)


MAIN = main_table(M5S, "M5")
save(MAIN, "o2_main_table_M5_stratified.csv")
print("\nMAIN TABLE (stratified M5): cost ratio (95% CI) with patient clusters and with province clusters (t, G-1 df).\n"
      "* = CI excludes 1. Province-cluster CIs are the more honest ones for facility-level terms (ownership, specialty,\n"
      "province); province terms themselves have no valid province-clustered SE.")
print(MAIN[["stratum", "term", "level_n", "patient_CI", "province_CI"]].to_string(index=False))

# pooled additive billing-mode ratio (v1 type), for comparison with the stratified standardised ratio
pb = LONG[(LONG.spec.str.contains("pooled")) & (LONG.cluster == "patient") & (LONG.term.str.contains("billing_mode"))]
print("\nPooled ADDITIVE billing-mode ratio (Global vs itemised) by block [v1-type estimate; assumes one constant ratio]:")
print(pb[["model", "term", "ratio", "lo", "hi"]].round(3).to_string(index=False))
save(pb, "o2_pooled_billing_ratio_by_block.csv")


# =============================================================================
# 6. AVERAGE MARGINAL COSTS + STANDARDISED GLOBAL-vs-ITEMISED CONTRAST
# =============================================================================
hdr("5. AVERAGE MARGINAL COSTS (g-computation from the stratified model)")


def amc_tables(core, d, model):
    own, att = [], []
    groups = ["All"] + sorted(d.group.unique())
    for s in ["Global", "Other"]:
        pop_s = d[d.stratum == s]
        for g in groups:
            over = None if g == "All" else {"group": g}
            pop = pop_s if g == "All" else pop_s[pop_s.group == g]
            if len(pop) < 20:
                continue
            m, gr = margin(core, s, pop)                                  # fitted mean in the observed group
            lm, lg = np.log(m), gr / m
            ci = est_ci(core, lm, lg)
            row = {"model": model, "stratum": s, "group": g, "n": len(pop), "observed_mean": pop.cost_intl.mean(),
                   "observed_median": pop.cost_intl.median(), "AMC": m, "obs_over_AMC": pop.cost_intl.mean() / m}
            for cl, (se, c, p) in ci.items():
                row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(lm - c * se)), float(np.exp(lm + c * se))
            own.append(row)
        # group counterfactual (everyone in the stratum set to group g): standardised AMC
        if True:
            for g in groups[1:]:
                m, gr = margin(core, s, pop_s, {"group": g})
                ci = est_ci(core, np.log(m), gr / m)
                row = {"model": model, "stratum": s, "group": f"standardised: all {s} episodes as {g}", "n": len(pop_s),
                       "observed_mean": np.nan, "observed_median": np.nan, "AMC": m, "obs_over_AMC": np.nan}
                for cl, (se, c, p) in ci.items():
                    row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(np.log(m) - c * se)), float(np.exp(np.log(m) + c * se))
                own.append(row)
    popG = d[d.stratum == "Global"]
    for g in groups:
        pop = popG if g == "All" else popG[popG.group == g]
        if len(pop) < 20:
            continue
        m1, g1 = margin(core, "Global", pop)
        m2, g2 = margin(core, "Other", pop)
        lr, glr = np.log(m1 / m2), g1 / m1 - g2 / m2
        cir = est_ci(core, lr, glr)
        cid = est_ci(core, m1 - m2, g1 - g2)
        row = {"model": model, "population": f"Global episodes ({g})", "n_Global": len(pop),
               "share_on_common_support": support_share(core, "Other", pop), "observed_Global_mean": pop.cost_intl.mean(),
               "fitted_Global_mean": m1, "if_itemised_mean": m2, "ratio_Global_over_itemised": m1 / m2,
               "diff_Int$": m1 - m2}
        for cl, (se, c, p) in cir.items():
            row[f"ratio_lo_{cl}"], row[f"ratio_hi_{cl}"], row[f"ratio_p_{cl}"] = \
                float(np.exp(lr - c * se)), float(np.exp(lr + c * se)), p
        for cl, (se, c, p) in cid.items():
            row[f"diff_lo_{cl}"], row[f"diff_hi_{cl}"] = (m1 - m2) - c * se, (m1 - m2) + c * se
        att.append(row)
    return pd.DataFrame(own), pd.DataFrame(att)


# The marginal costs (means) come from the quasi-Poisson-log fit: its score equations make the fitted mean equal the
# observed mean in every cell of the categorical covariates, so it is the mean-preserving estimator of expenditure.
# The Gamma-log fit (used for the cost RATIOS, as in the protocol) over-weights low-cost episodes in the heavy-tailed
# itemised stratum and misses the observed itemised Gallbladder mean by ~13%; its AMCs are saved for comparison.
M5Q = fit_block(D, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp")
AMC, ATT = amc_tables(M5Q, D, "M5 quasi-Poisson-log (primary for means)")
AMC_G, ATT_G = amc_tables(M5S, D, "M5 Gamma-log (comparison)")


def boot_ci(d, B, key):
    """Cluster bootstrap (whole refit) for the AMCs and the Global/itemised ratio. The delta-method CI above is
    conditional on the observed covariate mix, so it is too narrow for population means (it is ~0 for the stratum
    mean under the quasi-Poisson fit); the bootstrap includes that sampling variation."""
    cl = d[key].astype(str).to_numpy()
    uc, inv = np.unique(cl, return_inverse=True)
    idx_by = [np.where(inv == i)[0] for i in range(len(uc))]
    ra, rt = [], []
    for _ in range(B):
        pick = RNG.integers(0, len(uc), len(uc))
        db = d.iloc[np.concatenate([idx_by[i] for i in pick])].reset_index(drop=True)
        try:
            cb = fit_block(db, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp", cov=False)
            a_, t_ = amc_tables(cb, db, "b")
        except Exception:
            continue
        ra.append(a_.set_index(["stratum", "group"]).AMC)
        rt.append(t_.set_index("population")[["ratio_Global_over_itemised", "diff_Int$"]])
    ra, rt = pd.concat(ra, axis=1), pd.concat(rt, axis=1)
    return ra, rt


NB = min(NBOOT, 100)
for key, tag in [("cluster_id", "patient"), ("province_model", "province")]:
    ra, rt = boot_ci(D, NB, key)
    q = ra.quantile([.025, .975], axis=1).T
    AMC[f"lo_boot_{tag}"] = [q.loc[(r.stratum, r.group), 0.025] if (r.stratum, r.group) in q.index else np.nan for r in AMC.itertuples()]
    AMC[f"hi_boot_{tag}"] = [q.loc[(r.stratum, r.group), 0.975] if (r.stratum, r.group) in q.index else np.nan for r in AMC.itertuples()]
    ratio = pd.concat([rt.iloc[:, 0::2]], axis=1) if False else rt
    rr = rt["ratio_Global_over_itemised"]
    dd = rt["diff_Int$"]
    qr, qd = rr.quantile([.025, .975], axis=1).T, dd.quantile([.025, .975], axis=1).T
    ATT[f"ratio_lo_boot_{tag}"] = [qr.loc[p_, 0.025] for p_ in ATT.population]
    ATT[f"ratio_hi_boot_{tag}"] = [qr.loc[p_, 0.975] for p_ in ATT.population]
    ATT[f"diff_lo_boot_{tag}"] = [qd.loc[p_, 0.025] for p_ in ATT.population]
    ATT[f"diff_hi_boot_{tag}"] = [qd.loc[p_, 0.975] for p_ in ATT.population]
# the bootstrap CIs replace the delta-method CIs as the reported ones (delta columns are kept, suffix _delta)
for tag in ["patient", "province"]:
    AMC = AMC.rename(columns={f"lo_{tag}": f"lo_{tag}_delta", f"hi_{tag}": f"hi_{tag}_delta"})
    ATT = ATT.rename(columns={f"ratio_lo_{tag}": f"ratio_lo_{tag}_delta", f"ratio_hi_{tag}": f"ratio_hi_{tag}_delta"})
    AMC[f"lo_{tag}"], AMC[f"hi_{tag}"] = AMC[f"lo_boot_{tag}"], AMC[f"hi_boot_{tag}"]
    ATT[f"ratio_lo_{tag}"], ATT[f"ratio_hi_{tag}"] = ATT[f"ratio_lo_boot_{tag}"], ATT[f"ratio_hi_boot_{tag}"]

save(AMC, "o2_amc_stratified_M5.csv")
save(ATT, "o2_global_vs_itemised_standardised_M5.csv")
save(AMC_G, "o2_amc_stratified_M5_gamma.csv")
save(ATT_G, "o2_global_vs_itemised_standardised_M5_gamma.csv")
print("Average marginal cost (Int$) by stratum and group, with the observed mean (obs_over_AMC should be ~ 1):")
print(AMC[AMC.n.notna()][["stratum", "group", "n", "observed_mean", "AMC", "lo_patient", "hi_patient", "lo_province",
                          "hi_province", "obs_over_AMC"]].round(2).to_string(index=False))
print("\nStandardised Global-vs-itemised contrast on the Global episodes (what they would have cost if billed itemised):")
print(ATT[["population", "n_Global", "share_on_common_support", "observed_Global_mean", "if_itemised_mean",
           "ratio_Global_over_itemised", "ratio_lo_patient", "ratio_hi_patient", "ratio_lo_province", "ratio_hi_province"]]
      .round(3).to_string(index=False))
ATT_BY_BLOCK = []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    a = amc_tables(fit_block(D, blk_cats(m, False), NUMS[m], "stratified", fam="qp"), D, m)[1]
    ATT_BY_BLOCK.append(a[a.population == "Global episodes (All)"])
ATT_BY_BLOCK = pd.concat(ATT_BY_BLOCK)
save(ATT_BY_BLOCK, "o2_global_vs_itemised_by_block.csv")
print("\nGlobal/itemised ratio across blocks (stratified quasi-Poisson models):")
print(ATT_BY_BLOCK[["model", "fitted_Global_mean", "if_itemised_mean", "ratio_Global_over_itemised",
                    "ratio_lo_patient", "ratio_hi_patient"]].round(3).to_string(index=False))

# check of the v1 additive AMC against observed cells
chk = []
for s in ["Global", "Other"]:
    for g in sorted(D.group.unique()):
        pop = D[(D.stratum == s) & (D.group == g)]
        mu_p = pred_part(M5P, "All", pop)[0].mean()
        a = AMC_G[(AMC.stratum == s) & (AMC.group == g)]
        chk.append({"stratum": s, "group": g, "n": len(pop), "observed_mean": pop.cost_intl.mean(),
                    "pooled_additive_fitted": mu_p, "stratified_gamma_fitted": float(a.AMC.iloc[0]),
                    "stratified_qp_fitted": float(AMC[(AMC.stratum == s) & (AMC.group == g)].AMC.iloc[0]),
                    "pooled_error_pct": 100 * (mu_p / pop.cost_intl.mean() - 1),
                    "stratified_gamma_error_pct": 100 * (float(a.AMC.iloc[0]) / pop.cost_intl.mean() - 1),
                    "stratified_qp_error_pct": 100 * (float(AMC[(AMC.stratum == s) & (AMC.group == g)].AMC.iloc[0]) / pop.cost_intl.mean() - 1)})
CHK = pd.DataFrame(chk)
save(CHK, "o2_amc_fit_to_observed_pooled_vs_stratified.csv")
print("\nFitted vs observed cell means (quasi-Poisson reproduces them by construction, Gamma-log and the additive model do not):")
print(CHK.round(2).to_string(index=False))


# =============================================================================
# 7. DIAGNOSTICS, WITHIN EACH STRATUM
# =============================================================================
hdr("6. DIAGNOSTICS (Park, link, Cook's distance + refit, quasi-Poisson), within strata")


def ols_cluster(x, y, groups):
    Xo = np.column_stack([np.ones(len(x)), x])
    b = np.linalg.solve(Xo.T @ Xo, Xo.T @ y)
    e = y - Xo @ b
    meat, g = cluster_meat(Xo * e[:, None], groups)
    bread = np.linalg.inv(Xo.T @ Xo)
    V = (g / (g - 1)) * bread @ meat @ bread
    return b, np.sqrt(np.diag(V)), g


diag, cook_tabs = [], []
cook_flag_all = np.zeros(len(M5S["d"]), bool)
for s, b in M5S["blocks"].items():
    r = b["rows"]
    Xs, ys, mu, eta = b["X"], M5S["y"][r], M5S["mu"][r], M5S["eta"][r]
    pat = M5S["groups"]["patient"][r]
    e = ys - mu
    ok = e != 0
    (a0, a1), (s0, s1), g = ols_cluster(np.log(mu[ok]), np.log(e[ok] ** 2), pat[ok])
    # link test
    Xl = np.column_stack([Xs, eta ** 2 - 0])
    fl = fit_glm(Xl, ys)
    covl, _ = cov_robust(Xl, ys, fl, pat)
    tl = fl["beta"][-1] / np.sqrt(covl[-1, -1])
    # Cook's distance (Gamma, working weight 1)
    brd = np.linalg.pinv(Xs.T @ Xs)
    h = np.einsum("ij,jk,ik->i", Xs, brd, Xs)
    rp = (ys - mu) / mu
    k = Xs.shape[1]
    cook = rp ** 2 * h / (k * M5S["phi"][s] * (1 - h) ** 2)
    flag = cook > COOK_MULT / len(r)
    cook_flag_all[r[flag]] = True
    diag.append({"stratum": s, "n": len(r), "terms": k, "dispersion": M5S["phi"][s],
                 "park_slope": a1, "park_se": s1, "park_reading": ("Gamma (slope ~2)" if abs(a1 - 2) < 2 * s1 else
                                                                  ("closer to inverse Gaussian (3)" if a1 > 2 else "closer to Poisson-like (1)")),
                 "link_t": float(tl), "link_p": float(2 * stats.norm.sf(abs(tl))), "max_cook": float(cook.max()),
                 "n_cook_flag": int(flag.sum()), "max_leverage": float(h.max()), "n_leverage_gt_0.2": int((h > .2).sum())})
DIAG = pd.DataFrame(diag)
save(DIAG, "o2_diagnostics_by_stratum.csv")
print(DIAG.round(3).to_string(index=False))
print("Park slope within each stratum guides the variance function (2 = Gamma). Robust (sandwich) SEs are used throughout,\n"
      "so a moderate departure widens nothing; the quasi-Poisson refit below shows whether it matters for the estimates.")

dref = M5S["d"][~cook_flag_all]
M5C = fit_block(dref, BLOCKS["M5"], NUMS["M5"], "stratified")
tm, tc, tq = make_tab(M5S, "M5"), make_tab(M5C, "M5 cook-excl"), make_tab(M5Q, "M5 quasi-Poisson")
tm = tm[tm.cluster == "patient"].set_index("term_full")
tc, tq = tc[tc.cluster == "patient"].set_index("term_full"), tq[tq.cluster == "patient"].set_index("term_full")
ROB = tm[["stratum", "term", "level_n", "ratio", "lo", "hi"]].copy()
ROB["ratio_cook_excl"] = tc["ratio"].reindex(ROB.index)
ROB["ratio_quasiPoisson"] = tq["ratio"].reindex(ROB.index)
ROB["cook_pct_change"] = 100 * (ROB.ratio_cook_excl / ROB.ratio - 1)
ROB["qp_pct_change"] = 100 * (ROB.ratio_quasiPoisson / ROB.ratio - 1)
ROB = ROB.reset_index(drop=True)
save(ROB, "o2_robustness_cook_quasipoisson.csv")
print(f"\nCook refit excludes {int(cook_flag_all.sum())} episodes (> {COOK_MULT:g}/n within stratum). Key terms:")
kt = ROB[ROB.term.str.contains(KEY_RE)]
print(kt.round(3).to_string(index=False))
# AMC/ATT effect of the influential episodes
attC = amc_tables(M5C, M5C["d"], "M5 cook-excl")[1]
attQ = ATT_G
print("\nGlobal/itemised ratio: quasi-Poisson (primary)", ATT.ratio_Global_over_itemised.iloc[0].round(3), "| Gamma-log",
      ATT_G.ratio_Global_over_itemised.iloc[0].round(3), "| Gamma, Cook-excluded", attC.ratio_Global_over_itemised.iloc[0].round(3))


# =============================================================================
# 8. FIVE-FOLD OUT-OF-FOLD PREDICTION ERROR
# =============================================================================
hdr("7. OUT-OF-FOLD PREDICTION (5 folds, grouped by patient)")
ids = pd.factorize(D.cluster_id)[0]
uid = np.unique(ids)
RNG2 = np.random.default_rng(SEED)
RNG2.shuffle(uid)
fm = dict(zip(uid, np.arange(len(uid)) % N_FOLDS))
FOLD = np.array([fm[i] for i in ids])
oof_rows, cal_rows = [], []
for m in ["M1", "M3", "M5"]:
    for mode in ["pooled", "stratified", "stratified_qp"]:
        pred = np.full(len(D), np.nan)
        for k in range(N_FOLDS):
            tr, te = D[FOLD != k], D[FOLD == k]
            core = fit_block(tr, blk_cats(m, mode == "pooled"), NUMS[m], mode.replace("_qp", ""), fam="qp" if mode.endswith("qp") else "gamma", cov=False)
            p = predict_rows(core, te)
            lo_, hi_ = tr.cost_intl.min(), tr.cost_intl.max()
            pred[(FOLD == k)] = np.clip(p, lo_, hi_)
        y = D.cost_intl.to_numpy()
        for s in ["All", "Global", "Other"]:
            mk = np.ones(len(D), bool) if s == "All" else (D.stratum == s).to_numpy()
            err = y[mk] - pred[mk]
            oof_rows.append({"model": m, "spec": mode, "stratum": s, "n": int(mk.sum()), "MAE": np.abs(err).mean(),
                             "RMSE": np.sqrt((err ** 2).mean()), "MAE_log": np.abs(np.log(y[mk]) - np.log(pred[mk])).mean(),
                             "mean_observed": y[mk].mean(), "mean_predicted": pred[mk].mean(),
                             "calibration_ratio": pred[mk].mean() / y[mk].mean(),
                             "relative_MAE_pct": 100 * np.abs(err).mean() / y[mk].mean()})
            if m == "M5" and s != "All" or (m == "M5" and s == "All"):
                q = pd.qcut(pd.Series(pred[mk]).rank(method="first"), 10, labels=False)
                c = pd.DataFrame({"decile": q + 1, "pred": pred[mk], "obs": y[mk]}).groupby("decile").mean().reset_index()
                c["model"], c["spec"], c["stratum"] = m, mode, s
                cal_rows.append(c.rename(columns={"pred": "mean_predicted", "obs": "mean_observed"}))
OOF = pd.DataFrame(oof_rows)
CAL = pd.concat(cal_rows, ignore_index=True)
save(OOF, "o2_oof_error.csv")
save(CAL, "o2_oof_calibration_deciles.csv")
print(OOF.round(3).to_string(index=False))
c5 = CAL[(CAL.model == "M5") & (CAL.stratum == "All")].copy()
c5["ratio"] = c5.mean_predicted / c5.mean_observed
print("\nM5 calibration by predicted decile, all admissions (ratio = predicted/observed):")
print(c5.pivot(index="decile", columns="spec", values="ratio").round(3).to_string())


# =============================================================================
# 9. CONCENTRATION
# =============================================================================
hdr("8. CONCENTRATION OF EXPENDITURE")


def conc_stats(x):
    x = np.sort(np.asarray(x, float))
    n, tot = len(x), x.sum()
    cum = np.concatenate([[0], np.cumsum(x) / tot])
    p = np.linspace(0, 1, n + 1)
    gini = 1 - 2 * float(np.sum((cum[1:] + cum[:-1]) / 2 * np.diff(p)))
    out = {"n": n, "total_intl": tot, "gini": gini}
    for q in (1, 5, 10, 20):
        out[f"top{q}pct_share"] = 100 * x[-max(int(np.ceil(q / 100 * n)), 1):].sum() / tot
    return out


def conc_boot(x, B):
    x = np.asarray(x, float)
    r = [conc_stats(RNG.choice(x, len(x))) for _ in range(B)]
    return (np.percentile([v["top10pct_share"] for v in r], [2.5, 97.5]), np.percentile([v["gini"] for v in r], [2.5, 97.5]))


crow = []
masks = [("All (primary sample)", pd.Series(True, index=D.index))] + \
        [(f"group={g}", D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())] + \
        [(f"{g} x {b}", (D.group == g) & (D.billing_mode == b)) for g in sorted(D.group.unique()) for b in sorted(D.billing_mode.unique())] + \
        [("All, flagged extremes removed", ~D.flag_extreme_any)]
for lab, mask in masks:
    x = D.loc[mask, "cost_intl"]
    if len(x) < 20:
        continue
    (t_lo, t_hi), (g_lo, g_hi) = conc_boot(x, NBOOT)
    crow.append({"stratum": lab, **conc_stats(x), "top10_lo": t_lo, "top10_hi": t_hi, "gini_lo": g_lo, "gini_hi": g_hi})
CONC = pd.DataFrame(crow)
save(CONC, "o2_concentration.csv")
print(CONC.round(2).to_string(index=False))
lz = []
for lab, mask in [("All", pd.Series(True, index=D.index))] + [(g, D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())]:
    xs = np.sort(D.loc[mask, "cost_intl"].to_numpy())
    cum = np.concatenate([[0], np.cumsum(xs) / xs.sum()])
    pp = np.linspace(0, 1, len(xs) + 1)
    for q in np.linspace(0, 1, 101):
        lz.append({"stratum": lab, "cum_share_episodes": q, "cum_share_cost": float(np.interp(q, pp, cum))})
LZ = pd.DataFrame(lz)
save(LZ, "o2_lorenz_points.csv")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for lab in ["All"] + sorted(D.group.unique()):
        s = LZ[LZ.stratum == lab]
        ax[0].plot(s.cum_share_episodes, s.cum_share_cost, label=lab, lw=2 if lab == "All" else 1.4)
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
    ax[0].set(title="Lorenz curve of episode cost by group", xlabel="Cumulative share of episodes",
              ylabel="Cumulative share of expenditure")
    ax[0].legend(frameon=False)
    for k, spec in enumerate(["pooled", "stratified_qp"]):
        for s, col in [("Global", "C0"), ("Other", "C3")]:
            c8 = CAL[(CAL.model == "M5") & (CAL.stratum == s) & (CAL.spec == spec)]
            ax[1 + k].plot(c8.mean_predicted, c8.mean_observed, "o-", color=col, label=s)
    for a, t in zip(ax[1:], ["M5 pooled additive: OOF calibration", "M5 stratified quasi-Poisson: OOF calibration"]):
        lim = [0, max(CAL.mean_predicted.max(), CAL.mean_observed.max()) * 1.05]
        a.plot(lim, lim, "k--", lw=.8)
        a.set(title=t, xlabel="Mean predicted (Int$)", ylabel="Mean observed (Int$)")
        a.legend(frameon=False, title="billing mode")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o2_lorenz_and_calibration.png", dpi=200)
    plt.close(fig)
    print("Figure saved: o2_lorenz_and_calibration.png")
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")


# =============================================================================
# 10. HYPOTHESES 2 AND 3
# =============================================================================
hdr("9. HYPOTHESIS 2 (provinces) AND HYPOTHESIS 3 (K40: general surgery vs urology)")


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


h2 = []
for m in ["M3", "M4", "M5"]:
    core = CORE_S[m]
    for s, b in core["blocks"].items():
        idx = [b["cols"].start + j for j, nm in enumerate(b["names"]) if nm.startswith("C(province_model)")]
        if m == "M3":
            kw = stats.kruskal(*[g.cost_intl.values for _, g in D[D.stratum == s].groupby("province_model") if len(g) >= 5])
            h2.append({"model": "crude (Kruskal-Wallis)", "stratum": s, "df": np.nan, "chi2": float(kw.statistic),
                       "p": float(kw.pvalue), "note": "unadjusted"})
        if idx and m != "M3":
            chi2, df, p = wald(core, idx)
            prov = make_tab(core, m)
            prov = prov[(prov.stratum == s) & (prov.cluster == "patient") & prov.term.str.startswith("C(province_model)")]
            prov = prov[~prov.sparse_flag]
            h2.append({"model": m, "stratum": s, "df": df, "chi2": chi2, "p": p,
                       "n_provinces_modelled": df + 1, "min_ratio": prov.ratio.min(), "max_ratio": prov.ratio.max(),
                       "note": "Wald, patient-clustered; province-level SE not valid (28 clusters)"})
H2 = pd.DataFrame(h2)
save(H2, "o2_h2_province_tests.csv")
print(H2.round(4).to_string(index=False))
prov_main = make_tab(M5S, "M5")
prov_main = prov_main[prov_main.term.str.startswith("C(province_model)") & (prov_main.cluster == "patient")]
save(prov_main.drop(columns=["term_full"]), "o2_h2_province_ratios_M5.csv")
for s in ["Global", "Other"]:
    pm = prov_main[(prov_main.stratum == s) & ~prov_main.sparse_flag].sort_values("ratio")
    print(f"\n{s}: adjusted province cost ratios (vs the largest province in that stratum), extremes:")
    print(pm[["term", "level_n", "ratio", "lo", "hi"]].iloc[[0, 1, 2, -3, -2, -1]].round(2).to_string(index=False))

SIG = []
for s in ["Global", "Other"]:
    pass  # random-intercept variance components are computed with sensitivity 8 below


def h3_variant(d, which, cl_boot=False):
    cats = ["age_group", "sex", "insurance_fund_model", "ownership", "discharge_status_model", "any_surgical_service"]
    if which == "crude":
        cats = []
    if which == "adjusted_provFE":
        cats = cats + ["province_model"]
    d = d.copy()
    d["spec_h3"] = d["specialty_h3"].where(d["specialty_h3"].isin(["General surgery", "Urology"]), "x")
    d = d[d.spec_h3 != "x"]
    return d, cats + ["spec_h3"], ["los_cov"] if which != "crude" else []


h3 = []
K40 = D[D.icd10 == "K40"].copy()
for s in ["Other", "Global", "Both (pooled, +billing mode)"]:
    sub = K40 if s.startswith("Both") else K40[K40.stratum == s]
    for which in ["crude", "adjusted", "adjusted_provFE"]:
        d3, c3, n3 = h3_variant(sub, which)
        nu = (d3.specialty_h3 == "Urology").sum()
        if nu < 30 or (d3.specialty_h3 == "General surgery").sum() < 30:
            h3.append({"stratum": s, "variant": which, "n": len(d3), "n_urology": int(nu), "note": "too few urology episodes"})
            continue
        if s.startswith("Both"):
            c3 = c3 + ["billing_mode"]
        core = fit_block(d3.assign(stratum="Other"), c3, n3, "pooled")
        tt = make_tab(core, "K40")
        tt = tt[tt.term.str.contains("spec_h3")]
        rp = tt[tt.cluster == "patient"].iloc[0]
        rq = tt[tt.cluster == "province"].iloc[0]
        # province-level cluster bootstrap CI
        provs = d3.province_model.unique()
        bs = []
        grp = {p: d3[d3.province_model == p] for p in provs}
        for _ in range(min(NBOOT, 100)):
            pick = RNG.choice(provs, len(provs))
            db = pd.concat([grp[p] for p in pick], ignore_index=True)
            if db.spec_h3.nunique() < 2:
                continue
            try:
                cb = fit_block(db.assign(stratum="Other"), c3, n3, "pooled", cov=False)
                nmb = [i for i, nm in enumerate(cb["names"]) if "spec_h3" in nm]
                if nmb:
                    bs.append(np.exp(cb["beta"][nmb[0]]))
            except Exception:
                pass
        h3.append({"stratum": s, "variant": which, "n": len(d3), "n_urology": int(nu), "urology_vs_general_ratio": rp.ratio,
                   "lo_patient": rp.lo, "hi_patient": rp.hi, "p_patient": rp.p,
                   "lo_province_t": rq.lo, "hi_province_t": rq.hi, "p_province_t": rq.p,
                   "lo_province_bootstrap": np.percentile(bs, 2.5) if len(bs) > 20 else np.nan,
                   "hi_province_bootstrap": np.percentile(bs, 97.5) if len(bs) > 20 else np.nan, "note": ""})
H3 = pd.DataFrame(h3)
save(H3, "o2_h3_k40_specialty.csv")
print("\nH3 (K40, urology vs general surgery). Urology is concentrated in itemised billing, so the itemised stratum is "
      "the primary comparison; the specialty term is a facility-level exposure, so the province CIs are the honest ones:")
print(H3.round(3).to_string(index=False))


# =============================================================================
# 11. SENSITIVITY ANALYSES 1-8 (stratified M5)
# =============================================================================
hdr("10. SENSITIVITY ANALYSES 1-8 (stratified M5)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"),
        2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"),
        4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS covariate capped at P95", None)}
SL, SATT = pd.DataFrame(), pd.DataFrame()


def key_rows(core, s_id, name):
    t = make_tab(core, "M5")
    t = t[(t.cluster == "patient") & t.term.str.contains(KEY_RE)].copy()
    t["sens"], t["sens_name"] = s_id, name
    return t


if RUN_SENS:
    stabs, satt = [], []
    for s_id, (name, keepcol) in SENS.items():
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        if s_id == 7:
            sub["los_cov"] = sub["los_capped_p95"]
        core = fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified")
        stabs.append(key_rows(core, s_id, name))
        if "Global" in core["blocks"] and "Other" in core["blocks"]:
            coreq = fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp")
            a = amc_tables(coreq, sub.reset_index(drop=True), "M5")[1].assign(sens=s_id, sens_name=name)
            satt.append(a)
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}, strata = {list(core['blocks'])}")
    SL = pd.concat(stabs, ignore_index=True)
    SATT = pd.concat(satt, ignore_index=True) if satt else pd.DataFrame()


class LaplaceGamma:
    """Gamma-log GLMM with a province random intercept, Laplace-approximated ML (as lme4::glmer nAGQ = 1)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def _l(self, eta, nu):
        e = np.exp(-eta)
        l = nu * np.log(nu) - gammaln(nu) + (nu - 1) * np.log(self.y) - nu * self.y * e - nu * eta
        return l, nu * (self.y * e - 1), -nu * self.y * e

    def negll(self, theta):
        beta, sig, nu = theta[:self.k], np.exp(theta[self.k]), np.exp(theta[self.k + 1])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            l, d1, d2 = self._l(xb + u[self.g], nu)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        l, d1, d2 = self._l(xb + u[self.g], nu)
        h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
        return -(l.sum() - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0, nu0):
        th0 = np.concatenate([beta0, [np.log(0.3)], [np.log(nu0)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()
if RUN_SENS8:
    hdr("10b. SENSITIVITY 8: province as a random intercept (Gamma-log, Laplace ML), within each stratum")
    s8, vc8 = [], []
    for s, b in M5S["blocks"].items():
        t0 = time.time()
        try:
            ds = D[D.stratum == s].reset_index(drop=True)
            cats = [v for v in BLOCKS["M5"] if v != "province_model" and not (s == "Global" and v == "any_surgical_service")]
            cats = list(dict.fromkeys(cats + ["period"]))
            cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
            dm = apply_cmap(ds, cm)
            X8, nm8, _ = build_spec(dm, cats, ["los_cov"], {"period": min(dm["period"].unique())})
            y8 = ds.cost_intl.to_numpy(float)
            g8 = pd.factorize(ds.province_model)[0]
            f0 = fit_glm(X8, y8)
            m8 = LaplaceGamma(X8, y8, g8)
            theta = m8.fit(f0["beta"], 1 / max(f0["phi"], 1e-3))
            H = m8.hessian(theta)
            for _ in range(2):
                gr = optimize.approx_fprime(theta, m8.negll, 1e-6)
                try:
                    cand = theta - np.linalg.solve(H, gr)
                except np.linalg.LinAlgError:
                    break
                if m8.negll(cand) <= m8.negll(theta) + 1e-9:
                    theta = cand
                    H = m8.hessian(theta)
            se8 = np.sqrt(np.clip(np.diag(np.linalg.pinv(H)), 0, None))
            k8 = m8.k
            t8 = pd.DataFrame({"outcome": "cost", "model": "M5 random intercept", "stratum": s, "term": nm8[1:], "n": len(ds),
                               "level_n": [int(X8[:, j].sum()) if nm8[j].startswith("C(") else np.nan for j in range(1, k8)],
                               "ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                               "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]), "sens": 8,
                               "sens_name": "Province random intercept"})
            s8.append(t8)
            sig = float(np.exp(theta[k8]))
            vc8.append({"stratum": s, "n": len(ds), "provinces": m8.G, "sigma_u": sig,
                        "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])),
                        "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                        "province_cost_ratio_1sd": float(np.exp(sig)),
                        "gamma_dispersion": float(1 / np.exp(theta[k8 + 1]))})
            print(f"  {s}: n = {len(ds):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (a province 1 SD above the mean costs "
                  f"x{np.exp(sig):.2f}), {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  Sensitivity 8 ({s}) FAILED ({type(e).__name__}: {e})")
    if s8:
        S8, VC8 = pd.concat(s8, ignore_index=True), pd.DataFrame(vc8)
        save(S8, "o2_sens8_random_intercept_long.csv")
        save(VC8, "o2_sens8_variance_components.csv")
        print(VC8.round(3).to_string(index=False))
        SL = pd.concat([SL, S8[S8.term.str.contains(KEY_RE)]], ignore_index=True)
if len(SL):
    SL["analysis_status"] = "pre-specified sensitivity analyses, re-run on the post-hoc stratified specification"
    save(SL, "o2_sensitivity_long.csv")
if len(SATT):
    save(SATT, "o2_sensitivity_global_vs_itemised.csv")


# =============================================================================
# 12. EXPLORATORY
# =============================================================================
hdr("11. EXPLORATORY (label as such)")
if RUN_EXTRA and len(ALT):
    try:
        core0 = fit_block(ALT, BLOCKS["M5"], NUMS["M5"], "stratified")
        x0 = make_tab(core0, "X0 " + ALT_LABEL)
        save(x0.drop(columns=["term_full"]), "o2_X0_other_sample_M5.csv")
        k0 = x0[(x0.cluster == "patient") & x0.term.str.contains(KEY_RE)]
        print(f"X0: stratified M5 on {ALT_LABEL} (n = {len(ALT):,}); key terms:")
        print(k0[["stratum", "term", "ratio", "lo", "hi"]].round(3).to_string(index=False))
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    # leave-one-province-out on the Global/itemised ratio and group ratios
    lop = []
    for p in top_prov:
        sub = D[D.province_model != p]
        a = amc_tables(fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp"), sub.reset_index(drop=True), "M5")[1]
        lop.append({"province_left_out": p, "n": len(sub), "ratio_Global_over_itemised": a.ratio_Global_over_itemised.iloc[0]})
    LOP = pd.DataFrame(lop)
    save(LOP, "o2_X_leave_one_province_out.csv")
    print("\nLeave-one-province-out (largest provinces): Global/itemised standardised ratio")
    print(LOP.round(3).to_string(index=False))


# =============================================================================
# 13. RELIABILITY VERDICT
# =============================================================================
hdr("12. RELIABILITY VERDICT (A / B / C) for each stratified M5 estimate")
rows = []
base = make_tab(M5S, "M5")
bp = base[(base.cluster == "patient") & base.term.str.contains(KEY_RE)]
bq = base[(base.cluster == "province")].set_index("term_full")
for _, r in bp.iterrows():
    s = SL[(SL.stratum == r.stratum) & (SL.term == r.term)] if len(SL) else pd.DataFrame()
    s = s[s.sens.notna()] if len(s) else s
    if "stratum" not in SL.columns or not len(s):
        s = pd.DataFrame(columns=["ratio"])
    dev = 100 * np.abs(s.ratio / r.ratio - 1) if len(s) else pd.Series(dtype=float)
    same_dir = bool(((s.ratio > 1) == (r.ratio > 1)).all()) if len(s) else True
    rb = ROB[(ROB.stratum == r.stratum) & (ROB.term == r.term)]
    cook_ch = float(abs(rb.cook_pct_change.iloc[0])) if len(rb) else np.nan
    qp_ch = float(abs(rb.qp_pct_change.iloc[0])) if len(rb) else np.nan
    q = bq.loc[r.term_full]
    sig_p, sig_v = hit(r), hit(q)
    reasons = []
    if r.sparse_flag:
        reasons.append("level has <30 episodes")
    if not same_dir:
        reasons.append("direction changes in a sensitivity analysis")
    if len(dev) and dev.max() > STABLE_PCT:
        reasons.append(f"changes up to {dev.max():.0f}% across sensitivity analyses")
    if cook_ch > COOK_PCT:
        reasons.append(f"Cook refit changes it by {cook_ch:.0f}%")
    if qp_ch > COOK_PCT:
        reasons.append(f"quasi-Poisson changes it by {qp_ch:.0f}%")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    verdict = "C" if (r.sparse_flag or not same_dir) else ("A" if not reasons else "B")
    rows.append({"stratum": r.stratum, "term": r.term, "ratio": r.ratio, "lo": r.lo, "hi": r.hi, "CI_excludes_1": sig_p,
                 "level_n": r.level_n, "n_sens": len(s), "max_pct_change_sens": float(dev.max()) if len(dev) else np.nan,
                 "same_direction_all": same_dir, "cook_pct": cook_ch, "qp_pct": qp_ch,
                 "province_cluster_CI_excludes_1": sig_v, "verdict": verdict, "reasons": "; ".join(reasons)})
# the standardised Global vs itemised contrast
for _, r in ATT.iterrows():
    pop = r.population
    s = SATT[SATT.population == pop] if len(SATT) else pd.DataFrame()
    dev = 100 * np.abs(s.ratio_Global_over_itemised / r.ratio_Global_over_itemised - 1) if len(s) else pd.Series(dtype=float)
    same = bool(((s.ratio_Global_over_itemised > 1) == (r.ratio_Global_over_itemised > 1)).all()) if len(s) else True
    ratio_hit = lambda lo, hi: bool(lo > 1 or hi < 1)
    sp, sv = ratio_hit(r.ratio_lo_patient, r.ratio_hi_patient), ratio_hit(r.ratio_lo_province, r.ratio_hi_province)
    reasons = []
    if r.share_on_common_support < 0.9:
        reasons.append(f"only {100 * r.share_on_common_support:.0f}% of Global episodes on common support")
    if not same:
        reasons.append("direction changes")
    if len(dev) and dev.max() > STABLE_PCT:
        reasons.append(f"changes up to {dev.max():.0f}% across sensitivity analyses")
    if sp != sv:
        reasons.append("conclusion differs between patient and province clusters")
    rows.append({"stratum": "Global vs itemised (ATT)", "term": pop, "ratio": r.ratio_Global_over_itemised,
                 "lo": r.ratio_lo_patient, "hi": r.ratio_hi_patient, "CI_excludes_1": sp, "level_n": r.n_Global,
                 "n_sens": len(s), "max_pct_change_sens": float(dev.max()) if len(dev) else np.nan,
                 "same_direction_all": same, "cook_pct": np.nan, "qp_pct": np.nan, "province_cluster_CI_excludes_1": sv,
                 "verdict": "C" if not same else ("A" if not reasons else "B"), "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o2_reliability_verdict.csv")
print("A = stable under every check; B = direction stable but size or inference depends on the check named; "
      "C = sparse level or direction not stable (do not interpret).")
print(REL[["stratum", "term", "ratio", "lo", "hi", "verdict", "reasons"]].round(2).to_string(index=False))
print("\nVerdict counts:", REL.verdict.value_counts().to_dict())


# =============================================================================
# 14. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)),
        "n_Global": int((D.stratum == "Global").sum()), "n_Other": int((D.stratum == "Other").sum()),
        "rare_min": RARE_MIN, "boot": NBOOT, "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "deviations_from_protocol": ["admissions are the primary sample", "model stratified by billing mode",
                                     "year-month period dummies instead of year + month",
                                     "rare categories collapsed within strata",
                                     "any_surgical_service not estimated in the Global stratum"]}
(OUT_DIR / "o2_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")

In [20]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 3 (patient cost-sharing) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional).
Same engine and conventions as objective2_expenditure.py v2 (stratified by billing mode, year-month dummies,
rare categories collapsed within strata, cluster-robust SEs, A/B/C reliability verdict).

OUTCOME   patient share = insured_share_irr / total_cost_irr, kept only when the episode is RECONCILED
          (financing gap <= 5%, both shares present) and the share lies in [0, 1] (protocol, O3 primary sample).
MODEL     Fractional logit (binomial quasi-likelihood GLM, logit link), blocks M1-M5 as in Objectives 1-2
          (M2 = + fund, ownership [+ billing mode in the pooled comparison]; M3 = + group, surgical service,
          discharge status; M4 = + province; M5 = + specialty). LOS is not a covariate in O3 (a post-admission
          quantity); sensitivity 7 adds it. Average marginal effects (AME) in percentage points by g-computation
          with delta-method CIs (patient clusters AND province clusters).
PRIMARY SAMPLE  Admissions (Global + Inpatient) that are reconciled with a valid share, as in O1 v3 / O2 v2
          (Emergency Observation and Temporary admissions are not admissions). The protocol sample (all reconciled
          episodes) is fitted as exploratory X0. This is a deviation from the frozen protocol: state it.

WHY THE MODEL IS STRATIFIED BY BILLING MODE (same lesson as O2, checked on this file)
          Global episodes have a near-fixed share (median 8%, IQR 6-10%); itemised episodes are spread from 12% to
          76% (median 55%) and depend heavily on ownership and fund. One additive billing-mode term cannot describe
          both. The primary model is therefore fitted separately in the Global and itemised strata (jointly, as a
          block-diagonal fit so contrasts and clustered covariances are valid). The pooled additive M1-M5 is kept
          for comparison only. Rare levels (< 30 episodes inside a stratum) are collapsed; any_surgical_service is
          dropped in a stratum where one level has < 30 episodes.

SECTIONS
  2  Sample flow; reconciliation by group, code, ownership, billing mode
  3  Descriptives: share by group/code/billing/ownership/fund/specialty/province/period; share of episodes above 10/20/40%
  4  Selection: who is reconciled (the primary sample is a self-selected subset)
  5  Models M1-M5 (pooled comparison and stratified primary); main table (odds ratios, AME in pp)
  6  Global vs itemised, standardised (what Global episodes would pay if billed itemised), by block
  7  Diagnostics within strata: link test, Cook + refit, probit link, fit to observed cells; 5-fold out-of-fold error
  8  Hypothesis 2 (provinces): Wald tests, adjusted ratios, province random intercept (sensitivity 8)
  9  Unreconciled episodes: scenarios 1-4 and whether the conclusions hold under scenarios 1 and 4
 10  Sensitivity analyses 1-8
 11  Exploratory (label as such): protocol sample, inverse-probability-of-reconciliation weights, leave-one-province-out
 12  Reliability verdict (A / B / C)

SCENARIO B  The protocol asks for "Scenario B, as defined in the cholera paper". That definition is not in the
          protocol or the analysis file. SCENARIO_B below is a PROVISIONAL stand-in (recorded shares rescaled to
          the recorded financing base: insured / (insured + requested)). Replace the function with the cholera
          paper's definition before reporting scenario 3.

USAGE
  python objective3_cost_sharing.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 200]   (bootstraps use at most 100 replicates)
Outputs: folder objective3_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import expit

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "o3-v1-stratified"
SEED = 42
RARE_MIN = 30
SPARSE_N_MIN = 30
TOP_PROVINCES = 5
COOK_MULT = 4.0
N_FOLDS = 5
THRESHOLDS = (0.10, 0.20, 0.40)      # share of episodes with patient share above these (protocol)
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
STABLE_PP = 2.0                      # an AME is "stable" if it moves < max(STABLE_PP, 25% of |AME|) pp
STABLE_REL = 0.25
TIME_PP_TOL = 2.0

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=200)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective3_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service",
           "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
COLLAPSE_VARS = ["ownership", "specialty_model", "province_model", "insurance_fund_model",
                 "discharge_status_model", "sex", "age_group", "group", "any_surgical_service", "period"]
KEY_VARS = ["group", "ownership", "insurance_fund_model", "any_surgical_service"]     # verdict / sensitivity AMEs
REPORT_VARS = KEY_VARS + ["specialty_model", "province_model", "discharge_status_model", "age_group", "sex"]
Z975 = stats.norm.ppf(0.975)
T0 = time.time()


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE SAMPLES
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "total_cost_irr", "insured_share_irr", "requested_total_irr", "fin_gap",
        "reconciled", "patient_share", "patient_share_gt1", "los_capped_p95", "cluster_id", "any_surgical_service",
        "specialty_model", "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O3_O4",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O3_O4", "reconciled", "patient_share_gt1", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
if "specialty_h3" in DALL.columns:
    DALL["specialty_h3"] = DALL["specialty_h3"].fillna("Missing").astype(str)
for c in ["total_cost_irr", "insured_share_irr", "requested_total_irr", "fin_gap", "patient_share", "los_capped_p95",
          "los_capped_p99"]:
    if c in DALL.columns:
        DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")
DALL["ps_ok"] = DALL["in_O3_O4"] & DALL["patient_share"].between(0, 1)
if (DALL["patient_share"].notna() & ~DALL["reconciled"]).any():
    print("  Note: the file also holds a recorded share for non-reconciled episodes; the primary analysis uses reconciled "
          "episodes only (in_O3_O4), the others enter through scenarios 1-4.")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
valid_total = DALL["total_cost_irr"] > 0

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
adm = DALL[is_adm]
flow += [{"step": "Admissions (Global + Inpatient)", "n": len(adm)},
         {"step": "  with both insured and requested shares present", "n": int((adm.insured_share_irr.notna() & adm.requested_total_irr.notna()).sum())},
         {"step": "  reconciled (gap <= 5%, both shares present)", "n": int(adm.reconciled.sum())},
         {"step": "  reconciled but patient share > 1 (excluded, counted)", "n": int((adm.reconciled & adm.patient_share_gt1).sum())},
         {"step": "O3 primary sample: reconciled admissions with share in [0, 1]", "n": int(adm.ps_ok.sum())}]
D = (DALL[is_adm & DALL.ps_ok] if args.sample == "admissions" else DALL[DALL.ps_ok]).copy().reset_index(drop=True)
ALT = (DALL[DALL.ps_ok] if args.sample == "admissions" else DALL[is_adm & DALL.ps_ok]).copy().reset_index(drop=True)
SAMPLE_LABEL = "reconciled admissions (Global + Inpatient)" if args.sample == "admissions" else "all reconciled episodes (protocol sample)"
ALT_LABEL = "all reconciled episodes (protocol sample)" if args.sample == "admissions" else "reconciled admissions"
flow.append({"step": f"PRIMARY SAMPLE O3: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o3_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary O3 sample.")
print(f"\nPatient share: median {D.patient_share.median():.3f}, IQR {D.patient_share.quantile(.25):.3f}-"
      f"{D.patient_share.quantile(.75):.3f}, mean {D.patient_share.mean():.3f}")

# reconciliation by group, code, ownership, billing mode (primary-sample retention)
adm_all = DALL[is_adm].copy()
rec_rows = []
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("ownership", "Ownership"), ("billing_mode", "Billing mode"),
                 ("province_model", "Province")]:
    for lev, g in adm_all.groupby(var):
        rec_rows.append({"variable": lab, "level": lev, "n_admissions": len(g), "n_reconciled_valid": int(g.ps_ok.sum()),
                         "pct_kept": 100 * g.ps_ok.mean()})
REC = pd.DataFrame(rec_rows)
save(REC, "o3_reconciliation_by_group_code_ownership.csv")
print("\nShare of admissions kept in the O3 primary sample (protocol: report by group and ownership):")
print(REC[REC.variable.isin(["Group", "Ownership", "Billing mode"])].round(1).to_string(index=False))
print("By ICD-10 code:", REC[REC.variable == "ICD-10 code"].set_index("level").pct_kept.round(1).to_dict())


# Common reference levels in BOTH strata (so effects are comparable across strata and do not flip when the sample changes).
def build_refs(d):
    refs = {"ownership": "Public University-Affiliated", "group": "Hernia",
            "insurance_fund_model": "Rural Health Insurance Fund", "sex": "Male", "age_group": "40-59",
            "discharge_status_model": "Complete Recovery", "any_surgical_service": "Yes", "billing_mode": "Global"}
    refs["specialty_model"] = d["specialty_model"].value_counts().index[0]
    ct = pd.crosstab(d["province_model"], d["stratum"])
    ct = ct[~ct.index.isin(["Other", "Missing"])]
    refs["province_model"] = ct.min(axis=1).idxmax()          # largest province present in BOTH strata
    return {k: v for k, v in refs.items() if v in set(d[k].astype(str))}


REFS = build_refs(D)
print("Reference levels (both strata):", REFS)
MIN_SUPPORT = 10          # episodes per level and province needed for a within-province comparison
FAC_VARS = ("ownership", "specialty_model")     # facility-level exposures: compared only within provinces

# =============================================================================
# 2. ENGINE: block-diagonal (stratified) or pooled fractional-logit / fractional-probit GLM
# =============================================================================
def _dummies(series, levels):
    codes = pd.Categorical(series.astype(str), categories=levels).codes
    return (codes[:, None] == np.arange(1, len(levels))[None, :]).astype(float)


def build_spec(d, cats, nums, refs=None):
    refs = refs if refs is not None else {}
    cols, names, levs = [np.ones((len(d), 1))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        cols.append(_dummies(s, levs[v]))
        names += [f"C({v})[T.{lev}]" for lev in levs[v][1:]]
    for v in nums:
        cols.append(d[v].to_numpy(float)[:, None])
        names.append(v)
    X = np.hstack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], {"cats": list(cats), "nums": list(nums), "levs": levs, "keep": keep}


def apply_spec(d, spec):
    cols = [np.ones((len(d), 1))]
    for v in spec["cats"]:
        cols.append(_dummies(d[v], spec["levs"][v]))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float)[:, None])
    return np.hstack(cols)[:, spec["keep"]]


def merge_periods(vc, minn):
    groups = [[p, [p], int(vc[p])] for p in sorted(vc.index)]
    while len(groups) > 1 and min(g[2] for g in groups) < minn:
        i = int(np.argmin([g[2] for g in groups]))
        j = i + 1 if i + 1 < len(groups) else i - 1
        groups[j][1] += groups[i][1]
        groups[j][2] += groups[i][2]
        groups.pop(i)
    return {m: g[0] for g in groups for m in g[1]}


def make_cmap(ds, vars_, minn=RARE_MIN):
    """Collapse levels with < minn episodes inside this stratum (to 'Other', or to the modal level)."""
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        if v == "period":
            m = merge_periods(vc, minn)
        else:
            rare = [l for l in vc.index if vc[l] < minn]
            m = {}
            if rare:
                total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
                tgt = "Other" if (total >= minn and v not in ("any_surgical_service", "group", "sex")) else vc.index[0]
                m = {l: tgt for l in rare}
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def apply_cmap(d, cm):
    d = d.copy()
    for v, m in cm["map"].items():
        s0 = d[v].astype(str)
        s = s0.map(m).fillna(s0) if m else s0
        seen = cm["seen"][v]
        d[v] = s.where(s.isin(seen), "Other" if "Other" in seen else "__unseen__")
    return d


def _link(fam):
    if fam == "probit":
        return (lambda e: stats.norm.cdf(e)), (lambda e, mu: stats.norm.pdf(e))
    return (lambda e: expit(e)), (lambda e, mu: mu * (1 - mu))


def _bdev(y, mu, w):
    mu = np.clip(mu, 1e-12, 1 - 1e-12)
    t1 = np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0.0)
    t2 = np.where(y < 1, (1 - y) * np.log(np.where(y < 1, 1 - y, 1) / (1 - mu)), 0.0)
    return 2 * np.sum(w * (t1 + t2))


def fit_glm(X, y, fam="logit", w=None):
    """Fractional response GLM (binomial quasi-likelihood), IRLS with step-halving on the binomial deviance.
    fam 'logit' (primary) or 'probit' (link-robustness check). Optional case weights w."""
    n, k = X.shape
    w = np.ones(n) if w is None else np.asarray(w, float)
    inv, dmu = _link(fam)
    p0 = np.clip(np.average(y, weights=w), 1e-4, 1 - 1e-4)
    beta = np.zeros(k)
    beta[0] = np.log(p0 / (1 - p0)) if fam == "logit" else stats.norm.ppf(p0)
    eta = X @ beta
    mu = np.clip(inv(eta), 1e-9, 1 - 1e-9)
    d_old = _bdev(y, mu, w)
    conv = False
    for _ in range(200):
        dm = np.maximum(dmu(eta, mu), 1e-12)
        V = mu * (1 - mu)
        W = w * dm ** 2 / V
        z = eta + (y - mu) / dm
        A, rhs = (X.T * W) @ X, (X.T * W) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        step = 1.0
        for _h in range(30):
            cand = beta + step * (b_new - beta)
            eta_c = np.clip(X @ cand, -30, 30)
            mu_c = np.clip(inv(eta_c), 1e-9, 1 - 1e-9)
            d_new = _bdev(y, mu_c, w)
            if np.isfinite(d_new) and d_new <= d_old + 1e-10 * abs(d_old):
                break
            step /= 2
        done = np.abs(cand - beta).max() < 1e-9 or abs(d_old - d_new) < 1e-12 * max(abs(d_old), 1)
        beta, eta, mu, d_old = cand, eta_c, mu_c, d_new
        if done:
            conv = True
            break
    dm = np.maximum(dmu(eta, mu), 1e-12)
    V = mu * (1 - mu)
    W = w * dm ** 2 / V
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * W) @ X), "conv": conv, "W": W, "dm": dm,
            "V": V, "w": w, "phi": float(np.sum(w * (y - mu) ** 2 / V) / max(w.sum() - k, 1))}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups):
    n, k = X.shape
    s = f["w"] * (y - f["mu"]) * f["dm"] / f["V"]
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_block(d, cats, nums, mode="stratified", fam="logit", cov=True, minn=RARE_MIN, weights=None):
    d = d.reset_index(drop=True)
    wts = None if weights is None else np.asarray(weights, float)
    if mode == "pooled":
        parts = {"All": np.ones(len(d), bool)}
    else:
        parts = {s: (d["stratum"] == s).to_numpy() for s in ["Global", "Other"] if (d["stratum"] == s).sum() >= 50}
    blocks, Xs, off = {}, [], 0
    for s, rows in parts.items():
        ds = d.loc[rows]
        cs = [c for c in cats if not (mode == "stratified" and c == "billing_mode")]
        cs = list(dict.fromkeys(cs + ["period"]))
        cm = make_cmap(ds, [v for v in cs if v in COLLAPSE_VARS], minn)
        dm_ = apply_cmap(ds, cm)
        refs = {**REFS, "period": min(dm_["period"].unique())}
        X_s, nm_s, spec = build_spec(dm_, cs, nums, refs)
        blocks[s] = {"rows": np.where(rows)[0], "cols": slice(off, off + X_s.shape[1]), "names": nm_s, "spec": spec,
                     "cmap": cm, "X": X_s,
                     "lev_n": [int(X_s[:, j].sum()) if nm.startswith("C(") else np.nan for j, nm in enumerate(nm_s)]}
        off += X_s.shape[1]
        Xs.append((rows, X_s))
    K = off
    y = d["patient_share"].to_numpy(float)
    beta, eta, mu = np.zeros(K), np.zeros(len(d)), np.zeros(len(d))
    W, dm, V, wv = np.zeros(len(d)), np.zeros(len(d)), np.zeros(len(d)), np.ones(len(d))
    bread, conv = np.zeros((K, K)), True
    for b in blocks.values():
        r = b["rows"]
        fb = fit_glm(b["X"], y[r], fam, None if wts is None else wts[r])
        beta[b["cols"]], eta[r], mu[r], W[r], dm[r], V[r], wv[r] = fb["beta"], fb["eta"], fb["mu"], fb["W"], fb["dm"], fb["V"], fb["w"]
        bread[b["cols"], b["cols"]] = fb["bread"]
        b["phi"] = fb["phi"]
        conv &= fb["conv"]
    f = {"beta": beta, "eta": eta, "mu": mu, "bread": bread, "W": W, "dm": dm, "V": V, "w": wv}
    X = None
    if cov:
        X = np.zeros((len(d), K))
        for (rows, X_s), b in zip(Xs, blocks.values()):
            X[np.ix_(np.where(rows)[0], np.arange(b["cols"].start, b["cols"].stop))] = X_s
    allnames = [(f"{s}|{nm}" if mode == "stratified" else nm) for s, b in blocks.items() for nm in b["names"]]
    lev_n = [x for b in blocks.values() for x in b["lev_n"]]
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl]) for cl in groups} if cov else {}
    return {"mode": mode, "fam": fam, "n": len(d), "names": allnames, "lev_n": lev_n, "beta": beta, "covs": covs, "X": X,
            "y": y, "mu": mu, "eta": eta, "W": W, "dm": dm, "V": V, "w": wv, "phi": {s: b["phi"] for s, b in blocks.items()},
            "conv": conv, "blocks": blocks, "d": d, "K": K, "groups": groups}


def make_tab(core, model, label="patient_share"):
    out = []
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = core["beta"] / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term_full": core["names"], "n": core["n"],
                          "level_n": core["lev_n"], "odds_ratio": np.exp(core["beta"]),
                          "lo": np.exp(core["beta"] - crit * se), "hi": np.exp(core["beta"] + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        if core["mode"] == "stratified":
            t["stratum"] = t.term_full.str.split("|", n=1).str[0]
            t["term"] = t.term_full.str.split("|", n=1).str[1]
        else:
            t["stratum"], t["term"] = "All", t["term_full"]
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[(t["term"] != "Intercept") & ~t["term"].str.startswith("C(period)")])
    return pd.concat(out, ignore_index=True)


def crit_p(core, cl, est, se):
    G = core["covs"][cl][1]
    if cl == "province":
        return stats.t.ppf(0.975, G - 1), (2 * stats.t.sf(abs(est / se), G - 1) if se > 0 else np.nan)
    return Z975, (2 * stats.norm.sf(abs(est / se)) if se > 0 else np.nan)


# --- prediction, margins (g-computation) with gradients ----------------------
def pred_part(core, s, dpop, over=None):
    b = core["blocks"][s]
    dm_ = apply_cmap(dpop, b["cmap"])
    for k, v in (over or {}).items():
        dm_[k] = v
    X = apply_spec(dm_, b["spec"])
    eta = np.clip(X @ core["beta"][b["cols"]], -30, 30)
    inv, dmu = _link(core["fam"])
    mu = inv(eta)
    return mu, dmu(eta, mu), X


def margin(core, s, dpop, over=None):
    mu, dm_, X = pred_part(core, s, dpop, over)
    g = np.zeros(core["K"])
    g[core["blocks"][s]["cols"]] = (dm_ @ X) / len(mu)
    return float(mu.mean()), g


def predict_rows(core, dte):
    mu = np.full(len(dte), np.nan)
    for s, b in core["blocks"].items():
        m = (dte["stratum"] == s).to_numpy() if core["mode"] == "stratified" else np.ones(len(dte), bool)
        if m.any():
            mu[m] = pred_part(core, s, dte[m])[0]
    return mu


def est_se(core, grad):
    return {cl: float(np.sqrt(max(grad @ core["covs"][cl][0] @ grad, 0))) for cl in core["covs"]}


def ame_table(core, d, vars_, model):
    """AME in percentage points of each level vs the stratum reference (g-computation, delta-method CIs).
    Facility-level exposures (ownership, specialty) are compared only WITHIN provinces where both the level and the
    reference occur with >= MIN_SUPPORT episodes (otherwise the effect is extrapolated into provinces where the level
    never occurs and cannot be separated from the province). If no province qualifies the AME is not identified (NaN)."""
    rows = []
    d = d.reset_index(drop=True)
    for s, b in core["blocks"].items():
        pop = d[(d["stratum"] == s).to_numpy()] if core["mode"] == "stratified" else d
        dpc = apply_cmap(pop, b["cmap"])
        for v in vars_:
            if v not in b["spec"]["levs"]:
                continue
            levs = b["spec"]["levs"][v]
            ref = levs[0]
            ct = pd.crosstab(pop["province_model"].to_numpy(), dpc[v].to_numpy()) if v in FAC_VARS else None
            m0_all, g0_all = margin(core, s, pop, {v: ref})
            for lev in levs[1:]:
                nm = f"C({v})[T.{lev}]"
                if nm not in b["names"]:
                    continue
                m1_all, _ = margin(core, s, pop, {v: lev})
                pr, n_prov, n_obs = pop, np.nan, b["lev_n"][b["names"].index(nm)]
                if v in FAC_VARS:
                    okp = ct.index[(ct[lev] >= MIN_SUPPORT) & (ct[ref] >= MIN_SUPPORT)] if (lev in ct and ref in ct) else []
                    n_prov = len(okp)
                    pr = pop[pop["province_model"].isin(okp)]
                    n_obs = int((dpc[v] == lev)[pop["province_model"].isin(okp).to_numpy()].sum())
                row = {"model": model, "stratum": s, "variable": v, "level": lev, "reference": ref,
                       "level_n": b["lev_n"][b["names"].index(nm)], "n_support_provinces": n_prov,
                       "n_support_episodes": n_obs, "AME_unrestricted_pp": 100 * (m1_all - m0_all)}
                if len(pr) < 30 or n_obs < 1:
                    row.update({"AME_pp": np.nan, "mean_ref_pp": np.nan, "sparse_flag": True,
                                "note": "not identified within any province"})
                    rows.append(row)
                    continue
                m0, g0 = margin(core, s, pr, {v: ref})
                m1, g1 = margin(core, s, pr, {v: lev})
                a_, gr = 100 * (m1 - m0), 100 * (g1 - g0)
                row.update({"AME_pp": a_, "mean_ref_pp": 100 * m0, "note": ""})
                for cl, se in est_se(core, gr).items():
                    c, p = crit_p(core, cl, a_, se)
                    row[f"lo_{cl}"], row[f"hi_{cl}"], row[f"p_{cl}"] = a_ - c * se, a_ + c * se, p
                row["sparse_flag"] = bool(row["level_n"] < SPARSE_N_MIN or (v in FAC_VARS and n_obs < SPARSE_N_MIN))
                rows.append(row)
    return pd.DataFrame(rows)


def att_table(core, d, model):
    """Global episodes: patient share as billed (Global model) vs if billed itemised (itemised model), standardised.
    Primary rows use only Global episodes on COMMON SUPPORT (province, specialty, fund and ownership levels with >= 30
    episodes in the itemised stratum); the unrestricted row is kept for transparency."""
    d = d.reset_index(drop=True)
    popG0 = d[d["stratum"] == "Global"]
    if "Global" not in core["blocks"] or "Other" not in core["blocks"]:
        return pd.DataFrame()
    O = d[d["stratum"] == "Other"]
    ok = np.ones(len(popG0), bool)
    for v in ["province_model", "specialty_model", "insurance_fund_model", "ownership"]:
        ok &= (popG0[v].map(O[v].value_counts()).fillna(0) >= RARE_MIN).to_numpy()
    popG = popG0[ok]
    rows = []
    for lab, base, g in [("All", popG, "All")] + [(g, popG, g) for g in sorted(d.group.unique())] + \
            [("All, unrestricted", popG0, "All")]:
        pop = base if g == "All" else base[base.group == g]
        if len(pop) < 20:
            continue
        m1, g1 = margin(core, "Global", pop)
        m2, g2 = margin(core, "Other", pop)
        row = {"model": model, "population": f"Global episodes ({lab})", "n_Global": len(pop),
               "share_of_Global_on_support": len(popG) / len(popG0),
               "observed_Global_mean_pp": 100 * pop.patient_share.mean(), "fitted_Global_pp": 100 * m1,
               "if_itemised_pp": 100 * m2, "diff_pp": 100 * (m1 - m2), "ratio_Global_over_itemised": m1 / m2}
        for cl, se in est_se(core, 100 * (g1 - g2)).items():
            c, p = crit_p(core, cl, 100 * (m1 - m2), se)
            row[f"diff_lo_{cl}"], row[f"diff_hi_{cl}"], row[f"diff_p_{cl}"] = 100 * (m1 - m2) - c * se, 100 * (m1 - m2) + c * se, p
        rows.append(row)
    return pd.DataFrame(rows)


def hit(lo, hi, null=0.0):
    return bool(lo > null or hi < null)


# =============================================================================
# 3. DESCRIPTIVES AND THRESHOLDS
# =============================================================================
hdr("2. DESCRIPTIVES (patient share, %)")


def cl_prop(x, cl):
    """Cluster-robust proportion and 95% CI."""
    x = np.asarray(x, float)
    p = x.mean()
    cid = pd.factorize(cl)[0]
    s = pd.Series(x - p).groupby(cid).sum().to_numpy()
    se = np.sqrt((len(s) / (len(s) - 1)) * np.sum(s ** 2)) / len(x) if len(s) > 1 else np.nan
    return p, p - Z975 * se, p + Z975 * se


def cstats(g):
    s = 100 * g["patient_share"]
    row = {"n": len(g), "mean": s.mean(), "sd": s.std(), "median": s.median(), "p25": s.quantile(.25), "p75": s.quantile(.75),
           "median_IQR": f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})"}
    for t in THRESHOLDS:
        p, lo, hi = cl_prop(g["patient_share"] > t, g["cluster_id"])
        row[f"pct_gt_{int(t * 100)}"] = 100 * p
        row[f"pct_gt_{int(t * 100)}_lo"], row[f"pct_gt_{int(t * 100)}_hi"] = 100 * lo, 100 * hi
    return row


desc = [{"variable": "PRIMARY SAMPLE", "stratum": "All", "level": "All", **cstats(D)}]
for s, g in D.groupby("stratum"):
    desc.append({"variable": "Billing mode", "stratum": s, "level": s, **cstats(g)})
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("ownership", "Ownership"),
                 ("insurance_fund_model", "Fund"), ("specialty_model", "Specialty"), ("province_model", "Province"),
                 ("period", "Period (Jalali year-month)"), ("any_surgical_service", "Surgical service")]:
    for (s, lev), g in D.groupby(["stratum", var]):
        desc.append({"variable": lab, "stratum": s, "level": lev, **cstats(g)})
    for lev, g in D.groupby(var):
        desc.append({"variable": lab, "stratum": "All", "level": lev, **cstats(g)})
DESC = pd.DataFrame(desc)
save(DESC, "o3_descriptives_and_thresholds.csv")
cols = ["variable", "stratum", "level", "n", "median_IQR", "mean", "pct_gt_10", "pct_gt_20", "pct_gt_40"]
print(DESC[DESC.variable.isin(["PRIMARY SAMPLE", "Billing mode"])][cols].round(1).to_string(index=False))
print("\nBy group x billing mode:")
print(DESC[(DESC.variable == "Group") & (DESC.stratum != "All")][cols].round(1).to_string(index=False))
print("\nBy ownership x billing mode:")
print(DESC[(DESC.variable == "Ownership") & (DESC.stratum != "All")][cols].round(1).to_string(index=False))

# =============================================================================
# 4. SELECTION: who is reconciled?
# =============================================================================
hdr("3. SELECTION INTO THE PRIMARY SAMPLE (reconciled episodes)")
adm_sel = DALL[is_adm & valid_total].reset_index(drop=True).copy()
adm_sel["rec_y"] = adm_sel["ps_ok"].astype(float)
SEL_CATS = ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service", "province_model",
            "billing_mode"]
cm_sel = make_cmap(adm_sel, [v for v in SEL_CATS + ["period"] if v in COLLAPSE_VARS])
dsel = apply_cmap(adm_sel, cm_sel)
Xsel, nsel, specsel = build_spec(dsel, SEL_CATS + ["period"], [], {**REFS, "period": min(dsel.period.unique())})
fsel = fit_glm(Xsel, adm_sel["rec_y"].to_numpy(), "logit")
covsel, Gsel = cov_robust(Xsel, adm_sel["rec_y"].to_numpy(), fsel, pd.factorize(adm_sel.cluster_id)[0])
sesel = np.sqrt(np.clip(np.diag(covsel), 0, None))
SEL = pd.DataFrame({"term": nsel, "odds_ratio_reconciled": np.exp(fsel["beta"]), "lo": np.exp(fsel["beta"] - Z975 * sesel),
                    "hi": np.exp(fsel["beta"] + Z975 * sesel)})
SEL = SEL[(SEL.term != "Intercept") & ~SEL.term.str.startswith("C(period)") & ~SEL.term.str.startswith("C(province")]
# AUC
pr, yy = fsel["mu"], adm_sel["rec_y"].to_numpy()
ranks = stats.rankdata(pr)
auc = (ranks[yy == 1].sum() - (yy == 1).sum() * ((yy == 1).sum() + 1) / 2) / ((yy == 1).sum() * (yy == 0).sum())
save(SEL, "o3_selection_model.csv")
print(f"Admissions with valid total cost: {len(adm_sel):,}; reconciled with share in [0,1]: {int(yy.sum()):,} "
      f"({100 * yy.mean():.1f}%); selection model AUC = {auc:.3f}")
print(SEL.round(3).to_string(index=False))
cmp_rows = []
for var in ["group", "billing_mode", "ownership", "insurance_fund_model", "any_surgical_service", "age_group", "adm_year"]:
    t = pd.crosstab(adm_sel[var], adm_sel["ps_ok"], normalize="columns").mul(100)
    t.columns = ["pct_among_not_reconciled", "pct_among_reconciled"]
    t = t.reset_index().rename(columns={var: "level"})
    t.insert(0, "variable", var)
    cmp_rows.append(t)
SELC = pd.concat(cmp_rows)
SELC["difference_pp"] = SELC.pct_among_reconciled - SELC.pct_among_not_reconciled
save(SELC, "o3_reconciled_vs_not.csv")
print("\nReconciled vs not (column %):")
print(SELC.round(1).to_string(index=False))
print("A material imbalance means the primary sample is a selected subset: that is why scenarios 1-4 (section 9) and the "
      "inverse-probability-weighted analysis (X1) are reported.")
PROB_REC = pd.Series(fsel["mu"], index=adm_sel.episode_no.values)

# What the unreconciled episodes look like, and whether reconciliation is ignorable given covariates
adm_sel["ins_share"] = adm_sel.insured_share_irr / adm_sel.total_cost_irr
adm_sel["req_share"] = (adm_sel.requested_total_irr / adm_sel.total_cost_irr).clip(0, 1)
adm_sel["unaccounted"] = (adm_sel.total_cost_irr - adm_sel.insured_share_irr - adm_sel.requested_total_irr) / adm_sel.total_cost_irr
adm_sel["reconciled_valid"] = np.where(adm_sel.ps_ok, "reconciled", "not reconciled")
MONEY = adm_sel.groupby(["stratum", "reconciled_valid"]).agg(
    n=("episode_no", "size"), insured_share_mean=("ins_share", "mean"), requested_share_mean=("req_share", "mean"),
    unaccounted_mean=("unaccounted", "mean"), unaccounted_median=("unaccounted", "median"),
    share_unaccounted_gt_50pct=("unaccounted", lambda x: (x > .5).mean()),
    share_unaccounted_negative=("unaccounted", lambda x: (x < -.05).mean())).reset_index()
save(MONEY, "o3_money_decomposition_by_reconciliation.csv")
print("\nShare of total cost recorded as insured / requested / unaccounted, by billing mode and reconciliation:")
print(MONEY.round(3).to_string(index=False))
print("Unreconciled episodes always have money MISSING (total > insured + requested, never the reverse); where it went is "
      "not recorded. Their recorded insured share is therefore a lower bound, and 1 - requested share an upper bound.")
ign = []
for s_ in ["Global", "Other"]:
    ds = adm_sel[adm_sel.stratum == s_].reset_index(drop=True)
    cats = BLOCKS["M5"] + ["period"]
    cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
    dm_ = apply_cmap(ds, cm)
    Xa, nma, _ = build_spec(dm_, cats, [], {**REFS, "period": min(dm_.period.unique())})
    ya = ds.req_share.to_numpy()
    X2 = np.column_stack([Xa, ds.ps_ok.to_numpy(float)])
    fa = fit_glm(X2, ya, "logit")
    cva, _ = cov_robust(X2, ya, fa, pd.factorize(ds.cluster_id)[0])
    b_, se_ = fa["beta"][-1], np.sqrt(cva[-1, -1])
    lin = X2[:, :-1] @ fa["beta"][:-1]
    ame_ = 100 * (expit(lin + b_) - expit(lin)).mean()
    ign.append({"stratum": s_, "n": len(ds), "insurer_share_reconciled_pct": 100 * ya[ds.ps_ok.to_numpy()].mean(),
                "insurer_share_not_reconciled_pct": 100 * ya[~ds.ps_ok.to_numpy()].mean(),
                "adjusted_difference_pp": ame_, "logit_coef": b_, "se": se_})
IGN = pd.DataFrame(ign)
save(IGN, "o3_reconciliation_ignorability.csv")
print("\nInsurer's share of the bill (observed for ALL admissions): reconciled vs not, adjusted for the M5 covariates.")
print(IGN.round(3).to_string(index=False))
print("A large adjusted difference means unreconciled episodes differ in how the bill is financed even among comparable "
      "episodes, so the primary sample cannot be assumed representative (compare the scenarios).")

# =============================================================================
# 5. MODELS M1-M5
# =============================================================================
hdr("4. MODELS M1-M5 (fractional logit; patient and province clusters)")


def blk_cats(m, pooled):
    c = list(BLOCKS[m])
    return c + ["billing_mode"] if pooled and m != "M1" else c


CORE_P, CORE_S, TABS = {}, {}, []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cp = fit_block(D, blk_cats(m, True), [], "pooled")
    cs = fit_block(D, blk_cats(m, False), [], "stratified")
    CORE_P[m], CORE_S[m] = cp, cs
    TABS += [make_tab(cp, m).assign(spec="pooled additive (comparison only)"),
             make_tab(cs, m).assign(spec="stratified by billing mode (PRIMARY)")]
    print(f"  {m}: pooled terms {len(cp['names']) - 1} (quasi-binomial dispersion {cp['phi']['All']:.3f}) | stratified terms "
          f"{len(cs['names']) - len(cs['blocks'])}, dispersion " + ", ".join(f"{s} {v:.3f}" for s, v in cs["phi"].items())
          + ("" if cs["conv"] and cp["conv"] else "  ** did not converge **"))
LONG = pd.concat(TABS, ignore_index=True)
save(LONG, "o3_models_long_oddsratios.csv")
M5S, M5P = CORE_S["M5"], CORE_P["M5"]
prov_n = D.province_model.value_counts()
top_prov = prov_n.index[:TOP_PROVINCES].tolist()

AME_ALL = ame_table(M5S, D, REPORT_VARS, "M5 stratified")
save(AME_ALL, "o3_ame_M5_stratified_pp.csv")

# identifiability of facility-level terms: are the levels spread over provinces, or confined to a few?
idr = []
for s_ in ["Global", "Other"]:
    ds = D[D.stratum == s_]
    for v in FAC_VARS:
        for lev, g in ds.groupby(v):
            vc = g.province_model.value_counts()
            a = AME_ALL[(AME_ALL.stratum == s_) & (AME_ALL.variable == v) & (AME_ALL.level == lev)]
            idr.append({"stratum": s_, "variable": v, "level": lev, "n": len(g), "n_provinces": g.province_model.nunique(),
                        "top_province": vc.index[0], "top_province_share_pct": 100 * vc.iloc[0] / len(g),
                        "support_provinces_vs_reference": float(a.n_support_provinces.iloc[0]) if len(a) else np.nan,
                        "support_episodes": float(a.n_support_episodes.iloc[0]) if len(a) else np.nan,
                        "concentrated": bool(g.province_model.nunique() <= 3 or 100 * vc.iloc[0] / len(g) > 70 or
                                             (len(a) and np.isfinite(a.n_support_provinces.iloc[0]) and a.n_support_provinces.iloc[0] <= 2))})
IDENT = pd.DataFrame(idr)
save(IDENT, "o3_facility_identifiability.csv")
print("\nIdentifiability of facility-level exposures (ownership, specialty): a level confined to a few provinces cannot be "
      "separated from the province or hospital.")
print(IDENT[IDENT.variable == "ownership"].round(1).to_string(index=False))
IDENT_FLAG = {(r.stratum, r.variable, r.level): r.concentrated for r in IDENT.itertuples()}


def fmt_ame(r, cl):
    star = " *" if hit(r[f"lo_{cl}"], r[f"hi_{cl}"]) else ""
    sp = " [n<30]" if r.sparse_flag else ""
    return f"{r.AME_pp:+.1f} ({r[f'lo_{cl}']:+.1f} to {r[f'hi_{cl}']:+.1f}){star}{sp}"


mt = AME_ALL[AME_ALL.variable.isin(KEY_VARS + ["specialty_model"]) |
             ((AME_ALL.variable == "province_model") & AME_ALL.level.isin(top_prov))].copy()
mt["AME_patient_CI"] = mt.apply(lambda r: fmt_ame(r, "patient"), axis=1)
mt["AME_province_CI"] = mt.apply(lambda r: fmt_ame(r, "province"), axis=1)
orm = make_tab(M5S, "M5")
orm = orm[orm.cluster == "patient"].set_index(["stratum", "term"])
mt["odds_ratio"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "odds_ratio"] for r in mt.itertuples()]
mt["OR_lo"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "lo"] for r in mt.itertuples()]
mt["OR_hi"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "hi"] for r in mt.itertuples()]
save(mt, "o3_main_table_M5_stratified.csv")
print("\nMAIN TABLE (stratified M5): average marginal effect on patient share, percentage points vs the stratum reference.\n"
      "Patient-cluster and province-cluster (t, G-1 df) 95% CIs; * = CI excludes 0. Facility-level terms (ownership,\n"
      "specialty, province) are best read with the province-cluster CI.")
print(mt[["stratum", "variable", "level", "reference", "level_n", "mean_ref_pp", "AME_patient_CI", "AME_province_CI"]]
      .round(1).to_string(index=False))

pb = AME_ALL.iloc[0:0]
bm = []
for m in ["M2", "M3", "M4", "M5"]:
    a = ame_table(CORE_P[m], D, ["billing_mode"], f"{m} pooled additive")
    bm.append(a)
BM = pd.concat(bm)
save(BM, "o3_pooled_billing_AME_by_block.csv")
print("\nPooled ADDITIVE billing-mode AME (itemised vs Global, pp) [v1-type estimate; one constant effect]:")
print(BM[["model", "AME_pp", "lo_patient", "hi_patient"]].round(2).to_string(index=False))

# =============================================================================
# 6. GLOBAL vs ITEMISED, STANDARDISED
# =============================================================================
hdr("5. GLOBAL vs ITEMISED, standardised on the Global episodes")
ATT = att_table(M5S, D, "M5")
ATT_BLOCKS = pd.concat([att_table(CORE_S[m], D, m).query("population == 'Global episodes (All)'") for m in CORE_S])


def boot_att(d, B, key):
    cl = d[key].astype(str).to_numpy()
    uc, inv = np.unique(cl, return_inverse=True)
    idx_by = [np.where(inv == i)[0] for i in range(len(uc))]
    out = []
    for _ in range(B):
        pick = RNG.integers(0, len(uc), len(uc))
        db = d.iloc[np.concatenate([idx_by[i] for i in pick])].reset_index(drop=True)
        try:
            cb = fit_block(db, BLOCKS["M5"], [], "stratified", cov=False)
            out.append(att_table(cb, db, "b").set_index("population")["diff_pp"])
        except Exception:
            continue
    return pd.concat(out, axis=1)


NB = min(NBOOT, 100)
for key, tag in [("cluster_id", "patient"), ("province_model", "province")]:
    bt = boot_att(D, NB, key)
    q = bt.quantile([.025, .975], axis=1).T
    ATT[f"boot_lo_{tag}"] = [q.loc[p_, 0.025] if p_ in q.index else np.nan for p_ in ATT.population]
    ATT[f"boot_hi_{tag}"] = [q.loc[p_, 0.975] if p_ in q.index else np.nan for p_ in ATT.population]
save(ATT, "o3_global_vs_itemised_standardised_M5.csv")
save(ATT_BLOCKS, "o3_global_vs_itemised_by_block.csv")
print("Patient share (pp) of the Global episodes as billed vs if billed itemised (itemised model applied to them):")
print(ATT[["population", "n_Global", "observed_Global_mean_pp", "if_itemised_pp", "diff_pp", "boot_lo_patient", "boot_hi_patient",
           "boot_lo_province", "boot_hi_province"]].round(1).to_string(index=False))
print("\nAcross blocks (delta-method CI, patient clusters):")
print(ATT_BLOCKS[["model", "fitted_Global_pp", "if_itemised_pp", "diff_pp", "diff_lo_patient", "diff_hi_patient"]].round(1).to_string(index=False))

# fit to observed cell means (fractional logit reproduces them for categorical covariates)
chk = []
for s in ["Global", "Other"]:
    for var in ["group", "ownership"]:
        for lev, g in D[D.stratum == s].groupby(var):
            if len(g) < 30:
                continue
            fitted_s = pred_part(M5S, s, g)[0].mean()
            row = {"stratum": s, "variable": var, "level": lev, "n": len(g), "observed_pp": 100 * g.patient_share.mean(),
                   "stratified_fitted_pp": 100 * fitted_s}
            if var in M5P["blocks"]["All"]["spec"]["levs"]:
                row["pooled_additive_fitted_pp"] = 100 * pred_part(M5P, "All", g)[0].mean()
            chk.append(row)
CHK = pd.DataFrame(chk)
CHK["pooled_error_pp"] = CHK.pooled_additive_fitted_pp - CHK.observed_pp
CHK["stratified_error_pp"] = CHK.stratified_fitted_pp - CHK.observed_pp
save(CHK, "o3_fit_to_observed_cells.csv")
print("\nFitted vs observed cell means (pp): the stratified model reproduces them; the pooled additive does not:")
print(CHK.round(2).to_string(index=False))

# =============================================================================
# 7. DIAGNOSTICS AND OUT-OF-FOLD ERROR
# =============================================================================
hdr("6. DIAGNOSTICS within strata (link test, Cook + refit, probit link) and 5-fold out-of-fold error")
diag = []
cook_flag_all = np.zeros(len(M5S["d"]), bool)
for s, b in M5S["blocks"].items():
    r = b["rows"]
    Xs_, ys = b["X"], M5S["y"][r]
    mu, eta, W = M5S["mu"][r], M5S["eta"][r], M5S["W"][r]
    pat = M5S["groups"]["patient"][r]
    Xl = np.column_stack([Xs_, eta ** 2])
    fl = fit_glm(Xl, ys, "logit")
    covl, _ = cov_robust(Xl, ys, fl, pat)
    tl = fl["beta"][-1] / np.sqrt(covl[-1, -1])
    sw = np.sqrt(W)
    Z = Xs_ * sw[:, None]
    h = np.einsum("ij,jk,ik->i", Z, np.linalg.pinv(Z.T @ Z), Z)
    rp = (ys - mu) / np.sqrt(M5S["V"][r])
    k = Xs_.shape[1]
    cook = rp ** 2 * h / (k * b["phi"] * (1 - h) ** 2)
    flag = cook > COOK_MULT / len(r)
    cook_flag_all[r[flag]] = True
    diag.append({"stratum": s, "n": len(r), "terms": k, "dispersion": b["phi"], "link_t": float(tl),
                 "link_p": float(2 * stats.norm.sf(abs(tl))), "max_cook": float(cook.max()), "n_cook_flag": int(flag.sum()),
                 "max_leverage": float(h.max()), "n_leverage_gt_0.2": int((h > .2).sum()),
                 "share_exact_zero": float((ys == 0).mean())})
DIAG = pd.DataFrame(diag)
save(DIAG, "o3_diagnostics_by_stratum.csv")
print(DIAG.round(3).to_string(index=False))
print("A significant link test means some structure in the mean is missing (check the cell fit and out-of-fold calibration "
      "below for the practical size of the problem).")

M5C = fit_block(M5S["d"][~cook_flag_all], BLOCKS["M5"], [], "stratified", cov=False)
M5Pr = fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit", cov=False)
ame_c = ame_table(fit_block(M5S["d"][~cook_flag_all], BLOCKS["M5"], [], "stratified"), M5S["d"][~cook_flag_all], KEY_VARS, "cook")
ame_p = ame_table(fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit"), D, KEY_VARS, "probit")
key = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].copy()
for nm, a in [("cook_excl", ame_c), ("probit", ame_p)]:
    key = key.merge(a[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": f"AME_{nm}"}),
                    on=["stratum", "variable", "level"], how="left")
    key[f"{nm}_change_pp"] = key[f"AME_{nm}"] - key["AME_pp"]
save(key, "o3_robustness_cook_probit.csv")
print(f"\nCook refit excludes {int(cook_flag_all.sum())} episodes (> {COOK_MULT:g}/n within stratum); probit link refit. "
      f"AME (pp) of key terms:")
print(key.round(2).to_string(index=False))
atp = att_table(fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit", cov=False), D, "probit").iloc[0]
atc = att_table(M5C, M5C["d"], "cook").iloc[0]
print(f"\nGlobal vs itemised (diff, pp): logit {ATT.diff_pp.iloc[0]:.1f} | probit {atp.diff_pp:.1f} | Cook-excluded {atc.diff_pp:.1f}")

# out-of-fold
ids = pd.factorize(D.cluster_id)[0]
uid = np.unique(ids)
np.random.default_rng(SEED).shuffle(uid)
fm = dict(zip(uid, np.arange(len(uid)) % N_FOLDS))
FOLD = np.array([fm[i] for i in ids])
oof_rows, cal_rows = [], []
for m in ["M1", "M3", "M5"]:
    for mode in ["pooled", "stratified"]:
        pred = np.full(len(D), np.nan)
        for k in range(N_FOLDS):
            tr, te = D[FOLD != k], D[FOLD == k]
            core = fit_block(tr, blk_cats(m, mode == "pooled"), [], mode, cov=False)
            pred[FOLD == k] = np.clip(predict_rows(core, te), 0, 1)
        y = D.patient_share.to_numpy()
        for s in ["All", "Global", "Other"]:
            mk = np.ones(len(D), bool) if s == "All" else (D.stratum == s).to_numpy()
            e = y[mk] - pred[mk]
            oof_rows.append({"model": m, "spec": mode, "stratum": s, "n": int(mk.sum()), "MAE_pp": 100 * np.abs(e).mean(),
                             "RMSE_pp": 100 * np.sqrt((e ** 2).mean()), "mean_observed_pp": 100 * y[mk].mean(),
                             "mean_predicted_pp": 100 * pred[mk].mean(), "calibration_diff_pp": 100 * (pred[mk].mean() - y[mk].mean()),
                             "R2_oof": 1 - np.sum(e ** 2) / np.sum((y[mk] - y[mk].mean()) ** 2)})
            if m == "M5":
                q = pd.qcut(pd.Series(pred[mk]).rank(method="first"), 10, labels=False)
                c = pd.DataFrame({"decile": q + 1, "pred": 100 * pred[mk], "obs": 100 * y[mk]}).groupby("decile").mean().reset_index()
                c["model"], c["spec"], c["stratum"] = m, mode, s
                cal_rows.append(c.rename(columns={"pred": "mean_predicted_pp", "obs": "mean_observed_pp"}))
OOF, CAL = pd.DataFrame(oof_rows), pd.concat(cal_rows, ignore_index=True)
save(OOF, "o3_oof_error.csv")
save(CAL, "o3_oof_calibration_deciles.csv")
print("\n" + OOF.round(2).to_string(index=False))
c5 = CAL[(CAL.model == "M5") & (CAL.stratum == "All")].copy()
c5["diff_pp"] = c5.mean_predicted_pp - c5.mean_observed_pp
print("\nM5 calibration by predicted decile, all admissions (predicted minus observed, pp):")
print(c5.pivot(index="decile", columns="spec", values="diff_pp").round(2).to_string())

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for s, col in [("Global", "C0"), ("Other", "C3")]:
        ax[0].hist(100 * D.loc[D.stratum == s, "patient_share"], bins=50, alpha=.6, color=col, label=s, density=True)
    ax[0].set(title="Patient share by billing mode", xlabel="Patient share (%)", ylabel="Density")
    ax[0].legend(frameon=False)
    for k, spec in enumerate(["pooled", "stratified"]):
        for s, col in [("Global", "C0"), ("Other", "C3")]:
            c8 = CAL[(CAL.model == "M5") & (CAL.stratum == s) & (CAL.spec == spec)]
            ax[1 + k].plot(c8.mean_predicted_pp, c8.mean_observed_pp, "o-", color=col, label=s)
        ax[1 + k].plot([0, 100], [0, 100], "k--", lw=.8)
        ax[1 + k].set(title=f"M5 {spec}: out-of-fold calibration", xlabel="Mean predicted (%)", ylabel="Mean observed (%)")
        ax[1 + k].legend(frameon=False, title="billing mode")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o3_distribution_and_calibration.png", dpi=200)
    plt.close(fig)
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")

# =============================================================================
# 8. HYPOTHESIS 2: provinces
# =============================================================================
hdr("7. HYPOTHESIS 2: patient share differs across provinces after case-mix adjustment")


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


h2 = []
for s in ["Global", "Other"]:
    sub = D[D.stratum == s]
    kw = stats.kruskal(*[g.patient_share.values for _, g in sub.groupby("province_model") if len(g) >= 5])
    h2.append({"model": "crude (Kruskal-Wallis)", "stratum": s, "df": np.nan, "chi2": float(kw.statistic), "p": float(kw.pvalue)})
for m in ["M4", "M5"]:
    core = CORE_S[m]
    for s, b in core["blocks"].items():
        idx = [b["cols"].start + j for j, nm in enumerate(b["names"]) if nm.startswith("C(province_model)")]
        if idx:
            chi2, df, p = wald(core, idx)
            h2.append({"model": m, "stratum": s, "df": df, "chi2": chi2, "p": p,
                       "note": "Wald, patient-clustered; province-level SE not valid (28 clusters)"})
H2 = pd.DataFrame(h2)
save(H2, "o3_h2_province_tests.csv")
print(H2.round(4).to_string(index=False))
pm = AME_ALL[(AME_ALL.variable == "province_model") & ~AME_ALL.sparse_flag]
save(pm, "o3_h2_province_AME_M5.csv")
for s in ["Global", "Other"]:
    x = pm[pm.stratum == s].sort_values("AME_pp")
    ref = x.reference.iloc[0]
    print(f"\n{s}: adjusted province AME (pp) vs {ref}; extremes (modelled provinces with >= 30 episodes):")
    print(x[["level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].iloc[[0, 1, 2, -3, -2, -1]].round(1).to_string(index=False))


class LaplaceFrac:
    """Fractional-logit with a province random intercept, Laplace-approximated pseudo-ML (Bernoulli kernel)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def negll(self, theta):
        beta, sig = theta[:self.k], np.exp(theta[self.k])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            eta = xb + u[self.g]
            p = expit(eta)
            d1, d2 = self.y - p, -p * (1 - p)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        eta = xb + u[self.g]
        p = expit(eta)
        h = np.bincount(self.g, p * (1 - p), self.G) + 1 / sig ** 2
        ll = np.sum(self.y * eta - np.logaddexp(0, eta))
        return -(ll - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0):
        th0 = np.concatenate([beta0, [np.log(0.3)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()


def fit_ri(d, label):
    s8, vc8 = [], []
    for s in ["Global", "Other"]:
        ds = d[d.stratum == s].reset_index(drop=True)
        if len(ds) < 200:
            continue
        t0 = time.time()
        try:
            cats = [v for v in BLOCKS["M5"] if v != "province_model"] + ["period"]
            cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
            dm_ = apply_cmap(ds, cm)
            X8, nm8, _ = build_spec(dm_, cats, [], {**REFS, "period": min(dm_["period"].unique())})
            y8 = ds.patient_share.to_numpy(float)
            g8 = pd.factorize(ds.province_model)[0]
            f0 = fit_glm(X8, y8, "logit")
            m8 = LaplaceFrac(X8, y8, g8)
            theta = m8.fit(f0["beta"])
            H = m8.hessian(theta)
            se8 = np.sqrt(np.clip(np.diag(np.linalg.pinv(H)), 0, None))
            k8 = m8.k
            s8.append(pd.DataFrame({"outcome": "patient_share", "model": "M5 random intercept", "stratum": s, "term": nm8[1:],
                                    "n": len(ds), "level_n": [int(X8[:, j].sum()) if nm8[j].startswith("C(") else np.nan for j in range(1, k8)],
                                    "odds_ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                                    "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]), "sens": 8, "sens_name": label}))
            sig = float(np.exp(theta[k8]))
            vc8.append({"stratum": s, "n": len(ds), "provinces": m8.G, "sigma_u": sig,
                        "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])), "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                        "median_odds_ratio": float(np.exp(0.6745 * np.sqrt(2) * sig)),
                        "note": "pseudo-likelihood: SE of sigma_u approximate"})
            print(f"  {s}: n = {len(ds):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (median odds ratio between two random "
                  f"provinces {np.exp(0.6745 * np.sqrt(2) * sig):.2f}), {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  RI ({s}) FAILED ({type(e).__name__}: {e})")
    return (pd.concat(s8, ignore_index=True) if s8 else pd.DataFrame()), (pd.DataFrame(vc8) if vc8 else pd.DataFrame())


if RUN_SENS8:
    print("\nProvince random intercept, within each stratum (= sensitivity analysis 8):")
    S8, VC8 = fit_ri(D, "Province random intercept")
    if len(VC8):
        save(VC8, "o3_sens8_variance_components.csv")
        save(S8, "o3_sens8_random_intercept_long.csv")
        print(VC8.round(3).to_string(index=False))

# =============================================================================
# 9. UNRECONCILED EPISODES: SCENARIOS 1-4
# =============================================================================
hdr("8. UNRECONCILED EPISODES: scenarios for the patient share (all admissions with a valid total cost)")


def scenario_b(d):
    """PROVISIONAL stand-in for the cholera paper's Scenario B (not defined in the protocol or the analysis file):
    recorded shares rescaled to the recorded financing base, insured / (insured + requested). Episodes without both
    shares keep the recorded lower bound. REPLACE with the cholera paper's definition before reporting."""
    base = d["insured_share_irr"] + d["requested_total_irr"]
    ps = (d["insured_share_irr"] / base).where(base > 0)
    return ps.fillna(d["ps_scen1"]).clip(0, 1)


SC = DALL[is_adm & valid_total].copy().reset_index(drop=True)
tot = SC["total_cost_irr"]
SC["ps_scen1"] = (SC["insured_share_irr"] / tot).clip(0, 1)
isrec = SC["ps_ok"].to_numpy()
own_mean = SC[isrec].groupby("ownership")["patient_share"].mean()
own_bill_mean = SC[isrec].groupby(["ownership", "stratum"])["patient_share"].mean()
SC["ps_scen2"] = np.where(isrec, SC["patient_share"], SC["ownership"].map(own_mean))
SC["ps_scen2b"] = np.where(isrec, SC["patient_share"],
                           [own_bill_mean.get((o, s), own_mean.get(o, np.nan)) for o, s in zip(SC.ownership, SC.stratum)])
SC["ps_scen3"] = np.where(isrec, SC["patient_share"], scenario_b(SC))
upper = np.fmax((tot - SC["requested_total_irr"]) / tot, SC["ps_scen1"]).clip(0, 1)
SC["ps_scen4"] = np.where(isrec, SC["patient_share"], upper)
if "ps_scen3_upper_bound" in DALL.columns:
    diff_up = (SC["ps_scen4"] - DALL.set_index("episode_no").loc[SC.episode_no, "ps_scen3_upper_bound"].to_numpy())
    print(f"  Check: upper bound recomputed vs file column: max |difference| = {np.nanmax(np.abs(diff_up)):.2e}")
SCEN = {"1 recorded lower bound": "ps_scen1", "2 ownership-group mean": "ps_scen2",
        "2b ownership x billing-mode mean (exploratory)": "ps_scen2b",
        "3 'Scenario B' (PROVISIONAL stand-in)": "ps_scen3", "4 upper bound": "ps_scen4"}
sc_rows = []
for lab, col in SCEN.items():
    for stratum_lab, mk in [("All admissions", np.ones(len(SC), bool)), ("Global", (SC.stratum == "Global").to_numpy()),
                            ("Itemised", (SC.stratum == "Other").to_numpy())]:
        x = SC.loc[mk, col]
        okx = x.notna().to_numpy()
        p, lo, hi = cl_prop(x[okx], SC.loc[mk, "cluster_id"][okx])
        sc_rows.append({"scenario": lab, "population": stratum_lab, "n": int(okx.sum()), "mean_share_pct": 100 * p,
                        "lo": 100 * lo, "hi": 100 * hi,
                        "pct_gt_20": 100 * (x[okx] > .2).mean(), "pct_gt_40": 100 * (x[okx] > .4).mean()})
SCS = pd.DataFrame(sc_rows)
sc_rec = SC.loc[isrec, "patient_share"]
SCS = pd.concat([pd.DataFrame([{"scenario": "Reconciled episodes only (primary sample)", "population": "All admissions",
                                "n": int(isrec.sum()), "mean_share_pct": 100 * sc_rec.mean(),
                                "lo": 100 * cl_prop(sc_rec, SC.loc[isrec, "cluster_id"])[1],
                                "hi": 100 * cl_prop(sc_rec, SC.loc[isrec, "cluster_id"])[2],
                                "pct_gt_20": 100 * (sc_rec > .2).mean(), "pct_gt_40": 100 * (sc_rec > .4).mean()}]), SCS])
save(SCS, "o3_scenarios_pooled_mean.csv")
print("Pooled mean patient share under each scenario (reconciled episodes keep their observed share in every scenario):")
print(SCS.round(1).to_string(index=False))
rg = []
for lab, col in SCEN.items():
    for var in ["group", "ownership"]:
        for lev, g in SC.groupby(var):
            rg.append({"scenario": lab, var if False else "variable": var, "level": lev, "n": int(g[col].notna().sum()),
                       "mean_share_pct": 100 * g[col].mean()})
save(pd.DataFrame(rg), "o3_scenarios_by_group_ownership.csv")

# do the conclusions hold under scenarios 1 and 4 (and the others)?
print("\nConclusions under each scenario (stratified M5 refitted on all admissions, scenario share as outcome):")
conc_rows, att_rows = [], []
for lab, col in [("0 primary (reconciled only)", None)] + list(SCEN.items()):
    if col is None:
        dd, ame_s, att_s = D, AME_ALL[AME_ALL.variable.isin(KEY_VARS)].assign(scenario=lab), ATT.assign(scenario=lab)
    else:
        dd = SC.copy()
        dd["patient_share"] = dd[col]
        dd = dd[dd["patient_share"].notna()].reset_index(drop=True)
        cr = fit_block(dd, BLOCKS["M5"], [], "stratified")
        ame_s = ame_table(cr, dd, KEY_VARS, lab).assign(scenario=lab)
        att_s = att_table(cr, dd, lab).assign(scenario=lab)
    conc_rows.append(ame_s)
    att_rows.append(att_s)
SCAME = pd.concat(conc_rows, ignore_index=True)
SCATT = pd.concat(att_rows, ignore_index=True)
save(SCAME, "o3_scenarios_AME_key_terms.csv")
save(SCATT, "o3_scenarios_global_vs_itemised.csv")
base_ame = AME_ALL[AME_ALL.variable.isin(KEY_VARS)].set_index(["stratum", "variable", "level"]).AME_pp
hold = []
for (s_, v_, l_), a0 in base_ame.items():
    row = {"stratum": s_, "variable": v_, "level": l_, "primary_AME_pp": a0}
    ok_all = True
    for lab in SCEN:
        x = SCAME[(SCAME.scenario == lab) & (SCAME.stratum == s_) & (SCAME.variable == v_) & (SCAME.level == l_)]
        a1 = float(x.AME_pp.iloc[0]) if len(x) else np.nan
        row[f"AME_scenario_{lab.split()[0]}"] = a1
    for lab in ["1 recorded lower bound", "4 upper bound"]:
        a1 = row[f"AME_scenario_{lab.split()[0]}"]
        row[f"same_sign_scen{lab.split()[0]}"] = bool(np.sign(a1) == np.sign(a0)) if np.isfinite(a1) else np.nan
    row["holds_under_1_and_4"] = bool(row["same_sign_scen1"] and row["same_sign_scen4"]) if np.isfinite(row["AME_scenario_1"]) and np.isfinite(row["AME_scenario_4"]) else np.nan
    hold.append(row)
HOLD = pd.DataFrame(hold)
save(HOLD, "o3_scenario_conclusions_hold.csv")
print(HOLD.round(2).to_string(index=False))
a1 = SCATT[(SCATT.population == "Global episodes (All)")][["scenario", "observed_Global_mean_pp", "if_itemised_pp", "diff_pp"]]
print("\nGlobal vs itemised (standardised on Global episodes) under each scenario:")
print(a1.round(1).to_string(index=False))
print("PROTOCOL RULE: the primary conclusions must hold under scenarios 1 and 4, otherwise the Discussion must say they do not "
      "(see 'holds_under_1_and_4'; ownership effects are partly built in under scenario 2).")


# =============================================================================
# 10. SENSITIVITY ANALYSES 1-8
# =============================================================================
hdr("9. SENSITIVITY ANALYSES 1-8 (stratified M5; AME in pp)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"), 2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"), 4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS (P95 cap) added as a covariate", None)}
SL, SATT = pd.DataFrame(), pd.DataFrame()
if RUN_SENS:
    sl, sa = [], []
    for s_id, (name, keepcol) in SENS.items():
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        nums = []
        if s_id == 7:
            sub["los_cov"] = sub["los_capped_p95"]
            sub = sub[sub.los_cov.notna()]
            nums = ["los_cov"]
        core = fit_block(sub, BLOCKS["M5"], nums, "stratified")
        sl.append(ame_table(core, sub, KEY_VARS, name).assign(sens=s_id, sens_name=name))
        if len(core["blocks"]) == 2:
            sa.append(att_table(core, sub, name).assign(sens=s_id, sens_name=name))
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}, strata = {list(core['blocks'])}")
    # sensitivity 9-10 (added after the O3 diagnostics): the 5% reconciliation tolerance is fixed by the protocol, but the
    # results move with it, so wider tolerances are reported. Patient share is the recorded insured / total in all of them.
    adm_t = DALL[is_adm].copy()
    both_t = adm_t.insured_share_irr.notna() & adm_t.requested_total_irr.notna()
    for s_id, tol in [(9, 0.10), (10, 0.20)]:
        sub = adm_t[both_t & (adm_t.fin_gap <= tol) & adm_t.patient_share.between(0, 1)].reset_index(drop=True)
        name = f"Reconciliation tolerance {int(tol * 100)}% (instead of 5%)"
        core = fit_block(sub, BLOCKS["M5"], [], "stratified")
        sl.append(ame_table(core, sub, KEY_VARS, name).assign(sens=s_id, sens_name=name))
        sa.append(att_table(core, sub, name).assign(sens=s_id, sens_name=name))
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}")
    SL = pd.concat(sl, ignore_index=True)
    SATT = pd.concat(sa, ignore_index=True) if sa else pd.DataFrame()
    save(SL, "o3_sensitivity_AME.csv")
    if len(SATT):
        save(SATT, "o3_sensitivity_global_vs_itemised.csv")
    sdir = []
    for r in AME_ALL[AME_ALL.variable.isin(KEY_VARS)].itertuples():
        x = SL[(SL.stratum == r.stratum) & (SL.variable == r.variable) & (SL.level == r.level)]
        if len(x):
            sdir.append({"stratum": r.stratum, "variable": r.variable, "level": r.level, "main_AME_pp": r.AME_pp,
                         "n_sens": len(x), "same_sign": int((np.sign(x.AME_pp) == np.sign(r.AME_pp)).sum()),
                         "max_abs_change_pp": float((x.AME_pp - r.AME_pp).abs().max())})
    save(pd.DataFrame(sdir), "o3_sensitivity_direction.csv")
    print(pd.DataFrame(sdir).round(2).to_string(index=False))

# =============================================================================
# 11. EXPLORATORY
# =============================================================================
hdr("10. EXPLORATORY (label as such)")
IPW_AME = pd.DataFrame()
if RUN_EXTRA:
    try:
        core0 = fit_block(ALT, BLOCKS["M5"], [], "stratified")
        x0 = ame_table(core0, ALT, KEY_VARS, "X0 " + ALT_LABEL)
        save(x0, "o3_X0_other_sample_AME.csv")
        print(f"X0: stratified M5 on {ALT_LABEL} (n = {len(ALT):,}); key AME (pp):")
        print(x0[["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].round(1).to_string(index=False))
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    try:
        pr_ = PROB_REC.reindex(D.episode_no.values).to_numpy()
        w = 1 / np.clip(pr_, np.quantile(pr_, 0.01), 1)
        w = w / w.mean()
        coreW = fit_block(D, BLOCKS["M5"], [], "stratified", weights=w)
        IPW_AME = ame_table(coreW, D, KEY_VARS, "X1 inverse-probability-of-reconciliation weights")
        save(IPW_AME, "o3_X1_ipw_AME.csv")
        cmpw = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp"]].merge(
            IPW_AME[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": "AME_ipw"}), on=["stratum", "variable", "level"])
        cmpw["change_pp"] = cmpw.AME_ipw - cmpw.AME_pp
        print("\nX1: weighting reconciled episodes by 1/P(reconciled) (observable selection only), key AME (pp):")
        print(cmpw.round(2).to_string(index=False))
        print("(Weights correct only for selection on the covariates in the selection model, not on unobserved factors.)")
    except Exception as e:
        print(f"X1 failed: {type(e).__name__}: {e}")
    lop = []
    for p in top_prov:
        sub = D[D.province_model != p]
        a = att_table(fit_block(sub, BLOCKS["M5"], [], "stratified", cov=False), sub, "M5").iloc[0]
        lop.append({"province_left_out": p, "n": len(sub), "diff_pp_Global_minus_itemised": a.diff_pp})
    LOP = pd.DataFrame(lop)
    save(LOP, "o3_X_leave_one_province_out.csv")
    print("\nLeave-one-province-out (largest provinces): standardised Global-minus-itemised difference (pp)")
    print(LOP.round(2).to_string(index=False))

# =============================================================================
# 12. RELIABILITY VERDICT
# =============================================================================
hdr("11. RELIABILITY VERDICT (A / B / C) for the key AME estimates")
rows = []
for r in AME_ALL[AME_ALL.variable.isin(KEY_VARS)].itertuples():
    x = SL[(SL.stratum == r.stratum) & (SL.variable == r.variable) & (SL.level == r.level)] if len(SL) else pd.DataFrame()
    dev = (x.AME_pp - r.AME_pp).abs() if len(x) else pd.Series(dtype=float)
    tol = max(STABLE_PP, STABLE_REL * abs(r.AME_pp))
    same = bool((np.sign(x.AME_pp) == np.sign(r.AME_pp)).all()) if len(x) else True
    ri_same, ri_note = True, ""
    if len(S8):
        t8 = S8[(S8.stratum == r.stratum) & (S8.term == f"C({r.variable})[T.{r.level}]")]
        if len(t8):
            orm_ = float(orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "odds_ratio"])
            ri_same = bool((t8.odds_ratio.iloc[0] > 1) == (orm_ > 1))
    kk = key[(key.stratum == r.stratum) & (key.variable == r.variable) & (key.level == r.level)]
    cook_ch = float(abs(kk.cook_excl_change_pp.iloc[0])) if len(kk) else np.nan
    prob_ch = float(abs(kk.probit_change_pp.iloc[0])) if len(kk) else np.nan
    sig_p, sig_v = hit(r.lo_patient, r.hi_patient), hit(r.lo_province, r.hi_province)
    conc = bool(IDENT_FLAG.get((r.stratum, r.variable, r.level), False))
    not_ident = not np.isfinite(r.AME_pp)
    hh = HOLD[(HOLD.stratum == r.stratum) & (HOLD.variable == r.variable) & (HOLD.level == r.level)]
    scen_ok = bool(hh.holds_under_1_and_4.iloc[0]) if len(hh) and pd.notna(hh.holds_under_1_and_4.iloc[0]) else True
    reasons = []
    if r.sparse_flag:
        reasons.append("level has <30 episodes")
    if not same:
        reasons.append("sign changes in a sensitivity analysis")
    if not ri_same:
        reasons.append("sign differs with province random intercept")
    if len(dev) and dev.max() > tol:
        reasons.append(f"moves up to {dev.max():.1f} pp across sensitivity analyses")
    if cook_ch > tol:
        reasons.append(f"Cook refit moves it {cook_ch:.1f} pp")
    if prob_ch > tol:
        reasons.append(f"probit link moves it {prob_ch:.1f} pp")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    if not scen_ok:
        reasons.append("sign differs under scenario 1 or 4")
    if conc:
        reasons.append("level confined to <=3 provinces, >70% in one, or compared with the reference in <=2 provinces: hard to separate from province/hospital")
    if not_ident:
        reasons.append("not identified within any province")
    verdict = "C" if (r.sparse_flag or not same or not ri_same or not scen_ok or not_ident) else ("A" if not reasons else "B")
    rows.append({"stratum": r.stratum, "variable": r.variable, "level": r.level, "level_n": r.level_n, "AME_pp": r.AME_pp,
                 "lo_patient": r.lo_patient, "hi_patient": r.hi_patient, "CI_excludes_0": sig_p,
                 "lo_province": r.lo_province, "hi_province": r.hi_province, "province_CI_excludes_0": sig_v,
                 "max_change_sens_pp": float(dev.max()) if len(dev) else np.nan, "tolerance_pp": tol,
                 "cook_change_pp": cook_ch, "probit_change_pp": prob_ch, "verdict": verdict, "reasons": "; ".join(reasons)})
for r in ATT.itertuples():
    pop = r.population
    x = SATT[SATT.population == pop] if len(SATT) else pd.DataFrame()
    dev = (x.diff_pp - r.diff_pp).abs() if len(x) else pd.Series(dtype=float)
    tol = max(STABLE_PP, STABLE_REL * abs(r.diff_pp))
    same = bool((np.sign(x.diff_pp) == np.sign(r.diff_pp)).all()) if len(x) else True
    s1 = SCATT[(SCATT.population == pop) & SCATT.scenario.isin(["1 recorded lower bound", "4 upper bound"])]
    scen_same = bool((np.sign(s1.diff_pp) == np.sign(r.diff_pp)).all()) if len(s1) else True
    sig_p = hit(r.boot_lo_patient, r.boot_hi_patient)
    sig_v = hit(r.boot_lo_province, r.boot_hi_province)
    reasons = []
    if not same:
        reasons.append("sign changes in a sensitivity analysis")
    if len(dev) and dev.max() > tol:
        reasons.append(f"moves up to {dev.max():.1f} pp across sensitivity analyses")
    if not scen_same:
        reasons.append("sign differs under scenario 1 or 4")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    rows.append({"stratum": "Global vs itemised (standardised)", "variable": "billing mode", "level": pop, "level_n": r.n_Global,
                 "AME_pp": r.diff_pp, "lo_patient": r.boot_lo_patient, "hi_patient": r.boot_hi_patient, "CI_excludes_0": sig_p,
                 "lo_province": r.boot_lo_province, "hi_province": r.boot_hi_province, "province_CI_excludes_0": sig_v,
                 "max_change_sens_pp": float(dev.max()) if len(dev) else np.nan, "tolerance_pp": tol, "cook_change_pp": np.nan,
                 "probit_change_pp": np.nan, "verdict": "C" if (not same or not scen_same) else ("A" if not reasons else "B"),
                 "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o3_reliability_verdict.csv")
print("A = stable under every check; B = sign stable but size or inference depends on the check named; "
      "C = sparse level, or sign not stable (sensitivity analyses, random intercept, scenarios 1 and 4): do not interpret.")
print(REL[["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient", "verdict", "reasons"]].round(1).to_string(index=False))
print("\nVerdict counts:", REL.verdict.value_counts().to_dict())

# =============================================================================
# 13. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)),
        "n_Global": int((D.stratum == "Global").sum()), "n_Other": int((D.stratum == "Other").sum()),
        "rare_min": RARE_MIN, "boot": NBOOT, "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scenario_B": "PROVISIONAL stand-in: insured/(insured+requested); replace with the cholera paper definition",
        "deviations_from_protocol": ["reconciled ADMISSIONS (Global + Inpatient) are the primary sample",
                                     "model stratified by billing mode",
                                     "year-month period dummies instead of year + month",
                                     "rare categories collapsed within strata",
                                     "sensitivity 7 adds LOS (P95 cap) as a covariate because LOS is not in the O3 model"]}
(OUT_DIR / "o3_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")


1. LOAD AND VERIFY   [elapsed 0.0 min]
Rows: 15,601; columns: 70
  Note: the file also holds a recorded share for non-reconciled episodes; the primary analysis uses reconciled episodes only (in_O3_O4), the others enter through scenarios 1-4.
                                                         step     n
                         Episodes in the frozen analysis file 15601
                                           record type Global  8462
                                        record type Inpatient  6223
                            record type Emergency Observation   889
                    record type Temporary Inpatient Admission    27
                              Admissions (Global + Inpatient) 14685
               with both insured and requested shares present 14685
                  reconciled (gap <= 5%, both shares present)  6950
         reconciled but patient share > 1 (excluded, counted)     0
O3 primary sample: reconciled admissions with share in [0, 1]  6950
PRIMARY S

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 3 (patient cost-sharing) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional).
Same engine and conventions as objective2_expenditure.py v2 (stratified by billing mode, year-month dummies,
rare categories collapsed within strata, cluster-robust SEs, A/B/C reliability verdict).

OUTCOME   patient share = insured_share_irr / total_cost_irr, kept only when the episode is RECONCILED
          (financing gap <= 5%, both shares present) and the share lies in [0, 1] (protocol, O3 primary sample).
MODEL     Fractional logit (binomial quasi-likelihood GLM, logit link), blocks M1-M5 as in Objectives 1-2
          (M2 = + fund, ownership [+ billing mode in the pooled comparison]; M3 = + group, surgical service,
          discharge status; M4 = + province; M5 = + specialty). LOS is not a covariate in O3 (a post-admission
          quantity); sensitivity 7 adds it. Average marginal effects (AME) in percentage points by g-computation
          with delta-method CIs (patient clusters AND province clusters).
PRIMARY SAMPLE  Admissions (Global + Inpatient) that are reconciled with a valid share, as in O1 v3 / O2 v2
          (Emergency Observation and Temporary admissions are not admissions). The protocol sample (all reconciled
          episodes) is fitted as exploratory X0. This is a deviation from the frozen protocol: state it.

WHY THE MODEL IS STRATIFIED BY BILLING MODE (same lesson as O2, checked on this file)
          Global episodes have a near-fixed share (median 8%, IQR 6-10%); itemised episodes are spread from 12% to
          76% (median 55%) and depend heavily on ownership and fund. One additive billing-mode term cannot describe
          both. The primary model is therefore fitted separately in the Global and itemised strata (jointly, as a
          block-diagonal fit so contrasts and clustered covariances are valid). The pooled additive M1-M5 is kept
          for comparison only. Rare levels (< 30 episodes inside a stratum) are collapsed; any_surgical_service is
          dropped in a stratum where one level has < 30 episodes.

SECTIONS
  2  Sample flow; reconciliation by group, code, ownership, billing mode
  3  Descriptives: share by group/code/billing/ownership/fund/specialty/province/period; share of episodes above 10/20/40%
  4  Selection: who is reconciled (the primary sample is a self-selected subset)
  5  Models M1-M5 (pooled comparison and stratified primary); main table (odds ratios, AME in pp)
  6  Global vs itemised, standardised (what Global episodes would pay if billed itemised), by block
  7  Diagnostics within strata: link test, Cook + refit, probit link, fit to observed cells; 5-fold out-of-fold error
  8  Hypothesis 2 (provinces): Wald tests, adjusted ratios, province random intercept (sensitivity 8)
  9  Unreconciled episodes: scenarios 1-4 and whether the conclusions hold under scenarios 1 and 4
 10  Sensitivity analyses 1-8
 11  Exploratory (label as such): protocol sample, inverse-probability-of-reconciliation weights, leave-one-province-out
 12  Reliability verdict (A / B / C)

SCENARIO B  The protocol asks for "Scenario B, as defined in the cholera paper". That definition is not in the
          protocol or the analysis file. SCENARIO_B below is a PROVISIONAL stand-in (recorded shares rescaled to
          the recorded financing base: insured / (insured + requested)). Replace the function with the cholera
          paper's definition before reporting scenario 3.

USAGE
  python objective3_cost_sharing.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 200]   (bootstraps use at most 100 replicates)
Outputs: folder objective3_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import expit

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "o3-v1-stratified"
SEED = 42
RARE_MIN = 30
SPARSE_N_MIN = 30
TOP_PROVINCES = 5
COOK_MULT = 4.0
N_FOLDS = 5
THRESHOLDS = (0.10, 0.20, 0.40)      # share of episodes with patient share above these (protocol)
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
STABLE_PP = 2.0                      # an AME is "stable" if it moves < max(STABLE_PP, 25% of |AME|) pp
STABLE_REL = 0.25
TIME_PP_TOL = 2.0

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=200)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective3_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service",
           "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
COLLAPSE_VARS = ["ownership", "specialty_model", "province_model", "insurance_fund_model",
                 "discharge_status_model", "sex", "age_group", "group", "any_surgical_service", "period"]
KEY_VARS = ["group", "ownership", "insurance_fund_model", "any_surgical_service"]     # verdict / sensitivity AMEs
REPORT_VARS = KEY_VARS + ["specialty_model", "province_model", "discharge_status_model", "age_group", "sex"]
Z975 = stats.norm.ppf(0.975)
T0 = time.time()


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE SAMPLES
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "total_cost_irr", "insured_share_irr", "requested_total_irr", "fin_gap",
        "reconciled", "patient_share", "patient_share_gt1", "los_capped_p95", "cluster_id", "any_surgical_service",
        "specialty_model", "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O3_O4",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O3_O4", "reconciled", "patient_share_gt1", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
if "specialty_h3" in DALL.columns:
    DALL["specialty_h3"] = DALL["specialty_h3"].fillna("Missing").astype(str)
for c in ["total_cost_irr", "insured_share_irr", "requested_total_irr", "fin_gap", "patient_share", "los_capped_p95",
          "los_capped_p99"]:
    if c in DALL.columns:
        DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")
DALL["ps_ok"] = DALL["in_O3_O4"] & DALL["patient_share"].between(0, 1)
if (DALL["patient_share"].notna() & ~DALL["reconciled"]).any():
    print("  Note: the file also holds a recorded share for non-reconciled episodes; the primary analysis uses reconciled "
          "episodes only (in_O3_O4), the others enter through scenarios 1-4.")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
valid_total = DALL["total_cost_irr"] > 0

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
adm = DALL[is_adm]
flow += [{"step": "Admissions (Global + Inpatient)", "n": len(adm)},
         {"step": "  with both insured and requested shares present", "n": int((adm.insured_share_irr.notna() & adm.requested_total_irr.notna()).sum())},
         {"step": "  reconciled (gap <= 5%, both shares present)", "n": int(adm.reconciled.sum())},
         {"step": "  reconciled but patient share > 1 (excluded, counted)", "n": int((adm.reconciled & adm.patient_share_gt1).sum())},
         {"step": "O3 primary sample: reconciled admissions with share in [0, 1]", "n": int(adm.ps_ok.sum())}]
D = (DALL[is_adm & DALL.ps_ok] if args.sample == "admissions" else DALL[DALL.ps_ok]).copy().reset_index(drop=True)
ALT = (DALL[DALL.ps_ok] if args.sample == "admissions" else DALL[is_adm & DALL.ps_ok]).copy().reset_index(drop=True)
SAMPLE_LABEL = "reconciled admissions (Global + Inpatient)" if args.sample == "admissions" else "all reconciled episodes (protocol sample)"
ALT_LABEL = "all reconciled episodes (protocol sample)" if args.sample == "admissions" else "reconciled admissions"
flow.append({"step": f"PRIMARY SAMPLE O3: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o3_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary O3 sample.")
print(f"\nPatient share: median {D.patient_share.median():.3f}, IQR {D.patient_share.quantile(.25):.3f}-"
      f"{D.patient_share.quantile(.75):.3f}, mean {D.patient_share.mean():.3f}")

# reconciliation by group, code, ownership, billing mode (primary-sample retention)
adm_all = DALL[is_adm].copy()
rec_rows = []
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("ownership", "Ownership"), ("billing_mode", "Billing mode"),
                 ("province_model", "Province")]:
    for lev, g in adm_all.groupby(var):
        rec_rows.append({"variable": lab, "level": lev, "n_admissions": len(g), "n_reconciled_valid": int(g.ps_ok.sum()),
                         "pct_kept": 100 * g.ps_ok.mean()})
REC = pd.DataFrame(rec_rows)
save(REC, "o3_reconciliation_by_group_code_ownership.csv")
print("\nShare of admissions kept in the O3 primary sample (protocol: report by group and ownership):")
print(REC[REC.variable.isin(["Group", "Ownership", "Billing mode"])].round(1).to_string(index=False))
print("By ICD-10 code:", REC[REC.variable == "ICD-10 code"].set_index("level").pct_kept.round(1).to_dict())


# Common reference levels in BOTH strata (so effects are comparable across strata and do not flip when the sample changes).
def build_refs(d):
    refs = {"ownership": "Public University-Affiliated", "group": "Hernia",
            "insurance_fund_model": "Rural Health Insurance Fund", "sex": "Male", "age_group": "40-59",
            "discharge_status_model": "Complete Recovery", "any_surgical_service": "Yes", "billing_mode": "Global"}
    refs["specialty_model"] = d["specialty_model"].value_counts().index[0]
    ct = pd.crosstab(d["province_model"], d["stratum"])
    ct = ct[~ct.index.isin(["Other", "Missing"])]
    refs["province_model"] = ct.min(axis=1).idxmax()          # largest province present in BOTH strata
    return {k: v for k, v in refs.items() if v in set(d[k].astype(str))}


REFS = build_refs(D)
print("Reference levels (both strata):", REFS)
MIN_SUPPORT = 10          # episodes per level and province needed for a within-province comparison
FAC_VARS = ("ownership", "specialty_model")     # facility-level exposures: compared only within provinces

# =============================================================================
# 2. ENGINE: block-diagonal (stratified) or pooled fractional-logit / fractional-probit GLM
# =============================================================================
def _dummies(series, levels):
    codes = pd.Categorical(series.astype(str), categories=levels).codes
    return (codes[:, None] == np.arange(1, len(levels))[None, :]).astype(float)


def build_spec(d, cats, nums, refs=None):
    refs = refs if refs is not None else {}
    cols, names, levs = [np.ones((len(d), 1))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        cols.append(_dummies(s, levs[v]))
        names += [f"C({v})[T.{lev}]" for lev in levs[v][1:]]
    for v in nums:
        cols.append(d[v].to_numpy(float)[:, None])
        names.append(v)
    X = np.hstack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], {"cats": list(cats), "nums": list(nums), "levs": levs, "keep": keep}


def apply_spec(d, spec):
    cols = [np.ones((len(d), 1))]
    for v in spec["cats"]:
        cols.append(_dummies(d[v], spec["levs"][v]))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float)[:, None])
    return np.hstack(cols)[:, spec["keep"]]


def merge_periods(vc, minn):
    groups = [[p, [p], int(vc[p])] for p in sorted(vc.index)]
    while len(groups) > 1 and min(g[2] for g in groups) < minn:
        i = int(np.argmin([g[2] for g in groups]))
        j = i + 1 if i + 1 < len(groups) else i - 1
        groups[j][1] += groups[i][1]
        groups[j][2] += groups[i][2]
        groups.pop(i)
    return {m: g[0] for g in groups for m in g[1]}


def make_cmap(ds, vars_, minn=RARE_MIN):
    """Collapse levels with < minn episodes inside this stratum (to 'Other', or to the modal level)."""
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        if v == "period":
            m = merge_periods(vc, minn)
        else:
            rare = [l for l in vc.index if vc[l] < minn]
            m = {}
            if rare:
                total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
                tgt = "Other" if (total >= minn and v not in ("any_surgical_service", "group", "sex")) else vc.index[0]
                m = {l: tgt for l in rare}
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def apply_cmap(d, cm):
    d = d.copy()
    for v, m in cm["map"].items():
        s0 = d[v].astype(str)
        s = s0.map(m).fillna(s0) if m else s0
        seen = cm["seen"][v]
        d[v] = s.where(s.isin(seen), "Other" if "Other" in seen else "__unseen__")
    return d


def _link(fam):
    if fam == "probit":
        return (lambda e: stats.norm.cdf(e)), (lambda e, mu: stats.norm.pdf(e))
    return (lambda e: expit(e)), (lambda e, mu: mu * (1 - mu))


def _bdev(y, mu, w):
    mu = np.clip(mu, 1e-12, 1 - 1e-12)
    t1 = np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0.0)
    t2 = np.where(y < 1, (1 - y) * np.log(np.where(y < 1, 1 - y, 1) / (1 - mu)), 0.0)
    return 2 * np.sum(w * (t1 + t2))


def fit_glm(X, y, fam="logit", w=None):
    """Fractional response GLM (binomial quasi-likelihood), IRLS with step-halving on the binomial deviance.
    fam 'logit' (primary) or 'probit' (link-robustness check). Optional case weights w."""
    n, k = X.shape
    w = np.ones(n) if w is None else np.asarray(w, float)
    inv, dmu = _link(fam)
    p0 = np.clip(np.average(y, weights=w), 1e-4, 1 - 1e-4)
    beta = np.zeros(k)
    beta[0] = np.log(p0 / (1 - p0)) if fam == "logit" else stats.norm.ppf(p0)
    eta = X @ beta
    mu = np.clip(inv(eta), 1e-9, 1 - 1e-9)
    d_old = _bdev(y, mu, w)
    conv = False
    for _ in range(200):
        dm = np.maximum(dmu(eta, mu), 1e-12)
        V = mu * (1 - mu)
        W = w * dm ** 2 / V
        z = eta + (y - mu) / dm
        A, rhs = (X.T * W) @ X, (X.T * W) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        step = 1.0
        for _h in range(30):
            cand = beta + step * (b_new - beta)
            eta_c = np.clip(X @ cand, -30, 30)
            mu_c = np.clip(inv(eta_c), 1e-9, 1 - 1e-9)
            d_new = _bdev(y, mu_c, w)
            if np.isfinite(d_new) and d_new <= d_old + 1e-10 * abs(d_old):
                break
            step /= 2
        done = np.abs(cand - beta).max() < 1e-9 or abs(d_old - d_new) < 1e-12 * max(abs(d_old), 1)
        beta, eta, mu, d_old = cand, eta_c, mu_c, d_new
        if done:
            conv = True
            break
    dm = np.maximum(dmu(eta, mu), 1e-12)
    V = mu * (1 - mu)
    W = w * dm ** 2 / V
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * W) @ X), "conv": conv, "W": W, "dm": dm,
            "V": V, "w": w, "phi": float(np.sum(w * (y - mu) ** 2 / V) / max(w.sum() - k, 1))}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups):
    n, k = X.shape
    s = f["w"] * (y - f["mu"]) * f["dm"] / f["V"]
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_block(d, cats, nums, mode="stratified", fam="logit", cov=True, minn=RARE_MIN, weights=None):
    d = d.reset_index(drop=True)
    wts = None if weights is None else np.asarray(weights, float)
    if mode == "pooled":
        parts = {"All": np.ones(len(d), bool)}
    else:
        parts = {s: (d["stratum"] == s).to_numpy() for s in ["Global", "Other"] if (d["stratum"] == s).sum() >= 50}
    blocks, Xs, off = {}, [], 0
    for s, rows in parts.items():
        ds = d.loc[rows]
        cs = [c for c in cats if not (mode == "stratified" and c == "billing_mode")]
        cs = list(dict.fromkeys(cs + ["period"]))
        cm = make_cmap(ds, [v for v in cs if v in COLLAPSE_VARS], minn)
        dm_ = apply_cmap(ds, cm)
        refs = {**REFS, "period": min(dm_["period"].unique())}
        X_s, nm_s, spec = build_spec(dm_, cs, nums, refs)
        blocks[s] = {"rows": np.where(rows)[0], "cols": slice(off, off + X_s.shape[1]), "names": nm_s, "spec": spec,
                     "cmap": cm, "X": X_s,
                     "lev_n": [int(X_s[:, j].sum()) if nm.startswith("C(") else np.nan for j, nm in enumerate(nm_s)]}
        off += X_s.shape[1]
        Xs.append((rows, X_s))
    K = off
    y = d["patient_share"].to_numpy(float)
    beta, eta, mu = np.zeros(K), np.zeros(len(d)), np.zeros(len(d))
    W, dm, V, wv = np.zeros(len(d)), np.zeros(len(d)), np.zeros(len(d)), np.ones(len(d))
    bread, conv = np.zeros((K, K)), True
    for b in blocks.values():
        r = b["rows"]
        fb = fit_glm(b["X"], y[r], fam, None if wts is None else wts[r])
        beta[b["cols"]], eta[r], mu[r], W[r], dm[r], V[r], wv[r] = fb["beta"], fb["eta"], fb["mu"], fb["W"], fb["dm"], fb["V"], fb["w"]
        bread[b["cols"], b["cols"]] = fb["bread"]
        b["phi"] = fb["phi"]
        conv &= fb["conv"]
    f = {"beta": beta, "eta": eta, "mu": mu, "bread": bread, "W": W, "dm": dm, "V": V, "w": wv}
    X = None
    if cov:
        X = np.zeros((len(d), K))
        for (rows, X_s), b in zip(Xs, blocks.values()):
            X[np.ix_(np.where(rows)[0], np.arange(b["cols"].start, b["cols"].stop))] = X_s
    allnames = [(f"{s}|{nm}" if mode == "stratified" else nm) for s, b in blocks.items() for nm in b["names"]]
    lev_n = [x for b in blocks.values() for x in b["lev_n"]]
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl]) for cl in groups} if cov else {}
    return {"mode": mode, "fam": fam, "n": len(d), "names": allnames, "lev_n": lev_n, "beta": beta, "covs": covs, "X": X,
            "y": y, "mu": mu, "eta": eta, "W": W, "dm": dm, "V": V, "w": wv, "phi": {s: b["phi"] for s, b in blocks.items()},
            "conv": conv, "blocks": blocks, "d": d, "K": K, "groups": groups}


def make_tab(core, model, label="patient_share"):
    out = []
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = core["beta"] / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term_full": core["names"], "n": core["n"],
                          "level_n": core["lev_n"], "odds_ratio": np.exp(core["beta"]),
                          "lo": np.exp(core["beta"] - crit * se), "hi": np.exp(core["beta"] + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        if core["mode"] == "stratified":
            t["stratum"] = t.term_full.str.split("|", n=1).str[0]
            t["term"] = t.term_full.str.split("|", n=1).str[1]
        else:
            t["stratum"], t["term"] = "All", t["term_full"]
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[(t["term"] != "Intercept") & ~t["term"].str.startswith("C(period)")])
    return pd.concat(out, ignore_index=True)


def crit_p(core, cl, est, se):
    G = core["covs"][cl][1]
    if cl == "province":
        return stats.t.ppf(0.975, G - 1), (2 * stats.t.sf(abs(est / se), G - 1) if se > 0 else np.nan)
    return Z975, (2 * stats.norm.sf(abs(est / se)) if se > 0 else np.nan)


# --- prediction, margins (g-computation) with gradients ----------------------
def pred_part(core, s, dpop, over=None):
    b = core["blocks"][s]
    dm_ = apply_cmap(dpop, b["cmap"])
    for k, v in (over or {}).items():
        dm_[k] = v
    X = apply_spec(dm_, b["spec"])
    eta = np.clip(X @ core["beta"][b["cols"]], -30, 30)
    inv, dmu = _link(core["fam"])
    mu = inv(eta)
    return mu, dmu(eta, mu), X


def margin(core, s, dpop, over=None):
    mu, dm_, X = pred_part(core, s, dpop, over)
    g = np.zeros(core["K"])
    g[core["blocks"][s]["cols"]] = (dm_ @ X) / len(mu)
    return float(mu.mean()), g


def predict_rows(core, dte):
    mu = np.full(len(dte), np.nan)
    for s, b in core["blocks"].items():
        m = (dte["stratum"] == s).to_numpy() if core["mode"] == "stratified" else np.ones(len(dte), bool)
        if m.any():
            mu[m] = pred_part(core, s, dte[m])[0]
    return mu


def est_se(core, grad):
    return {cl: float(np.sqrt(max(grad @ core["covs"][cl][0] @ grad, 0))) for cl in core["covs"]}


def ame_table(core, d, vars_, model):
    """AME in percentage points of each level vs the stratum reference (g-computation, delta-method CIs).
    Facility-level exposures (ownership, specialty) are compared only WITHIN provinces where both the level and the
    reference occur with >= MIN_SUPPORT episodes (otherwise the effect is extrapolated into provinces where the level
    never occurs and cannot be separated from the province). If no province qualifies the AME is not identified (NaN)."""
    rows = []
    d = d.reset_index(drop=True)
    for s, b in core["blocks"].items():
        pop = d[(d["stratum"] == s).to_numpy()] if core["mode"] == "stratified" else d
        dpc = apply_cmap(pop, b["cmap"])
        for v in vars_:
            if v not in b["spec"]["levs"]:
                continue
            levs = b["spec"]["levs"][v]
            ref = levs[0]
            ct = pd.crosstab(pop["province_model"].to_numpy(), dpc[v].to_numpy()) if v in FAC_VARS else None
            m0_all, g0_all = margin(core, s, pop, {v: ref})
            for lev in levs[1:]:
                nm = f"C({v})[T.{lev}]"
                if nm not in b["names"]:
                    continue
                m1_all, _ = margin(core, s, pop, {v: lev})
                pr, n_prov, n_obs = pop, np.nan, b["lev_n"][b["names"].index(nm)]
                if v in FAC_VARS:
                    okp = ct.index[(ct[lev] >= MIN_SUPPORT) & (ct[ref] >= MIN_SUPPORT)] if (lev in ct and ref in ct) else []
                    n_prov = len(okp)
                    pr = pop[pop["province_model"].isin(okp)]
                    n_obs = int((dpc[v] == lev)[pop["province_model"].isin(okp).to_numpy()].sum())
                row = {"model": model, "stratum": s, "variable": v, "level": lev, "reference": ref,
                       "level_n": b["lev_n"][b["names"].index(nm)], "n_support_provinces": n_prov,
                       "n_support_episodes": n_obs, "AME_unrestricted_pp": 100 * (m1_all - m0_all)}
                if len(pr) < 30 or n_obs < 1:
                    row.update({"AME_pp": np.nan, "mean_ref_pp": np.nan, "sparse_flag": True,
                                "note": "not identified within any province"})
                    rows.append(row)
                    continue
                m0, g0 = margin(core, s, pr, {v: ref})
                m1, g1 = margin(core, s, pr, {v: lev})
                a_, gr = 100 * (m1 - m0), 100 * (g1 - g0)
                row.update({"AME_pp": a_, "mean_ref_pp": 100 * m0, "note": ""})
                for cl, se in est_se(core, gr).items():
                    c, p = crit_p(core, cl, a_, se)
                    row[f"lo_{cl}"], row[f"hi_{cl}"], row[f"p_{cl}"] = a_ - c * se, a_ + c * se, p
                row["sparse_flag"] = bool(row["level_n"] < SPARSE_N_MIN or (v in FAC_VARS and n_obs < SPARSE_N_MIN))
                rows.append(row)
    return pd.DataFrame(rows)


def att_table(core, d, model):
    """Global episodes: patient share as billed (Global model) vs if billed itemised (itemised model), standardised.
    Primary rows use only Global episodes on COMMON SUPPORT (province, specialty, fund and ownership levels with >= 30
    episodes in the itemised stratum); the unrestricted row is kept for transparency."""
    d = d.reset_index(drop=True)
    popG0 = d[d["stratum"] == "Global"]
    if "Global" not in core["blocks"] or "Other" not in core["blocks"]:
        return pd.DataFrame()
    O = d[d["stratum"] == "Other"]
    ok = np.ones(len(popG0), bool)
    for v in ["province_model", "specialty_model", "insurance_fund_model", "ownership"]:
        ok &= (popG0[v].map(O[v].value_counts()).fillna(0) >= RARE_MIN).to_numpy()
    popG = popG0[ok]
    rows = []
    for lab, base, g in [("All", popG, "All")] + [(g, popG, g) for g in sorted(d.group.unique())] + \
            [("All, unrestricted", popG0, "All")]:
        pop = base if g == "All" else base[base.group == g]
        if len(pop) < 20:
            continue
        m1, g1 = margin(core, "Global", pop)
        m2, g2 = margin(core, "Other", pop)
        row = {"model": model, "population": f"Global episodes ({lab})", "n_Global": len(pop),
               "share_of_Global_on_support": len(popG) / len(popG0),
               "observed_Global_mean_pp": 100 * pop.patient_share.mean(), "fitted_Global_pp": 100 * m1,
               "if_itemised_pp": 100 * m2, "diff_pp": 100 * (m1 - m2), "ratio_Global_over_itemised": m1 / m2}
        for cl, se in est_se(core, 100 * (g1 - g2)).items():
            c, p = crit_p(core, cl, 100 * (m1 - m2), se)
            row[f"diff_lo_{cl}"], row[f"diff_hi_{cl}"], row[f"diff_p_{cl}"] = 100 * (m1 - m2) - c * se, 100 * (m1 - m2) + c * se, p
        rows.append(row)
    return pd.DataFrame(rows)


def hit(lo, hi, null=0.0):
    return bool(lo > null or hi < null)


# =============================================================================
# 3. DESCRIPTIVES AND THRESHOLDS
# =============================================================================
hdr("2. DESCRIPTIVES (patient share, %)")


def cl_prop(x, cl):
    """Cluster-robust proportion and 95% CI."""
    x = np.asarray(x, float)
    p = x.mean()
    cid = pd.factorize(cl)[0]
    s = pd.Series(x - p).groupby(cid).sum().to_numpy()
    se = np.sqrt((len(s) / (len(s) - 1)) * np.sum(s ** 2)) / len(x) if len(s) > 1 else np.nan
    return p, p - Z975 * se, p + Z975 * se


def cstats(g):
    s = 100 * g["patient_share"]
    row = {"n": len(g), "mean": s.mean(), "sd": s.std(), "median": s.median(), "p25": s.quantile(.25), "p75": s.quantile(.75),
           "median_IQR": f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})"}
    for t in THRESHOLDS:
        p, lo, hi = cl_prop(g["patient_share"] > t, g["cluster_id"])
        row[f"pct_gt_{int(t * 100)}"] = 100 * p
        row[f"pct_gt_{int(t * 100)}_lo"], row[f"pct_gt_{int(t * 100)}_hi"] = 100 * lo, 100 * hi
    return row


desc = [{"variable": "PRIMARY SAMPLE", "stratum": "All", "level": "All", **cstats(D)}]
for s, g in D.groupby("stratum"):
    desc.append({"variable": "Billing mode", "stratum": s, "level": s, **cstats(g)})
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("ownership", "Ownership"),
                 ("insurance_fund_model", "Fund"), ("specialty_model", "Specialty"), ("province_model", "Province"),
                 ("period", "Period (Jalali year-month)"), ("any_surgical_service", "Surgical service")]:
    for (s, lev), g in D.groupby(["stratum", var]):
        desc.append({"variable": lab, "stratum": s, "level": lev, **cstats(g)})
    for lev, g in D.groupby(var):
        desc.append({"variable": lab, "stratum": "All", "level": lev, **cstats(g)})
DESC = pd.DataFrame(desc)
save(DESC, "o3_descriptives_and_thresholds.csv")
cols = ["variable", "stratum", "level", "n", "median_IQR", "mean", "pct_gt_10", "pct_gt_20", "pct_gt_40"]
print(DESC[DESC.variable.isin(["PRIMARY SAMPLE", "Billing mode"])][cols].round(1).to_string(index=False))
print("\nBy group x billing mode:")
print(DESC[(DESC.variable == "Group") & (DESC.stratum != "All")][cols].round(1).to_string(index=False))
print("\nBy ownership x billing mode:")
print(DESC[(DESC.variable == "Ownership") & (DESC.stratum != "All")][cols].round(1).to_string(index=False))

# =============================================================================
# 4. SELECTION: who is reconciled?
# =============================================================================
hdr("3. SELECTION INTO THE PRIMARY SAMPLE (reconciled episodes)")
adm_sel = DALL[is_adm & valid_total].reset_index(drop=True).copy()
adm_sel["rec_y"] = adm_sel["ps_ok"].astype(float)
SEL_CATS = ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service", "province_model",
            "billing_mode"]
cm_sel = make_cmap(adm_sel, [v for v in SEL_CATS + ["period"] if v in COLLAPSE_VARS])
dsel = apply_cmap(adm_sel, cm_sel)
Xsel, nsel, specsel = build_spec(dsel, SEL_CATS + ["period"], [], {**REFS, "period": min(dsel.period.unique())})
fsel = fit_glm(Xsel, adm_sel["rec_y"].to_numpy(), "logit")
covsel, Gsel = cov_robust(Xsel, adm_sel["rec_y"].to_numpy(), fsel, pd.factorize(adm_sel.cluster_id)[0])
sesel = np.sqrt(np.clip(np.diag(covsel), 0, None))
SEL = pd.DataFrame({"term": nsel, "odds_ratio_reconciled": np.exp(fsel["beta"]), "lo": np.exp(fsel["beta"] - Z975 * sesel),
                    "hi": np.exp(fsel["beta"] + Z975 * sesel)})
SEL = SEL[(SEL.term != "Intercept") & ~SEL.term.str.startswith("C(period)") & ~SEL.term.str.startswith("C(province")]
# AUC
pr, yy = fsel["mu"], adm_sel["rec_y"].to_numpy()
ranks = stats.rankdata(pr)
auc = (ranks[yy == 1].sum() - (yy == 1).sum() * ((yy == 1).sum() + 1) / 2) / ((yy == 1).sum() * (yy == 0).sum())
save(SEL, "o3_selection_model.csv")
print(f"Admissions with valid total cost: {len(adm_sel):,}; reconciled with share in [0,1]: {int(yy.sum()):,} "
      f"({100 * yy.mean():.1f}%); selection model AUC = {auc:.3f}")
print(SEL.round(3).to_string(index=False))
cmp_rows = []
for var in ["group", "billing_mode", "ownership", "insurance_fund_model", "any_surgical_service", "age_group", "adm_year"]:
    t = pd.crosstab(adm_sel[var], adm_sel["ps_ok"], normalize="columns").mul(100)
    t.columns = ["pct_among_not_reconciled", "pct_among_reconciled"]
    t = t.reset_index().rename(columns={var: "level"})
    t.insert(0, "variable", var)
    cmp_rows.append(t)
SELC = pd.concat(cmp_rows)
SELC["difference_pp"] = SELC.pct_among_reconciled - SELC.pct_among_not_reconciled
save(SELC, "o3_reconciled_vs_not.csv")
print("\nReconciled vs not (column %):")
print(SELC.round(1).to_string(index=False))
print("A material imbalance means the primary sample is a selected subset: that is why scenarios 1-4 (section 9) and the "
      "inverse-probability-weighted analysis (X1) are reported.")
PROB_REC = pd.Series(fsel["mu"], index=adm_sel.episode_no.values)

# What the unreconciled episodes look like, and whether reconciliation is ignorable given covariates
adm_sel["ins_share"] = adm_sel.insured_share_irr / adm_sel.total_cost_irr
adm_sel["req_share"] = (adm_sel.requested_total_irr / adm_sel.total_cost_irr).clip(0, 1)
adm_sel["unaccounted"] = (adm_sel.total_cost_irr - adm_sel.insured_share_irr - adm_sel.requested_total_irr) / adm_sel.total_cost_irr
adm_sel["reconciled_valid"] = np.where(adm_sel.ps_ok, "reconciled", "not reconciled")
MONEY = adm_sel.groupby(["stratum", "reconciled_valid"]).agg(
    n=("episode_no", "size"), insured_share_mean=("ins_share", "mean"), requested_share_mean=("req_share", "mean"),
    unaccounted_mean=("unaccounted", "mean"), unaccounted_median=("unaccounted", "median"),
    share_unaccounted_gt_50pct=("unaccounted", lambda x: (x > .5).mean()),
    share_unaccounted_negative=("unaccounted", lambda x: (x < -.05).mean())).reset_index()
save(MONEY, "o3_money_decomposition_by_reconciliation.csv")
print("\nShare of total cost recorded as insured / requested / unaccounted, by billing mode and reconciliation:")
print(MONEY.round(3).to_string(index=False))
print("Unreconciled episodes always have money MISSING (total > insured + requested, never the reverse); where it went is "
      "not recorded. Their recorded insured share is therefore a lower bound, and 1 - requested share an upper bound.")
ign = []
for s_ in ["Global", "Other"]:
    ds = adm_sel[adm_sel.stratum == s_].reset_index(drop=True)
    cats = BLOCKS["M5"] + ["period"]
    cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
    dm_ = apply_cmap(ds, cm)
    Xa, nma, _ = build_spec(dm_, cats, [], {**REFS, "period": min(dm_.period.unique())})
    ya = ds.req_share.to_numpy()
    X2 = np.column_stack([Xa, ds.ps_ok.to_numpy(float)])
    fa = fit_glm(X2, ya, "logit")
    cva, _ = cov_robust(X2, ya, fa, pd.factorize(ds.cluster_id)[0])
    b_, se_ = fa["beta"][-1], np.sqrt(cva[-1, -1])
    lin = X2[:, :-1] @ fa["beta"][:-1]
    ame_ = 100 * (expit(lin + b_) - expit(lin)).mean()
    ign.append({"stratum": s_, "n": len(ds), "insurer_share_reconciled_pct": 100 * ya[ds.ps_ok.to_numpy()].mean(),
                "insurer_share_not_reconciled_pct": 100 * ya[~ds.ps_ok.to_numpy()].mean(),
                "adjusted_difference_pp": ame_, "logit_coef": b_, "se": se_})
IGN = pd.DataFrame(ign)
save(IGN, "o3_reconciliation_ignorability.csv")
print("\nInsurer's share of the bill (observed for ALL admissions): reconciled vs not, adjusted for the M5 covariates.")
print(IGN.round(3).to_string(index=False))
print("A large adjusted difference means unreconciled episodes differ in how the bill is financed even among comparable "
      "episodes, so the primary sample cannot be assumed representative (compare the scenarios).")

# =============================================================================
# 5. MODELS M1-M5
# =============================================================================
hdr("4. MODELS M1-M5 (fractional logit; patient and province clusters)")


def blk_cats(m, pooled):
    c = list(BLOCKS[m])
    return c + ["billing_mode"] if pooled and m != "M1" else c


CORE_P, CORE_S, TABS = {}, {}, []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cp = fit_block(D, blk_cats(m, True), [], "pooled")
    cs = fit_block(D, blk_cats(m, False), [], "stratified")
    CORE_P[m], CORE_S[m] = cp, cs
    TABS += [make_tab(cp, m).assign(spec="pooled additive (comparison only)"),
             make_tab(cs, m).assign(spec="stratified by billing mode (PRIMARY)")]
    print(f"  {m}: pooled terms {len(cp['names']) - 1} (quasi-binomial dispersion {cp['phi']['All']:.3f}) | stratified terms "
          f"{len(cs['names']) - len(cs['blocks'])}, dispersion " + ", ".join(f"{s} {v:.3f}" for s, v in cs["phi"].items())
          + ("" if cs["conv"] and cp["conv"] else "  ** did not converge **"))
LONG = pd.concat(TABS, ignore_index=True)
save(LONG, "o3_models_long_oddsratios.csv")
M5S, M5P = CORE_S["M5"], CORE_P["M5"]
prov_n = D.province_model.value_counts()
top_prov = prov_n.index[:TOP_PROVINCES].tolist()

AME_ALL = ame_table(M5S, D, REPORT_VARS, "M5 stratified")
save(AME_ALL, "o3_ame_M5_stratified_pp.csv")

# identifiability of facility-level terms: are the levels spread over provinces, or confined to a few?
idr = []
for s_ in ["Global", "Other"]:
    ds = D[D.stratum == s_]
    for v in FAC_VARS:
        for lev, g in ds.groupby(v):
            vc = g.province_model.value_counts()
            a = AME_ALL[(AME_ALL.stratum == s_) & (AME_ALL.variable == v) & (AME_ALL.level == lev)]
            idr.append({"stratum": s_, "variable": v, "level": lev, "n": len(g), "n_provinces": g.province_model.nunique(),
                        "top_province": vc.index[0], "top_province_share_pct": 100 * vc.iloc[0] / len(g),
                        "support_provinces_vs_reference": float(a.n_support_provinces.iloc[0]) if len(a) else np.nan,
                        "support_episodes": float(a.n_support_episodes.iloc[0]) if len(a) else np.nan,
                        "concentrated": bool(g.province_model.nunique() <= 3 or 100 * vc.iloc[0] / len(g) > 70 or
                                             (len(a) and np.isfinite(a.n_support_provinces.iloc[0]) and a.n_support_provinces.iloc[0] <= 2))})
IDENT = pd.DataFrame(idr)
save(IDENT, "o3_facility_identifiability.csv")
print("\nIdentifiability of facility-level exposures (ownership, specialty): a level confined to a few provinces cannot be "
      "separated from the province or hospital.")
print(IDENT[IDENT.variable == "ownership"].round(1).to_string(index=False))
IDENT_FLAG = {(r.stratum, r.variable, r.level): r.concentrated for r in IDENT.itertuples()}


def fmt_ame(r, cl):
    star = " *" if hit(r[f"lo_{cl}"], r[f"hi_{cl}"]) else ""
    sp = " [n<30]" if r.sparse_flag else ""
    return f"{r.AME_pp:+.1f} ({r[f'lo_{cl}']:+.1f} to {r[f'hi_{cl}']:+.1f}){star}{sp}"


mt = AME_ALL[AME_ALL.variable.isin(KEY_VARS + ["specialty_model"]) |
             ((AME_ALL.variable == "province_model") & AME_ALL.level.isin(top_prov))].copy()
mt["AME_patient_CI"] = mt.apply(lambda r: fmt_ame(r, "patient"), axis=1)
mt["AME_province_CI"] = mt.apply(lambda r: fmt_ame(r, "province"), axis=1)
orm = make_tab(M5S, "M5")
orm = orm[orm.cluster == "patient"].set_index(["stratum", "term"])
mt["odds_ratio"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "odds_ratio"] for r in mt.itertuples()]
mt["OR_lo"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "lo"] for r in mt.itertuples()]
mt["OR_hi"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "hi"] for r in mt.itertuples()]
save(mt, "o3_main_table_M5_stratified.csv")
print("\nMAIN TABLE (stratified M5): average marginal effect on patient share, percentage points vs the stratum reference.\n"
      "Patient-cluster and province-cluster (t, G-1 df) 95% CIs; * = CI excludes 0. Facility-level terms (ownership,\n"
      "specialty, province) are best read with the province-cluster CI.")
print(mt[["stratum", "variable", "level", "reference", "level_n", "mean_ref_pp", "AME_patient_CI", "AME_province_CI"]]
      .round(1).to_string(index=False))

pb = AME_ALL.iloc[0:0]
bm = []
for m in ["M2", "M3", "M4", "M5"]:
    a = ame_table(CORE_P[m], D, ["billing_mode"], f"{m} pooled additive")
    bm.append(a)
BM = pd.concat(bm)
save(BM, "o3_pooled_billing_AME_by_block.csv")
print("\nPooled ADDITIVE billing-mode AME (itemised vs Global, pp) [v1-type estimate; one constant effect]:")
print(BM[["model", "AME_pp", "lo_patient", "hi_patient"]].round(2).to_string(index=False))

# =============================================================================
# 6. GLOBAL vs ITEMISED, STANDARDISED
# =============================================================================
hdr("5. GLOBAL vs ITEMISED, standardised on the Global episodes")
ATT = att_table(M5S, D, "M5")
ATT_BLOCKS = pd.concat([att_table(CORE_S[m], D, m).query("population == 'Global episodes (All)'") for m in CORE_S])


def boot_att(d, B, key):
    cl = d[key].astype(str).to_numpy()
    uc, inv = np.unique(cl, return_inverse=True)
    idx_by = [np.where(inv == i)[0] for i in range(len(uc))]
    out = []
    for _ in range(B):
        pick = RNG.integers(0, len(uc), len(uc))
        db = d.iloc[np.concatenate([idx_by[i] for i in pick])].reset_index(drop=True)
        try:
            cb = fit_block(db, BLOCKS["M5"], [], "stratified", cov=False)
            out.append(att_table(cb, db, "b").set_index("population")["diff_pp"])
        except Exception:
            continue
    return pd.concat(out, axis=1)


NB = min(NBOOT, 100)
for key, tag in [("cluster_id", "patient"), ("province_model", "province")]:
    bt = boot_att(D, NB, key)
    q = bt.quantile([.025, .975], axis=1).T
    ATT[f"boot_lo_{tag}"] = [q.loc[p_, 0.025] if p_ in q.index else np.nan for p_ in ATT.population]
    ATT[f"boot_hi_{tag}"] = [q.loc[p_, 0.975] if p_ in q.index else np.nan for p_ in ATT.population]
save(ATT, "o3_global_vs_itemised_standardised_M5.csv")
save(ATT_BLOCKS, "o3_global_vs_itemised_by_block.csv")
print("Patient share (pp) of the Global episodes as billed vs if billed itemised (itemised model applied to them):")
print(ATT[["population", "n_Global", "observed_Global_mean_pp", "if_itemised_pp", "diff_pp", "boot_lo_patient", "boot_hi_patient",
           "boot_lo_province", "boot_hi_province"]].round(1).to_string(index=False))
print("\nAcross blocks (delta-method CI, patient clusters):")
print(ATT_BLOCKS[["model", "fitted_Global_pp", "if_itemised_pp", "diff_pp", "diff_lo_patient", "diff_hi_patient"]].round(1).to_string(index=False))

# fit to observed cell means (fractional logit reproduces them for categorical covariates)
chk = []
for s in ["Global", "Other"]:
    for var in ["group", "ownership"]:
        for lev, g in D[D.stratum == s].groupby(var):
            if len(g) < 30:
                continue
            fitted_s = pred_part(M5S, s, g)[0].mean()
            row = {"stratum": s, "variable": var, "level": lev, "n": len(g), "observed_pp": 100 * g.patient_share.mean(),
                   "stratified_fitted_pp": 100 * fitted_s}
            if var in M5P["blocks"]["All"]["spec"]["levs"]:
                row["pooled_additive_fitted_pp"] = 100 * pred_part(M5P, "All", g)[0].mean()
            chk.append(row)
CHK = pd.DataFrame(chk)
CHK["pooled_error_pp"] = CHK.pooled_additive_fitted_pp - CHK.observed_pp
CHK["stratified_error_pp"] = CHK.stratified_fitted_pp - CHK.observed_pp
save(CHK, "o3_fit_to_observed_cells.csv")
print("\nFitted vs observed cell means (pp): the stratified model reproduces them; the pooled additive does not:")
print(CHK.round(2).to_string(index=False))

# =============================================================================
# 7. DIAGNOSTICS AND OUT-OF-FOLD ERROR
# =============================================================================
hdr("6. DIAGNOSTICS within strata (link test, Cook + refit, probit link) and 5-fold out-of-fold error")
diag = []
cook_flag_all = np.zeros(len(M5S["d"]), bool)
for s, b in M5S["blocks"].items():
    r = b["rows"]
    Xs_, ys = b["X"], M5S["y"][r]
    mu, eta, W = M5S["mu"][r], M5S["eta"][r], M5S["W"][r]
    pat = M5S["groups"]["patient"][r]
    Xl = np.column_stack([Xs_, eta ** 2])
    fl = fit_glm(Xl, ys, "logit")
    covl, _ = cov_robust(Xl, ys, fl, pat)
    tl = fl["beta"][-1] / np.sqrt(covl[-1, -1])
    sw = np.sqrt(W)
    Z = Xs_ * sw[:, None]
    h = np.einsum("ij,jk,ik->i", Z, np.linalg.pinv(Z.T @ Z), Z)
    rp = (ys - mu) / np.sqrt(M5S["V"][r])
    k = Xs_.shape[1]
    cook = rp ** 2 * h / (k * b["phi"] * (1 - h) ** 2)
    flag = cook > COOK_MULT / len(r)
    cook_flag_all[r[flag]] = True
    diag.append({"stratum": s, "n": len(r), "terms": k, "dispersion": b["phi"], "link_t": float(tl),
                 "link_p": float(2 * stats.norm.sf(abs(tl))), "max_cook": float(cook.max()), "n_cook_flag": int(flag.sum()),
                 "max_leverage": float(h.max()), "n_leverage_gt_0.2": int((h > .2).sum()),
                 "share_exact_zero": float((ys == 0).mean())})
DIAG = pd.DataFrame(diag)
save(DIAG, "o3_diagnostics_by_stratum.csv")
print(DIAG.round(3).to_string(index=False))
print("A significant link test means some structure in the mean is missing (check the cell fit and out-of-fold calibration "
      "below for the practical size of the problem).")

M5C = fit_block(M5S["d"][~cook_flag_all], BLOCKS["M5"], [], "stratified", cov=False)
M5Pr = fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit", cov=False)
ame_c = ame_table(fit_block(M5S["d"][~cook_flag_all], BLOCKS["M5"], [], "stratified"), M5S["d"][~cook_flag_all], KEY_VARS, "cook")
ame_p = ame_table(fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit"), D, KEY_VARS, "probit")
key = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].copy()
for nm, a in [("cook_excl", ame_c), ("probit", ame_p)]:
    key = key.merge(a[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": f"AME_{nm}"}),
                    on=["stratum", "variable", "level"], how="left")
    key[f"{nm}_change_pp"] = key[f"AME_{nm}"] - key["AME_pp"]
save(key, "o3_robustness_cook_probit.csv")
print(f"\nCook refit excludes {int(cook_flag_all.sum())} episodes (> {COOK_MULT:g}/n within stratum); probit link refit. "
      f"AME (pp) of key terms:")
print(key.round(2).to_string(index=False))
atp = att_table(fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit", cov=False), D, "probit").iloc[0]
atc = att_table(M5C, M5C["d"], "cook").iloc[0]
print(f"\nGlobal vs itemised (diff, pp): logit {ATT.diff_pp.iloc[0]:.1f} | probit {atp.diff_pp:.1f} | Cook-excluded {atc.diff_pp:.1f}")

# out-of-fold
ids = pd.factorize(D.cluster_id)[0]
uid = np.unique(ids)
np.random.default_rng(SEED).shuffle(uid)
fm = dict(zip(uid, np.arange(len(uid)) % N_FOLDS))
FOLD = np.array([fm[i] for i in ids])
oof_rows, cal_rows = [], []
for m in ["M1", "M3", "M5"]:
    for mode in ["pooled", "stratified"]:
        pred = np.full(len(D), np.nan)
        for k in range(N_FOLDS):
            tr, te = D[FOLD != k], D[FOLD == k]
            core = fit_block(tr, blk_cats(m, mode == "pooled"), [], mode, cov=False)
            pred[FOLD == k] = np.clip(predict_rows(core, te), 0, 1)
        y = D.patient_share.to_numpy()
        for s in ["All", "Global", "Other"]:
            mk = np.ones(len(D), bool) if s == "All" else (D.stratum == s).to_numpy()
            e = y[mk] - pred[mk]
            oof_rows.append({"model": m, "spec": mode, "stratum": s, "n": int(mk.sum()), "MAE_pp": 100 * np.abs(e).mean(),
                             "RMSE_pp": 100 * np.sqrt((e ** 2).mean()), "mean_observed_pp": 100 * y[mk].mean(),
                             "mean_predicted_pp": 100 * pred[mk].mean(), "calibration_diff_pp": 100 * (pred[mk].mean() - y[mk].mean()),
                             "R2_oof": 1 - np.sum(e ** 2) / np.sum((y[mk] - y[mk].mean()) ** 2)})
            if m == "M5":
                q = pd.qcut(pd.Series(pred[mk]).rank(method="first"), 10, labels=False)
                c = pd.DataFrame({"decile": q + 1, "pred": 100 * pred[mk], "obs": 100 * y[mk]}).groupby("decile").mean().reset_index()
                c["model"], c["spec"], c["stratum"] = m, mode, s
                cal_rows.append(c.rename(columns={"pred": "mean_predicted_pp", "obs": "mean_observed_pp"}))
OOF, CAL = pd.DataFrame(oof_rows), pd.concat(cal_rows, ignore_index=True)
save(OOF, "o3_oof_error.csv")
save(CAL, "o3_oof_calibration_deciles.csv")
print("\n" + OOF.round(2).to_string(index=False))
c5 = CAL[(CAL.model == "M5") & (CAL.stratum == "All")].copy()
c5["diff_pp"] = c5.mean_predicted_pp - c5.mean_observed_pp
print("\nM5 calibration by predicted decile, all admissions (predicted minus observed, pp):")
print(c5.pivot(index="decile", columns="spec", values="diff_pp").round(2).to_string())

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for s, col in [("Global", "C0"), ("Other", "C3")]:
        ax[0].hist(100 * D.loc[D.stratum == s, "patient_share"], bins=50, alpha=.6, color=col, label=s, density=True)
    ax[0].set(title="Patient share by billing mode", xlabel="Patient share (%)", ylabel="Density")
    ax[0].legend(frameon=False)
    for k, spec in enumerate(["pooled", "stratified"]):
        for s, col in [("Global", "C0"), ("Other", "C3")]:
            c8 = CAL[(CAL.model == "M5") & (CAL.stratum == s) & (CAL.spec == spec)]
            ax[1 + k].plot(c8.mean_predicted_pp, c8.mean_observed_pp, "o-", color=col, label=s)
        ax[1 + k].plot([0, 100], [0, 100], "k--", lw=.8)
        ax[1 + k].set(title=f"M5 {spec}: out-of-fold calibration", xlabel="Mean predicted (%)", ylabel="Mean observed (%)")
        ax[1 + k].legend(frameon=False, title="billing mode")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o3_distribution_and_calibration.png", dpi=200)
    plt.close(fig)
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")

# =============================================================================
# 8. HYPOTHESIS 2: provinces
# =============================================================================
hdr("7. HYPOTHESIS 2: patient share differs across provinces after case-mix adjustment")


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


h2 = []
for s in ["Global", "Other"]:
    sub = D[D.stratum == s]
    kw = stats.kruskal(*[g.patient_share.values for _, g in sub.groupby("province_model") if len(g) >= 5])
    h2.append({"model": "crude (Kruskal-Wallis)", "stratum": s, "df": np.nan, "chi2": float(kw.statistic), "p": float(kw.pvalue)})
for m in ["M4", "M5"]:
    core = CORE_S[m]
    for s, b in core["blocks"].items():
        idx = [b["cols"].start + j for j, nm in enumerate(b["names"]) if nm.startswith("C(province_model)")]
        if idx:
            chi2, df, p = wald(core, idx)
            h2.append({"model": m, "stratum": s, "df": df, "chi2": chi2, "p": p,
                       "note": "Wald, patient-clustered; province-level SE not valid (28 clusters)"})
H2 = pd.DataFrame(h2)
save(H2, "o3_h2_province_tests.csv")
print(H2.round(4).to_string(index=False))
pm = AME_ALL[(AME_ALL.variable == "province_model") & ~AME_ALL.sparse_flag]
save(pm, "o3_h2_province_AME_M5.csv")
for s in ["Global", "Other"]:
    x = pm[pm.stratum == s].sort_values("AME_pp")
    ref = x.reference.iloc[0]
    print(f"\n{s}: adjusted province AME (pp) vs {ref}; extremes (modelled provinces with >= 30 episodes):")
    print(x[["level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].iloc[[0, 1, 2, -3, -2, -1]].round(1).to_string(index=False))


class LaplaceFrac:
    """Fractional-logit with a province random intercept, Laplace-approximated pseudo-ML (Bernoulli kernel)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def negll(self, theta):
        beta, sig = theta[:self.k], np.exp(theta[self.k])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            eta = xb + u[self.g]
            p = expit(eta)
            d1, d2 = self.y - p, -p * (1 - p)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        eta = xb + u[self.g]
        p = expit(eta)
        h = np.bincount(self.g, p * (1 - p), self.G) + 1 / sig ** 2
        ll = np.sum(self.y * eta - np.logaddexp(0, eta))
        return -(ll - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0):
        th0 = np.concatenate([beta0, [np.log(0.3)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()


def fit_ri(d, label):
    s8, vc8 = [], []
    for s in ["Global", "Other"]:
        ds = d[d.stratum == s].reset_index(drop=True)
        if len(ds) < 200:
            continue
        t0 = time.time()
        try:
            cats = [v for v in BLOCKS["M5"] if v != "province_model"] + ["period"]
            cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
            dm_ = apply_cmap(ds, cm)
            X8, nm8, _ = build_spec(dm_, cats, [], {**REFS, "period": min(dm_["period"].unique())})
            y8 = ds.patient_share.to_numpy(float)
            g8 = pd.factorize(ds.province_model)[0]
            f0 = fit_glm(X8, y8, "logit")
            m8 = LaplaceFrac(X8, y8, g8)
            theta = m8.fit(f0["beta"])
            H = m8.hessian(theta)
            se8 = np.sqrt(np.clip(np.diag(np.linalg.pinv(H)), 0, None))
            k8 = m8.k
            s8.append(pd.DataFrame({"outcome": "patient_share", "model": "M5 random intercept", "stratum": s, "term": nm8[1:],
                                    "n": len(ds), "level_n": [int(X8[:, j].sum()) if nm8[j].startswith("C(") else np.nan for j in range(1, k8)],
                                    "odds_ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                                    "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]), "sens": 8, "sens_name": label}))
            sig = float(np.exp(theta[k8]))
            vc8.append({"stratum": s, "n": len(ds), "provinces": m8.G, "sigma_u": sig,
                        "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])), "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                        "median_odds_ratio": float(np.exp(0.6745 * np.sqrt(2) * sig)),
                        "note": "pseudo-likelihood: SE of sigma_u approximate"})
            print(f"  {s}: n = {len(ds):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (median odds ratio between two random "
                  f"provinces {np.exp(0.6745 * np.sqrt(2) * sig):.2f}), {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  RI ({s}) FAILED ({type(e).__name__}: {e})")
    return (pd.concat(s8, ignore_index=True) if s8 else pd.DataFrame()), (pd.DataFrame(vc8) if vc8 else pd.DataFrame())


if RUN_SENS8:
    print("\nProvince random intercept, within each stratum (= sensitivity analysis 8):")
    S8, VC8 = fit_ri(D, "Province random intercept")
    if len(VC8):
        save(VC8, "o3_sens8_variance_components.csv")
        save(S8, "o3_sens8_random_intercept_long.csv")
        print(VC8.round(3).to_string(index=False))

# =============================================================================
# 9. UNRECONCILED EPISODES: SCENARIOS 1-4
# =============================================================================
hdr("8. UNRECONCILED EPISODES: scenarios for the patient share (all admissions with a valid total cost)")


def scenario_b(d):
    """PROVISIONAL stand-in for the cholera paper's Scenario B (not defined in the protocol or the analysis file):
    recorded shares rescaled to the recorded financing base, insured / (insured + requested). Episodes without both
    shares keep the recorded lower bound. REPLACE with the cholera paper's definition before reporting."""
    base = d["insured_share_irr"] + d["requested_total_irr"]
    ps = (d["insured_share_irr"] / base).where(base > 0)
    return ps.fillna(d["ps_scen1"]).clip(0, 1)


SC = DALL[is_adm & valid_total].copy().reset_index(drop=True)
tot = SC["total_cost_irr"]
SC["ps_scen1"] = (SC["insured_share_irr"] / tot).clip(0, 1)
isrec = SC["ps_ok"].to_numpy()
own_mean = SC[isrec].groupby("ownership")["patient_share"].mean()
own_bill_mean = SC[isrec].groupby(["ownership", "stratum"])["patient_share"].mean()
SC["ps_scen2"] = np.where(isrec, SC["patient_share"], SC["ownership"].map(own_mean))
SC["ps_scen2b"] = np.where(isrec, SC["patient_share"],
                           [own_bill_mean.get((o, s), own_mean.get(o, np.nan)) for o, s in zip(SC.ownership, SC.stratum)])
SC["ps_scen3"] = np.where(isrec, SC["patient_share"], scenario_b(SC))
upper = np.fmax((tot - SC["requested_total_irr"]) / tot, SC["ps_scen1"]).clip(0, 1)
SC["ps_scen4"] = np.where(isrec, SC["patient_share"], upper)
if "ps_scen3_upper_bound" in DALL.columns:
    diff_up = (SC["ps_scen4"] - DALL.set_index("episode_no").loc[SC.episode_no, "ps_scen3_upper_bound"].to_numpy())
    print(f"  Check: upper bound recomputed vs file column: max |difference| = {np.nanmax(np.abs(diff_up)):.2e}")
SCEN = {"1 recorded lower bound": "ps_scen1", "2 ownership-group mean": "ps_scen2",
        "2b ownership x billing-mode mean (exploratory)": "ps_scen2b",
        "3 'Scenario B' (PROVISIONAL stand-in)": "ps_scen3", "4 upper bound": "ps_scen4"}
sc_rows = []
for lab, col in SCEN.items():
    for stratum_lab, mk in [("All admissions", np.ones(len(SC), bool)), ("Global", (SC.stratum == "Global").to_numpy()),
                            ("Itemised", (SC.stratum == "Other").to_numpy())]:
        x = SC.loc[mk, col]
        okx = x.notna().to_numpy()
        p, lo, hi = cl_prop(x[okx], SC.loc[mk, "cluster_id"][okx])
        sc_rows.append({"scenario": lab, "population": stratum_lab, "n": int(okx.sum()), "mean_share_pct": 100 * p,
                        "lo": 100 * lo, "hi": 100 * hi,
                        "pct_gt_20": 100 * (x[okx] > .2).mean(), "pct_gt_40": 100 * (x[okx] > .4).mean()})
SCS = pd.DataFrame(sc_rows)
sc_rec = SC.loc[isrec, "patient_share"]
SCS = pd.concat([pd.DataFrame([{"scenario": "Reconciled episodes only (primary sample)", "population": "All admissions",
                                "n": int(isrec.sum()), "mean_share_pct": 100 * sc_rec.mean(),
                                "lo": 100 * cl_prop(sc_rec, SC.loc[isrec, "cluster_id"])[1],
                                "hi": 100 * cl_prop(sc_rec, SC.loc[isrec, "cluster_id"])[2],
                                "pct_gt_20": 100 * (sc_rec > .2).mean(), "pct_gt_40": 100 * (sc_rec > .4).mean()}]), SCS])
save(SCS, "o3_scenarios_pooled_mean.csv")
print("Pooled mean patient share under each scenario (reconciled episodes keep their observed share in every scenario):")
print(SCS.round(1).to_string(index=False))
rg = []
for lab, col in SCEN.items():
    for var in ["group", "ownership"]:
        for lev, g in SC.groupby(var):
            rg.append({"scenario": lab, var if False else "variable": var, "level": lev, "n": int(g[col].notna().sum()),
                       "mean_share_pct": 100 * g[col].mean()})
save(pd.DataFrame(rg), "o3_scenarios_by_group_ownership.csv")

# do the conclusions hold under scenarios 1 and 4 (and the others)?
print("\nConclusions under each scenario (stratified M5 refitted on all admissions, scenario share as outcome):")
conc_rows, att_rows = [], []
for lab, col in [("0 primary (reconciled only)", None)] + list(SCEN.items()):
    if col is None:
        dd, ame_s, att_s = D, AME_ALL[AME_ALL.variable.isin(KEY_VARS)].assign(scenario=lab), ATT.assign(scenario=lab)
    else:
        dd = SC.copy()
        dd["patient_share"] = dd[col]
        dd = dd[dd["patient_share"].notna()].reset_index(drop=True)
        cr = fit_block(dd, BLOCKS["M5"], [], "stratified")
        ame_s = ame_table(cr, dd, KEY_VARS, lab).assign(scenario=lab)
        att_s = att_table(cr, dd, lab).assign(scenario=lab)
    conc_rows.append(ame_s)
    att_rows.append(att_s)
SCAME = pd.concat(conc_rows, ignore_index=True)
SCATT = pd.concat(att_rows, ignore_index=True)
save(SCAME, "o3_scenarios_AME_key_terms.csv")
save(SCATT, "o3_scenarios_global_vs_itemised.csv")
base_ame = AME_ALL[AME_ALL.variable.isin(KEY_VARS)].set_index(["stratum", "variable", "level"]).AME_pp
hold = []
for (s_, v_, l_), a0 in base_ame.items():
    row = {"stratum": s_, "variable": v_, "level": l_, "primary_AME_pp": a0}
    ok_all = True
    for lab in SCEN:
        x = SCAME[(SCAME.scenario == lab) & (SCAME.stratum == s_) & (SCAME.variable == v_) & (SCAME.level == l_)]
        a1 = float(x.AME_pp.iloc[0]) if len(x) else np.nan
        row[f"AME_scenario_{lab.split()[0]}"] = a1
    for lab in ["1 recorded lower bound", "4 upper bound"]:
        a1 = row[f"AME_scenario_{lab.split()[0]}"]
        row[f"same_sign_scen{lab.split()[0]}"] = bool(np.sign(a1) == np.sign(a0)) if np.isfinite(a1) else np.nan
    row["holds_under_1_and_4"] = bool(row["same_sign_scen1"] and row["same_sign_scen4"]) if np.isfinite(row["AME_scenario_1"]) and np.isfinite(row["AME_scenario_4"]) else np.nan
    hold.append(row)
HOLD = pd.DataFrame(hold)
save(HOLD, "o3_scenario_conclusions_hold.csv")
print(HOLD.round(2).to_string(index=False))
a1 = SCATT[(SCATT.population == "Global episodes (All)")][["scenario", "observed_Global_mean_pp", "if_itemised_pp", "diff_pp"]]
print("\nGlobal vs itemised (standardised on Global episodes) under each scenario:")
print(a1.round(1).to_string(index=False))
print("PROTOCOL RULE: the primary conclusions must hold under scenarios 1 and 4, otherwise the Discussion must say they do not "
      "(see 'holds_under_1_and_4'; ownership effects are partly built in under scenario 2).")


# =============================================================================
# 10. SENSITIVITY ANALYSES 1-8
# =============================================================================
hdr("9. SENSITIVITY ANALYSES 1-8 (stratified M5; AME in pp)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"), 2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"), 4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS (P95 cap) added as a covariate", None)}
SL, SATT = pd.DataFrame(), pd.DataFrame()
if RUN_SENS:
    sl, sa = [], []
    for s_id, (name, keepcol) in SENS.items():
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        nums = []
        if s_id == 7:
            sub["los_cov"] = sub["los_capped_p95"]
            sub = sub[sub.los_cov.notna()]
            nums = ["los_cov"]
        core = fit_block(sub, BLOCKS["M5"], nums, "stratified")
        sl.append(ame_table(core, sub, KEY_VARS, name).assign(sens=s_id, sens_name=name))
        if len(core["blocks"]) == 2:
            sa.append(att_table(core, sub, name).assign(sens=s_id, sens_name=name))
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}, strata = {list(core['blocks'])}")
    # sensitivity 9-10 (added after the O3 diagnostics): the 5% reconciliation tolerance is fixed by the protocol, but the
    # results move with it, so wider tolerances are reported. Patient share is the recorded insured / total in all of them.
    adm_t = DALL[is_adm].copy()
    both_t = adm_t.insured_share_irr.notna() & adm_t.requested_total_irr.notna()
    for s_id, tol in [(9, 0.10), (10, 0.20)]:
        sub = adm_t[both_t & (adm_t.fin_gap <= tol) & adm_t.patient_share.between(0, 1)].reset_index(drop=True)
        name = f"Reconciliation tolerance {int(tol * 100)}% (instead of 5%)"
        core = fit_block(sub, BLOCKS["M5"], [], "stratified")
        sl.append(ame_table(core, sub, KEY_VARS, name).assign(sens=s_id, sens_name=name))
        sa.append(att_table(core, sub, name).assign(sens=s_id, sens_name=name))
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}")
    SL = pd.concat(sl, ignore_index=True)
    SATT = pd.concat(sa, ignore_index=True) if sa else pd.DataFrame()
    save(SL, "o3_sensitivity_AME.csv")
    if len(SATT):
        save(SATT, "o3_sensitivity_global_vs_itemised.csv")
    sdir = []
    for r in AME_ALL[AME_ALL.variable.isin(KEY_VARS)].itertuples():
        x = SL[(SL.stratum == r.stratum) & (SL.variable == r.variable) & (SL.level == r.level)]
        if len(x):
            sdir.append({"stratum": r.stratum, "variable": r.variable, "level": r.level, "main_AME_pp": r.AME_pp,
                         "n_sens": len(x), "same_sign": int((np.sign(x.AME_pp) == np.sign(r.AME_pp)).sum()),
                         "max_abs_change_pp": float((x.AME_pp - r.AME_pp).abs().max())})
    save(pd.DataFrame(sdir), "o3_sensitivity_direction.csv")
    print(pd.DataFrame(sdir).round(2).to_string(index=False))

# =============================================================================
# 11. EXPLORATORY
# =============================================================================
hdr("10. EXPLORATORY (label as such)")
IPW_AME = pd.DataFrame()
if RUN_EXTRA:
    try:
        core0 = fit_block(ALT, BLOCKS["M5"], [], "stratified")
        x0 = ame_table(core0, ALT, KEY_VARS, "X0 " + ALT_LABEL)
        save(x0, "o3_X0_other_sample_AME.csv")
        print(f"X0: stratified M5 on {ALT_LABEL} (n = {len(ALT):,}); key AME (pp):")
        print(x0[["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].round(1).to_string(index=False))
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    try:
        pr_ = PROB_REC.reindex(D.episode_no.values).to_numpy()
        w = 1 / np.clip(pr_, np.quantile(pr_, 0.01), 1)
        w = w / w.mean()
        coreW = fit_block(D, BLOCKS["M5"], [], "stratified", weights=w)
        IPW_AME = ame_table(coreW, D, KEY_VARS, "X1 inverse-probability-of-reconciliation weights")
        save(IPW_AME, "o3_X1_ipw_AME.csv")
        cmpw = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp"]].merge(
            IPW_AME[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": "AME_ipw"}), on=["stratum", "variable", "level"])
        cmpw["change_pp"] = cmpw.AME_ipw - cmpw.AME_pp
        print("\nX1: weighting reconciled episodes by 1/P(reconciled) (observable selection only), key AME (pp):")
        print(cmpw.round(2).to_string(index=False))
        print("(Weights correct only for selection on the covariates in the selection model, not on unobserved factors.)")
    except Exception as e:
        print(f"X1 failed: {type(e).__name__}: {e}")
    lop = []
    for p in top_prov:
        sub = D[D.province_model != p]
        a = att_table(fit_block(sub, BLOCKS["M5"], [], "stratified", cov=False), sub, "M5").iloc[0]
        lop.append({"province_left_out": p, "n": len(sub), "diff_pp_Global_minus_itemised": a.diff_pp})
    LOP = pd.DataFrame(lop)
    save(LOP, "o3_X_leave_one_province_out.csv")
    print("\nLeave-one-province-out (largest provinces): standardised Global-minus-itemised difference (pp)")
    print(LOP.round(2).to_string(index=False))

# =============================================================================
# 12. RELIABILITY VERDICT
# =============================================================================
hdr("11. RELIABILITY VERDICT (A / B / C) for the key AME estimates")
rows = []
for r in AME_ALL[AME_ALL.variable.isin(KEY_VARS)].itertuples():
    x = SL[(SL.stratum == r.stratum) & (SL.variable == r.variable) & (SL.level == r.level)] if len(SL) else pd.DataFrame()
    dev = (x.AME_pp - r.AME_pp).abs() if len(x) else pd.Series(dtype=float)
    tol = max(STABLE_PP, STABLE_REL * abs(r.AME_pp))
    same = bool((np.sign(x.AME_pp) == np.sign(r.AME_pp)).all()) if len(x) else True
    ri_same, ri_note = True, ""
    if len(S8):
        t8 = S8[(S8.stratum == r.stratum) & (S8.term == f"C({r.variable})[T.{r.level}]")]
        if len(t8):
            orm_ = float(orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "odds_ratio"])
            ri_same = bool((t8.odds_ratio.iloc[0] > 1) == (orm_ > 1))
    kk = key[(key.stratum == r.stratum) & (key.variable == r.variable) & (key.level == r.level)]
    cook_ch = float(abs(kk.cook_excl_change_pp.iloc[0])) if len(kk) else np.nan
    prob_ch = float(abs(kk.probit_change_pp.iloc[0])) if len(kk) else np.nan
    sig_p, sig_v = hit(r.lo_patient, r.hi_patient), hit(r.lo_province, r.hi_province)
    conc = bool(IDENT_FLAG.get((r.stratum, r.variable, r.level), False))
    not_ident = not np.isfinite(r.AME_pp)
    hh = HOLD[(HOLD.stratum == r.stratum) & (HOLD.variable == r.variable) & (HOLD.level == r.level)]
    scen_ok = bool(hh.holds_under_1_and_4.iloc[0]) if len(hh) and pd.notna(hh.holds_under_1_and_4.iloc[0]) else True
    reasons = []
    if r.sparse_flag:
        reasons.append("level has <30 episodes")
    if not same:
        reasons.append("sign changes in a sensitivity analysis")
    if not ri_same:
        reasons.append("sign differs with province random intercept")
    if len(dev) and dev.max() > tol:
        reasons.append(f"moves up to {dev.max():.1f} pp across sensitivity analyses")
    if cook_ch > tol:
        reasons.append(f"Cook refit moves it {cook_ch:.1f} pp")
    if prob_ch > tol:
        reasons.append(f"probit link moves it {prob_ch:.1f} pp")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    if not scen_ok:
        reasons.append("sign differs under scenario 1 or 4")
    if conc:
        reasons.append("level confined to <=3 provinces, >70% in one, or compared with the reference in <=2 provinces: hard to separate from province/hospital")
    if not_ident:
        reasons.append("not identified within any province")
    verdict = "C" if (r.sparse_flag or not same or not ri_same or not scen_ok or not_ident) else ("A" if not reasons else "B")
    rows.append({"stratum": r.stratum, "variable": r.variable, "level": r.level, "level_n": r.level_n, "AME_pp": r.AME_pp,
                 "lo_patient": r.lo_patient, "hi_patient": r.hi_patient, "CI_excludes_0": sig_p,
                 "lo_province": r.lo_province, "hi_province": r.hi_province, "province_CI_excludes_0": sig_v,
                 "max_change_sens_pp": float(dev.max()) if len(dev) else np.nan, "tolerance_pp": tol,
                 "cook_change_pp": cook_ch, "probit_change_pp": prob_ch, "verdict": verdict, "reasons": "; ".join(reasons)})
for r in ATT.itertuples():
    pop = r.population
    x = SATT[SATT.population == pop] if len(SATT) else pd.DataFrame()
    dev = (x.diff_pp - r.diff_pp).abs() if len(x) else pd.Series(dtype=float)
    tol = max(STABLE_PP, STABLE_REL * abs(r.diff_pp))
    same = bool((np.sign(x.diff_pp) == np.sign(r.diff_pp)).all()) if len(x) else True
    s1 = SCATT[(SCATT.population == pop) & SCATT.scenario.isin(["1 recorded lower bound", "4 upper bound"])]
    scen_same = bool((np.sign(s1.diff_pp) == np.sign(r.diff_pp)).all()) if len(s1) else True
    sig_p = hit(r.boot_lo_patient, r.boot_hi_patient)
    sig_v = hit(r.boot_lo_province, r.boot_hi_province)
    reasons = []
    if not same:
        reasons.append("sign changes in a sensitivity analysis")
    if len(dev) and dev.max() > tol:
        reasons.append(f"moves up to {dev.max():.1f} pp across sensitivity analyses")
    if not scen_same:
        reasons.append("sign differs under scenario 1 or 4")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    rows.append({"stratum": "Global vs itemised (standardised)", "variable": "billing mode", "level": pop, "level_n": r.n_Global,
                 "AME_pp": r.diff_pp, "lo_patient": r.boot_lo_patient, "hi_patient": r.boot_hi_patient, "CI_excludes_0": sig_p,
                 "lo_province": r.boot_lo_province, "hi_province": r.boot_hi_province, "province_CI_excludes_0": sig_v,
                 "max_change_sens_pp": float(dev.max()) if len(dev) else np.nan, "tolerance_pp": tol, "cook_change_pp": np.nan,
                 "probit_change_pp": np.nan, "verdict": "C" if (not same or not scen_same) else ("A" if not reasons else "B"),
                 "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o3_reliability_verdict.csv")
print("A = stable under every check; B = sign stable but size or inference depends on the check named; "
      "C = sparse level, or sign not stable (sensitivity analyses, random intercept, scenarios 1 and 4): do not interpret.")
print(REL[["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient", "verdict", "reasons"]].round(1).to_string(index=False))
print("\nVerdict counts:", REL.verdict.value_counts().to_dict())

# =============================================================================
# 13. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)),
        "n_Global": int((D.stratum == "Global").sum()), "n_Other": int((D.stratum == "Other").sum()),
        "rare_min": RARE_MIN, "boot": NBOOT, "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scenario_B": "PROVISIONAL stand-in: insured/(insured+requested); replace with the cholera paper definition",
        "deviations_from_protocol": ["reconciled ADMISSIONS (Global + Inpatient) are the primary sample",
                                     "model stratified by billing mode",
                                     "year-month period dummies instead of year + month",
                                     "rare categories collapsed within strata",
                                     "sensitivity 7 adds LOS (P95 cap) as a covariate because LOS is not in the O3 model"]}
(OUT_DIR / "o3_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 4 (joint profiles), VERSION 3.1 - final, after the v1/v2/v3 diagnostics. POST-HOC re-specification: label it so.
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran"

Runs on the FROZEN analysis file (read only). numpy, pandas, scipy only (matplotlib optional).

QUESTION THE ANALYSIS NOW ASKS
  v1 and v2 showed that cost, claim lines and patient share are organised by hospital ownership (a tariff regime: patient
  share ~0.10 in public university hospitals, ~0.76 in private) and by province (recording practice). Pooled or
  ownership-stratified clusters mostly re-discover those. The question is therefore: AFTER removing ownership x province
  differences, do itemised admissions fall into stable joint profiles of LOS, claim lines, cost and patient share?

WHAT IS FIXED HERE, BEFORE v3 IS RUN
  Sample      Reconciled ADMISSIONS (Global + Inpatient), ITEMISED only (Global packages have tariff-like costs).
  Inputs      ln(1+LOS capped at group P99), ln(1+claim lines), ln(1+cost Int$), patient share. Each is then centred on its
              ownership x province cell mean (cells < 15 episodes use the ownership mean) and z-scored. [PRIMARY, A]
              Unadjusted inputs are analysed only to show how much structure is ownership/province (analysis B).
  Clustering  k-means, 50 random starts, fixed seed, k = 2..6.
  Choice of k Among k with every cluster >= 5%, weakest-cluster mean COLD-start bootstrap Jaccard (one-to-one matching)
              >= 0.75, silhouette >= 0.25 and above the maximum of Gaussian-copula null data: the SMALLEST k within 0.02
              of the best silhouette. If none qualifies: no profiles are reported (description only, every cluster Tier 3).
  Hard criteria (all clusters)  size >= 5%; bootstrap Jaccard >= 0.75; solution passes the k rule; the choice of k is
              secure (chosen k within 0.02 of the best silhouette in >= 70% of bootstrap resamples); the solution is
              reproducible across 5 random seeds (min pairwise ARI >= 0.90).
  Soft criteria (per cluster)   (i) not province-captured (provinces sending >= 60% of >= 20 episodes into the cluster,
              at >= 1.5x its share, supply < 40% of it); (ii) split-half replication ACROSS PROVINCES (provinces split at
              random into two halves, 12 repeats; matched Jaccard of the cluster between the half-A centres applied to
              half B and half B's own solution, mean >= 0.60); (iii) recovered (matched Jaccard >= 0.60) in every VALID
              sensitivity analysis. S1 (surgical service only) is not valid for a cluster with >= 25% non-surgical episodes.
  Solution flag If the clusters still agree with ownership or province (bias-corrected Cramer's V >= 0.50) the solution is
              flagged "organised by administrative variables" and no cluster can be Tier 1.
  Tiers       Tier 1 = all hard + all soft criteria and no flag. Tier 2 = all hard criteria and at most ONE soft failure (or
              the flag only). Tier 3 = anything else: do not interpret as a profile.
  Sensitivity S1 surgical service only; S2 LOS capped at P95; S3 exclude K81; S4 exclude flagged extremes; S5 exclude
              'Doctor of medicine'; S6 leave out the 3 largest provinces; S7 reconciliation tolerance 10%.
              (Anonymous episodes removed nothing and are dropped; province random intercept does not apply to clustering.)
  Membership  Ridge multinomial model of cluster membership on the M3 block (+ year), and again with province.

CHANGES IN 3.1 (after the v3 output; post hoc)
  * S8 selection-weighted clustering: only 40-50% of itemised admissions are reconciled and the sample over-represents
    episodes with many claim lines (standardised difference 0.31) and under-represents public university hospitals and
    Khuzestan, Mazandaran, Kurdistan. Inverse-probability weights from a ridge-logistic selection model (ownership, province,
    group, surgical service, fund, sex, age, year; truncated at the 1st/99th percentile) feed a WEIGHTED k-means; the
    clusters must be recovered (matched Jaccard >= 0.60), like in every other sensitivity analysis.
  * S1 is not interpretable for this kind of solution when the clusters are partly defined by surgical status (a cluster with
    >= 25% non-surgical episodes disappears when they are removed, which re-arranges the other clusters). The table says so;
    S1 still counts for the clusters it is valid for, and the solution-level ARI is printed.
  * Each cluster gets a "distinctive composition" line (levels of group, ICD-10, specialty, ownership, fund, sex, age with
    lift >= 1.5 and n >= 30, and the share of non-surgical episodes) so that clusters are described by what they contain.

DEVIATIONS FROM THE WRITTEN PROTOCOL (list them in the paper)
  itemised admissions only; inputs centred on ownership x province; parsimony rule, copula null, k-security and seed
  criteria in the choice of k; cold-start bootstrap; tier rules; revised sensitivity set; year instead of year-month.

USAGE
  python objective4_joint_profiles_v3_1.py [analysis_file.csv] [--boot 200] [--starts 50] [--null-b 20] [--k-boot 60] [--quick]
Outputs: folder objective4_v31_outputs next to the analysis file.
"""
import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION (all thresholds are fixed here, before the run)
# =============================================================================
INPUT_CSV = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
                 r"\analysis_ready_final\digestive_surgical_analysis_file.csv")
VERSION = "o4-v3.1"
SEED = 42
K_RANGE = list(range(2, 7))
MIN_CLUSTER_SHARE = 0.05
MIN_JACCARD = 0.75
MIN_SILHOUETTE = 0.25
PARSIMONY_TOL = 0.02
BOOT_STARTS = 5
CAPTURE_FRAC, CAPTURE_MAX, MIN_PROV_N = 0.60, 0.40, 20
REPLICATION_MIN = 0.60
SENS_MIN = 0.60
MIN_STRATUM_N = 300
RARE_MIN = 30
RIDGE_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
N_FOLDS = 5
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
PU = "Public University-Affiliated"
FEATURES = ["LOS (capped, days)", "Claim lines", "Cost (Int$)", "Patient share"]
CELL_MIN = 15
SPLITS = 12
SEED_REPS = 5
SOL_ADMIN_V = 0.50
M3 = ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service", "discharge_status_model",
      "adm_year"]
BLOCK_OF = {"age_group": "predisposing", "sex": "predisposing", "insurance_fund_model": "enabling", "ownership": "enabling",
            "group": "need", "any_surgical_service": "need", "discharge_status_model": "need", "adm_year": "time",
            "province_model": "geography"}
COLLAPSE_VARS = ["ownership", "insurance_fund_model", "discharge_status_model", "sex", "age_group", "group",
                 "any_surgical_service", "province_model"]
REFS = {"ownership": PU, "group": "Hernia", "insurance_fund_model": "Rural Health Insurance Fund", "sex": "Male",
        "age_group": "40-59", "discharge_status_model": "Complete Recovery", "any_surgical_service": "Yes"}

ap = argparse.ArgumentParser()
ap.add_argument("input", nargs="?", default=None)
ap.add_argument("--boot", type=int, default=200)
ap.add_argument("--starts", type=int, default=50)
ap.add_argument("--null-b", type=int, default=20)
ap.add_argument("--k-boot", type=int, default=60)
ap.add_argument("--sens-starts", type=int, default=20)
ap.add_argument("--quick", action="store_true", help="tiny settings for a smoke test only")
args, _ = ap.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
NBOOT, NSTARTS, NULL_B = args.boot, args.starts, args.null_b
KBOOT, SENS_STARTS = args.k_boot, args.sens_starts
if args.quick:
    NBOOT, NSTARTS, NULL_B = 6, 5, 3
    KBOOT, SENS_STARTS = 4, 5
OUT_DIR = INPUT_CSV.parent / "objective4_v31_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 50)
T0 = time.time()


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


# =============================================================================
# CLUSTERING ENGINE
# =============================================================================
def kmeans_fit(Z, k, starts, rng, max_iter=60, tol=1e-4, w=None):
    n, d = Z.shape
    C = np.stack([Z[rng.choice(n, k, replace=False)] for _ in range(starts)])
    x2 = (Z ** 2).sum(1)
    ar = np.arange(k)
    for _ in range(max_iter):
        D2 = x2[None, :, None] - 2 * np.einsum("nd,skd->snk", Z, C) + (C ** 2).sum(2)[:, None, :]
        lab = D2.argmin(2)
        oh = (lab[:, :, None] == ar[None, None, :]).astype(float)
        if w is not None:
            oh = oh * w[None, :, None]
        cnt = oh.sum(1)
        newC = np.where(cnt[:, :, None] > 0, np.einsum("snk,nd->skd", oh, Z) / np.maximum(cnt[:, :, None], 1), C)
        shift = np.abs(newC - C).max()
        C = newC
        if shift < tol:
            break
    D2 = x2[None, :, None] - 2 * np.einsum("nd,skd->snk", Z, C) + (C ** 2).sum(2)[:, None, :]
    lab = D2.argmin(2)
    inertia = (np.take_along_axis(D2, lab[:, :, None], 2)[:, :, 0] * (1 if w is None else w[None, :])).sum(1)
    empty = np.array([len(np.unique(l)) < k for l in lab])
    inertia = np.where(empty, np.inf, inertia)
    b = int(np.argmin(inertia))
    return lab[b], C[b], float(inertia[b])


def fit(Z, k, starts, rng, w=None):
    lab, C, inertia = kmeans_fit(Z, k, starts, rng, w=w)
    order = np.argsort(-np.bincount(lab, minlength=k))
    inv = np.empty_like(order)
    inv[order] = np.arange(k)
    return inv[lab], C[order], inertia


def silhouette(Z, lab, chunk=1000):
    k = lab.max() + 1
    n = len(Z)
    oh = (lab[:, None] == np.arange(k)[None, :]).astype(float)
    cnt = oh.sum(0)
    x2 = (Z ** 2).sum(1)
    s = np.empty(n)
    for i in range(0, n, chunk):
        Zi = Z[i:i + chunk]
        D = np.sqrt(np.maximum(x2[i:i + chunk, None] - 2 * Zi @ Z.T + x2[None, :], 0))
        S = D @ oh
        li = lab[i:i + chunk]
        own = S[np.arange(len(li)), li]
        a = own / np.maximum(cnt[li] - 1, 1)
        M = S / np.maximum(cnt[None, :], 1)
        M[np.arange(len(li)), li] = np.inf
        b = M.min(1)
        si = (b - a) / np.maximum(np.maximum(a, b), 1e-12)
        si[cnt[li] <= 1] = 0
        s[i:i + chunk] = si
    return s


def contingency(a, b, ka=None, kb=None):
    ka = ka or a.max() + 1
    kb = kb or b.max() + 1
    return np.bincount(a * kb + b, minlength=ka * kb).reshape(ka, kb)


def ari(a, b):
    c = contingency(a, b)
    comb = lambda x: x * (x - 1) / 2
    sij, sa, sb, n = comb(c).sum(), comb(c.sum(1)).sum(), comb(c.sum(0)).sum(), comb(c.sum())
    exp = sa * sb / n
    den = 0.5 * (sa + sb) - exp
    return float((sij - exp) / den) if den != 0 else 1.0


def assign(Z, C):
    D2 = (Z ** 2).sum(1)[:, None] - 2 * Z @ C.T + (C ** 2).sum(1)[None, :]
    return D2.argmin(1)


def hungarian(l_ref, l_new, k):
    """One-to-one Jaccard matching of new clusters to reference clusters. Returns matched Jaccard per reference
    cluster and the map new label -> reference label."""
    c = contingency(l_ref, l_new, k, k).astype(float)
    J = c / np.maximum(c.sum(1)[:, None] + c.sum(0)[None, :] - c, 1)
    r, cidx = linear_sum_assignment(-J)
    out = np.zeros(k)
    out[r] = J[r, cidx]
    mp = np.empty(k, int)
    mp[cidx] = r
    return out, mp


def unit_index(d):
    codes = pd.factorize(d["cluster_id"])[0]
    order = np.argsort(codes, kind="stable")
    return np.split(order, np.cumsum(np.bincount(codes))[:-1])


def boot_stab(Z, lab, k, units, B, rng):
    """Patient-level bootstrap, cold start, one-to-one matching. Returns the B x k Jaccard matrix, the B ARIs and the
    pointwise membership confidence (share of resamples in which the episode returns to its own cluster)."""
    n = len(Z)
    jac, aris = np.zeros((B, k)), np.zeros(B)
    hits, cnt = np.zeros(n), np.zeros(n)
    for b in range(B):
        pick = rng.integers(0, len(units), len(units))
        ids = np.concatenate([units[i] for i in pick])
        _, Cb, _ = kmeans_fit(Z[ids], k, BOOT_STARTS, rng)
        U = np.unique(ids)
        lb = assign(Z[U], Cb)
        jac[b], mp = hungarian(lab[U], lb, k)
        aris[b] = ari(lab[U], lb)
        hits[U] += (mp[lb] == lab[U])
        cnt[U] += 1
    return jac, aris, hits / np.maximum(cnt, 1)


def zscore(F):
    sd = F.std(0)
    return (F - F.mean(0)) / np.where(sd > 0, sd, 1)


def feats(d, los="los_capped_p99"):
    return np.column_stack([np.log1p(d[los].to_numpy(float)), np.log1p(d["claim_lines"].to_numpy(float)),
                            np.log1p(d["cost_intl"].to_numpy(float)), d["patient_share"].to_numpy(float)])


def residualise(d, F):
    """Remove province means (provinces with < 15 episodes in the stratum share one pooled mean)."""
    key = d["province_model"].where(d["province_model"].map(d["province_model"].value_counts()) >= 15, "pooled").to_numpy()
    return F - pd.DataFrame(F).groupby(key).transform("mean").to_numpy()


def copula_null_sil(F, k, B, rng):
    """Silhouettes of k-means on Gaussian-copula null data: same marginals and rank correlations as F, no joint cluster
    structure. Skewed or bimodal inputs (patient share) cannot produce an excess by themselves."""
    n, dd = F.shape
    G = stats.norm.ppf((stats.rankdata(F, axis=0) - 0.5) / n)
    R = np.corrcoef(G.T)
    out = []
    for _ in range(B):
        Gs = rng.multivariate_normal(np.zeros(dd), R, size=n)
        Us = stats.norm.cdf(Gs)
        Fs = np.column_stack([np.quantile(F[:, j], Us[:, j]) for j in range(dd)])
        Zs = zscore(Fs)
        lab, _, _ = fit(Zs, k, 10, rng)
        sub = rng.choice(n, min(3000, n), replace=False)
        out.append(silhouette(Zs[sub], lab[sub]).mean() if len(np.unique(lab[sub])) > 1 else np.nan)
    return np.array(out)


def cramers_bc(a, b):
    ct = pd.crosstab(np.asarray(a), np.asarray(b)).to_numpy().astype(float)
    n = ct.sum()
    r, c = ct.shape
    if r < 2 or c < 2 or n < 3:
        return np.nan
    e = ct.sum(1, keepdims=True) * ct.sum(0, keepdims=True) / n
    phi2c = max(0.0, ((ct - e) ** 2 / e).sum() / n - (r - 1) * (c - 1) / (n - 1))
    rc, cc = r - (r - 1) ** 2 / (n - 1), c - (c - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2c / max(min(rc - 1, cc - 1), 1e-12)))




def medians(d, lab):
    d = d.copy()
    d["cluster"] = lab + 1
    q = lambda p: (lambda x: x.quantile(p))
    return d.groupby("cluster").agg(
        n=("episode_no", "size"), LOS_med=("los_capped_p99", "median"), LOS_p25=("los_capped_p99", q(.25)), LOS_p75=("los_capped_p99", q(.75)),
        lines_med=("claim_lines", "median"), lines_p25=("claim_lines", q(.25)), lines_p75=("claim_lines", q(.75)),
        cost_med=("cost_intl", "median"), cost_p25=("cost_intl", q(.25)), cost_p75=("cost_intl", q(.75)),
        share_med=("patient_share", "median"), share_p25=("patient_share", q(.25)), share_p75=("patient_share", q(.75)),
        pct_surgical=("any_surgical_service", lambda x: 100 * (x == "Yes").mean())).reset_index()


def composition(d, lab):
    d = d.copy()
    d["cluster"] = lab + 1
    rows = []
    for var in ["group", "icd10", "ownership", "insurance_fund_model", "specialty_model", "province_model", "any_surgical_service"]:
        ct = pd.crosstab(d[var], d["cluster"])
        overall = ct.sum(1) / ct.sum().sum()
        for lev in ct.index:
            for c in ct.columns:
                rows.append({"variable": var, "level": lev, "cluster": c, "n": int(ct.loc[lev, c]),
                             "pct_of_cluster": 100 * ct.loc[lev, c] / ct[c].sum(), "pct_overall": 100 * overall[lev],
                             "lift": (ct.loc[lev, c] / ct[c].sum()) / overall[lev]})
    return pd.DataFrame(rows)


def province_capture(d, lab, k):
    ct = pd.crosstab(d["province_model"].to_numpy(), lab).reindex(columns=range(k), fill_value=0)
    ntot = ct.sum(1)
    rows = []
    for j in range(k):
        col = ct[j]
        share_c = col / max(col.sum(), 1)
        frac_in = col / ntot
        lift = frac_in / max(col.sum() / ntot.sum(), 1e-12)          # guards against large clusters 'capturing' by chance
        cap = float(share_c[(frac_in >= CAPTURE_FRAC) & (lift >= 1.5) & (ntot >= MIN_PROV_N)].sum())
        rows.append({"cluster": j + 1, "top_province": share_c.idxmax(), "top_province_pct": 100 * share_c.max(),
                     "province_capture": cap, "effective_n_provinces": float(1 / (share_c ** 2).sum())})
    return pd.DataFrame(rows)



def design(d, cats, refs=None):
    ds = d.copy()
    for v in [x for x in cats if x in COLLAPSE_VARS]:
        vc = ds[v].astype(str).value_counts()
        rare = [l for l in vc.index if vc[l] < RARE_MIN]
        if rare:
            tgt = "Other" if (vc[rare].sum() >= RARE_MIN and v not in ("any_surgical_service", "group", "sex")) else vc.index[0]
            ds[v] = ds[v].astype(str).replace({l: tgt for l in rare})
    cols, names, blocks = [], [], []
    for v in cats:
        s = ds[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = (refs or {}).get(v)
        ref = ref if ref in order else order[0]
        for lev in [x for x in order if x != ref]:
            cols.append((s == lev).to_numpy(float))
            names.append(f"{v}[{lev}] (vs {ref})")
            blocks.append(BLOCK_OF.get(v, "other"))
    X = np.column_stack(cols) if cols else np.zeros((len(ds), 0))
    keep = [j for j in range(X.shape[1]) if X[:, j].std() > 0]
    return X[:, keep], [names[j] for j in keep], [blocks[j] for j in keep]


def ridge_multinom(X, y, K, lam):
    n, p = X.shape
    Xa = np.column_stack([np.ones(n), X])
    Y = np.eye(K)[y]

    def f(w):
        W = w.reshape(p + 1, K)
        S = Xa @ W
        lse = logsumexp(S, axis=1)
        nll = -np.mean(np.sum(Y * S, axis=1) - lse)
        P = np.exp(S - lse[:, None])
        G = Xa.T @ (P - Y) / n
        G[1:] += lam * W[1:]
        return nll + 0.5 * lam * np.sum(W[1:] ** 2), G.ravel()
    r = optimize.minimize(f, np.zeros((p + 1) * K), jac=True, method="L-BFGS-B", options={"maxiter": 2000, "maxfun": 5000})
    return r.x.reshape(p + 1, K)


def proba(X, W):
    S = np.column_stack([np.ones(len(X)), X]) @ W
    return np.exp(S - logsumexp(S, axis=1)[:, None])


def membership(d, lab, K, cats, tag):
    dd = d.reset_index(drop=True)
    X, names, blocks = design(dd, cats, REFS)
    uid = pd.factorize(dd["cluster_id"])[0]
    u = np.unique(uid)
    np.random.default_rng(SEED).shuffle(u)
    fm = dict(zip(u, np.arange(len(u)) % N_FOLDS))
    fold = np.array([fm[i] for i in uid])

    def cv(Xm, lam):
        P = np.zeros((len(Xm), K))
        for f_ in range(N_FOLDS):
            tr, te = fold != f_, fold == f_
            P[te] = proba(Xm[te], ridge_multinom(Xm[tr], lab[tr], K, lam))
        return float(-np.mean(np.log(np.clip(P[np.arange(len(lab)), lab], 1e-12, 1)))), float((P.argmax(1) == lab).mean())
    grid = [{"lambda": lam, "cv_logloss": cv(X, lam)[0]} for lam in RIDGE_GRID]
    G = pd.DataFrame(grid)
    lam = float(G.loc[G.cv_logloss.idxmin(), "lambda"])
    ll, acc = cv(X, lam)
    prior = np.bincount(lab, minlength=K) / len(lab)
    null_ll = float(-np.mean(np.log(np.clip(prior[lab], 1e-12, 1))))
    imp = []
    for blk in sorted(set(blocks)):
        keep = [j for j, b in enumerate(blocks) if b != blk]
        if keep:
            ll_b, acc_b = cv(X[:, keep], lam)
            imp.append({"model": tag, "block_dropped": blk, "cv_logloss_increase": ll_b - ll, "cv_accuracy": acc_b})
    summ = {"model": tag, "n": len(lab), "clusters": K, "lambda": lam, "lambda_at_grid_edge": lam in (RIDGE_GRID[0], RIDGE_GRID[-1]),
            "cv_logloss": ll, "cv_logloss_null": null_ll, "logloss_reduction_pct": 100 * (null_ll - ll) / null_ll,
            "cv_accuracy": acc, "baseline_accuracy": float(prior.max()), "terms": len(names)}
    return summ, pd.DataFrame(imp)




def adjust_inputs(d, F):
    """Centre every input on its ownership x province cell mean (cells < CELL_MIN use the ownership mean)."""
    own = d["ownership"].to_numpy()
    key = pd.Series(own + "|" + d["province_model"].to_numpy())
    use = key.map(key.value_counts()) >= CELL_MIN
    k2 = np.where(use, key, own)
    return F - pd.DataFrame(F).groupby(k2).transform("mean").to_numpy()


# =============================================================================
# 1. LOAD, VERIFY, SAMPLE
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "cost_intl", "claim_lines", "los_days", "los_capped_p99", "los_capped_p95",
        "patient_share", "fin_gap", "reconciled", "in_O4", "cluster_id", "any_surgical_service", "specialty_model",
        "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type", "discharge_status",
        "age_group", "sex", "adm_year", "insured_share_irr", "total_cost_irr", "keep_sens3_not_K81",
        "keep_sens4_no_extremes", "keep_sens6_not_doctor_of_medicine"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
DALL = raw.copy()
for c in ["any_surgical_service", "reconciled", "in_O4"] + [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model", "specialty_model",
          "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["cost_intl", "claim_lines", "los_days", "los_capped_p99", "los_capped_p95", "patient_share", "fin_gap"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
cnt_ds = DALL["discharge_status"].value_counts()
DALL["discharge_status_model"] = DALL["discharge_status"].where(~DALL["discharge_status"].isin(cnt_ds[cnt_ds < RARE_MIN].index), "Other")
DALL["itemised"] = DALL["billing_mode"] != "Global"
valid_inputs = (DALL["patient_share"].between(0, 1) & DALL["los_capped_p99"].notna() & DALL["claim_lines"].notna()
                & (DALL["cost_intl"] > 0))
DALL["O4_ok"] = DALL["in_O4"] & DALL["reconciled"] & valid_inputs
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
D = DALL[is_adm & DALL.O4_ok & DALL.itemised].reset_index(drop=True)
flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)},
        {"step": "Admissions (Global + Inpatient)", "n": int(is_adm.sum())},
        {"step": "  itemised (Global packages excluded)", "n": int((is_adm & DALL.itemised).sum())},
        {"step": "  reconciled with all four inputs: PRIMARY SAMPLE", "n": len(D)}]
save(pd.DataFrame(flow), "o4v31_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
if len(D) < 300:
    fail("Primary sample too small.")
units = unit_index(D)
print(f"patients (resampling units): {len(units):,}")

# who is in the reconciled sample (selection)
base = DALL[is_adm & DALL.itemised].copy()
base["in_sample"] = base["episode_no"].isin(D["episode_no"])
sel = []
for v in ["ownership", "group", "any_surgical_service"]:
    t = base.groupby(v)["in_sample"].agg(["size", "mean"]).reset_index()
    t.columns = ["level", "n_itemised_admissions", "pct_in_sample"]
    t["pct_in_sample"] *= 100
    t.insert(0, "variable", v)
    sel.append(t)
t = base.groupby("province_model")["in_sample"].agg(["size", "mean"]).reset_index()
t.columns = ["level", "n_itemised_admissions", "pct_in_sample"]
t = t[t.n_itemised_admissions >= 100]
t["pct_in_sample"] *= 100
t.insert(0, "variable", "province_model (n>=100)")
SEL = pd.concat(sel + [t], ignore_index=True)
save(SEL, "o4v31_selection_into_sample.csv")
print("\nShare of itemised admissions that enter the sample (reconciled), by characteristic:")
print(SEL.round(1).to_string(index=False))
smd = []
for nm, col in [("ln(1+cost Int$)", "cost_intl"), ("ln(1+claim lines)", "claim_lines"), ("ln(1+LOS days)", "los_days")]:
    x = np.log1p(base[col].astype(float))
    a, b = x[base.in_sample].dropna(), x[~base.in_sample].dropna()
    smd.append({"variable": nm, "mean_in": a.mean(), "mean_out": b.mean(),
                "std_diff": (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)})
SMD = pd.DataFrame(smd)
save(SMD, "o4v31_selection_smd.csv")
print(SMD.round(3).to_string(index=False))
# inverse-probability-of-selection weights (observed covariates only)
selcats = ["ownership", "province_model", "group", "any_surgical_service", "insurance_fund_model", "sex", "age_group", "adm_year"]
Xs, _, _ = design(base.reset_index(drop=True), selcats, REFS)
ys = base["in_sample"].to_numpy().astype(int)
Ws = ridge_multinom(Xs, ys, 2, 1e-3)
ps = np.clip(proba(Xs, Ws)[:, 1], 0.02, 1)
w_all = pd.Series(1 / ps, index=base["episode_no"].values)[base["in_sample"].to_numpy()]
lo, hi = np.percentile(w_all, [1, 99])
w_all = w_all.clip(lo, hi)
W_D = w_all.reindex(D["episode_no"].values).to_numpy()
W_D = W_D / W_D.mean()
ESS = float(W_D.sum() ** 2 / (W_D ** 2).sum())
xl = np.log1p(base["claim_lines"].astype(float)).to_numpy()
ins = base["in_sample"].to_numpy()
wb = pd.Series(1 / ps, index=base.index)[ins].clip(lo, hi).to_numpy()
mw = float(np.average(xl[ins], weights=wb))
print(f"Selection weights: claim-lines mean in sample {xl[ins].mean():.3f}, weighted {mw:.3f}, not in sample "
      f"{xl[~ins].mean():.3f}; effective sample size of the weights {ESS:.0f} of {len(D)}.")
print("Only 40-50% of itemised admissions are reconciled and the share varies by province; the results describe the "
      "reconciled subset only.")

# =============================================================================
# 2. WHY ADJUST: inputs by ownership and province (descriptive)
# =============================================================================
hdr("2. INPUTS BY OWNERSHIP AND PROVINCE (descriptive)")
own_t = D.groupby("ownership").agg(n=("episode_no", "size"), share_med=("patient_share", "median"),
                                   share_iqr=("patient_share", lambda x: x.quantile(.75) - x.quantile(.25)),
                                   los_med=("los_capped_p99", "median"), lines_med=("claim_lines", "median"),
                                   cost_med=("cost_intl", "median")).reset_index()
prov_t = D.groupby("province_model").agg(n=("episode_no", "size"), share_med=("patient_share", "median"),
                                         los_med=("los_capped_p99", "median"), lines_med=("claim_lines", "median"),
                                         cost_med=("cost_intl", "median"),
                                         pct_public_univ=("ownership", lambda x: 100 * (x == PU).mean())).reset_index()
prov_t = prov_t[prov_t.n >= 40].sort_values("n", ascending=False)
save(own_t, "o4v31_inputs_by_ownership.csv")
save(prov_t, "o4v31_inputs_by_province.csv")
print(own_t.round(2).to_string(index=False))
print("\nProvinces with >= 40 episodes:")
print(prov_t.round(2).to_string(index=False))
F_raw = feats(D)
F_adj = adjust_inputs(D, F_raw)
r2 = 1 - F_adj.var(0) / F_raw.var(0)
R2T = pd.DataFrame({"input": FEATURES, "share_of_variance_explained_by_ownership_x_province": r2})
save(R2T, "o4v31_variance_explained_by_ownership_province.csv")
print("\nShare of each input's variance explained by ownership x province cells (removed in the primary analysis):")
print(R2T.round(3).to_string(index=False))

# =============================================================================
# 3. B: UNADJUSTED CLUSTERING (how much is ownership / province)
# =============================================================================
hdr("3. ANALYSIS B: UNADJUSTED INPUTS - HOW MUCH STRUCTURE IS OWNERSHIP / PROVINCE?")
rngB = np.random.default_rng(SEED + 5)
ZB = zscore(F_raw)
own_cls = np.where(D.ownership == PU, "PU", "other")
rowsB = []
for k in K_RANGE:
    lab, _, _ = fit(ZB, k, NSTARTS, rngB)
    rowsB.append({"k": k, "silhouette": float(silhouette(ZB, lab).mean()), "ARI_ownership_class": ari(lab, pd.factorize(own_cls)[0]),
                  "V_ownership_class": cramers_bc(lab, own_cls), "V_ownership": cramers_bc(lab, D.ownership),
                  "V_province": cramers_bc(lab, D.province_model)})
TB = pd.DataFrame(rowsB)
save(TB, "o4v31_B_unadjusted_association.csv")
print(TB.round(3).to_string(index=False))
print("High V with ownership / province = the unadjusted clusters are administrative groupings, not patterns of care.")

# =============================================================================
# 4. A: PRIMARY ANALYSIS (inputs centred on ownership x province)
# =============================================================================
hdr("4. ANALYSIS A (PRIMARY): INPUTS CENTRED ON OWNERSHIP x PROVINCE")
rng = np.random.default_rng(SEED)
ZA = zscore(F_adj)
n = len(ZA)
rows, store = [], {}
for k in K_RANGE:
    lab, C, _ = fit(ZA, k, NSTARTS, rng)
    sil_i = silhouette(ZA, lab)
    jac, aris, conf = boot_stab(ZA, lab, k, units, NBOOT, rng)
    nul = copula_null_sil(F_adj, k, NULL_B, rng)
    sizes = np.bincount(lab, minlength=k)
    mj = jac.mean(0)
    row = {"k": k, "silhouette": float(sil_i.mean()), "copula_null_max": float(np.nanmax(nul)),
           "excess_over_null": float(sil_i.mean() - np.nanmax(nul)), "min_cluster_share": float(sizes.min() / n),
           "min_jaccard_cold": float(mj.min()), "mean_jaccard_cold": float(mj.mean()), "ari_boot_mean": float(aris.mean()),
           "cluster_sizes": " / ".join(map(str, sizes))}
    row["passes_rule"] = bool(row["min_cluster_share"] >= MIN_CLUSTER_SHARE and row["min_jaccard_cold"] >= MIN_JACCARD
                              and row["silhouette"] >= MIN_SILHOUETTE and row["excess_over_null"] > 0)
    rows.append(row)
    store[k] = {"lab": lab, "C": C, "sil_i": sil_i, "jac": jac, "conf": conf, "sizes": sizes}
    print(f"  k={k}: silhouette {row['silhouette']:.3f} (copula null max {row['copula_null_max']:.3f}), min share "
          f"{row['min_cluster_share']:.3f}, min cold Jaccard {row['min_jaccard_cold']:.2f}, passes: {row['passes_rule']}")
T = pd.DataFrame(rows)

# security of the choice of k: silhouette bootstrap over patients
hdr("4b. HOW SECURE IS THE CHOICE OF k?")
sil_boot = np.zeros((KBOOT, len(K_RANGE)))
for b in range(KBOOT):
    pick = rng.integers(0, len(units), len(units))
    ids = np.unique(np.concatenate([units[i] for i in pick]))
    Zb = ZA[ids]
    for j, k in enumerate(K_RANGE):
        lb, _, _ = fit(Zb, k, 10, rng)
        sil_boot[b, j] = silhouette(Zb, lb).mean() if len(np.unique(lb)) > 1 else np.nan
ok = T[T.passes_rule]
if len(ok):
    best = ok.silhouette.max()
    K = int(ok[ok.silhouette >= best - PARSIMONY_TOL].k.min())
    passes = True
else:
    cand = T[T.min_cluster_share >= MIN_CLUSTER_SHARE]
    K = int((cand if len(cand) else T).sort_values("silhouette", ascending=False).k.iloc[0])
    passes = False
jK = K_RANGE.index(K)
within = float(np.mean(sil_boot[:, jK] >= np.nanmax(sil_boot, axis=1) - PARSIMONY_TOL))
winner = pd.Series(np.array(K_RANGE)[np.nanargmax(sil_boot, axis=1)]).value_counts(normalize=True).reindex(K_RANGE, fill_value=0)
K_SECURE = bool(within >= 0.70)
print("Share of resamples in which each k has the highest silhouette: " + ", ".join(f"k={k}: {100 * v:.0f}%" for k, v in winner.items()))
print(f"Chosen k = {K}: within {PARSIMONY_TOL} of the best silhouette in {100 * within:.0f}% of resamples -> secure: {K_SECURE}")
T["boot_winner_share"] = winner.values
T["chosen"] = (T.k == K) & passes
save(T, "o4v31_A_k_selection.csv")
print(T.round(3).to_string(index=False))

# reproducibility across seeds
S = store[K]
labs = [S["lab"]] + [fit(ZA, K, NSTARTS, np.random.default_rng(SEED + 100 + i))[0] for i in range(SEED_REPS - 1)]
min_ari_seed = min(ari(labs[i], labs[j]) for i in range(len(labs)) for j in range(i + 1, len(labs)))
SEED_OK = bool(min_ari_seed >= 0.90)
print(f"Reproducibility across {SEED_REPS} seeds at k = {K}: minimum pairwise ARI {min_ari_seed:.3f} -> {SEED_OK}")
print(f"\n>>> k = {K}" + ("" if passes else "  (NO k passes the rule: DESCRIPTION ONLY, every cluster is Tier 3)"))

lab = S["lab"]
z_centres = pd.DataFrame([ZA[lab == j].mean(0) for j in range(K)], columns=FEATURES)
med = medians(D, lab)
desc = []
for j in range(K):
    tg = [f"{s} {'high' if z > .5 else 'low'} (vs own hospital type/province)" for z, s in zip(z_centres.iloc[j], ["LOS", "lines", "cost", "share"]) if abs(z) > .5]
    desc.append(", ".join(tg) if tg else "near average")
med["description"] = desc
comp0 = composition(D, lab)
dist = []
for j in range(K):
    c0 = comp0[(comp0.cluster == j + 1) & (comp0.n >= 30) & (comp0.lift >= 1.5)
               & comp0.variable.isin(["group", "icd10", "specialty_model", "ownership", "insurance_fund_model"])]
    c0 = c0.sort_values("lift", ascending=False).head(4)
    items = [f"{r.level} ({r.pct_of_cluster:.0f}%, x{r.lift:.1f})" for r in c0.itertuples()]
    ns = 100 * float((D.any_surgical_service.to_numpy()[lab == j] == "No").mean())
    items.append(f"non-surgical {ns:.0f}%")
    dist.append("; ".join(items))
med["distinctive_composition"] = dist
save(med, "o4v31_A_profiles_medians.csv")
save(z_centres.assign(cluster=np.arange(1, K + 1)), "o4v31_A_profiles_centres_z.csv")
comp = composition(D, lab)
save(comp, "o4v31_A_profiles_composition.csv")
print("\nMedians of the inputs by cluster (original scale; the clusters are defined on centred inputs):")
print(med[["cluster", "n", "LOS_med", "lines_med", "cost_med", "share_med", "pct_surgical"]].round(2).to_string(index=False))
for r in med.itertuples():
    print(f"  cluster {r.cluster}: {r.description}\n     distinctive composition: {r.distinctive_composition}")
for var in ["ownership", "group"]:
    print(f"\n{var}: % of each cluster")
    print(comp[comp.variable == var].pivot(index="level", columns="cluster", values="pct_of_cluster").round(1).to_string())

# solution-level association with administrative variables
V_own, V_prov = cramers_bc(lab, D.ownership), cramers_bc(lab, D.province_model)
ARI_own = ari(lab, pd.factorize(D.ownership)[0])
FLAG = bool((V_own >= SOL_ADMIN_V) or (V_prov >= SOL_ADMIN_V))
cap = province_capture(D, lab, K)
print(f"\nCramer's V (bias-corrected) of clusters with ownership = {V_own:.2f}, with province = {V_prov:.2f}; "
      f"ARI with ownership {ARI_own:.2f}. Flag 'organised by administrative variables': {FLAG}")
print(cap.round(2).to_string(index=False))
save(pd.DataFrame([{"k": K, "V_ownership": V_own, "V_province": V_prov, "ARI_ownership": ARI_own, "flag_admin": FLAG,
                    "k_secure": K_SECURE, "within_tol_share": within, "seed_min_ARI": min_ari_seed}]), "o4v31_A_solution_checks.csv")

# split-half replication across provinces
hdr("4c. SPLIT-HALF REPLICATION ACROSS PROVINCES")
provs = D["province_model"].to_numpy()
up = np.unique(provs)
rep2 = np.full((SPLITS, K), np.nan)
rs = np.random.default_rng(SEED + 7)
for s_ in range(SPLITS):
    perm = rs.permutation(up)
    A_p = set(perm[: len(perm) // 2])
    inA = np.array([p in A_p for p in provs])
    if inA.sum() < 100 or (~inA).sum() < 100:
        continue
    for half in (inA, ~inA):
        labh, _, _ = fit(ZA[half], K, 30, rs)
        mj, _ = hungarian(lab[half], labh, K)        # primary cluster j (ref) vs the half's own solution
        rep2[s_] = np.where(np.isnan(rep2[s_]), mj, (rep2[s_] + mj) / 2)
rep_primary = np.nanmean(rep2, axis=0)
print("Primary cluster recovered in random province halves (mean matched Jaccard): " + str(np.round(rep_primary, 2).tolist()))

# sensitivity analyses
hdr("4d. SENSITIVITY ANALYSES S1-S7 (matched Jaccard per primary cluster)")
ref_pos = pd.Series(np.arange(n), index=D["episode_no"].values)
top3 = D.province_model.value_counts().index[:3]
tol = DALL[is_adm & DALL.itemised & (DALL.fin_gap <= 0.10) & DALL.insured_share_irr.notna() & DALL.total_cost_irr.gt(0) & valid_inputs]
SENS = {"S1 surgical service only": (D[D.any_surgical_service == "Yes"], "los_capped_p99"),
        "S2 LOS capped at P95": (D[D.los_capped_p95.notna()], "los_capped_p95"),
        "S3 exclude K81": (D[D.keep_sens3_not_K81], "los_capped_p99"),
        "S4 exclude flagged extremes": (D[D.keep_sens4_no_extremes], "los_capped_p99"),
        "S5 exclude Doctor of medicine": (D[D.keep_sens6_not_doctor_of_medicine], "los_capped_p99"),
        "S6 leave out 3 largest provinces": (D[~D.province_model.isin(top3)], "los_capped_p99"),
        "S7 reconciliation tolerance 10%": (tol, "los_capped_p99")}
srows = []
non_surg = (D.any_surgical_service == "No").to_numpy()
for sname, (dn, los) in SENS.items():
    dn = dn.reset_index(drop=True)
    Fn = feats(dn, los)
    labn, _, _ = fit(zscore(adjust_inputs(dn, Fn)), K, SENS_STARTS, rng)
    m = dn["episode_no"].isin(ref_pos.index).to_numpy()
    l_ref = lab[ref_pos.loc[dn.loc[m, "episode_no"]].to_numpy()]
    mj, _ = hungarian(l_ref, labn[m], K)
    for j in range(K):
        valid = True
        if sname.startswith("S1") and non_surg[lab == j].mean() >= 0.25:
            valid = False
        srows.append({"sens": sname, "n": len(dn), "n_common": int(m.sum()), "cluster": j + 1, "matched_jaccard": float(mj[j]),
                      "ARI": ari(l_ref, labn[m]), "valid_for_cluster": valid})
# S8: selection-weighted clustering (same sample, weighted k-means on the same adjusted inputs)
lab8, _, _ = fit(ZA, K, SENS_STARTS, rng, w=W_D)
mj8, _ = hungarian(lab, lab8, K)
for j in range(K):
    srows.append({"sens": "S8 selection-weighted (IPW)", "n": n, "n_common": n, "cluster": j + 1, "matched_jaccard": float(mj8[j]),
                  "ARI": ari(lab, lab8), "valid_for_cluster": True})
SENSDF = pd.DataFrame(srows)
save(SENSDF, "o4v31_A_sensitivity.csv")
print(SENSDF.assign(m=np.where(SENSDF.valid_for_cluster, SENSDF.matched_jaccard, np.nan)).pivot(index="sens", columns="cluster", values="m").round(2).to_string())
print("(blank = analysis not valid for that cluster)")
s1_ari = float(SENSDF[SENSDF.sens.str.startswith("S1")].ARI.iloc[0])
print(f"S1 solution-level ARI with the primary labels: {s1_ari:.2f}. Clusters that are partly defined by non-surgical episodes "
      f"cannot be recovered when those episodes are removed, and the remaining clusters re-arrange: S1 shows how much the "
      f"solution depends on the non-surgical episodes, not instability of the clusters it is not valid for.")

# tiers
mjb = S["jac"].mean(0)
trows = []
for j in range(K):
    sj = SENSDF[(SENSDF.cluster == j + 1) & SENSDF.valid_for_cluster].matched_jaccard
    c = {"cluster": j + 1, "n": int(S["sizes"][j]), "share_pct": 100 * S["sizes"][j] / n, "description": desc[j],
         "boot_jaccard_cold": float(mjb[j]), "cluster_silhouette": float(S["sil_i"][lab == j].mean()),
         "province_capture": float(cap.province_capture[j]), "top_province": cap.top_province[j],
         "split_half_jaccard": float(rep_primary[j]), "sens_min_matched_jaccard": float(sj.min()) if len(sj) else np.nan}
    c["size_ok"] = c["share_pct"] >= 100 * MIN_CLUSTER_SHARE
    c["stability_ok"] = c["boot_jaccard_cold"] >= MIN_JACCARD
    c["rule_ok"] = bool(passes)
    c["k_secure"] = K_SECURE
    c["seed_ok"] = SEED_OK
    c["province_ok"] = c["province_capture"] < CAPTURE_MAX
    c["replication_ok"] = c["split_half_jaccard"] >= REPLICATION_MIN
    c["sensitivity_ok"] = bool(len(sj) and sj.min() >= SENS_MIN)
    c["separation_ok"] = c["cluster_silhouette"] >= MIN_SILHOUETTE
    hard = c["size_ok"] and c["stability_ok"] and c["rule_ok"] and c["k_secure"] and c["seed_ok"]
    soft_fail = [x for x, ok_ in [("province-captured", c["province_ok"]), ("not replicated across province halves", c["replication_ok"]),
                                  ("not recovered in every valid sensitivity analysis", c["sensitivity_ok"]),
                                  ("diffuse: cluster silhouette < 0.25", c["separation_ok"])] if not ok_]
    if hard and not soft_fail and not FLAG:
        c["tier"] = 1
    elif hard and (len(soft_fail) + int(FLAG) <= 1):
        c["tier"] = 2
    else:
        c["tier"] = 3
    fails = ([] if c["size_ok"] else ["too small"]) + ([] if c["stability_ok"] else ["unstable under bootstrap"]) + \
            ([] if c["rule_ok"] else ["solution fails the k rule"]) + ([] if c["k_secure"] else ["choice of k not secure"]) + \
            ([] if c["seed_ok"] else ["not reproducible across seeds"]) + soft_fail + (["clusters still organised by ownership/province"] if FLAG else [])
    c["reasons"] = "; ".join(fails)
    trows.append(c)
TIER = pd.DataFrame(trows)
save(TIER, "o4v31_A_cluster_tiers.csv")

# membership models
hdr("5. MEMBERSHIP MODELS (ridge multinomial, patient-grouped cross-validation)")
cats = list(M3)
ms1, mi1 = membership(D, lab, K, cats, "M3 (+ year)")
ms2, mi2 = membership(D, lab, K, cats + ["province_model"], "M3 + province")
MS, MI = pd.DataFrame([ms1, ms2]), pd.concat([mi1, mi2], ignore_index=True)
save(MS, "o4v31_A_membership_summary.csv")
save(MI, "o4v31_A_membership_block_importance.csv")
print(MS.round(3).to_string(index=False))
print(MI.round(3).to_string(index=False))
print("Reading: a small reduction in log-loss means that case mix, fund and facility type do not explain the clusters.")
pd.DataFrame({"episode_no": D["episode_no"], "cluster": lab + 1, "confidence": S["conf"]}).to_csv(
    OUT_DIR / "o4v31_A_episode_assignments.csv", index=False, encoding="utf-8-sig")

# =============================================================================
# 6. SUMMARY
# =============================================================================
hdr("6. SUMMARY: WHICH PROFILES CAN BE REPORTED?")
show = ["cluster", "n", "share_pct", "description", "boot_jaccard_cold", "province_capture", "top_province", "split_half_jaccard",
        "sens_min_matched_jaccard", "tier", "reasons"]
print(TIER[show].round(2).to_string(index=False))
n1, n2, n3 = [int((TIER.tier == t).sum()) for t in (1, 2, 3)]
print(f"\nTier 1 (report as profiles): {n1}; Tier 2 (report with the named weakness): {n2}; Tier 3 (do not interpret): {n3}")
if n1 + n2 == 0:
    print("RESULT: no reliable distinct joint profiles. Report O4 as a negative finding: LOS, claim lines, cost and patient share "
          "are organised by ownership (tariff regime) and province practice; after removing those, no stable, replicable "
          "groups remain.")
else:
    print("RESULT: some groups pass the hard criteria; report ONLY Tier 1-2 clusters, with the weaknesses named, and "
          "label the analysis post hoc / exploratory.")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.5, 0.7 * K + 2.2))
    M = z_centres.to_numpy()
    im = ax.imshow(M, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(4), ["LOS", "Lines", "Cost", "Share"])
    ax.set_yticks(range(K), [f"C{j + 1} (n={int(TIER.n[j])}, tier {TIER.tier[j]})" for j in range(K)])
    ax.set_title("Primary analysis: centres in z-units (inputs centred on ownership x province)")
    for a in range(K):
        for b in range(4):
            ax.text(b, a, f"{M[a, b]:+.1f}", ha="center", va="center", fontsize=9)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o4v31_profile_heatmap.png", dpi=200)
    plt.close(fig)
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")

info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "seed": SEED, "starts": NSTARTS, "boot": NBOOT, "null_b": NULL_B, "k_boot": KBOOT,
        "n_primary": int(n), "chosen_k": K, "k_rule_passed": bool(passes), "k_secure": K_SECURE, "seed_ok": SEED_OK,
        "flag_admin": FLAG, "tiers": {"1": n1, "2": n2, "3": n3},
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
        "post_hoc_respecification": True,
        "thresholds": {"min_cluster_share": MIN_CLUSTER_SHARE, "min_jaccard": MIN_JACCARD, "min_silhouette": MIN_SILHOUETTE,
                       "parsimony_tol": PARSIMONY_TOL, "capture_max": CAPTURE_MAX, "replication_min": REPLICATION_MIN,
                       "sens_min": SENS_MIN, "admin_V": SOL_ADMIN_V},
        "deviations_from_protocol": ["itemised admissions only", "inputs centred on ownership x province",
                                     "parsimony rule + copula null + k-security + seed reproducibility in the choice of k",
                                     "cold-start bootstrap with one-to-one matching", "tier rules replace A/B/C",
                                     "revised sensitivity set S1-S7", "year instead of year-month in the membership model"]}
(OUT_DIR / "o4v31_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")


1. LOAD AND VERIFY   [elapsed 0.0 min]
                                             step     n
             Episodes in the frozen analysis file 15601
                  Admissions (Global + Inpatient) 14685
              itemised (Global packages excluded)  6223
  reconciled with all four inputs: PRIMARY SAMPLE  2674
patients (resampling units): 2,647

Share of itemised admissions that enter the sample (reconciled), by characteristic:
               variable                            level  n_itemised_admissions  pct_in_sample
              ownership                       Charitable                    112           77.7
              ownership          Non-Governmental Public                    698           58.9
              ownership                          Private                   2920           43.4
              ownership Public Non-University-Affiliated                    113           85.0
              ownership     Public University-Affiliated                   2380      

In [29]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Stage B of the analysis protocol: the three gaps to close on the RAW CLAIM LINES, before episodes are built.
Affects ALL objectives (O1-O4): duplicate lines double-count cost (O2, O3, O4), the LOS column feeds O1, O2 and O4,
and anonymous episodes can merge or split admissions (patient clusters in every model).

  B1  Exact duplicate claim lines (identical in every mapped column, across sheets): counted per sheet, dropped, and the
      cost they would have added is reported. The protocol puts this step BEFORE the episode key is built.
  B2  LOS from dates: Jalali admission and discharge dates -> days, compared with the recorded LOS column, per EPISODE
      (the build script takes the maximum recorded LOS over the lines of an episode). Reports agreement and the episodes
      that disagree, and writes the lines you need to add to the build script so that los_from_dates reaches the analysis file
      BEFORE the dates are dropped.
  B3  Missing patient IDs: share of lines and of episodes per sheet whose key would be a signature (anonymous).

This script only AUDITS. It does not build episodes. Run it on the same Excel file as the build script, then (if B1 finds
duplicates) add the drop step to the build script, add los_from_dates, rerun the build and the whole chain.

COLUMN MAP. The Persian headers are mapped through ALIASES below. I have not seen your build script, so the aliases are
GUESSES: if the script stops with "unmapped canonical columns", copy the accepted aliases from the build script into
ALIASES (or pass --map-json my_map.json with {"canonical": ["header1", "header2"]}).

USAGE
  python stage_b_raw_checks.py raw_claims.xlsx [--out stage_b_outputs] [--map-json map.json] [--los-tol 1.0]
"""
import argparse
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

CODES = ["K40", "K41", "K42", "K43", "K45", "K46", "K80", "K81", "K60", "K61", "K62", "K64"]
REQUIRED = ["icd10", "specialty", "adm_date", "dis_date", "total_cost"]
OPTIONAL = ["patient_id", "los", "insured_share", "requested_share", "paid_share", "age", "sex", "ownership", "fund",
            "record_type", "province", "service_group"]
ALIASES = {  # canonical -> accepted headers (after text normalisation). EDIT to match the build script.
    "patient_id": ["patient_id", "patient_no", "شناسه بیمار", "کد بیمار", "شماره بیمار", "کد ملی"],
    "icd10": ["icd10", "icd_10", "کد تشخیص", "تشخیص", "کد بیماری"],
    "specialty": ["specialty", "تخصص", "تخصص پزشک"],
    "adm_date": ["adm_date", "admission_date", "تاریخ پذیرش", "تاریخ بستری"],
    "dis_date": ["dis_date", "discharge_date", "تاریخ ترخیص"],
    "total_cost": ["total_cost", "total_cost_irr", "هزینه کل", "جمع کل", "مبلغ کل"],
    "insured_share": ["insured_share", "سهم بیمه شده", "سهم بیمار"],
    "requested_share": ["requested_share", "سهم درخواستی", "مبلغ درخواستی"],
    "paid_share": ["paid_share", "سهم پرداختی", "مبلغ پرداختی"],
    "los": ["los", "los_days", "مدت اقامت", "طول مدت بستری", "روز بستری"],
    "age": ["age", "سن"], "sex": ["sex", "gender", "جنس"],
    "ownership": ["ownership", "مالکیت", "نوع مالکیت"],
    "fund": ["fund", "insurance_fund", "صندوق", "صندوق بیمه"],
    "record_type": ["record_type", "نوع پرونده", "نوع رکورد"],
    "province": ["province", "استان"],
    "service_group": ["service_group", "گروه خدمت", "گروه خدمات"],
}
PERSIAN_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def norm_text(x):
    if pd.isna(x):
        return x
    s = str(x).replace("ي", "ی").replace("ك", "ک").replace("‌", " ").translate(PERSIAN_DIGITS)
    return re.sub(r"\s+", " ", s).strip()


def jalali_day_number(y, m, d):
    """Linear day count for a Jalali date (differences between two dates are exact days; valid 1178-1633)."""
    y = y + 1595
    return (-355668 + 365 * y + (y // 33) * 8 + ((y % 33) + 3) // 4 + d + np.where(m < 7, (m - 1) * 31, (m - 7) * 30 + 186))


def parse_jalali(s):
    """'1401/05/12', '1401-5-12', '14010512' (Persian digits allowed) -> (y, m, d) float arrays, NaN when invalid."""
    t = s.map(norm_text).astype("string")
    parts = t.str.extract(r"^\s*(\d{4})\D?(\d{1,2})\D?(\d{1,2})")
    y, m, d = (pd.to_numeric(parts[i], errors="coerce") for i in range(3))
    ok = y.between(1300, 1500) & m.between(1, 12) & d.between(1, 31)
    return y.where(ok), m.where(ok), d.where(ok)


# ---- NOTEBOOK USERS: put the path of the raw claims workbook here (or leave None and use the command line) ----
RAW_FILE = None          # e.g. r"C:\Users\me\project\raw_claims.xlsx"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("raw", nargs="?", default=None)
    ap.add_argument("--out", default="stage_b_outputs")
    ap.add_argument("--map-json", default=None)
    ap.add_argument("--los-tol", type=float, default=1.0, help="days of disagreement counted as a mismatch")
    a, _ = ap.parse_known_args()          # ignores the extra arguments Jupyter passes
    a.raw = RAW_FILE or a.raw
    if not a.raw:
        raise SystemExit("Set RAW_FILE at the top of the script (or pass the raw workbook on the command line).")
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    aliases = {k: [norm_text(v) for v in vs] for k, vs in ALIASES.items()}
    if a.map_json:
        for k, vs in json.loads(Path(a.map_json).read_text(encoding="utf-8")).items():
            aliases[k] = aliases.get(k, []) + [norm_text(v) for v in vs]

    xl = pd.read_excel(a.raw, sheet_name=None, dtype=str)
    frames, skipped, unmapped_headers = [], [], {}
    for sh, df in xl.items():
        df = df.copy()
        df.columns = [norm_text(c) for c in df.columns]
        ren = {}
        for canon, al in aliases.items():
            hit = [c for c in df.columns if c in al]
            if hit:
                ren[hit[0]] = canon
        unmapped_headers[sh] = [c for c in df.columns if c not in ren]
        df = df.rename(columns=ren)
        miss = [c for c in REQUIRED if c not in df.columns]
        if miss:
            skipped.append({"sheet": sh, "missing_required": ", ".join(miss), "rows": len(df)})
            continue
        df["sheet"] = sh
        frames.append(df)
    if skipped:
        print("Sheets skipped (missing required columns):")
        print(pd.DataFrame(skipped).to_string(index=False))
        print("\nHeaders that were not mapped, per sheet (use them to extend ALIASES):")
        for sh, cols in unmapped_headers.items():
            print(f"  {sh}: {cols}")
    if not frames:
        sys.exit("STOP: unmapped canonical columns in every sheet. Edit ALIASES or pass --map-json.")
    L = pd.concat(frames, ignore_index=True)
    for c in L.columns:
        if c != "sheet":
            L[c] = L[c].map(norm_text)
    L["icd10"] = L["icd10"].astype("string").str.upper().str.extract(r"([A-Z]\d{2})", expand=False)
    n_read = len(L)
    L = L[L["icd10"].isin(CODES)].copy()
    kept = L.groupby("sheet").size().rename("lines_kept")
    read = pd.concat([f.groupby("sheet").size() for f in frames]).rename("lines_read")
    lines_tab = pd.concat([read, kept], axis=1).fillna(0).astype(int)
    print(f"\nLines read (sheets with required columns): {n_read:,}; kept (12 codes): {len(L):,}")
    for c in ["total_cost", "insured_share", "requested_share", "paid_share", "los", "age"]:
        if c in L.columns:
            L[c] = pd.to_numeric(L[c].astype("string").str.replace(",", "", regex=False), errors="coerce")

    # ------------------------------------------------------------------ B1 duplicates
    cols = [c for c in L.columns if c != "sheet"]
    dup_any = L.duplicated(subset=cols, keep="first")
    dup_tab = (L.assign(dup=dup_any).groupby("sheet").agg(lines=("dup", "size"), duplicate_lines=("dup", "sum"),
                                                          duplicate_cost=("total_cost", lambda s: s[dup_any.loc[s.index]].sum())))
    dup_tab["pct_duplicate"] = 100 * dup_tab.duplicate_lines / dup_tab.lines
    # duplicates ACROSS sheets only (the first copy sits on a different sheet)
    h = pd.util.hash_pandas_object(L[cols].astype(str), index=False)
    first_sheet = L.assign(h=h).groupby("h")["sheet"].first()
    cross = int((h[dup_any].map(first_sheet).values != L.loc[dup_any, "sheet"].values).sum())
    print("\nB1. EXACT DUPLICATE CLAIM LINES")
    print(dup_tab.round(2).to_string())
    print(f"Total duplicate lines: {int(dup_any.sum()):,} of {len(L):,} ({100 * dup_any.mean():.2f}%); on a different sheet from the "
          f"first copy: {cross:,}; cost double-counted if kept: {dup_tab.duplicate_cost.sum():,.0f} IRR "
          f"({100 * dup_tab.duplicate_cost.sum() / max(L.total_cost.sum(), 1):.2f}% of total).")
    print("Caution: a repeated line within an episode can also be a genuine repeated service (same code, same day, same amount). "
          "Report both counts; drop only lines that are identical on ALL columns INCLUDING the claim/line identifier if the extract has one.")
    dup_tab.reset_index().to_csv(out / "stage_b1_duplicates_by_sheet.csv", index=False, encoding="utf-8-sig")
    Ld = L[~dup_any].copy()

    # ------------------------------------------------------------------ B2 LOS from dates
    ya, ma, da = parse_jalali(Ld["adm_date"])
    yd, md, dd = parse_jalali(Ld["dis_date"])
    Ld["adm_dn"] = jalali_day_number(ya, ma, da)
    Ld["dis_dn"] = jalali_day_number(yd, md, dd)
    Ld["los_dates"] = Ld["dis_dn"] - Ld["adm_dn"]
    Ld["adm_key"] = Ld["adm_date"].astype("string")
    Ld["dis_key"] = Ld["dis_date"].astype("string")
    pid = Ld["patient_id"] if "patient_id" in Ld.columns else pd.Series(pd.NA, index=Ld.index)
    Ld["anonymous"] = pid.isna() | (pid.astype("string").str.strip() == "")
    sig_cols = [c for c in ["adm_date", "dis_date", "age", "sex", "ownership", "fund", "record_type", "province", "icd10"] if c in Ld.columns]
    _sig = Ld[sig_cols].astype(str).apply(lambda col: col).agg("|".join, axis=1) if len(Ld) else pd.Series(dtype=str)
    Ld["episode_key"] = np.where(Ld["anonymous"], "SIG|" + _sig,
                                 "PID|" + pid.astype(str) + "|" + Ld["adm_date"].astype(str) + "|" + Ld["dis_date"].astype(str))
    print("\nB2. LOS FROM DATES vs RECORDED LOS (per episode; recorded = maximum over the lines, as in the build script)")
    bad_dates = int(Ld["los_dates"].isna().sum())
    neg = int((Ld["los_dates"] < 0).sum())
    print(f"Lines with an unparseable Jalali date: {bad_dates:,}; with discharge before admission: {neg:,}")
    if "los" in Ld.columns:
        E = Ld.groupby("episode_key").agg(los_dates=("los_dates", "first"), los_recorded=("los", "max"),
                                          lines=("los", "size"), anonymous=("anonymous", "first"),
                                          sheet=("sheet", "first")).reset_index()
        E = E.dropna(subset=["los_dates", "los_recorded"])
        E["diff"] = E["los_recorded"] - E["los_dates"]
        E["mismatch"] = E["diff"].abs() > a.los_tol
        E["zero_day_stay_recorded_fraction"] = (E["los_dates"] == 0) & (E["los_recorded"] > 0)
        cmp_tab = pd.DataFrame([{"episodes_compared": len(E), "corr": E[["los_dates", "los_recorded"]].corr().iloc[0, 1],
                                 "median_diff_recorded_minus_dates": E["diff"].median(),
                                 "mean_diff": E["diff"].mean(), "pct_within_0.5d": 100 * (E["diff"].abs() <= 0.5).mean(),
                                 f"pct_mismatch_gt_{a.los_tol:g}d": 100 * E["mismatch"].mean(),
                                 "pct_recorded_fractional": 100 * ((E.los_recorded % 1) != 0).mean(),
                                 "pct_dates_same_day_but_recorded_positive": 100 * E["zero_day_stay_recorded_fraction"].mean()}])
        print(cmp_tab.round(3).T.to_string(header=False))
        print("Reading: the protocol defines an overnight stay as LOS >= 1 day on a FRACTIONAL recorded LOS. Calendar-date LOS is an "
              "integer (same-day discharge = 0). If the two differ systematically, report the overnight-stay results under both.")
        cmp_tab.to_csv(out / "stage_b2_los_comparison_summary.csv", index=False, encoding="utf-8-sig")
        E.drop(columns=["episode_key"]).to_csv(out / "stage_b2_los_by_episode_no_ids.csv", index=False, encoding="utf-8-sig")
        by_sheet = E.groupby("sheet").agg(episodes=("diff", "size"), pct_mismatch=("mismatch", "mean"), median_diff=("diff", "median"))
        by_sheet["pct_mismatch"] *= 100
        by_sheet.reset_index().to_csv(out / "stage_b2_los_mismatch_by_sheet.csv", index=False, encoding="utf-8-sig")
        print(by_sheet.round(2).to_string())
    else:
        print("No recorded LOS column was mapped: only the date-based LOS can be computed.")
    print("\nTo carry this into the analysis file, add to the build script BEFORE the dates are dropped (episode level):\n"
          "    los_from_dates = (jalali_day_number(dis_y, dis_m, dis_d) - jalali_day_number(adm_y, adm_m, adm_d))\n"
          "    # take the same value for every line of the episode; keep the recorded LOS column as well, and report both.")

    # ------------------------------------------------------------------ B3 anonymous
    print("\nB3. MISSING PATIENT IDs (episodes keyed by a signature)")
    anon_lines = Ld.groupby("sheet")["anonymous"].agg(lines="size", anonymous_lines="sum")
    anon_lines["pct_anonymous_lines"] = 100 * anon_lines.anonymous_lines / anon_lines.lines
    ep = Ld.groupby(["sheet", "episode_key"])["anonymous"].first().reset_index()
    anon_eps = ep.groupby("sheet")["anonymous"].agg(episodes="size", anonymous_episodes="sum")
    anon_eps["pct_anonymous_episodes"] = 100 * anon_eps.anonymous_episodes / anon_eps.episodes
    A = anon_lines.join(anon_eps)
    print(A.round(2).to_string())
    print(f"Overall: {100 * Ld.anonymous.mean():.2f}% of lines; {100 * ep.anonymous.mean():.2f}% of episodes.")
    # how many signature episodes could be a merge of several real admissions (many lines + long signature group)
    sig = Ld[Ld.anonymous].groupby("episode_key").agg(lines=("icd10", "size"))
    if len(sig):
        print(f"Signature episodes with unusually many lines (> 30): {(sig.lines > 30).sum():,} of {len(sig):,} "
              "(possible merges of separate admissions).")
    A.reset_index().to_csv(out / "stage_b3_anonymous_by_sheet.csv", index=False, encoding="utf-8-sig")
    lines_tab.reset_index().rename(columns={"index": "sheet"}).to_csv(out / "stage_b0_lines_by_sheet.csv", index=False, encoding="utf-8-sig")

    summary = {"lines_read": int(n_read), "lines_kept": int(len(L)), "duplicate_lines": int(dup_any.sum()),
               "duplicate_cost_irr": float(dup_tab.duplicate_cost.sum()), "pct_anonymous_lines": float(100 * Ld.anonymous.mean()),
               "pct_anonymous_episodes": float(100 * ep.anonymous.mean()), "sheets_skipped": skipped}
    (out / "stage_b_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nDone. Outputs in {out.resolve()}. Put the counts in the flow table (lines read / kept / duplicates dropped).")


if __name__ == "__main__":
    main()

SystemExit: Set RAW_FILE at the top of the script (or pass the raw workbook on the command line).

In [34]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 1 (resource utilisation) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".   VERSION 3

Runs on the FROZEN analysis file written by clean_digestive_surgical_episodes.py (read only).
Needs only numpy, pandas and scipy (no statsmodels / patsy): the GLM, Firth and cluster-robust
engines are written out below.  They were checked against statsmodels 0.14.5 on the real file:
dispersion, rate ratios and Firth odds ratios agree to four decimals.

OUTCOMES                                                         MODEL
  1. Claim lines per episode        quasi-Poisson, log link       rate ratio (RR)
  2. Any overnight stay (LOS >= 1)  Firth penalised logistic      odds ratio (OR)
  3. LOS among overnight stays      Gamma-log GLM on P99-capped   ratio of means
                                    LOS; quasi-Poisson as check
BLOCKS (Andersen), every model also carries Jalali year and month
  M1 age group, sex | M2 + fund, ownership, billing mode | M3 + group, surgical service, discharge
  status | M4 + province | M5 + specialty

------------------------------------------------------------------------------------------------
WHAT CHANGED IN VERSION 3 AND WHY (all of it is driven by the diagnostic output on the real file)
------------------------------------------------------------------------------------------------
  1. PRIMARY SAMPLE = ADMISSIONS (record types Global + Inpatient, n = 14,685).
     Emergency Observation (889 episodes: LOS 0.08 d, 1% overnight, 8% surgical) and Temporary
     inpatient admission (27) are not admissions, yet they sat inside the "Other" billing mode.
     They inflated the surgical-service odds ratio from about 5 to 12.5 and pulled every Global-vs-
     other contrast.  The protocol's sample (all valid episodes) is still fitted and compared
     (o1_sample_comparison.csv; use --sample all to make it primary).  THIS IS A DEVIATION FROM THE
     FROZEN PROTOCOL: state it in the Methods and label it as made after seeing the diagnostics.
  2. BILLING MODE IS ANALYSED BY PROVINCE.  The Global-minus-itemised gap in overnight share runs from
     -21 to +66 percentage points across provinces (billing x province chi2 = 501).  A pooled OR is an
     average of opposite effects.  The script now fits province-specific billing effects, pools them
     with a DerSimonian-Laird random-effects model and reports I2, tau and the 95% prediction interval.
  3. MODELS STRATIFIED BY BILLING MODE (Global only / Inpatient only) because group x billing and
     surgical service x billing interactions are large (e.g. Gallbladder overnight OR 9.1 in Global vs
     3.1 in Inpatient; claim-line RR 0.93 vs 1.15).  Surgical service is almost constant inside Global
     (99.6%), so its effect is only estimable in Inpatient.
  4. TWO SETS OF STANDARD ERRORS.  Patient clusters are almost singletons (15,214 clusters for 15,601
     episodes), so they equal ordinary robust SEs; province clusters are 2-8x wider for facility-level
     exposures (ICC after M3 = 0.18 for claim lines).  Both are reported; province-cluster CIs use t with
     G-1 df.  Province coefficients themselves have no province-cluster SE (they are absorbed).
  5. SPARSE CELLS (kept from v2): level flagged when n < 30 or < 10 in an outcome class; Firth sandwich
     SE replaced by profile penalised-likelihood CI (M5) or the larger model SE when the model SE is
     > 1.5x the sandwich SE.
  6. EXPLICIT RELIABILITY VERDICT per main-table estimate (o1_reliability_summary.csv):
        A  direction and size hold      B  direction holds, size varies by billing mode
        C  not robust (the column `why` lists the failed criteria)
     The pre-specified rule (direction holds in all sensitivity analyses) is one of the criteria.
  7. Tables now print the number of episodes behind each level; the specialty and province rows with
     fewer than 100 episodes are marked as not interpretable on their own.
  Sensitivity 8 (province random intercept) is still NOT run here; the province-clustered models and
  the by-province random-effects analysis are Python stand-ins, not replacements.

OUTPUTS (folder objective1_outputs next to the analysis file)
  o1_sample_flow.csv, o1_descriptives.csv
  o1_models_long.csv (patient clusters) / o1_models_province_cluster.csv (M5, province clusters)
  o1_main_table.csv, o1_main_table_stratified.csv
  o1_billing_by_province.csv, o1_billing_pooled.csv
  o1_reliability_summary.csv
  o1_sensitivity_long.csv, o1_sensitivity_direction.csv
  o1_sample_comparison.csv, o1_exploratory_long.csv, o1_exploratory_core_terms.csv
  o1_diagnostics_*.csv (dispersion, gvif, m5_vs_m3, sparse_cells, firth_vs_ml, interactions,
                        icc_province, se_by_cluster_level, los_bins, los_top_values, overnight_cutoffs,
                        record_type)
  o1_run_info.json

USAGE
  python objective1_resource_utilisation.py [analysis_file.csv] [--sample admissions|all]
                                            [--no-sens] [--no-extra] [--no-profile]
"""

import argparse
import hashlib
import json
import platform
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import expit

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
RARE_MIN = 30             # protocol rule (discharge status, counts only)
VIF_FLAG = 5.0
TOP_PROVINCES = 5
SPARSE_N_MIN = 30         # level flagged when it has fewer episodes than this ...
SPARSE_CELL_MIN = 10      # ... or fewer than this in either outcome class (separation risk)
SE_NARROW_RATIO = 1.5     # model SE / sandwich SE above this -> sandwich not trusted
MIN_LEVEL_N = 100         # levels below this are shown but marked "not interpretable"
SPREAD_FLAG = 1.5         # stratum-specific ratios differing by more than this factor = "size varies"
PROV_MIN_PER_MODE = 30    # provinces enter the billing-by-province analysis with >= this many per mode
OVERNIGHT_CUTOFFS = [0.5, 0.75]
LOPO_N = 8
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--no-profile", action="store_true")
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_EXTRA, RUN_PROFILE = not args.no_sens, not args.no_extra, not args.no_profile
OUT_DIR = INPUT_CSV.parent / "objective1_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode",
           "group", "any_surgical_service", "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
TIME = ["adm_year", "adm_month"]
NO_BILLING = lambda b: [t for t in b if t != "billing_mode"]
OUTCOMES = {"claim_lines": ("claim_lines", "quasipoisson"),
            "overnight": ("overnight", "firth"),
            "los": ("los_capped_p99", "gamma")}
EFFECT_LABEL = {"claim_lines": "RR", "overnight": "OR", "los": "ratio", "los_qp": "RR"}
CORE_VARS = ["age_group", "billing_mode", "group", "any_surgical_service"]
REPORT_VARS = ["ownership", "billing_mode", "group", "any_surgical_service", "specialty_model", "province_model"]


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


# =============================================================================
# 1. LOAD, VERIFY, DEFINE THE PRIMARY SAMPLE
# =============================================================================
print("=" * 78)
print("1. LOAD AND VERIFY")
print("=" * 78)
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need_cols = ["episode_no", "icd10", "group", "claim_lines", "los_days", "overnight", "los_capped_p99",
             "los_capped_p95", "cluster_id", "any_surgical_service", "specialty_model", "province_model",
             "insurance_fund_model", "ownership", "billing_mode", "record_type", "discharge_status",
             "age_group", "sex", "adm_year", "adm_month", "anonymous", "keep_sens1_surgical",
             "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
             "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine"]
missing = [c for c in need_cols if c not in raw.columns]
if missing:
    fail(f"Columns missing from the analysis file: {missing}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous"] + [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["claim_lines", "los_days", "overnight", "los_capped_p99", "los_capped_p95"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")

cnt = DALL["discharge_status"].value_counts()
rare = cnt[cnt < RARE_MIN].index.tolist()
DALL["discharge_status_model"] = DALL["discharge_status"].where(~DALL["discharge_status"].isin(rare), "Other")
print(f"discharge_status: {len(cnt)} categories, {len(rare)} collapsed to 'Other'.")

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
flow.append({"step": "  of which record type Global", "n": int((DALL.record_type == "Global").sum())})
flow.append({"step": "  of which record type Inpatient (itemised)", "n": int((DALL.record_type == "Inpatient").sum())})
for t in NON_ADMISSION:
    flow.append({"step": f"  of which record type {t} (not an admission)", "n": int((DALL.record_type == t).sum())})
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types found: {unknown_rt}; they are treated as admissions.")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
D = DALL[is_adm].copy() if args.sample == "admissions" else DALL.copy()
ALT = DALL.copy() if args.sample == "admissions" else DALL[is_adm].copy()
SAMPLE_LABEL = ("admissions (Global + Inpatient)" if args.sample == "admissions"
                else "all valid episodes (protocol sample)")
ALT_LABEL = "all valid episodes (protocol sample)" if args.sample == "admissions" else "admissions only"
flow.append({"step": f"PRIMARY SAMPLE: {SAMPLE_LABEL}", "n": len(D)})
save(pd.DataFrame(flow), "o1_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))

print("\nOutcome checks (primary sample)")
print(f"  claim_lines: median {D.claim_lines.median():.0f}, IQR {D.claim_lines.quantile(.25):.0f}-"
      f"{D.claim_lines.quantile(.75):.0f}, min {D.claim_lines.min():.0f}, max {D.claim_lines.max():.0f}, "
      f"missing {D.claim_lines.isna().sum()}")
print(f"  overnight stay: {100 * D.overnight.mean():.1f}% (missing {D.overnight.isna().sum()})")
print(f"  LOS among overnight stays: n = {int((D.overnight == 1).sum()):,}, "
      f"median {D.loc[D.overnight == 1, 'los_days'].median():.2f} d")

CAT_VARS = sorted({v for b in BLOCKS.values() for v in b} | set(TIME))
REFS = {v: D[v].value_counts().index[0] for v in CAT_VARS}
REFS["billing_mode"] = "Other"             # Other = itemised inpatient in the primary sample
REFS["any_surgical_service"] = "No"
REFS["adm_month"] = "1"
print("Reference levels:", {k: REFS[k] for k in BLOCKS["M5"]})
CUT_COL = {}
for c in OVERNIGHT_CUTOFFS:
    col = f"overnight_c{int(round(c * 100)):03d}"
    DALL[col] = np.where(DALL["los_days"].notna(), (DALL["los_days"] >= c).astype(float), np.nan)
    CUT_COL[c] = col
    D[col] = DALL.loc[D.index, col]
    ALT[col] = DALL.loc[ALT.index, col]


# =============================================================================
# 2. MODEL ENGINE (numpy / scipy)
# =============================================================================
def design(d, terms, extra_inter=()):
    """Dummy design matrix. Columns are named like C(var)[T.level]; the reference is REFS[var]
    (or the most common level in this sample when REFS[var] is absent). Constant columns are dropped."""
    cols, names, levs = [np.ones(len(d))], ["Intercept"], {}
    for v in terms:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = REFS.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        for lev in levs[v][1:]:
            cols.append((s == lev).to_numpy(float))
            names.append(f"C({v})[T.{lev}]")
    for a, b in extra_inter:
        for la in levs[a][1:]:
            for lb in levs[b][1:]:
                cols.append(((d[a].astype(str) == la) & (d[b].astype(str) == lb)).to_numpy(float))
                names.append(f"C({a})[T.{la}]:C({b})[T.{lb}]")
    X = np.column_stack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], levs


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def glm_log(X, y, groups, fam):
    """Log-link GLM (Poisson or Gamma) by IRLS with Liang-Zeger cluster-robust covariance (CR1)."""
    n, k = X.shape
    wf = (lambda m: m) if fam == "poisson" else (lambda m: np.ones_like(m))
    beta = np.zeros(k)
    beta[0] = np.log(y.mean())
    mu = np.full(n, y.mean())
    eta = np.log(mu)
    for _ in range(100):
        w = wf(mu)
        z = eta + (y - mu) / mu
        XtW = X.T * w
        b_new = np.linalg.solve(XtW @ X, XtW @ z)
        eta = X @ b_new
        mu = np.exp(np.clip(eta, -30, 30))
        done = np.abs(b_new - beta).max() < 1e-10
        beta = b_new
        if done:
            break
    bread = np.linalg.inv((X.T * wf(mu)) @ X)
    V = mu if fam == "poisson" else mu ** 2
    meat, g = cluster_meat(X * ((y - mu) / V * mu)[:, None], groups)
    cov = (g / (g - 1)) * ((n - 1) / (n - k)) * bread @ meat @ bread
    return {"beta": beta, "cov": cov, "G": g, "disp": float(np.sum((y - mu) ** 2 / V) / (n - k)),
            "resid": (y - mu) / np.sqrt(V), "conv": bool(done)}


def _pen_ll(beta, X, y):
    eta = X @ beta
    w = expit(eta) * (1 - expit(eta))
    ev = np.linalg.eigvalsh(X.T @ (X * w[:, None]))
    return np.sum(y * eta - np.logaddexp(0, eta)) + 0.5 * np.sum(np.log(ev[ev > 1e-10]))


def firth_irls(X, y, fixed=None, beta0=None, maxiter=200, tol=1e-8):
    """Firth penalised logistic regression; fixed=(j, value) holds beta_j fixed (profile likelihood)."""
    n, k = X.shape
    beta = np.zeros(k) if beta0 is None else np.array(beta0, float)
    free = np.arange(k)
    if fixed is not None:
        beta[fixed[0]] = fixed[1]
        free = np.array([i for i in range(k) if i != fixed[0]])
    conv = False
    for _ in range(maxiter):
        p = expit(X @ beta)
        w = p * (1 - p)
        info = X.T @ (X * w[:, None])
        inv = np.linalg.pinv(info)
        A = X * np.sqrt(w)[:, None]
        h = np.einsum("ij,ij->i", A @ inv, A)
        u = X.T @ (y - p + h * (0.5 - p))
        step = np.zeros(k)
        if fixed is None:
            step = inv @ u
        else:
            step[free] = np.linalg.pinv(info[np.ix_(free, free)]) @ u[free]
        mx = np.abs(step).max()
        if mx > 5:
            step *= 5 / mx
        f0, t = _pen_ll(beta, X, y), 1.0
        while t > 1e-4 and _pen_ll(beta + t * step, X, y) < f0 - 1e-10:
            t /= 2
        beta = beta + t * step
        # converged when the step is tiny OR the penalised score is ~0 (the line search can stall on
        # floating-point noise in sparse designs although the penalised likelihood is already flat)
        if np.abs(t * step).max() < tol or np.abs(u[free]).max() < 1e-4:
            conv = True
            break
    return beta, conv, _pen_ll(beta, X, y)


def firth_fit(X, y, groups):
    n, k = X.shape
    beta, conv, pl = firth_irls(X, y)
    p = expit(X @ beta)
    w = p * (1 - p)
    inv = np.linalg.pinv(X.T @ (X * w[:, None]))
    A = X * np.sqrt(w)[:, None]
    h = np.einsum("ij,ij->i", A @ inv, A)
    meat, g = cluster_meat(X * (y - p + h * (0.5 - p))[:, None], groups)
    cov = ((g / (g - 1)) * ((n - 1) / max(n - k, 1)) if g > 1 else 1.0) * inv @ meat @ inv
    return {"beta": beta, "cov": cov, "cov_model": inv, "G": g, "conv": conv, "pl": pl,
            "resid": (y - p) / np.sqrt(w)}


def firth_profile_ci(X, y, j, beta, pl_max, se_hint, level=0.95):
    """Profile penalised-likelihood CI and LR p-value for coefficient j (valid under separation)."""
    crit = stats.chi2.ppf(level, 1)

    def g(b):
        return 2 * (pl_max - firth_irls(X, y, fixed=(j, b), beta0=beta)[2]) - crit

    bounds = []
    for sgn in (-1, 1):
        step, found = max(se_hint, 0.25), None
        for _ in range(10):
            if g(beta[j] + sgn * step) > 0:
                found = beta[j] + sgn * step
                break
            step *= 2
        if found is None:
            bounds.append(sgn * np.inf)
        else:
            a, c = (beta[j], found) if sgn > 0 else (found, beta[j])
            bounds.append(optimize.brentq(g, a, c, xtol=1e-4))
    pl0 = firth_irls(X, y, fixed=(j, 0.0), beta0=beta)[2]
    return bounds[0], bounds[1], stats.chi2.sf(max(2 * (pl_max - pl0), 0.0), 1)


def ml_logit(X, y, maxiter=60):
    """Plain maximum-likelihood logistic regression (damped Newton), only to show separation."""
    beta = np.zeros(X.shape[1])
    for _ in range(maxiter):
        p = expit(X @ beta)
        w = np.clip(p * (1 - p), 1e-12, None)
        step = np.linalg.pinv(X.T @ (X * w[:, None])) @ (X.T @ (y - p))
        mx = np.abs(step).max()
        if mx > 3:
            step *= 3 / mx
        beta = beta + step
        if np.abs(step).max() < 1e-8:
            break
    w = np.clip(expit(X @ beta) * (1 - expit(X @ beta)), 1e-12, None)
    return beta, np.sqrt(np.diag(np.linalg.pinv(X.T @ (X * w[:, None]))))


def icc_anova(resid, groups):
    df = pd.DataFrame({"r": np.asarray(resid, float), "g": np.asarray(groups)})
    gm = df.groupby("g")["r"]
    k, n = gm.ngroups, len(df)
    ni = gm.size().to_numpy()
    ssb = np.sum(ni * (gm.mean().to_numpy() - df["r"].mean()) ** 2)
    ssw = np.sum((df["r"] - gm.transform("mean")) ** 2)
    msb, msw = ssb / (k - 1), ssw / (n - k)
    n0 = (n - np.sum(ni ** 2) / n) / (k - 1)
    return (msb - msw) / (msb + (n0 - 1) * msw)


def term_flags(X, names, y_bin):
    """Sparse-cell flag per column: level n < 30, or (binary outcome) < 10 in an outcome class."""
    flags, ns = {}, {}
    for j, nm in enumerate(names):
        if nm == "Intercept":
            continue
        n_l = int(X[:, j].sum())
        f = n_l < SPARSE_N_MIN
        if y_bin is not None and n_l > 0:
            n1 = int((X[:, j] * y_bin).sum())
            f = f or min(n1, n_l - n1) < SPARSE_CELL_MIN
        flags[nm], ns[nm] = bool(f), n_l
    return flags, ns


def fit_core(d, key, terms, y_col=None, family=None, cluster="patient", extra_inter=(), profile=False):
    outcome_col, fam = OUTCOMES[key]
    outcome_col = y_col or outcome_col
    fam = family or fam
    sub = d.dropna(subset=[outcome_col]).copy()
    if key == "los":                                   # LOS among overnight stays only
        sub = sub[(sub["overnight"] == 1) & (sub[outcome_col] > 0)]
    X, names, _ = design(sub, terms + TIME, extra_inter)
    y = sub[outcome_col].to_numpy(float)
    groups = pd.factorize(sub["cluster_id" if cluster == "patient" else "province_model"])[0]
    core = {"n": len(sub), "fam": fam, "cluster": cluster, "names": names, "profile": {}, "sub_index": sub.index}
    if fam == "firth":
        f = firth_fit(X, y, groups)
        se_rob, se_mod = np.sqrt(np.diag(f["cov"])), np.sqrt(np.diag(f["cov_model"]))
        narrow = (se_mod / np.where(se_rob > 0, se_rob, np.nan) > SE_NARROW_RATIO) & (np.array(names) != "Intercept")
        se = np.where(narrow, np.maximum(se_rob, se_mod), se_rob)
        if profile and narrow.any():
            for j in np.where(narrow)[0]:
                try:
                    core["profile"][int(j)] = firth_profile_ci(X, y, int(j), f["beta"], f["pl"], se_mod[j])
                except Exception as e:
                    print(f"    profile CI failed for {names[j]}: {type(e).__name__}: {e}")
        core.update(beta=f["beta"], se=se, se_sandwich=se_rob, se_model=se_mod, cov=f["cov"], G=f["G"],
                    narrow=narrow, converged=f["conv"], dispersion=np.nan, family_label="Firth logit",
                    resid=f["resid"], X=X, y=y)
        core["flags"], core["lev_n"] = term_flags(X, names, y)
    else:
        r = glm_log(X, y, groups, "poisson" if fam == "quasipoisson" else "gamma")
        core.update(beta=r["beta"], se=np.sqrt(np.diag(r["cov"])), se_sandwich=np.sqrt(np.diag(r["cov"])),
                    se_model=np.full(len(names), np.nan), cov=r["cov"], G=r["G"], narrow=np.zeros(len(names), bool),
                    converged=r["conv"], dispersion=r["disp"], resid=r["resid"], X=X, y=y,
                    family_label="quasi-Poisson (Poisson + robust SE)" if fam == "quasipoisson" else "Gamma-log")
        core["flags"], core["lev_n"] = term_flags(X, names, None)
    return core


def make_tab(core, label, model, extra=None):
    beta, se, names, G = core["beta"], core["se"], core["names"], core["G"]
    use_t = core["cluster"] == "province"
    crit = stats.t.ppf(0.975, G - 1) if use_t else stats.norm.ppf(0.975)
    lo, hi = beta - crit * se, beta + crit * se
    z = beta / np.where(se > 0, se, np.nan)
    p = 2 * (stats.t.sf(np.abs(z), G - 1) if use_t else stats.norm.sf(np.abs(z)))
    base = "cluster-robust SE (province, t)" if use_t else "cluster-robust SE (patient)"
    method = np.array([base] * len(names), dtype=object)
    for j in np.where(core["narrow"])[0]:
        method[j] = "model-based SE (sandwich too narrow)"
    for j, (l, h_, pj) in core["profile"].items():
        lo[j], hi[j], p[j] = l, h_, pj
        method[j] = "profile penalised likelihood"
    out = pd.DataFrame({
        "outcome": label, "model": model, "term": names, "n": core["n"], "level_n": [core["lev_n"].get(t, np.nan) for t in names],
        "ratio": np.exp(beta), "lo": np.exp(lo), "hi": np.exp(hi), "p": p, "se_log": se,
        "se_log_sandwich": core["se_sandwich"], "se_log_model": core["se_model"], "ci_method": method,
        "sparse_flag": [bool(core["flags"].get(t, False)) for t in names], "cluster": core["cluster"]})
    if extra:
        for k, v in extra.items():
            out[k] = v
    return out[out["term"] != "Intercept"]


def fit_one(d, key, model, terms, label=None, quiet=False, **kw):
    label = label or key
    try:
        core = fit_core(d, key, terms, **kw)
        tab = make_tab(core, label, model)
        disp = {"outcome": label, "model": model, "n": core["n"], "family": core["family_label"],
                "dispersion": core["dispersion"], "converged": core["converged"], "cluster": core["cluster"]}
        if not quiet:
            extra = f", {int(core['narrow'].sum())} term(s) with sandwich SE too narrow" if core["narrow"].any() else ""
            print(f"  {label:<12} {model}: n = {core['n']:,}, terms = {len(tab)}"
                  + (f", dispersion = {disp['dispersion']:.2f}" if np.isfinite(disp["dispersion"]) else "")
                  + extra + ("" if disp["converged"] else "  ** did not converge **"))
        return tab, disp, core
    except Exception as e:
        print(f"  {label:<12} {model}: FAILED ({type(e).__name__}: {e})")
        return None, None, None


# =============================================================================
# 3. DESCRIPTIVES
# =============================================================================
def iqr_str(s, dec=1):
    s = s.dropna()
    return "" if len(s) == 0 else f"{s.median():.{dec}f} ({s.quantile(.25):.{dec}f}-{s.quantile(.75):.{dec}f})"


def describe(d, var, label):
    rows = []
    for lev, g in ([("All", d)] if var is None else d.groupby(var, observed=True)):
        gi = g[g["overnight"] == 1]
        rows.append({"variable": label, "level": lev, "episodes": len(g),
                     "claim_lines_median_IQR": iqr_str(g["claim_lines"], 0),
                     "overnight_n": int((g["overnight"] == 1).sum()),
                     "overnight_pct": round(100 * g["overnight"].mean(), 1),
                     "LOS_all_median_IQR": iqr_str(g["los_days"], 2),
                     "LOS_overnight_median_IQR": iqr_str(gi["los_days"], 2),
                     "surgical_service_pct": round(100 * (g["any_surgical_service"] == "Yes").mean(), 1)})
    return rows


print("\n" + "=" * 78)
print("2. DESCRIPTIVES")
print("=" * 78)
desc = describe(DALL, None, "All valid episodes (protocol sample)") + describe(DALL, "record_type", "Record type (all episodes)")
desc += describe(D, None, "PRIMARY SAMPLE")
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("billing_mode", "Billing mode"),
                 ("specialty_model", "Specialty")]:
    desc += describe(D, var, lab)
desc = pd.DataFrame(desc)
save(desc, "o1_descriptives.csv")
print(desc[desc.variable.isin(["All valid episodes (protocol sample)", "Record type (all episodes)", "PRIMARY SAMPLE",
                               "Billing mode", "Group"])].to_string(index=False))

lb = pd.cut(DALL["los_days"], [0, .25, .5, .75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0, np.inf], right=False)
los_bins = (pd.crosstab(D["group"], pd.cut(D["los_days"], lb.cat.categories, right=False), normalize="index") * 100).round(1)
los_bins.columns = [str(c) for c in los_bins.columns]
save(los_bins.reset_index(), "o1_diagnostics_los_bins.csv")
top = D["los_days"].round(2).value_counts().head(15).rename_axis("los_rounded").reset_index(name="episodes")
save(top, "o1_diagnostics_los_top_values.csv")
cut = pd.DataFrame({"cutoff_days": [0.25, 0.5, 0.75, 1.0, 1.25]})
cut["overnight_pct"] = [round(100 * (D["los_days"].dropna() >= c).mean(), 1) for c in cut.cutoff_days]
save(cut, "o1_diagnostics_overnight_cutoffs.csv")
save(pd.DataFrame(describe(DALL, "record_type", "Record type")), "o1_diagnostics_record_type.csv")

sp_rows = []
for v in sorted(set(BLOCKS["M5"]) | set(TIME)):
    for lev, r in D.groupby(v)["overnight"].agg(n="size", overnight="sum").iterrows():
        n_l, n1 = int(r["n"]), int(r["overnight"])
        sp_rows.append({"variable": v, "level": lev, "n": n_l, "overnight": n1, "not_overnight": n_l - n1,
                        "flag_n_lt30": n_l < SPARSE_N_MIN, "flag_n_lt100": n_l < MIN_LEVEL_N,
                        "flag_separation_risk": min(n1, n_l - n1) < SPARSE_CELL_MIN or n_l < SPARSE_N_MIN})
SPARSE = pd.DataFrame(sp_rows)
save(SPARSE, "o1_diagnostics_sparse_cells.csv")
print(f"\nSparse cells (n < {SPARSE_N_MIN} or < {SPARSE_CELL_MIN} in an outcome class): "
      f"{int(SPARSE.flag_separation_risk.sum())} levels; levels with n < {MIN_LEVEL_N}: {int(SPARSE.flag_n_lt100.sum())}.")

# =============================================================================
# 4. MAIN MODELS M1-M5 (patient clusters, as pre-specified)
# =============================================================================
print("\n" + "=" * 78)
print(f"3. MODELS M1-M5, primary sample: {SAMPLE_LABEL}, n = {len(D):,}")
print("=" * 78)
warnings.filterwarnings("ignore")
long_tabs, disp_rows, CORE = [], [], {}
for model, blocks in BLOCKS.items():
    prof = RUN_PROFILE and model == "M5"
    for key in OUTCOMES:
        tab, disp, core = fit_one(D, key, model, blocks, profile=prof)
        if tab is not None:
            long_tabs.append(tab); disp_rows.append(disp); CORE[(key, model)] = core
    tab, disp, core = fit_one(D, "los", model, blocks, family="quasipoisson", label="los_qp")
    if tab is not None:
        long_tabs.append(tab); disp_rows.append(disp); CORE[("los_qp", model)] = core
LONG = pd.concat(long_tabs, ignore_index=True)
LONG["analysis_status"] = "pre-specified models; primary sample redefined after diagnostics"
save(LONG, "o1_models_long.csv")
save(pd.DataFrame(disp_rows), "o1_diagnostics_dispersion.csv")

# M5 with province-clustered SEs (all outcomes)
print("\nModel M5 with province-clustered SEs (28 clusters; t with G-1 df)")
pc_tabs = []
for key in list(OUTCOMES) + ["los_qp"]:
    k_, fam_ = ("los", "quasipoisson") if key == "los_qp" else (key, None)
    tab, _, _ = fit_one(D, k_, "M5", BLOCKS["M5"], label=key, family=fam_, cluster="province")
    if tab is not None:
        pc_tabs.append(tab)
PC = pd.concat(pc_tabs, ignore_index=True)
PC = PC[~PC.term.str.contains("province_model")]       # province coefficients have no province-cluster SE
PC["analysis_status"] = "exploratory: province-clustered SE"
save(PC, "o1_models_province_cluster.csv")

# =============================================================================
# 5. STRATIFIED MODELS BY BILLING MODE, M5 (Global only / itemised only)
# =============================================================================
print("\n" + "=" * 78)
print("4. M5 STRATIFIED BY BILLING MODE (Global only | Other = itemised only)")
print("=" * 78)
ST_tabs = []
for lab, mask in [("Global", D.billing_mode == "Global"), ("Other", D.billing_mode != "Global")]:
    sub = D[mask]
    for key in OUTCOMES:
        for cl in ["patient", "province"]:
            tab, _, _ = fit_one(sub, key, f"M5 | {lab}", NO_BILLING(BLOCKS["M5"]), label=key, cluster=cl,
                                profile=RUN_PROFILE and cl == "patient", quiet=(cl == "province"))
            if tab is not None:
                tab["stratum"] = lab
                ST_tabs.append(tab)
ST = pd.concat(ST_tabs, ignore_index=True)
ST["analysis_status"] = "exploratory (added after diagnostics)"
save(ST, "o1_models_stratified.csv")

# =============================================================================
# 6. BILLING MODE BY PROVINCE, random-effects pooling
# =============================================================================
print("\n" + "=" * 78)
print("5. BILLING-MODE EFFECT BY PROVINCE (Global vs itemised), random-effects pooling")
print("=" * 78)


def drsl(b, se):
    w = 1 / se ** 2
    fe = np.sum(w * b) / w.sum()
    Q = np.sum(w * (b - fe) ** 2)
    k = len(b)
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - np.sum(w ** 2) / w.sum()))
    ws = 1 / (se ** 2 + tau2)
    re, se_re = np.sum(ws * b) / ws.sum(), np.sqrt(1 / ws.sum())
    t = stats.t.ppf(.975, k - 2)
    pi_half = t * np.sqrt(tau2 + se_re ** 2)
    return {"k": k, "fixed": fe, "re": re, "se_re": se_re, "tau": np.sqrt(tau2),
            "I2_pct": 100 * max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0, "Q": Q,
            "Q_p": stats.chi2.sf(Q, k - 1), "pi_lo": re - pi_half, "pi_hi": re + pi_half}


BP_rows, BP_pool = [], []
if args.sample == "all":
    print("  (skipped: the billing-by-province analysis needs the admissions-only sample, where "
          "Other = itemised inpatient)")
    BPD = None
else:
    ct = pd.crosstab(D.province_model, D.billing_mode)
    PROV = [p for p in ct.index if ct.loc[p].min() >= PROV_MIN_PER_MODE and p != "Other"]
    BPD = D[D.province_model.isin(PROV)]
    ref_prov = BPD.province_model.value_counts().index[0]
    REFS_SAVE = REFS.get("province_model")
    REFS["province_model"] = ref_prov
    print(f"  {len(PROV)} provinces with >= {PROV_MIN_PER_MODE} Global and >= {PROV_MIN_PER_MODE} itemised "
          f"episodes ({len(BPD):,} episodes); reference province: {ref_prov}")
    for key in OUTCOMES:
        core = fit_core(BPD, key, BLOCKS["M5"], extra_inter=[("billing_mode", "province_model")])
        names, beta, cov = core["names"], core["beta"], core["cov"]
        jb = names.index("C(billing_mode)[T.Global]")
        est, sel, used = [], [], []
        for p in PROV:
            L = np.zeros(len(names)); L[jb] = 1
            nm = f"C(billing_mode)[T.Global]:C(province_model)[T.{p}]"
            if nm in names:
                L[names.index(nm)] = 1
            elif p != ref_prov:
                continue
            e, s = float(L @ beta), float(np.sqrt(L @ cov @ L))
            g_ = BPD[BPD.province_model == p]
            raw_ = {"overnight_Global_pct": 100 * g_[g_.billing_mode == "Global"].overnight.mean(),
                    "overnight_Other_pct": 100 * g_[g_.billing_mode != "Global"].overnight.mean()}
            BP_rows.append({"outcome": key, "province": p, "n_Global": int((g_.billing_mode == "Global").sum()),
                            "n_Other": int((g_.billing_mode != "Global").sum()), "ratio": np.exp(e),
                            "lo": np.exp(e - 1.96 * s), "hi": np.exp(e + 1.96 * s), **raw_})
            est.append(e); sel.append(s); used.append(p)
        est, sel = np.array(est), np.array(sel)
        r = drsl(est, sel)
        pool = {"outcome": key, "provinces": r["k"], "pooled_ratio_RE": np.exp(r["re"]),
                "pooled_lo": np.exp(r["re"] - 1.96 * r["se_re"]), "pooled_hi": np.exp(r["re"] + 1.96 * r["se_re"]),
                "pooled_ratio_fixed": np.exp(r["fixed"]), "I2_pct": r["I2_pct"], "tau_log": r["tau"],
                "Q_p": r["Q_p"], "pred_interval_lo": np.exp(r["pi_lo"]), "pred_interval_hi": np.exp(r["pi_hi"]),
                "provinces_ratio_below_1": int((est < 0).sum()), "provinces_ratio_above_1": int((est > 0).sum()),
                "min_province_ratio": np.exp(est.min()), "max_province_ratio": np.exp(est.max()),
                "same_direction_all_provinces": bool((est < 0).all() or (est > 0).all())}
        BP_pool.append(pool)
        print(f"  {key:<12} RE ratio {pool['pooled_ratio_RE']:.2f} ({pool['pooled_lo']:.2f}-{pool['pooled_hi']:.2f}); "
              f"I2 = {pool['I2_pct']:.0f}%; 95% PI {pool['pred_interval_lo']:.2f}-{pool['pred_interval_hi']:.2f}; "
              f"provinces <1 / >1: {pool['provinces_ratio_below_1']}/{pool['provinces_ratio_above_1']}")
    if REFS_SAVE is not None:
        REFS["province_model"] = REFS_SAVE
    BP = pd.DataFrame(BP_rows)
    BP["analysis_status"] = "exploratory (added after diagnostics)"
    BPP = pd.DataFrame(BP_pool)
    BPP["analysis_status"] = "exploratory (added after diagnostics)"
    save(BP, "o1_billing_by_province.csv")
    save(BPP, "o1_billing_pooled.csv")

# =============================================================================
# 7. MAIN TABLE (M5)
# =============================================================================
vc = D["province_model"].value_counts()
TOP_PROV = [p for p in vc.index if p != "Other"][:TOP_PROVINCES]
SELECT = [("billing_mode", None), ("group", None), ("specialty_model", None), ("province_model", TOP_PROV)]
LEVELS = {v: [REFS[v]] + [x for x in D[v].value_counts().index if x != REFS[v]] for v in REFS if v in D.columns}


def fmt(r, mark_small=True):
    s = f"{r.ratio:.2f} ({r.lo:.2f}-{r.hi:.2f})"
    s += "" if pd.isna(r.p) else (f"; p={r.p:.3f}" if r.p >= 0.001 else "; p<0.001")
    if r.sparse_flag:
        s += " \u2020"
    elif mark_small and pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N:
        s += " \u2021"
    return s


def lookup(tabs, key, model, term, cluster="patient"):
    m = tabs[(tabs.outcome == key) & (tabs.model == model) & (tabs.term == term) & (tabs.cluster == cluster)]
    return m.iloc[0] if len(m) else None


def main_table(model="M5"):
    rows = []
    for var, only in SELECT:
        levels = LEVELS[var] if only is None else [REFS[var]] + [x for x in only if x != REFS[var]]
        for lev in levels:
            row = {"variable": var, "level": lev, "episodes": int((D[var] == lev).sum())}
            for key in ["claim_lines", "overnight", "los", "los_qp"]:
                lab = f"{EFFECT_LABEL[key]} {key}"
                if lev == REFS[var]:
                    row[lab] = "1 (ref)"
                    row[lab + " | province-cluster CI"] = "1 (ref)"
                    continue
                term = f"C({var})[T.{lev}]"
                a = lookup(LONG, key, model, term)
                row[lab] = fmt(a) if a is not None else ""
                b = lookup(PC.assign(model=model), key, model, term, "province")
                row[lab + " | province-cluster CI"] = (
                    f"{b.ratio:.2f} ({b.lo:.2f}-{b.hi:.2f}); p={b.p:.3f}" if b is not None
                    else ("not defined" if var == "province_model" else ""))
            rows.append(row)
    return pd.DataFrame(rows)


MAIN = main_table()
save(MAIN, "o1_main_table.csv")
print("\n" + "=" * 78)
print("6. MAIN TABLE, model M5 (patient-clustered CI | province-clustered CI)")
print("=" * 78)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 38)
print(MAIN[["variable", "level", "episodes", "RR claim_lines", "OR overnight", "ratio los"]].to_string(index=False))
print("\u2020 sparse cell (< 30 episodes or < 10 in an outcome class); \u2021 fewer than "
      f"{MIN_LEVEL_N} episodes: not interpretable on its own.")
print("The billing-mode row is a pooled average across provinces whose effects differ; "
      "use section 5 (by province, random effects) for billing mode.")

ST_MAIN_ROWS = []
for var, only in [("group", None), ("any_surgical_service", None), ("ownership", None), ("specialty_model", None),
                  ("province_model", TOP_PROV)]:
    levels = LEVELS[var] if only is None else [REFS[var]] + [x for x in only if x != REFS[var]]
    for lev in levels:
        row = {"variable": var, "level": lev}
        for stratum in ["Global", "Other"]:
            sub = D[D.billing_mode == "Global"] if stratum == "Global" else D[D.billing_mode != "Global"]
            row[f"episodes_{stratum}"] = int((sub[var] == lev).sum())
            for key in ["claim_lines", "overnight", "los"]:
                lab = f"{EFFECT_LABEL[key]} {key} | {stratum}"
                if lev == REFS[var]:
                    row[lab] = "1 (ref)"
                    continue
                m = ST[(ST.outcome == key) & (ST.stratum == stratum) & (ST.term == f"C({var})[T.{lev}]") & (ST.cluster == "patient")]
                row[lab] = fmt(m.iloc[0]) if len(m) else "not estimable"
        ST_MAIN_ROWS.append(row)
STM = pd.DataFrame(ST_MAIN_ROWS)
save(STM, "o1_main_table_stratified.csv")
print("\nStratified by billing mode (group and surgical service):")
print(STM[STM.variable.isin(["group", "any_surgical_service"])][["variable", "level", "RR claim_lines | Global",
      "RR claim_lines | Other", "OR overnight | Global", "OR overnight | Other"]].to_string(index=False))

# =============================================================================
# 8. DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 78)
print("7. DIAGNOSTICS")
print("=" * 78)


def gvif_table(X, names):
    keep = [i for i, nm in enumerate(names) if nm != "Intercept" and X[:, i].std() > 0]
    term_of = {i: names[i].split("[T.")[0] for i in keep}
    R = np.corrcoef(X[:, keep], rowvar=False)
    sign, ld_all = np.linalg.slogdet(R)
    rows = []
    for term in dict.fromkeys(term_of.values()):
        idx = [j for j, i in enumerate(keep) if term_of[i] == term]
        rest = [j for j in range(len(keep)) if j not in set(idx)]
        s1, ld_t = np.linalg.slogdet(R[np.ix_(idx, idx)])
        s2, ld_r = np.linalg.slogdet(R[np.ix_(rest, rest)])
        g = np.exp(ld_t + ld_r - ld_all) if sign > 0 and s1 > 0 and s2 > 0 else np.nan
        rows.append({"term": term, "df": len(idx), "GVIF": g, "VIF_equivalent": g ** (1 / len(idx)) if np.isfinite(g) else np.nan})
    out = pd.DataFrame(rows)
    out["flag_gt5"] = out["VIF_equivalent"] > VIF_FLAG
    return out


c5 = CORE[("claim_lines", "M5")]
GV = gvif_table(c5["X"], c5["names"])
save(GV, "o1_diagnostics_gvif.csv")
print("GVIF (M5): VIF_equivalent = GVIF^(1/(2df)) flag > 5 ->", "none flagged" if not GV.flag_gt5.any()
      else GV[GV.flag_gt5].term.tolist())

shared = [t for t in LONG.term.unique() if "province_model" not in t and "specialty_model" not in t]
m3 = LONG[(LONG.model == "M3") & LONG.term.isin(shared)].set_index(["outcome", "term"])["ratio"]
m5 = LONG[(LONG.model == "M5") & LONG.term.isin(shared)].set_index(["outcome", "term"])["ratio"]
CH = pd.concat([m3.rename("ratio_M3"), m5.rename("ratio_M5")], axis=1).dropna()
CH["pct_change_M5_vs_M3"] = 100 * (CH["ratio_M5"] / CH["ratio_M3"] - 1)
CH = CH.reset_index()
save(CH, "o1_diagnostics_m5_vs_m3.csv")
big = CH[(CH.pct_change_M5_vs_M3.abs() > 10) & CH.term.str.contains("ownership|billing|group|surgical")]
print(f"M5 vs M3: {int((CH.pct_change_M5_vs_M3.abs() > 10).sum())} of {len(CH)} estimates move > 10%; among the "
      f"main exposures: {len(big)}")
if len(big):
    print(big.round(2).to_string(index=False))
print("Dispersion, M3/M5:")
print(pd.DataFrame(disp_rows).query("model in ['M3','M5']").round(2).to_string(index=False))

try:
    cf = CORE[("overnight", "M5")]
    bml, seml = ml_logit(cf["X"], cf["y"])
    FM = pd.DataFrame({"term": cf["names"], "beta_firth": cf["beta"], "se_sandwich": cf["se_sandwich"],
                       "se_model": cf["se_model"], "beta_ml": bml, "se_ml": seml})
    FM["se_ratio_model_over_sandwich"] = FM.se_model / FM.se_sandwich
    FM["flag_sandwich_too_narrow"] = (FM.se_ratio_model_over_sandwich > SE_NARROW_RATIO) & (FM.term != "Intercept")
    save(FM, "o1_diagnostics_firth_vs_ml.csv")
    print("Sandwich SE too narrow (separation) for:", FM[FM.flag_sandwich_too_narrow].term.tolist() or "none")
except Exception as e:
    print(f"Firth vs ML failed: {type(e).__name__}: {e}")

icc_rows = []
for key in OUTCOMES:
    c3 = CORE[(key, "M3")]
    prov = D.loc[c3["sub_index"], "province_model"].to_numpy()
    icc_rows.append({"outcome": key, "icc_province_after_M3": icc_anova(c3["resid"], prov),
                     "n": c3["n"], "provinces": int(pd.Series(prov).nunique())})
ICC = pd.DataFrame(icc_rows)
save(ICC, "o1_diagnostics_icc_province.csv")
print("Province ICC after M3:", {r.outcome: round(r.icc_province_after_M3, 3) for r in ICC.itertuples()})

M5L = LONG[(LONG.model == "M5") & LONG.outcome.isin(list(OUTCOMES))]
CMP = M5L[["outcome", "term", "ratio", "lo", "hi", "se_log"]].merge(
    PC[PC.outcome.isin(list(OUTCOMES))][["outcome", "term", "ratio", "lo", "hi", "se_log"]],
    on=["outcome", "term"], suffixes=("_patient", "_province"))
CMP["se_ratio_province_over_patient"] = CMP.se_log_province / CMP.se_log_patient
CMP["sig_patient"] = (CMP.lo_patient > 1) | (CMP.hi_patient < 1)
CMP["sig_province"] = (CMP.lo_province > 1) | (CMP.hi_province < 1)
save(CMP, "o1_diagnostics_se_by_cluster_level.csv")
mainx = CMP[CMP.term.str.contains("ownership|billing|group|surgical")]
print(f"Province-clustered SE is {mainx.se_ratio_province_over_patient.median():.1f}x the patient-clustered SE "
      f"(median, main exposures); significance lost with province clusters: "
      f"{int((mainx.sig_patient & ~mainx.sig_province).sum())} of {int(mainx.sig_patient.sum())} significant terms.")

# interaction (heterogeneity) tests
INT_ROWS = []
for a, b in [("billing_mode", "group"), ("billing_mode", "any_surgical_service"), ("group", "any_surgical_service"),
             ("billing_mode", "province_model")]:
    if args.sample == "all" and a == "billing_mode" and b == "province_model":
        continue
    for key in OUTCOMES:
        try:
            core = fit_core(D, key, BLOCKS["M5"], extra_inter=[(a, b)])
            idx = [i for i, nm in enumerate(core["names"]) if ":" in nm and f"C({a})" in nm and f"C({b})" in nm]
            bb, V = core["beta"][idx], core["cov"][np.ix_(idx, idx)]
            chi2 = float(bb @ np.linalg.pinv(V) @ bb)
            INT_ROWS.append({"interaction": f"{a} x {b}", "outcome": key, "chi2": chi2, "df": len(idx),
                             "p_value": stats.chi2.sf(chi2, len(idx))})
        except Exception as e:
            print(f"  interaction {a} x {b} / {key} FAILED: {type(e).__name__}: {e}")
INT = pd.DataFrame(INT_ROWS)
INT["significant_bonferroni"] = INT.p_value < 0.05 / max(len(INT), 1)
save(INT, "o1_diagnostics_interactions.csv")
print("Interaction (heterogeneity) tests, M5:")
print(INT.round(4).to_string(index=False))

# =============================================================================
# 9. SENSITIVITY ANALYSES 1-7 (M5, primary sample)
# =============================================================================
SENS = {1: ("Surgical service only", "keep_sens1_surgical", None),
        2: ("Exclude Global packages", "keep_sens2_itemised", None),
        3: ("Exclude K81", "keep_sens3_not_K81", None),
        4: ("Exclude flagged extremes", "keep_sens4_no_extremes", None),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous", None),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine", None),
        7: ("LOS capped at P95", None, "los_capped_p95")}
SL = pd.DataFrame()
if RUN_SENS:
    print("\n" + "=" * 78)
    print("8. SENSITIVITY ANALYSES 1-7 (M5, primary sample)")
    print("=" * 78)
    tabs = []
    for s, (name, keepcol, los_col) in SENS.items():
        print(f"\nSensitivity {s}: {name}")
        sub = D[D[keepcol]] if keepcol else D
        for key in OUTCOMES:
            if s == 7 and key != "los":
                continue
            tab, _, _ = fit_one(sub, key, "M5", BLOCKS["M5"], y_col=los_col if key == "los" else None)
            if tab is not None:
                tab["sens"], tab["sens_name"] = s, name
                tabs.append(tab)
    SL = pd.concat(tabs, ignore_index=True)
    SL["analysis_status"] = "pre-specified analyses; primary sample redefined after diagnostics"
    save(SL, "o1_sensitivity_long.csv")

# =============================================================================
# 10. EXPLORATORY: protocol sample, cut-offs, tiny cells, leave-one-province-out
# =============================================================================
EXP = []


def add_exp(code, name, d, keys=("claim_lines", "overnight", "los"), y_cols=None, core_only=True, label_n=True):
    print(f"\nExploratory {code}: {name} (n = {len(d):,})")
    for key in keys:
        tab, _, _ = fit_one(d, key, "M5", BLOCKS["M5"], y_col=(y_cols or {}).get(key))
        if tab is not None:
            if core_only:
                tab = tab[tab.term.str.startswith(tuple(f"C({v})[" for v in CORE_VARS))]
            tab["analysis"], tab["analysis_name"] = code, name
            EXP.append(tab)


if RUN_EXTRA:
    print("\n" + "=" * 78)
    print("9. EXPLORATORY ANALYSES (report as exploratory)")
    print("=" * 78)
    add_exp("X0", f"Other sample: {ALT_LABEL}", ALT, core_only=False)
    for c in OVERNIGHT_CUTOFFS:
        add_exp(f"X5_{c}", f"Overnight stay defined as LOS >= {c} d", D, keys=("overnight",),
                y_cols={"overnight": CUT_COL[c]})
    tiny = pd.Series(False, index=D.index)
    for v in ["sex", "insurance_fund_model", "adm_year"]:
        vcn = D[v].value_counts()
        tiny |= D[v].isin(vcn[vcn < SPARSE_N_MIN].index)
    add_exp("X6", f"Episodes in cells < {SPARSE_N_MIN} (sex, fund, year) removed ({int(tiny.sum())})", D[~tiny])
    for p in [p for p in D["province_model"].value_counts().index if p != "Other"][:LOPO_N]:
        add_exp(f"X7_{p}", f"Without {p}", D[D["province_model"] != p])
    # X8 (added in the gap-closing step): Stage C consistency flags (money inconsistencies, duplicate episodes,
    # patient conflicts) were created but never used; this refits M5 without the flagged episodes. EXPLORATORY.
    if "keep_xsens_consistent" in D.columns:
        _kx = to_bool(D["keep_xsens_consistent"])
        add_exp("X8", f"Exclude flagged money inconsistencies / duplicates / patient conflicts ({int((~_kx).sum())} removed)", D[_kx])
    EXPL = pd.concat(EXP, ignore_index=True)
    EXPL["analysis_status"] = "exploratory (added after diagnostics)"
    save(EXPL, "o1_exploratory_long.csv")
    core_tab = EXPL[EXPL.term.str.startswith(tuple(f"C({v})[" for v in CORE_VARS))].merge(
        LONG[LONG.model == "M5"][["outcome", "term", "ratio"]].rename(columns={"ratio": "primary_M5_ratio"}),
        on=["outcome", "term"], how="left")
    save(core_tab[["analysis", "analysis_name", "outcome", "term", "n", "ratio", "lo", "hi", "p", "primary_M5_ratio",
                   "sparse_flag", "ci_method"]], "o1_exploratory_core_terms.csv")
    x0 = EXPL[EXPL.analysis == "X0"].merge(LONG[LONG.model == "M5"][["outcome", "term", "ratio", "lo", "hi"]],
                                           on=["outcome", "term"], suffixes=("_other_sample", "_primary"))
    x0 = x0[x0.term.str.contains("billing|group|surgical|ownership|age_group")]
    x0["pct_difference"] = 100 * (x0.ratio_other_sample / x0.ratio_primary - 1)
    x0["other_sample"] = ALT_LABEL
    save(x0[["outcome", "term", "other_sample", "ratio_primary", "ratio_other_sample", "pct_difference"]],
         "o1_sample_comparison.csv")
    print("\nSample comparison (main exposures), primary vs", ALT_LABEL)
    print(x0[x0.term.str.contains("billing|surgical|group")][["outcome", "term", "ratio_primary", "ratio_other_sample",
          "pct_difference"]].round(2).to_string(index=False))

# =============================================================================
# 11. RELIABILITY SUMMARY
# =============================================================================
print("\n" + "=" * 78)
print("10. RELIABILITY SUMMARY OF MAIN-TABLE ESTIMATES")
print("=" * 78)
prefix = tuple(f"C({v})[" for v in REPORT_VARS)
rel_rows = []
for key in OUTCOMES:
    main = LONG[(LONG.model == "M5") & (LONG.outcome == key) & LONG.term.str.startswith(prefix)]
    for _, r in main.iterrows():
        sig_p = bool((r.lo > 1) or (r.hi < 1))
        pcr = PC[(PC.outcome == key) & (PC.term == r.term)]
        sig_pc = None if "province_model" in r.term else (bool((pcr.iloc[0].lo > 1) or (pcr.iloc[0].hi < 1)) if len(pcr) else None)
        s = SL[(SL.outcome == key) & (SL.term == r.term)] if len(SL) else pd.DataFrame()
        sens_n = len(s)
        sens_dir = int(((s.ratio > 1) == (r.ratio > 1)).sum()) if sens_n else 0
        sens_sig = int((((s.ratio > 1) == (r.ratio > 1)) & ((s.lo > 1) | (s.hi < 1))).sum()) if sens_n else 0
        st = ST[(ST.outcome == key) & (ST.term == r.term) & (ST.cluster == "patient")]
        rg = st[st.stratum == "Global"].ratio
        ro = st[st.stratum == "Other"].ratio
        if "billing_mode" in r.term:
            strata_dir, spread = None, np.nan
        elif len(rg) and len(ro):
            strata_dir = bool(((rg.iloc[0] > 1) == (r.ratio > 1)) and ((ro.iloc[0] > 1) == (r.ratio > 1)))
            spread = float(max(rg.iloc[0], ro.iloc[0]) / min(rg.iloc[0], ro.iloc[0]))
        else:
            strata_dir, spread = None, np.nan
        why = []
        if "billing_mode" in r.term:
            bpp = BPP[BPP.outcome == key] if args.sample == "admissions" else pd.DataFrame()
            if len(bpp):
                b = bpp.iloc[0]
                re_sig = (b.pooled_lo > 1) or (b.pooled_hi < 1)
                if not b.same_direction_all_provinces:
                    why.append(f"direction differs across provinces ({int(b.provinces_ratio_below_1)} below 1, "
                               f"{int(b.provinces_ratio_above_1)} above 1)")
                if not re_sig:
                    why.append("random-effects pooled CI includes 1")
                verdict = "B" if (b.same_direction_all_provinces and re_sig) else "C"
                if verdict == "B":
                    why.append("same direction in every province; size varies (see I2 / prediction interval)")
            else:
                verdict = "C"; why.append("billing-by-province analysis not run")
        else:
            if not sig_p:
                why.append("CI includes 1 (patient clusters)")
            if sig_pc is False:
                why.append("CI includes 1 with province clusters")
            if sens_n and sens_dir < sens_n:
                why.append(f"direction fails in {sens_n - sens_dir} of {sens_n} sensitivity analyses")
            if strata_dir is False:
                why.append("direction differs between Global-only and itemised-only models")
            if strata_dir is None:
                why.append("not estimable in one billing stratum")
            if r.sparse_flag:
                why.append("sparse cell")
            if pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N:
                why.append(f"fewer than {MIN_LEVEL_N} episodes")
            hard = [w for w in why if not w.startswith("not estimable")]
            ok = (not hard) and strata_dir is not False
            if "province_model" in r.term:
                ok = sig_p and (not r.sparse_flag) and (sens_n == 0 or sens_dir == sens_n) and strata_dir is not False \
                    and not (pd.notna(r.level_n) and r.level_n < MIN_LEVEL_N)
                why.append("province coefficient: cluster-robust SE not defined at province level; "
                           "treat as descriptive of recording/case-mix differences")
            if not ok:
                verdict = "C"
            elif pd.notna(spread) and spread > SPREAD_FLAG:
                verdict = "B"; why.append(f"size differs between billing strata (ratio of ratios {spread:.1f})")
            else:
                verdict = "A"
        rel_rows.append({"outcome": key, "term": r.term, "level_n": r.level_n, "ratio": round(r.ratio, 3),
                         "CI_patient": f"{r.lo:.2f}-{r.hi:.2f}",
                         "CI_province": (f"{pcr.iloc[0].lo:.2f}-{pcr.iloc[0].hi:.2f}" if len(pcr) else ""),
                         "sens_tested": sens_n, "sens_same_direction": sens_dir, "sens_same_direction_and_sig": sens_sig,
                         "ratio_Global_only": round(rg.iloc[0], 3) if len(rg) else np.nan,
                         "ratio_Other_only": round(ro.iloc[0], 3) if len(ro) else np.nan,
                         "strata_spread": round(spread, 2) if pd.notna(spread) else np.nan,
                         "sparse_flag": bool(r.sparse_flag), "verdict": verdict, "why": "; ".join(why)})
REL = pd.DataFrame(rel_rows)
save(REL, "o1_reliability_summary.csv")
save(REL[REL.term.str.contains("ownership|billing|group|surgical")] if len(REL) else REL, "o1_reliability_main_exposures.csv")
print(REL.groupby(["outcome", "verdict"]).size().unstack(fill_value=0))
print("\nMain exposures:")
print(REL[REL.term.str.contains("billing|group|surgical|ownership")][["outcome", "term", "ratio", "CI_patient", "CI_province",
      "verdict", "why"]].to_string(index=False, max_colwidth=70))

# =============================================================================
# 12. RUN INFO
# =============================================================================
info = {"script_version": "3", "input_file": str(INPUT_CSV), "input_sha256": sha256(INPUT_CSV),
        "rows_in_file": int(len(DALL)), "primary_sample": SAMPLE_LABEL, "primary_sample_n": int(len(D)),
        "reference_levels": {k: REFS[k] for k in BLOCKS["M5"] + TIME}, "top_provinces_in_table": TOP_PROV,
        "settings": {"sparse_n_min": SPARSE_N_MIN, "sparse_cell_min": SPARSE_CELL_MIN, "se_narrow_ratio": SE_NARROW_RATIO,
                     "min_level_n": MIN_LEVEL_N, "spread_flag": SPREAD_FLAG, "prov_min_per_mode": PROV_MIN_PER_MODE,
                     "profile_run": RUN_PROFILE, "overnight_cutoffs_exploratory": OVERNIGHT_CUTOFFS, "lopo_provinces": LOPO_N},
        "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                     "scipy": scipy.__version__},
        "notes": ["Primary sample redefined to admissions (Global + Inpatient) after the diagnostics: a deviation from the frozen protocol",
                  "Sensitivity 8 (province random intercept) not run here; province-clustered SEs and the by-province "
                  "random-effects analysis are Python stand-ins",
                  "Firth sandwich SE replaced by profile likelihood CI (M5) or the larger model SE where the model SE "
                  "is > 1.5x the sandwich SE",
                  "Billing x province, stratified, X0-X7, interaction and reliability analyses were added after the "
                  "diagnostic run and must be labelled exploratory"]}
with open(OUT_DIR / "o1_run_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2)
print("\n" + "=" * 78)
print(f"DONE. Outputs in {OUT_DIR}")
print("Sensitivity 8 (province as random intercept) is not run here; run it in R (lme4::glmer / glmmTMB) on the same file.")
print("=" * 78)

1. LOAD AND VERIFY
Rows: 15,601; columns: 70
discharge_status: 9 categories, 4 collapsed to 'Other'.
                                                                   step     n
                                   Episodes in the frozen analysis file 15601
                                            of which record type Global  8462
                              of which record type Inpatient (itemised)  6223
          of which record type Emergency Observation (not an admission)   889
  of which record type Temporary Inpatient Admission (not an admission)    27
                        PRIMARY SAMPLE: admissions (Global + Inpatient) 14685

Outcome checks (primary sample)
  claim_lines: median 4, IQR 2-10, min 1, max 23, missing 0
  overnight stay: 72.6% (missing 0)
  LOS among overnight stays: n = 10,660, median 1.61 d
Reference levels: {'age_group': '40-59', 'sex': 'Male', 'insurance_fund_model': 'Rural Health Insurance Fund', 'ownership': 'Public University-Affiliated', 'billing_mode

In [35]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 2 (expenditure) of the analysis protocol  --  VERSION 2 (corrected after diagnostics on the real file)
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional).

WHAT CHANGED FROM v1 (each change was forced by a diagnostic on digestive_surgical_analysis_file.csv)
  1  PRIMARY MODEL IS STRATIFIED BY BILLING MODE (Global package vs itemised "Other"), fitted as ONE fully
     interacted block-diagonal Gamma-log GLM (so patient/province-clustered covariances and Global-vs-Other
     contrasts are valid). v1 forced one set of covariate effects and one constant Global/Other ratio on two
     cost-generating processes that differ enormously (dispersion 0.05 vs 0.24, different ownership/surgical/
     province effects). The additive pooled model is still fitted (M1-M5, protocol comparable) but is labelled
     "pooled additive, known to be mis-specified" and is not used for conclusions.
  2  TIME: the nominal tariff moves in steps (about +45-50% in the first two months of 1402) and drifts inside
     1401. Separate year and month dummies cannot represent this; v2 uses year-month (period) dummies.
     The 10 episodes of 1400-12 are merged into 1401-01; any further period with < RARE_MIN episodes in a
     stratum is merged into its chronological neighbour.
  3  Categories with < RARE_MIN (30) episodes WITHIN A STRATUM are collapsed (to "Other", or to the modal level
     if the Other bucket would still be < 30) so no coefficient rests on a handful of episodes.
  4  POSITIVITY: any_surgical_service is not estimable in the Global stratum (only ~31 non-surgical Global
     episodes) and is dropped there; it stays in the itemised stratum.
  5  AVERAGE MARGINAL COSTS come from the stratified model (g-computation, delta-method CIs with patient AND
     province clusters) and are compared with the observed cell means; the Global-vs-itemised contrast is
     the standardised effect on the Global episodes (ATT): what the Global episodes would have cost if billed
     itemised, with the share of episodes on common support reported.
  6  Diagnostics (Park, link, Cook + refit, quasi-Poisson check) are run WITHIN each stratum; five-fold
     patient-grouped out-of-fold validation compares pooled vs stratified models by stratum.
  7  H1 compares variation after removing tariff steps (ICD-10 x period, and x province cells).
     H2 is tested within each stratum, H3 is shown crude / adjusted / province FE with a province-level
     cluster bootstrap CI (the province coefficient has no valid province-clustered SE otherwise).
  8  Sensitivity analyses 1-8 are re-run on the stratified specification; the A/B/C reliability verdict is
     computed from them (sparse level, direction, size of change, Cook refit, quasi-Poisson, province
     cluster, province random intercept).

PRIMARY SAMPLE  Admissions (record types Global + Inpatient), as in Objective 1 v3. Protocol sample (all valid
          episodes) is fitted as exploratory X0. This and the billing-mode stratification are deviations from
          the frozen protocol: state them in the Methods and label the analyses post hoc.
OUTCOME   cost_intl (Int$ = IRR/118,411.24, nominal, not inflation-adjusted).

USAGE
  python objective2_expenditure.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 200]   (AMC and H3 bootstraps use at most 100 replicates)
Outputs: folder objective2_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import gammaln

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "v2-stratified"
SEED = 42
RARE_MIN = 30                       # minimum episodes per category level within a stratum
SPARSE_N_MIN = 30
TOP_PROVINCES = 5
COOK_MULT = 4.0                     # Cook's distance threshold = COOK_MULT / n
N_FOLDS = 5
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
STABLE_PCT = 15.0                   # max % change across sensitivity analyses for an "A" estimate
COOK_PCT = 10.0                     # max % change in the Cook refit / quasi-Poisson for an "A" estimate

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=200)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service",
           "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
NUMS = {"M1": [], "M2": [], "M3": ["los_cov"], "M4": ["los_cov"], "M5": ["los_cov"]}   # LOS enters in M3
COLLAPSE_VARS = ["ownership", "specialty_model", "province_model", "insurance_fund_model",
                 "discharge_status_model", "sex", "age_group", "period"]
KEY_RE = r"C\(group\)|any_surgical_service|C\(ownership\)|los_cov"
Z975 = stats.norm.ppf(0.975)


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


T0 = time.time()


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE THE PRIMARY SAMPLE
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "cost_intl", "total_cost_irr", "los_days", "los_capped_p99",
        "los_capped_p95", "cluster_id", "any_surgical_service", "specialty_model", "specialty_h3",
        "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O2",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O2", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "specialty_h3", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["cost_intl", "total_cost_irr", "los_days", "los_capped_p99", "los_capped_p95"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
if (DALL["cost_intl"] <= 0).any() or DALL["cost_intl"].isna().any():
    fail("cost_intl has missing or non-positive values; the frozen file should not.")
DALL["los_cov"] = DALL["los_capped_p99"]
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"          # 10 episodes; see header
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
valid = DALL["in_O2"] & DALL["los_cov"].notna()
flow.append({"step": "O2 complete cases (cost and LOS present), all record types", "n": int(valid.sum())})
D = (DALL[is_adm & valid] if args.sample == "admissions" else DALL[valid]).copy().reset_index(drop=True)
ALT = (DALL[valid] if args.sample == "admissions" else DALL[is_adm & valid]).copy().reset_index(drop=True)
SAMPLE_LABEL = "admissions (Global + Inpatient)" if args.sample == "admissions" else "all valid episodes (protocol sample)"
ALT_LABEL = "all valid episodes (protocol sample)" if args.sample == "admissions" else "admissions only"
flow.append({"step": f"PRIMARY SAMPLE O2: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o2_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
print(f"\nCost (Int$): median {D.cost_intl.median():,.0f}, IQR {D.cost_intl.quantile(.25):,.0f}-"
      f"{D.cost_intl.quantile(.75):,.0f}, mean {D.cost_intl.mean():,.0f}, max {D.cost_intl.max():,.0f}")
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary sample.")


# =============================================================================
# 2. MODEL ENGINE: block-diagonal (stratified) or pooled Gamma-log / quasi-Poisson-log GLM
# =============================================================================
def build_spec(d, cats, nums, refs=None):
    refs = refs if refs is not None else {}
    cols, names, levs = [np.ones((len(d), 1))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        cols.append(_dummies(s, levs[v]))
        names += [f"C({v})[T.{lev}]" for lev in levs[v][1:]]
    for v in nums:
        cols.append(d[v].to_numpy(float)[:, None])
        names.append(v)
    X = np.hstack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], {"cats": list(cats), "nums": list(nums), "levs": levs, "keep": keep}


def _dummies(series, levels):
    """One-hot (non-reference levels) in a single vectorised pass."""
    codes = pd.Categorical(series.astype(str), categories=levels).codes
    return (codes[:, None] == np.arange(1, len(levels))[None, :]).astype(float)


def apply_spec(d, spec):
    cols = [np.ones((len(d), 1))]
    for v in spec["cats"]:
        cols.append(_dummies(d[v], spec["levs"][v]))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float)[:, None])
    return np.hstack(cols)[:, spec["keep"]]


def merge_periods(vc, minn):
    groups = [[p, [p], int(vc[p])] for p in sorted(vc.index)]          # [label, members, count]
    while len(groups) > 1 and min(g[2] for g in groups) < minn:
        i = int(np.argmin([g[2] for g in groups]))
        j = i + 1 if i + 1 < len(groups) else i - 1
        groups[j][1] += groups[i][1]
        groups[j][2] += groups[i][2]
        groups.pop(i)
    return {m: g[0] for g in groups for m in g[1]}


def make_cmap(ds, vars_, minn=RARE_MIN):
    """Collapse levels with < minn episodes inside this stratum."""
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        if v == "period":
            m = merge_periods(vc, minn)
        else:
            rare = [l for l in vc.index if vc[l] < minn]
            m = {}
            if rare:
                total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
                tgt = "Other" if total >= minn else vc.index[0]
                m = {l: tgt for l in rare}
                if tgt == "Other" and "Other" in vc.index:
                    pass
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def apply_cmap(d, cm):
    d = d.copy()
    for v, m in cm["map"].items():
        s0 = d[v].astype(str)
        s = s0.map(m).fillna(s0) if m else s0
        seen = cm["seen"][v]
        d[v] = s.where(s.isin(seen), "Other" if "Other" in seen else "__unseen__")
    return d


def fit_glm(X, y, fam="gamma"):
    """Log-link GLM by IRLS with step-halving on the deviance (plain IRLS diverges on the itemised stratum).
    Gamma: V = mu^2 (working weight 1). quasi-Poisson ('qp'): V = mu."""
    n, k = X.shape

    def dev(mu):
        if fam == "gamma":
            return 2 * np.sum(-np.log(y / mu) + (y - mu) / mu)
        return 2 * np.sum(y * np.log(y / mu) - (y - mu))

    beta = np.zeros(k)
    beta[0] = np.log(y.mean())
    eta = X @ beta
    mu = np.exp(eta)
    d_old = dev(mu)
    conv = False
    for _ in range(300):
        w = np.ones(n) if fam == "gamma" else mu
        z = eta + (y - mu) / mu
        A, rhs = (X.T * w) @ X, (X.T * w) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        step = 1.0
        for _h in range(30):
            cand = beta + step * (b_new - beta)
            eta_c = np.clip(X @ cand, -30, 30)
            mu_c = np.exp(eta_c)
            d_new = dev(mu_c)
            if np.isfinite(d_new) and d_new <= d_old + 1e-10 * abs(d_old):
                break
            step /= 2
        done = np.abs(cand - beta).max() < 1e-9 or abs(d_old - d_new) < 1e-12 * abs(d_old)
        beta, eta, mu, d_old = cand, eta_c, mu_c, d_new
        if done:
            conv = True
            break
    w = np.ones(n) if fam == "gamma" else mu
    V = mu ** 2 if fam == "gamma" else mu
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * w) @ X), "conv": conv,
            "phi": float(np.sum((y - mu) ** 2 / V) / max(n - k, 1)), "V": V}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups, fam="gamma"):
    n, k = X.shape
    s = (y - f["mu"]) / f["mu"] if fam == "gamma" else (y - f["mu"])
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_block(d, cats, nums, mode="stratified", fam="gamma", cov=True, minn=RARE_MIN):
    """mode 'stratified': one separate model per billing-mode stratum, fitted jointly as a block-diagonal GLM
    (any_surgical_service is dropped in the Global stratum). mode 'pooled': one additive model, billing_mode
    as a covariate (v1 specification, kept for comparison only)."""
    d = d.reset_index(drop=True)
    if mode == "pooled":
        parts = {"All": np.ones(len(d), bool)}
    else:
        parts = {s: (d["stratum"] == s).to_numpy() for s in ["Global", "Other"] if (d["stratum"] == s).sum() >= 50}
    blocks, Xs, names, off = {}, [], [], 0
    for s, rows in parts.items():
        ds = d.loc[rows]
        cs = [c for c in cats if not (mode == "stratified" and c in ("any_surgical_service", "billing_mode") and
                                      (s == "Global" or c == "billing_mode"))]
        cs = list(dict.fromkeys(cs + ["period"]))
        cm = make_cmap(ds, [v for v in cs if v in COLLAPSE_VARS], minn)
        dm = apply_cmap(ds, cm)
        refs = {"period": min(dm["period"].unique())}
        X_s, nm_s, spec = build_spec(dm, cs, nums, refs)
        blocks[s] = {"rows": np.where(rows)[0], "cols": slice(off, off + X_s.shape[1]), "names": nm_s,
                     "spec": spec, "cmap": cm, "X": X_s,
                     "lev_n": [int(X_s[:, j].sum()) if nm.startswith("C(") else np.nan for j, nm in enumerate(nm_s)]}
        off += X_s.shape[1]
        Xs.append((rows, X_s))
    K = off
    y = d["cost_intl"].to_numpy(float)
    # each stratum is fitted on its own (identical to the block-diagonal joint fit, ~5x cheaper)
    beta, eta, mu, V = np.zeros(K), np.zeros(len(d)), np.zeros(len(d)), np.zeros(len(d))
    bread, conv = np.zeros((K, K)), True
    for b in blocks.values():
        r = b["rows"]
        fb = fit_glm(b["X"], y[r], fam)
        beta[b["cols"]], eta[r], mu[r], V[r] = fb["beta"], fb["eta"], fb["mu"], fb["V"]
        bread[b["cols"], b["cols"]] = fb["bread"] if cov else 0
        conv &= fb["conv"]
    f = {"beta": beta, "eta": eta, "mu": mu, "V": V, "bread": bread, "conv": conv}
    X = None
    if cov:   # dense joint design only needed for the cluster-robust covariance (clusters cross strata)
        X = np.zeros((len(d), K))
        for (rows, X_s), b in zip(Xs, blocks.values()):
            X[np.ix_(np.where(rows)[0], np.arange(b["cols"].start, b["cols"].stop))] = X_s
    pre = {s: s for s in blocks}
    allnames = [(f"{s}|{nm}" if mode == "stratified" else nm) for s, b in blocks.items() for nm in b["names"]]
    lev_n = [x for b in blocks.values() for x in b["lev_n"]]
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl], fam) for cl in groups} if cov else {}
    phi = {}
    for s, b in blocks.items():
        r = b["rows"]
        phi[s] = float(np.sum((y[r] - f["mu"][r]) ** 2 / f["V"][r]) / max(len(r) - b["X"].shape[1], 1))
    return {"mode": mode, "fam": fam, "n": len(d), "names": allnames, "lev_n": lev_n, "beta": f["beta"],
            "covs": covs, "X": X, "y": y, "mu": f["mu"], "eta": f["eta"], "phi": phi, "conv": f["conv"],
            "blocks": blocks, "d": d, "bread": f["bread"], "V": f["V"], "K": K, "groups": groups}


def make_tab(core, model, label="cost"):
    out = []
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = core["beta"] / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term_full": core["names"], "n": core["n"],
                          "level_n": core["lev_n"], "ratio": np.exp(core["beta"]),
                          "lo": np.exp(core["beta"] - crit * se), "hi": np.exp(core["beta"] + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        if core["mode"] == "stratified":
            t["stratum"] = t.term_full.str.split("|", n=1).str[0]
            t["term"] = t.term_full.str.split("|", n=1).str[1]
        else:
            t["stratum"], t["term"] = "All", t["term_full"]
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[(t["term"] != "Intercept") & ~t["term"].str.startswith("C(period)")])
    return pd.concat(out, ignore_index=True)


def hit(r):
    return bool((r.lo > 1) or (r.hi < 1))


def crit_p(core, cl, est, se):
    G = core["covs"][cl][1]
    if cl == "province":
        c = stats.t.ppf(0.975, G - 1)
        return c, 2 * stats.t.sf(abs(est / se), G - 1) if se > 0 else np.nan
    return Z975, 2 * stats.norm.sf(abs(est / se)) if se > 0 else np.nan


# --- prediction, g-computation margins with gradients ------------------------
def pred_part(core, s, dpop, over=None):
    b = core["blocks"][s]
    dm = apply_cmap(dpop, b["cmap"])
    for k, v in (over or {}).items():
        dm[k] = v
    X = apply_spec(dm, b["spec"])
    return np.exp(np.clip(X @ core["beta"][b["cols"]], -30, 30)), X


def margin(core, s, dpop, over=None):
    mu, X = pred_part(core, s, dpop, over)
    g = np.zeros(core["K"])
    g[core["blocks"][s]["cols"]] = (mu @ X) / len(mu)
    return float(mu.mean()), g


def predict_rows(core, dte):
    mu = np.full(len(dte), np.nan)
    for s, b in core["blocks"].items():
        m = (dte["stratum"] == s).to_numpy() if core["mode"] == "stratified" else np.ones(len(dte), bool)
        if m.any():
            mu[m] = pred_part(core, s, dte[m])[0]
    return mu


def est_ci(core, est, grad, as_ratio=True):
    rows = {}
    for cl in core["covs"]:
        cov = core["covs"][cl][0]
        se = float(np.sqrt(max(grad @ cov @ grad, 0)))
        c, p = crit_p(core, cl, est, se)
        rows[cl] = (se, c, p)
    return rows


def support_share(core, s, dpop):
    b = core["blocks"][s]
    ok = np.ones(len(dpop), bool)
    for v in COLLAPSE_VARS:
        if v in b["cmap"]["map"] and v not in ("period", "sex", "age_group"):
            m = b["cmap"]["map"][v]
            ok &= ~dpop[v].astype(str).isin(m.keys()).to_numpy()
    return float(ok.mean())


# =============================================================================
# 3. DESCRIPTIVES
# =============================================================================
hdr("2. DESCRIPTIVES (cost, Int$)")


def cstats(s):
    s = s.dropna()
    m = s.mean()
    return {"n": len(s), "mean": m, "sd": s.std(), "median": s.median(), "p25": s.quantile(.25),
            "p75": s.quantile(.75), "p5": s.quantile(.05), "p95": s.quantile(.95), "p99": s.quantile(.99),
            "max": s.max(), "cv": s.std() / m if m > 0 else np.nan,
            "median_IQR": f"{s.median():,.0f} ({s.quantile(.25):,.0f}-{s.quantile(.75):,.0f})"}


def describe(d, var, label):
    rows = []
    for lev, g in ([("All", d)] if var is None else d.groupby(var, observed=True)):
        rows.append({"variable": label, "level": lev, **cstats(g["cost_intl"])})
    return rows


desc = describe(DALL[valid], None, "All valid episodes (protocol sample)")
desc += describe(DALL[valid], "record_type", "Record type (all valid episodes)")
desc += describe(D, None, "PRIMARY SAMPLE")
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("billing_mode", "Billing mode"),
                 ("specialty_model", "Specialty"), ("province_model", "Province"), ("ownership", "Ownership"),
                 ("period", "Period (Jalali year-month)")]:
    desc += describe(D, var, lab)
D["grp_x_bill"] = D["group"] + " | " + D["billing_mode"]
desc += describe(D, "grp_x_bill", "Group x billing mode")
DESC = pd.DataFrame(desc)
save(DESC, "o2_descriptives.csv")
print(DESC[DESC.variable.isin(["All valid episodes (protocol sample)", "Record type (all valid episodes)",
                               "PRIMARY SAMPLE", "Group", "Billing mode", "Group x billing mode"])]
      [["variable", "level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
print("\nBy ICD-10 code:\n", DESC[DESC.variable == "ICD-10 code"][["level", "n", "median_IQR", "mean", "cv"]].round(2).to_string(index=False))
print("\nBy period:\n", DESC[DESC.variable == "Period (Jalali year-month)"][["level", "n", "median_IQR", "mean"]].round(1).to_string(index=False))


# =============================================================================
# 4. HYPOTHESIS 1: Global vs itemised cost variation within the same code (tariff steps removed)
# =============================================================================
hdr("3. HYPOTHESIS 1: Global vs itemised cost variation, same ICD-10 code")


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj, run = np.empty_like(p), 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i])
        adj[i] = min(run, 1.0)
    return adj


def tariff_stats(x):
    med = x.median()
    return {"n": len(x), "median": med, "mean": x.mean(), "sd": x.std(), "cv": x.std() / x.mean(),
            "sd_log": np.log(x).std(),
            "qcd": (x.quantile(.75) - x.quantile(.25)) / (x.quantile(.75) + x.quantile(.25)),
            "pct_within_10pct_of_median": 100 * ((x - med).abs() <= 0.10 * med).mean(),
            "pct_within_25pct_of_median": 100 * ((x - med).abs() <= 0.25 * med).mean()}


H1_cells, H1_cmp = [], []
for code, g in D.groupby("icd10"):
    gg, oo = g.loc[g.stratum == "Global", "cost_intl"], g.loc[g.stratum == "Other", "cost_intl"]
    for lab, x in [("Global", gg), ("Other", oo)]:
        if len(x) >= 5:
            H1_cells.append({"icd10": code, "billing_mode": lab, **tariff_stats(x)})
    if len(gg) < 10 or len(oo) < 10:
        H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo), "note": "too few in a mode"})
        continue
    lg, lo_ = np.log(gg.to_numpy()), np.log(oo.to_numpy())
    bf = stats.levene(lg, lo_, center="median")
    bs = [np.log(RNG.choice(gg.to_numpy(), len(gg))).std() / np.log(RNG.choice(oo.to_numpy(), len(oo))).std()
          for _ in range(NBOOT)]
    H1_cmp.append({"icd10": code, "n_Global": len(gg), "n_Other": len(oo),
                   "sd_log_ratio_G_over_O": lg.std() / lo_.std(), "sd_log_lo": np.percentile(bs, 2.5),
                   "sd_log_hi": np.percentile(bs, 97.5), "cv_ratio_G_over_O": (gg.std() / gg.mean()) / (oo.std() / oo.mean()),
                   "median_ratio_G_over_O": gg.median() / oo.median(), "brown_forsythe_p": float(bf.pvalue), "note": ""})
H1c = pd.DataFrame(H1_cmp)
ok = H1c["brown_forsythe_p"].notna()
H1c.loc[ok, "p_holm"] = holm(H1c.loc[ok, "brown_forsythe_p"])
H1c["Global_less_variable"] = (H1c["sd_log_ratio_G_over_O"] < 1) & (H1c["p_holm"] < 0.05)
save(pd.DataFrame(H1_cells), "o2_h1_cost_by_code_and_billing.csv")
save(H1c, "o2_h1_variation_by_code.csv")
print(pd.DataFrame(H1_cells).round(2).to_string(index=False))
print("\nPer code (ratio of SD(log cost), Global/itemised; < 1 = Global less variable). The itemised cost also moves "
      "with the nominal tariff, so see the adjusted comparison below:")
print(H1c.round(3).to_string(index=False))


def pooled_within(lc, cell):
    n = np.bincount(cell)
    m = np.bincount(cell, lc) / np.maximum(n, 1)
    ss = float(np.sum((lc - m[cell]) ** 2))
    df = len(lc) - np.count_nonzero(n)
    tot = float(np.sum((lc - lc.mean()) ** 2))
    return np.sqrt(ss / max(df, 1)), 1 - ss / tot


h1a = []
for lab, keys in [("ICD-10 code", ["icd10"]), ("ICD-10 x period", ["icd10", "period"]),
                  ("ICD-10 x period x province", ["icd10", "period", "province_model"])]:
    cellid = D[keys].astype(str).agg("|".join, axis=1)
    cnt = pd.crosstab(cellid, D.stratum)
    good = cnt.index[(cnt["Global"] >= 5) & (cnt["Other"] >= 5)]
    sub = D[cellid.isin(good)]
    if len(sub) < 100:
        continue
    cid = pd.factorize(cellid[cellid.isin(good)])[0]
    lc = np.log(sub.cost_intl.to_numpy())
    res = {}
    for s in ["Global", "Other"]:
        m = (sub.stratum == s).to_numpy()
        res[s] = pooled_within(lc[m], pd.factorize(cid[m])[0])
    bs = []
    for _ in range(NBOOT):
        r = {}
        for s in ["Global", "Other"]:
            idx = np.where((sub.stratum == s).to_numpy())[0]
            idx = RNG.choice(idx, len(idx))
            r[s] = pooled_within(lc[idx], pd.factorize(cid[idx])[0])[0]
        bs.append(r["Global"] / r["Other"])
    sub_s = sub.assign(_c=cellid[cellid.isin(good)])
    pct = {}
    for s in ["Global", "Other"]:
        x = sub_s[sub_s.stratum == s]
        med = x.groupby("_c").cost_intl.transform("median")
        pct[s] = (100 * ((x.cost_intl - med).abs() <= .10 * med).mean(), 100 * ((x.cost_intl - med).abs() <= .25 * med).mean())
    h1a.append({"cells": lab, "n_cells": len(good), "n_Global": int((sub.stratum == "Global").sum()),
                "n_Other": int((sub.stratum == "Other").sum()),
                "within_cell_SD_log_Global": res["Global"][0], "within_cell_SD_log_Other": res["Other"][0],
                "SD_ratio_G_over_O": res["Global"][0] / res["Other"][0],
                "ratio_lo": np.percentile(bs, 2.5), "ratio_hi": np.percentile(bs, 97.5),
                "R2_cells_Global": res["Global"][1], "R2_cells_Other": res["Other"][1],
                "pct_within10_Global": pct["Global"][0], "pct_within10_Other": pct["Other"][0],
                "pct_within25_Global": pct["Global"][1], "pct_within25_Other": pct["Other"][1]})
H1A = pd.DataFrame(h1a)
H1A["supports_H1"] = H1A["ratio_hi"] < 1
save(H1A, "o2_h1_adjusted_variation.csv")
print("\nPooled within-cell variation (cells with >= 5 episodes in BOTH modes; bootstrap CI):")
print(H1A.round(3).to_string(index=False))
print("Reading: after removing code and period (tariff steps), the Global package varies less than itemised billing "
      "if the SD ratio and its CI are below 1. The R2 columns show how much of the Global cost variance is a tariff cell.")


# =============================================================================
# 5. MODELS M1-M5 : pooled additive (comparison) and stratified (primary)
# =============================================================================
hdr("4. MODELS M1-M5 (Gamma-log; patient and province clusters)")
CATS_ALL = BLOCKS["M5"] + ["billing_mode"]


def blk_cats(m, pooled):
    c = list(BLOCKS[m])
    return c + ["billing_mode"] if pooled and m != "M1" else c


CORE_P, CORE_S, TABS = {}, {}, []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cp = fit_block(D, blk_cats(m, True), NUMS[m], "pooled")
    cs = fit_block(D, blk_cats(m, False), NUMS[m], "stratified")
    CORE_P[m], CORE_S[m] = cp, cs
    TABS += [make_tab(cp, m).assign(spec="pooled additive (comparison only)"),
             make_tab(cs, m).assign(spec="stratified by billing mode (PRIMARY)")]
    print(f"  {m}: pooled terms {len(cp['names']) - 1}, dispersion {cp['phi']['All']:.3f} | stratified terms "
          f"{len(cs['names']) - len(cs['blocks'])}, dispersion " + ", ".join(f"{s} {v:.3f}" for s, v in cs["phi"].items()))
LONG = pd.concat(TABS, ignore_index=True)
save(LONG, "o2_models_long.csv")
LONG_S = LONG[LONG.spec.str.contains("PRIMARY")]
M5S, M5P = CORE_S["M5"], CORE_P["M5"]
prov_n = D.province_model.value_counts()
top_prov = prov_n.index[:TOP_PROVINCES].tolist()


def fmt(r):
    star = " *" if hit(r) else ""
    sp = " [n<30]" if r.sparse_flag else ""
    return f"{r.ratio:.2f} ({r.lo:.2f}-{r.hi:.2f}){star}{sp}"


def main_table(core, model):
    t = make_tab(core, model)
    rows = []
    for s in ["Global", "Other"]:
        for _, r in t[(t.stratum == s) & (t.cluster == "patient")].iterrows():
            if not (r.term.startswith(("C(group)", "C(any_surgical", "C(ownership", "C(specialty_model")) or
                    r.term == "los_cov" or any(r.term.endswith(f"[T.{p}]") and "province" in r.term for p in top_prov)):
                continue
            q = t[(t.stratum == s) & (t.cluster == "province") & (t.term == r.term)].iloc[0]
            rows.append({"stratum": s, "term": r.term, "level_n": r.level_n, "ratio": r.ratio, "lo_patient": r.lo,
                         "hi_patient": r.hi, "p_patient": r.p, "lo_province": q.lo, "hi_province": q.hi,
                         "p_province": q.p, "patient_CI": fmt(r), "province_CI": fmt(q),
                         "sparse_flag": r.sparse_flag})
    return pd.DataFrame(rows)


MAIN = main_table(M5S, "M5")
save(MAIN, "o2_main_table_M5_stratified.csv")
print("\nMAIN TABLE (stratified M5): cost ratio (95% CI) with patient clusters and with province clusters (t, G-1 df).\n"
      "* = CI excludes 1. Province-cluster CIs are the more honest ones for facility-level terms (ownership, specialty,\n"
      "province); province terms themselves have no valid province-clustered SE.")
print(MAIN[["stratum", "term", "level_n", "patient_CI", "province_CI"]].to_string(index=False))

# pooled additive billing-mode ratio (v1 type), for comparison with the stratified standardised ratio
pb = LONG[(LONG.spec.str.contains("pooled")) & (LONG.cluster == "patient") & (LONG.term.str.contains("billing_mode"))]
print("\nPooled ADDITIVE billing-mode ratio (Global vs itemised) by block [v1-type estimate; assumes one constant ratio]:")
print(pb[["model", "term", "ratio", "lo", "hi"]].round(3).to_string(index=False))
save(pb, "o2_pooled_billing_ratio_by_block.csv")


# =============================================================================
# 6. AVERAGE MARGINAL COSTS + STANDARDISED GLOBAL-vs-ITEMISED CONTRAST
# =============================================================================
hdr("5. AVERAGE MARGINAL COSTS (g-computation from the stratified model)")


def amc_tables(core, d, model):
    own, att = [], []
    groups = ["All"] + sorted(d.group.unique())
    for s in ["Global", "Other"]:
        pop_s = d[d.stratum == s]
        for g in groups:
            over = None if g == "All" else {"group": g}
            pop = pop_s if g == "All" else pop_s[pop_s.group == g]
            if len(pop) < 20:
                continue
            m, gr = margin(core, s, pop)                                  # fitted mean in the observed group
            lm, lg = np.log(m), gr / m
            ci = est_ci(core, lm, lg)
            row = {"model": model, "stratum": s, "group": g, "n": len(pop), "observed_mean": pop.cost_intl.mean(),
                   "observed_median": pop.cost_intl.median(), "AMC": m, "obs_over_AMC": pop.cost_intl.mean() / m}
            for cl, (se, c, p) in ci.items():
                row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(lm - c * se)), float(np.exp(lm + c * se))
            own.append(row)
        # group counterfactual (everyone in the stratum set to group g): standardised AMC
        if True:
            for g in groups[1:]:
                m, gr = margin(core, s, pop_s, {"group": g})
                ci = est_ci(core, np.log(m), gr / m)
                row = {"model": model, "stratum": s, "group": f"standardised: all {s} episodes as {g}", "n": len(pop_s),
                       "observed_mean": np.nan, "observed_median": np.nan, "AMC": m, "obs_over_AMC": np.nan}
                for cl, (se, c, p) in ci.items():
                    row[f"lo_{cl}"], row[f"hi_{cl}"] = float(np.exp(np.log(m) - c * se)), float(np.exp(np.log(m) + c * se))
                own.append(row)
    popG = d[d.stratum == "Global"]
    for g in groups:
        pop = popG if g == "All" else popG[popG.group == g]
        if len(pop) < 20:
            continue
        m1, g1 = margin(core, "Global", pop)
        m2, g2 = margin(core, "Other", pop)
        lr, glr = np.log(m1 / m2), g1 / m1 - g2 / m2
        cir = est_ci(core, lr, glr)
        cid = est_ci(core, m1 - m2, g1 - g2)
        row = {"model": model, "population": f"Global episodes ({g})", "n_Global": len(pop),
               "share_on_common_support": support_share(core, "Other", pop), "observed_Global_mean": pop.cost_intl.mean(),
               "fitted_Global_mean": m1, "if_itemised_mean": m2, "ratio_Global_over_itemised": m1 / m2,
               "diff_Int$": m1 - m2}
        for cl, (se, c, p) in cir.items():
            row[f"ratio_lo_{cl}"], row[f"ratio_hi_{cl}"], row[f"ratio_p_{cl}"] = \
                float(np.exp(lr - c * se)), float(np.exp(lr + c * se)), p
        for cl, (se, c, p) in cid.items():
            row[f"diff_lo_{cl}"], row[f"diff_hi_{cl}"] = (m1 - m2) - c * se, (m1 - m2) + c * se
        att.append(row)
    return pd.DataFrame(own), pd.DataFrame(att)


# The marginal costs (means) come from the quasi-Poisson-log fit: its score equations make the fitted mean equal the
# observed mean in every cell of the categorical covariates, so it is the mean-preserving estimator of expenditure.
# The Gamma-log fit (used for the cost RATIOS, as in the protocol) over-weights low-cost episodes in the heavy-tailed
# itemised stratum and misses the observed itemised Gallbladder mean by ~13%; its AMCs are saved for comparison.
M5Q = fit_block(D, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp")
AMC, ATT = amc_tables(M5Q, D, "M5 quasi-Poisson-log (primary for means)")
AMC_G, ATT_G = amc_tables(M5S, D, "M5 Gamma-log (comparison)")


def boot_ci(d, B, key):
    """Cluster bootstrap (whole refit) for the AMCs and the Global/itemised ratio. The delta-method CI above is
    conditional on the observed covariate mix, so it is too narrow for population means (it is ~0 for the stratum
    mean under the quasi-Poisson fit); the bootstrap includes that sampling variation."""
    cl = d[key].astype(str).to_numpy()
    uc, inv = np.unique(cl, return_inverse=True)
    idx_by = [np.where(inv == i)[0] for i in range(len(uc))]
    ra, rt = [], []
    for _ in range(B):
        pick = RNG.integers(0, len(uc), len(uc))
        db = d.iloc[np.concatenate([idx_by[i] for i in pick])].reset_index(drop=True)
        try:
            cb = fit_block(db, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp", cov=False)
            a_, t_ = amc_tables(cb, db, "b")
        except Exception:
            continue
        ra.append(a_.set_index(["stratum", "group"]).AMC)
        rt.append(t_.set_index("population")[["ratio_Global_over_itemised", "diff_Int$"]])
    ra, rt = pd.concat(ra, axis=1), pd.concat(rt, axis=1)
    return ra, rt


NB = min(NBOOT, 100)
for key, tag in [("cluster_id", "patient"), ("province_model", "province")]:
    ra, rt = boot_ci(D, NB, key)
    q = ra.quantile([.025, .975], axis=1).T
    AMC[f"lo_boot_{tag}"] = [q.loc[(r.stratum, r.group), 0.025] if (r.stratum, r.group) in q.index else np.nan for r in AMC.itertuples()]
    AMC[f"hi_boot_{tag}"] = [q.loc[(r.stratum, r.group), 0.975] if (r.stratum, r.group) in q.index else np.nan for r in AMC.itertuples()]
    ratio = pd.concat([rt.iloc[:, 0::2]], axis=1) if False else rt
    rr = rt["ratio_Global_over_itemised"]
    dd = rt["diff_Int$"]
    qr, qd = rr.quantile([.025, .975], axis=1).T, dd.quantile([.025, .975], axis=1).T
    ATT[f"ratio_lo_boot_{tag}"] = [qr.loc[p_, 0.025] for p_ in ATT.population]
    ATT[f"ratio_hi_boot_{tag}"] = [qr.loc[p_, 0.975] for p_ in ATT.population]
    ATT[f"diff_lo_boot_{tag}"] = [qd.loc[p_, 0.025] for p_ in ATT.population]
    ATT[f"diff_hi_boot_{tag}"] = [qd.loc[p_, 0.975] for p_ in ATT.population]
# the bootstrap CIs replace the delta-method CIs as the reported ones (delta columns are kept, suffix _delta)
for tag in ["patient", "province"]:
    AMC = AMC.rename(columns={f"lo_{tag}": f"lo_{tag}_delta", f"hi_{tag}": f"hi_{tag}_delta"})
    ATT = ATT.rename(columns={f"ratio_lo_{tag}": f"ratio_lo_{tag}_delta", f"ratio_hi_{tag}": f"ratio_hi_{tag}_delta"})
    AMC[f"lo_{tag}"], AMC[f"hi_{tag}"] = AMC[f"lo_boot_{tag}"], AMC[f"hi_boot_{tag}"]
    ATT[f"ratio_lo_{tag}"], ATT[f"ratio_hi_{tag}"] = ATT[f"ratio_lo_boot_{tag}"], ATT[f"ratio_hi_boot_{tag}"]

save(AMC, "o2_amc_stratified_M5.csv")
save(ATT, "o2_global_vs_itemised_standardised_M5.csv")
save(AMC_G, "o2_amc_stratified_M5_gamma.csv")
save(ATT_G, "o2_global_vs_itemised_standardised_M5_gamma.csv")
print("Average marginal cost (Int$) by stratum and group, with the observed mean (obs_over_AMC should be ~ 1):")
print(AMC[AMC.n.notna()][["stratum", "group", "n", "observed_mean", "AMC", "lo_patient", "hi_patient", "lo_province",
                          "hi_province", "obs_over_AMC"]].round(2).to_string(index=False))
print("\nStandardised Global-vs-itemised contrast on the Global episodes (what they would have cost if billed itemised):")
print(ATT[["population", "n_Global", "share_on_common_support", "observed_Global_mean", "if_itemised_mean",
           "ratio_Global_over_itemised", "ratio_lo_patient", "ratio_hi_patient", "ratio_lo_province", "ratio_hi_province"]]
      .round(3).to_string(index=False))
ATT_BY_BLOCK = []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    a = amc_tables(fit_block(D, blk_cats(m, False), NUMS[m], "stratified", fam="qp"), D, m)[1]
    ATT_BY_BLOCK.append(a[a.population == "Global episodes (All)"])
ATT_BY_BLOCK = pd.concat(ATT_BY_BLOCK)
save(ATT_BY_BLOCK, "o2_global_vs_itemised_by_block.csv")
print("\nGlobal/itemised ratio across blocks (stratified quasi-Poisson models):")
print(ATT_BY_BLOCK[["model", "fitted_Global_mean", "if_itemised_mean", "ratio_Global_over_itemised",
                    "ratio_lo_patient", "ratio_hi_patient"]].round(3).to_string(index=False))

# check of the v1 additive AMC against observed cells
chk = []
for s in ["Global", "Other"]:
    for g in sorted(D.group.unique()):
        pop = D[(D.stratum == s) & (D.group == g)]
        mu_p = pred_part(M5P, "All", pop)[0].mean()
        a = AMC_G[(AMC.stratum == s) & (AMC.group == g)]
        chk.append({"stratum": s, "group": g, "n": len(pop), "observed_mean": pop.cost_intl.mean(),
                    "pooled_additive_fitted": mu_p, "stratified_gamma_fitted": float(a.AMC.iloc[0]),
                    "stratified_qp_fitted": float(AMC[(AMC.stratum == s) & (AMC.group == g)].AMC.iloc[0]),
                    "pooled_error_pct": 100 * (mu_p / pop.cost_intl.mean() - 1),
                    "stratified_gamma_error_pct": 100 * (float(a.AMC.iloc[0]) / pop.cost_intl.mean() - 1),
                    "stratified_qp_error_pct": 100 * (float(AMC[(AMC.stratum == s) & (AMC.group == g)].AMC.iloc[0]) / pop.cost_intl.mean() - 1)})
CHK = pd.DataFrame(chk)
save(CHK, "o2_amc_fit_to_observed_pooled_vs_stratified.csv")
print("\nFitted vs observed cell means (quasi-Poisson reproduces them by construction, Gamma-log and the additive model do not):")
print(CHK.round(2).to_string(index=False))


# =============================================================================
# 7. DIAGNOSTICS, WITHIN EACH STRATUM
# =============================================================================
hdr("6. DIAGNOSTICS (Park, link, Cook's distance + refit, quasi-Poisson), within strata")


def ols_cluster(x, y, groups):
    Xo = np.column_stack([np.ones(len(x)), x])
    b = np.linalg.solve(Xo.T @ Xo, Xo.T @ y)
    e = y - Xo @ b
    meat, g = cluster_meat(Xo * e[:, None], groups)
    bread = np.linalg.inv(Xo.T @ Xo)
    V = (g / (g - 1)) * bread @ meat @ bread
    return b, np.sqrt(np.diag(V)), g


diag, cook_tabs = [], []
cook_flag_all = np.zeros(len(M5S["d"]), bool)
for s, b in M5S["blocks"].items():
    r = b["rows"]
    Xs, ys, mu, eta = b["X"], M5S["y"][r], M5S["mu"][r], M5S["eta"][r]
    pat = M5S["groups"]["patient"][r]
    e = ys - mu
    ok = e != 0
    (a0, a1), (s0, s1), g = ols_cluster(np.log(mu[ok]), np.log(e[ok] ** 2), pat[ok])
    # link test
    Xl = np.column_stack([Xs, eta ** 2 - 0])
    fl = fit_glm(Xl, ys)
    covl, _ = cov_robust(Xl, ys, fl, pat)
    tl = fl["beta"][-1] / np.sqrt(covl[-1, -1])
    # Cook's distance (Gamma, working weight 1)
    brd = np.linalg.pinv(Xs.T @ Xs)
    h = np.einsum("ij,jk,ik->i", Xs, brd, Xs)
    rp = (ys - mu) / mu
    k = Xs.shape[1]
    cook = rp ** 2 * h / (k * M5S["phi"][s] * (1 - h) ** 2)
    flag = cook > COOK_MULT / len(r)
    cook_flag_all[r[flag]] = True
    diag.append({"stratum": s, "n": len(r), "terms": k, "dispersion": M5S["phi"][s],
                 "park_slope": a1, "park_se": s1, "park_reading": ("Gamma (slope ~2)" if abs(a1 - 2) < 2 * s1 else
                                                                  ("closer to inverse Gaussian (3)" if a1 > 2 else "closer to Poisson-like (1)")),
                 "link_t": float(tl), "link_p": float(2 * stats.norm.sf(abs(tl))), "max_cook": float(cook.max()),
                 "n_cook_flag": int(flag.sum()), "max_leverage": float(h.max()), "n_leverage_gt_0.2": int((h > .2).sum())})
DIAG = pd.DataFrame(diag)
save(DIAG, "o2_diagnostics_by_stratum.csv")
print(DIAG.round(3).to_string(index=False))
print("Park slope within each stratum guides the variance function (2 = Gamma). Robust (sandwich) SEs are used throughout,\n"
      "so a moderate departure widens nothing; the quasi-Poisson refit below shows whether it matters for the estimates.")

dref = M5S["d"][~cook_flag_all]
M5C = fit_block(dref, BLOCKS["M5"], NUMS["M5"], "stratified")
tm, tc, tq = make_tab(M5S, "M5"), make_tab(M5C, "M5 cook-excl"), make_tab(M5Q, "M5 quasi-Poisson")
tm = tm[tm.cluster == "patient"].set_index("term_full")
tc, tq = tc[tc.cluster == "patient"].set_index("term_full"), tq[tq.cluster == "patient"].set_index("term_full")
ROB = tm[["stratum", "term", "level_n", "ratio", "lo", "hi"]].copy()
ROB["ratio_cook_excl"] = tc["ratio"].reindex(ROB.index)
ROB["ratio_quasiPoisson"] = tq["ratio"].reindex(ROB.index)
ROB["cook_pct_change"] = 100 * (ROB.ratio_cook_excl / ROB.ratio - 1)
ROB["qp_pct_change"] = 100 * (ROB.ratio_quasiPoisson / ROB.ratio - 1)
ROB = ROB.reset_index(drop=True)
save(ROB, "o2_robustness_cook_quasipoisson.csv")
print(f"\nCook refit excludes {int(cook_flag_all.sum())} episodes (> {COOK_MULT:g}/n within stratum). Key terms:")
kt = ROB[ROB.term.str.contains(KEY_RE)]
print(kt.round(3).to_string(index=False))
# AMC/ATT effect of the influential episodes
attC = amc_tables(M5C, M5C["d"], "M5 cook-excl")[1]
attQ = ATT_G
print("\nGlobal/itemised ratio: quasi-Poisson (primary)", ATT.ratio_Global_over_itemised.iloc[0].round(3), "| Gamma-log",
      ATT_G.ratio_Global_over_itemised.iloc[0].round(3), "| Gamma, Cook-excluded", attC.ratio_Global_over_itemised.iloc[0].round(3))


# =============================================================================
# 8. FIVE-FOLD OUT-OF-FOLD PREDICTION ERROR
# =============================================================================
hdr("7. OUT-OF-FOLD PREDICTION (5 folds, grouped by patient)")
ids = pd.factorize(D.cluster_id)[0]
uid = np.unique(ids)
RNG2 = np.random.default_rng(SEED)
RNG2.shuffle(uid)
fm = dict(zip(uid, np.arange(len(uid)) % N_FOLDS))
FOLD = np.array([fm[i] for i in ids])
oof_rows, cal_rows = [], []
for m in ["M1", "M3", "M5"]:
    for mode in ["pooled", "stratified", "stratified_qp"]:
        pred = np.full(len(D), np.nan)
        for k in range(N_FOLDS):
            tr, te = D[FOLD != k], D[FOLD == k]
            core = fit_block(tr, blk_cats(m, mode == "pooled"), NUMS[m], mode.replace("_qp", ""), fam="qp" if mode.endswith("qp") else "gamma", cov=False)
            p = predict_rows(core, te)
            lo_, hi_ = tr.cost_intl.min(), tr.cost_intl.max()
            pred[(FOLD == k)] = np.clip(p, lo_, hi_)
        y = D.cost_intl.to_numpy()
        for s in ["All", "Global", "Other"]:
            mk = np.ones(len(D), bool) if s == "All" else (D.stratum == s).to_numpy()
            err = y[mk] - pred[mk]
            oof_rows.append({"model": m, "spec": mode, "stratum": s, "n": int(mk.sum()), "MAE": np.abs(err).mean(),
                             "RMSE": np.sqrt((err ** 2).mean()), "MAE_log": np.abs(np.log(y[mk]) - np.log(pred[mk])).mean(),
                             "mean_observed": y[mk].mean(), "mean_predicted": pred[mk].mean(),
                             "calibration_ratio": pred[mk].mean() / y[mk].mean(),
                             "relative_MAE_pct": 100 * np.abs(err).mean() / y[mk].mean()})
            if m == "M5" and s != "All" or (m == "M5" and s == "All"):
                q = pd.qcut(pd.Series(pred[mk]).rank(method="first"), 10, labels=False)
                c = pd.DataFrame({"decile": q + 1, "pred": pred[mk], "obs": y[mk]}).groupby("decile").mean().reset_index()
                c["model"], c["spec"], c["stratum"] = m, mode, s
                cal_rows.append(c.rename(columns={"pred": "mean_predicted", "obs": "mean_observed"}))
OOF = pd.DataFrame(oof_rows)
CAL = pd.concat(cal_rows, ignore_index=True)
save(OOF, "o2_oof_error.csv")
save(CAL, "o2_oof_calibration_deciles.csv")
print(OOF.round(3).to_string(index=False))
c5 = CAL[(CAL.model == "M5") & (CAL.stratum == "All")].copy()
c5["ratio"] = c5.mean_predicted / c5.mean_observed
print("\nM5 calibration by predicted decile, all admissions (ratio = predicted/observed):")
print(c5.pivot(index="decile", columns="spec", values="ratio").round(3).to_string())


# =============================================================================
# 9. CONCENTRATION
# =============================================================================
hdr("8. CONCENTRATION OF EXPENDITURE")


def conc_stats(x):
    x = np.sort(np.asarray(x, float))
    n, tot = len(x), x.sum()
    cum = np.concatenate([[0], np.cumsum(x) / tot])
    p = np.linspace(0, 1, n + 1)
    gini = 1 - 2 * float(np.sum((cum[1:] + cum[:-1]) / 2 * np.diff(p)))
    out = {"n": n, "total_intl": tot, "gini": gini}
    for q in (1, 5, 10, 20):
        out[f"top{q}pct_share"] = 100 * x[-max(int(np.ceil(q / 100 * n)), 1):].sum() / tot
    return out


def conc_boot(x, B):
    x = np.asarray(x, float)
    r = [conc_stats(RNG.choice(x, len(x))) for _ in range(B)]
    return (np.percentile([v["top10pct_share"] for v in r], [2.5, 97.5]), np.percentile([v["gini"] for v in r], [2.5, 97.5]))


crow = []
masks = [("All (primary sample)", pd.Series(True, index=D.index))] + \
        [(f"group={g}", D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())] + \
        [(f"{g} x {b}", (D.group == g) & (D.billing_mode == b)) for g in sorted(D.group.unique()) for b in sorted(D.billing_mode.unique())] + \
        [("All, flagged extremes removed", ~D.flag_extreme_any)]
for lab, mask in masks:
    x = D.loc[mask, "cost_intl"]
    if len(x) < 20:
        continue
    (t_lo, t_hi), (g_lo, g_hi) = conc_boot(x, NBOOT)
    crow.append({"stratum": lab, **conc_stats(x), "top10_lo": t_lo, "top10_hi": t_hi, "gini_lo": g_lo, "gini_hi": g_hi})
CONC = pd.DataFrame(crow)
save(CONC, "o2_concentration.csv")
print(CONC.round(2).to_string(index=False))
lz = []
for lab, mask in [("All", pd.Series(True, index=D.index))] + [(g, D.group == g) for g in sorted(D.group.unique())] + \
        [(f"billing={b}", D.billing_mode == b) for b in sorted(D.billing_mode.unique())]:
    xs = np.sort(D.loc[mask, "cost_intl"].to_numpy())
    cum = np.concatenate([[0], np.cumsum(xs) / xs.sum()])
    pp = np.linspace(0, 1, len(xs) + 1)
    for q in np.linspace(0, 1, 101):
        lz.append({"stratum": lab, "cum_share_episodes": q, "cum_share_cost": float(np.interp(q, pp, cum))})
LZ = pd.DataFrame(lz)
save(LZ, "o2_lorenz_points.csv")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for lab in ["All"] + sorted(D.group.unique()):
        s = LZ[LZ.stratum == lab]
        ax[0].plot(s.cum_share_episodes, s.cum_share_cost, label=lab, lw=2 if lab == "All" else 1.4)
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
    ax[0].set(title="Lorenz curve of episode cost by group", xlabel="Cumulative share of episodes",
              ylabel="Cumulative share of expenditure")
    ax[0].legend(frameon=False)
    for k, spec in enumerate(["pooled", "stratified_qp"]):
        for s, col in [("Global", "C0"), ("Other", "C3")]:
            c8 = CAL[(CAL.model == "M5") & (CAL.stratum == s) & (CAL.spec == spec)]
            ax[1 + k].plot(c8.mean_predicted, c8.mean_observed, "o-", color=col, label=s)
    for a, t in zip(ax[1:], ["M5 pooled additive: OOF calibration", "M5 stratified quasi-Poisson: OOF calibration"]):
        lim = [0, max(CAL.mean_predicted.max(), CAL.mean_observed.max()) * 1.05]
        a.plot(lim, lim, "k--", lw=.8)
        a.set(title=t, xlabel="Mean predicted (Int$)", ylabel="Mean observed (Int$)")
        a.legend(frameon=False, title="billing mode")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o2_lorenz_and_calibration.png", dpi=200)
    plt.close(fig)
    print("Figure saved: o2_lorenz_and_calibration.png")
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")


# =============================================================================
# 10. HYPOTHESES 2 AND 3
# =============================================================================
hdr("9. HYPOTHESIS 2 (provinces) AND HYPOTHESIS 3 (K40: general surgery vs urology)")


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


h2 = []
for m in ["M3", "M4", "M5"]:
    core = CORE_S[m]
    for s, b in core["blocks"].items():
        idx = [b["cols"].start + j for j, nm in enumerate(b["names"]) if nm.startswith("C(province_model)")]
        if m == "M3":
            kw = stats.kruskal(*[g.cost_intl.values for _, g in D[D.stratum == s].groupby("province_model") if len(g) >= 5])
            h2.append({"model": "crude (Kruskal-Wallis)", "stratum": s, "df": np.nan, "chi2": float(kw.statistic),
                       "p": float(kw.pvalue), "note": "unadjusted"})
        if idx and m != "M3":
            chi2, df, p = wald(core, idx)
            prov = make_tab(core, m)
            prov = prov[(prov.stratum == s) & (prov.cluster == "patient") & prov.term.str.startswith("C(province_model)")]
            prov = prov[~prov.sparse_flag]
            h2.append({"model": m, "stratum": s, "df": df, "chi2": chi2, "p": p,
                       "n_provinces_modelled": df + 1, "min_ratio": prov.ratio.min(), "max_ratio": prov.ratio.max(),
                       "note": "Wald, patient-clustered; province-level SE not valid (28 clusters)"})
H2 = pd.DataFrame(h2)
save(H2, "o2_h2_province_tests.csv")
print(H2.round(4).to_string(index=False))
prov_main = make_tab(M5S, "M5")
prov_main = prov_main[prov_main.term.str.startswith("C(province_model)") & (prov_main.cluster == "patient")]
save(prov_main.drop(columns=["term_full"]), "o2_h2_province_ratios_M5.csv")
for s in ["Global", "Other"]:
    pm = prov_main[(prov_main.stratum == s) & ~prov_main.sparse_flag].sort_values("ratio")
    print(f"\n{s}: adjusted province cost ratios (vs the largest province in that stratum), extremes:")
    print(pm[["term", "level_n", "ratio", "lo", "hi"]].iloc[[0, 1, 2, -3, -2, -1]].round(2).to_string(index=False))

SIG = []
for s in ["Global", "Other"]:
    pass  # random-intercept variance components are computed with sensitivity 8 below


def h3_variant(d, which, cl_boot=False):
    cats = ["age_group", "sex", "insurance_fund_model", "ownership", "discharge_status_model", "any_surgical_service"]
    if which == "crude":
        cats = []
    if which == "adjusted_provFE":
        cats = cats + ["province_model"]
    d = d.copy()
    d["spec_h3"] = d["specialty_h3"].where(d["specialty_h3"].isin(["General surgery", "Urology"]), "x")
    d = d[d.spec_h3 != "x"]
    return d, cats + ["spec_h3"], ["los_cov"] if which != "crude" else []


h3 = []
K40 = D[D.icd10 == "K40"].copy()
for s in ["Other", "Global", "Both (pooled, +billing mode)"]:
    sub = K40 if s.startswith("Both") else K40[K40.stratum == s]
    for which in ["crude", "adjusted", "adjusted_provFE"]:
        d3, c3, n3 = h3_variant(sub, which)
        nu = (d3.specialty_h3 == "Urology").sum()
        if nu < 30 or (d3.specialty_h3 == "General surgery").sum() < 30:
            h3.append({"stratum": s, "variant": which, "n": len(d3), "n_urology": int(nu), "note": "too few urology episodes"})
            continue
        if s.startswith("Both"):
            c3 = c3 + ["billing_mode"]
        core = fit_block(d3.assign(stratum="Other"), c3, n3, "pooled")
        tt = make_tab(core, "K40")
        tt = tt[tt.term.str.contains("spec_h3")]
        rp = tt[tt.cluster == "patient"].iloc[0]
        rq = tt[tt.cluster == "province"].iloc[0]
        # province-level cluster bootstrap CI
        provs = d3.province_model.unique()
        bs = []
        grp = {p: d3[d3.province_model == p] for p in provs}
        for _ in range(min(NBOOT, 100)):
            pick = RNG.choice(provs, len(provs))
            db = pd.concat([grp[p] for p in pick], ignore_index=True)
            if db.spec_h3.nunique() < 2:
                continue
            try:
                cb = fit_block(db.assign(stratum="Other"), c3, n3, "pooled", cov=False)
                nmb = [i for i, nm in enumerate(cb["names"]) if "spec_h3" in nm]
                if nmb:
                    bs.append(np.exp(cb["beta"][nmb[0]]))
            except Exception:
                pass
        h3.append({"stratum": s, "variant": which, "n": len(d3), "n_urology": int(nu), "urology_vs_general_ratio": rp.ratio,
                   "lo_patient": rp.lo, "hi_patient": rp.hi, "p_patient": rp.p,
                   "lo_province_t": rq.lo, "hi_province_t": rq.hi, "p_province_t": rq.p,
                   "lo_province_bootstrap": np.percentile(bs, 2.5) if len(bs) > 20 else np.nan,
                   "hi_province_bootstrap": np.percentile(bs, 97.5) if len(bs) > 20 else np.nan, "note": ""})
H3 = pd.DataFrame(h3)
save(H3, "o2_h3_k40_specialty.csv")
print("\nH3 (K40, urology vs general surgery). Urology is concentrated in itemised billing, so the itemised stratum is "
      "the primary comparison; the specialty term is a facility-level exposure, so the province CIs are the honest ones:")
print(H3.round(3).to_string(index=False))


# =============================================================================
# 11. SENSITIVITY ANALYSES 1-8 (stratified M5)
# =============================================================================
hdr("10. SENSITIVITY ANALYSES 1-8 (stratified M5)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"),
        2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"),
        4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS covariate capped at P95", None)}
SL, SATT = pd.DataFrame(), pd.DataFrame()


def key_rows(core, s_id, name):
    t = make_tab(core, "M5")
    t = t[(t.cluster == "patient") & t.term.str.contains(KEY_RE)].copy()
    t["sens"], t["sens_name"] = s_id, name
    return t


if RUN_SENS:
    stabs, satt = [], []
    for s_id, (name, keepcol) in SENS.items():
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        if s_id == 7:
            sub["los_cov"] = sub["los_capped_p95"]
        core = fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified")
        stabs.append(key_rows(core, s_id, name))
        if "Global" in core["blocks"] and "Other" in core["blocks"]:
            coreq = fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp")
            a = amc_tables(coreq, sub.reset_index(drop=True), "M5")[1].assign(sens=s_id, sens_name=name)
            satt.append(a)
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}, strata = {list(core['blocks'])}")
    SL = pd.concat(stabs, ignore_index=True)
    SATT = pd.concat(satt, ignore_index=True) if satt else pd.DataFrame()


class LaplaceGamma:
    """Gamma-log GLMM with a province random intercept, Laplace-approximated ML (as lme4::glmer nAGQ = 1)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def _l(self, eta, nu):
        e = np.exp(-eta)
        l = nu * np.log(nu) - gammaln(nu) + (nu - 1) * np.log(self.y) - nu * self.y * e - nu * eta
        return l, nu * (self.y * e - 1), -nu * self.y * e

    def negll(self, theta):
        beta, sig, nu = theta[:self.k], np.exp(theta[self.k]), np.exp(theta[self.k + 1])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            l, d1, d2 = self._l(xb + u[self.g], nu)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        l, d1, d2 = self._l(xb + u[self.g], nu)
        h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
        return -(l.sum() - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0, nu0):
        th0 = np.concatenate([beta0, [np.log(0.3)], [np.log(nu0)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()
if RUN_SENS8:
    hdr("10b. SENSITIVITY 8: province as a random intercept (Gamma-log, Laplace ML), within each stratum")
    s8, vc8 = [], []
    for s, b in M5S["blocks"].items():
        t0 = time.time()
        try:
            ds = D[D.stratum == s].reset_index(drop=True)
            cats = [v for v in BLOCKS["M5"] if v != "province_model" and not (s == "Global" and v == "any_surgical_service")]
            cats = list(dict.fromkeys(cats + ["period"]))
            cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
            dm = apply_cmap(ds, cm)
            X8, nm8, _ = build_spec(dm, cats, ["los_cov"], {"period": min(dm["period"].unique())})
            y8 = ds.cost_intl.to_numpy(float)
            g8 = pd.factorize(ds.province_model)[0]
            f0 = fit_glm(X8, y8)
            m8 = LaplaceGamma(X8, y8, g8)
            theta = m8.fit(f0["beta"], 1 / max(f0["phi"], 1e-3))
            H = m8.hessian(theta)
            for _ in range(2):
                gr = optimize.approx_fprime(theta, m8.negll, 1e-6)
                try:
                    cand = theta - np.linalg.solve(H, gr)
                except np.linalg.LinAlgError:
                    break
                if m8.negll(cand) <= m8.negll(theta) + 1e-9:
                    theta = cand
                    H = m8.hessian(theta)
            se8 = np.sqrt(np.clip(np.diag(np.linalg.pinv(H)), 0, None))
            k8 = m8.k
            t8 = pd.DataFrame({"outcome": "cost", "model": "M5 random intercept", "stratum": s, "term": nm8[1:], "n": len(ds),
                               "level_n": [int(X8[:, j].sum()) if nm8[j].startswith("C(") else np.nan for j in range(1, k8)],
                               "ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                               "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]), "sens": 8,
                               "sens_name": "Province random intercept"})
            s8.append(t8)
            sig = float(np.exp(theta[k8]))
            vc8.append({"stratum": s, "n": len(ds), "provinces": m8.G, "sigma_u": sig,
                        "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])),
                        "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                        "province_cost_ratio_1sd": float(np.exp(sig)),
                        "gamma_dispersion": float(1 / np.exp(theta[k8 + 1]))})
            print(f"  {s}: n = {len(ds):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (a province 1 SD above the mean costs "
                  f"x{np.exp(sig):.2f}), {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  Sensitivity 8 ({s}) FAILED ({type(e).__name__}: {e})")
    if s8:
        S8, VC8 = pd.concat(s8, ignore_index=True), pd.DataFrame(vc8)
        save(S8, "o2_sens8_random_intercept_long.csv")
        save(VC8, "o2_sens8_variance_components.csv")
        print(VC8.round(3).to_string(index=False))
        SL = pd.concat([SL, S8[S8.term.str.contains(KEY_RE)]], ignore_index=True)
if len(SL):
    SL["analysis_status"] = "pre-specified sensitivity analyses, re-run on the post-hoc stratified specification"
    save(SL, "o2_sensitivity_long.csv")
if len(SATT):
    save(SATT, "o2_sensitivity_global_vs_itemised.csv")


# =============================================================================
# 12. EXPLORATORY
# =============================================================================
hdr("11. EXPLORATORY (label as such)")
if RUN_EXTRA and len(ALT):
    try:
        core0 = fit_block(ALT, BLOCKS["M5"], NUMS["M5"], "stratified")
        x0 = make_tab(core0, "X0 " + ALT_LABEL)
        save(x0.drop(columns=["term_full"]), "o2_X0_other_sample_M5.csv")
        k0 = x0[(x0.cluster == "patient") & x0.term.str.contains(KEY_RE)]
        print(f"X0: stratified M5 on {ALT_LABEL} (n = {len(ALT):,}); key terms:")
        print(k0[["stratum", "term", "ratio", "lo", "hi"]].round(3).to_string(index=False))
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    # X8 (added in the gap-closing step): Stage C consistency flags were created but never used. EXPLORATORY.
    try:
        _kx = to_bool(D["keep_xsens_consistent"])
        subx = D[_kx]
        core8 = fit_block(subx, BLOCKS["M5"], NUMS["M5"], "stratified")
        x8 = make_tab(core8, f"X8 consistent records only ({int((~_kx).sum())} flagged episodes removed)")
        save(x8.drop(columns=["term_full"]), "o2_X8_consistent_records_M5.csv")
        k8 = x8[(x8.cluster == "patient") & x8.term.str.contains(KEY_RE)][["stratum", "term", "ratio", "lo", "hi"]]
        k8 = k8.merge(make_tab(M5S, "M5").query("cluster == 'patient'")[["stratum", "term", "ratio"]].rename(columns={"ratio": "primary_ratio"}),
                      on=["stratum", "term"], how="left")
        k8["pct_change"] = 100 * (k8.ratio / k8.primary_ratio - 1)
        save(k8, "o2_X8_consistent_records_key_terms.csv")
        print(f"\nX8: stratified M5 without flagged episodes (n = {len(subx):,}); key terms vs primary:")
        print(k8.round(3).to_string(index=False))
    except Exception as e:
        print(f"X8 failed: {type(e).__name__}: {e}")
    # leave-one-province-out on the Global/itemised ratio and group ratios
    lop = []
    for p in top_prov:
        sub = D[D.province_model != p]
        a = amc_tables(fit_block(sub, BLOCKS["M5"], NUMS["M5"], "stratified", fam="qp"), sub.reset_index(drop=True), "M5")[1]
        lop.append({"province_left_out": p, "n": len(sub), "ratio_Global_over_itemised": a.ratio_Global_over_itemised.iloc[0]})
    LOP = pd.DataFrame(lop)
    save(LOP, "o2_X_leave_one_province_out.csv")
    print("\nLeave-one-province-out (largest provinces): Global/itemised standardised ratio")
    print(LOP.round(3).to_string(index=False))


# =============================================================================
# 13. RELIABILITY VERDICT
# =============================================================================
hdr("12. RELIABILITY VERDICT (A / B / C) for each stratified M5 estimate")
rows = []
base = make_tab(M5S, "M5")
bp = base[(base.cluster == "patient") & base.term.str.contains(KEY_RE)]
bq = base[(base.cluster == "province")].set_index("term_full")
for _, r in bp.iterrows():
    s = SL[(SL.stratum == r.stratum) & (SL.term == r.term)] if len(SL) else pd.DataFrame()
    s = s[s.sens.notna()] if len(s) else s
    if "stratum" not in SL.columns or not len(s):
        s = pd.DataFrame(columns=["ratio"])
    dev = 100 * np.abs(s.ratio / r.ratio - 1) if len(s) else pd.Series(dtype=float)
    same_dir = bool(((s.ratio > 1) == (r.ratio > 1)).all()) if len(s) else True
    rb = ROB[(ROB.stratum == r.stratum) & (ROB.term == r.term)]
    cook_ch = float(abs(rb.cook_pct_change.iloc[0])) if len(rb) else np.nan
    qp_ch = float(abs(rb.qp_pct_change.iloc[0])) if len(rb) else np.nan
    q = bq.loc[r.term_full]
    sig_p, sig_v = hit(r), hit(q)
    reasons = []
    if r.sparse_flag:
        reasons.append("level has <30 episodes")
    if not same_dir:
        reasons.append("direction changes in a sensitivity analysis")
    if len(dev) and dev.max() > STABLE_PCT:
        reasons.append(f"changes up to {dev.max():.0f}% across sensitivity analyses")
    if cook_ch > COOK_PCT:
        reasons.append(f"Cook refit changes it by {cook_ch:.0f}%")
    if qp_ch > COOK_PCT:
        reasons.append(f"quasi-Poisson changes it by {qp_ch:.0f}%")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    verdict = "C" if (r.sparse_flag or not same_dir) else ("A" if not reasons else "B")
    rows.append({"stratum": r.stratum, "term": r.term, "ratio": r.ratio, "lo": r.lo, "hi": r.hi, "CI_excludes_1": sig_p,
                 "level_n": r.level_n, "n_sens": len(s), "max_pct_change_sens": float(dev.max()) if len(dev) else np.nan,
                 "same_direction_all": same_dir, "cook_pct": cook_ch, "qp_pct": qp_ch,
                 "province_cluster_CI_excludes_1": sig_v, "verdict": verdict, "reasons": "; ".join(reasons)})
# the standardised Global vs itemised contrast
for _, r in ATT.iterrows():
    pop = r.population
    s = SATT[SATT.population == pop] if len(SATT) else pd.DataFrame()
    dev = 100 * np.abs(s.ratio_Global_over_itemised / r.ratio_Global_over_itemised - 1) if len(s) else pd.Series(dtype=float)
    same = bool(((s.ratio_Global_over_itemised > 1) == (r.ratio_Global_over_itemised > 1)).all()) if len(s) else True
    ratio_hit = lambda lo, hi: bool(lo > 1 or hi < 1)
    sp, sv = ratio_hit(r.ratio_lo_patient, r.ratio_hi_patient), ratio_hit(r.ratio_lo_province, r.ratio_hi_province)
    reasons = []
    if r.share_on_common_support < 0.9:
        reasons.append(f"only {100 * r.share_on_common_support:.0f}% of Global episodes on common support")
    if not same:
        reasons.append("direction changes")
    if len(dev) and dev.max() > STABLE_PCT:
        reasons.append(f"changes up to {dev.max():.0f}% across sensitivity analyses")
    if sp != sv:
        reasons.append("conclusion differs between patient and province clusters")
    rows.append({"stratum": "Global vs itemised (ATT)", "term": pop, "ratio": r.ratio_Global_over_itemised,
                 "lo": r.ratio_lo_patient, "hi": r.ratio_hi_patient, "CI_excludes_1": sp, "level_n": r.n_Global,
                 "n_sens": len(s), "max_pct_change_sens": float(dev.max()) if len(dev) else np.nan,
                 "same_direction_all": same, "cook_pct": np.nan, "qp_pct": np.nan, "province_cluster_CI_excludes_1": sv,
                 "verdict": "C" if not same else ("A" if not reasons else "B"), "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o2_reliability_verdict.csv")
print("A = stable under every check; B = direction stable but size or inference depends on the check named; "
      "C = sparse level or direction not stable (do not interpret).")
print(REL[["stratum", "term", "ratio", "lo", "hi", "verdict", "reasons"]].round(2).to_string(index=False))
print("\nVerdict counts:", REL.verdict.value_counts().to_dict())


# =============================================================================
# 14. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)),
        "n_Global": int((D.stratum == "Global").sum()), "n_Other": int((D.stratum == "Other").sum()),
        "rare_min": RARE_MIN, "boot": NBOOT, "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "deviations_from_protocol": ["admissions are the primary sample", "model stratified by billing mode",
                                     "year-month period dummies instead of year + month",
                                     "rare categories collapsed within strata",
                                     "any_surgical_service not estimated in the Global stratum"]}
(OUT_DIR / "o2_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")


1. LOAD AND VERIFY   [elapsed 0.0 min]
Rows: 15,601; columns: 70
                                                      step     n
                      Episodes in the frozen analysis file 15601
                                        record type Global  8462
                                     record type Inpatient  6223
                         record type Emergency Observation   889
                 record type Temporary Inpatient Admission    27
O2 complete cases (cost and LOS present), all record types 15601
        PRIMARY SAMPLE O2: admissions (Global + Inpatient) 14685
                                            stratum Global  8462
                                             stratum Other  6223

Cost (Int$): median 436, IQR 301-708, mean 602, max 37,984

2. DESCRIPTIVES (cost, Int$)   [elapsed 0.0 min]
                            variable                         level     n      median_IQR   mean   cv
All valid episodes (protocol sample)                           All 15601 

In [36]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Objective 3 (patient cost-sharing) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional).
Same engine and conventions as objective2_expenditure.py v2 (stratified by billing mode, year-month dummies,
rare categories collapsed within strata, cluster-robust SEs, A/B/C reliability verdict).

OUTCOME   patient share = insured_share_irr / total_cost_irr, kept only when the episode is RECONCILED
          (financing gap <= 5%, both shares present) and the share lies in [0, 1] (protocol, O3 primary sample).
MODEL     Fractional logit (binomial quasi-likelihood GLM, logit link), blocks M1-M5 as in Objectives 1-2
          (M2 = + fund, ownership [+ billing mode in the pooled comparison]; M3 = + group, surgical service,
          discharge status; M4 = + province; M5 = + specialty). LOS is not a covariate in O3 (a post-admission
          quantity); sensitivity 7 adds it. Average marginal effects (AME) in percentage points by g-computation
          with delta-method CIs (patient clusters AND province clusters).
PRIMARY SAMPLE  Admissions (Global + Inpatient) that are reconciled with a valid share, as in O1 v3 / O2 v2
          (Emergency Observation and Temporary admissions are not admissions). The protocol sample (all reconciled
          episodes) is fitted as exploratory X0. This is a deviation from the frozen protocol: state it.

WHY THE MODEL IS STRATIFIED BY BILLING MODE (same lesson as O2, checked on this file)
          Global episodes have a near-fixed share (median 8%, IQR 6-10%); itemised episodes are spread from 12% to
          76% (median 55%) and depend heavily on ownership and fund. One additive billing-mode term cannot describe
          both. The primary model is therefore fitted separately in the Global and itemised strata (jointly, as a
          block-diagonal fit so contrasts and clustered covariances are valid). The pooled additive M1-M5 is kept
          for comparison only. Rare levels (< 30 episodes inside a stratum) are collapsed; any_surgical_service is
          dropped in a stratum where one level has < 30 episodes.

SECTIONS
  2  Sample flow; reconciliation by group, code, ownership, billing mode
  3  Descriptives: share by group/code/billing/ownership/fund/specialty/province/period; share of episodes above 10/20/40%
  4  Selection: who is reconciled (the primary sample is a self-selected subset)
  5  Models M1-M5 (pooled comparison and stratified primary); main table (odds ratios, AME in pp)
  6  Global vs itemised, standardised (what Global episodes would pay if billed itemised), by block
  7  Diagnostics within strata: link test, Cook + refit, probit link, fit to observed cells; 5-fold out-of-fold error
  8  Hypothesis 2 (provinces): Wald tests, adjusted ratios, province random intercept (sensitivity 8)
  9  Unreconciled episodes: scenarios 1-4 and whether the conclusions hold under scenarios 1 and 4
 10  Sensitivity analyses 1-8
 11  Exploratory (label as such): protocol sample, inverse-probability-of-reconciliation weights, leave-one-province-out
 12  Reliability verdict (A / B / C)

SCENARIO B  The protocol asks for "Scenario B, as defined in the cholera paper". That definition is not in the
          protocol or the analysis file. SCENARIO_B below is a PROVISIONAL stand-in (recorded shares rescaled to
          the recorded financing base: insured / (insured + requested)). Replace the function with the cholera
          paper's definition before reporting scenario 3.

USAGE
  python objective3_cost_sharing.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-sens8] [--no-extra] [--boot 200]   (bootstraps use at most 100 replicates)
Outputs: folder objective3_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.special import expit

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "o3-v1-stratified"
SEED = 42
RARE_MIN = 30
SPARSE_N_MIN = 30
TOP_PROVINCES = 5
COOK_MULT = 4.0
N_FOLDS = 5
THRESHOLDS = (0.10, 0.20, 0.40)      # share of episodes with patient share above these (protocol)
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
STABLE_PP = 2.0                      # an AME is "stable" if it moves < max(STABLE_PP, 25% of |AME|) pp
STABLE_REL = 0.25
TIME_PP_TOL = 2.0

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="admissions")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-sens8", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=200)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_SENS8, RUN_EXTRA = not args.no_sens, not (args.no_sens or args.no_sens8), not args.no_extra
NBOOT = args.boot
OUT_DIR = INPUT_CSV.parent / "objective3_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BLOCKS = {
    "M1": ["age_group", "sex"],
    "M2": ["age_group", "sex", "insurance_fund_model", "ownership"],
    "M3": ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service",
           "discharge_status_model"],
}
BLOCKS["M4"] = BLOCKS["M3"] + ["province_model"]
BLOCKS["M5"] = BLOCKS["M4"] + ["specialty_model"]
COLLAPSE_VARS = ["ownership", "specialty_model", "province_model", "insurance_fund_model",
                 "discharge_status_model", "sex", "age_group", "group", "any_surgical_service", "period"]
KEY_VARS = ["group", "ownership", "insurance_fund_model", "any_surgical_service"]     # verdict / sensitivity AMEs
REPORT_VARS = KEY_VARS + ["specialty_model", "province_model", "discharge_status_model", "age_group", "sex"]
Z975 = stats.norm.ppf(0.975)
T0 = time.time()


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE SAMPLES
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "total_cost_irr", "insured_share_irr", "requested_total_irr", "fin_gap",
        "reconciled", "patient_share", "patient_share_gt1", "los_capped_p95", "cluster_id", "any_surgical_service",
        "specialty_model", "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "anonymous", "in_O3_O4",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine", "flag_extreme_any"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "anonymous", "in_O3_O4", "reconciled", "patient_share_gt1", "flag_extreme_any"] + \
         [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model",
          "specialty_model", "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
if "specialty_h3" in DALL.columns:
    DALL["specialty_h3"] = DALL["specialty_h3"].fillna("Missing").astype(str)
for c in ["total_cost_irr", "insured_share_irr", "requested_total_irr", "fin_gap", "patient_share", "los_capped_p95",
          "los_capped_p99"]:
    if c in DALL.columns:
        DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(
        ~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")
DALL["ps_ok"] = DALL["in_O3_O4"] & DALL["patient_share"].between(0, 1)
if (DALL["patient_share"].notna() & ~DALL["reconciled"]).any():
    print("  Note: the file also holds a recorded share for non-reconciled episodes; the primary analysis uses reconciled "
          "episodes only (in_O3_O4), the others enter through scenarios 1-4.")
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
unknown_rt = sorted(set(DALL.record_type) - set(ADMISSION_TYPES) - set(NON_ADMISSION))
if unknown_rt:
    print(f"  WARNING: unexpected record types treated as admissions: {unknown_rt}")
valid_total = DALL["total_cost_irr"] > 0

flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
adm = DALL[is_adm]
flow += [{"step": "Admissions (Global + Inpatient)", "n": len(adm)},
         {"step": "  with both insured and requested shares present", "n": int((adm.insured_share_irr.notna() & adm.requested_total_irr.notna()).sum())},
         {"step": "  reconciled (gap <= 5%, both shares present)", "n": int(adm.reconciled.sum())},
         {"step": "  reconciled but patient share > 1 (excluded, counted)", "n": int((adm.reconciled & adm.patient_share_gt1).sum())},
         {"step": "O3 primary sample: reconciled admissions with share in [0, 1]", "n": int(adm.ps_ok.sum())}]
D = (DALL[is_adm & DALL.ps_ok] if args.sample == "admissions" else DALL[DALL.ps_ok]).copy().reset_index(drop=True)
ALT = (DALL[DALL.ps_ok] if args.sample == "admissions" else DALL[is_adm & DALL.ps_ok]).copy().reset_index(drop=True)
SAMPLE_LABEL = "reconciled admissions (Global + Inpatient)" if args.sample == "admissions" else "all reconciled episodes (protocol sample)"
ALT_LABEL = "all reconciled episodes (protocol sample)" if args.sample == "admissions" else "reconciled admissions"
flow.append({"step": f"PRIMARY SAMPLE O3: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o3_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary O3 sample.")
print(f"\nPatient share: median {D.patient_share.median():.3f}, IQR {D.patient_share.quantile(.25):.3f}-"
      f"{D.patient_share.quantile(.75):.3f}, mean {D.patient_share.mean():.3f}")

# reconciliation by group, code, ownership, billing mode (primary-sample retention)
adm_all = DALL[is_adm].copy()
rec_rows = []
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("ownership", "Ownership"), ("billing_mode", "Billing mode"),
                 ("province_model", "Province")]:
    for lev, g in adm_all.groupby(var):
        rec_rows.append({"variable": lab, "level": lev, "n_admissions": len(g), "n_reconciled_valid": int(g.ps_ok.sum()),
                         "pct_kept": 100 * g.ps_ok.mean()})
REC = pd.DataFrame(rec_rows)
save(REC, "o3_reconciliation_by_group_code_ownership.csv")
print("\nShare of admissions kept in the O3 primary sample (protocol: report by group and ownership):")
print(REC[REC.variable.isin(["Group", "Ownership", "Billing mode"])].round(1).to_string(index=False))
print("By ICD-10 code:", REC[REC.variable == "ICD-10 code"].set_index("level").pct_kept.round(1).to_dict())


# Common reference levels in BOTH strata (so effects are comparable across strata and do not flip when the sample changes).
def build_refs(d):
    refs = {"ownership": "Public University-Affiliated", "group": "Hernia",
            "insurance_fund_model": "Rural Health Insurance Fund", "sex": "Male", "age_group": "40-59",
            "discharge_status_model": "Complete Recovery", "any_surgical_service": "Yes", "billing_mode": "Global"}
    refs["specialty_model"] = d["specialty_model"].value_counts().index[0]
    ct = pd.crosstab(d["province_model"], d["stratum"])
    ct = ct[~ct.index.isin(["Other", "Missing"])]
    refs["province_model"] = ct.min(axis=1).idxmax()          # largest province present in BOTH strata
    return {k: v for k, v in refs.items() if v in set(d[k].astype(str))}


REFS = build_refs(D)
print("Reference levels (both strata):", REFS)
MIN_SUPPORT = 10          # episodes per level and province needed for a within-province comparison
FAC_VARS = ("ownership", "specialty_model")     # facility-level exposures: compared only within provinces

# =============================================================================
# 2. ENGINE: block-diagonal (stratified) or pooled fractional-logit / fractional-probit GLM
# =============================================================================
def _dummies(series, levels):
    codes = pd.Categorical(series.astype(str), categories=levels).codes
    return (codes[:, None] == np.arange(1, len(levels))[None, :]).astype(float)


def build_spec(d, cats, nums, refs=None):
    refs = refs if refs is not None else {}
    cols, names, levs = [np.ones((len(d), 1))], ["Intercept"], {}
    for v in cats:
        s = d[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = refs.get(v, order[0])
        ref = ref if ref in order else order[0]
        levs[v] = [ref] + [x for x in order if x != ref]
        cols.append(_dummies(s, levs[v]))
        names += [f"C({v})[T.{lev}]" for lev in levs[v][1:]]
    for v in nums:
        cols.append(d[v].to_numpy(float)[:, None])
        names.append(v)
    X = np.hstack(cols)
    keep = [i for i in range(X.shape[1]) if i == 0 or X[:, i].std() > 0]
    return X[:, keep], [names[i] for i in keep], {"cats": list(cats), "nums": list(nums), "levs": levs, "keep": keep}


def apply_spec(d, spec):
    cols = [np.ones((len(d), 1))]
    for v in spec["cats"]:
        cols.append(_dummies(d[v], spec["levs"][v]))
    for v in spec["nums"]:
        cols.append(d[v].to_numpy(float)[:, None])
    return np.hstack(cols)[:, spec["keep"]]


def merge_periods(vc, minn):
    groups = [[p, [p], int(vc[p])] for p in sorted(vc.index)]
    while len(groups) > 1 and min(g[2] for g in groups) < minn:
        i = int(np.argmin([g[2] for g in groups]))
        j = i + 1 if i + 1 < len(groups) else i - 1
        groups[j][1] += groups[i][1]
        groups[j][2] += groups[i][2]
        groups.pop(i)
    return {m: g[0] for g in groups for m in g[1]}


def make_cmap(ds, vars_, minn=RARE_MIN):
    """Collapse levels with < minn episodes inside this stratum (to 'Other', or to the modal level)."""
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        if v == "period":
            m = merge_periods(vc, minn)
        else:
            rare = [l for l in vc.index if vc[l] < minn]
            m = {}
            if rare:
                total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
                tgt = "Other" if (total >= minn and v not in ("any_surgical_service", "group", "sex")) else vc.index[0]
                m = {l: tgt for l in rare}
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def apply_cmap(d, cm):
    d = d.copy()
    for v, m in cm["map"].items():
        s0 = d[v].astype(str)
        s = s0.map(m).fillna(s0) if m else s0
        seen = cm["seen"][v]
        d[v] = s.where(s.isin(seen), "Other" if "Other" in seen else "__unseen__")
    return d


def _link(fam):
    if fam == "probit":
        return (lambda e: stats.norm.cdf(e)), (lambda e, mu: stats.norm.pdf(e))
    return (lambda e: expit(e)), (lambda e, mu: mu * (1 - mu))


def _bdev(y, mu, w):
    mu = np.clip(mu, 1e-12, 1 - 1e-12)
    t1 = np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0.0)
    t2 = np.where(y < 1, (1 - y) * np.log(np.where(y < 1, 1 - y, 1) / (1 - mu)), 0.0)
    return 2 * np.sum(w * (t1 + t2))


def fit_glm(X, y, fam="logit", w=None):
    """Fractional response GLM (binomial quasi-likelihood), IRLS with step-halving on the binomial deviance.
    fam 'logit' (primary) or 'probit' (link-robustness check). Optional case weights w."""
    n, k = X.shape
    w = np.ones(n) if w is None else np.asarray(w, float)
    inv, dmu = _link(fam)
    p0 = np.clip(np.average(y, weights=w), 1e-4, 1 - 1e-4)
    beta = np.zeros(k)
    beta[0] = np.log(p0 / (1 - p0)) if fam == "logit" else stats.norm.ppf(p0)
    eta = X @ beta
    mu = np.clip(inv(eta), 1e-9, 1 - 1e-9)
    d_old = _bdev(y, mu, w)
    conv = False
    for _ in range(200):
        dm = np.maximum(dmu(eta, mu), 1e-12)
        V = mu * (1 - mu)
        W = w * dm ** 2 / V
        z = eta + (y - mu) / dm
        A, rhs = (X.T * W) @ X, (X.T * W) @ z
        try:
            b_new = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            b_new = np.linalg.pinv(A) @ rhs
        step = 1.0
        for _h in range(30):
            cand = beta + step * (b_new - beta)
            eta_c = np.clip(X @ cand, -30, 30)
            mu_c = np.clip(inv(eta_c), 1e-9, 1 - 1e-9)
            d_new = _bdev(y, mu_c, w)
            if np.isfinite(d_new) and d_new <= d_old + 1e-10 * abs(d_old):
                break
            step /= 2
        done = np.abs(cand - beta).max() < 1e-9 or abs(d_old - d_new) < 1e-12 * max(abs(d_old), 1)
        beta, eta, mu, d_old = cand, eta_c, mu_c, d_new
        if done:
            conv = True
            break
    dm = np.maximum(dmu(eta, mu), 1e-12)
    V = mu * (1 - mu)
    W = w * dm ** 2 / V
    return {"beta": beta, "eta": eta, "mu": mu, "bread": np.linalg.pinv((X.T * W) @ X), "conv": conv, "W": W, "dm": dm,
            "V": V, "w": w, "phi": float(np.sum(w * (y - mu) ** 2 / V) / max(w.sum() - k, 1))}


def cluster_meat(S, groups):
    G = pd.DataFrame(S).groupby(groups).sum().to_numpy()
    return G.T @ G, G.shape[0]


def cov_robust(X, y, f, groups):
    n, k = X.shape
    s = f["w"] * (y - f["mu"]) * f["dm"] / f["V"]
    meat, g = cluster_meat(X * s[:, None], groups)
    if g < 2:
        return np.full((k, k), np.nan), g
    return (g / (g - 1)) * ((n - 1) / max(n - k, 1)) * f["bread"] @ meat @ f["bread"], g


def fit_block(d, cats, nums, mode="stratified", fam="logit", cov=True, minn=RARE_MIN, weights=None):
    d = d.reset_index(drop=True)
    wts = None if weights is None else np.asarray(weights, float)
    if mode == "pooled":
        parts = {"All": np.ones(len(d), bool)}
    else:
        parts = {s: (d["stratum"] == s).to_numpy() for s in ["Global", "Other"] if (d["stratum"] == s).sum() >= 50}
    blocks, Xs, off = {}, [], 0
    for s, rows in parts.items():
        ds = d.loc[rows]
        cs = [c for c in cats if not (mode == "stratified" and c == "billing_mode")]
        cs = list(dict.fromkeys(cs + ["period"]))
        cm = make_cmap(ds, [v for v in cs if v in COLLAPSE_VARS], minn)
        dm_ = apply_cmap(ds, cm)
        refs = {**REFS, "period": min(dm_["period"].unique())}
        X_s, nm_s, spec = build_spec(dm_, cs, nums, refs)
        blocks[s] = {"rows": np.where(rows)[0], "cols": slice(off, off + X_s.shape[1]), "names": nm_s, "spec": spec,
                     "cmap": cm, "X": X_s,
                     "lev_n": [int(X_s[:, j].sum()) if nm.startswith("C(") else np.nan for j, nm in enumerate(nm_s)]}
        off += X_s.shape[1]
        Xs.append((rows, X_s))
    K = off
    y = d["patient_share"].to_numpy(float)
    beta, eta, mu = np.zeros(K), np.zeros(len(d)), np.zeros(len(d))
    W, dm, V, wv = np.zeros(len(d)), np.zeros(len(d)), np.zeros(len(d)), np.ones(len(d))
    bread, conv = np.zeros((K, K)), True
    for b in blocks.values():
        r = b["rows"]
        fb = fit_glm(b["X"], y[r], fam, None if wts is None else wts[r])
        beta[b["cols"]], eta[r], mu[r], W[r], dm[r], V[r], wv[r] = fb["beta"], fb["eta"], fb["mu"], fb["W"], fb["dm"], fb["V"], fb["w"]
        bread[b["cols"], b["cols"]] = fb["bread"]
        b["phi"] = fb["phi"]
        conv &= fb["conv"]
    f = {"beta": beta, "eta": eta, "mu": mu, "bread": bread, "W": W, "dm": dm, "V": V, "w": wv}
    X = None
    if cov:
        X = np.zeros((len(d), K))
        for (rows, X_s), b in zip(Xs, blocks.values()):
            X[np.ix_(np.where(rows)[0], np.arange(b["cols"].start, b["cols"].stop))] = X_s
    allnames = [(f"{s}|{nm}" if mode == "stratified" else nm) for s, b in blocks.items() for nm in b["names"]]
    lev_n = [x for b in blocks.values() for x in b["lev_n"]]
    groups = {"patient": pd.factorize(d["cluster_id"])[0], "province": pd.factorize(d["province_model"])[0]}
    covs = {cl: cov_robust(X, y, f, groups[cl]) for cl in groups} if cov else {}
    return {"mode": mode, "fam": fam, "n": len(d), "names": allnames, "lev_n": lev_n, "beta": beta, "covs": covs, "X": X,
            "y": y, "mu": mu, "eta": eta, "W": W, "dm": dm, "V": V, "w": wv, "phi": {s: b["phi"] for s, b in blocks.items()},
            "conv": conv, "blocks": blocks, "d": d, "K": K, "groups": groups}


def make_tab(core, model, label="patient_share"):
    out = []
    for cl, (cov, G) in core["covs"].items():
        se = np.sqrt(np.clip(np.diag(cov), 0, None))
        use_t = cl == "province"
        crit = stats.t.ppf(0.975, max(G - 1, 1)) if use_t else Z975
        zz = core["beta"] / np.where(se > 0, se, np.nan)
        p = 2 * (stats.t.sf(np.abs(zz), max(G - 1, 1)) if use_t else stats.norm.sf(np.abs(zz)))
        t = pd.DataFrame({"outcome": label, "model": model, "term_full": core["names"], "n": core["n"],
                          "level_n": core["lev_n"], "odds_ratio": np.exp(core["beta"]),
                          "lo": np.exp(core["beta"] - crit * se), "hi": np.exp(core["beta"] + crit * se), "p": p,
                          "se_log": se, "cluster": cl})
        if core["mode"] == "stratified":
            t["stratum"] = t.term_full.str.split("|", n=1).str[0]
            t["term"] = t.term_full.str.split("|", n=1).str[1]
        else:
            t["stratum"], t["term"] = "All", t["term_full"]
        t["sparse_flag"] = t["level_n"] < SPARSE_N_MIN
        out.append(t[(t["term"] != "Intercept") & ~t["term"].str.startswith("C(period)")])
    return pd.concat(out, ignore_index=True)


def crit_p(core, cl, est, se):
    G = core["covs"][cl][1]
    if cl == "province":
        return stats.t.ppf(0.975, G - 1), (2 * stats.t.sf(abs(est / se), G - 1) if se > 0 else np.nan)
    return Z975, (2 * stats.norm.sf(abs(est / se)) if se > 0 else np.nan)


# --- prediction, margins (g-computation) with gradients ----------------------
def pred_part(core, s, dpop, over=None):
    b = core["blocks"][s]
    dm_ = apply_cmap(dpop, b["cmap"])
    for k, v in (over or {}).items():
        dm_[k] = v
    X = apply_spec(dm_, b["spec"])
    eta = np.clip(X @ core["beta"][b["cols"]], -30, 30)
    inv, dmu = _link(core["fam"])
    mu = inv(eta)
    return mu, dmu(eta, mu), X


def margin(core, s, dpop, over=None):
    mu, dm_, X = pred_part(core, s, dpop, over)
    g = np.zeros(core["K"])
    g[core["blocks"][s]["cols"]] = (dm_ @ X) / len(mu)
    return float(mu.mean()), g


def predict_rows(core, dte):
    mu = np.full(len(dte), np.nan)
    for s, b in core["blocks"].items():
        m = (dte["stratum"] == s).to_numpy() if core["mode"] == "stratified" else np.ones(len(dte), bool)
        if m.any():
            mu[m] = pred_part(core, s, dte[m])[0]
    return mu


def est_se(core, grad):
    return {cl: float(np.sqrt(max(grad @ core["covs"][cl][0] @ grad, 0))) for cl in core["covs"]}


def ame_table(core, d, vars_, model):
    """AME in percentage points of each level vs the stratum reference (g-computation, delta-method CIs).
    Facility-level exposures (ownership, specialty) are compared only WITHIN provinces where both the level and the
    reference occur with >= MIN_SUPPORT episodes (otherwise the effect is extrapolated into provinces where the level
    never occurs and cannot be separated from the province). If no province qualifies the AME is not identified (NaN)."""
    rows = []
    d = d.reset_index(drop=True)
    for s, b in core["blocks"].items():
        pop = d[(d["stratum"] == s).to_numpy()] if core["mode"] == "stratified" else d
        dpc = apply_cmap(pop, b["cmap"])
        for v in vars_:
            if v not in b["spec"]["levs"]:
                continue
            levs = b["spec"]["levs"][v]
            ref = levs[0]
            ct = pd.crosstab(pop["province_model"].to_numpy(), dpc[v].to_numpy()) if v in FAC_VARS else None
            m0_all, g0_all = margin(core, s, pop, {v: ref})
            for lev in levs[1:]:
                nm = f"C({v})[T.{lev}]"
                if nm not in b["names"]:
                    continue
                m1_all, _ = margin(core, s, pop, {v: lev})
                pr, n_prov, n_obs = pop, np.nan, b["lev_n"][b["names"].index(nm)]
                if v in FAC_VARS:
                    okp = ct.index[(ct[lev] >= MIN_SUPPORT) & (ct[ref] >= MIN_SUPPORT)] if (lev in ct and ref in ct) else []
                    n_prov = len(okp)
                    pr = pop[pop["province_model"].isin(okp)]
                    n_obs = int((dpc[v] == lev)[pop["province_model"].isin(okp).to_numpy()].sum())
                row = {"model": model, "stratum": s, "variable": v, "level": lev, "reference": ref,
                       "level_n": b["lev_n"][b["names"].index(nm)], "n_support_provinces": n_prov,
                       "n_support_episodes": n_obs, "AME_unrestricted_pp": 100 * (m1_all - m0_all)}
                if len(pr) < 30 or n_obs < 1:
                    row.update({"AME_pp": np.nan, "mean_ref_pp": np.nan, "sparse_flag": True,
                                "note": "not identified within any province"})
                    rows.append(row)
                    continue
                m0, g0 = margin(core, s, pr, {v: ref})
                m1, g1 = margin(core, s, pr, {v: lev})
                a_, gr = 100 * (m1 - m0), 100 * (g1 - g0)
                row.update({"AME_pp": a_, "mean_ref_pp": 100 * m0, "note": ""})
                for cl, se in est_se(core, gr).items():
                    c, p = crit_p(core, cl, a_, se)
                    row[f"lo_{cl}"], row[f"hi_{cl}"], row[f"p_{cl}"] = a_ - c * se, a_ + c * se, p
                row["sparse_flag"] = bool(row["level_n"] < SPARSE_N_MIN or (v in FAC_VARS and n_obs < SPARSE_N_MIN))
                rows.append(row)
    return pd.DataFrame(rows)


def att_table(core, d, model):
    """Global episodes: patient share as billed (Global model) vs if billed itemised (itemised model), standardised.
    Primary rows use only Global episodes on COMMON SUPPORT (province, specialty, fund and ownership levels with >= 30
    episodes in the itemised stratum); the unrestricted row is kept for transparency."""
    d = d.reset_index(drop=True)
    popG0 = d[d["stratum"] == "Global"]
    if "Global" not in core["blocks"] or "Other" not in core["blocks"]:
        return pd.DataFrame()
    O = d[d["stratum"] == "Other"]
    ok = np.ones(len(popG0), bool)
    for v in ["province_model", "specialty_model", "insurance_fund_model", "ownership"]:
        ok &= (popG0[v].map(O[v].value_counts()).fillna(0) >= RARE_MIN).to_numpy()
    popG = popG0[ok]
    rows = []
    for lab, base, g in [("All", popG, "All")] + [(g, popG, g) for g in sorted(d.group.unique())] + \
            [("All, unrestricted", popG0, "All")]:
        pop = base if g == "All" else base[base.group == g]
        if len(pop) < 20:
            continue
        m1, g1 = margin(core, "Global", pop)
        m2, g2 = margin(core, "Other", pop)
        row = {"model": model, "population": f"Global episodes ({lab})", "n_Global": len(pop),
               "share_of_Global_on_support": len(popG) / len(popG0),
               "observed_Global_mean_pp": 100 * pop.patient_share.mean(), "fitted_Global_pp": 100 * m1,
               "if_itemised_pp": 100 * m2, "diff_pp": 100 * (m1 - m2), "ratio_Global_over_itemised": m1 / m2}
        for cl, se in est_se(core, 100 * (g1 - g2)).items():
            c, p = crit_p(core, cl, 100 * (m1 - m2), se)
            row[f"diff_lo_{cl}"], row[f"diff_hi_{cl}"], row[f"diff_p_{cl}"] = 100 * (m1 - m2) - c * se, 100 * (m1 - m2) + c * se, p
        rows.append(row)
    return pd.DataFrame(rows)


def hit(lo, hi, null=0.0):
    return bool(lo > null or hi < null)


# =============================================================================
# 3. DESCRIPTIVES AND THRESHOLDS
# =============================================================================
hdr("2. DESCRIPTIVES (patient share, %)")


def cl_prop(x, cl):
    """Cluster-robust proportion and 95% CI."""
    x = np.asarray(x, float)
    p = x.mean()
    cid = pd.factorize(cl)[0]
    s = pd.Series(x - p).groupby(cid).sum().to_numpy()
    se = np.sqrt((len(s) / (len(s) - 1)) * np.sum(s ** 2)) / len(x) if len(s) > 1 else np.nan
    return p, p - Z975 * se, p + Z975 * se


def cstats(g):
    s = 100 * g["patient_share"]
    row = {"n": len(g), "mean": s.mean(), "sd": s.std(), "median": s.median(), "p25": s.quantile(.25), "p75": s.quantile(.75),
           "median_IQR": f"{s.median():.1f} ({s.quantile(.25):.1f}-{s.quantile(.75):.1f})"}
    for t in THRESHOLDS:
        p, lo, hi = cl_prop(g["patient_share"] > t, g["cluster_id"])
        row[f"pct_gt_{int(t * 100)}"] = 100 * p
        row[f"pct_gt_{int(t * 100)}_lo"], row[f"pct_gt_{int(t * 100)}_hi"] = 100 * lo, 100 * hi
    return row


desc = [{"variable": "PRIMARY SAMPLE", "stratum": "All", "level": "All", **cstats(D)}]
for s, g in D.groupby("stratum"):
    desc.append({"variable": "Billing mode", "stratum": s, "level": s, **cstats(g)})
for var, lab in [("group", "Group"), ("icd10", "ICD-10 code"), ("ownership", "Ownership"),
                 ("insurance_fund_model", "Fund"), ("specialty_model", "Specialty"), ("province_model", "Province"),
                 ("period", "Period (Jalali year-month)"), ("any_surgical_service", "Surgical service")]:
    for (s, lev), g in D.groupby(["stratum", var]):
        desc.append({"variable": lab, "stratum": s, "level": lev, **cstats(g)})
    for lev, g in D.groupby(var):
        desc.append({"variable": lab, "stratum": "All", "level": lev, **cstats(g)})
DESC = pd.DataFrame(desc)
save(DESC, "o3_descriptives_and_thresholds.csv")
cols = ["variable", "stratum", "level", "n", "median_IQR", "mean", "pct_gt_10", "pct_gt_20", "pct_gt_40"]
print(DESC[DESC.variable.isin(["PRIMARY SAMPLE", "Billing mode"])][cols].round(1).to_string(index=False))
print("\nBy group x billing mode:")
print(DESC[(DESC.variable == "Group") & (DESC.stratum != "All")][cols].round(1).to_string(index=False))
print("\nBy ownership x billing mode:")
print(DESC[(DESC.variable == "Ownership") & (DESC.stratum != "All")][cols].round(1).to_string(index=False))

# =============================================================================
# 4. SELECTION: who is reconciled?
# =============================================================================
hdr("3. SELECTION INTO THE PRIMARY SAMPLE (reconciled episodes)")
adm_sel = DALL[is_adm & valid_total].reset_index(drop=True).copy()
adm_sel["rec_y"] = adm_sel["ps_ok"].astype(float)
SEL_CATS = ["age_group", "sex", "insurance_fund_model", "ownership", "group", "any_surgical_service", "province_model",
            "billing_mode"]
cm_sel = make_cmap(adm_sel, [v for v in SEL_CATS + ["period"] if v in COLLAPSE_VARS])
dsel = apply_cmap(adm_sel, cm_sel)
Xsel, nsel, specsel = build_spec(dsel, SEL_CATS + ["period"], [], {**REFS, "period": min(dsel.period.unique())})
fsel = fit_glm(Xsel, adm_sel["rec_y"].to_numpy(), "logit")
covsel, Gsel = cov_robust(Xsel, adm_sel["rec_y"].to_numpy(), fsel, pd.factorize(adm_sel.cluster_id)[0])
sesel = np.sqrt(np.clip(np.diag(covsel), 0, None))
SEL = pd.DataFrame({"term": nsel, "odds_ratio_reconciled": np.exp(fsel["beta"]), "lo": np.exp(fsel["beta"] - Z975 * sesel),
                    "hi": np.exp(fsel["beta"] + Z975 * sesel)})
SEL = SEL[(SEL.term != "Intercept") & ~SEL.term.str.startswith("C(period)") & ~SEL.term.str.startswith("C(province")]
# AUC
pr, yy = fsel["mu"], adm_sel["rec_y"].to_numpy()
ranks = stats.rankdata(pr)
auc = (ranks[yy == 1].sum() - (yy == 1).sum() * ((yy == 1).sum() + 1) / 2) / ((yy == 1).sum() * (yy == 0).sum())
save(SEL, "o3_selection_model.csv")
print(f"Admissions with valid total cost: {len(adm_sel):,}; reconciled with share in [0,1]: {int(yy.sum()):,} "
      f"({100 * yy.mean():.1f}%); selection model AUC = {auc:.3f}")
print(SEL.round(3).to_string(index=False))
cmp_rows = []
for var in ["group", "billing_mode", "ownership", "insurance_fund_model", "any_surgical_service", "age_group", "adm_year"]:
    t = pd.crosstab(adm_sel[var], adm_sel["ps_ok"], normalize="columns").mul(100)
    t.columns = ["pct_among_not_reconciled", "pct_among_reconciled"]
    t = t.reset_index().rename(columns={var: "level"})
    t.insert(0, "variable", var)
    cmp_rows.append(t)
SELC = pd.concat(cmp_rows)
SELC["difference_pp"] = SELC.pct_among_reconciled - SELC.pct_among_not_reconciled
save(SELC, "o3_reconciled_vs_not.csv")
print("\nReconciled vs not (column %):")
print(SELC.round(1).to_string(index=False))
print("A material imbalance means the primary sample is a selected subset: that is why scenarios 1-4 (section 9) and the "
      "inverse-probability-weighted analysis (X1) are reported.")
PROB_REC = pd.Series(fsel["mu"], index=adm_sel.episode_no.values)

# What the unreconciled episodes look like, and whether reconciliation is ignorable given covariates
adm_sel["ins_share"] = adm_sel.insured_share_irr / adm_sel.total_cost_irr
adm_sel["req_share"] = (adm_sel.requested_total_irr / adm_sel.total_cost_irr).clip(0, 1)
adm_sel["unaccounted"] = (adm_sel.total_cost_irr - adm_sel.insured_share_irr - adm_sel.requested_total_irr) / adm_sel.total_cost_irr
adm_sel["reconciled_valid"] = np.where(adm_sel.ps_ok, "reconciled", "not reconciled")
MONEY = adm_sel.groupby(["stratum", "reconciled_valid"]).agg(
    n=("episode_no", "size"), insured_share_mean=("ins_share", "mean"), requested_share_mean=("req_share", "mean"),
    unaccounted_mean=("unaccounted", "mean"), unaccounted_median=("unaccounted", "median"),
    share_unaccounted_gt_50pct=("unaccounted", lambda x: (x > .5).mean()),
    share_unaccounted_negative=("unaccounted", lambda x: (x < -.05).mean())).reset_index()
save(MONEY, "o3_money_decomposition_by_reconciliation.csv")
print("\nShare of total cost recorded as insured / requested / unaccounted, by billing mode and reconciliation:")
print(MONEY.round(3).to_string(index=False))
print("Unreconciled episodes always have money MISSING (total > insured + requested, never the reverse); where it went is "
      "not recorded. Their recorded insured share is therefore a lower bound, and 1 - requested share an upper bound.")
ign = []
for s_ in ["Global", "Other"]:
    ds = adm_sel[adm_sel.stratum == s_].reset_index(drop=True)
    cats = BLOCKS["M5"] + ["period"]
    cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
    dm_ = apply_cmap(ds, cm)
    Xa, nma, _ = build_spec(dm_, cats, [], {**REFS, "period": min(dm_.period.unique())})
    ya = ds.req_share.to_numpy()
    X2 = np.column_stack([Xa, ds.ps_ok.to_numpy(float)])
    fa = fit_glm(X2, ya, "logit")
    cva, _ = cov_robust(X2, ya, fa, pd.factorize(ds.cluster_id)[0])
    b_, se_ = fa["beta"][-1], np.sqrt(cva[-1, -1])
    lin = X2[:, :-1] @ fa["beta"][:-1]
    ame_ = 100 * (expit(lin + b_) - expit(lin)).mean()
    ign.append({"stratum": s_, "n": len(ds), "insurer_share_reconciled_pct": 100 * ya[ds.ps_ok.to_numpy()].mean(),
                "insurer_share_not_reconciled_pct": 100 * ya[~ds.ps_ok.to_numpy()].mean(),
                "adjusted_difference_pp": ame_, "logit_coef": b_, "se": se_})
IGN = pd.DataFrame(ign)
save(IGN, "o3_reconciliation_ignorability.csv")
print("\nInsurer's share of the bill (observed for ALL admissions): reconciled vs not, adjusted for the M5 covariates.")
print(IGN.round(3).to_string(index=False))
print("A large adjusted difference means unreconciled episodes differ in how the bill is financed even among comparable "
      "episodes, so the primary sample cannot be assumed representative (compare the scenarios).")

# =============================================================================
# 5. MODELS M1-M5
# =============================================================================
hdr("4. MODELS M1-M5 (fractional logit; patient and province clusters)")


def blk_cats(m, pooled):
    c = list(BLOCKS[m])
    return c + ["billing_mode"] if pooled and m != "M1" else c


CORE_P, CORE_S, TABS = {}, {}, []
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cp = fit_block(D, blk_cats(m, True), [], "pooled")
    cs = fit_block(D, blk_cats(m, False), [], "stratified")
    CORE_P[m], CORE_S[m] = cp, cs
    TABS += [make_tab(cp, m).assign(spec="pooled additive (comparison only)"),
             make_tab(cs, m).assign(spec="stratified by billing mode (PRIMARY)")]
    print(f"  {m}: pooled terms {len(cp['names']) - 1} (quasi-binomial dispersion {cp['phi']['All']:.3f}) | stratified terms "
          f"{len(cs['names']) - len(cs['blocks'])}, dispersion " + ", ".join(f"{s} {v:.3f}" for s, v in cs["phi"].items())
          + ("" if cs["conv"] and cp["conv"] else "  ** did not converge **"))
LONG = pd.concat(TABS, ignore_index=True)
save(LONG, "o3_models_long_oddsratios.csv")
M5S, M5P = CORE_S["M5"], CORE_P["M5"]
prov_n = D.province_model.value_counts()
top_prov = prov_n.index[:TOP_PROVINCES].tolist()

AME_ALL = ame_table(M5S, D, REPORT_VARS, "M5 stratified")
save(AME_ALL, "o3_ame_M5_stratified_pp.csv")

# identifiability of facility-level terms: are the levels spread over provinces, or confined to a few?
idr = []
for s_ in ["Global", "Other"]:
    ds = D[D.stratum == s_]
    for v in FAC_VARS:
        for lev, g in ds.groupby(v):
            vc = g.province_model.value_counts()
            a = AME_ALL[(AME_ALL.stratum == s_) & (AME_ALL.variable == v) & (AME_ALL.level == lev)]
            idr.append({"stratum": s_, "variable": v, "level": lev, "n": len(g), "n_provinces": g.province_model.nunique(),
                        "top_province": vc.index[0], "top_province_share_pct": 100 * vc.iloc[0] / len(g),
                        "support_provinces_vs_reference": float(a.n_support_provinces.iloc[0]) if len(a) else np.nan,
                        "support_episodes": float(a.n_support_episodes.iloc[0]) if len(a) else np.nan,
                        "concentrated": bool(g.province_model.nunique() <= 3 or 100 * vc.iloc[0] / len(g) > 70 or
                                             (len(a) and np.isfinite(a.n_support_provinces.iloc[0]) and a.n_support_provinces.iloc[0] <= 2))})
IDENT = pd.DataFrame(idr)
save(IDENT, "o3_facility_identifiability.csv")
print("\nIdentifiability of facility-level exposures (ownership, specialty): a level confined to a few provinces cannot be "
      "separated from the province or hospital.")
print(IDENT[IDENT.variable == "ownership"].round(1).to_string(index=False))
IDENT_FLAG = {(r.stratum, r.variable, r.level): r.concentrated for r in IDENT.itertuples()}


def fmt_ame(r, cl):
    star = " *" if hit(r[f"lo_{cl}"], r[f"hi_{cl}"]) else ""
    sp = " [n<30]" if r.sparse_flag else ""
    return f"{r.AME_pp:+.1f} ({r[f'lo_{cl}']:+.1f} to {r[f'hi_{cl}']:+.1f}){star}{sp}"


mt = AME_ALL[AME_ALL.variable.isin(KEY_VARS + ["specialty_model"]) |
             ((AME_ALL.variable == "province_model") & AME_ALL.level.isin(top_prov))].copy()
mt["AME_patient_CI"] = mt.apply(lambda r: fmt_ame(r, "patient"), axis=1)
mt["AME_province_CI"] = mt.apply(lambda r: fmt_ame(r, "province"), axis=1)
orm = make_tab(M5S, "M5")
orm = orm[orm.cluster == "patient"].set_index(["stratum", "term"])
mt["odds_ratio"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "odds_ratio"] for r in mt.itertuples()]
mt["OR_lo"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "lo"] for r in mt.itertuples()]
mt["OR_hi"] = [orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "hi"] for r in mt.itertuples()]
save(mt, "o3_main_table_M5_stratified.csv")
print("\nMAIN TABLE (stratified M5): average marginal effect on patient share, percentage points vs the stratum reference.\n"
      "Patient-cluster and province-cluster (t, G-1 df) 95% CIs; * = CI excludes 0. Facility-level terms (ownership,\n"
      "specialty, province) are best read with the province-cluster CI.")
print(mt[["stratum", "variable", "level", "reference", "level_n", "mean_ref_pp", "AME_patient_CI", "AME_province_CI"]]
      .round(1).to_string(index=False))

pb = AME_ALL.iloc[0:0]
bm = []
for m in ["M2", "M3", "M4", "M5"]:
    a = ame_table(CORE_P[m], D, ["billing_mode"], f"{m} pooled additive")
    bm.append(a)
BM = pd.concat(bm)
save(BM, "o3_pooled_billing_AME_by_block.csv")
print("\nPooled ADDITIVE billing-mode AME (itemised vs Global, pp) [v1-type estimate; one constant effect]:")
print(BM[["model", "AME_pp", "lo_patient", "hi_patient"]].round(2).to_string(index=False))

# =============================================================================
# 6. GLOBAL vs ITEMISED, STANDARDISED
# =============================================================================
hdr("5. GLOBAL vs ITEMISED, standardised on the Global episodes")
ATT = att_table(M5S, D, "M5")
ATT_BLOCKS = pd.concat([att_table(CORE_S[m], D, m).query("population == 'Global episodes (All)'") for m in CORE_S])


def boot_att(d, B, key):
    cl = d[key].astype(str).to_numpy()
    uc, inv = np.unique(cl, return_inverse=True)
    idx_by = [np.where(inv == i)[0] for i in range(len(uc))]
    out = []
    for _ in range(B):
        pick = RNG.integers(0, len(uc), len(uc))
        db = d.iloc[np.concatenate([idx_by[i] for i in pick])].reset_index(drop=True)
        try:
            cb = fit_block(db, BLOCKS["M5"], [], "stratified", cov=False)
            out.append(att_table(cb, db, "b").set_index("population")["diff_pp"])
        except Exception:
            continue
    return pd.concat(out, axis=1)


NB = min(NBOOT, 100)
for key, tag in [("cluster_id", "patient"), ("province_model", "province")]:
    bt = boot_att(D, NB, key)
    q = bt.quantile([.025, .975], axis=1).T
    ATT[f"boot_lo_{tag}"] = [q.loc[p_, 0.025] if p_ in q.index else np.nan for p_ in ATT.population]
    ATT[f"boot_hi_{tag}"] = [q.loc[p_, 0.975] if p_ in q.index else np.nan for p_ in ATT.population]
save(ATT, "o3_global_vs_itemised_standardised_M5.csv")
save(ATT_BLOCKS, "o3_global_vs_itemised_by_block.csv")
print("Patient share (pp) of the Global episodes as billed vs if billed itemised (itemised model applied to them):")
print(ATT[["population", "n_Global", "observed_Global_mean_pp", "if_itemised_pp", "diff_pp", "boot_lo_patient", "boot_hi_patient",
           "boot_lo_province", "boot_hi_province"]].round(1).to_string(index=False))
print("\nAcross blocks (delta-method CI, patient clusters):")
print(ATT_BLOCKS[["model", "fitted_Global_pp", "if_itemised_pp", "diff_pp", "diff_lo_patient", "diff_hi_patient"]].round(1).to_string(index=False))

# fit to observed cell means (fractional logit reproduces them for categorical covariates)
chk = []
for s in ["Global", "Other"]:
    for var in ["group", "ownership"]:
        for lev, g in D[D.stratum == s].groupby(var):
            if len(g) < 30:
                continue
            fitted_s = pred_part(M5S, s, g)[0].mean()
            row = {"stratum": s, "variable": var, "level": lev, "n": len(g), "observed_pp": 100 * g.patient_share.mean(),
                   "stratified_fitted_pp": 100 * fitted_s}
            if var in M5P["blocks"]["All"]["spec"]["levs"]:
                row["pooled_additive_fitted_pp"] = 100 * pred_part(M5P, "All", g)[0].mean()
            chk.append(row)
CHK = pd.DataFrame(chk)
CHK["pooled_error_pp"] = CHK.pooled_additive_fitted_pp - CHK.observed_pp
CHK["stratified_error_pp"] = CHK.stratified_fitted_pp - CHK.observed_pp
save(CHK, "o3_fit_to_observed_cells.csv")
print("\nFitted vs observed cell means (pp): the stratified model reproduces them; the pooled additive does not:")
print(CHK.round(2).to_string(index=False))

# =============================================================================
# 7. DIAGNOSTICS AND OUT-OF-FOLD ERROR
# =============================================================================
hdr("6. DIAGNOSTICS within strata (link test, Cook + refit, probit link) and 5-fold out-of-fold error")
diag = []
cook_flag_all = np.zeros(len(M5S["d"]), bool)
for s, b in M5S["blocks"].items():
    r = b["rows"]
    Xs_, ys = b["X"], M5S["y"][r]
    mu, eta, W = M5S["mu"][r], M5S["eta"][r], M5S["W"][r]
    pat = M5S["groups"]["patient"][r]
    Xl = np.column_stack([Xs_, eta ** 2])
    fl = fit_glm(Xl, ys, "logit")
    covl, _ = cov_robust(Xl, ys, fl, pat)
    tl = fl["beta"][-1] / np.sqrt(covl[-1, -1])
    sw = np.sqrt(W)
    Z = Xs_ * sw[:, None]
    h = np.einsum("ij,jk,ik->i", Z, np.linalg.pinv(Z.T @ Z), Z)
    rp = (ys - mu) / np.sqrt(M5S["V"][r])
    k = Xs_.shape[1]
    cook = rp ** 2 * h / (k * b["phi"] * (1 - h) ** 2)
    flag = cook > COOK_MULT / len(r)
    cook_flag_all[r[flag]] = True
    diag.append({"stratum": s, "n": len(r), "terms": k, "dispersion": b["phi"], "link_t": float(tl),
                 "link_p": float(2 * stats.norm.sf(abs(tl))), "max_cook": float(cook.max()), "n_cook_flag": int(flag.sum()),
                 "max_leverage": float(h.max()), "n_leverage_gt_0.2": int((h > .2).sum()),
                 "share_exact_zero": float((ys == 0).mean())})
DIAG = pd.DataFrame(diag)
save(DIAG, "o3_diagnostics_by_stratum.csv")
print(DIAG.round(3).to_string(index=False))
print("A significant link test means some structure in the mean is missing (check the cell fit and out-of-fold calibration "
      "below for the practical size of the problem).")

M5C = fit_block(M5S["d"][~cook_flag_all], BLOCKS["M5"], [], "stratified", cov=False)
M5Pr = fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit", cov=False)
ame_c = ame_table(fit_block(M5S["d"][~cook_flag_all], BLOCKS["M5"], [], "stratified"), M5S["d"][~cook_flag_all], KEY_VARS, "cook")
ame_p = ame_table(fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit"), D, KEY_VARS, "probit")
key = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].copy()
for nm, a in [("cook_excl", ame_c), ("probit", ame_p)]:
    key = key.merge(a[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": f"AME_{nm}"}),
                    on=["stratum", "variable", "level"], how="left")
    key[f"{nm}_change_pp"] = key[f"AME_{nm}"] - key["AME_pp"]
save(key, "o3_robustness_cook_probit.csv")
print(f"\nCook refit excludes {int(cook_flag_all.sum())} episodes (> {COOK_MULT:g}/n within stratum); probit link refit. "
      f"AME (pp) of key terms:")
print(key.round(2).to_string(index=False))
atp = att_table(fit_block(D, BLOCKS["M5"], [], "stratified", fam="probit", cov=False), D, "probit").iloc[0]
atc = att_table(M5C, M5C["d"], "cook").iloc[0]
print(f"\nGlobal vs itemised (diff, pp): logit {ATT.diff_pp.iloc[0]:.1f} | probit {atp.diff_pp:.1f} | Cook-excluded {atc.diff_pp:.1f}")

# out-of-fold
ids = pd.factorize(D.cluster_id)[0]
uid = np.unique(ids)
np.random.default_rng(SEED).shuffle(uid)
fm = dict(zip(uid, np.arange(len(uid)) % N_FOLDS))
FOLD = np.array([fm[i] for i in ids])
oof_rows, cal_rows = [], []
for m in ["M1", "M3", "M5"]:
    for mode in ["pooled", "stratified"]:
        pred = np.full(len(D), np.nan)
        for k in range(N_FOLDS):
            tr, te = D[FOLD != k], D[FOLD == k]
            core = fit_block(tr, blk_cats(m, mode == "pooled"), [], mode, cov=False)
            pred[FOLD == k] = np.clip(predict_rows(core, te), 0, 1)
        y = D.patient_share.to_numpy()
        for s in ["All", "Global", "Other"]:
            mk = np.ones(len(D), bool) if s == "All" else (D.stratum == s).to_numpy()
            e = y[mk] - pred[mk]
            oof_rows.append({"model": m, "spec": mode, "stratum": s, "n": int(mk.sum()), "MAE_pp": 100 * np.abs(e).mean(),
                             "RMSE_pp": 100 * np.sqrt((e ** 2).mean()), "mean_observed_pp": 100 * y[mk].mean(),
                             "mean_predicted_pp": 100 * pred[mk].mean(), "calibration_diff_pp": 100 * (pred[mk].mean() - y[mk].mean()),
                             "R2_oof": 1 - np.sum(e ** 2) / np.sum((y[mk] - y[mk].mean()) ** 2)})
            if m == "M5":
                q = pd.qcut(pd.Series(pred[mk]).rank(method="first"), 10, labels=False)
                c = pd.DataFrame({"decile": q + 1, "pred": 100 * pred[mk], "obs": 100 * y[mk]}).groupby("decile").mean().reset_index()
                c["model"], c["spec"], c["stratum"] = m, mode, s
                cal_rows.append(c.rename(columns={"pred": "mean_predicted_pp", "obs": "mean_observed_pp"}))
OOF, CAL = pd.DataFrame(oof_rows), pd.concat(cal_rows, ignore_index=True)
save(OOF, "o3_oof_error.csv")
save(CAL, "o3_oof_calibration_deciles.csv")
print("\n" + OOF.round(2).to_string(index=False))
c5 = CAL[(CAL.model == "M5") & (CAL.stratum == "All")].copy()
c5["diff_pp"] = c5.mean_predicted_pp - c5.mean_observed_pp
print("\nM5 calibration by predicted decile, all admissions (predicted minus observed, pp):")
print(c5.pivot(index="decile", columns="spec", values="diff_pp").round(2).to_string())

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for s, col in [("Global", "C0"), ("Other", "C3")]:
        ax[0].hist(100 * D.loc[D.stratum == s, "patient_share"], bins=50, alpha=.6, color=col, label=s, density=True)
    ax[0].set(title="Patient share by billing mode", xlabel="Patient share (%)", ylabel="Density")
    ax[0].legend(frameon=False)
    for k, spec in enumerate(["pooled", "stratified"]):
        for s, col in [("Global", "C0"), ("Other", "C3")]:
            c8 = CAL[(CAL.model == "M5") & (CAL.stratum == s) & (CAL.spec == spec)]
            ax[1 + k].plot(c8.mean_predicted_pp, c8.mean_observed_pp, "o-", color=col, label=s)
        ax[1 + k].plot([0, 100], [0, 100], "k--", lw=.8)
        ax[1 + k].set(title=f"M5 {spec}: out-of-fold calibration", xlabel="Mean predicted (%)", ylabel="Mean observed (%)")
        ax[1 + k].legend(frameon=False, title="billing mode")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o3_distribution_and_calibration.png", dpi=200)
    plt.close(fig)
except Exception as e:
    print(f"(figure skipped: {type(e).__name__}: {e})")

# =============================================================================
# 8. HYPOTHESIS 2: provinces
# =============================================================================
hdr("7. HYPOTHESIS 2: patient share differs across provinces after case-mix adjustment")


def wald(core, idx, cluster="patient"):
    b = core["beta"][idx]
    V = core["covs"][cluster][0][np.ix_(idx, idx)]
    chi2 = float(b @ np.linalg.pinv(V) @ b)
    return chi2, len(idx), float(stats.chi2.sf(chi2, len(idx)))


h2 = []
for s in ["Global", "Other"]:
    sub = D[D.stratum == s]
    kw = stats.kruskal(*[g.patient_share.values for _, g in sub.groupby("province_model") if len(g) >= 5])
    h2.append({"model": "crude (Kruskal-Wallis)", "stratum": s, "df": np.nan, "chi2": float(kw.statistic), "p": float(kw.pvalue)})
for m in ["M4", "M5"]:
    core = CORE_S[m]
    for s, b in core["blocks"].items():
        idx = [b["cols"].start + j for j, nm in enumerate(b["names"]) if nm.startswith("C(province_model)")]
        if idx:
            chi2, df, p = wald(core, idx)
            h2.append({"model": m, "stratum": s, "df": df, "chi2": chi2, "p": p,
                       "note": "Wald, patient-clustered; province-level SE not valid (28 clusters)"})
H2 = pd.DataFrame(h2)
save(H2, "o3_h2_province_tests.csv")
print(H2.round(4).to_string(index=False))
pm = AME_ALL[(AME_ALL.variable == "province_model") & ~AME_ALL.sparse_flag]
save(pm, "o3_h2_province_AME_M5.csv")
for s in ["Global", "Other"]:
    x = pm[pm.stratum == s].sort_values("AME_pp")
    ref = x.reference.iloc[0]
    print(f"\n{s}: adjusted province AME (pp) vs {ref}; extremes (modelled provinces with >= 30 episodes):")
    print(x[["level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].iloc[[0, 1, 2, -3, -2, -1]].round(1).to_string(index=False))


class LaplaceFrac:
    """Fractional-logit with a province random intercept, Laplace-approximated pseudo-ML (Bernoulli kernel)."""

    def __init__(self, X, y, g):
        self.X, self.y, self.g = X, y, g
        self.n, self.k = X.shape
        self.G = int(g.max()) + 1
        self.u = np.zeros(self.G)

    def negll(self, theta):
        beta, sig = theta[:self.k], np.exp(theta[self.k])
        xb = self.X @ beta
        u = self.u.copy()
        for _ in range(40):
            eta = xb + u[self.g]
            p = expit(eta)
            d1, d2 = self.y - p, -p * (1 - p)
            gr = np.bincount(self.g, d1, self.G) - u / sig ** 2
            h = -np.bincount(self.g, d2, self.G) + 1 / sig ** 2
            step = gr / h
            u = u + np.clip(step, -2, 2)
            if np.abs(step).max() < 1e-9:
                break
        self.u = u
        eta = xb + u[self.g]
        p = expit(eta)
        h = np.bincount(self.g, p * (1 - p), self.G) + 1 / sig ** 2
        ll = np.sum(self.y * eta - np.logaddexp(0, eta))
        return -(ll - 0.5 * np.sum(u ** 2) / sig ** 2 - self.G * np.log(sig) - 0.5 * np.sum(np.log(h)))

    def fit(self, beta0):
        th0 = np.concatenate([beta0, [np.log(0.3)]])
        r = optimize.minimize(self.negll, th0, method="L-BFGS-B", options={"maxiter": 500, "maxfun": 200000})
        r = optimize.minimize(self.negll, r.x, method="BFGS", options={"maxiter": 200, "gtol": 1e-5})
        return r.x

    def hessian(self, theta, eps=1e-4):
        p = len(theta)
        H = np.zeros((p, p))
        grad = lambda t: optimize.approx_fprime(t, self.negll, 1e-6)
        for j in range(p):
            e = np.zeros(p)
            e[j] = eps
            H[:, j] = (grad(theta + e) - grad(theta - e)) / (2 * eps)
        return (H + H.T) / 2


S8, VC8 = pd.DataFrame(), pd.DataFrame()


def fit_ri(d, label):
    s8, vc8 = [], []
    for s in ["Global", "Other"]:
        ds = d[d.stratum == s].reset_index(drop=True)
        if len(ds) < 200:
            continue
        t0 = time.time()
        try:
            cats = [v for v in BLOCKS["M5"] if v != "province_model"] + ["period"]
            cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
            dm_ = apply_cmap(ds, cm)
            X8, nm8, _ = build_spec(dm_, cats, [], {**REFS, "period": min(dm_["period"].unique())})
            y8 = ds.patient_share.to_numpy(float)
            g8 = pd.factorize(ds.province_model)[0]
            f0 = fit_glm(X8, y8, "logit")
            m8 = LaplaceFrac(X8, y8, g8)
            theta = m8.fit(f0["beta"])
            H = m8.hessian(theta)
            se8 = np.sqrt(np.clip(np.diag(np.linalg.pinv(H)), 0, None))
            k8 = m8.k
            s8.append(pd.DataFrame({"outcome": "patient_share", "model": "M5 random intercept", "stratum": s, "term": nm8[1:],
                                    "n": len(ds), "level_n": [int(X8[:, j].sum()) if nm8[j].startswith("C(") else np.nan for j in range(1, k8)],
                                    "odds_ratio": np.exp(theta[1:k8]), "lo": np.exp(theta[1:k8] - Z975 * se8[1:k8]),
                                    "hi": np.exp(theta[1:k8] + Z975 * se8[1:k8]), "sens": 8, "sens_name": label}))
            sig = float(np.exp(theta[k8]))
            vc8.append({"stratum": s, "n": len(ds), "provinces": m8.G, "sigma_u": sig,
                        "sigma_u_lo": float(np.exp(theta[k8] - Z975 * se8[k8])), "sigma_u_hi": float(np.exp(theta[k8] + Z975 * se8[k8])),
                        "median_odds_ratio": float(np.exp(0.6745 * np.sqrt(2) * sig)),
                        "note": "pseudo-likelihood: SE of sigma_u approximate"})
            print(f"  {s}: n = {len(ds):,}, provinces = {m8.G}, sigma_u = {sig:.3f} (median odds ratio between two random "
                  f"provinces {np.exp(0.6745 * np.sqrt(2) * sig):.2f}), {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  RI ({s}) FAILED ({type(e).__name__}: {e})")
    return (pd.concat(s8, ignore_index=True) if s8 else pd.DataFrame()), (pd.DataFrame(vc8) if vc8 else pd.DataFrame())


if RUN_SENS8:
    print("\nProvince random intercept, within each stratum (= sensitivity analysis 8):")
    S8, VC8 = fit_ri(D, "Province random intercept")
    if len(VC8):
        save(VC8, "o3_sens8_variance_components.csv")
        save(S8, "o3_sens8_random_intercept_long.csv")
        print(VC8.round(3).to_string(index=False))

# =============================================================================
# 9. UNRECONCILED EPISODES: SCENARIOS 1-4
# =============================================================================
hdr("8. UNRECONCILED EPISODES: scenarios for the patient share (all admissions with a valid total cost)")


def scenario_b(d):
    """PROVISIONAL stand-in for the cholera paper's Scenario B (not defined in the protocol or the analysis file):
    recorded shares rescaled to the recorded financing base, insured / (insured + requested). Episodes without both
    shares keep the recorded lower bound. REPLACE with the cholera paper's definition before reporting."""
    base = d["insured_share_irr"] + d["requested_total_irr"]
    ps = (d["insured_share_irr"] / base).where(base > 0)
    return ps.fillna(d["ps_scen1"]).clip(0, 1)


SC = DALL[is_adm & valid_total].copy().reset_index(drop=True)
tot = SC["total_cost_irr"]
SC["ps_scen1"] = (SC["insured_share_irr"] / tot).clip(0, 1)
isrec = SC["ps_ok"].to_numpy()
own_mean = SC[isrec].groupby("ownership")["patient_share"].mean()
own_bill_mean = SC[isrec].groupby(["ownership", "stratum"])["patient_share"].mean()
SC["ps_scen2"] = np.where(isrec, SC["patient_share"], SC["ownership"].map(own_mean))
SC["ps_scen2b"] = np.where(isrec, SC["patient_share"],
                           [own_bill_mean.get((o, s), own_mean.get(o, np.nan)) for o, s in zip(SC.ownership, SC.stratum)])
SC["ps_scen3"] = np.where(isrec, SC["patient_share"], scenario_b(SC))
upper = np.fmax((tot - SC["requested_total_irr"]) / tot, SC["ps_scen1"]).clip(0, 1)
SC["ps_scen4"] = np.where(isrec, SC["patient_share"], upper)
if "ps_scen3_upper_bound" in DALL.columns:
    diff_up = (SC["ps_scen4"] - DALL.set_index("episode_no").loc[SC.episode_no, "ps_scen3_upper_bound"].to_numpy())
    print(f"  Check: upper bound recomputed vs file column: max |difference| = {np.nanmax(np.abs(diff_up)):.2e}")
SCEN = {"1 recorded lower bound": "ps_scen1", "2 ownership-group mean": "ps_scen2",
        "2b ownership x billing-mode mean (exploratory)": "ps_scen2b",
        "3 'Scenario B' (PROVISIONAL stand-in)": "ps_scen3", "4 upper bound": "ps_scen4"}
sc_rows = []
for lab, col in SCEN.items():
    for stratum_lab, mk in [("All admissions", np.ones(len(SC), bool)), ("Global", (SC.stratum == "Global").to_numpy()),
                            ("Itemised", (SC.stratum == "Other").to_numpy())]:
        x = SC.loc[mk, col]
        okx = x.notna().to_numpy()
        p, lo, hi = cl_prop(x[okx], SC.loc[mk, "cluster_id"][okx])
        sc_rows.append({"scenario": lab, "population": stratum_lab, "n": int(okx.sum()), "mean_share_pct": 100 * p,
                        "lo": 100 * lo, "hi": 100 * hi,
                        "pct_gt_20": 100 * (x[okx] > .2).mean(), "pct_gt_40": 100 * (x[okx] > .4).mean()})
SCS = pd.DataFrame(sc_rows)
sc_rec = SC.loc[isrec, "patient_share"]
SCS = pd.concat([pd.DataFrame([{"scenario": "Reconciled episodes only (primary sample)", "population": "All admissions",
                                "n": int(isrec.sum()), "mean_share_pct": 100 * sc_rec.mean(),
                                "lo": 100 * cl_prop(sc_rec, SC.loc[isrec, "cluster_id"])[1],
                                "hi": 100 * cl_prop(sc_rec, SC.loc[isrec, "cluster_id"])[2],
                                "pct_gt_20": 100 * (sc_rec > .2).mean(), "pct_gt_40": 100 * (sc_rec > .4).mean()}]), SCS])
save(SCS, "o3_scenarios_pooled_mean.csv")
print("Pooled mean patient share under each scenario (reconciled episodes keep their observed share in every scenario):")
print(SCS.round(1).to_string(index=False))
rg = []
for lab, col in SCEN.items():
    for var in ["group", "ownership"]:
        for lev, g in SC.groupby(var):
            rg.append({"scenario": lab, var if False else "variable": var, "level": lev, "n": int(g[col].notna().sum()),
                       "mean_share_pct": 100 * g[col].mean()})
save(pd.DataFrame(rg), "o3_scenarios_by_group_ownership.csv")

# do the conclusions hold under scenarios 1 and 4 (and the others)?
print("\nConclusions under each scenario (stratified M5 refitted on all admissions, scenario share as outcome):")
conc_rows, att_rows = [], []
for lab, col in [("0 primary (reconciled only)", None)] + list(SCEN.items()):
    if col is None:
        dd, ame_s, att_s = D, AME_ALL[AME_ALL.variable.isin(KEY_VARS)].assign(scenario=lab), ATT.assign(scenario=lab)
    else:
        dd = SC.copy()
        dd["patient_share"] = dd[col]
        dd = dd[dd["patient_share"].notna()].reset_index(drop=True)
        cr = fit_block(dd, BLOCKS["M5"], [], "stratified")
        ame_s = ame_table(cr, dd, KEY_VARS, lab).assign(scenario=lab)
        att_s = att_table(cr, dd, lab).assign(scenario=lab)
    conc_rows.append(ame_s)
    att_rows.append(att_s)
SCAME = pd.concat(conc_rows, ignore_index=True)
SCATT = pd.concat(att_rows, ignore_index=True)
save(SCAME, "o3_scenarios_AME_key_terms.csv")
save(SCATT, "o3_scenarios_global_vs_itemised.csv")
base_ame = AME_ALL[AME_ALL.variable.isin(KEY_VARS)].set_index(["stratum", "variable", "level"]).AME_pp
hold = []
for (s_, v_, l_), a0 in base_ame.items():
    row = {"stratum": s_, "variable": v_, "level": l_, "primary_AME_pp": a0}
    ok_all = True
    for lab in SCEN:
        x = SCAME[(SCAME.scenario == lab) & (SCAME.stratum == s_) & (SCAME.variable == v_) & (SCAME.level == l_)]
        a1 = float(x.AME_pp.iloc[0]) if len(x) else np.nan
        row[f"AME_scenario_{lab.split()[0]}"] = a1
    for lab in ["1 recorded lower bound", "4 upper bound"]:
        a1 = row[f"AME_scenario_{lab.split()[0]}"]
        row[f"same_sign_scen{lab.split()[0]}"] = bool(np.sign(a1) == np.sign(a0)) if np.isfinite(a1) else np.nan
    row["holds_under_1_and_4"] = bool(row["same_sign_scen1"] and row["same_sign_scen4"]) if np.isfinite(row["AME_scenario_1"]) and np.isfinite(row["AME_scenario_4"]) else np.nan
    hold.append(row)
HOLD = pd.DataFrame(hold)
save(HOLD, "o3_scenario_conclusions_hold.csv")
print(HOLD.round(2).to_string(index=False))
a1 = SCATT[(SCATT.population == "Global episodes (All)")][["scenario", "observed_Global_mean_pp", "if_itemised_pp", "diff_pp"]]
print("\nGlobal vs itemised (standardised on Global episodes) under each scenario:")
print(a1.round(1).to_string(index=False))
print("PROTOCOL RULE: the primary conclusions must hold under scenarios 1 and 4, otherwise the Discussion must say they do not "
      "(see 'holds_under_1_and_4'; ownership effects are partly built in under scenario 2).")


# =============================================================================
# 10. SENSITIVITY ANALYSES 1-8
# =============================================================================
hdr("9. SENSITIVITY ANALYSES 1-8 (stratified M5; AME in pp)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"), 2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"), 4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"),
        7: ("LOS (P95 cap) added as a covariate", None)}
SL, SATT = pd.DataFrame(), pd.DataFrame()
if RUN_SENS:
    sl, sa = [], []
    for s_id, (name, keepcol) in SENS.items():
        sub = D[D[keepcol]].copy() if keepcol else D.copy()
        nums = []
        if s_id == 7:
            sub["los_cov"] = sub["los_capped_p95"]
            sub = sub[sub.los_cov.notna()]
            nums = ["los_cov"]
        core = fit_block(sub, BLOCKS["M5"], nums, "stratified")
        sl.append(ame_table(core, sub, KEY_VARS, name).assign(sens=s_id, sens_name=name))
        if len(core["blocks"]) == 2:
            sa.append(att_table(core, sub, name).assign(sens=s_id, sens_name=name))
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}, strata = {list(core['blocks'])}")
    # sensitivity 9-10 (added after the O3 diagnostics): the 5% reconciliation tolerance is fixed by the protocol, but the
    # results move with it, so wider tolerances are reported. Patient share is the recorded insured / total in all of them.
    adm_t = DALL[is_adm].copy()
    both_t = adm_t.insured_share_irr.notna() & adm_t.requested_total_irr.notna()
    for s_id, tol in [(9, 0.10), (10, 0.20)]:
        sub = adm_t[both_t & (adm_t.fin_gap <= tol) & adm_t.patient_share.between(0, 1)].reset_index(drop=True)
        name = f"Reconciliation tolerance {int(tol * 100)}% (instead of 5%)"
        core = fit_block(sub, BLOCKS["M5"], [], "stratified")
        sl.append(ame_table(core, sub, KEY_VARS, name).assign(sens=s_id, sens_name=name))
        sa.append(att_table(core, sub, name).assign(sens=s_id, sens_name=name))
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}")
    SL = pd.concat(sl, ignore_index=True)
    SATT = pd.concat(sa, ignore_index=True) if sa else pd.DataFrame()
    save(SL, "o3_sensitivity_AME.csv")
    if len(SATT):
        save(SATT, "o3_sensitivity_global_vs_itemised.csv")
    sdir = []
    for r in AME_ALL[AME_ALL.variable.isin(KEY_VARS)].itertuples():
        x = SL[(SL.stratum == r.stratum) & (SL.variable == r.variable) & (SL.level == r.level)]
        if len(x):
            sdir.append({"stratum": r.stratum, "variable": r.variable, "level": r.level, "main_AME_pp": r.AME_pp,
                         "n_sens": len(x), "same_sign": int((np.sign(x.AME_pp) == np.sign(r.AME_pp)).sum()),
                         "max_abs_change_pp": float((x.AME_pp - r.AME_pp).abs().max())})
    save(pd.DataFrame(sdir), "o3_sensitivity_direction.csv")
    print(pd.DataFrame(sdir).round(2).to_string(index=False))

# =============================================================================
# 11. EXPLORATORY
# =============================================================================
hdr("10. EXPLORATORY (label as such)")
IPW_AME = pd.DataFrame()
if RUN_EXTRA:
    try:
        core0 = fit_block(ALT, BLOCKS["M5"], [], "stratified")
        x0 = ame_table(core0, ALT, KEY_VARS, "X0 " + ALT_LABEL)
        save(x0, "o3_X0_other_sample_AME.csv")
        print(f"X0: stratified M5 on {ALT_LABEL} (n = {len(ALT):,}); key AME (pp):")
        print(x0[["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient"]].round(1).to_string(index=False))
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    try:
        pr_ = PROB_REC.reindex(D.episode_no.values).to_numpy()
        w = 1 / np.clip(pr_, np.quantile(pr_, 0.01), 1)
        w = w / w.mean()
        coreW = fit_block(D, BLOCKS["M5"], [], "stratified", weights=w)
        IPW_AME = ame_table(coreW, D, KEY_VARS, "X1 inverse-probability-of-reconciliation weights")
        save(IPW_AME, "o3_X1_ipw_AME.csv")
        cmpw = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp"]].merge(
            IPW_AME[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": "AME_ipw"}), on=["stratum", "variable", "level"])
        cmpw["change_pp"] = cmpw.AME_ipw - cmpw.AME_pp
        print("\nX1: weighting reconciled episodes by 1/P(reconciled) (observable selection only), key AME (pp):")
        print(cmpw.round(2).to_string(index=False))
        print("(Weights correct only for selection on the covariates in the selection model, not on unobserved factors.)")
    except Exception as e:
        print(f"X1 failed: {type(e).__name__}: {e}")
    # X8 (added in the gap-closing step): Stage C consistency flags were created but never used. EXPLORATORY.
    try:
        _kx = to_bool(D["keep_xsens_consistent"])
        subx = D[_kx]
        core8 = fit_block(subx, BLOCKS["M5"], [], "stratified")
        x8 = ame_table(core8, subx, KEY_VARS, f"X8 consistent records only ({int((~_kx).sum())} flagged episodes removed)")
        save(x8, "o3_X8_consistent_records_AME.csv")
        c8 = AME_ALL[AME_ALL.variable.isin(KEY_VARS)][["stratum", "variable", "level", "level_n", "AME_pp"]].merge(
            x8[["stratum", "variable", "level", "AME_pp"]].rename(columns={"AME_pp": "AME_X8"}), on=["stratum", "variable", "level"])
        c8["change_pp"] = c8.AME_X8 - c8.AME_pp
        c8["same_sign"] = np.sign(c8.AME_X8) == np.sign(c8.AME_pp)
        save(c8, "o3_X8_consistent_records_vs_primary.csv")
        print(f"\nX8: stratified M5 without flagged episodes (n = {len(subx):,}); key AME (pp) vs primary:")
        print(c8.round(2).to_string(index=False))
    except Exception as e:
        print(f"X8 failed: {type(e).__name__}: {e}")
    lop = []
    for p in top_prov:
        sub = D[D.province_model != p]
        a = att_table(fit_block(sub, BLOCKS["M5"], [], "stratified", cov=False), sub, "M5").iloc[0]
        lop.append({"province_left_out": p, "n": len(sub), "diff_pp_Global_minus_itemised": a.diff_pp})
    LOP = pd.DataFrame(lop)
    save(LOP, "o3_X_leave_one_province_out.csv")
    print("\nLeave-one-province-out (largest provinces): standardised Global-minus-itemised difference (pp)")
    print(LOP.round(2).to_string(index=False))

# =============================================================================
# 12. RELIABILITY VERDICT
# =============================================================================
hdr("11. RELIABILITY VERDICT (A / B / C) for the key AME estimates")
rows = []
for r in AME_ALL[AME_ALL.variable.isin(KEY_VARS)].itertuples():
    x = SL[(SL.stratum == r.stratum) & (SL.variable == r.variable) & (SL.level == r.level)] if len(SL) else pd.DataFrame()
    dev = (x.AME_pp - r.AME_pp).abs() if len(x) else pd.Series(dtype=float)
    tol = max(STABLE_PP, STABLE_REL * abs(r.AME_pp))
    same = bool((np.sign(x.AME_pp) == np.sign(r.AME_pp)).all()) if len(x) else True
    ri_same, ri_note = True, ""
    if len(S8):
        t8 = S8[(S8.stratum == r.stratum) & (S8.term == f"C({r.variable})[T.{r.level}]")]
        if len(t8):
            orm_ = float(orm.loc[(r.stratum, f"C({r.variable})[T.{r.level}]"), "odds_ratio"])
            ri_same = bool((t8.odds_ratio.iloc[0] > 1) == (orm_ > 1))
    kk = key[(key.stratum == r.stratum) & (key.variable == r.variable) & (key.level == r.level)]
    cook_ch = float(abs(kk.cook_excl_change_pp.iloc[0])) if len(kk) else np.nan
    prob_ch = float(abs(kk.probit_change_pp.iloc[0])) if len(kk) else np.nan
    sig_p, sig_v = hit(r.lo_patient, r.hi_patient), hit(r.lo_province, r.hi_province)
    conc = bool(IDENT_FLAG.get((r.stratum, r.variable, r.level), False))
    not_ident = not np.isfinite(r.AME_pp)
    hh = HOLD[(HOLD.stratum == r.stratum) & (HOLD.variable == r.variable) & (HOLD.level == r.level)]
    scen_ok = bool(hh.holds_under_1_and_4.iloc[0]) if len(hh) and pd.notna(hh.holds_under_1_and_4.iloc[0]) else True
    reasons = []
    if r.sparse_flag:
        reasons.append("level has <30 episodes")
    if not same:
        reasons.append("sign changes in a sensitivity analysis")
    if not ri_same:
        reasons.append("sign differs with province random intercept")
    if len(dev) and dev.max() > tol:
        reasons.append(f"moves up to {dev.max():.1f} pp across sensitivity analyses")
    if cook_ch > tol:
        reasons.append(f"Cook refit moves it {cook_ch:.1f} pp")
    if prob_ch > tol:
        reasons.append(f"probit link moves it {prob_ch:.1f} pp")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    if not scen_ok:
        reasons.append("sign differs under scenario 1 or 4")
    if conc:
        reasons.append("level confined to <=3 provinces, >70% in one, or compared with the reference in <=2 provinces: hard to separate from province/hospital")
    if not_ident:
        reasons.append("not identified within any province")
    verdict = "C" if (r.sparse_flag or not same or not ri_same or not scen_ok or not_ident) else ("A" if not reasons else "B")
    rows.append({"stratum": r.stratum, "variable": r.variable, "level": r.level, "level_n": r.level_n, "AME_pp": r.AME_pp,
                 "lo_patient": r.lo_patient, "hi_patient": r.hi_patient, "CI_excludes_0": sig_p,
                 "lo_province": r.lo_province, "hi_province": r.hi_province, "province_CI_excludes_0": sig_v,
                 "max_change_sens_pp": float(dev.max()) if len(dev) else np.nan, "tolerance_pp": tol,
                 "cook_change_pp": cook_ch, "probit_change_pp": prob_ch, "verdict": verdict, "reasons": "; ".join(reasons)})
for r in ATT.itertuples():
    pop = r.population
    x = SATT[SATT.population == pop] if len(SATT) else pd.DataFrame()
    dev = (x.diff_pp - r.diff_pp).abs() if len(x) else pd.Series(dtype=float)
    tol = max(STABLE_PP, STABLE_REL * abs(r.diff_pp))
    same = bool((np.sign(x.diff_pp) == np.sign(r.diff_pp)).all()) if len(x) else True
    s1 = SCATT[(SCATT.population == pop) & SCATT.scenario.isin(["1 recorded lower bound", "4 upper bound"])]
    scen_same = bool((np.sign(s1.diff_pp) == np.sign(r.diff_pp)).all()) if len(s1) else True
    sig_p = hit(r.boot_lo_patient, r.boot_hi_patient)
    sig_v = hit(r.boot_lo_province, r.boot_hi_province)
    reasons = []
    if not same:
        reasons.append("sign changes in a sensitivity analysis")
    if len(dev) and dev.max() > tol:
        reasons.append(f"moves up to {dev.max():.1f} pp across sensitivity analyses")
    if not scen_same:
        reasons.append("sign differs under scenario 1 or 4")
    if sig_p != sig_v:
        reasons.append("conclusion differs between patient and province clusters")
    rows.append({"stratum": "Global vs itemised (standardised)", "variable": "billing mode", "level": pop, "level_n": r.n_Global,
                 "AME_pp": r.diff_pp, "lo_patient": r.boot_lo_patient, "hi_patient": r.boot_hi_patient, "CI_excludes_0": sig_p,
                 "lo_province": r.boot_lo_province, "hi_province": r.boot_hi_province, "province_CI_excludes_0": sig_v,
                 "max_change_sens_pp": float(dev.max()) if len(dev) else np.nan, "tolerance_pp": tol, "cook_change_pp": np.nan,
                 "probit_change_pp": np.nan, "verdict": "C" if (not same or not scen_same) else ("A" if not reasons else "B"),
                 "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o3_reliability_verdict.csv")
print("A = stable under every check; B = sign stable but size or inference depends on the check named; "
      "C = sparse level, or sign not stable (sensitivity analyses, random intercept, scenarios 1 and 4): do not interpret.")
print(REL[["stratum", "variable", "level", "level_n", "AME_pp", "lo_patient", "hi_patient", "verdict", "reasons"]].round(1).to_string(index=False))
print("\nVerdict counts:", REL.verdict.value_counts().to_dict())

# =============================================================================
# 13. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)),
        "n_Global": int((D.stratum == "Global").sum()), "n_Other": int((D.stratum == "Other").sum()),
        "rare_min": RARE_MIN, "boot": NBOOT, "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scenario_B": "PROVISIONAL stand-in: insured/(insured+requested); replace with the cholera paper definition",
        "deviations_from_protocol": ["reconciled ADMISSIONS (Global + Inpatient) are the primary sample",
                                     "model stratified by billing mode",
                                     "year-month period dummies instead of year + month",
                                     "rare categories collapsed within strata",
                                     "sensitivity 7 adds LOS (P95 cap) as a covariate because LOS is not in the O3 model"]}
(OUT_DIR / "o3_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")


1. LOAD AND VERIFY   [elapsed 0.0 min]
Rows: 15,601; columns: 70
  Note: the file also holds a recorded share for non-reconciled episodes; the primary analysis uses reconciled episodes only (in_O3_O4), the others enter through scenarios 1-4.
                                                         step     n
                         Episodes in the frozen analysis file 15601
                                           record type Global  8462
                                        record type Inpatient  6223
                            record type Emergency Observation   889
                    record type Temporary Inpatient Admission    27
                              Admissions (Global + Inpatient) 14685
               with both insured and requested shares present 14685
                  reconciled (gap <= 5%, both shares present)  6950
         reconciled but patient share > 1 (excluded, counted)     0
O3 primary sample: reconciled admissions with share in [0, 1]  6950
PRIMARY S

In [37]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
OBJECTIVE 4 AS WRITTEN IN THE PROTOCOL (supplementary analysis). Same code as the first O4 script with these changes only:
  * default sample = all reconciled episodes (the protocol sample; use --sample admissions for the admissions version);
  * bootstrap = COLD start (5 random starts per resample) with one-to-one Jaccard matching, because the warm start
    flattered the stability numbers;
  * "recovers billing mode alone" = ARI with billing mode >= 0.50 (chance-corrected) instead of an 85% agreement, which
    rises mechanically with k; ARI with ownership class is reported next to it.
The post-hoc re-specification that answers the same question after the diagnostics is objective4_joint_profiles_v3_1.py.
Report THIS script's tables (all k, all values, including those that fail the rule) as the protocol-specified O4.

Objective 4 (joint profiles) of the analysis protocol:
"cost, resource use and patient cost-sharing in common digestive-surgical admissions in Iran".

Runs on the FROZEN analysis file (read only). Needs only numpy, pandas and scipy (matplotlib optional); k-means,
silhouette, adjusted Rand index, Jaccard stability and the ridge multinomial model are implemented here, so the
script runs in the same environment as Objectives 1-3.

PROTOCOL (fixed in advance)
  Input      LOS (capped at the group P99), claim lines, cost in Int$ and patient share per episode.
             ln(1 + x) on the first three, then z-score all four.
  Sample     Reconciled episodes (patient share needed). As in O1-O3 the primary sample is reconciled ADMISSIONS
             (Global + Inpatient); the protocol sample (all reconciled episodes) is run as exploratory X0.
  Clustering k-means, 50 random starts, fixed seed, k = 2 to 8.
  Choice of k  Highest mean silhouette among the k where EVERY cluster holds >= 5% of episodes and the bootstrap
             Jaccard stability of EVERY cluster is >= 0.75. All k are reported, including those that fail.
  Stability  200 bootstrap resamples (patients as resampling units); cluster-wise Jaccard (Hennig), adjusted Rand
             index against the full-sample solution, silhouette.
  Profiles   Medians of the four inputs per cluster; composition by group, billing mode, specialty, province
             (and ownership, fund). Ridge multinomial model of membership on the M3 block.
  Global packages  Clustering is run twice, on all episodes and on itemised episodes only. If the all-episode
             solution recovers billing mode alone, the itemised solution is primary.
             "Recovers billing mode alone" is made operational here (it is not defined in the protocol):
             clusters agree with billing mode (each cluster labelled by its majority mode) for at least 85% of episodes.
  Sensitivity  Analyses 1-7 repeat the primary analysis on the altered sample (8, province as a random intercept,
             does not apply to unsupervised clustering and is not run).

CHOICES MADE HERE THAT THE PROTOCOL LEAVES OPEN (state them in the Methods)
  * Bootstrap runs use the full-sample centres plus 2 random starts (3 starts) rather than 50 random starts,
    because a bad local optimum in a resample is not instability of the data and 200 x 7 x 50 fits is slow.
  * Jaccard stability of a cluster = mean over bootstrap samples of its best Jaccard match among the resample's
    clusters, on the episodes present in both (Hennig 2007); the stability of a solution is its WEAKEST cluster.
  * The bootstrap silhouette is computed on a random subsample of 1,500 episodes per resample (the full-sample
    silhouette is exact).
  * Ridge penalty chosen by 5-fold patient-grouped cross-validation (log-loss).
  * Sensitivity analyses use fewer bootstrap resamples (--sens-boot, default 50).

Reliability additions (after the diagnostics on Objectives 2-3): a permuted-data silhouette baseline (is there
structure beyond the marginal distributions?), per-cluster A/B/C verdicts from stability and sensitivity recovery.

USAGE
  python objective4_joint_profiles.py [analysis_file.csv] [--sample admissions|all]
         [--no-sens] [--no-extra] [--boot 200] [--sens-boot 50] [--starts 50]
Outputs: folder objective4_outputs next to the analysis file.
"""

import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_CSV = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases"
    r"\analysis_ready_final\digestive_surgical_analysis_file.csv"
)
VERSION = "o4-protocol-as-written"
SEED = 42
K_RANGE = list(range(2, 9))
MIN_CLUSTER_SHARE = 0.05
MIN_JACCARD = 0.75
ARI_BILLING = 0.50                 # "recovers billing mode alone": chance-corrected agreement (ARI) with billing mode >= 0.50
PURITY_AGREEMENT = 0.85             # solution 'recovers billing mode alone' if clusters match billing mode (majority vote) for >= 85% of episodes
RARE_MIN = 30
BOOT_STARTS = 5                     # bootstrap fits: COLD start, 5 random starts per resample (no full-sample centres)
SIL_SUBSAMPLE = 1500
RIDGE_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
N_FOLDS = 5
NON_ADMISSION = ["Emergency Observation", "Temporary Inpatient Admission"]
ADMISSION_TYPES = ["Global", "Inpatient"]
FEATURES = ["LOS (capped, days)", "Claim lines", "Cost (Int$)", "Patient share"]
M3 = ["age_group", "sex", "insurance_fund_model", "ownership", "billing_mode", "group", "any_surgical_service",
      "discharge_status_model"]
BLOCK_OF = {"age_group": "predisposing", "sex": "predisposing", "insurance_fund_model": "enabling",
            "ownership": "enabling", "billing_mode": "enabling", "group": "need", "any_surgical_service": "need",
            "discharge_status_model": "need", "period": "time"}
COLLAPSE_VARS = ["ownership", "insurance_fund_model", "discharge_status_model", "sex", "age_group", "group",
                 "any_surgical_service", "period"]

parser = argparse.ArgumentParser()
parser.add_argument("input", nargs="?", default=None)
parser.add_argument("--sample", choices=["admissions", "all"], default="all")
parser.add_argument("--no-sens", action="store_true")
parser.add_argument("--no-extra", action="store_true")
parser.add_argument("--boot", type=int, default=200)
parser.add_argument("--sens-boot", type=int, default=50)
parser.add_argument("--starts", type=int, default=50)
args, _ = parser.parse_known_args([] if IN_NOTEBOOK else None)
if args.input:
    INPUT_CSV = Path(args.input)
RUN_SENS, RUN_EXTRA = not args.no_sens, not args.no_extra
NBOOT, NBOOT_SENS, NSTARTS = args.boot, args.sens_boot, args.starts
OUT_DIR = INPUT_CSV.parent / "objective4_protocol_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
T0 = time.time()


def fail(msg):
    print(f"\nSTOP: {msg}")
    if IN_NOTEBOOK:
        raise RuntimeError(msg)
    sys.exit(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def to_bool(s):
    return s if s.dtype == bool else s.astype(str).str.strip().str.upper().isin(["TRUE", "T", "1", "YES"])


def save(df, name):
    df.to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")


def hdr(t):
    print("\n" + "=" * 78 + f"\n{t}   [elapsed {(time.time() - T0) / 60:.1f} min]\n" + "=" * 78)


# =============================================================================
# 1. LOAD, VERIFY, DEFINE SAMPLES
# =============================================================================
hdr("1. LOAD AND VERIFY")
if not INPUT_CSV.exists():
    fail(f"Input file not found: {INPUT_CSV}")
raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False)
need = ["episode_no", "icd10", "group", "cost_intl", "claim_lines", "los_days", "los_capped_p99", "los_capped_p95",
        "patient_share", "fin_gap", "reconciled", "in_O3_O4", "in_O4", "cluster_id", "any_surgical_service",
        "specialty_model", "province_model", "insurance_fund_model", "ownership", "billing_mode", "record_type",
        "discharge_status", "age_group", "sex", "adm_year", "adm_month", "insured_share_irr", "total_cost_irr",
        "keep_sens1_surgical", "keep_sens2_itemised", "keep_sens3_not_K81", "keep_sens4_no_extremes",
        "keep_sens5_not_anonymous", "keep_sens6_not_doctor_of_medicine"]
miss = [c for c in need if c not in raw.columns]
if miss:
    fail(f"Columns missing from the analysis file: {miss}")
if not raw["episode_no"].is_unique:
    fail("episode_no is not unique.")
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")

DALL = raw.copy()
for c in ["any_surgical_service", "reconciled", "in_O3_O4", "in_O4"] + [c for c in DALL.columns if c.startswith("keep_sens")]:
    DALL[c] = to_bool(DALL[c])
DALL["any_surgical_service"] = np.where(DALL["any_surgical_service"], "Yes", "No")
DALL["adm_year"] = DALL["adm_year"].astype("Int64").astype(str)
DALL["adm_month"] = DALL["adm_month"].astype("Int64").astype(str)
DALL["cluster_id"] = DALL["cluster_id"].astype(str)
for c in ["discharge_status", "sex", "age_group", "ownership", "insurance_fund_model", "province_model", "specialty_model",
          "billing_mode", "group", "icd10", "record_type"]:
    DALL[c] = DALL[c].fillna("Missing").astype(str)
for c in ["cost_intl", "claim_lines", "los_days", "los_capped_p99", "los_capped_p95", "patient_share", "fin_gap"]:
    DALL[c] = pd.to_numeric(DALL[c], errors="coerce")
if "discharge_status_model" not in DALL.columns:
    cnt = DALL["discharge_status"].value_counts()
    DALL["discharge_status_model"] = DALL["discharge_status"].where(~DALL["discharge_status"].isin(cnt[cnt < RARE_MIN].index), "Other")
DALL["discharge_status_model"] = DALL["discharge_status_model"].fillna("Missing").astype(str)
DALL["period"] = DALL["adm_year"] + "-" + DALL["adm_month"].str.zfill(2)
DALL.loc[DALL["period"] == "1400-12", "period"] = "1401-01"
DALL["stratum"] = np.where(DALL["billing_mode"] == "Global", "Global", "Other")
DALL["O4_ok"] = (DALL["in_O4"] & DALL["reconciled"] & DALL["patient_share"].between(0, 1) & DALL["los_capped_p99"].notna()
                 & DALL["claim_lines"].notna() & (DALL["cost_intl"] > 0))
is_adm = ~DALL.record_type.isin(NON_ADMISSION)
flow = [{"step": "Episodes in the frozen analysis file", "n": len(DALL)}]
for t in ADMISSION_TYPES + NON_ADMISSION:
    flow.append({"step": f"  record type {t}", "n": int((DALL.record_type == t).sum())})
flow += [{"step": "Admissions (Global + Inpatient)", "n": int(is_adm.sum())},
         {"step": "  reconciled with all four inputs present (O4 sample)", "n": int((is_adm & DALL.O4_ok).sum())}]
D = (DALL[is_adm & DALL.O4_ok] if args.sample == "admissions" else DALL[DALL.O4_ok]).copy().reset_index(drop=True)
ALT = (DALL[DALL.O4_ok] if args.sample == "admissions" else DALL[is_adm & DALL.O4_ok]).copy().reset_index(drop=True)
SAMPLE_LABEL = "reconciled admissions (Global + Inpatient)" if args.sample == "admissions" else "all reconciled episodes (protocol sample)"
ALT_LABEL = "all reconciled episodes (protocol sample)" if args.sample == "admissions" else "reconciled admissions"
flow.append({"step": f"PRIMARY SAMPLE O4: {SAMPLE_LABEL}", "n": len(D)})
for s in ["Global", "Other"]:
    flow.append({"step": f"  stratum {s}", "n": int((D.stratum == s).sum())})
save(pd.DataFrame(flow), "o4_sample_flow.csv")
print(pd.DataFrame(flow).to_string(index=False))
if D.stratum.nunique() < 2:
    fail("Only one billing-mode stratum in the primary O4 sample.")


def feature_matrix(d, los_col="los_capped_p99"):
    F = np.column_stack([np.log1p(d[los_col].to_numpy(float)), np.log1p(d["claim_lines"].to_numpy(float)),
                         np.log1p(d["cost_intl"].to_numpy(float)), d["patient_share"].to_numpy(float)])
    return F


def zscore(F):
    return (F - F.mean(0)) / F.std(0)


desc_rows = []
for lab, g in [("All", D)] + [(s, D[D.stratum == s]) for s in ["Global", "Other"]]:
    for name, col in [("LOS (capped P99)", "los_capped_p99"), ("Claim lines", "claim_lines"), ("Cost (Int$)", "cost_intl"),
                      ("Patient share", "patient_share")]:
        x = g[col]
        desc_rows.append({"sample": lab, "input": name, "n": len(x), "median": x.median(), "p25": x.quantile(.25), "p75": x.quantile(.75),
                          "mean": x.mean(), "sd": x.std()})
save(pd.DataFrame(desc_rows), "o4_input_descriptives.csv")
CORR = pd.DataFrame(np.corrcoef(zscore(feature_matrix(D)).T), index=FEATURES, columns=FEATURES)
save(CORR.reset_index(), "o4_input_correlations.csv")
print("\nInputs (original scale):")
print(pd.DataFrame(desc_rows).round(2).to_string(index=False))
print("\nCorrelations of the transformed inputs:")
print(CORR.round(2).to_string())


# =============================================================================
# 2. CLUSTERING ENGINE (numpy): k-means, silhouette, ARI, Jaccard
# =============================================================================
def kmeans_fit(Z, k, starts, rng, init_centres=None, max_iter=60, tol=1e-4):
    """k-means (Lloyd) with several starts run in one vectorised batch; returns the lowest-inertia solution.
    Starts are random draws of k distinct observations; optional extra start from init_centres."""
    n, d = Z.shape
    C = np.stack([Z[rng.choice(n, k, replace=False)] for _ in range(starts)])
    if init_centres is not None:
        C = np.concatenate([C, init_centres[None]], axis=0)
    x2 = (Z ** 2).sum(1)
    ar = np.arange(k)
    for _ in range(max_iter):
        D2 = x2[None, :, None] - 2 * np.einsum("nd,skd->snk", Z, C) + (C ** 2).sum(2)[:, None, :]
        lab = D2.argmin(2)
        oh = (lab[:, :, None] == ar[None, None, :]).astype(float)
        cnt = oh.sum(1)
        newC = np.where(cnt[:, :, None] > 0, np.einsum("snk,nd->skd", oh, Z) / np.maximum(cnt[:, :, None], 1), C)
        shift = np.abs(newC - C).max()
        C = newC
        if shift < tol:
            break
    D2 = x2[None, :, None] - 2 * np.einsum("nd,skd->snk", Z, C) + (C ** 2).sum(2)[:, None, :]
    lab = D2.argmin(2)
    inertia = np.take_along_axis(D2, lab[:, :, None], 2).sum((1, 2))
    empty = np.array([len(np.unique(l)) < k for l in lab])
    inertia = np.where(empty, np.inf, inertia)
    b = int(np.argmin(inertia))
    return lab[b], C[b], float(inertia[b])


def relabel_by_size(lab, C):
    order = np.argsort(-np.bincount(lab, minlength=len(C)))
    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))
    return inv[lab], C[order]


def silhouette(Z, lab, chunk=1000):
    k = lab.max() + 1
    n = len(Z)
    oh = (lab[:, None] == np.arange(k)[None, :]).astype(float)
    cnt = oh.sum(0)
    x2 = (Z ** 2).sum(1)
    s = np.empty(n)
    for i in range(0, n, chunk):
        Zi = Z[i:i + chunk]
        D = np.sqrt(np.maximum(x2[i:i + chunk, None] - 2 * Zi @ Z.T + x2[None, :], 0))
        S = D @ oh                                             # sum of distances to each cluster
        li = lab[i:i + chunk]
        own = S[np.arange(len(li)), li]
        a = own / np.maximum(cnt[li] - 1, 1)
        M = S / np.maximum(cnt[None, :], 1)
        M[np.arange(len(li)), li] = np.inf
        b = M.min(1)
        si = (b - a) / np.maximum(np.maximum(a, b), 1e-12)
        si[cnt[li] <= 1] = 0
        s[i:i + chunk] = si
    return s


def contingency(a, b):
    ka, kb = a.max() + 1, b.max() + 1
    return np.bincount(a * kb + b, minlength=ka * kb).reshape(ka, kb)


def ari(a, b):
    c = contingency(a, b)
    comb = lambda x: x * (x - 1) / 2
    sij, sa, sb, n = comb(c).sum(), comb(c.sum(1)).sum(), comb(c.sum(0)).sum(), comb(c.sum())
    exp = sa * sb / n
    return float((sij - exp) / (0.5 * (sa + sb) - exp)) if (0.5 * (sa + sb) - exp) != 0 else 1.0


def nmi(a, b):
    c = contingency(a, b).astype(float)
    n = c.sum()
    pa, pb, pab = c.sum(1) / n, c.sum(0) / n, c / n
    nz = pab > 0
    mi = np.sum(pab[nz] * np.log(pab[nz] / (pa[:, None] * pb[None, :])[nz]))
    ha, hb = -np.sum(pa[pa > 0] * np.log(pa[pa > 0])), -np.sum(pb[pb > 0] * np.log(pb[pb > 0]))
    return float(mi / np.sqrt(ha * hb)) if ha > 0 and hb > 0 else 0.0


def jaccard_best(a, b):
    """One-to-one (Hungarian) Jaccard matching of the clusters of b to the clusters of a, on the same observations;
    returns the matched Jaccard for each cluster of a (0 when a cluster has no partner)."""
    c = contingency(a, b).astype(float)
    ua, ub = c.sum(1)[:, None], c.sum(0)[None, :]
    J = c / np.maximum(ua + ub - c, 1)
    r, cc = linear_sum_assignment(-J)
    out = np.zeros(c.shape[0])
    out[r] = J[r, cc]
    return out


def assign(Z, C):
    D2 = (Z ** 2).sum(1)[:, None] - 2 * Z @ C.T + (C ** 2).sum(1)[None, :]
    return D2.argmin(1)


def cluster_units(d):
    codes, _ = pd.factorize(d["cluster_id"])
    idx_by = [np.where(codes == i)[0] for i in range(codes.max() + 1)]
    return idx_by


def analyse(d, label, nboot, starts=NSTARTS, seed=SEED, los_col="los_capped_p99", verbose=True, ks=K_RANGE):
    """Full protocol pipeline on one sample: k-means for k in ks, silhouette, bootstrap stability, selection rule."""
    rng = np.random.default_rng(seed)
    F = feature_matrix(d, los_col)
    Z = zscore(F)
    n = len(Z)
    units = cluster_units(d)
    rows, per_cluster, sols = [], [], {}
    for k in ks:
        lab, C, inertia = kmeans_fit(Z, k, starts, rng)
        lab, C = relabel_by_size(lab, C)
        sil_i = silhouette(Z, lab)
        sizes = np.bincount(lab, minlength=k)
        jac = np.zeros((nboot, k))
        aris, sils = np.zeros(nboot), np.zeros(nboot)
        for b in range(nboot):
            pick = rng.integers(0, len(units), len(units))
            ids = np.concatenate([units[i] for i in pick])
            lb, Cb, _ = kmeans_fit(Z[ids], k, BOOT_STARTS, rng)
            U = np.unique(ids)
            lbU = assign(Z[U], Cb)                       # labels of the unique resampled episodes
            lbU = np.unique(lbU, return_inverse=True)[1]
            l0U = lab[U]
            jac[b] = jaccard_best(l0U, lbU)[np.arange(k)] if lbU.max() + 1 >= 1 else 0
            aris[b] = ari(l0U, lbU)
            sub = rng.choice(len(U), min(SIL_SUBSAMPLE, len(U)), replace=False)
            lsub = lbU[sub]
            sils[b] = silhouette(Z[U][sub], lsub).mean() if len(np.unique(lsub)) > 1 else np.nan
        mj = jac.mean(0)
        row = {"analysis": label, "k": k, "n": n, "silhouette": float(sil_i.mean()), "inertia": inertia,
               "min_cluster_share": float(sizes.min() / n), "cluster_sizes": " / ".join(str(x) for x in sizes),
               "min_jaccard": float(mj.min()), "mean_jaccard": float(mj.mean()), "ari_mean": float(np.nanmean(aris)),
               "ari_sd": float(np.nanstd(aris)), "boot_silhouette_mean": float(np.nanmean(sils)), "n_boot": nboot}
        row["size_ok"] = row["min_cluster_share"] >= MIN_CLUSTER_SHARE
        row["stability_ok"] = row["min_jaccard"] >= MIN_JACCARD
        row["passes_rule"] = bool(row["size_ok"] and row["stability_ok"])
        rows.append(row)
        for j in range(k):
            per_cluster.append({"analysis": label, "k": k, "cluster": j + 1, "n": int(sizes[j]), "share": sizes[j] / n,
                                "jaccard_mean": float(mj[j]), "jaccard_sd": float(jac[:, j].std()),
                                "jaccard_p05": float(np.percentile(jac[:, j], 5)),
                                "silhouette_cluster": float(sil_i[lab == j].mean())})
        sols[k] = {"labels": lab, "centres": C, "silhouette_i": sil_i}
        if verbose:
            print(f"  [{label}] k={k}: silhouette {row['silhouette']:.3f}, min share {row['min_cluster_share']:.3f}, "
                  f"min Jaccard {row['min_jaccard']:.2f}, ARI {row['ari_mean']:.2f}, passes rule: {row['passes_rule']}")
    T = pd.DataFrame(rows)
    ok = T[T.passes_rule]
    chosen = int(ok.loc[ok.silhouette.idxmax(), "k"]) if len(ok) else None
    return {"label": label, "table": T, "clusters": pd.DataFrame(per_cluster), "sols": sols, "chosen_k": chosen, "Z": Z, "F": F,
            "data": d}


def billing_recovery(res, k):
    """Does the solution at k recover billing mode alone?"""
    d, lab = res["data"], res["sols"][k]["labels"]
    bill = (d["stratum"] == "Global").astype(int).to_numpy()
    ct = pd.crosstab(lab, d["stratum"].to_numpy())
    pure_share = ct.max(1) / ct.sum(1)
    in_pure = ct.max(1).sum() / ct.sum().sum()          # weighted purity: agreement with billing mode by majority vote
    return {"k": k, "ARI_vs_billing_mode": ari(lab, bill), "NMI_vs_billing_mode": nmi(lab, bill),
            "billing_mode_agreement": float(in_pure), "n_clusters_ge90pct_one_mode": int((pure_share >= 0.90).sum()),
            "ARI_vs_ownership_class": ari(lab, (d["ownership"] == "Public University-Affiliated").astype(int).to_numpy()),
            "recovers_billing_mode_alone": bool(ari(lab, bill) >= ARI_BILLING)}


# =============================================================================
# 3. CLUSTERING: ALL EPISODES AND ITEMISED ONLY
# =============================================================================
hdr(f"2. K-MEANS, k = {K_RANGE[0]}-{K_RANGE[-1]} ({NSTARTS} random starts; {NBOOT} bootstrap resamples)")
print("All reconciled admissions:")
RES_ALL = analyse(D, "All episodes", NBOOT)
hdr("2b. ITEMISED EPISODES ONLY")
D_ITEM = D[D.stratum == "Other"].reset_index(drop=True)
RES_ITEM = analyse(D_ITEM, "Itemised only", NBOOT)
STAB = pd.concat([RES_ALL["table"], RES_ITEM["table"]], ignore_index=True)
CL = pd.concat([RES_ALL["clusters"], RES_ITEM["clusters"]], ignore_index=True)
save(STAB, "o4_k_selection_all_k.csv")
save(CL, "o4_cluster_stability_by_cluster.csv")
cols = ["analysis", "k", "silhouette", "min_cluster_share", "min_jaccard", "mean_jaccard", "ari_mean", "boot_silhouette_mean",
        "size_ok", "stability_ok", "passes_rule", "cluster_sizes"]
print("\nAll k (the protocol asks for every value, including those that fail the rule):")
print(STAB[cols].round(3).to_string(index=False))
for r in (RES_ALL, RES_ITEM):
    print(f"\n{r['label']}: chosen k = {r['chosen_k']}" if r["chosen_k"] else
          f"\n{r['label']}: NO k satisfies the rule (size >= {MIN_CLUSTER_SHARE:.0%} and Jaccard >= {MIN_JACCARD}).")

hdr("3. DO THE ALL-EPISODE SOLUTIONS JUST RECOVER BILLING MODE?")
BR = pd.DataFrame([billing_recovery(RES_ALL, k) for k in K_RANGE])
save(BR, "o4_billing_mode_recovery_all_episodes.csv")
print(BR.round(3).to_string(index=False))
k_all = RES_ALL["chosen_k"]
if k_all is None:
    PRIMARY, why = RES_ITEM, "no k passes the rule in the all-episode analysis, so the itemised solution is primary"
    print(f"\nDecision: {why}.")
else:
    rec = BR[BR.k == k_all].iloc[0]
    if rec.recovers_billing_mode_alone:
        PRIMARY, why = RES_ITEM, f"the all-episode solution (k = {k_all}) recovers billing mode alone"
    else:
        PRIMARY, why = RES_ALL, (f"the all-episode solution (k = {k_all}) does not recover billing mode alone "
                                 f"({100 * rec.billing_mode_agreement:.0f}% of episodes agree with billing mode, "
                                 f"ARI {rec.ARI_vs_billing_mode:.2f})")
    print(f"\nDecision (protocol rule): {why}. PRIMARY = {PRIMARY['label']}.")
if PRIMARY["chosen_k"] is None:
    best = PRIMARY["table"][PRIMARY["table"].size_ok].sort_values("silhouette", ascending=False)
    K_PRIMARY = int(best.k.iloc[0]) if len(best) else K_RANGE[0]
    print(f"The primary analysis has no k that passes the rule; k = {K_PRIMARY} (highest silhouette with every cluster >= 5%) "
          f"is shown for DESCRIPTION ONLY and must be reported as failing the stability rule.")
    PRIMARY_PASSES = False
else:
    K_PRIMARY, PRIMARY_PASSES = PRIMARY["chosen_k"], True
D_PRIM = PRIMARY["data"]
LAB = PRIMARY["sols"][K_PRIMARY]["labels"]
print(f"Primary solution: {PRIMARY['label']}, k = {K_PRIMARY}, n = {len(D_PRIM):,}")
DECISION = pd.DataFrame([{"primary_analysis": PRIMARY["label"], "k": K_PRIMARY, "passes_rule": PRIMARY_PASSES, "reason": why,
                          "all_episode_k": k_all, "itemised_k": RES_ITEM["chosen_k"]}])
save(DECISION, "o4_primary_solution_decision.csv")

# =============================================================================
# 4. PROFILES
# =============================================================================
hdr("4. PROFILES OF THE PRIMARY SOLUTION")


def profiles(res, k, tag):
    d, lab = res["data"].copy(), res["sols"][k]["labels"]
    d["cluster"] = lab + 1
    med = d.groupby("cluster").agg(n=("episode_no", "size"), LOS_median=("los_capped_p99", "median"),
                                   LOS_p25=("los_capped_p99", lambda x: x.quantile(.25)), LOS_p75=("los_capped_p99", lambda x: x.quantile(.75)),
                                   lines_median=("claim_lines", "median"), lines_p25=("claim_lines", lambda x: x.quantile(.25)),
                                   lines_p75=("claim_lines", lambda x: x.quantile(.75)), cost_median=("cost_intl", "median"),
                                   cost_p25=("cost_intl", lambda x: x.quantile(.25)), cost_p75=("cost_intl", lambda x: x.quantile(.75)),
                                   share_median=("patient_share", "median"), share_p25=("patient_share", lambda x: x.quantile(.25)),
                                   share_p75=("patient_share", lambda x: x.quantile(.75))).reset_index()
    med["pct_of_episodes"] = 100 * med.n / med.n.sum()
    Zc = res["Z"]
    cz = pd.DataFrame([Zc[lab == j].mean(0) for j in range(k)], columns=[f"z_{f}" for f in FEATURES])
    cz.insert(0, "cluster", np.arange(1, k + 1))
    names = []
    for j in range(k):
        tags = []
        for f, short in zip(cz.iloc[j, 1:], ["LOS", "lines", "cost", "share"]):
            if f > 0.5:
                tags.append(f"{short} high")
            elif f < -0.5:
                tags.append(f"{short} low")
        names.append(", ".join(tags) if tags else "near average")
    med["description"] = names
    comp = []
    for var, lab_v in [("group", "Group"), ("billing_mode", "Billing mode"), ("ownership", "Ownership"),
                       ("insurance_fund_model", "Fund"), ("specialty_model", "Specialty"), ("province_model", "Province"),
                       ("icd10", "ICD-10 code"), ("any_surgical_service", "Surgical service"), ("period", "Period")]:
        ct = pd.crosstab(d[var], d["cluster"])
        overall = ct.sum(1) / ct.sum().sum()
        for lev in ct.index:
            for c in ct.columns:
                comp.append({"variable": lab_v, "level": lev, "cluster": c, "n": int(ct.loc[lev, c]),
                             "pct_of_cluster": 100 * ct.loc[lev, c] / ct[c].sum(), "pct_of_level_in_cluster": 100 * ct.loc[lev, c] / ct.loc[lev].sum(),
                             "pct_overall": 100 * overall[lev],
                             "lift": (ct.loc[lev, c] / ct[c].sum()) / overall[lev]})
    COMP = pd.DataFrame(comp)
    save(med, f"o4_profiles_medians_{tag}.csv")
    save(cz, f"o4_profiles_centres_z_{tag}.csv")
    save(COMP, f"o4_profiles_composition_{tag}.csv")
    return med, cz, COMP


MED, CZ, COMP = profiles(PRIMARY, K_PRIMARY, "primary")
print(f"Medians (IQR) of the four inputs by cluster (original scale), primary solution k = {K_PRIMARY}:")
show = MED.copy()
for nm, a, b, c in [("LOS (days)", "LOS_median", "LOS_p25", "LOS_p75"), ("Claim lines", "lines_median", "lines_p25", "lines_p75"),
                    ("Cost (Int$)", "cost_median", "cost_p25", "cost_p75"), ("Patient share", "share_median", "share_p25", "share_p75")]:
    f = (lambda x: f"{x:.2f}") if nm == "Patient share" else (lambda x: f"{x:,.1f}" if nm == "LOS (days)" else f"{x:,.0f}")
    show[nm] = [f"{f(x)} ({f(y)}-{f(z)})" for x, y, z in zip(show[a], show[b], show[c])]
print(show[["cluster", "n", "pct_of_episodes", "LOS (days)", "Claim lines", "Cost (Int$)", "Patient share", "description"]].round(1).to_string(index=False))
for var in ["Billing mode", "Group", "Ownership", "Surgical service"]:
    t = COMP[COMP.variable == var].pivot(index="level", columns="cluster", values="pct_of_cluster").round(1)
    print(f"\n{var}: % of each cluster"); print(t.to_string())
sp = COMP[(COMP.variable.isin(["Specialty", "Province"])) & (COMP.n >= 30)].sort_values("lift", ascending=False)
print("\nSpecialties and provinces most over-represented in a cluster (lift = share in cluster / overall share; n >= 30):")
print(sp.head(12)[["variable", "level", "cluster", "n", "pct_of_cluster", "pct_overall", "lift"]].round(2).to_string(index=False))
if PRIMARY is not RES_ALL:
    MED_A, _, _ = profiles(RES_ALL, k_all if k_all else K_PRIMARY, "all_episodes")
    print(f"\nAll-episode solution (k = {k_all if k_all else K_PRIMARY}) medians, for comparison:")
    print(MED_A[["cluster", "n", "LOS_median", "lines_median", "cost_median", "share_median", "description"]].round(2).to_string(index=False))

# =============================================================================
# 5. RIDGE MULTINOMIAL MODEL OF MEMBERSHIP (M3 block)
# =============================================================================
hdr("5. RIDGE MULTINOMIAL MODEL OF CLUSTER MEMBERSHIP (M3 block)")


def make_cmap(ds, vars_, minn=RARE_MIN):
    mp, seen = {}, {}
    for v in vars_:
        vc = ds[v].astype(str).value_counts()
        rare = [l for l in vc.index if vc[l] < minn]
        m = {}
        if rare:
            total = vc[rare].sum() + (vc["Other"] if ("Other" in vc.index and "Other" not in rare) else 0)
            tgt = "Other" if (total >= minn and v not in ("any_surgical_service", "group", "sex")) else vc.index[0]
            m = {l: tgt for l in rare}
        mp[v] = m
        seen[v] = {m.get(l, l) for l in vc.index}
    return {"map": mp, "seen": seen}


def design(d, cats, refs=None):
    ds = d.copy()
    cm = make_cmap(ds, [v for v in cats if v in COLLAPSE_VARS])
    for v, m in cm["map"].items():
        s0 = ds[v].astype(str)
        ds[v] = s0.map(m).fillna(s0) if m else s0
    cols, names, blocks = [], [], []
    for v in cats:
        s = ds[v].astype(str)
        order = s.value_counts().index.tolist()
        ref = (refs or {}).get(v)
        ref = ref if ref in order else order[0]
        for lev in [x for x in order if x != ref]:
            cols.append((s == lev).to_numpy(float))
            names.append(f"C({v})[T.{lev}] (vs {ref})")
            blocks.append(BLOCK_OF.get(v, "other"))
    X = np.column_stack(cols) if cols else np.zeros((len(ds), 0))
    keep = [j for j in range(X.shape[1]) if X[:, j].std() > 0]
    return X[:, keep], [names[j] for j in keep], [blocks[j] for j in keep]


def ridge_multinom(X, y, K, lam, W0=None):
    n, p = X.shape
    Xa = np.column_stack([np.ones(n), X])
    Y = np.eye(K)[y]

    def f(w):
        W = w.reshape(p + 1, K)
        S = Xa @ W
        lse = logsumexp(S, axis=1)
        nll = -np.mean(np.sum(Y * S, axis=1) - lse)
        P = np.exp(S - lse[:, None])
        G = Xa.T @ (P - Y) / n
        pen = 0.5 * lam * np.sum(W[1:] ** 2)
        G[1:] += lam * W[1:]
        return nll + pen, G.ravel()

    w0 = np.zeros((p + 1) * K) if W0 is None else W0.ravel()
    r = optimize.minimize(f, w0, jac=True, method="L-BFGS-B", options={"maxiter": 2000, "maxfun": 5000})
    return r.x.reshape(p + 1, K)


def predict_proba(X, W):
    S = np.column_stack([np.ones(len(X)), X]) @ W
    return np.exp(S - logsumexp(S, axis=1)[:, None])


def membership_model(d, lab, tag, cats, refs):
    K = lab.max() + 1
    dd = d.reset_index(drop=True)
    X, names, blocks = design(dd, cats + ["period"], refs)
    uid = pd.factorize(dd["cluster_id"])[0]
    u = np.unique(uid)
    np.random.default_rng(SEED).shuffle(u)
    fm = dict(zip(u, np.arange(len(u)) % N_FOLDS))
    fold = np.array([fm[i] for i in uid])

    def cv_loss(Xm, lam):
        ll, acc = [], []
        P = np.zeros((len(Xm), K))
        for k_ in range(N_FOLDS):
            tr, te = fold != k_, fold == k_
            W = ridge_multinom(Xm[tr], lab[tr], K, lam)
            P[te] = predict_proba(Xm[te], W)
        return float(-np.mean(np.log(np.clip(P[np.arange(len(lab)), lab], 1e-12, 1)))), float((P.argmax(1) == lab).mean()), P

    grid = []
    for lam in RIDGE_GRID:
        ll, acc, _ = cv_loss(X, lam)
        grid.append({"lambda": lam, "cv_logloss": ll, "cv_accuracy": acc})
    G = pd.DataFrame(grid)
    lam = float(G.loc[G.cv_logloss.idxmin(), "lambda"])
    ll, acc, P = cv_loss(X, lam)
    prior = np.bincount(lab, minlength=K) / len(lab)
    null_ll = float(-np.mean(np.log(prior[lab])))
    base_acc = float(prior.max())
    W = ridge_multinom(X, lab, K, lam)
    ref_c = 0
    coef = []
    for j, nm in enumerate(names):
        for c in range(1, K):
            b = W[j + 1, c] - W[j + 1, ref_c]
            coef.append({"analysis": tag, "term": nm, "block": blocks[j], "cluster": c + 1, "reference_cluster": 1,
                         "coef_logodds": b, "odds_ratio": float(np.exp(b)), "n_level": int(X[:, j].sum())})
    COEF = pd.DataFrame(coef)
    imp = []
    for blk in sorted(set(blocks)):
        keep = [j for j, b in enumerate(blocks) if b != blk]
        ll_b, acc_b, _ = cv_loss(X[:, keep], lam)
        imp.append({"analysis": tag, "block_dropped": blk, "cv_logloss": ll_b, "increase_vs_full": ll_b - ll, "cv_accuracy": acc_b})
    IMP = pd.DataFrame(imp)
    summ = {"analysis": tag, "n": len(lab), "clusters": K, "lambda": lam, "cv_logloss": ll, "cv_logloss_null": null_ll,
            "logloss_reduction_pct": 100 * (null_ll - ll) / null_ll, "cv_accuracy": acc, "accuracy_largest_cluster_baseline": base_acc,
            "terms": len(names)}
    return summ, G, COEF, IMP


REFS = {"ownership": "Public University-Affiliated", "group": "Hernia", "insurance_fund_model": "Rural Health Insurance Fund",
        "sex": "Male", "age_group": "40-59", "discharge_status_model": "Complete Recovery", "any_surgical_service": "Yes",
        "billing_mode": "Global"}
cats_m3 = [c for c in M3 if not (PRIMARY is RES_ITEM and c == "billing_mode")]
MM_SUM, MM_GRID, MM_COEF, MM_IMP = membership_model(D_PRIM, LAB, PRIMARY["label"], cats_m3, REFS)
save(pd.DataFrame([MM_SUM]), "o4_membership_model_summary.csv")
save(MM_GRID, "o4_membership_model_lambda_grid.csv")
save(MM_COEF, "o4_membership_model_coefficients.csv")
save(MM_IMP, "o4_membership_model_block_importance.csv")
print(pd.DataFrame([MM_SUM]).round(3).to_string(index=False))
print("\nWhich block of factors predicts membership (cross-validated log-loss when the block is dropped; larger = more important):")
print(MM_IMP.round(4).to_string(index=False))
print("\nLargest penalised odds ratios (vs cluster 1). Penalised estimates have no valid CIs: read direction and relative size only.")
cc = MM_COEF[MM_COEF.n_level >= RARE_MIN].copy()
cc["abs_log"] = cc.coef_logodds.abs()
print(cc[cc.block != "time"].sort_values("abs_log", ascending=False).head(14)[["term", "cluster", "odds_ratio", "n_level"]].round(2).to_string(index=False))

# =============================================================================
# 6. STRUCTURE BEYOND THE MARGINAL DISTRIBUTIONS (permutation baseline)
# =============================================================================
hdr("6. IS THERE CLUSTER STRUCTURE BEYOND THE MARGINALS? (permuted-data silhouette baseline)")
rngp = np.random.default_rng(SEED + 1)
Zp0 = PRIMARY["Z"]
perm_rows = []
for k in K_RANGE:
    sils = []
    for _ in range(5):
        Zp = np.column_stack([rngp.permutation(Zp0[:, j]) for j in range(Zp0.shape[1])])
        lab_, _, _ = kmeans_fit(Zp, k, 5, rngp)
        sub = rngp.choice(len(Zp), min(3000, len(Zp)), replace=False)
        sils.append(silhouette(Zp[sub], lab_[sub]).mean() if len(np.unique(lab_[sub])) > 1 else np.nan)
    real = PRIMARY["table"].set_index("k").loc[k, "silhouette"]
    perm_rows.append({"analysis": PRIMARY["label"], "k": k, "silhouette_real": real, "silhouette_permuted_mean": np.nanmean(sils),
                      "silhouette_permuted_max": np.nanmax(sils), "excess_over_permuted_max": real - np.nanmax(sils)})
PERM = pd.DataFrame(perm_rows)
save(PERM, "o4_permutation_silhouette_baseline.csv")
print(PERM.round(3).to_string(index=False))
print("Excess over the permuted maximum > 0 means the clusters reflect dependence between the four inputs, not just their shapes. "
      "Silhouettes of 0.25-0.5 indicate weak-to-reasonable structure; read the profiles as typical patterns, not sharp types.")

# =============================================================================
# 7. SENSITIVITY ANALYSES 1-7
# =============================================================================
hdr("7. SENSITIVITY ANALYSES 1-7 (primary analysis repeated on the altered sample)")
SENS = {1: ("Surgical service only", "keep_sens1_surgical"), 2: ("Exclude Global packages", "keep_sens2_itemised"),
        3: ("Exclude K81", "keep_sens3_not_K81"), 4: ("Exclude flagged extremes", "keep_sens4_no_extremes"),
        5: ("Exclude anonymous episodes", "keep_sens5_not_anonymous"),
        6: ("Exclude 'Doctor of medicine'", "keep_sens6_not_doctor_of_medicine"), 7: ("LOS capped at P95", None)}


def match_clusters(l_ref, l_new):
    """Hungarian matching of new clusters to reference clusters on common episodes; returns matched Jaccard per ref cluster."""
    c = contingency(l_ref, l_new).astype(float)
    ua, ub = c.sum(1)[:, None], c.sum(0)[None, :]
    J = c / np.maximum(ua + ub - c, 1)
    r, cidx = linear_sum_assignment(-J)
    out = np.zeros(c.shape[0])
    out[r] = J[r, cidx]
    return out


def recover(res_new, d_ref_idx, k, ref_lab, ref_Z_index):
    """Compare the new solution at k with the primary solution on common episodes."""
    d_new = res_new["data"]
    common = d_new["episode_no"].isin(ref_Z_index.index)
    l_new = res_new["sols"][k]["labels"][common.to_numpy()]
    l_ref = ref_lab[ref_Z_index.loc[d_new.loc[common, "episode_no"].values].values]
    return ari(l_ref, l_new), match_clusters(l_ref, l_new), int(common.sum())


ref_index = pd.Series(np.arange(len(D_PRIM)), index=D_PRIM["episode_no"].values)
SENS_ROWS, SENS_CL = [], []
if RUN_SENS:
    for s_id, (name, keepcol) in SENS.items():
        base = D_PRIM
        sub = base[base[keepcol]].reset_index(drop=True) if keepcol else base.copy()
        if PRIMARY is RES_ITEM and s_id == 2:
            print(f"  sensitivity 2: identical to the primary sample (itemised only): skipped.")
            continue
        r = analyse(sub, f"sens {s_id}", NBOOT_SENS, los_col="los_capped_p95" if s_id == 7 else "los_capped_p99", verbose=False,
                    seed=SEED + s_id, ks=[K_PRIMARY])
        a, mj, ncom = recover(r, None, K_PRIMARY, LAB, ref_index)
        rowk = r["table"].set_index("k").loc[K_PRIMARY]
        SENS_ROWS.append({"sens": s_id, "name": name, "n": len(sub), "primary_k_passes_rule_in_sens": bool(rowk.passes_rule),
                          "silhouette_at_primary_k": rowk.silhouette, "min_jaccard_at_primary_k": rowk.min_jaccard,
                          "ARI_vs_primary": a, "n_common": ncom, "min_matched_jaccard": float(mj.min())})
        for j, v in enumerate(mj):
            SENS_CL.append({"sens": s_id, "name": name, "primary_cluster": j + 1, "matched_jaccard": float(v)})
        print(f"  sensitivity {s_id}: {name}: n = {len(sub):,}; at k = {K_PRIMARY}: silhouette "
              f"{rowk.silhouette:.3f}, ARI vs primary {a:.2f}, weakest matched cluster {mj.min():.2f}")
    print("  sensitivity 8 (province as a random intercept) does not apply to unsupervised clustering: not run.")
SENSDF, SENSCL = pd.DataFrame(SENS_ROWS), pd.DataFrame(SENS_CL)
if len(SENSDF):
    save(SENSDF, "o4_sensitivity_summary.csv")
    save(SENSCL, "o4_sensitivity_cluster_recovery.csv")
    print("\n" + SENSDF.round(3).to_string(index=False))

# =============================================================================
# 8. EXPLORATORY
# =============================================================================
hdr("8. EXPLORATORY (label as such)")
if RUN_EXTRA:
    # X0: protocol sample (all reconciled episodes, including emergency observation) at the primary k
    try:
        d0 = ALT if PRIMARY is RES_ALL else ALT[ALT.stratum == "Other"].reset_index(drop=True)
        r0 = analyse(d0, "X0 " + ALT_LABEL, NBOOT_SENS, verbose=False, ks=[K_PRIMARY])
        print(f"X0 ({ALT_LABEL}{'' if PRIMARY is RES_ALL else ', itemised only'}), n = {len(d0):,}, k = {K_PRIMARY}: silhouette "
              f"{r0['table'].silhouette.iloc[0]:.3f}, min Jaccard {r0['table'].min_jaccard.iloc[0]:.2f}")
        a0, mj0, nc0 = recover(r0, None, K_PRIMARY, LAB, ref_index)
        print(f"   ARI vs primary on the {nc0:,} common episodes: {a0:.2f}; weakest matched cluster Jaccard {mj0.min():.2f}")
        save(pd.DataFrame([{"sample": ALT_LABEL, "n": len(d0), "k": K_PRIMARY, "silhouette": r0['table'].silhouette.iloc[0],
                            "min_jaccard": r0['table'].min_jaccard.iloc[0], "ARI_vs_primary": a0, "min_matched_jaccard": float(mj0.min())}]),
             "o4_X0_other_sample.csv")
    except Exception as e:
        print(f"X0 failed: {type(e).__name__}: {e}")
    # X1: reconciliation tolerance 10% (patient share = recorded insured / total)
    try:
        adm_t = DALL[is_adm].copy()
        both = adm_t.insured_share_irr.notna() & adm_t.total_cost_irr.gt(0)
        sub = adm_t[both & (adm_t.fin_gap <= 0.10) & adm_t.patient_share.between(0, 1) & adm_t.los_capped_p99.notna()
                    & adm_t.claim_lines.notna() & (adm_t.cost_intl > 0)].reset_index(drop=True)
        if PRIMARY is RES_ITEM:
            sub = sub[sub.stratum == "Other"].reset_index(drop=True)
        r1 = analyse(sub, "X1 tolerance 10%", NBOOT_SENS, verbose=False, ks=[K_PRIMARY])
        a1, mj1, nc1 = recover(r1, None, K_PRIMARY, LAB, ref_index)
        print(f"X1 (reconciliation tolerance 10%), n = {len(sub):,}, k = {K_PRIMARY}: silhouette {r1['table'].silhouette.iloc[0]:.3f}, "
              f"ARI vs primary on {nc1:,} common episodes {a1:.2f}")
        save(pd.DataFrame([{"sample": "tolerance 10%", "n": len(sub), "k": K_PRIMARY, "silhouette": r1['table'].silhouette.iloc[0],
                            "ARI_vs_primary": a1, "min_matched_jaccard": float(mj1.min())}]), "o4_X1_tolerance10.csv")
    except Exception as e:
        print(f"X1 failed: {type(e).__name__}: {e}")

# =============================================================================
# 9. FIGURES
# =============================================================================
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
    for lab_, res in [("All episodes", RES_ALL), ("Itemised only", RES_ITEM)]:
        t = res["table"]
        ax[0].plot(t.k, t.silhouette, "o-", label=lab_)
        ax[1].plot(t.k, t.min_jaccard, "o-", label=lab_)
    ax[0].set(title="Mean silhouette by k", xlabel="k", ylabel="Silhouette")
    ax[1].axhline(MIN_JACCARD, color="k", ls="--", lw=.8)
    ax[1].set(title="Weakest cluster's bootstrap Jaccard", xlabel="k", ylabel="Jaccard")
    ax[0].legend(frameon=False)
    Zc = PRIMARY["Z"] - PRIMARY["Z"].mean(0)
    U, S, Vt = np.linalg.svd(Zc, full_matrices=False)
    pc = Zc @ Vt[:2].T
    sel = np.random.default_rng(SEED).choice(len(pc), min(4000, len(pc)), replace=False)
    sc = ax[2].scatter(pc[sel, 0], pc[sel, 1], c=LAB[sel], cmap="tab10", s=5, alpha=.6)
    ax[2].set(title=f"{PRIMARY['label']}, k = {K_PRIMARY} (first two principal components)", xlabel="PC1", ylabel="PC2")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o4_selection_and_clusters.png", dpi=200)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6.5, 0.7 * K_PRIMARY + 2))
    M = CZ.iloc[:, 1:].to_numpy()
    im = ax.imshow(M, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(4), ["LOS", "Lines", "Cost", "Patient share"])
    ax.set_yticks(range(K_PRIMARY), [f"Cluster {j + 1} (n={int((LAB == j).sum())})" for j in range(K_PRIMARY)])
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:+.1f}", ha="center", va="center", fontsize=9)
    plt.colorbar(im, label="mean z-score")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "o4_profile_heatmap.png", dpi=200)
    plt.close(fig)
except Exception as e:
    print(f"(figures skipped: {type(e).__name__}: {e})")

# =============================================================================
# 10. RELIABILITY VERDICT
# =============================================================================
hdr("9. RELIABILITY VERDICT (A / B / C) per cluster of the primary solution")
rows = []
cl_prim = PRIMARY["clusters"][PRIMARY["clusters"].k == K_PRIMARY].set_index("cluster")
for j in range(K_PRIMARY):
    st = cl_prim.loc[j + 1]
    sens_j = (SENSCL[(SENSCL.primary_cluster == j + 1) & ~((SENSCL.sens == 2) & (PRIMARY is not RES_ITEM))].matched_jaccard
              if len(SENSCL) else pd.Series(dtype=float))   # sens 2 (drop Global) is not a valid recovery test for an all-episode solution
    med_sens = float(sens_j.median()) if len(sens_j) else np.nan
    min_sens = float(sens_j.min()) if len(sens_j) else np.nan
    reasons = []
    if st.share < MIN_CLUSTER_SHARE:
        reasons.append(f"holds only {100 * st.share:.1f}% of episodes")
    if st.jaccard_mean < MIN_JACCARD:
        reasons.append(f"bootstrap Jaccard {st.jaccard_mean:.2f} < {MIN_JACCARD}")
    elif st.jaccard_mean < 0.85:
        reasons.append(f"bootstrap Jaccard {st.jaccard_mean:.2f} (moderately stable)")
    if len(sens_j) and min_sens < 0.6:
        reasons.append(f"recovered poorly in a sensitivity analysis (Jaccard {min_sens:.2f})")
    elif len(sens_j) and med_sens < 0.8:
        reasons.append(f"median recovery across sensitivity analyses {med_sens:.2f}")
    if not PRIMARY_PASSES:
        reasons.append("the solution fails the k-selection rule")
    sil_c = st.silhouette_cluster
    if sil_c < 0.25:
        reasons.append(f"cluster silhouette {sil_c:.2f} (diffuse)")
    if st.share < MIN_CLUSTER_SHARE or st.jaccard_mean < MIN_JACCARD or (len(sens_j) and min_sens < 0.6) or not PRIMARY_PASSES:
        v = "C"
    elif not reasons:
        v = "A"
    else:
        v = "B"
    rows.append({"cluster": j + 1, "n": int(st.n), "share_pct": 100 * st.share, "description": MED.loc[MED.cluster == j + 1, "description"].iloc[0],
                 "bootstrap_jaccard": st.jaccard_mean, "cluster_silhouette": sil_c, "sens_median_matched_jaccard": med_sens,
                 "sens_min_matched_jaccard": min_sens, "verdict": v, "reasons": "; ".join(reasons)})
REL = pd.DataFrame(rows)
save(REL, "o4_reliability_verdict.csv")
print("A = stable under bootstrap and every sensitivity analysis; B = usable but with the weakness named; "
      "C = unstable or too small (do not interpret as a distinct profile).")
print(REL.round(2).to_string(index=False))
sol_sil = PRIMARY["table"].set_index("k").loc[K_PRIMARY, "silhouette"]
pm = PERM.set_index("k").loc[K_PRIMARY]
print(f"\nSolution level: silhouette {sol_sil:.3f} (permuted-data maximum {pm.silhouette_permuted_max:.3f}); passes the "
      f"protocol rule: {PRIMARY_PASSES}; verdict counts: {REL.verdict.value_counts().to_dict()}")

# =============================================================================
# 11. RUN INFO
# =============================================================================
info = {"script_version": VERSION, "run_time": time.strftime("%Y-%m-%d %H:%M:%S"), "input": str(INPUT_CSV),
        "input_sha256": sha256(INPUT_CSV), "primary_sample": SAMPLE_LABEL, "n_primary": int(len(D)), "n_itemised": int(len(D_ITEM)),
        "primary_analysis": PRIMARY["label"], "primary_k": K_PRIMARY, "primary_passes_rule": PRIMARY_PASSES, "decision": why,
        "seed": SEED, "starts": NSTARTS, "boot": NBOOT, "sens_boot": NBOOT_SENS, "boot_starts": BOOT_STARTS,
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
        "choices_left_open_by_protocol": ["billing-mode recovery rule (>= 85% agreement of clusters with billing mode)", "Jaccard stability of a solution = weakest cluster",
                                          "bootstrap with full-sample centres + random starts", "bootstrap silhouette on 1,500-episode subsample",
                                          "ridge penalty by 5-fold patient-grouped CV", "sensitivity 8 not applicable"],
        "deviations_from_protocol": ["reconciled ADMISSIONS (Global + Inpatient) are the primary sample"]}
(OUT_DIR / "o4_run_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"\nDone. Outputs in: {OUT_DIR}")


1. LOAD AND VERIFY   [elapsed 0.0 min]
Rows: 15,601; columns: 70
                                                        step     n
                        Episodes in the frozen analysis file 15601
                                          record type Global  8462
                                       record type Inpatient  6223
                           record type Emergency Observation   889
                   record type Temporary Inpatient Admission    27
                             Admissions (Global + Inpatient) 14685
         reconciled with all four inputs present (O4 sample)  6950
PRIMARY SAMPLE O4: all reconciled episodes (protocol sample)  7709
                                              stratum Global  4276
                                               stratum Other  3433

Inputs (original scale):
sample            input    n  median    p25    p75   mean     sd
   All LOS (capped P99) 7709    1.15   0.70   1.69   1.38   1.21
   All      Claim lines 7709    4.00   2.

In [39]:
"""
Consolidation (protocol robustness rule + RECORD flow).  Affects ALL objectives (O1-O4).

Reads the output folders you already have and writes three tables:
  robustness_table_all_objectives.csv : one row per estimate/cluster with verdict (A/B/C or tier), whether its direction
                                        held in ALL sensitivity analyses, and the protocol call "robust" / "not robust".
  robustness_summary.csv              : counts per objective and verdict.
  master_flow_table.csv               : Stage A-C flow (flow_table.csv, if present) + per-objective sample flows.

Protocol rule: a conclusion is robust only if its direction holds in all sensitivity analyses AND its reliability verdict is A
(O4: Tier 1).  Verdict B/C or any direction change = not robust.  Nothing is recomputed; it only reads your outputs.

Usage:
  python consolidate_robustness_and_flow.py ROOT_FOLDER [--stage-c-dir analysis_ready_final] [--o4 v31|protocol]
ROOT_FOLDER holds objective1_outputs, objective2_outputs, objective3_outputs and objective4_v31_outputs
(and optionally objective4_protocol_outputs).
"""
import argparse
from pathlib import Path
import numpy as np
import pandas as pd

# ---- NOTEBOOK USERS: set these two paths here (leave None to use the command line instead) ----
ROOT_FOLDER = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases\analysis_ready_final"
STAGE_C_DIR = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases\analysis_ready_final"

ap = argparse.ArgumentParser()
ap.add_argument("root", nargs="?", default=None)
ap.add_argument("--stage-c-dir", default=None, help="folder with flow_table.csv from Stage C")
a, _ = ap.parse_known_args()          # ignores the extra arguments Jupyter passes
a.root = ROOT_FOLDER or a.root
a.stage_c_dir = STAGE_C_DIR or a.stage_c_dir
if not a.root:
    raise SystemExit("Set ROOT_FOLDER at the top of the script (or pass the folder on the command line).")
R = Path(a.root)
OUT = R / "consolidated_outputs"
OUT.mkdir(exist_ok=True)


def find(name):
    hits = sorted(R.rglob(name))
    return pd.read_csv(hits[0], encoding="utf-8-sig") if hits else None


rows = []

# ---------------- O1 ----------------
d = find("o1_reliability_summary.csv")
if d is not None:
    for _, r in d.iterrows():
        n_s, n_same = r.get("sens_tested", 0), r.get("sens_same_direction", 0)
        dir_ok = bool(n_s == n_same) if n_s > 0 else np.nan
        rows.append(dict(objective="O1", outcome=r["outcome"], stratum="all", estimate=r["term"],
                         effect=r["ratio"], verdict=r["verdict"], n_sens=n_s, direction_all_sens=dir_ok,
                         reasons=r.get("why", "")))
# ---------------- O2 ----------------
d = find("o2_reliability_verdict.csv")
if d is not None:
    for _, r in d.iterrows():
        rows.append(dict(objective="O2", outcome="cost (ratio)", stratum=r["stratum"], estimate=r["term"],
                         effect=r["ratio"], verdict=r["verdict"], n_sens=r.get("n_sens", np.nan),
                         direction_all_sens=r.get("same_direction_all", np.nan), reasons=r.get("reasons", "")))
# ---------------- O3 ----------------
d = find("o3_reliability_verdict.csv")
sd = find("o3_sensitivity_direction.csv")
if d is not None:
    dirmap = {}
    nmap = {}
    if sd is not None and {"same_sign"}.issubset(sd.columns):
        keys = [c for c in ["stratum", "variable", "level"] if c in sd.columns]
        dirmap = sd.groupby(keys)["same_sign"].all().to_dict()
        nmap = (sd.groupby(keys)["n_sens"].max().to_dict() if "n_sens" in sd.columns else {})
    for _, r in d.iterrows():
        k = (r["stratum"], r["variable"], r["level"])
        rows.append(dict(objective="O3", outcome="patient share (AME, pp)", stratum=r["stratum"],
                         estimate=f'{r["variable"]}: {r["level"]}', effect=r["AME_pp"], verdict=r["verdict"],
                         n_sens=nmap.get(k, np.nan),
                         direction_all_sens=dirmap.get(k, np.nan) if dirmap else np.nan,
                         reasons=r.get("reasons", "")))
# ---------------- O4 (post hoc v3.1 and protocol) ----------------
for tag, fname, label in [("v31", "o4v31_A_cluster_tiers.csv", "O4 (post hoc v3.1)")]:
    d = find(fname)
    if d is not None:
        for _, r in d.iterrows():
            rows.append(dict(objective=label, outcome="joint profile", stratum="itemised reconciled admissions",
                             estimate=f'cluster {r["cluster"]}: {str(r["description"])[:60]}', effect=r["share_pct"],
                             verdict=f'Tier {r["tier"]}', n_sens="", direction_all_sens=bool(r.get("sensitivity_ok", np.nan)),
                             reasons=r.get("reasons", "")))
d = find("o4_reliability_verdict.csv")
if d is not None:
    for _, r in d.iterrows():
        rows.append(dict(objective="O4 (protocol-specified)", outcome="joint profile", stratum="primary solution",
                         estimate=f'cluster {r.get("cluster", "")}', effect=r.get("share_pct", np.nan),
                         verdict=r.get("verdict", ""), n_sens="", direction_all_sens=np.nan,
                         reasons=r.get("reasons", "")))

T = pd.DataFrame(rows)
if T.empty:
    raise SystemExit("No output tables found under " + str(R))


def call(r):
    v = str(r["verdict"])
    top = v in ("A", "Tier 1")
    dirn = r["direction_all_sens"]
    if "province coefficient" in str(r["reasons"]):
        return "descriptive only (province indicator, not a conclusion)" if top else "descriptive only; not robust (verdict " + v + ")"
    if not top:
        return "not robust (verdict " + v + ")"
    if pd.isna(dirn):
        # verdict A is only given when the direction held in every sensitivity analysis run for that estimate
        return "robust (verdict A already requires an unchanged direction)"
    return "robust" if bool(dirn) else "not robust (direction changed in a sensitivity analysis)"


T["protocol_robustness_call"] = T.apply(call, axis=1)
T.to_csv(OUT / "robustness_table_all_objectives.csv", index=False, encoding="utf-8-sig")
S = (T.groupby(["objective", "verdict"]).size().unstack(fill_value=0))
S["n_total"] = S.sum(axis=1)
S["n_robust_by_protocol_rule"] = T[T.protocol_robustness_call.str.startswith("robust")].groupby("objective").size()
S["n_province_descriptive"] = T[T.protocol_robustness_call.str.startswith("descriptive")].groupby("objective").size()
S = S.fillna(0).astype(int)
S.to_csv(OUT / "robustness_summary.csv", encoding="utf-8-sig")
print("\nROBUSTNESS SUMMARY (verdict counts per objective)\n", S.to_string())
print("\nRows whose conclusion is robust by the protocol rule:")
print(T[T.protocol_robustness_call.str.startswith("robust")][["objective", "stratum", "estimate", "effect", "verdict"]].to_string(index=False))

# ---------------- master flow ----------------
F = []
sc = Path(a.stage_c_dir) / "flow_table.csv" if a.stage_c_dir else None
hits = [sc] if sc is not None and sc.exists() else sorted(R.rglob("flow_table.csv"))
if hits:
    f = pd.read_csv(hits[0], encoding="utf-8-sig")
    f.insert(0, "stage", "Stages A-C (build and cleaning)")
    f.columns = ["stage"] + list(f.columns[1:])
    F.append(f)
else:
    print("\n[note] Stage C flow_table.csv not found: pass --stage-c-dir to include the build/cleaning flow.")
for name, lab in [("o1_sample_flow.csv", "Objective 1"), ("o2_sample_flow.csv", "Objective 2"),
                  ("o3_sample_flow.csv", "Objective 3"), ("o4v31_sample_flow.csv", "Objective 4 (post hoc v3.1)"),
                  ("o4_sample_flow.csv", "Objective 4 (protocol-specified)")]:
    f = find(name)
    if f is not None:
        f.insert(0, "stage", lab)
        F.append(f)
M = pd.concat(F, ignore_index=True, sort=False)
M.to_csv(OUT / "master_flow_table.csv", index=False, encoding="utf-8-sig")
print("\nMASTER FLOW TABLE written:", len(M), "rows ->", OUT / "master_flow_table.csv")
print("Outputs in:", OUT)


ROBUSTNESS SUMMARY (verdict counts per objective)
 verdict                   A   B    C  Tier 2  n_total  n_robust_by_protocol_rule  n_province_descriptive
objective                                                                                               
O1                       30  12  117       0      159                         12                      81
O2                        9   4    4       0       17                          9                       0
O3                        8  13    5       0       26                          8                       0
O4 (post hoc v3.1)        0   0    0       3        3                          0                       0
O4 (protocol-specified)   2   0    1       0        3                          2                       0

Rows whose conclusion is robust by the protocol rule:
              objective                           stratum                                                                estimate    effect verdict
          

In [45]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Stage B audit of the RAW CLAIM-LINE Excel file (before episodes are built).

  B1  Exact duplicate claim lines (identical in every column, across sheets): counted, cost reported.
  B2  LOS from Jalali dates vs recorded LOS, per episode (recorded = maximum over lines, as in the build script).
  B3  Missing patient IDs: share of lines / episodes per sheet that would be keyed by a signature (anonymous).

It only AUDITS; it does not build episodes.

HOW TO RUN
  Jupyter / Spyder / VS Code interactive:  edit the CONFIG block below, then run the whole cell/file.
  Anaconda Prompt:  python stage_b_raw_checks.py "C:\\path\\raw_claims.xlsx" --out stage_b_outputs

IMPORTANT: RAW_FILE must be the RAW claim-line Excel file (Persian headers, Jalali dates),
NOT digestive_surgical_episodes_english.csv (that file is already episode-level and has no dates).
"""
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# =============================== CONFIG (edit here when running in Jupyter) ===============================
RAW_FILE = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Diseases_of_the_digestive_syste.xlsx"
OUT_DIR = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases\stage_b_outputs"
MAP_JSON = None      # optional path to a json file {"canonical": ["header1", "header2"]}
LOS_TOL = 1.0        # days of disagreement counted as a mismatch
# ============================================================================================================

CODES = ["K40", "K41", "K42", "K43", "K45", "K46", "K80", "K81", "K60", "K61", "K62", "K64"]
REQUIRED = ["icd10", "specialty", "adm_date", "dis_date", "total_cost"]
# canonical -> accepted headers (after text normalisation). If sheets are skipped, extend these with the
# headers printed under "Headers that were not mapped" (or copy the aliases from build_episodes_digestive_surgical.py).
ALIASES = {
    "patient_id": ["patient_id", "patient_no", "NATURAL_PERSON_NATIONAL_CODE2", "شناسه بیمار", "کد بیمار", "شماره بیمار", "کد ملی"],
    "icd10": ["icd10", "icd_10", "کد تشخیص", "تشخیص", "کد بیماری"],
    "specialty": ["specialty", "تخصص", "تخصص پزشک"],
    "adm_date": ["adm_date", "admission_date", "تاریخ پذیرش", "تاریخ بستری"],
    "dis_date": ["dis_date", "discharge_date", "تاریخ ترخیص"],
    "total_cost": ["total_cost", "total_cost_irr", "هزینه کل - ریال", "هزینه کل", "جمع کل", "مبلغ کل"],
    "insured_share": ["insured_share", "سهم بیمه شده - ریال", "سهم بیمه شده", "سهم بیمار"],
    "requested_share": ["requested_share", "مجموع سهم های درخواستی - ریال", "سهم درخواستی", "مبلغ درخواستی"],
    "paid_share": ["paid_share", "مجموع سهم های پرداختی - ریال", "سهم پرداختی", "مبلغ پرداختی"],
    "los": ["los", "los_days", "طول درمان", "مدت اقامت", "طول مدت بستری", "روز بستری"],
    "age": ["age", "سن"],
    "sex": ["sex", "gender", "جنسیت", "جنس"],
    "ownership": ["ownership", "مالکیت", "نوع مالکیت"],
    "fund": ["fund", "insurance_fund", "صندوق", "صندوق بیمه"],
    "record_type": ["record_type", "نوع پرونده", "نوع رکورد"],
    "province": ["province", "استان"],
    "service_group": ["service_group", "گروه خدمت", "گروه خدمات"],
}
PERSIAN_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def norm_text(x):
    if pd.isna(x):
        return x
    s = str(x).replace("ي", "ی").replace("ك", "ک").replace("\u200c", " ").translate(PERSIAN_DIGITS)
    return re.sub(r"\s+", " ", s).strip()


# ---------------------------------------------------------------- Jalali -> day number (jalaali-js algorithm)
_BREAKS = [-61, 9, 38, 199, 426, 686, 756, 818, 1111, 1181, 1210, 1635, 2060, 2097, 2192, 2262, 2324, 2394, 2456, 3178]


def _div(a, b):
    return int(a / b)  # truncation toward zero, as in jalaali-js


def _mod(a, b):
    return a - _div(a, b) * b


def _jal_cal(jy):
    gy = jy + 621
    leap_j = -14
    jp = _BREAKS[0]
    jump = 0
    for jm in _BREAKS[1:]:
        jump = jm - jp
        if jy < jm:
            break
        leap_j += _div(jump, 33) * 8 + _div(_mod(jump, 33), 4)
        jp = jm
    n = jy - jp
    leap_j += _div(n, 33) * 8 + _div(_mod(n, 33) + 3, 4)
    if _mod(jump, 33) == 4 and jump - n == 4:
        leap_j += 1
    leap_g = _div(gy, 4) - _div((_div(gy, 100) + 1) * 3, 4) - 150
    march = 20 + leap_j - leap_g
    return gy, march


def _g2d(gy, gm, gd):
    d = _div((gy + _div(gm - 8, 6) + 100100) * 1461, 4) + _div(153 * _mod(gm + 9, 12) + 2, 5) + gd - 34840408
    d = d - _div(_div(gy + 100100 + _div(gm - 8, 6), 100) * 3, 4) + 752
    return d


def j2d(jy, jm, jd):
    """Day number of a Jalali date; differences between two dates are exact days."""
    gy, march = _jal_cal(jy)
    return _g2d(gy, 3, march) + (jm - 1) * 31 - _div(jm, 7) * (jm - 7) + jd - 1


def _selftest_calendar():
    # 1399 and 1403 are leap years, 1398 and 1400 are not
    ok = (j2d(1399, 1, 1) - j2d(1398, 1, 1) == 365 and j2d(1400, 1, 1) - j2d(1399, 1, 1) == 366
          and j2d(1404, 1, 1) - j2d(1403, 1, 1) == 366 and j2d(1403, 1, 1) - j2d(1402, 1, 1) == 365)
    if not ok:
        print("WARNING: Jalali calendar self-test failed; date-based LOS may be wrong.")


def parse_jalali_to_daynum(s):
    """Series of Jalali date strings ('1402/05/12', '1402-5-12', '14020512', Persian digits ok) -> float day numbers."""
    t = s.map(norm_text).astype("string")
    parts = t.str.extract(r"^\s*(\d{4})\D?(\d{1,2})\D?(\d{1,2})")
    y, m, d = (pd.to_numeric(parts[i], errors="coerce") for i in range(3))
    ok = y.between(1300, 1500) & m.between(1, 12) & d.between(1, 31)
    key = pd.DataFrame({"y": y.where(ok), "m": m.where(ok), "d": d.where(ok)})
    uniq = key.dropna().drop_duplicates()
    lut = {(int(r.y), int(r.m), int(r.d)): j2d(int(r.y), int(r.m), int(r.d)) for r in uniq.itertuples()}  # few unique dates
    return pd.Series([lut.get((int(a), int(b), int(c)), np.nan) if pd.notna(a) else np.nan
                      for a, b, c in zip(key.y, key.m, key.d)], index=s.index, dtype="float64")


# ---------------------------------------------------------------- main
def run_stage_b(raw, out_dir="stage_b_outputs", map_json=None, los_tol=1.0):
    raw = Path(raw)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    _selftest_calendar()

    if not raw.exists():
        folder = raw.parent
        print(f"STOP: file not found: {raw}")
        if folder.exists():
            found = sorted(p.name for p in folder.iterdir() if p.suffix.lower() in (".xlsx", ".xls", ".xlsm"))
            print("Excel files in that folder:", found if found else "none")
        return None
    if raw.suffix.lower() == ".csv":
        head = pd.read_csv(raw, nrows=2)
        if "episode_no" in head.columns:
            print("STOP: this is the EPISODE-level CSV (it has episode_no and no dates). Stage B needs the RAW claim-line "
                  "Excel file (the input of build_episodes_digestive_surgical.py).")
            return None

    aliases = {k: [norm_text(v) for v in vs] for k, vs in ALIASES.items()}
    if map_json:
        for k, vs in json.loads(Path(map_json).read_text(encoding="utf-8")).items():
            aliases[k] = aliases.get(k, []) + [norm_text(v) for v in vs]

    xl = pd.read_excel(raw, sheet_name=None, dtype=str)
    frames, skipped, unmapped = [], [], {}
    for sh, df in xl.items():
        df = df.copy()
        df.columns = [norm_text(c) for c in df.columns]
        ren = {}
        for canon, al in aliases.items():
            al_l = [str(x).lower() for x in al]
            hit = [c for c in df.columns if str(c).lower() in al_l]
            if hit:
                ren[hit[0]] = canon
        unmapped[sh] = [c for c in df.columns if c not in ren]
        df = df.rename(columns=ren)
        miss = [c for c in REQUIRED if c not in df.columns]
        if miss:
            skipped.append({"sheet": sh, "missing_required": ", ".join(miss), "rows": len(df)})
            continue
        df["sheet"] = sh
        frames.append(df)
    if skipped:
        print("Sheets skipped (missing required columns):")
        print(pd.DataFrame(skipped).to_string(index=False))
        print("\nHeaders that were not mapped, per sheet (use them to extend ALIASES):")
        for sh, cols in unmapped.items():
            print(f"  {sh}: {cols}")
    if not frames:
        print("STOP: no sheet has all required columns. Extend ALIASES (or use MAP_JSON) with the headers printed above.")
        return None

    L = pd.concat(frames, ignore_index=True)
    for c in L.columns:
        if c != "sheet":
            L[c] = L[c].map(norm_text)
    L["icd10"] = L["icd10"].astype("string").str.upper().str.extract(r"([A-Z]\d{2})", expand=False)
    n_read = len(L)
    read = L.groupby("sheet").size().rename("lines_read")
    L = L[L["icd10"].isin(CODES)].copy()
    kept = L.groupby("sheet").size().rename("lines_kept")
    lines_tab = pd.concat([read, kept], axis=1).fillna(0).astype(int)
    print(f"\nLines read (sheets with required columns): {n_read:,}; kept (12 codes): {len(L):,}")
    print(lines_tab.to_string())
    for c in ["total_cost", "insured_share", "requested_share", "paid_share", "los", "age"]:
        if c in L.columns:
            L[c] = pd.to_numeric(L[c].astype("string").str.replace(",", "", regex=False), errors="coerce")

    # ------------------------------------------------------------ B1 duplicates
    cols = [c for c in L.columns if c != "sheet"]  # every column, mapped or not
    dup_any = L.duplicated(subset=cols, keep="first")
    dup_tab = pd.DataFrame({"lines": L.groupby("sheet").size(),
                            "duplicate_lines": dup_any.groupby(L["sheet"]).sum(),
                            "duplicate_cost_irr": L.loc[dup_any].groupby("sheet")["total_cost"].sum()}).fillna(0)
    dup_tab["pct_duplicate"] = 100 * dup_tab.duplicate_lines / dup_tab.lines
    h = pd.util.hash_pandas_object(L[cols].astype(str), index=False)
    first_sheet = L.assign(h=h).groupby("h")["sheet"].first()
    cross = int((h[dup_any].map(first_sheet).values != L.loc[dup_any, "sheet"].values).sum())
    tot_cost = max(float(L["total_cost"].sum()), 1.0)
    print("\nB1. EXACT DUPLICATE CLAIM LINES")
    print(dup_tab.round(2).to_string())
    print(f"Total duplicate lines: {int(dup_any.sum()):,} of {len(L):,} ({100 * dup_any.mean():.2f}%); "
          f"first copy on a different sheet: {cross:,}; cost double-counted if kept: "
          f"{dup_tab.duplicate_cost_irr.sum():,.0f} IRR ({100 * dup_tab.duplicate_cost_irr.sum() / tot_cost:.2f}% of total).")
    print("Caution: an identical line inside one episode can be a genuine repeated service. Drop only lines identical on ALL "
          "columns including any claim/line identifier. Report both counts (all duplicates; cross-sheet duplicates).")
    dup_tab.reset_index().to_csv(out / "stage_b1_duplicates_by_sheet.csv", index=False, encoding="utf-8-sig")
    Ld = L[~dup_any].copy()

    # ------------------------------------------------------------ episode key (same logic as the build script)
    pid = Ld["patient_id"] if "patient_id" in Ld.columns else pd.Series(pd.NA, index=Ld.index)
    Ld["anonymous"] = (pid.isna() | (pid.astype("string").str.strip() == "")).to_numpy()
    sig_cols = [c for c in ["adm_date", "dis_date", "age", "sex", "ownership", "fund", "record_type", "province", "icd10"]
                if c in Ld.columns]
    sig = Ld[sig_cols[0]].astype(str)
    for c in sig_cols[1:]:
        sig = sig + "|" + Ld[c].astype(str)
    pidkey = pid.astype(str) + "|" + Ld["adm_date"].astype(str) + "|" + Ld["dis_date"].astype(str)
    Ld["episode_key"] = np.where(Ld["anonymous"], "SIG|" + sig, "PID|" + pidkey)

    # ------------------------------------------------------------ B2 LOS from dates
    Ld["adm_dn"] = parse_jalali_to_daynum(Ld["adm_date"])
    Ld["dis_dn"] = parse_jalali_to_daynum(Ld["dis_date"])
    Ld["los_dates"] = Ld["dis_dn"] - Ld["adm_dn"]
    print("\nB2. LOS FROM DATES vs RECORDED LOS (per episode)")
    print(f"Lines with an unparseable Jalali date: {int(Ld['los_dates'].isna().sum()):,}; "
          f"discharge before admission: {int((Ld['los_dates'] < 0).sum()):,}")
    if "los" in Ld.columns:
        E = (Ld.groupby("episode_key")
             .agg(los_dates=("los_dates", "first"), los_recorded=("los", "max"), lines=("los", "size"),
                  anonymous=("anonymous", "first"), sheet=("sheet", "first")).reset_index())
        E = E.dropna(subset=["los_dates", "los_recorded"]).copy()
        E["diff"] = E["los_recorded"] - E["los_dates"]
        E["mismatch"] = E["diff"].abs() > los_tol
        E["same_day_but_recorded_positive"] = (E["los_dates"] == 0) & (E["los_recorded"] > 0)
        cmp_tab = pd.DataFrame([{
            "episodes_compared": len(E),
            "correlation": E[["los_dates", "los_recorded"]].corr().iloc[0, 1],
            "median_diff_recorded_minus_dates": E["diff"].median(),
            "mean_diff": E["diff"].mean(),
            "pct_within_0.5d": 100 * (E["diff"].abs() <= 0.5).mean(),
            f"pct_mismatch_gt_{los_tol:g}d": 100 * E["mismatch"].mean(),
            "pct_recorded_fractional": 100 * ((E["los_recorded"] % 1) != 0).mean(),
            "pct_same_day_but_recorded_positive": 100 * E["same_day_but_recorded_positive"].mean()}])
        print(cmp_tab.round(3).T.to_string(header=False))
        print("Reading: overnight stay = recorded LOS >= 1 day (fractional). Date-based LOS is an integer (same-day = 0). "
              "If they differ systematically, report overnight-stay results under both definitions.")
        cmp_tab.to_csv(out / "stage_b2_los_comparison_summary.csv", index=False, encoding="utf-8-sig")
        E.drop(columns=["episode_key"]).to_csv(out / "stage_b2_los_by_episode_no_ids.csv", index=False, encoding="utf-8-sig")
        by_sheet = E.groupby("sheet").agg(episodes=("diff", "size"), pct_mismatch=("mismatch", "mean"),
                                          median_diff=("diff", "median"))
        by_sheet["pct_mismatch"] *= 100
        by_sheet.reset_index().to_csv(out / "stage_b2_los_mismatch_by_sheet.csv", index=False, encoding="utf-8-sig")
        print(by_sheet.round(2).to_string())
    else:
        print("No recorded LOS column was mapped: only date-based LOS can be computed (extend ALIASES['los']).")
    print("\nTo carry this into the analysis file, add to the build script BEFORE the dates are dropped (episode level):\n"
          "    los_from_dates = j2d(dis_y, dis_m, dis_d) - j2d(adm_y, adm_m, adm_d)   # same for every line of the episode\n"
          "    # keep the recorded LOS column too and report both. Copy j2d() from this script.")

    # ------------------------------------------------------------ B3 anonymous
    print("\nB3. MISSING PATIENT IDs (episodes keyed by a signature)")
    anon_lines = Ld.groupby("sheet")["anonymous"].agg(lines="size", anonymous_lines="sum")
    anon_lines["pct_anonymous_lines"] = 100 * anon_lines.anonymous_lines / anon_lines.lines
    ep = Ld.groupby(["sheet", "episode_key"])["anonymous"].first().reset_index()
    anon_eps = ep.groupby("sheet")["anonymous"].agg(episodes="size", anonymous_episodes="sum")
    anon_eps["pct_anonymous_episodes"] = 100 * anon_eps.anonymous_episodes / anon_eps.episodes
    A = anon_lines.join(anon_eps)
    print(A.round(2).to_string())
    print(f"Overall: {100 * Ld['anonymous'].mean():.2f}% of lines; {100 * ep['anonymous'].mean():.2f}% of episodes.")
    sigl = Ld[Ld["anonymous"]].groupby("episode_key").size()
    if len(sigl):
        print(f"Signature episodes with unusually many lines (> 30): {int((sigl > 30).sum()):,} of {len(sigl):,} "
              "(possible merges of separate admissions).")
    A.reset_index().to_csv(out / "stage_b3_anonymous_by_sheet.csv", index=False, encoding="utf-8-sig")
    lines_tab.reset_index().to_csv(out / "stage_b0_lines_by_sheet.csv", index=False, encoding="utf-8-sig")

    summary = {"lines_read": int(n_read), "lines_kept": int(len(L)), "duplicate_lines": int(dup_any.sum()),
               "duplicate_lines_cross_sheet": cross, "duplicate_cost_irr": float(dup_tab.duplicate_cost_irr.sum()),
               "pct_anonymous_lines": float(100 * Ld["anonymous"].mean()),
               "pct_anonymous_episodes": float(100 * ep["anonymous"].mean()), "sheets_skipped": skipped}
    (out / "stage_b_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nDone. Outputs in {out.resolve()}. Put the counts in the flow table (lines read / kept / duplicates dropped).")
    return summary


if __name__ == "__main__":
    in_notebook = "ipykernel" in sys.modules
    if len(sys.argv) > 1 and not in_notebook:  # command-line use
        import argparse
        ap = argparse.ArgumentParser()
        ap.add_argument("raw")
        ap.add_argument("--out", default="stage_b_outputs")
        ap.add_argument("--map-json", default=None)
        ap.add_argument("--los-tol", type=float, default=1.0)
        a = ap.parse_args()
        run_stage_b(a.raw, a.out, a.map_json, a.los_tol)
    else:  # Jupyter / interactive: use the CONFIG block at the top
        run_stage_b(RAW_FILE, OUT_DIR, MAP_JSON, LOS_TOL)


Lines read (sheets with required columns): 317,702; kept (12 codes): 93,422
        lines_read  lines_kept
sheet                         
Sheet1      317702       93422

B1. EXACT DUPLICATE CLAIM LINES
        lines  duplicate_lines  duplicate_cost_irr  pct_duplicate
sheet                                                            
Sheet1  93422                0                 0.0            0.0
Total duplicate lines: 0 of 93,422 (0.00%); first copy on a different sheet: 0; cost double-counted if kept: 0 IRR (0.00% of total).
Caution: an identical line inside one episode can be a genuine repeated service. Drop only lines identical on ALL columns including any claim/line identifier. Report both counts (all duplicates; cross-sheet duplicates).

B2. LOS FROM DATES vs RECORDED LOS (per episode)
Lines with an unparseable Jalali date: 0; discharge before admission: 0
episodes_compared                   15601.000
correlation                             0.990
median_diff_recorded_minus_dates

In [46]:
   import pandas as pd
   df = pd.read_csv(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Digestive_diseases\digestive_surgical_episodes_english.csv")
   print(df.groupby(["adm_year", "adm_month"]).size())
   print(df["adm_year"].value_counts(normalize=True).mul(100).round(1))

adm_year  adm_month
1400      12            10
1401      1            429
          2            624
          3            683
          4            585
          5            638
          6            653
          7            530
          8            631
          9            583
          10           651
          11           674
          12           615
1402      1            474
          2            649
          3            795
          4            687
          5            817
          6            660
          7            653
          8            731
          9            655
          10           769
          11           759
          12           646
dtype: int64
adm_year
1402    53.2
1401    46.8
1400     0.1
Name: proportion, dtype: float64
